# Data Loading and Readiness Verification

## Environment Setup and Input File Inventory

### Inspect uploaded files

In [10]:
from pathlib import Path
import os

# Colab upload directory
DATA_DIR = Path("/content")

# Expected output directory
OUTPUT_DIR = Path("/content/phase3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inspect files available in the Colab environment
available_files = sorted(
    [
        {
            "file_name": file.name,
            "extension": file.suffix.lower(),
            "size_mb": round(file.stat().st_size / (1024 ** 2), 3),
            "file_path": str(file)
        }
        for file in DATA_DIR.iterdir()
        if file.is_file()
    ],
    key=lambda x: x["file_name"].lower()
)

print("=" * 75)
print("SUPERR SELECTOR SEASON 2 — PHASE 3")
print("SECTION 3A.1 — INPUT FILE INVENTORY")
print("=" * 75)

print(f"\nFiles detected: {len(available_files)}")
print(f"Output directory: {OUTPUT_DIR}\n")

for index, file_info in enumerate(available_files, start=1):
    print(
        f"{index:02d}. {file_info['file_name']}"
        f" | Type: {file_info['extension'] or 'No extension'}"
        f" | Size: {file_info['size_mb']} MB"
    )

print("\n" + "=" * 75)
print("FILE INVENTORY COMPLETED")
print("=" * 75)


SUPERR SELECTOR SEASON 2 — PHASE 3
SECTION 3A.1 — INPUT FILE INVENTORY

Files detected: 14
Output directory: /content/phase3_outputs

01. ball_by_ball.csv | Type: .csv | Size: 39.169 MB
02. competition_team_player_mapping.csv | Type: .csv | Size: 0.442 MB
03. competitions.csv | Type: .csv | Size: 0.007 MB
04. matches.csv | Type: .csv | Size: 0.118 MB
05. Phase 0 report.md | Type: .md | Size: 0.017 MB
06. Phase 2 report.md | Type: .md | Size: 0.009 MB
07. phase2_completion_summary.csv | Type: .csv | Size: 0.0 MB
08. phase2_exception_register.csv | Type: .csv | Size: 0.007 MB
09. phase2_integrated_cleaned_deliveries.parquet | Type: .parquet | Size: 3.297 MB
10. phase2_legal_ball_evidence.csv | Type: .csv | Size: 50.938 MB
11. phase2_unexplained_extended_over_register.csv | Type: .csv | Size: 0.004 MB
12. playing_eleven.csv | Type: .csv | Size: 0.922 MB
13. SupeRR_Selector_S2_Phase_1_Report.docx | Type: .docx | Size: 0.043 MB
14. teams.csv | Type: .csv | Size: 0.007 MB

FILE INVENTORY COM

### Load all analytical and source datasets

In [11]:
# Install the package required to read the Phase 1 DOCX report

%pip install -q python-docx

# Verify installation
from docx import Document

print("python-docx installed and imported successfully.")


python-docx installed and imported successfully.


In [12]:
import pandas as pd
from pathlib import Path
from docx import Document

BASE_DIR = Path("/content")

# ------------------------------------------------------------
# 1. Define all input paths
# ------------------------------------------------------------

file_paths = {
    "integrated": BASE_DIR / "phase2_integrated_cleaned_deliveries.parquet",
    "legal_evidence": BASE_DIR / "phase2_legal_ball_evidence.csv",
    "exceptions": BASE_DIR / "phase2_exception_register.csv",
    "extended_overs": BASE_DIR / "phase2_unexplained_extended_over_register.csv",
    "completion": BASE_DIR / "phase2_completion_summary.csv",
    "source_ball": BASE_DIR / "ball_by_ball.csv",
    "source_matches": BASE_DIR / "matches.csv",
    "phase0": BASE_DIR / "Phase 0 report.md",
    "phase1": BASE_DIR / "SupeRR_Selector_S2_Phase_1_Report.docx",
    "phase2": BASE_DIR / "Phase 2 report.md"
}

# ------------------------------------------------------------
# 2. Verify file availability
# ------------------------------------------------------------

missing_files = [
    name for name, path in file_paths.items()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"The following required files are missing: {missing_files}"
    )

print("=" * 75)
print("PHASE 3A.2 — INPUT LOADING")
print("=" * 75)

# ------------------------------------------------------------
# 3. Load Phase 2 analytical datasets
# ------------------------------------------------------------

df_deliveries = pd.read_parquet(file_paths["integrated"])

df_legal_evidence = pd.read_csv(
    file_paths["legal_evidence"],
    low_memory=False
)

df_exception_register = pd.read_csv(
    file_paths["exceptions"],
    low_memory=False
)

df_extended_over_register = pd.read_csv(
    file_paths["extended_overs"],
    low_memory=False
)

df_completion_summary = pd.read_csv(
    file_paths["completion"],
    low_memory=False
)

# ------------------------------------------------------------
# 4. Load original source datasets separately
# ------------------------------------------------------------

df_source_ball = pd.read_csv(
    file_paths["source_ball"],
    low_memory=False
)

df_source_matches = pd.read_csv(
    file_paths["source_matches"],
    low_memory=False
)

# ------------------------------------------------------------
# 5. Load project reports
# ------------------------------------------------------------

phase0_report = file_paths["phase0"].read_text(
    encoding="utf-8"
)

phase1_document = Document(file_paths["phase1"])

phase1_report_text = "\n".join(
    paragraph.text
    for paragraph in phase1_document.paragraphs
)

phase2_report = file_paths["phase2"].read_text(
    encoding="utf-8"
)

# ------------------------------------------------------------
# 6. Report dataset dimensions
# ------------------------------------------------------------

loaded_datasets = {
    "Integrated deliveries": df_deliveries,
    "Legal-ball evidence": df_legal_evidence,
    "Exception register": df_exception_register,
    "Extended-over register": df_extended_over_register,
    "Completion summary": df_completion_summary,
    "Original ball-by-ball": df_source_ball,
    "Original matches": df_source_matches
}

print("\nDATASET DIMENSIONS")
print("-" * 75)

for name, dataframe in loaded_datasets.items():
    print(
        f"{name:<30} "
        f"{dataframe.shape[0]:>10,} rows × "
        f"{dataframe.shape[1]:>3,} columns"
    )

# ------------------------------------------------------------
# 7. Report project report loading status
# ------------------------------------------------------------

print("\nPROJECT REPORTS")
print("-" * 75)

print(f"Phase 0: {len(phase0_report):,} characters")
print(f"Phase 1: {len(phase1_report_text):,} characters")
print(f"Phase 2: {len(phase2_report):,} characters")

# ------------------------------------------------------------
# 8. Inspect integrated dataset schema
# ------------------------------------------------------------

print("\nINTEGRATED DATASET SCHEMA")
print("-" * 75)

for index, column in enumerate(df_deliveries.columns, start=1):
    print(f"{index:02d}. {column:<35} {df_deliveries[column].dtype}")

# ------------------------------------------------------------
# 9. Inspect original match metadata schema
# ------------------------------------------------------------

print("\nORIGINAL MATCHES SCHEMA")
print("-" * 75)

for column in df_source_matches.columns:
    print(f"{column:<35} {df_source_matches[column].dtype}")

# ------------------------------------------------------------
# 10. Inspect memory usage
# ------------------------------------------------------------

print("\nMEMORY USAGE")
print("-" * 75)

for name, dataframe in loaded_datasets.items():
    memory_mb = (
        dataframe.memory_usage(deep=True).sum() / (1024 ** 2)
    )
    print(f"{name:<30} {memory_mb:>10.2f} MB")

print("\n" + "=" * 75)
print("INPUT LOADING COMPLETED")
print("=" * 75)

PHASE 3A.2 — INPUT LOADING

DATASET DIMENSIONS
---------------------------------------------------------------------------
Integrated deliveries             192,235 rows ×  61 columns
Legal-ball evidence               192,235 rows ×  36 columns
Exception register                     24 rows ×  17 columns
Extended-over register                147 rows ×  12 columns
Completion summary                     14 rows ×   2 columns
Original ball-by-ball             200,999 rows ×  33 columns
Original matches                      883 rows ×  22 columns

PROJECT REPORTS
---------------------------------------------------------------------------
Phase 0: 17,427 characters
Phase 1: 10,750 characters
Phase 2: 9,391 characters

INTEGRATED DATASET SCHEMA
---------------------------------------------------------------------------
01. match_id                            int64
02. innings                             int64
03. overid                              int64
04. over_ball_num                   

### Primary population and delivery-key verification

In [13]:
import pandas as pd

print("=" * 75)
print("PHASE 3A.3 — PRIMARY POPULATION INTEGRITY AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Expected population
# ------------------------------------------------------------

EXPECTED_DELIVERIES = 192235
EXPECTED_MATCHES = 813

# ------------------------------------------------------------
# 2. Basic population counts
# ------------------------------------------------------------

actual_deliveries = len(df_deliveries)
actual_matches = df_deliveries["match_id"].nunique()

print("\n1. POPULATION COUNTS")
print("-" * 75)

print(f"Expected regulation deliveries : {EXPECTED_DELIVERIES:,}")
print(f"Observed regulation deliveries : {actual_deliveries:,}")
print(f"Expected eligible matches      : {EXPECTED_MATCHES:,}")
print(f"Observed distinct matches      : {actual_matches:,}")

# ------------------------------------------------------------
# 3. Innings population
# ------------------------------------------------------------

print("\n2. INNINGS DISTRIBUTION")
print("-" * 75)

innings_distribution = (
    df_deliveries["innings"]
    .value_counts(dropna=False)
    .sort_index()
)

print(innings_distribution.to_string())

observed_innings = set(
    df_deliveries["innings"].dropna().unique()
)

print(f"\nObserved innings values: {sorted(observed_innings)}")

# ------------------------------------------------------------
# 4. Delivery-key integrity
# ------------------------------------------------------------

print("\n3. DELIVERY-KEY INTEGRITY")
print("-" * 75)

delivery_key = [
    "match_id",
    "innings",
    "innings_ball_num"
]

missing_key_values = (
    df_deliveries[delivery_key]
    .isna()
    .sum()
)

duplicate_key_count = (
    df_deliveries
    .duplicated(subset=delivery_key)
    .sum()
)

print("Missing values in delivery-key columns:")
print(missing_key_values.to_string())

print(f"\nDuplicate delivery keys: {duplicate_key_count:,}")

# ------------------------------------------------------------
# 5. Competition and format distribution
# ------------------------------------------------------------

print("\n4. COMPETITION DISTRIBUTION")
print("-" * 75)

competition_distribution = (
    df_deliveries.groupby(
        ["match_format", "comp_id", "comp_name"],
        dropna=False
    )["match_id"]
    .nunique()
    .reset_index(name="distinct_matches")
)

format_distribution = (
    df_deliveries.groupby("match_format", dropna=False)["match_id"]
    .nunique()
    .sort_index()
)

print("Distinct matches by match format:")
print(format_distribution.to_string())

print("\nDistinct competitions represented:")
print(df_deliveries["comp_id"].nunique())

# ------------------------------------------------------------
# 6. Team identity completeness
# ------------------------------------------------------------

print("\n5. TEAM IDENTITY COMPLETENESS")
print("-" * 75)

team_columns = [
    "batting_team_id",
    "bowling_team_id",
    "batting_team_name",
    "bowling_team_name"
]

team_missingness = df_deliveries[team_columns].isna().sum()

print(team_missingness.to_string())

# ------------------------------------------------------------
# 7. Match reference integrity
# ------------------------------------------------------------

print("\n6. MATCH REFERENCE INTEGRITY")
print("-" * 75)

integrated_match_ids = set(df_deliveries["match_id"].unique())
source_match_ids = set(df_source_matches["match_id"].unique())

unmatched_match_ids = integrated_match_ids - source_match_ids
unused_source_match_ids = source_match_ids - integrated_match_ids

print(f"Integrated distinct matches : {len(integrated_match_ids):,}")
print(f"Original distinct matches   : {len(source_match_ids):,}")
print(f"Integrated IDs not in source: {len(unmatched_match_ids):,}")
print(f"Source IDs not in integrated: {len(unused_source_match_ids):,}")

# ------------------------------------------------------------
# 8. Consolidated validation results
# ------------------------------------------------------------

checks = {
    "Delivery count": actual_deliveries == EXPECTED_DELIVERIES,
    "Eligible match count": actual_matches == EXPECTED_MATCHES,
    "Only innings 1 and 2": observed_innings == {1, 2},
    "No missing delivery-key values": missing_key_values.sum() == 0,
    "No duplicate delivery keys": duplicate_key_count == 0,
    "No unmatched match IDs": len(unmatched_match_ids) == 0,
    "Complete batting/bowling team identities":
        team_missingness.sum() == 0
}

print("\n7. VALIDATION SUMMARY")
print("-" * 75)

for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<8} | {check_name}")

print("\n" + "=" * 75)

if all(checks.values()):
    print("PRIMARY POPULATION INTEGRITY CHECKS PASSED")
else:
    print("POPULATION DISCREPANCIES DETECTED — REVIEW REQUIRED")

print("=" * 75)

PHASE 3A.3 — PRIMARY POPULATION INTEGRITY AUDIT

1. POPULATION COUNTS
---------------------------------------------------------------------------
Expected regulation deliveries : 192,235
Observed regulation deliveries : 192,235
Expected eligible matches      : 813
Observed distinct matches      : 813

2. INNINGS DISTRIBUTION
---------------------------------------------------------------------------
innings
1    100333
2     91902

Observed innings values: [np.int64(1), np.int64(2)]

3. DELIVERY-KEY INTEGRITY
---------------------------------------------------------------------------
Missing values in delivery-key columns:
match_id            0
innings             0
innings_ball_num    0

Duplicate delivery keys: 0

4. COMPETITION DISTRIBUTION
---------------------------------------------------------------------------
Distinct matches by match format:
match_format
T20     362
T20I    451

Distinct competitions represented:
136

5. TEAM IDENTITY COMPLETENESS
----------------------------

### Source-match exclusion audit

In [14]:
import pandas as pd
from pathlib import Path

print("=" * 75)
print("PHASE 3A.4 — SOURCE-MATCH EXCLUSION AUDIT")
print("=" * 75)

OUTPUT_DIR = Path("/content/phase3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Verify source match-key uniqueness before joining.
source_match_duplicates = df_source_matches["match_id"].duplicated().sum()

print("\n1. SOURCE MATCH-KEY CHECK")
print("-" * 75)
print(f"Original match records       : {len(df_source_matches):,}")
print(f"Distinct original match IDs  : {df_source_matches['match_id'].nunique():,}")
print(f"Duplicate source match IDs   : {source_match_duplicates:,}")

assert source_match_duplicates == 0, (
    "Original matches.csv contains duplicate match IDs. "
    "Resolve this before performing a one-to-one exclusion audit."
)

# 2. Identify source matches absent from the integrated population.
integrated_match_ids = set(df_deliveries["match_id"].unique())

source_match_audit = df_source_matches.copy()
source_match_audit["phase2_population_status"] = (
    source_match_audit["match_id"]
    .isin(integrated_match_ids)
    .map({True: "included", False: "absent_from_integrated"})
)

excluded_matches = source_match_audit.loc[
    source_match_audit["phase2_population_status"]
    == "absent_from_integrated"
].copy()

included_matches = source_match_audit.loc[
    source_match_audit["phase2_population_status"] == "included"
].copy()

print("\n2. POPULATION RECONCILIATION")
print("-" * 75)
print(f"Source match records                  : {len(source_match_audit):,}")
print(f"Included source matches               : {len(included_matches):,}")
print(f"Source matches absent from integrated : {len(excluded_matches):,}")
print(
    "Reconciliation total                  : "
    f"{len(included_matches) + len(excluded_matches):,}"
)

assert len(included_matches) == 813
assert len(excluded_matches) == 70
assert len(included_matches) + len(excluded_matches) == len(source_match_audit)

# 3. Check whether excluded matches have records in the original ball-by-ball file.
source_delivery_counts = (
    df_source_ball.groupby("match_id", dropna=False)
    .size()
    .rename("original_ball_by_ball_records")
    .reset_index()
)

excluded_matches = excluded_matches.merge(
    source_delivery_counts,
    on="match_id",
    how="left",
    validate="one_to_one",
    indicator="source_delivery_match_status"
)

excluded_matches["original_ball_by_ball_records"] = (
    excluded_matches["original_ball_by_ball_records"].fillna(0).astype("int64")
)

print("\n3. ORIGINAL DELIVERY-RECORD PRESENCE")
print("-" * 75)
print(
    excluded_matches["source_delivery_match_status"]
    .value_counts(dropna=False)
    .to_string()
)
print(
    "\nOriginal delivery-record count summary for excluded matches:"
)
print(
    excluded_matches["original_ball_by_ball_records"]
    .describe()
    .to_string()
)

# 4. Summarize available exclusion metadata without inferring reasons.
metadata_columns = [
    column for column in [
        "comp_id",
        "stage",
        "result",
        "knock_out",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target"
    ]
    if column in excluded_matches.columns
]

print("\n4. EXCLUDED MATCH METADATA")
print("-" * 75)

for column in metadata_columns:
    print(f"\n{column}:")
    print(
        excluded_matches[column]
        .value_counts(dropna=False)
        .head(20)
        .to_string()
    )

# 5. Match-format metadata is not in the original matches.csv.
# Attach it from the integrated dataset only for included matches.
integrated_match_metadata = (
    df_deliveries[
        ["match_id", "match_format", "comp_name"]
    ]
    .drop_duplicates(subset=["match_id"])
)

assert not integrated_match_metadata["match_id"].duplicated().any()

print("\n5. INCLUDED MATCH FORMAT CROSS-CHECK")
print("-" * 75)
print(
    integrated_match_metadata["match_format"]
    .value_counts(dropna=False)
    .to_string()
)

# 6. Export the excluded source-match records with audit fields.
excluded_output_path = (
    OUTPUT_DIR / "phase3A_source_matches_absent_from_integrated.csv"
)
excluded_matches.to_csv(excluded_output_path, index=False)

print("\n6. EXPORT")
print("-" * 75)
print(f"Excluded-match audit saved: {excluded_output_path}")
print(f"Rows exported             : {len(excluded_matches):,}")

print("\n7. VALIDATION SUMMARY")
print("-" * 75)
checks = {
    "Source match IDs are unique": source_match_duplicates == 0,
    "Included matches reconcile to 813": len(included_matches) == 813,
    "Absent matches reconcile to 70": len(excluded_matches) == 70,
    "All source matches accounted for":
        len(included_matches) + len(excluded_matches)
        == len(source_match_audit),
    "Exclusion audit has one row per excluded match":
        excluded_matches["match_id"].is_unique,
    "Exported exclusion list has 70 rows":
        excluded_output_path.exists()
        and len(pd.read_csv(excluded_output_path)) == 70
}

for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL':<8} | {check_name}")

print("\n" + "=" * 75)
if all(checks.values()):
    print("SOURCE-MATCH EXCLUSION AUDIT COMPLETED")
else:
    print("EXCLUSION AUDIT DISCREPANCY — REVIEW REQUIRED")
print("=" * 75)

PHASE 3A.4 — SOURCE-MATCH EXCLUSION AUDIT

1. SOURCE MATCH-KEY CHECK
---------------------------------------------------------------------------
Original match records       : 883
Distinct original match IDs  : 883
Duplicate source match IDs   : 0

2. POPULATION RECONCILIATION
---------------------------------------------------------------------------
Source match records                  : 883
Included source matches               : 813
Source matches absent from integrated : 70
Reconciliation total                  : 883

3. ORIGINAL DELIVERY-RECORD PRESENCE
---------------------------------------------------------------------------
source_delivery_match_status
both          54
left_only     16
right_only     0

Original delivery-record count summary for excluded matches:
count     70.000000
mean     124.828571
std      102.724194
min        0.000000
25%       15.250000
50%      120.000000
75%      238.250000
max      262.000000

4. EXCLUDED MATCH METADATA
---------------------------

### Reconcile exclusions against the Phase 2 exception register

In [15]:
import pandas as pd

print("=" * 75)
print("PHASE 3A.5 — PHASE 2 EXCEPTION REGISTER RECONCILIATION")
print("=" * 75)

# 1. Inspect exception-register structure.
print("\n1. EXCEPTION REGISTER STRUCTURE")
print("-" * 75)
print(f"Rows    : {len(df_exception_register):,}")
print(f"Columns : {len(df_exception_register.columns):,}")
print("\nColumn names and data types:")
print(
    pd.DataFrame({
        "column": df_exception_register.columns,
        "dtype": df_exception_register.dtypes.astype(str).values,
        "non_null": df_exception_register.notna().sum().values,
        "null": df_exception_register.isna().sum().values
    }).to_string(index=False)
)

# 2. Display a sample to establish the register's row grain and terminology.
print("\n2. EXCEPTION REGISTER SAMPLE")
print("-" * 75)
print(df_exception_register.head(12).to_string(index=False))

# 3. Search column names for possible match identifiers.
possible_match_columns = [
    column for column in df_exception_register.columns
    if any(term in column.lower() for term in ["match", "game", "fixture"])
]

print("\n3. POSSIBLE MATCH-IDENTIFIER COLUMNS")
print("-" * 75)
print(possible_match_columns if possible_match_columns else "None identified by column name.")

# 4. Inspect values in likely descriptive fields.
descriptive_columns = [
    column for column in df_exception_register.columns
    if any(
        term in column.lower()
        for term in ["exception", "issue", "reason", "category", "status", "description", "finding"]
    )
]

print("\n4. EXCEPTION CATEGORY / DESCRIPTION FIELDS")
print("-" * 75)
if descriptive_columns:
    for column in descriptive_columns:
        print(f"\n{column}:")
        print(
            df_exception_register[column]
            .value_counts(dropna=False)
            .head(25)
            .to_string()
        )
else:
    print("No obvious descriptive fields identified by column name.")

# 5. Check whether excluded match IDs appear in any plausible match-ID column.
print("\n5. EXCLUDED MATCH ID RECONCILIATION")
print("-" * 75)

excluded_ids = set(excluded_matches["match_id"].dropna().astype("int64"))
match_column_results = {}

for column in possible_match_columns:
    values = pd.to_numeric(
        df_exception_register[column], errors="coerce"
    ).dropna()

    register_ids = set(values.astype("int64"))
    matched_ids = excluded_ids.intersection(register_ids)

    match_column_results[column] = {
        "register_distinct_numeric_values": len(register_ids),
        "excluded_match_ids_found": len(matched_ids),
        "excluded_match_ids_not_found": len(excluded_ids - matched_ids)
    }

    print(f"\nColumn: {column}")
    print(f"Distinct numeric values in register : {len(register_ids):,}")
    print(f"Excluded match IDs found            : {len(matched_ids):,}")
    print(f"Excluded match IDs not found        : {len(excluded_ids - matched_ids):,}")

if not possible_match_columns:
    print(
        "No candidate match-ID column exists. "
        "A direct match-level reconciliation cannot yet be performed."
    )

# 6. Preserve an audit summary without assigning inferred exclusion reasons.
exception_reconciliation_summary = pd.DataFrame([
    {
        "register_column": column,
        **result
    }
    for column, result in match_column_results.items()
])

summary_path = (
    OUTPUT_DIR / "phase3A_exception_register_reconciliation_summary.csv"
)
exception_reconciliation_summary.to_csv(summary_path, index=False)

print("\n6. AUDIT SUMMARY EXPORT")
print("-" * 75)
print(f"Saved: {summary_path}")
print(f"Candidate match columns inspected: {len(possible_match_columns)}")

print("\n" + "=" * 75)
print("EXCEPTION REGISTER INSPECTION COMPLETED — INTERPRETATION PENDING")
print("=" * 75)

PHASE 3A.5 — PHASE 2 EXCEPTION REGISTER RECONCILIATION

1. EXCEPTION REGISTER STRUCTURE
---------------------------------------------------------------------------
Rows    : 24
Columns : 17

Column names and data types:
          column   dtype  non_null  null
    exception_id  object        24     0
  exception_type  object        24     0
         dataset  object        24     0
        match_id float64        23     1
affected_records   int64        24     0
           field  object        24     0
     observation  object        24     0
       treatment  object        24     0
          status  object        24     0
         innings float64         9    15
          overid float64         9    15
innings_ball_num float64         9    15
       bowler_id float64         9    15
    extras1_runs float64         9    15
     extras_type  object        10    14
   extras_2_runs float64         9    15
            runs float64         9    15

2. EXCEPTION REGISTER SAMPLE
------------

### Classify excluded matches by source-data availability

In [16]:
import pandas as pd
from pathlib import Path

print("=" * 75)
print("PHASE 3A.6 — EXCLUDED MATCH SOURCE-EVIDENCE CLASSIFICATION")
print("=" * 75)

# Start from the previously audited 70 excluded source matches.
excluded_source_audit = excluded_matches.copy()

# 1. Verify the expected audit population.
assert len(excluded_source_audit) == 70
assert excluded_source_audit["match_id"].is_unique

# 2. Classify strictly by original delivery-record presence.
excluded_source_audit["source_evidence_class"] = (
    excluded_source_audit["original_ball_by_ball_records"]
    .gt(0)
    .map({
        True: "SOURCE_DELIVERIES_PRESENT_NOT_IN_INTEGRATED",
        False: "NO_SOURCE_DELIVERY_RECORDS"
    })
)

# 3. Summarize evidence classes.
print("\n1. SOURCE-EVIDENCE CLASSIFICATION")
print("-" * 75)

class_summary = (
    excluded_source_audit.groupby("source_evidence_class", dropna=False)
    .agg(
        excluded_matches=("match_id", "nunique"),
        total_original_delivery_records=(
            "original_ball_by_ball_records", "sum"
        ),
        median_original_delivery_records=(
            "original_ball_by_ball_records", "median"
        ),
        minimum_original_delivery_records=(
            "original_ball_by_ball_records", "min"
        ),
        maximum_original_delivery_records=(
            "original_ball_by_ball_records", "max"
        )
    )
    .reset_index()
)

print(class_summary.to_string(index=False))

# 4. Cross-tabulate source-evidence class against result metadata.
print("\n2. RESULT METADATA BY SOURCE-EVIDENCE CLASS")
print("-" * 75)

result_cross_tab = pd.crosstab(
    excluded_source_audit["source_evidence_class"],
    excluded_source_audit["result"],
    dropna=False
)

print(result_cross_tab.to_string())

# 5. Cross-tabulate source-evidence class against competition.
print("\n3. COMPETITION DISTRIBUTION BY SOURCE-EVIDENCE CLASS")
print("-" * 75)

competition_cross_tab = pd.crosstab(
    excluded_source_audit["source_evidence_class"],
    excluded_source_audit["comp_id"],
    dropna=False
)

print(competition_cross_tab.to_string())

# 6. Produce a compact audit table.
audit_columns = [
    column for column in [
        "match_id",
        "comp_id",
        "match_number",
        "match_date",
        "stage",
        "result",
        "knock_out",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target",
        "original_ball_by_ball_records",
        "source_evidence_class"
    ]
    if column in excluded_source_audit.columns
]

excluded_match_classification = (
    excluded_source_audit[audit_columns]
    .sort_values(
        ["source_evidence_class", "comp_id", "match_id"]
    )
    .reset_index(drop=True)
)

print("\n4. CLASSIFIED MATCH LIST — FIRST 20 ROWS")
print("-" * 75)
print(excluded_match_classification.head(20).to_string(index=False))

# 7. Export the complete classified list and summary.
OUTPUT_DIR = Path("/content/phase3_outputs")

classification_path = (
    OUTPUT_DIR / "phase3A_excluded_match_source_evidence_classification.csv"
)
summary_path = (
    OUTPUT_DIR / "phase3A_excluded_match_source_evidence_summary.csv"
)

excluded_match_classification.to_csv(classification_path, index=False)
class_summary.to_csv(summary_path, index=False)

# 8. Validation.
checks = {
    "Exactly 70 excluded matches classified":
        len(excluded_match_classification) == 70,
    "No duplicate excluded match IDs":
        excluded_match_classification["match_id"].is_unique,
    "All matches assigned one evidence class":
        excluded_match_classification["source_evidence_class"].notna().all(),
    "54 matches have source delivery records":
        (
            excluded_match_classification["source_evidence_class"]
            == "SOURCE_DELIVERIES_PRESENT_NOT_IN_INTEGRATED"
        ).sum() == 54,
    "16 matches have no source delivery records":
        (
            excluded_match_classification["source_evidence_class"]
            == "NO_SOURCE_DELIVERY_RECORDS"
        ).sum() == 16,
    "Classified delivery-record total reconciles":
        excluded_match_classification[
            "original_ball_by_ball_records"
        ].sum()
        == excluded_source_audit["original_ball_by_ball_records"].sum(),
    "Both audit outputs exist":
        classification_path.exists() and summary_path.exists()
}

print("\n5. VALIDATION SUMMARY")
print("-" * 75)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL':<8} | {name}")

print("\n6. EXPORTS")
print("-" * 75)
print(classification_path)
print(summary_path)

print("\n" + "=" * 75)
if all(checks.values()):
    print("EXCLUDED MATCH SOURCE-EVIDENCE CLASSIFICATION PASSED")
else:
    print("CLASSIFICATION VALIDATION FAILED — REVIEW REQUIRED")
print("=" * 75)

PHASE 3A.6 — EXCLUDED MATCH SOURCE-EVIDENCE CLASSIFICATION

1. SOURCE-EVIDENCE CLASSIFICATION
---------------------------------------------------------------------------
                      source_evidence_class  excluded_matches  total_original_delivery_records  median_original_delivery_records  minimum_original_delivery_records  maximum_original_delivery_records
                 NO_SOURCE_DELIVERY_RECORDS                16                                0                               0.0                                  0                                  0
SOURCE_DELIVERIES_PRESENT_NOT_IN_INTEGRATED                54                             8738                             196.0                                  2                                262

2. RESULT METADATA BY SOURCE-EVIDENCE CLASS
---------------------------------------------------------------------------
result                                       Afghanistan A Won by 11 runs  Afghanistan A Won by 20 runs  Afghani

### Investigate the 54 excluded matches with source deliveries

In [32]:
# ============================================================
# PHASE 3A.7 — INVESTIGATION TABLES
# Complete, self-contained replacement cell
# ============================================================

# 1. Select the 54 excluded matches with source deliveries.
investigation_54_base = (
    excluded_source_audit.loc[
        excluded_source_audit["original_ball_by_ball_records"] > 0
    ].copy()
)

# 2. Reconstruct source innings evidence.
source_54 = df_source_ball.loc[
    df_source_ball["match_id"].isin(
        investigation_54_base["match_id"]
    )
].copy()

source_innings_detail = (
    source_54.groupby(["match_id", "innings"], dropna=False)
    .size()
    .reset_index(name="innings_delivery_records")
)

source_innings_summary = (
    source_innings_detail.groupby("match_id")
    .agg(
        source_innings_count=("innings", "nunique"),
        source_innings_values=(
            "innings",
            lambda x: ", ".join(
                sorted(x.dropna().astype(str).unique())
            )
        )
    )
    .reset_index()
)

innings_wide = (
    source_innings_detail.pivot(
        index="match_id",
        columns="innings",
        values="innings_delivery_records"
    )
    .rename(columns={
        1: "first_innings_rows",
        2: "second_innings_rows"
    })
    .reset_index()
)

# Keep only the intended innings columns.
innings_wide = innings_wide[
    [
        col for col in innings_wide.columns
        if col in [
            "match_id",
            "first_innings_rows",
            "second_innings_rows"
        ]
    ]
]

# 3. Build the match-level investigation.
investigation_54 = investigation_54_base.merge(
    source_innings_summary,
    on="match_id",
    how="left",
    validate="one_to_one"
)

investigation_54 = investigation_54.merge(
    innings_wide,
    on="match_id",
    how="left",
    validate="one_to_one"
)

for col in [
    "source_innings_count",
    "first_innings_rows",
    "second_innings_rows"
]:
    investigation_54[col] = (
        investigation_54[col].fillna(0).astype(int)
    )

investigation_54["source_innings_values"] = (
    investigation_54["source_innings_values"].fillna("")
)

investigation_54["source_delivery_rows"] = (
    investigation_54["original_ball_by_ball_records"].astype(int)
)

# 4. Reconstruct integrated delivery counts.
integrated_counts_54 = (
    df_deliveries.loc[
        df_deliveries["match_id"].isin(
            investigation_54["match_id"]
        )
    ]
    .groupby("match_id")
    .size()
    .reset_index(name="integrated_delivery_rows")
)

investigation_54 = investigation_54.merge(
    integrated_counts_54,
    on="match_id",
    how="left",
    validate="one_to_one"
)

investigation_54["integrated_delivery_rows"] = (
    investigation_54["integrated_delivery_rows"]
    .fillna(0)
    .astype(int)
)

# 5. Derive revised-innings metadata flag from source match metadata.
investigation_54["has_revised_innings_metadata"] = (
    investigation_54["first_inn_revised_overs"].notna()
    | investigation_54["second_inn_revised_overs"].notna()
    | investigation_54["revised_target"].notna()
)

# 6. Match-level investigation table.
match_table_columns = [
    "match_id",
    "comp_id",
    "match_number",
    "match_date",
    "stage",
    "result",
    "source_delivery_rows",
    "source_innings_count",
    "source_innings_values",
    "first_innings_rows",
    "second_innings_rows",
    "integrated_delivery_rows",
    "has_revised_innings_metadata"
]

match_table = (
    investigation_54[match_table_columns]
    .sort_values(["comp_id", "match_id"])
    .reset_index(drop=True)
)

# 7. Competition-level summary.
competition_summary_54 = (
    investigation_54.groupby("comp_id", dropna=False)
    .agg(
        excluded_matches=("match_id", "nunique"),
        source_delivery_rows=("source_delivery_rows", "sum"),
        two_innings_matches=(
            "source_innings_count",
            lambda x: int(x.eq(2).sum())
        ),
        revised_innings_metadata_matches=(
            "has_revised_innings_metadata",
            "sum"
        )
    )
    .reset_index()
)

competition_table = (
    competition_summary_54
    .sort_values(
        ["excluded_matches", "source_delivery_rows"],
        ascending=False
    )
    .reset_index(drop=True)
)

# 8. Overall investigation summary.
overall_summary_54 = pd.DataFrame([
    {
        "measure": "Matches investigated",
        "value": investigation_54["match_id"].nunique()
    },
    {
        "measure": "Source delivery records",
        "value": investigation_54["source_delivery_rows"].sum()
    },
    {
        "measure": "Integrated delivery records",
        "value": investigation_54["integrated_delivery_rows"].sum()
    },
    {
        "measure": "Competitions represented",
        "value": investigation_54["comp_id"].nunique()
    },
    {
        "measure": "Matches with exactly two observed innings",
        "value": investigation_54["source_innings_count"].eq(2).sum()
    },
    {
        "measure": "Matches with four observed innings",
        "value": investigation_54["source_innings_count"].eq(4).sum()
    },
    {
        "measure": "Matches with revised-innings metadata",
        "value": investigation_54[
            "has_revised_innings_metadata"
        ].sum()
    }
])

# 9. Validation.
investigation_validation = pd.DataFrame([
    {
        "validation": "Exactly 54 unique source matches",
        "status": (
            len(investigation_54) == 54
            and investigation_54["match_id"].nunique() == 54
        )
    },
    {
        "validation": "Source delivery records total 8,738",
        "status": investigation_54["source_delivery_rows"].sum() == 8738
    },
    {
        "validation": "Source deliveries reconcile to original source",
        "status": (
            investigation_54["source_delivery_rows"].sum()
            == len(source_54)
        )
    },
    {
        "validation": "No integrated delivery records",
        "status": investigation_54["integrated_delivery_rows"].sum() == 0
    },
    {
        "validation": "Competition count reconciles to match-level data",
        "status": (
            competition_table["comp_id"].nunique()
            == investigation_54["comp_id"].nunique()
        )
    },
    {
        "validation": "Competition match totals reconcile to 54",
        "status": competition_table["excluded_matches"].sum() == 54
    },
    {
        "validation": "Competition delivery totals reconcile to 8,738",
        "status": competition_table["source_delivery_rows"].sum() == 8738
    },
    {
        "validation": "No duplicate match IDs",
        "status": not investigation_54["match_id"].duplicated().any()
    },
    {
        "validation": "All required match-table columns exist",
        "status": set(match_table_columns).issubset(
            investigation_54.columns
        )
    }
])

investigation_validation["status"] = (
    investigation_validation["status"]
    .map({True: "PASS", False: "FAIL"})
)

# 10. Display all four tables.
print("1. MATCH-LEVEL INVESTIGATION")
display(match_table)

print("2. COMPETITION-LEVEL SUMMARY")
display(competition_table)

print("3. OVERALL INVESTIGATION SUMMARY")
display(overall_summary_54)

print("4. VALIDATION RESULTS")
display(investigation_validation)

# 11. Export only after all validations pass.
assert investigation_validation["status"].eq("PASS").all(), (
    "Validation failed. Do not treat these outputs as final."
)

match_table.to_csv(
    OUTPUT_DIR / "phase3A_54_match_investigation.csv",
    index=False
)

competition_table.to_csv(
    OUTPUT_DIR / "phase3A_54_competition_investigation.csv",
    index=False
)

overall_summary_54.to_csv(
    OUTPUT_DIR / "phase3A_54_overall_investigation.csv",
    index=False
)

investigation_validation.to_csv(
    OUTPUT_DIR / "phase3A_54_investigation_validation.csv",
    index=False
)

print("SUCCESS: All nine validations passed.")
print("SUCCESS: All four audit CSV files exported.")

1. MATCH-LEVEL INVESTIGATION


,match_id,comp_id,match_number,match_date,stage,result,source_delivery_rows,source_innings_count,source_innings_values,first_innings_rows,second_innings_rows,integrated_delivery_rows,has_revised_innings_metadata
0,15410,1439,5,2022-06-19,League,Match Abandoned,22,1,1,22,0,0,True
1,15546,1452,1,2022-07-02,League,No Result,82,1,1,82,0,0,True
2,17765,1611,3,2022-10-14,League,No Result,100,2,"1, 2",75,25,0,True
3,20226,1715,45,2023-05-03,League,Match Abandoned,118,1,1,118,0,0,False
4,21406,1799,4,2023-04-20,League,Match Abandoned,118,1,1,118,0,0,False
5,28333,2058,2,2023-12-29,Others,No Result,66,1,1,66,0,0,False
6,31707,2350,1,2024-04-18,League,No Result,2,1,1,2,0,0,True
7,37353,2455,5,2024-11-18,Others,No Result,30,1,1,30,0,0,False
8,35424,2605,2,2024-08-24,League,No Result,125,1,1,125,0,0,False
9,37085,2670,1,2024-10-18,League,Bangladesh A Won by 5 wickets,236,2,"1, 2",121,115,0,False


2. COMPETITION-LEVEL SUMMARY


,comp_id,excluded_matches,source_delivery_rows,two_innings_matches,revised_innings_metadata_matches
0,2670,15,3519,15,0
1,3289,15,3466,13,0
2,2725,3,316,1,0
3,2801,2,169,0,2
4,3226,2,149,0,1
5,2915,2,89,0,1
6,2605,1,125,0,0
7,3126,1,122,0,0
8,1715,1,118,0,0
9,1799,1,118,0,0


3. OVERALL INVESTIGATION SUMMARY


,measure,value
0,Matches investigated,54
1,Source delivery records,8738
2,Integrated delivery records,0
3,Competitions represented,21
4,Matches with exactly two observed innings,30
5,Matches with four observed innings,2
6,Matches with revised-innings metadata,10


4. VALIDATION RESULTS


,validation,status
0,Exactly 54 unique source matches,PASS
1,"Source delivery records total 8,738",PASS
2,Source deliveries reconcile to original source,PASS
3,No integrated delivery records,PASS
4,Competition count reconciles to match-level data,PASS
5,Competition match totals reconcile to 54,PASS
6,"Competition delivery totals reconcile to 8,738",PASS
7,No duplicate match IDs,PASS
8,All required match-table columns exist,PASS


SUCCESS: All nine validations passed.
SUCCESS: All four audit CSV files exported.


### Competition-level population lineage audit

In [33]:
print("PHASE 3A.8 — COMPETITION-LEVEL POPULATION LINEAGE AUDIT")

# 1. Source match counts by competition
source_competition = (
    df_source_matches.groupby("comp_id", dropna=False)
    .agg(
        source_matches=("match_id", "nunique")
    )
    .reset_index()
)

# 2. Integrated match counts by competition
integrated_competition = (
    df_deliveries[["comp_id", "match_id"]]
    .drop_duplicates()
    .groupby("comp_id", dropna=False)
    .agg(
        integrated_matches=("match_id", "nunique")
    )
    .reset_index()
)

# 3. Excluded match counts and delivery records by competition
excluded_competition = (
    excluded_source_audit.groupby("comp_id", dropna=False)
    .agg(
        excluded_matches=("match_id", "nunique"),
        excluded_source_delivery_rows=(
            "original_ball_by_ball_records", "sum"
        )
    )
    .reset_index()
)

# 4. Two-innings excluded match count by competition
two_innings_excluded = (
    investigation_54.assign(
        two_innings=investigation_54["source_innings_count"].eq(2)
    )
    .groupby("comp_id", dropna=False)
    .agg(
        excluded_matches_with_two_innings=("two_innings", "sum")
    )
    .reset_index()
)

# 5. Combine with explicit merge validation
competition_lineage = source_competition.merge(
    integrated_competition,
    on="comp_id",
    how="left",
    validate="one_to_one"
)

competition_lineage = competition_lineage.merge(
    excluded_competition,
    on="comp_id",
    how="left",
    validate="one_to_one"
)

competition_lineage = competition_lineage.merge(
    two_innings_excluded,
    on="comp_id",
    how="left",
    validate="one_to_one"
)

count_columns = [
    "integrated_matches",
    "excluded_matches",
    "excluded_source_delivery_rows",
    "excluded_matches_with_two_innings"
]

competition_lineage[count_columns] = (
    competition_lineage[count_columns].fillna(0).astype("int64")
)

# 6. Population reconciliation
competition_lineage["reconciled_matches"] = (
    competition_lineage["integrated_matches"]
    + competition_lineage["excluded_matches"]
)

competition_lineage["population_status"] = "PARTIAL_INCLUSION"

competition_lineage.loc[
    competition_lineage["integrated_matches"].eq(0),
    "population_status"
] = "NO_MATCHES_INTEGRATED"

competition_lineage.loc[
    competition_lineage["excluded_matches"].eq(0),
    "population_status"
] = "ALL_SOURCE_MATCHES_INTEGRATED"

competition_lineage["match_count_reconciles"] = (
    competition_lineage["source_matches"]
    == competition_lineage["reconciled_matches"]
)

# 7. Sort with most consequential exclusions first
competition_lineage = competition_lineage.sort_values(
    ["excluded_source_delivery_rows", "excluded_matches"],
    ascending=[False, False]
).reset_index(drop=True)

print("\n1. COMPETITION-LEVEL LINEAGE TABLE")
display(competition_lineage)

# 8. Status summary
status_summary = (
    competition_lineage.groupby("population_status", dropna=False)
    .agg(
        competitions=("comp_id", "nunique"),
        source_matches=("source_matches", "sum"),
        integrated_matches=("integrated_matches", "sum"),
        excluded_matches=("excluded_matches", "sum"),
        excluded_source_delivery_rows=(
            "excluded_source_delivery_rows", "sum"
        )
    )
    .reset_index()
)

print("\n2. POPULATION STATUS SUMMARY")
display(status_summary)

# 9. Focus on competitions 2670 and 3289
priority_competitions = competition_lineage.loc[
    competition_lineage["comp_id"].isin([2670, 3289])
].copy()

print("\n3. PRIORITY COMPETITIONS — 2670 AND 3289")
display(priority_competitions)

# 10. Validation
validation_table = pd.DataFrame([
    {
        "Validation": "All source competitions represented",
        "Status": (
            competition_lineage["comp_id"].nunique()
            == df_source_matches["comp_id"].nunique()
        )
    },
    {
        "Validation": "Every competition match count reconciles",
        "Status": competition_lineage["match_count_reconciles"].all()
    },
    {
        "Validation": "Source match total equals 883",
        "Status": competition_lineage["source_matches"].sum() == 883
    },
    {
        "Validation": "Integrated match total equals 813",
        "Status": competition_lineage["integrated_matches"].sum() == 813
    },
    {
        "Validation": "Excluded match total equals 70",
        "Status": competition_lineage["excluded_matches"].sum() == 70
    },
    {
        "Validation": "Excluded source delivery total equals 8,738",
        "Status": (
            competition_lineage["excluded_source_delivery_rows"].sum()
            == 8738
        )
    }
])

validation_table["Status"] = validation_table["Status"].map({
    True: "PASS",
    False: "FAIL"
})

print("\n4. VALIDATION RESULTS")
display(validation_table)

# 11. Export audit tables
OUTPUT_DIR = Path("/content/phase3_outputs")

competition_lineage.to_csv(
    OUTPUT_DIR / "phase3A_competition_population_lineage.csv",
    index=False
)

status_summary.to_csv(
    OUTPUT_DIR / "phase3A_competition_population_status_summary.csv",
    index=False
)

priority_competitions.to_csv(
    OUTPUT_DIR / "phase3A_priority_competition_lineage.csv",
    index=False
)

validation_table.to_csv(
    OUTPUT_DIR / "phase3A_competition_lineage_validation.csv",
    index=False
)

print("\n5. EXPORTS")
display(pd.DataFrame({
    "Output": [
        "phase3A_competition_population_lineage.csv",
        "phase3A_competition_population_status_summary.csv",
        "phase3A_priority_competition_lineage.csv",
        "phase3A_competition_lineage_validation.csv"
    ],
    "Status": ["Saved"] * 4
}))

PHASE 3A.8 — COMPETITION-LEVEL POPULATION LINEAGE AUDIT

1. COMPETITION-LEVEL LINEAGE TABLE


,comp_id,source_matches,integrated_matches,excluded_matches,excluded_source_delivery_rows,excluded_matches_with_two_innings,reconciled_matches,population_status,match_count_reconciles
0,2670,15,0,15,3519,15,15,NO_MATCHES_INTEGRATED,True
1,3289,15,0,15,3466,13,15,NO_MATCHES_INTEGRATED,True
2,2725,75,71,4,316,1,75,PARTIAL_INCLUSION,True
3,2801,3,1,2,169,0,3,PARTIAL_INCLUSION,True
4,3226,3,1,2,149,0,3,PARTIAL_INCLUSION,True
...,...,...,...,...,...,...,...,...,...
133,3690,5,5,0,0,0,5,ALL_SOURCE_MATCHES_INTEGRATED,True
134,3713,3,3,0,0,0,3,ALL_SOURCE_MATCHES_INTEGRATED,True
135,3739,1,1,0,0,0,1,ALL_SOURCE_MATCHES_INTEGRATED,True
136,3765,3,3,0,0,0,3,ALL_SOURCE_MATCHES_INTEGRATED,True



2. POPULATION STATUS SUMMARY


,population_status,competitions,source_matches,integrated_matches,excluded_matches,excluded_source_delivery_rows
0,ALL_SOURCE_MATCHES_INTEGRATED,106,453,453,0,0
1,NO_MATCHES_INTEGRATED,2,30,0,30,6985
2,PARTIAL_INCLUSION,30,400,360,40,1753



3. PRIORITY COMPETITIONS — 2670 AND 3289


,comp_id,source_matches,integrated_matches,excluded_matches,excluded_source_delivery_rows,excluded_matches_with_two_innings,reconciled_matches,population_status,match_count_reconciles
0,2670,15,0,15,3519,15,15,NO_MATCHES_INTEGRATED,True
1,3289,15,0,15,3466,13,15,NO_MATCHES_INTEGRATED,True



4. VALIDATION RESULTS


,Validation,Status
0,All source competitions represented,PASS
1,Every competition match count reconciles,PASS
2,Source match total equals 883,PASS
3,Integrated match total equals 813,PASS
4,Excluded match total equals 70,PASS
5,"Excluded source delivery total equals 8,738",PASS



5. EXPORTS


,Output,Status
0,phase3A_competition_population_lineage.csv,Saved
1,phase3A_competition_population_status_summary.csv,Saved
2,phase3A_priority_competition_lineage.csv,Saved
3,phase3A_competition_lineage_validation.csv,Saved


### Competition metadata lookup

In [34]:
# Load the newly uploaded competition lookup file.
df_source_competitions = pd.read_csv(
    "/content/competitions.csv",
    low_memory=False
)

# Inspect the lookup schema.
competition_schema = pd.DataFrame({
    "column": df_source_competitions.columns,
    "dtype": df_source_competitions.dtypes.astype(str).values,
    "non_null": df_source_competitions.notna().sum().values,
    "null": df_source_competitions.isna().sum().values
})

print("1. COMPETITION LOOKUP SCHEMA")
display(competition_schema)

# Retrieve the two priority competitions.
priority_competition_metadata = (
    df_source_competitions.loc[
        pd.to_numeric(
            df_source_competitions["comp_id"],
            errors="coerce"
        ).isin([2670, 3289])
    ]
    .copy()
    .sort_values("comp_id")
    .reset_index(drop=True)
)

print("\n2. PRIORITY COMPETITION METADATA")
display(priority_competition_metadata)

# Validate lookup uniqueness and coverage.
priority_lookup_validation = pd.DataFrame([
    {
        "Validation": "Competition lookup IDs are unique",
        "Status": (
            not df_source_competitions["comp_id"].duplicated().any()
        )
    },
    {
        "Validation": "Both priority competition IDs found",
        "Status": (
            set(priority_competition_metadata["comp_id"].astype(int))
            == {2670, 3289}
        )
    },
    {
        "Validation": "Exactly one record per priority competition",
        "Status": (
            len(priority_competition_metadata) == 2
            and priority_competition_metadata["comp_id"].nunique() == 2
        )
    }
])

priority_lookup_validation["Status"] = (
    priority_lookup_validation["Status"]
    .map({True: "PASS", False: "FAIL"})
)

print("\n3. LOOKUP VALIDATION")
display(priority_lookup_validation)

1. COMPETITION LOOKUP SCHEMA


,column,dtype,non_null,null
0,comp_id,int64,141,0
1,comp_name,object,141,0
2,match_format,object,141,0
3,season,object,141,0



2. PRIORITY COMPETITION METADATA


,comp_id,comp_name,match_format,season
0,2670,Mens T20 Emerging Teams Asia Cup 2024,T20,2024
1,3289,Asia Cup Rising Stars 2025,T20,2025



3. LOOKUP VALIDATION


,Validation,Status
0,Competition lookup IDs are unique,PASS
1,Both priority competition IDs found,PASS
2,Exactly one record per priority competition,PASS


## Quantify the impact on T20 and T20I populations

In [42]:
# 1. Attach competition metadata to the source match population.
source_match_format = df_source_matches.merge(
    df_source_competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ],
    on="comp_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# 2. Attach competition metadata to the integrated match population.
integrated_match_format = (
    df_deliveries[["match_id", "comp_id"]]
    .drop_duplicates()
    .merge(
        df_source_competitions[
            ["comp_id", "comp_name", "match_format", "season"]
        ],
        on="comp_id",
        how="left",
        validate="many_to_one",
        indicator=True
    )
)

# 3. Build match counts by competition and format.
source_format_counts = (
    source_match_format.groupby("match_format", dropna=False)
    .agg(source_matches=("match_id", "nunique"))
    .reset_index()
)

integrated_format_counts = (
    integrated_match_format.groupby("match_format", dropna=False)
    .agg(integrated_matches=("match_id", "nunique"))
    .reset_index()
)

excluded_match_format = excluded_source_audit[
    ["match_id", "comp_id", "original_ball_by_ball_records"]
].merge(
    df_source_competitions[
        ["comp_id", "match_format"]
    ],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

excluded_format_counts = (
    excluded_match_format.groupby("match_format", dropna=False)
    .agg(
        excluded_matches=("match_id", "nunique"),
        excluded_source_delivery_rows=(
            "original_ball_by_ball_records", "sum"
        )
    )
    .reset_index()
)

# 4. Count fully and partially excluded competitions by format.
competition_status_by_format = (
    competition_lineage.merge(
        df_source_competitions[
            ["comp_id", "match_format"]
        ],
        on="comp_id",
        how="left",
        validate="one_to_one"
    )
    .groupby("match_format", dropna=False)
    .agg(
        source_competitions=("comp_id", "nunique"),
        fully_excluded_competitions=(
            "population_status",
            lambda values: int(
                (values == "NO_MATCHES_INTEGRATED").sum()
            )
        ),
        partially_included_competitions=(
            "population_status",
            lambda values: int(
                (values == "PARTIAL_INCLUSION").sum()
            )
        ),
        fully_integrated_competitions=(
            "population_status",
            lambda values: int(
                (values == "ALL_SOURCE_MATCHES_INTEGRATED").sum()
            )
        )
    )
    .reset_index()
)

# 5. Combine all format-level summaries.
format_impact = (
    source_format_counts
    .merge(
        integrated_format_counts,
        on="match_format",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        excluded_format_counts,
        on="match_format",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        competition_status_by_format,
        on="match_format",
        how="outer",
        validate="one_to_one"
    )
    .fillna(0)
)

numeric_columns = [
    column for column in format_impact.columns
    if column != "match_format"
]

format_impact[numeric_columns] = (
    format_impact[numeric_columns].astype("int64")
)

format_impact["match_reconciliation"] = (
    format_impact["source_matches"]
    == (
        format_impact["integrated_matches"]
        + format_impact["excluded_matches"]
    )
)

print("1. FORMAT-LEVEL POPULATION IMPACT")
display(format_impact.sort_values("match_format").reset_index(drop=True))

# 6. Inspect competition lookup coverage.
source_comp_ids = set(df_source_matches["comp_id"].unique())
integrated_comp_ids = set(df_deliveries["comp_id"].unique())
lookup_comp_ids = set(df_source_competitions["comp_id"].unique())

lookup_coverage = pd.DataFrame([
    {
        "Measure": "Competition IDs in lookup",
        "Count": len(lookup_comp_ids)
    },
    {
        "Measure": "Competition IDs in source matches",
        "Count": len(source_comp_ids)
    },
    {
        "Measure": "Competition IDs in integrated population",
        "Count": len(integrated_comp_ids)
    },
    {
        "Measure": "Source competition IDs missing from lookup",
        "Count": len(source_comp_ids - lookup_comp_ids)
    },
    {
        "Measure": "Lookup competition IDs with no source matches",
        "Count": len(lookup_comp_ids - source_comp_ids)
    }
])

print("\n2. COMPETITION LOOKUP COVERAGE")
display(lookup_coverage)

# 7. Validation table.
format_validation = pd.DataFrame([
    {
        "Validation": "All source match formats mapped",
        "Status": source_match_format["_merge"].eq("both").all()
    },
    {
        "Validation": "All integrated match formats mapped",
        "Status": integrated_match_format["_merge"].eq("both").all()
    },
    {
        "Validation": "All excluded match formats mapped",
        "Status": excluded_match_format["match_format"].notna().all()
    },
    {
        "Validation": "Format-level match counts reconcile",
        "Status": format_impact["match_reconciliation"].all()
    },
    {
        "Validation": "Source matches total equals 883",
        "Status": format_impact["source_matches"].sum() == 883
    },
    {
        "Validation": "Integrated matches total equals 813",
        "Status": format_impact["integrated_matches"].sum() == 813
    },
    {
        "Validation": "Excluded matches total equals 70",
        "Status": format_impact["excluded_matches"].sum() == 70
    }
])

format_validation["Status"] = format_validation["Status"].map({
    True: "PASS",
    False: "FAIL"
})

print("\n3. VALIDATION RESULTS")
display(format_validation)

1. FORMAT-LEVEL POPULATION IMPACT


,match_format,source_matches,integrated_matches,excluded_matches,excluded_source_delivery_rows,source_competitions,fully_excluded_competitions,partially_included_competitions,fully_integrated_competitions,match_reconciliation
0,T20,401,362,39,7441,7,2,4,1,True
1,T20I,482,451,31,1297,131,0,26,105,True



2. COMPETITION LOOKUP COVERAGE


,Measure,Count
0,Competition IDs in lookup,141
1,Competition IDs in source matches,138
2,Competition IDs in integrated population,136
3,Source competition IDs missing from lookup,0
4,Lookup competition IDs with no source matches,3



3. VALIDATION RESULTS


,Validation,Status
0,All source match formats mapped,PASS
1,All integrated match formats mapped,PASS
2,All excluded match formats mapped,PASS
3,Format-level match counts reconcile,PASS
4,Source matches total equals 883,PASS
5,Integrated matches total equals 813,PASS
6,Excluded matches total equals 70,PASS


### Review Phase 2 population-selection rationale

In [43]:
# Search the Phase 2 report without modifying the original report text.
phase2_lines = phase2_report.splitlines()

search_terms = [
    "eligible",
    "eligibility",
    "excluded",
    "exclusion",
    "competition",
    "filter",
    "813",
    "T20I",
    "T20",
    "population",
    "match selection"
]

report_matches = []

for line_number, line in enumerate(phase2_lines, start=1):
    matched_terms = [
        term for term in search_terms
        if term.lower() in line.lower()
    ]

    if matched_terms:
        report_matches.append({
            "line_number": line_number,
            "matched_terms": ", ".join(matched_terms),
            "report_text": line.strip()
        })

phase2_report_search = pd.DataFrame(report_matches)

print("1. PHASE 2 REPORT — POPULATION-SELECTION REFERENCES")
display(phase2_report_search)

# Search the Phase 1 report for the original population definition.
phase1_lines = phase1_report_text.splitlines()

phase1_terms = [
    "eligible",
    "excluded",
    "competition",
    "T20I",
    "T20",
    "match format",
    "population"
]

phase1_matches = []

for line_number, line in enumerate(phase1_lines, start=1):
    matched_terms = [
        term for term in phase1_terms
        if term.lower() in line.lower()
    ]

    if matched_terms:
        phase1_matches.append({
            "line_number": line_number,
            "matched_terms": ", ".join(matched_terms),
            "report_text": line.strip()
        })

phase1_report_search = pd.DataFrame(phase1_matches)

print("\n2. PHASE 1 REPORT — ORIGINAL POPULATION REFERENCES")
display(phase1_report_search)

1. PHASE 2 REPORT — POPULATION-SELECTION REFERENCES


,line_number,matched_terms,report_text
0,11,eligibility,"Phase 2 focused on data cleaning and reconstruction of the cricket delivery dataset, including match eligibility, player identity, delivery classification, dismissal handling, innings and over sequence integrity, and preparation of an integrated analytical dataset."
1,13,"eligibility, 813, population","The approved target population contained 853 matches. After applying the established eligibility criteria, 813 matches were retained, containing 192,235 regulation deliveries. A further 26 Super Over deliveries were retained separately."
2,18,"eligible, 813",- 813 eligible matches
3,19,competition,- 136 competitions
4,29,"eligibility, population",## 2. Match Population and Eligibility
5,31,population,| Population measure | Records |
6,34,"eligible, 813",| Eligible matches | 813 |
7,35,excluded,| Excluded matches | 40 |
8,36,eligible,"| Eligible regulation deliveries | 192,235 |"
9,39,"eligible, T20I, T20, population",The target population consisted of 371 T20 matches and 482 T20I matches. The eligible population consisted of 362 T20 matches and 451 T20I matches.



2. PHASE 1 REPORT — ORIGINAL POPULATION REFERENCES


,line_number,matched_terms,report_text
0,5,population,Approved analysis population and data-quality assessment
1,10,"competition, population","Phase 1 audited six supplied CSV datasets: ball-by-ball deliveries, playing-XI/player registrations, matches, competition-team-player mappings, teams and competitions. The audit examined dataset structure, identifier relationships, player identity, team and competition consistency, delivery-level fields, playing-XI structure, competition coverage, match eligibility and the proposed primary delivery population."
2,11,"competition, T20I, T20, population","The approved primary population contains 813 matches: 362 T20 and 451 T20I. These matches are represented in 192,235 regulation deliveries from innings 1 and 2. Four Super Over matches contribute 26 additional deliveries, retained separately. The full target competition set contains 853 matches: 371 T20 and 482 T20I."
3,12,competition,"Most match, team, competition and player-mapping relationships reconciled. The audit also found field-semantic ambiguity and localized source inconsistencies. The user approved retaining those issues as documented limitations, without silent correction or inferred replacement values."
4,20,competition,"All home/away team IDs, match competition IDs, mapping competition IDs and match home/away team-competition pairs reconciled to their reference tables."
5,21,competition,"All actual delivery-role competition-team-player combinations were present in the mapping: striker 4,327/4,327; non-striker 4,165/4,165; bowler 3,101/3,101."
6,24,competition,"4. Team, competition and playing-XI consistency"
7,25,competition,"Of 5,646 unique playing-XI match-team-player combinations, 5,645 matched the competition-team-player mapping."
8,27,competition,All delivery-role competition-team-player combinations were mapped.
9,35,competition,5. Competition and season coverage


### Reconcile the documented 40 exclusions against the current 70

In [44]:
# Extract complete relevant passages from the Phase 2 report.
phase2_sections = [
    (1, 25),
    (29, 50),
    (155, 175),
    (225, 240)
]

phase2_passages = []

for start_line, end_line in phase2_sections:
    for line_number in range(start_line, min(end_line, len(phase2_lines)) + 1):
        text_line = phase2_lines[line_number - 1].strip()

        if text_line:
            phase2_passages.append({
                "section_range": f"{start_line}-{end_line}",
                "line_number": line_number,
                "report_text": text_line
            })

phase2_population_passages = pd.DataFrame(phase2_passages)

print("1. COMPLETE PHASE 2 POPULATION PASSAGES")
display(phase2_population_passages)

# Extract complete relevant Phase 1 passages for comparison.
phase1_sections = [
    (8, 15),
    (35, 46),
    (74, 84)
]

phase1_passages = []

for start_line, end_line in phase1_sections:
    for line_number in range(start_line, min(end_line, len(phase1_lines)) + 1):
        text_line = phase1_lines[line_number - 1].strip()

        if text_line:
            phase1_passages.append({
                "section_range": f"{start_line}-{end_line}",
                "line_number": line_number,
                "report_text": text_line
            })

phase1_population_passages = pd.DataFrame(phase1_passages)

print("\n2. COMPLETE PHASE 1 POPULATION PASSAGES")
display(phase1_population_passages)

1. COMPLETE PHASE 2 POPULATION PASSAGES


,section_range,line_number,report_text
0,1-25,1,# SupeRR Selector Season 2
1,1-25,2,## Phase 2 — Data Cleaning & Reconstruction Report
2,1-25,4,**Status:** Final validation and export completed; report pending review
3,1-25,5,**Primary output:** Integrated regulation-delivery dataset
4,1-25,7,---
5,1-25,9,## 1. Executive Summary
6,1-25,11,"Phase 2 focused on data cleaning and reconstruction of the cricket delivery dataset, including match eligibility, player identity, delivery classification, dismissal handling, innings and over sequence integrity, and preparation of an integrated analytical dataset."
7,1-25,13,"The approved target population contained 853 matches. After applying the established eligibility criteria, 813 matches were retained, containing 192,235 regulation deliveries. A further 26 Super Over deliveries were retained separately."
8,1-25,15,The final integrated dataset contains:
9,1-25,17,"- 192,235 regulation delivery records"



2. COMPLETE PHASE 1 POPULATION PASSAGES


,section_range,line_number,report_text
0,8-15,9,1. Executive summary
1,8-15,10,"Phase 1 audited six supplied CSV datasets: ball-by-ball deliveries, playing-XI/player registrations, matches, competition-team-player mappings, teams and competitions. The audit examined dataset structure, identifier relationships, player identity, team and competition consistency, delivery-level fields, playing-XI structure, competition coverage, match eligibility and the proposed primary delivery population."
2,8-15,11,"The approved primary population contains 813 matches: 362 T20 and 451 T20I. These matches are represented in 192,235 regulation deliveries from innings 1 and 2. Four Super Over matches contribute 26 additional deliveries, retained separately. The full target competition set contains 853 matches: 371 T20 and 482 T20I."
3,8-15,12,"Most match, team, competition and player-mapping relationships reconciled. The audit also found field-semantic ambiguity and localized source inconsistencies. The user approved retaining those issues as documented limitations, without silent correction or inferred replacement values."
4,8-15,13,2. Scope and dataset inventory
5,8-15,15,Zero exact duplicate rows does not establish uniqueness at every business key. The playing_eleven table contains repeated match-team-player registrations with differing batting-order values.
6,35-46,35,5. Competition and season coverage
7,35-46,37,"The competitions reference contains 141 records: 139 target IPL/T20I competitions and two out-of-scope T20 competitions with match data. Three target T20I competition records have no match records. Coverage spans IPL seasons 2022–2026 and T20I season labels 2022, 2022/23, 2023, 2023/24, 2024, 2024/25, 2025 and 2026. Missing delivery matches in the target set are abandoned matches."
8,35-46,38,6. Match eligibility and exclusions
9,35-46,40,The three abandoned matches with partial data contain only one innings each. The two No Result matches with two innings were inspected and found incomplete. Four Super Over matches are identifiable by match result and innings 3/4 records. Their regulation innings remain in the primary population.


In [45]:
import pandas as pd
from IPython.display import display

# Classify the 70 absent matches using observed result and source evidence.
reconciliation_70 = investigation_54.copy()

# Add the 16 matches with no source deliveries.
no_delivery_matches = excluded_source_audit.loc[
    excluded_source_audit["original_ball_by_ball_records"].eq(0)
].copy()

no_delivery_matches["source_innings_count"] = 0
no_delivery_matches["source_innings_values"] = ""
no_delivery_matches["first_innings_rows"] = 0
no_delivery_matches["second_innings_rows"] = 0

reconciliation_70 = pd.concat(
    [reconciliation_70, no_delivery_matches],
    ignore_index=True,
    sort=False
)

# Apply explicit evidence-based categories.
conditions = [
    (
        reconciliation_70["result"].eq("Match Abandoned")
        & reconciliation_70["original_ball_by_ball_records"].eq(0)
    ),
    (
        reconciliation_70["result"].eq("Match Abandoned")
        & reconciliation_70["original_ball_by_ball_records"].gt(0)
    ),
    (
        reconciliation_70["result"].eq("No Result")
        & reconciliation_70["source_innings_count"].eq(1)
    ),
    (
        reconciliation_70["result"].eq("No Result")
        & reconciliation_70["source_innings_count"].eq(2)
    )
]

categories = [
    "ABANDONED_NO_DELIVERIES",
    "ABANDONED_WITH_DELIVERIES",
    "NO_RESULT_ONE_INNINGS",
    "NO_RESULT_TWO_INNINGS"
]

reconciliation_70["reconciliation_category"] = "OTHER_RECORDED_RESULT"

reconciliation_70.loc[
    conditions[0], "reconciliation_category"
] = categories[0]

reconciliation_70.loc[
    conditions[1], "reconciliation_category"
] = categories[1]

reconciliation_70.loc[
    conditions[2], "reconciliation_category"
] = categories[2]

reconciliation_70.loc[
    conditions[3], "reconciliation_category"
] = categories[3]

# Summarize the categories.
category_summary = (
    reconciliation_70.groupby(
        "reconciliation_category",
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        source_delivery_records=(
            "original_ball_by_ball_records", "sum"
        )
    )
    .reset_index()
)

category_summary["phase2_reported_count"] = (
    category_summary["reconciliation_category"]
    .map({
        "ABANDONED_NO_DELIVERIES": 16,
        "ABANDONED_WITH_DELIVERIES": 3,
        "NO_RESULT_ONE_INNINGS": 19,
        "NO_RESULT_TWO_INNINGS": 2
    })
)

category_summary["count_difference"] = (
    category_summary["matches"]
    - category_summary["phase2_reported_count"]
)

print("1. DOCUMENTED EXCLUSION CATEGORY RECONCILIATION")
display(category_summary)

# Compare the 40 documented categories with the additional matches.
documented_categories = categories

reconciliation_70["documented_40_match_group"] = (
    reconciliation_70["reconciliation_category"]
    .isin(documented_categories)
    .map({
        True: "MATCHES_IN_PHASE2_DOCUMENTED_CATEGORIES",
        False: "ADDITIONAL_MATCHES_NOT_IN_DOCUMENTED_CATEGORIES"
    })
)

group_summary = (
    reconciliation_70.groupby(
        "documented_40_match_group",
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        source_delivery_records=(
            "original_ball_by_ball_records", "sum"
        ),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
)

print("\n2. DOCUMENTED 40 VERSUS ADDITIONAL MATCHES")
display(group_summary)

# Identify the additional matches in detail.
additional_matches = (
    reconciliation_70.loc[
        reconciliation_70["documented_40_match_group"]
        == "ADDITIONAL_MATCHES_NOT_IN_DOCUMENTED_CATEGORIES",
        [
            "match_id",
            "comp_id",
            "match_number",
            "match_date",
            "result",
            "original_ball_by_ball_records",
            "source_innings_count"
        ]
    ]
    .sort_values(["comp_id", "match_id"])
    .reset_index(drop=True)
)

print("\n3. ADDITIONAL MATCHES NOT EXPLAINED BY THE 40 DOCUMENTED CATEGORIES")
display(additional_matches)

# Validate the reconciliation.
validation = pd.DataFrame([
    {
        "Validation": "Total absent matches equal 70",
        "Status": len(reconciliation_70) == 70
    },
    {
        "Validation": "Abandoned without deliveries equals 16",
        "Status": (
            (reconciliation_70["reconciliation_category"]
             == "ABANDONED_NO_DELIVERIES").sum() == 16
        )
    },
    {
        "Validation": "Abandoned with deliveries equals 3",
        "Status": (
            (reconciliation_70["reconciliation_category"]
             == "ABANDONED_WITH_DELIVERIES").sum() == 3
        )
    },
    {
        "Validation": "No Result with one innings equals 19",
        "Status": (
            (reconciliation_70["reconciliation_category"]
             == "NO_RESULT_ONE_INNINGS").sum() == 19
        )
    },
    {
        "Validation": "No Result with two innings equals 2",
        "Status": (
            (reconciliation_70["reconciliation_category"]
             == "NO_RESULT_TWO_INNINGS").sum() == 2
        )
    },
    {
        "Validation": "Additional matches equal 30",
        "Status": len(additional_matches) == 30
    },
    {
        "Validation": "Documented categories total 40",
        "Status": (
            reconciliation_70["documented_40_match_group"]
            .eq("MATCHES_IN_PHASE2_DOCUMENTED_CATEGORIES")
            .sum() == 40
        )
    }
])

validation["Status"] = validation["Status"].map({
    True: "PASS",
    False: "FAIL"
})

print("\n4. VALIDATION RESULTS")
display(validation)

1. DOCUMENTED EXCLUSION CATEGORY RECONCILIATION


,reconciliation_category,matches,source_delivery_records,phase2_reported_count,count_difference
0,ABANDONED_NO_DELIVERIES,16,0,16.0,0.0
1,ABANDONED_WITH_DELIVERIES,3,258,3.0,0.0
2,NO_RESULT_ONE_INNINGS,19,1265,19.0,0.0
3,NO_RESULT_TWO_INNINGS,2,230,2.0,0.0
4,OTHER_RECORDED_RESULT,30,6985,NaN,NaN



2. DOCUMENTED 40 VERSUS ADDITIONAL MATCHES


,documented_40_match_group,matches,source_delivery_records,competitions
0,ADDITIONAL_MATCHES_NOT_IN_DOCUMENTED_CATEGORIES,30,6985,2
1,MATCHES_IN_PHASE2_DOCUMENTED_CATEGORIES,40,1753,30



3. ADDITIONAL MATCHES NOT EXPLAINED BY THE 40 DOCUMENTED CATEGORIES


,match_id,comp_id,match_number,match_date,result,original_ball_by_ball_records,source_innings_count
0,37085,2670,1,2024-10-18,Bangladesh A Won by 5 wickets,236,2
1,37086,2670,2,2024-10-18,Afghanistan A Won by 11 runs,244,2
2,37087,2670,3,2024-10-19,United Arab Emirates Won by 4 wickets,241,2
3,37088,2670,4,2024-10-19,India A Won by 7 runs,250,2
4,37089,2670,5,2024-10-20,Sri Lanka A Won by 42 runs,244,2
5,37090,2670,6,2024-10-20,Afghanistan A Won by 4 wickets,244,2
6,37091,2670,7,2024-10-21,Pakistan A Won by 74 runs,254,2
7,37092,2670,8,2024-10-21,India A Won by 7 wickets,172,2
8,37093,2670,9,2024-10-22,Hong Kong Won by 5 wickets,239,2
9,37094,2670,10,2024-10-22,Sri Lanka A Won by 19 runs,247,2



4. VALIDATION RESULTS


,Validation,Status
0,Total absent matches equal 70,PASS
1,Abandoned without deliveries equals 16,PASS
2,Abandoned with deliveries equals 3,PASS
3,No Result with one innings equals 19,PASS
4,No Result with two innings equals 2,PASS
5,Additional matches equal 30,PASS
6,Documented categories total 40,PASS


### Approved Population Boundary Review for the 30 Additional Matches

In [46]:
# Extract the relevant population statements from the Phase 1 and Phase 2 reports.
population_terms = [
    "approved primary population",
    "target population",
    "853 matches",
    "813 matches",
    "371 T20",
    "362 T20",
    "competition",
    "excluded"
]

report_evidence = []

for report_name, report_lines in [
    ("Phase 1 Report", phase1_lines),
    ("Phase 2 Report", phase2_lines)
]:
    for line_number, line in enumerate(report_lines, start=1):
        if any(term.lower() in line.lower() for term in population_terms):
            report_evidence.append({
                "report": report_name,
                "line_number": line_number,
                "report_text": line.strip()
            })

report_evidence_df = pd.DataFrame(report_evidence)

print("1. POPULATION-RELATED REPORT EVIDENCE")
display(report_evidence_df)

# Inspect the source metadata for the two additional competitions.
additional_competition_ids = [2670, 3289]

additional_competition_metadata = (
    df_source_competitions.loc[
        df_source_competitions["comp_id"].isin(
            additional_competition_ids
        )
    ]
    .sort_values("comp_id")
    .reset_index(drop=True)
)

print("2. METADATA FOR THE TWO ADDITIONAL COMPETITIONS")
display(additional_competition_metadata)

# Compare the competitions against the integrated analytical population.
integrated_competition_ids = set(
    df_deliveries["comp_id"].dropna().unique()
)

source_competition_population = (
    df_source_matches.loc[
        df_source_matches["comp_id"].isin(
            additional_competition_ids
        )
    ]
    .groupby("comp_id")
    .agg(
        source_matches=("match_id", "nunique")
    )
    .reset_index()
)

source_competition_population["integrated_matches"] = (
    source_competition_population["comp_id"].map(
        df_deliveries.groupby("comp_id")["match_id"].nunique()
    ).fillna(0).astype(int)
)

source_competition_population["present_in_integrated_population"] = (
    source_competition_population["comp_id"].isin(
        integrated_competition_ids
    )
)

print("3. SOURCE VERSUS INTEGRATED POPULATION")
display(source_competition_population)

1. POPULATION-RELATED REPORT EVIDENCE


,report,line_number,report_text
0,Phase 1 Report,10,"Phase 1 audited six supplied CSV datasets: ball-by-ball deliveries, playing-XI/player registrations, matches, competition-team-player mappings, teams and competitions. The audit examined dataset structure, identifier relationships, player identity, team and competition consistency, delivery-level fields, playing-XI structure, competition coverage, match eligibility and the proposed primary delivery population."
1,Phase 1 Report,11,"The approved primary population contains 813 matches: 362 T20 and 451 T20I. These matches are represented in 192,235 regulation deliveries from innings 1 and 2. Four Super Over matches contribute 26 additional deliveries, retained separately. The full target competition set contains 853 matches: 371 T20 and 482 T20I."
2,Phase 1 Report,12,"Most match, team, competition and player-mapping relationships reconciled. The audit also found field-semantic ambiguity and localized source inconsistencies. The user approved retaining those issues as documented limitations, without silent correction or inferred replacement values."
3,Phase 1 Report,20,"All home/away team IDs, match competition IDs, mapping competition IDs and match home/away team-competition pairs reconciled to their reference tables."
4,Phase 1 Report,21,"All actual delivery-role competition-team-player combinations were present in the mapping: striker 4,327/4,327; non-striker 4,165/4,165; bowler 3,101/3,101."
5,Phase 1 Report,24,"4. Team, competition and playing-XI consistency"
6,Phase 1 Report,25,"Of 5,646 unique playing-XI match-team-player combinations, 5,645 matched the competition-team-player mapping."
7,Phase 1 Report,27,All delivery-role competition-team-player combinations were mapped.
8,Phase 1 Report,35,5. Competition and season coverage
9,Phase 1 Report,37,"The competitions reference contains 141 records: 139 target IPL/T20I competitions and two out-of-scope T20 competitions with match data. Three target T20I competition records have no match records. Coverage spans IPL seasons 2022–2026 and T20I season labels 2022, 2022/23, 2023, 2023/24, 2024, 2024/25, 2025 and 2026. Missing delivery matches in the target set are abandoned matches."


2. METADATA FOR THE TWO ADDITIONAL COMPETITIONS


,comp_id,comp_name,match_format,season
0,2670,Mens T20 Emerging Teams Asia Cup 2024,T20,2024
1,3289,Asia Cup Rising Stars 2025,T20,2025


3. SOURCE VERSUS INTEGRATED POPULATION


,comp_id,source_matches,integrated_matches,present_in_integrated_population
0,2670,15,0,False
1,3289,15,0,False


## Inspect the full population and competition-scope statements

In [47]:
passage_requests = [
    {
        "report": "Phase 1 Report",
        "lines": phase1_lines,
        "start": 8,
        "end": 15
    },
    {
        "report": "Phase 1 Report",
        "lines": phase1_lines,
        "start": 33,
        "end": 43
    },
    {
        "report": "Phase 2 Report",
        "lines": phase2_lines,
        "start": 10,
        "end": 22
    },
    {
        "report": "Phase 2 Report",
        "lines": phase2_lines,
        "start": 32,
        "end": 49
    },
    {
        "report": "Phase 2 Report",
        "lines": phase2_lines,
        "start": 145,
        "end": 172
    }
]

full_passages = []

for request in passage_requests:
    for line_number in range(
        request["start"],
        min(request["end"], len(request["lines"])) + 1
    ):
        full_passages.append({
            "report": request["report"],
            "line_number": line_number,
            "full_text": request["lines"][line_number - 1].rstrip()
        })

full_passages_df = pd.DataFrame(full_passages)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

display(full_passages_df)

,report,line_number,full_text
0,Phase 1 Report,8,
1,Phase 1 Report,9,1. Executive summary
2,Phase 1 Report,10,"Phase 1 audited six supplied CSV datasets: ball-by-ball deliveries, playing-XI/player registrations, matches, competition-team-player mappings, teams and competitions. The audit examined dataset structure, identifier relationships, player identity, team and competition consistency, delivery-level fields, playing-XI structure, competition coverage, match eligibility and the proposed primary delivery population."
3,Phase 1 Report,11,"The approved primary population contains 813 matches: 362 T20 and 451 T20I. These matches are represented in 192,235 regulation deliveries from innings 1 and 2. Four Super Over matches contribute 26 additional deliveries, retained separately. The full target competition set contains 853 matches: 371 T20 and 482 T20I."
4,Phase 1 Report,12,"Most match, team, competition and player-mapping relationships reconciled. The audit also found field-semantic ambiguity and localized source inconsistencies. The user approved retaining those issues as documented limitations, without silent correction or inferred replacement values."
5,Phase 1 Report,13,2. Scope and dataset inventory
6,Phase 1 Report,14,
7,Phase 1 Report,15,Zero exact duplicate rows does not establish uniqueness at every business key. The playing_eleven table contains repeated match-team-player registrations with differing batting-order values.
8,Phase 1 Report,33,"Twenty-three match-team groups have zero played records. They belong to 12 abandoned matches, all with zero ball-by-ball deliveries."
9,Phase 1 Report,34,"There were 512 played-XI registrations with no striker, non-striker or bowler delivery involvement. For 511, the listed batting order was below the maximum observed striker order in the team innings, consistent with players who did not come to bat; this does not establish their exact on-field role. One exception is Mosaddek Hossain (player_id 3281), Bangladesh, match 51385: listed as is_played=True at batting order 6, but absent from all delivery roles despite a substantial Bangladesh second innings (121 deliveries, six wickets). The reason is unknown; retain as an unresolved XI/delivery-feed inconsistency."


## Population Reconciliation Closure

In [48]:
population_closure = pd.DataFrame([
    {
        "reconciliation_item": "Approved target population",
        "matches": 853,
        "status": "RECONCILED",
        "evidence": "Phase 1 and Phase 2 reports"
    },
    {
        "reconciliation_item": "Eligible integrated population",
        "matches": 813,
        "status": "RECONCILED",
        "evidence": "Phase 2 integrated dataset"
    },
    {
        "reconciliation_item": "Documented eligibility exclusions",
        "matches": 40,
        "status": "RECONCILED",
        "evidence": "All four Phase 2 exclusion categories matched"
    },
    {
        "reconciliation_item": "Out-of-scope T20 competition matches",
        "matches": 30,
        "status": "RECONCILED",
        "evidence": "Phase 1 report, competition coverage section"
    },
    {
        "reconciliation_item": "Current source matches",
        "matches": 883,
        "status": "RECONCILED",
        "evidence": "853 target matches plus 30 out-of-scope matches"
    }
])

display(population_closure)

# Arithmetic and population checks.
population_validation = pd.DataFrame([
    {
        "validation": "Target equals eligible plus documented exclusions",
        "status": 813 + 40 == 853
    },
    {
        "validation": "Source matches equal target plus out-of-scope matches",
        "status": 853 + 30 == 883
    },
    {
        "validation": "All 30 out-of-scope matches belong to two competitions",
        "status": (
            additional_matches["comp_id"].nunique() == 2
            and additional_matches["match_id"].nunique() == 30
        )
    },
    {
        "validation": "All 40 documented exclusions reconciled",
        "status": (
            category_summary.loc[
                category_summary["reconciliation_category"].isin(
                    categories
                ),
                "matches"
            ].sum() == 40
        )
    }
])

population_validation["status"] = (
    population_validation["status"]
    .map({True: "PASS", False: "FAIL"})
)

display(population_validation)

assert population_validation["status"].eq("PASS").all()

population_closure.to_csv(
    OUTPUT_DIR / "phase3A_population_reconciliation_closure.csv",
    index=False
)

population_validation.to_csv(
    OUTPUT_DIR / "phase3A_population_reconciliation_validation.csv",
    index=False
)

,reconciliation_item,matches,status,evidence
0,Approved target population,853,RECONCILED,Phase 1 and Phase 2 reports
1,Eligible integrated population,813,RECONCILED,Phase 2 integrated dataset
2,Documented eligibility exclusions,40,RECONCILED,All four Phase 2 exclusion categories matched
3,Out-of-scope T20 competition matches,30,RECONCILED,"Phase 1 report, competition coverage section"
4,Current source matches,883,RECONCILED,853 target matches plus 30 out-of-scope matches


,validation,status
0,Target equals eligible plus documented exclusions,PASS
1,Source matches equal target plus out-of-scope matches,PASS
2,All 30 out-of-scope matches belong to two competitions,PASS
3,All 40 documented exclusions reconciled,PASS


# Match-state feasibility

## Pre-delivery Match-State Field Availability Audit

In [49]:
required_field_groups = {
    "Match identity": [
        "match_id"
    ],
    "Innings identity": [
        "innings"
    ],
    "Team identity": [
        "batting_team_id",
        "bowling_team_id"
    ],
    "Delivery ordering": [
        "over_ball_num",
        "timestamp"
    ],
    "Scoring evidence": [
        "runs",
        "extras1_runs",
        "extras_2_runs",
        "boundary"
    ],
    "Dismissal evidence": [
        "wicket",
        "player_out_id",
        "dismissal_kind"
    ],
    "Innings length metadata": [
        "revised_overs"
    ]
}

available_columns = set(df_deliveries.columns)

field_audit_rows = []

for group_name, fields in required_field_groups.items():
    for field in fields:
        exists = field in available_columns

        if exists:
            missing_count = int(df_deliveries[field].isna().sum())
            non_missing_count = int(df_deliveries[field].notna().sum())
            dtype = str(df_deliveries[field].dtype)
        else:
            missing_count = None
            non_missing_count = None
            dtype = "NOT AVAILABLE"

        field_audit_rows.append({
            "field_group": group_name,
            "field": field,
            "available": exists,
            "dtype": dtype,
            "non_missing_rows": non_missing_count,
            "missing_rows": missing_count
        })

field_availability_audit = pd.DataFrame(field_audit_rows)

print("1. FIELD-LEVEL AVAILABILITY")
display(field_availability_audit)

field_group_summary = (
    field_availability_audit.groupby("field_group")
    .agg(
        fields_audited=("field", "count"),
        fields_available=("available", "sum"),
        fields_missing=("available", lambda x: int((~x).sum()))
    )
    .reset_index()
)

print("2. FIELD-GROUP SUMMARY")
display(field_group_summary)

1. FIELD-LEVEL AVAILABILITY


,field_group,field,available,dtype,non_missing_rows,missing_rows
0,Match identity,match_id,True,int64,192235.0,0.0
1,Innings identity,innings,True,int64,192235.0,0.0
2,Team identity,batting_team_id,True,int64,192235.0,0.0
3,Team identity,bowling_team_id,True,int64,192235.0,0.0
4,Delivery ordering,over_ball_num,True,int64,192235.0,0.0
5,Delivery ordering,timestamp,False,NOT AVAILABLE,NaN,NaN
6,Scoring evidence,runs,True,int64,192235.0,0.0
7,Scoring evidence,extras1_runs,True,int64,192235.0,0.0
8,Scoring evidence,extras_2_runs,True,int64,192235.0,0.0
9,Scoring evidence,boundary,False,NOT AVAILABLE,NaN,NaN


2. FIELD-GROUP SUMMARY


,field_group,fields_audited,fields_available,fields_missing
0,Delivery ordering,2,1,1
1,Dismissal evidence,3,0,3
2,Innings identity,1,1,0
3,Innings length metadata,1,0,1
4,Match identity,1,1,0
5,Scoring evidence,4,3,1
6,Team identity,2,2,0


## Integrated and Original Source Schema Mapping

In [50]:
schema_rows = []

for dataset_name, dataframe in [
    ("Phase 2 integrated deliveries", df_deliveries),
    ("Original ball-by-ball source", df_source_ball),
    ("Original matches source", df_source_matches)
]:
    for position, column in enumerate(dataframe.columns):
        schema_rows.append({
            "dataset": dataset_name,
            "column_position": position,
            "column_name": column,
            "dtype": str(dataframe[column].dtype),
            "non_missing_rows": int(dataframe[column].notna().sum()),
            "missing_rows": int(dataframe[column].isna().sum())
        })

complete_schema_audit = pd.DataFrame(schema_rows)

print("1. COMPLETE DATASET SCHEMA")
display(complete_schema_audit)

# Identify potentially relevant fields by name.
relevant_terms = [
    "run", "extra", "wicket", "dismiss", "out",
    "boundary", "over", "ball", "time", "date",
    "innings", "target", "revised", "legal"
]

relevant_schema = complete_schema_audit.loc[
    complete_schema_audit["column_name"].str.lower().apply(
        lambda name: any(term in name for term in relevant_terms)
    )
].reset_index(drop=True)

print("2. POTENTIALLY RELEVANT FIELD NAMES")
display(relevant_schema)

1. COMPLETE DATASET SCHEMA


,dataset,column_position,column_name,dtype,non_missing_rows,missing_rows
0,Phase 2 integrated deliveries,0,match_id,int64,192235,0
1,Phase 2 integrated deliveries,1,innings,int64,192235,0
2,Phase 2 integrated deliveries,2,overid,int64,192235,0
3,Phase 2 integrated deliveries,3,over_ball_num,int64,192235,0
4,Phase 2 integrated deliveries,4,innings_ball_num,int64,192235,0
...,...,...,...,...,...,...
111,Original matches source,17,second_inn_revised_overs,float64,25,858
112,Original matches source,18,revised_target,float64,23,860
113,Original matches source,19,knock_out,int64,883,0
114,Original matches source,20,win_team_id,float64,842,41


2. POTENTIALLY RELEVANT FIELD NAMES


,dataset,column_position,column_name,dtype,non_missing_rows,missing_rows
0,Phase 2 integrated deliveries,1,innings,int64,192235,0
1,Phase 2 integrated deliveries,2,overid,int64,192235,0
2,Phase 2 integrated deliveries,3,over_ball_num,int64,192235,0
3,Phase 2 integrated deliveries,4,innings_ball_num,int64,192235,0
4,Phase 2 integrated deliveries,16,runs,int64,192235,0
5,Phase 2 integrated deliveries,19,is_wicket,bool,192235,0
6,Phase 2 integrated deliveries,20,dismissal_type_id,float64,10392,181843
7,Phase 2 integrated deliveries,21,dismissal_type,object,10392,181843
8,Phase 2 integrated deliveries,22,dismissed_player_id,float64,10402,181833
9,Phase 2 integrated deliveries,23,extras_type,object,11373,180862


### Validate Candidate Delivery-State Fields

In [52]:
# 1. Profile the principal candidate fields.
candidate_fields = [
    "runs",
    "extras1_runs",
    "extras_2_runs",
    "is_wicket",
    "dismissal_type",
    "dismissal_type_normalized",
    "dismissed_player_id",
    "extras_type",
    "legal_ball_evidence_class",
    "legal_ball_evidence_status",
    "legal_ball_final_treatment"
]

field_profile_rows = []

for field in candidate_fields:
    if field not in df_deliveries.columns:
        continue

    series = df_deliveries[field]

    field_profile_rows.append({
        "field": field,
        "dtype": str(series.dtype),
        "non_missing": int(series.notna().sum()),
        "missing": int(series.isna().sum()),
        "unique_values": int(series.nunique(dropna=True)),
        "most_common_values": str(
            series.value_counts(dropna=False).head(12).to_dict()
        )
    })

candidate_field_profile = pd.DataFrame(field_profile_rows)

print("1. CANDIDATE FIELD PROFILE")
display(candidate_field_profile)


# 2. Inspect the encoded values of the scoring and extras fields.
value_profile_rows = []

for field in ["runs", "extras1_runs", "extras_2_runs"]:
    if field not in df_deliveries.columns:
        continue

    counts = (
        df_deliveries[field]
        .value_counts(dropna=False)
        .sort_index()
    )

    for value, count in counts.items():
        value_profile_rows.append({
            "field": field,
            "encoded_value": value,
            "delivery_count": int(count),
            "share_pct": round(
                100 * count / len(df_deliveries), 4
            )
        })

scoring_value_profile = pd.DataFrame(value_profile_rows)

print("2. SCORING AND EXTRAS VALUE DISTRIBUTIONS")
display(scoring_value_profile)


# 3. Test innings_ball_num sequence behaviour.
sequence_data = df_deliveries[
    [
        "match_id",
        "innings",
        "overid",
        "over_ball_num",
        "innings_ball_num"
    ]
].copy()

sequence_data = sequence_data.sort_values(
    ["match_id", "innings", "overid", "over_ball_num"],
    kind="stable"
)

sequence_data["sequence_difference"] = (
    sequence_data.groupby(["match_id", "innings"])["innings_ball_num"]
    .diff()
)

sequence_summary = (
    sequence_data.groupby(["match_id", "innings"])
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        unique_innings_ball_numbers=("innings_ball_num", "nunique"),
        minimum_innings_ball_num=("innings_ball_num", "min"),
        maximum_innings_ball_num=("innings_ball_num", "max"),
        duplicate_innings_ball_numbers=(
            "innings_ball_num",
            lambda values: int(values.duplicated().sum())
        ),
        non_increasing_transitions=(
            "sequence_difference",
            lambda values: int((values <= 0).sum())
        ),
        increments_of_one=(
            "sequence_difference",
            lambda values: int((values == 1).sum())
        ),
        increments_greater_than_one=(
            "sequence_difference",
            lambda values: int((values > 1).sum())
        )
    )
    .reset_index()
)

print("3. INNINGS SEQUENCE SUMMARY — FIRST 25 INNINGS")
display(sequence_summary.head(25))

sequence_overall = pd.DataFrame([{
    "innings_groups": len(sequence_summary),
    "groups_with_duplicate_numbers": int(
        (sequence_summary["duplicate_innings_ball_numbers"] > 0).sum()
    ),
    "groups_with_non_increasing_transitions": int(
        (sequence_summary["non_increasing_transitions"] > 0).sum()
    ),
    "groups_with_gaps": int(
        (sequence_summary["increments_greater_than_one"] > 0).sum()
    ),
    "groups_where_unique_count_equals_delivery_rows": int(
        (
            sequence_summary["unique_innings_ball_numbers"]
            == sequence_summary["delivery_rows"]
        ).sum()
    )
}])

print("4. OVERALL SEQUENCE VALIDATION")
display(sequence_overall)


# 4. Compare wicket indicator with dismissal evidence.
wicket_audit = pd.DataFrame([{
    "total_deliveries": len(df_deliveries),
    "is_wicket_true": int(df_deliveries["is_wicket"].sum()),
    "dismissal_type_present": int(
        df_deliveries["dismissal_type"].notna().sum()
    ),
    "dismissed_player_id_present": int(
        df_deliveries["dismissed_player_id"].notna().sum()
    ),
    "wicket_true_but_no_dismissal_type": int(
        (
            df_deliveries["is_wicket"]
            & df_deliveries["dismissal_type"].isna()
        ).sum()
    ),
    "wicket_false_but_dismissal_type_present": int(
        (
            ~df_deliveries["is_wicket"]
            & df_deliveries["dismissal_type"].notna()
        ).sum()
    ),
    "wicket_true_but_no_dismissed_player": int(
        (
            df_deliveries["is_wicket"]
            & df_deliveries["dismissed_player_id"].isna()
        ).sum()
    ),
    "wicket_false_but_dismissed_player_present": int(
        (
            ~df_deliveries["is_wicket"]
            & df_deliveries["dismissed_player_id"].notna()
        ).sum()
    )
}])

print("5. WICKET AND DISMISSAL CONSISTENCY")
display(wicket_audit)


# 5. Compare candidate fields in the original source and integrated data.
source_comparison_rows = []

for field in candidate_fields:
    if field not in df_source_ball.columns or field not in df_deliveries.columns:
        continue

    source_series = df_source_ball[field]
    integrated_series = df_deliveries[field]

    source_comparison_rows.append({
        "field": field,
        "source_rows": len(source_series),
        "integrated_rows": len(integrated_series),
        "source_unique_values": int(source_series.nunique(dropna=True)),
        "integrated_unique_values": int(integrated_series.nunique(dropna=True)),
        "source_non_missing": int(source_series.notna().sum()),
        "integrated_non_missing": int(integrated_series.notna().sum())
    })

source_comparison = pd.DataFrame(source_comparison_rows)

print("6. ORIGINAL SOURCE VS INTEGRATED FIELD COVERAGE")
display(source_comparison)

1. CANDIDATE FIELD PROFILE


,field,dtype,non_missing,missing,unique_values,most_common_values
0,runs,int64,192235,0,7,"{0: 75779, 1: 68986, 4: 22490, 6: 12296, 2: 12036, 3: 626, 5: 22}"
1,extras1_runs,int64,192235,0,7,"{6: 180862, 2: 7419, 3: 2635, 1: 557, 4: 505, 5: 256, 7: 1}"
2,extras_2_runs,int64,192235,0,3,"{0: 192200, 1: 19, 4: 16}"
3,is_wicket,bool,192235,0,2,"{False: 181843, True: 10392}"
4,dismissal_type,object,10392,181843,9,"{None: 181843, 'Caught': 7264, 'Bowled': 1675, 'LBW': 663, 'RunOut': 567, 'Stump': 198, 'Hit Wicket': 17, 'Retire Out': 6, 'Obstracting the field': 1, 'Obstructing the field': 1}"
5,dismissal_type_normalized,object,10392,181843,8,"{None: 181843, 'Caught': 7264, 'Bowled': 1675, 'LBW': 663, 'RunOut': 567, 'Stump': 198, 'Hit Wicket': 17, 'Retire Out': 6, 'Obstructing the field': 2}"
6,dismissed_player_id,float64,10402,181833,758,"{nan: 181833, 6281.0: 123, 1699.0: 119, 4117.0: 113, 2258.0: 108, 813.0: 106, 6288.0: 100, 3125.0: 94, 3330.0: 90, 126.0: 86, 7897.0: 85, 4986.0: 83}"
7,extras_type,object,11373,180862,6,"{None: 180862, 'wide': 7419, 'legbyes': 2635, 'Overstep': 557, 'byes': 505, 'other': 256, 'Penalty': 1}"
8,legal_ball_evidence_class,object,192235,0,7,"{'RECORDED_NO_EXTRAS': 180862, 'RECORDED_WIDE': 7419, 'RECORDED_LEGBYES': 2635, 'RECORDED_OVERSTEP': 557, 'RECORDED_BYES': 505, 'RECORDED_OTHER': 256, 'RECORDED_PENALTY': 1}"
9,legal_ball_evidence_status,object,192235,0,3,"{'NO_RECORDED_WIDE_OR_OVERSTEP': 184002, 'RECORDED_NON_LEGAL_EVIDENCE': 7976, 'UNRESOLVED': 257}"


2. SCORING AND EXTRAS VALUE DISTRIBUTIONS


,field,encoded_value,delivery_count,share_pct
0,runs,0,75779,39.4200
1,runs,1,68986,35.8863
2,runs,2,12036,6.2611
3,runs,3,626,0.3256
4,runs,4,22490,11.6992
5,runs,5,22,0.0114
6,runs,6,12296,6.3963
7,extras1_runs,1,557,0.2897
8,extras1_runs,2,7419,3.8593
9,extras1_runs,3,2635,1.3707


3. INNINGS SEQUENCE SUMMARY — FIRST 25 INNINGS


,match_id,innings,delivery_rows,unique_innings_ball_numbers,minimum_innings_ball_num,maximum_innings_ball_num,duplicate_innings_ball_numbers,non_increasing_transitions,increments_of_one,increments_greater_than_one
0,13511,1,121,121,1,121,0,0,120,0
1,13511,2,103,103,1,103,0,0,102,0
2,13512,1,128,128,1,128,0,0,127,0
3,13512,2,124,124,1,124,0,0,123,0
4,13513,1,125,125,1,125,0,0,124,0
5,13513,2,127,127,1,127,0,0,126,0
6,13514,1,128,128,1,128,0,0,127,0
7,13514,2,125,125,1,125,0,0,124,0
8,13515,1,127,127,1,127,0,0,126,0
9,13515,2,127,127,1,127,0,0,126,0


... (2 outputs left out)

5. WICKET AND DISMISSAL CONSISTENCY


,total_deliveries,is_wicket_true,dismissal_type_present,dismissed_player_id_present,wicket_true_but_no_dismissal_type,wicket_false_but_dismissal_type_present,wicket_true_but_no_dismissed_player,wicket_false_but_dismissed_player_present
0,192235,10392,10392,10402,0,0,1,11


6. ORIGINAL SOURCE VS INTEGRATED FIELD COVERAGE


,field,source_rows,integrated_rows,source_unique_values,integrated_unique_values,source_non_missing,integrated_non_missing
0,runs,200999,192235,7,7,200999,192235
1,extras1_runs,200999,192235,7,7,200999,192235
2,extras_2_runs,200999,192235,3,3,200999,192235
3,is_wicket,200999,192235,2,2,200999,192235
4,dismissal_type,200999,192235,9,9,10881,10392
5,dismissed_player_id,200999,192235,814,758,10892,10402
6,extras_type,200999,192235,6,6,11832,11373


### Legal-ball Evidence VS Delivery Sequence

In [53]:
# 1. Confirm the evidence categories and their population counts.
legal_evidence_distribution = (
    df_deliveries["legal_ball_final_treatment"]
    .value_counts(dropna=False)
    .rename_axis("legal_ball_final_treatment")
    .reset_index(name="delivery_count")
)

legal_evidence_distribution["share_pct"] = (
    100 * legal_evidence_distribution["delivery_count"]
    / len(df_deliveries)
).round(4)

print("1. LEGAL-BALL EVIDENCE DISTRIBUTION")
display(legal_evidence_distribution)


# 2. Build an innings-level comparison.
legal_sequence_data = df_deliveries[
    [
        "match_id",
        "innings",
        "innings_ball_num",
        "legal_ball_final_treatment"
    ]
].copy()

legal_sequence_data["recorded_non_legal"] = (
    legal_sequence_data["legal_ball_final_treatment"]
    == "NON_LEGAL_EVIDENCE"
)

legal_sequence_data["unresolved"] = (
    legal_sequence_data["legal_ball_final_treatment"]
    == "UNRESOLVED"
)

legal_sequence_data["no_recorded_wide_or_overstep"] = (
    legal_sequence_data["legal_ball_final_treatment"]
    == "NO_RECORDED_WIDE_OR_OVERSTEP"
)

legal_innings_audit = (
    legal_sequence_data.groupby(["match_id", "innings"])
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        minimum_sequence_number=("innings_ball_num", "min"),
        maximum_sequence_number=("innings_ball_num", "max"),
        recorded_non_legal_deliveries=("recorded_non_legal", "sum"),
        unresolved_deliveries=("unresolved", "sum"),
        no_recorded_wide_or_overstep_deliveries=(
            "no_recorded_wide_or_overstep", "sum"
        )
    )
    .reset_index()
)

legal_innings_audit["sequence_matches_delivery_count"] = (
    legal_innings_audit["maximum_sequence_number"]
    == legal_innings_audit["delivery_rows"]
)

legal_innings_audit["potential_legal_lower_bound"] = (
    legal_innings_audit["no_recorded_wide_or_overstep_deliveries"]
)

legal_innings_audit["potential_legal_upper_bound"] = (
    legal_innings_audit["delivery_rows"]
    - legal_innings_audit["recorded_non_legal_deliveries"]
)

print("2. INNINGS-LEVEL DELIVERY VS LEGAL-BALL EVIDENCE — FIRST 25")
display(legal_innings_audit.head(25))


# 3. Summarise the range of potentially legal deliveries.
legal_sequence_overall = pd.DataFrame([{
    "innings_groups": len(legal_innings_audit),
    "groups_where_sequence_max_equals_delivery_rows": int(
        legal_innings_audit["sequence_matches_delivery_count"].sum()
    ),
    "total_delivery_rows": int(
        legal_innings_audit["delivery_rows"].sum()
    ),
    "recorded_non_legal_deliveries": int(
        legal_innings_audit["recorded_non_legal_deliveries"].sum()
    ),
    "unresolved_deliveries": int(
        legal_innings_audit["unresolved_deliveries"].sum()
    ),
    "no_recorded_wide_or_overstep_deliveries": int(
        legal_innings_audit[
            "no_recorded_wide_or_overstep_deliveries"
        ].sum()
    ),
    "potential_legal_lower_bound": int(
        legal_innings_audit["potential_legal_lower_bound"].sum()
    ),
    "potential_legal_upper_bound": int(
        legal_innings_audit["potential_legal_upper_bound"].sum()
    ),
    "innings_over_120_delivery_rows": int(
        (legal_innings_audit["delivery_rows"] > 120).sum()
    ),
    "maximum_delivery_rows_in_an_innings": int(
        legal_innings_audit["delivery_rows"].max()
    )
}])

print("3. OVERALL LEGAL-BALL EVIDENCE BOUNDS")
display(legal_sequence_overall)


# 4. Inspect innings with more than 120 delivery records.
extended_innings_audit = (
    legal_innings_audit.loc[
        legal_innings_audit["delivery_rows"] > 120
    ]
    .sort_values(
        ["delivery_rows", "recorded_non_legal_deliveries"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("4. INNINGS WITH MORE THAN 120 DELIVERY RECORDS")
display(extended_innings_audit.head(40))


# 5. Verify that the evidence categories partition the population.
category_partition_check = pd.DataFrame([{
    "integrated_delivery_rows": len(df_deliveries),
    "classified_delivery_rows": int(
        legal_evidence_distribution["delivery_count"].sum()
    ),
    "partition_matches_population": (
        int(legal_evidence_distribution["delivery_count"].sum())
        == len(df_deliveries)
    ),
    "missing_evidence_classifications": int(
        df_deliveries["legal_ball_final_treatment"].isna().sum()
    )
}])

print("5. EVIDENCE PARTITION VALIDATION")
display(category_partition_check)

assert category_partition_check.loc[
    0, "partition_matches_population"
], "Legal-ball evidence categories do not partition the integrated population."

assert category_partition_check.loc[
    0, "missing_evidence_classifications"
] == 0, "Missing legal-ball evidence classifications detected."

assert (
    legal_innings_audit["sequence_matches_delivery_count"].all()
), "At least one innings sequence maximum does not equal its delivery-row count."

1. LEGAL-BALL EVIDENCE DISTRIBUTION


,legal_ball_final_treatment,delivery_count,share_pct
0,NO_RECORDED_WIDE_OR_OVERSTEP,184002,95.7172
1,NON_LEGAL_EVIDENCE,7976,4.1491
2,UNRESOLVED,257,0.1337


2. INNINGS-LEVEL DELIVERY VS LEGAL-BALL EVIDENCE — FIRST 25


,match_id,innings,delivery_rows,minimum_sequence_number,maximum_sequence_number,recorded_non_legal_deliveries,unresolved_deliveries,no_recorded_wide_or_overstep_deliveries,sequence_matches_delivery_count,potential_legal_lower_bound,potential_legal_upper_bound
0,13511,1,121,1,121,3,0,118,True,118,118
1,13511,2,103,1,103,0,0,103,True,103,103
2,13512,1,128,1,128,8,0,120,True,120,120
3,13512,2,124,1,124,4,0,120,True,120,120
4,13513,1,125,1,125,4,1,120,True,120,121
5,13513,2,127,1,127,7,0,120,True,120,120
6,13514,1,128,1,128,8,0,120,True,120,120
7,13514,2,125,1,125,5,0,120,True,120,120
8,13515,1,127,1,127,6,1,120,True,120,121
9,13515,2,127,1,127,7,1,119,True,119,120


3. OVERALL LEGAL-BALL EVIDENCE BOUNDS


,innings_groups,groups_where_sequence_max_equals_delivery_rows,total_delivery_rows,recorded_non_legal_deliveries,unresolved_deliveries,no_recorded_wide_or_overstep_deliveries,potential_legal_lower_bound,potential_legal_upper_bound,innings_over_120_delivery_rows,maximum_delivery_rows_in_an_innings
0,1626,1626,192235,7976,257,184002,184002,184259,1122,136


4. INNINGS WITH MORE THAN 120 DELIVERY RECORDS


,match_id,innings,delivery_rows,minimum_sequence_number,maximum_sequence_number,recorded_non_legal_deliveries,unresolved_deliveries,no_recorded_wide_or_overstep_deliveries,sequence_matches_delivery_count,potential_legal_lower_bound,potential_legal_upper_bound
0,20186,2,136,1,136,16,0,120,True,120,120
1,30341,2,136,1,136,16,0,120,True,120,120
2,36199,2,136,1,136,16,0,120,True,120,120
3,40712,2,136,1,136,16,0,120,True,120,120
4,40716,1,135,1,135,15,0,120,True,120,120
5,42028,1,135,1,135,15,0,120,True,120,120
6,27062,1,134,1,134,14,0,120,True,120,120
7,30071,1,134,1,134,13,1,120,True,120,121
8,38796,1,134,1,134,13,1,120,True,120,121
9,41508,2,133,1,133,16,1,116,True,116,117


5. EVIDENCE PARTITION VALIDATION


,integrated_delivery_rows,classified_delivery_rows,partition_matches_population,missing_evidence_classifications
0,192235,192235,True,0


### Scheduled Innings Length Metadata Audit

In [54]:
# 1. Identify all match-level fields potentially related to innings length,
#    match format, interruptions, targets, and match identity.
match_terms = [
    "match", "over", "innings", "inning", "target",
    "revised", "format", "competition", "date", "result"
]

match_schema_rows = []

for position, column in enumerate(df_source_matches.columns):
    if any(term in column.lower() for term in match_terms):
        series = df_source_matches[column]

        match_schema_rows.append({
            "column_position": position,
            "column_name": column,
            "dtype": str(series.dtype),
            "non_missing_rows": int(series.notna().sum()),
            "missing_rows": int(series.isna().sum()),
            "unique_values": int(series.nunique(dropna=True)),
            "sample_values": str(
                series.dropna().drop_duplicates().head(10).tolist()
            )
        })

match_metadata_schema = pd.DataFrame(match_schema_rows)

print("1. MATCH-LEVEL METADATA FIELDS")
display(match_metadata_schema)


# 2. Inspect the revised-innings metadata at match level.
revised_fields = [
    "match_id",
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

available_revised_fields = [
    field for field in revised_fields
    if field in df_source_matches.columns
]

revised_match_metadata = (
    df_source_matches[available_revised_fields]
    .copy()
)

revised_match_metadata["any_revised_metadata"] = (
    revised_match_metadata.drop(columns=["match_id"], errors="ignore")
    .notna()
    .any(axis=1)
)

print("2. MATCHES WITH REVISED-OVERS OR REVISED-TARGET METADATA")
display(
    revised_match_metadata.loc[
        revised_match_metadata["any_revised_metadata"]
    ].sort_values("match_id").reset_index(drop=True)
)


# 3. Summarise revised metadata patterns.
revised_summary_rows = []

for field in [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]:
    if field not in df_source_matches.columns:
        continue

    values = df_source_matches[field]

    revised_summary_rows.append({
        "field": field,
        "non_missing_matches": int(values.notna().sum()),
        "distinct_non_missing_values": int(values.nunique(dropna=True)),
        "minimum": values.min(),
        "median": values.median(),
        "maximum": values.max(),
        "most_common_values": str(
            values.value_counts(dropna=True).head(15).to_dict()
        )
    })

revised_metadata_summary = pd.DataFrame(revised_summary_rows)

print("3. REVISED METADATA VALUE SUMMARY")
display(revised_metadata_summary)


# 4. Check whether match-level revised metadata is constant across
#    delivery rows within each match in the integrated dataset.
integrated_revised_fields = [
    field for field in [
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target"
    ]
    if field in df_deliveries.columns
]

consistency_rows = []

for field in integrated_revised_fields:
    grouped = (
        df_deliveries.groupby("match_id")[field]
        .agg(
            non_missing_rows="count",
            distinct_non_missing_values="nunique"
        )
        .reset_index()
    )

    consistency_rows.append({
        "field": field,
        "matches_with_any_value": int(
            (grouped["non_missing_rows"] > 0).sum()
        ),
        "matches_with_multiple_values": int(
            (grouped["distinct_non_missing_values"] > 1).sum()
        ),
        "matches_with_one_value": int(
            (grouped["distinct_non_missing_values"] == 1).sum()
        ),
        "matches_with_no_value": int(
            (grouped["non_missing_rows"] == 0).sum()
        )
    })

integrated_metadata_consistency = pd.DataFrame(consistency_rows)

print("4. INTEGRATED REVISED-METADATA CONSISTENCY")
display(integrated_metadata_consistency)

1. MATCH-LEVEL METADATA FIELDS


,column_position,column_name,dtype,non_missing_rows,missing_rows,unique_values,sample_values
0,0,match_id,int64,883,0,883,"[13511, 13512, 13513, 13514, 13515, 13522, 13523, 13524, 13612, 13613]"
1,2,match_number,int64,883,0,75,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
2,3,match_date,object,883,0,686,"['2022-01-22', '2022-01-23', '2022-01-26', '2022-01-29', '2022-01-30', '2022-02-16', '2022-02-18', '2022-02-20', '2022-02-11', '2022-02-13']"
3,15,man_of_the_match,int64,883,0,293,"[0, 4986, 1699, 830, 286, 403, 3644, 4840, 1566, 2258]"
4,16,first_inn_revised_overs,float64,25,858,13,"[114.0, 84.0, 66.0, 48.0, 72.0, 102.0, 30.0, 42.0, 78.0, 54.0]"
5,17,second_inn_revised_overs,float64,25,858,12,"[114.0, 42.0, 48.0, 72.0, 90.0, 102.0, 60.0, 78.0, 30.0, 84.0]"
6,18,revised_target,float64,23,860,23,"[143.0, 56.0, 91.0, 130.0, 171.0, 104.0, 203.0, 152.0, 126.0, 94.0]"
7,21,result,object,883,0,670,"['West Indies Won by 9 Wickets', 'England Won by 1 Run', 'West Indies Won by 20 runs', 'England Won by 34 runs', 'West Indies Won by 17 runs', 'India Won by 6 Wickets', 'India Won by 8 Runs', 'India Won by 17 runs', 'Australia Won by 20 runs (D/L method)', 'Australia Won the one over Eliminator']"


2. MATCHES WITH REVISED-OVERS OR REVISED-TARGET METADATA


,match_id,first_inn_revised_overs,second_inn_revised_overs,revised_target,any_revised_metadata
0,13612,NaN,114.0,143.0,True
1,15410,114.0,NaN,NaN,True
2,15546,84.0,NaN,NaN,True
3,16753,66.0,NaN,NaN,True
4,16754,NaN,42.0,56.0,True
5,17395,48.0,48.0,91.0,True
6,17765,72.0,72.0,130.0,True
7,19959,66.0,NaN,NaN,True
8,21070,NaN,48.0,104.0,True
9,21071,102.0,102.0,203.0,True


3. REVISED METADATA VALUE SUMMARY


,field,non_missing_matches,distinct_non_missing_values,minimum,median,maximum,most_common_values
0,first_inn_revised_overs,25,13,30.0,72.0,114.0,"{54.0: 4, 66.0: 3, 84.0: 3, 102.0: 2, 114.0: 2, 108.0: 2, 30.0: 2, 90.0: 2, 48.0: 1, 72.0: 1, 42.0: 1, 78.0: 1, 60.0: 1}"
1,second_inn_revised_overs,25,12,30.0,72.0,114.0,"{90.0: 5, 114.0: 3, 48.0: 3, 42.0: 2, 54.0: 2, 102.0: 2, 60.0: 2, 30.0: 2, 72.0: 1, 78.0: 1, 84.0: 1, 66.0: 1}"
2,revised_target,23,23,33.0,125.0,213.0,"{143.0: 1, 56.0: 1, 91.0: 1, 130.0: 1, 171.0: 1, 104.0: 1, 203.0: 1, 152.0: 1, 126.0: 1, 94.0: 1, 116.0: 1, 78.0: 1, 69.0: 1, 96.0: 1, 147.0: 1}"


4. INTEGRATED REVISED-METADATA CONSISTENCY


,field,matches_with_any_value,matches_with_multiple_values,matches_with_one_value,matches_with_no_value
0,first_inn_revised_overs,15,0,15,798
1,second_inn_revised_overs,22,0,22,791
2,revised_target,22,0,22,791


### Reconcile Revised Metadata Between Source and Integrated Data

In [55]:
revised_fields = [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

# Construct one integrated metadata record per match.
integrated_match_metadata = (
    df_deliveries[
        ["match_id"] + revised_fields
    ]
    .groupby("match_id", as_index=False)
    .agg({
        field: "first" for field in revised_fields
    })
)

# Validate that integrated metadata is constant within every match.
integrated_within_match_check = (
    df_deliveries.groupby("match_id")[revised_fields]
    .nunique(dropna=True)
    .reset_index()
)

within_match_validation = pd.DataFrame([{
    "integrated_matches": len(integrated_within_match_check),
    "matches_with_multiple_first_inn_values": int(
        (
            integrated_within_match_check[
                "first_inn_revised_overs"
            ] > 1
        ).sum()
    ),
    "matches_with_multiple_second_inn_values": int(
        (
            integrated_within_match_check[
                "second_inn_revised_overs"
            ] > 1
        ).sum()
    ),
    "matches_with_multiple_revised_target_values": int(
        (
            integrated_within_match_check[
                "revised_target"
            ] > 1
        ).sum()
    )
}])

# Match-level one-to-one reconciliation.
source_revised_metadata = df_source_matches[
    ["match_id"] + revised_fields
].copy()

metadata_reconciliation = source_revised_metadata.merge(
    integrated_match_metadata,
    on="match_id",
    how="outer",
    suffixes=("_source", "_integrated"),
    indicator=True,
    validate="one_to_one"
)

metadata_reconciliation["population_status"] = (
    metadata_reconciliation["_merge"]
    .map({
        "both": "Present in source and integrated",
        "left_only": "Source only",
        "right_only": "Integrated only"
    })
)

# Compare source and integrated values only where both are non-missing.
for field in revised_fields:
    source_field = f"{field}_source"
    integrated_field = f"{field}_integrated"

    metadata_reconciliation[f"{field}_status"] = "Both missing"

    source_present = metadata_reconciliation[source_field].notna()
    integrated_present = metadata_reconciliation[integrated_field].notna()

    metadata_reconciliation.loc[
        source_present & ~integrated_present,
        f"{field}_status"
    ] = "Source value missing in integrated"

    metadata_reconciliation.loc[
        ~source_present & integrated_present,
        f"{field}_status"
    ] = "Integrated value absent in source"

    both_present = source_present & integrated_present

    equal_values = (
        metadata_reconciliation[source_field]
        == metadata_reconciliation[integrated_field]
    )

    metadata_reconciliation.loc[
        both_present & equal_values,
        f"{field}_status"
    ] = "Values match"

    metadata_reconciliation.loc[
        both_present & ~equal_values,
        f"{field}_status"
    ] = "Values differ"


# 1. Population reconciliation.
print("1. MATCH POPULATION RECONCILIATION")
display(
    metadata_reconciliation["_merge"]
    .value_counts()
    .rename_axis("match_presence")
    .reset_index(name="match_count")
)

# 2. Field-level metadata reconciliation.
field_reconciliation_rows = []

for field in revised_fields:
    status_counts = (
        metadata_reconciliation[f"{field}_status"]
        .value_counts()
        .to_dict()
    )

    field_reconciliation_rows.append({
        "field": field,
        "source_matches_with_value": int(
            metadata_reconciliation[
                f"{field}_source"
            ].notna().sum()
        ),
        "integrated_matches_with_value": int(
            metadata_reconciliation[
                f"{field}_integrated"
            ].notna().sum()
        ),
        "values_match": status_counts.get("Values match", 0),
        "source_value_missing_in_integrated": status_counts.get(
            "Source value missing in integrated", 0
        ),
        "integrated_value_absent_in_source": status_counts.get(
            "Integrated value absent in source", 0
        ),
        "values_differ": status_counts.get("Values differ", 0),
        "both_missing": status_counts.get("Both missing", 0)
    })

field_reconciliation_summary = pd.DataFrame(
    field_reconciliation_rows
)

print("2. FIELD-LEVEL REVISED METADATA RECONCILIATION")
display(field_reconciliation_summary)

# 3. Show every match with any metadata discrepancy.
discrepancy_mask = pd.Series(
    False, index=metadata_reconciliation.index
)

for field in revised_fields:
    discrepancy_mask |= (
        metadata_reconciliation[f"{field}_status"]
        != "Values match"
    ) & (
        metadata_reconciliation[f"{field}_source"].notna()
        | metadata_reconciliation[f"{field}_integrated"].notna()
    )

metadata_discrepancy_rows = (
    metadata_reconciliation.loc[
        discrepancy_mask,
        [
            "match_id",
            "population_status",
            "first_inn_revised_overs_source",
            "first_inn_revised_overs_integrated",
            "first_inn_revised_overs_status",
            "second_inn_revised_overs_source",
            "second_inn_revised_overs_integrated",
            "second_inn_revised_overs_status",
            "revised_target_source",
            "revised_target_integrated",
            "revised_target_status"
        ]
    ]
    .sort_values("match_id")
    .reset_index(drop=True)
)

print("3. MATCHES WITH REVISED METADATA DISCREPANCIES")
display(metadata_discrepancy_rows)

# 4. Validate the reconciliation structure.
reconciliation_validation = pd.DataFrame([{
    "source_match_rows": len(df_source_matches),
    "integrated_unique_matches": len(integrated_match_metadata),
    "reconciled_match_rows": len(metadata_reconciliation),
    "expected_union_match_rows": (
        df_source_matches["match_id"].nunique()
        + integrated_match_metadata["match_id"].nunique()
        - len(
            set(df_source_matches["match_id"])
            & set(integrated_match_metadata["match_id"])
        )
    ),
    "source_match_ids_unique": (
        df_source_matches["match_id"].is_unique
    ),
    "integrated_match_ids_unique": (
        integrated_match_metadata["match_id"].is_unique
    ),
    "within_match_metadata_consistent": (
        within_match_validation.loc[
            0,
            [
                "matches_with_multiple_first_inn_values",
                "matches_with_multiple_second_inn_values",
                "matches_with_multiple_revised_target_values"
            ]
        ].sum() == 0
    )
}])

print("4. RECONCILIATION VALIDATION")
display(reconciliation_validation)

assert df_source_matches["match_id"].is_unique
assert integrated_match_metadata["match_id"].is_unique
assert len(metadata_reconciliation) == reconciliation_validation.loc[
    0, "expected_union_match_rows"
]
assert reconciliation_validation.loc[
    0, "within_match_metadata_consistent"
]

1. MATCH POPULATION RECONCILIATION


,match_presence,match_count
0,both,813
1,left_only,70
2,right_only,0


2. FIELD-LEVEL REVISED METADATA RECONCILIATION


,field,source_matches_with_value,integrated_matches_with_value,values_match,source_value_missing_in_integrated,integrated_value_absent_in_source,values_differ,both_missing
0,first_inn_revised_overs,25,15,15,10,0,0,858
1,second_inn_revised_overs,25,22,22,3,0,0,858
2,revised_target,23,22,22,1,0,0,860


3. MATCHES WITH REVISED METADATA DISCREPANCIES


,match_id,population_status,first_inn_revised_overs_source,first_inn_revised_overs_integrated,first_inn_revised_overs_status,second_inn_revised_overs_source,second_inn_revised_overs_integrated,second_inn_revised_overs_status,revised_target_source,revised_target_integrated,revised_target_status
0,15410,Source only,114.0,NaN,Source value missing in integrated,NaN,NaN,Both missing,NaN,NaN,Both missing
1,15546,Source only,84.0,NaN,Source value missing in integrated,NaN,NaN,Both missing,NaN,NaN,Both missing
2,17765,Source only,72.0,NaN,Source value missing in integrated,72.0,NaN,Source value missing in integrated,130.0,NaN,Source value missing in integrated
3,31707,Source only,30.0,NaN,Source value missing in integrated,NaN,NaN,Both missing,NaN,NaN,Both missing
4,39983,Source only,54.0,NaN,Source value missing in integrated,NaN,NaN,Both missing,NaN,NaN,Both missing
5,39985,Source only,108.0,NaN,Source value missing in integrated,NaN,NaN,Both missing,NaN,NaN,Both missing
6,41975,Source only,108.0,NaN,Source value missing in integrated,NaN,NaN,Both missing,NaN,NaN,Both missing
7,46311,Source only,54.0,NaN,Source value missing in integrated,54.0,NaN,Source value missing in integrated,NaN,NaN,Both missing
8,46749,Source only,84.0,NaN,Source value missing in integrated,NaN,NaN,Both missing,NaN,NaN,Both missing
9,50408,Source only,54.0,NaN,Source value missing in integrated,54.0,NaN,Source value missing in integrated,NaN,NaN,Both missing


4. RECONCILIATION VALIDATION


,source_match_rows,integrated_unique_matches,reconciled_match_rows,expected_union_match_rows,source_match_ids_unique,integrated_match_ids_unique,within_match_metadata_consistent
0,883,813,883,883,True,True,True


### Scoring Field Semantics and Pre-delivery Score Feasibility

In [56]:
# 1. Display the complete integrated and source column names.
schema_name_rows = []

for dataset_name, dataframe in [
    ("Integrated deliveries", df_deliveries),
    ("Original ball-by-ball source", df_source_ball)
]:
    for position, column in enumerate(dataframe.columns):
        schema_name_rows.append({
            "dataset": dataset_name,
            "position": position,
            "column_name": column,
            "dtype": str(dataframe[column].dtype)
        })

complete_column_names = pd.DataFrame(schema_name_rows)

print("1. COMPLETE DELIVERY COLUMN NAMES")
display(complete_column_names)


# 2. Profile every field with a potentially scoring-related name.
scoring_terms = [
    "run", "extra", "score", "total", "batter",
    "bat", "wide", "bye", "penalty", "overthrow"
]

scoring_schema = complete_column_names.loc[
    complete_column_names["column_name"].str.lower().apply(
        lambda name: any(term in name for term in scoring_terms)
    )
].reset_index(drop=True)

print("2. SCORING-RELATED FIELD INVENTORY")
display(scoring_schema)


# 3. Inspect combinations of the encoded scoring fields.
scoring_combinations = (
    df_deliveries.groupby(
        ["runs", "extras1_runs", "extras_2_runs"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
    .reset_index(drop=True)
)

scoring_combinations["share_pct"] = (
    100 * scoring_combinations["delivery_count"]
    / len(df_deliveries)
).round(4)

print("3. OBSERVED RUNS / EXTRAS CODE COMBINATIONS")
display(scoring_combinations.head(40))


# 4. Inspect scoring field combinations by extras category.
extras_combinations = (
    df_deliveries.groupby(
        ["extras_type", "runs", "extras_2_runs"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(
        ["extras_type", "delivery_count"],
        ascending=[True, False],
        na_position="first"
    )
    .reset_index(drop=True)
)

print("4. RUNS AND SECONDARY FIELD BY EXTRAS TYPE")
display(extras_combinations.head(60))


# 5. Check whether the raw source and integrated values agree
#    for shared matches and delivery keys.
key_columns = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num"
]

comparison_fields = [
    "runs",
    "extras1_runs",
    "extras_2_runs",
    "is_wicket"
]

source_comparison = df_source_ball[
    key_columns + comparison_fields
].copy()

integrated_comparison = df_deliveries[
    key_columns + comparison_fields
].copy()

keyed_scoring_comparison = source_comparison.merge(
    integrated_comparison,
    on=key_columns,
    how="inner",
    suffixes=("_source", "_integrated"),
    validate="one_to_one"
)

comparison_rows = []

for field in comparison_fields:
    source_values = keyed_scoring_comparison[
        f"{field}_source"
    ]
    integrated_values = keyed_scoring_comparison[
        f"{field}_integrated"
    ]

    comparison_rows.append({
        "field": field,
        "shared_delivery_rows": len(keyed_scoring_comparison),
        "matching_values": int(
            (source_values == integrated_values).sum()
        ),
        "different_values": int(
            (source_values != integrated_values).sum()
        )
    })

source_integrated_scoring_check = pd.DataFrame(comparison_rows)

print("5. SOURCE VS INTEGRATED SCORING VALUE AGREEMENT")
display(source_integrated_scoring_check)


# 6. Validate the keyed comparison.
keyed_comparison_validation = pd.DataFrame([{
    "source_delivery_rows": len(df_source_ball),
    "integrated_delivery_rows": len(df_deliveries),
    "shared_keyed_rows": len(keyed_scoring_comparison),
    "integrated_population_rows_reconciled": (
        len(keyed_scoring_comparison) == len(df_deliveries)
    ),
    "duplicate_source_keys": int(
        df_source_ball.duplicated(key_columns).sum()
    ),
    "duplicate_integrated_keys": int(
        df_deliveries.duplicated(key_columns).sum()
    )
}])

print("6. KEYED COMPARISON VALIDATION")
display(keyed_comparison_validation)

assert keyed_comparison_validation.loc[
    0, "duplicate_source_keys"
] == 0

assert keyed_comparison_validation.loc[
    0, "duplicate_integrated_keys"
] == 0

assert keyed_comparison_validation.loc[
    0, "integrated_population_rows_reconciled"
]

1. COMPLETE DELIVERY COLUMN NAMES


,dataset,position,column_name,dtype
0,Integrated deliveries,0,match_id,int64
1,Integrated deliveries,1,innings,int64
2,Integrated deliveries,2,overid,int64
3,Integrated deliveries,3,over_ball_num,int64
4,Integrated deliveries,4,innings_ball_num,int64
5,Integrated deliveries,5,batting_team_id,int64
6,Integrated deliveries,6,bowling_team_id,int64
7,Integrated deliveries,7,striker_id,int64
8,Integrated deliveries,8,striker_name,object
9,Integrated deliveries,9,non_striker_id,int64


2. SCORING-RELATED FIELD INVENTORY


,dataset,position,column_name,dtype
0,Integrated deliveries,5,batting_team_id,int64
1,Integrated deliveries,13,batting_type,object
2,Integrated deliveries,15,batting_order,int64
3,Integrated deliveries,16,runs,int64
4,Integrated deliveries,23,extras_type,object
5,Integrated deliveries,24,extras1_runs,int64
6,Integrated deliveries,25,extras_2_runs,int64
7,Integrated deliveries,59,batting_team_name,object
8,Original ball-by-ball source,5,batting_team_id,int64
9,Original ball-by-ball source,13,batting_type,object


3. OBSERVED RUNS / EXTRAS CODE COMBINATIONS


,runs,extras1_runs,extras_2_runs,delivery_count,share_pct
0,1,6,0,68772,35.7750
1,0,6,0,64863,33.7415
2,4,6,0,22371,11.6373
3,6,6,0,12225,6.3594
4,2,6,0,11985,6.2346
5,0,2,0,7419,3.8593
6,0,3,0,2635,1.3707
7,3,6,0,623,0.3241
8,0,4,0,505,0.2627
9,0,1,0,211,0.1098


... (2 outputs left out)

5. SOURCE VS INTEGRATED SCORING VALUE AGREEMENT


,field,shared_delivery_rows,matching_values,different_values
0,runs,192235,192235,0
1,extras1_runs,192235,192235,0
2,extras_2_runs,192235,192235,0
3,is_wicket,192235,192235,0


6. KEYED COMPARISON VALIDATION


,source_delivery_rows,integrated_delivery_rows,shared_keyed_rows,integrated_population_rows_reconciled,duplicate_source_keys,duplicate_integrated_keys
0,200999,192235,192235,True,0,0


### Match-Level Score Evidence Inventory

In [57]:
# 1. Display every match-source column, not only previously
#    identified metadata fields.
match_source_schema_full = pd.DataFrame([
    {
        "column_position": position,
        "column_name": column,
        "dtype": str(df_source_matches[column].dtype),
        "non_missing_rows": int(df_source_matches[column].notna().sum()),
        "missing_rows": int(df_source_matches[column].isna().sum()),
        "unique_values": int(df_source_matches[column].nunique(dropna=True)),
        "sample_values": str(
            df_source_matches[column]
            .dropna()
            .drop_duplicates()
            .head(8)
            .tolist()
        )
    }
    for position, column in enumerate(df_source_matches.columns)
])

print("1. COMPLETE MATCH-SOURCE SCHEMA")
display(match_source_schema_full)


# 2. Identify potential score, innings, target, and result fields.
score_terms = [
    "score", "run", "total", "innings", "inning",
    "target", "result", "won", "margin", "overs",
    "wicket", "team"
]

match_score_schema = match_source_schema_full.loc[
    match_source_schema_full["column_name"].str.lower().apply(
        lambda name: any(term in name for term in score_terms)
    )
].reset_index(drop=True)

print("2. POTENTIALLY RELEVANT MATCH-LEVEL FIELDS")
display(match_score_schema)


# 3. Produce innings-level sums of the raw runs field as a
#    diagnostic only. These are not yet accepted as official scores.
raw_innings_run_diagnostic = (
    df_deliveries.groupby(
        ["match_id", "innings", "batting_team_id"],
        as_index=False
    )
    .agg(
        delivery_rows=("runs", "size"),
        raw_runs_sum=("runs", "sum"),
        wicket_events=("is_wicket", "sum"),
        extras_type_rows=("extras_type", "count"),
        nonzero_extras_2_rows=(
            "extras_2_runs",
            lambda values: int((values != 0).sum())
        )
    )
)

print("3. RAW RUNS SUM BY MATCH AND INNINGS — DIAGNOSTIC ONLY")
display(raw_innings_run_diagnostic.head(30))


# 4. Inspect result text patterns to identify whether results
#    contain score totals or only winning margins.
result_profile = (
    df_source_matches["result"]
    .value_counts(dropna=False)
    .head(40)
    .rename_axis("result_text")
    .reset_index(name="match_count")
)

print("4. MATCH RESULT TEXT DISTRIBUTION — FIRST 40")
display(result_profile)

1. COMPLETE MATCH-SOURCE SCHEMA


,column_position,column_name,dtype,non_missing_rows,missing_rows,unique_values,sample_values
0,0,match_id,int64,883,0,883,"[13511, 13512, 13513, 13514, 13515, 13522, 13523, 13524]"
1,1,comp_id,int64,883,0,138,"[1331, 1335, 1345, 1358, 1435, 1439, 1452, 1458]"
2,2,match_number,int64,883,0,75,"[1, 2, 3, 4, 5, 6, 7, 8]"
3,3,match_date,object,883,0,686,"['2022-01-22', '2022-01-23', '2022-01-26', '2022-01-29', '2022-01-30', '2022-02-16', '2022-02-18', '2022-02-20']"
4,4,stage,object,883,0,7,"['League', 'Qualifier', 'Eliminator', 'Final', 'Others', 'Semi Final', 'Super 4']"
5,5,venue_id,int64,883,0,98,"[54, 8, 44, 129, 17, 3, 46, 12]"
6,6,venue_country,object,883,0,17,"['West Indies', 'India', 'Australia', 'Sri Lanka', 'Zimbabwe', 'Ireland', 'United Kingdom', 'United States']"
7,7,home_team_id,int64,883,0,36,"[28, 18, 20, 1, 7, 4, 3508, 242]"
8,8,home_team,object,883,0,36,"['West Indies', 'India', 'Australia', 'Chennai Super Kings', 'Delhi Capitals', 'Punjab Kings', 'Gujarat Titans', 'Sunrisers Hyderabad']"
9,9,away_team_id,int64,883,0,36,"[22, 28, 27, 8, 2, 5, 3509, 3]"


2. POTENTIALLY RELEVANT MATCH-LEVEL FIELDS


,column_position,column_name,dtype,non_missing_rows,missing_rows,unique_values,sample_values
0,7,home_team_id,int64,883,0,36,"[28, 18, 20, 1, 7, 4, 3508, 242]"
1,8,home_team,object,883,0,36,"['West Indies', 'India', 'Australia', 'Chennai Super Kings', 'Delhi Capitals', 'Punjab Kings', 'Gujarat Titans', 'Sunrisers Hyderabad']"
2,9,away_team_id,int64,883,0,36,"[22, 28, 27, 8, 2, 5, 3509, 3]"
3,10,away_team,object,883,0,36,"['England', 'West Indies', 'Sri Lanka', 'Kolkata Knight Riders', 'Mumbai Indians', 'Royal Challengers Bengaluru', 'Lucknow Super Giants', 'Rajasthan Royals']"
4,11,home_team_captain_id,float64,839,44,89,"[494.0, 126.0, 250.0, 152.0, 4703.0, 992.0, 3330.0, 1568.0]"
5,12,away_team_captain_id,float64,838,45,98,"[478.0, 1596.0, 494.0, 2436.0, 3647.0, 126.0, 88.0, 953.0]"
6,13,toss_win_team_id,float64,868,15,37,"[28.0, 22.0, 18.0, 27.0, 20.0, 8.0, 7.0, 4.0]"
7,16,first_inn_revised_overs,float64,25,858,13,"[114.0, 84.0, 66.0, 48.0, 72.0, 102.0, 30.0, 42.0]"
8,17,second_inn_revised_overs,float64,25,858,12,"[114.0, 42.0, 48.0, 72.0, 90.0, 102.0, 60.0, 78.0]"
9,18,revised_target,float64,23,860,23,"[143.0, 56.0, 91.0, 130.0, 171.0, 104.0, 203.0, 152.0]"


3. RAW RUNS SUM BY MATCH AND INNINGS — DIAGNOSTIC ONLY


,match_id,innings,batting_team_id,delivery_rows,raw_runs_sum,wicket_events,extras_type_rows,nonzero_extras_2_rows
0,13511,1,22,121,100,10,3,0
1,13511,2,28,103,99,1,2,0
2,13512,1,22,128,163,8,8,0
3,13512,2,28,124,158,8,10,0
4,13513,1,28,125,211,5,6,0
5,13513,2,22,127,192,9,7,0
6,13514,1,22,128,182,6,11,0
7,13514,2,28,125,150,5,6,0
8,13515,1,28,127,168,4,10,0
9,13515,2,22,127,150,10,11,0


4. MATCH RESULT TEXT DISTRIBUTION — FIRST 40


,result_text,match_count
0,No Result,21
1,Match Abandoned,19
2,India Won by 7 wickets,8
3,Delhi Capitals Won by 6 wickets,7
4,Kolkata Knight Riders Won by 8 wickets,6
5,Punjab Kings Won by 5 wickets,5
6,India Won by 6 Wickets,5
7,Gujarat Titans Won by 7 wickets,5
8,West Indies Won by 5 wickets,4
9,Australia Won by 3 wickets,4


### Verify Innings Aggregations and Scoring Diagnostics

In [58]:
# 1. Recalculate innings-level summaries with explicit boolean counts.
innings_diagnostic = (
    df_deliveries.groupby(
        ["match_id", "innings", "batting_team_id"],
        as_index=False
    )
    .agg(
        delivery_rows=("match_id", "size"),
        recorded_runs_sum=("runs", "sum"),
        wicket_events=("is_wicket", "sum"),
        boundary_four_flags=("is_four", "sum"),
        boundary_six_flags=("is_six", "sum"),
        extras_type_rows=("extras_type", "count"),
        nonzero_extras_2_rows=(
            "extras_2_runs",
            lambda values: int((values != 0).sum())
        )
    )
)

print("1. VERIFIED INNINGS-LEVEL AGGREGATION — FIRST 30")
display(innings_diagnostic.head(30))


# 2. Validate wicket aggregation bounds and overall totals.
innings_aggregation_validation = pd.DataFrame([{
    "innings_groups": len(innings_diagnostic),
    "total_delivery_rows": int(
        innings_diagnostic["delivery_rows"].sum()
    ),
    "total_wicket_events": int(
        innings_diagnostic["wicket_events"].sum()
    ),
    "source_is_wicket_true_rows": int(
        df_deliveries["is_wicket"].sum()
    ),
    "innings_with_wickets_exceeding_delivery_rows": int(
        (
            innings_diagnostic["wicket_events"]
            > innings_diagnostic["delivery_rows"]
        ).sum()
    ),
    "innings_delivery_rows_reconcile": (
        int(innings_diagnostic["delivery_rows"].sum())
        == len(df_deliveries)
    ),
    "wicket_events_reconcile": (
        int(innings_diagnostic["wicket_events"].sum())
        == int(df_deliveries["is_wicket"].sum())
    )
}])

print("2. INNINGS AGGREGATION VALIDATION")
display(innings_aggregation_validation)


# 3. Test boundary flags against recorded runs on rows without
#    a recorded extras category. This is a consistency check only.
boundary_audit_data = df_deliveries.copy()

boundary_audit_data["no_recorded_extras_type"] = (
    boundary_audit_data["extras_type"].isna()
)

boundary_audit_data["runs_four"] = (
    boundary_audit_data["runs"] == 4
)

boundary_audit_data["runs_six"] = (
    boundary_audit_data["runs"] == 6
)

boundary_audit_rows = []

for subset_name, subset in [
    ("All deliveries", boundary_audit_data),
    (
        "No recorded extras type",
        boundary_audit_data.loc[
            boundary_audit_data["no_recorded_extras_type"]
        ]
    )
]:
    boundary_audit_rows.append({
        "subset": subset_name,
        "delivery_rows": len(subset),
        "is_four_true": int(subset["is_four"].sum()),
        "runs_equal_four": int(subset["runs_four"].sum()),
        "is_four_disagreement": int(
            (subset["is_four"] != subset["runs_four"]).sum()
        ),
        "is_six_true": int(subset["is_six"].sum()),
        "runs_equal_six": int(subset["runs_six"].sum()),
        "is_six_disagreement": int(
            (subset["is_six"] != subset["runs_six"]).sum()
        )
    })

boundary_consistency = pd.DataFrame(boundary_audit_rows)

print("3. BOUNDARY FLAG CONSISTENCY")
display(boundary_consistency)


# 4. Summarise recorded run values by extras category.
runs_by_extras = (
    boundary_audit_data.groupby(
        "extras_type",
        dropna=False
    )
    .agg(
        delivery_rows=("runs", "size"),
        run_sum=("runs", "sum"),
        minimum_runs=("runs", "min"),
        maximum_runs=("runs", "max"),
        distinct_run_values=("runs", "nunique")
    )
    .reset_index()
    .sort_values("delivery_rows", ascending=False)
)

print("4. RECORDED RUN VALUES BY EXTRAS CATEGORY")
display(runs_by_extras)


# 5. Final explicit checks.
assert innings_aggregation_validation.loc[
    0, "innings_delivery_rows_reconcile"
]

assert innings_aggregation_validation.loc[
    0, "wicket_events_reconcile"
]

assert innings_aggregation_validation.loc[
    0, "innings_with_wickets_exceeding_delivery_rows"
] == 0

1. VERIFIED INNINGS-LEVEL AGGREGATION — FIRST 30


,match_id,innings,batting_team_id,delivery_rows,recorded_runs_sum,wicket_events,boundary_four_flags,boundary_six_flags,extras_type_rows,nonzero_extras_2_rows
0,13511,1,22,121,100,10,6,6,3,0
1,13511,2,28,103,99,1,8,1,2,0
2,13512,1,22,128,163,8,20,5,8,0
3,13512,2,28,124,158,8,8,13,10,0
4,13513,1,28,125,211,5,12,16,6,0
5,13513,2,22,127,192,9,7,15,7,0
6,13514,1,22,128,182,6,9,15,11,0
7,13514,2,28,125,150,5,10,9,6,0
8,13515,1,28,127,168,4,7,12,10,0
9,13515,2,22,127,150,10,14,4,11,0


2. INNINGS AGGREGATION VALIDATION


,innings_groups,total_delivery_rows,total_wicket_events,source_is_wicket_true_rows,innings_with_wickets_exceeding_delivery_rows,innings_delivery_rows_reconcile,wicket_events_reconcile
0,1626,192235,10392,10392,0,True,True


3. BOUNDARY FLAG CONSISTENCY


,subset,delivery_rows,is_four_true,runs_equal_four,is_four_disagreement,is_six_true,runs_equal_six,is_six_disagreement
0,All deliveries,192235,22470,22490,20,12293,12296,3
1,No recorded extras type,180862,22351,22371,20,12222,12225,3


4. RECORDED RUN VALUES BY EXTRAS CATEGORY


,extras_type,delivery_rows,run_sum,minimum_runs,maximum_runs,distinct_run_values
6,NaN,180862,257555,0,6,7
5,wide,7419,0,0,0,1
3,legbyes,2635,0,0,0,1
0,Overstep,557,824,0,6,6
2,byes,505,0,0,0,1
4,other,256,403,0,6,6
1,Penalty,1,0,0,0,1


### Reconstruct Pre-Delivery Wickets Fallen

In [59]:
# Create a separate derived table; do not modify df_deliveries.
phase3_wicket_state = df_deliveries.copy()

# Establish deterministic within-innings delivery order.
phase3_wicket_state = phase3_wicket_state.sort_values(
    ["match_id", "innings", "innings_ball_num"],
    kind="stable"
).reset_index(drop=True)

# Count only wicket events from earlier deliveries.
phase3_wicket_state["pre_delivery_wickets_fallen"] = (
    phase3_wicket_state
    .groupby(["match_id", "innings"])["is_wicket"]
    .transform(
        lambda values: values.astype("int64").cumsum()
        - values.astype("int64")
    )
)

# Count wickets after the current delivery for validation only.
phase3_wicket_state["post_delivery_wickets_fallen"] = (
    phase3_wicket_state
    .groupby(["match_id", "innings"])["is_wicket"]
    .transform(
        lambda values: values.astype("int64").cumsum()
    )
)

# 1. Show a sample of reconstructed pre-delivery states.
print("1. PRE-DELIVERY WICKET STATE — FIRST 25 RECORDS")

display(
    phase3_wicket_state[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "is_wicket",
            "pre_delivery_wickets_fallen",
            "post_delivery_wickets_fallen"
        ]
    ].head(25)
)


# 2. Summarise reconstructed state by innings.
wicket_state_innings_summary = (
    phase3_wicket_state
    .groupby(["match_id", "innings"], as_index=False)
    .agg(
        delivery_rows=("match_id", "size"),
        first_pre_delivery_wickets=(
            "pre_delivery_wickets_fallen", "first"
        ),
        maximum_pre_delivery_wickets=(
            "pre_delivery_wickets_fallen", "max"
        ),
        final_post_delivery_wickets=(
            "post_delivery_wickets_fallen", "last"
        )
    )
)

print("2. INNINGS-LEVEL WICKET STATE SUMMARY — FIRST 25")

display(wicket_state_innings_summary.head(25))


# 3. Validate state reconstruction and current-delivery exclusion.
wicket_state_validation = pd.DataFrame([{
    "delivery_rows": len(phase3_wicket_state),
    "innings_groups": len(wicket_state_innings_summary),
    "innings_starting_at_zero_wickets": int(
        (
            wicket_state_innings_summary[
                "first_pre_delivery_wickets"
            ] == 0
        ).sum()
    ),
    "innings_not_starting_at_zero": int(
        (
            wicket_state_innings_summary[
                "first_pre_delivery_wickets"
            ] != 0
        ).sum()
    ),
    "negative_pre_delivery_states": int(
        (
            phase3_wicket_state[
                "pre_delivery_wickets_fallen"
            ] < 0
        ).sum()
    ),
    "pre_delivery_states_above_ten": int(
        (
            phase3_wicket_state[
                "pre_delivery_wickets_fallen"
            ] > 10
        ).sum()
    ),
    "current_wicket_leakage_rows": int(
        (
            phase3_wicket_state[
                "pre_delivery_wickets_fallen"
            ]
            !=
            phase3_wicket_state[
                "post_delivery_wickets_fallen"
            ]
            - phase3_wicket_state["is_wicket"].astype("int64")
        ).sum()
    ),
    "innings_wicket_totals_reconcile": (
        int(
            wicket_state_innings_summary[
                "final_post_delivery_wickets"
            ].sum()
        )
        == int(df_deliveries["is_wicket"].sum())
    )
}])

print("3. PRE-DELIVERY WICKET STATE VALIDATION")
display(wicket_state_validation)


# 4. Enforce the required checks.
assert len(phase3_wicket_state) == len(df_deliveries)

assert (
    wicket_state_validation.loc[
        0, "innings_not_starting_at_zero"
    ] == 0
)

assert (
    wicket_state_validation.loc[
        0, "negative_pre_delivery_states"
    ] == 0
)

assert (
    wicket_state_validation.loc[
        0, "pre_delivery_states_above_ten"
    ] == 0
)

assert (
    wicket_state_validation.loc[
        0, "current_wicket_leakage_rows"
    ] == 0
)

assert wicket_state_validation.loc[
    0, "innings_wicket_totals_reconcile"
]

print("All pre-delivery wicket-state validations passed.")

1. PRE-DELIVERY WICKET STATE — FIRST 25 RECORDS


,match_id,innings,innings_ball_num,is_wicket,pre_delivery_wickets_fallen,post_delivery_wickets_fallen
0,13511,1,1,False,0,0
1,13511,1,2,False,0,0
2,13511,1,3,False,0,0
3,13511,1,4,False,0,0
4,13511,1,5,False,0,0
5,13511,1,6,True,0,1
6,13511,1,7,False,1,1
7,13511,1,8,False,1,1
8,13511,1,9,False,1,1
9,13511,1,10,False,1,1


2. INNINGS-LEVEL WICKET STATE SUMMARY — FIRST 25


,match_id,innings,delivery_rows,first_pre_delivery_wickets,maximum_pre_delivery_wickets,final_post_delivery_wickets
0,13511,1,121,0,9,10
1,13511,2,103,0,1,1
2,13512,1,128,0,8,8
3,13512,2,124,0,8,8
4,13513,1,125,0,5,5
5,13513,2,127,0,9,9
6,13514,1,128,0,6,6
7,13514,2,125,0,5,5
8,13515,1,127,0,4,4
9,13515,2,127,0,9,10


3. PRE-DELIVERY WICKET STATE VALIDATION


,delivery_rows,innings_groups,innings_starting_at_zero_wickets,innings_not_starting_at_zero,negative_pre_delivery_states,pre_delivery_states_above_ten,current_wicket_leakage_rows,innings_wicket_totals_reconcile
0,192235,1626,1626,0,0,0,0,True


All pre-delivery wicket-state validations passed.


### Innings Structure and Team Identity Audit

In [60]:
required_columns = [
    "match_id",
    "innings",
    "batting_team_id"
]

missing_required = [
    col for col in required_columns
    if col not in df_deliveries.columns
]

assert not missing_required, (
    f"Required columns missing: {missing_required}"
)

optional_team_columns = [
    col for col in ["bowling_team_id"]
    if col in df_deliveries.columns
]

team_columns = ["batting_team_id"] + optional_team_columns


# 1. Summarise team identity within each innings.
aggregation_spec = {
    "delivery_rows": ("match_id", "size"),
    "batting_team_count": ("batting_team_id", "nunique"),
    "batting_team_missing_rows": (
        "batting_team_id",
        lambda values: int(values.isna().sum())
    )
}

if "bowling_team_id" in optional_team_columns:
    aggregation_spec.update({
        "bowling_team_count": ("bowling_team_id", "nunique"),
        "bowling_team_missing_rows": (
            "bowling_team_id",
            lambda values: int(values.isna().sum())
        )
    })

innings_team_audit = (
    df_deliveries
    .groupby(["match_id", "innings"], as_index=False)
    .agg(**aggregation_spec)
)

print("1. INNINGS-LEVEL TEAM IDENTITY AUDIT — FIRST 30")
display(innings_team_audit.head(30))


# 2. Summarise innings structure per match.
match_innings_audit = (
    df_deliveries
    .groupby("match_id", as_index=False)
    .agg(
        innings_count=("innings", "nunique"),
        innings_numbers=("innings", lambda x: tuple(sorted(x.dropna().unique()))),
        delivery_rows=("match_id", "size"),
        batting_teams_in_match=("batting_team_id", "nunique")
    )
)

print("2. MATCH-LEVEL INNINGS STRUCTURE — FIRST 30")
display(match_innings_audit.head(30))


# 3. Summarise observed innings numbers and structural exceptions.
innings_number_distribution = (
    df_deliveries
    .groupby("innings", dropna=False)
    .agg(
        delivery_rows=("match_id", "size"),
        innings_groups=("match_id", lambda x: int(
            df_deliveries.loc[x.index, ["match_id", "innings"]]
            .drop_duplicates()
            .shape[0]
        )),
        matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values("innings")
)

print("3. INNINGS NUMBER DISTRIBUTION")
display(innings_number_distribution)


# 4. Build a compact validation summary.
team_identity_validation = {
    "delivery_rows": len(df_deliveries),
    "innings_groups": len(innings_team_audit),
    "matches": int(df_deliveries["match_id"].nunique()),
    "innings_with_multiple_batting_teams": int(
        (innings_team_audit["batting_team_count"] > 1).sum()
    ),
    "innings_with_missing_batting_team": int(
        (innings_team_audit["batting_team_missing_rows"] > 0).sum()
    ),
    "matches_with_one_innings": int(
        (match_innings_audit["innings_count"] == 1).sum()
    ),
    "matches_with_two_innings": int(
        (match_innings_audit["innings_count"] == 2).sum()
    ),
    "matches_with_more_than_two_innings": int(
        (match_innings_audit["innings_count"] > 2).sum()
    ),
    "matches_with_duplicate_batting_team_across_innings": int(
        (
            match_innings_audit["batting_teams_in_match"]
            < match_innings_audit["innings_count"]
        ).sum()
    )
}

if "bowling_team_id" in optional_team_columns:
    team_identity_validation.update({
        "innings_with_multiple_bowling_teams": int(
            (innings_team_audit["bowling_team_count"] > 1).sum()
        ),
        "innings_with_missing_bowling_team": int(
            (innings_team_audit["bowling_team_missing_rows"] > 0).sum()
        )
    })

team_identity_validation = pd.DataFrame(
    [team_identity_validation]
)

print("4. TEAM AND INNINGS STRUCTURE VALIDATION")
display(team_identity_validation)


# 5. Display any innings with inconsistent or missing team identity.
innings_team_exceptions = innings_team_audit.loc[
    (innings_team_audit["batting_team_count"] != 1)
    | (innings_team_audit["batting_team_missing_rows"] > 0)
].copy()

if "bowling_team_id" in optional_team_columns:
    innings_team_exceptions = innings_team_audit.loc[
        (innings_team_audit["batting_team_count"] != 1)
        | (innings_team_audit["batting_team_missing_rows"] > 0)
        | (innings_team_audit["bowling_team_count"] != 1)
        | (innings_team_audit["bowling_team_missing_rows"] > 0)
    ].copy()

print("5. INNINGS WITH TEAM IDENTITY EXCEPTIONS")
display(innings_team_exceptions)


# 6. Confirm the audit has not changed the source delivery population.
assert len(df_deliveries) == 192235
assert innings_team_audit["delivery_rows"].sum() == len(df_deliveries)
assert match_innings_audit["delivery_rows"].sum() == len(df_deliveries)
assert innings_team_audit["match_id"].nunique() == df_deliveries["match_id"].nunique()

print("All population-preservation checks passed.")

1. INNINGS-LEVEL TEAM IDENTITY AUDIT — FIRST 30


,match_id,innings,delivery_rows,batting_team_count,batting_team_missing_rows,bowling_team_count,bowling_team_missing_rows
0,13511,1,121,1,0,1,0
1,13511,2,103,1,0,1,0
2,13512,1,128,1,0,1,0
3,13512,2,124,1,0,1,0
4,13513,1,125,1,0,1,0
5,13513,2,127,1,0,1,0
6,13514,1,128,1,0,1,0
7,13514,2,125,1,0,1,0
8,13515,1,127,1,0,1,0
9,13515,2,127,1,0,1,0


2. MATCH-LEVEL INNINGS STRUCTURE — FIRST 30


,match_id,innings_count,innings_numbers,delivery_rows,batting_teams_in_match
0,13511,2,"(1, 2)",224,2
1,13512,2,"(1, 2)",252,2
2,13513,2,"(1, 2)",252,2
3,13514,2,"(1, 2)",253,2
4,13515,2,"(1, 2)",254,2
5,13522,2,"(1, 2)",247,2
6,13523,2,"(1, 2)",257,2
7,13524,2,"(1, 2)",251,2
8,13612,2,"(1, 2)",242,2
9,13613,2,"(1, 2)",246,2


3. INNINGS NUMBER DISTRIBUTION


,innings,delivery_rows,innings_groups,matches
0,1,100333,813,813
1,2,91902,813,813


4. TEAM AND INNINGS STRUCTURE VALIDATION


,delivery_rows,innings_groups,matches,innings_with_multiple_batting_teams,innings_with_missing_batting_team,matches_with_one_innings,matches_with_two_innings,matches_with_more_than_two_innings,matches_with_duplicate_batting_team_across_innings,innings_with_multiple_bowling_teams,innings_with_missing_bowling_team
0,192235,1626,813,0,0,0,813,0,0,0,0


5. INNINGS WITH TEAM IDENTITY EXCEPTIONS


,match_id,innings,delivery_rows,batting_team_count,batting_team_missing_rows,bowling_team_count,bowling_team_missing_rows


All population-preservation checks passed.


## Cross-Innings Team Reversal Audit

In [61]:
# Build one row per match and innings with its unique team identities.
cross_innings_source = (
    df_deliveries[
        [
            "match_id",
            "innings",
            "batting_team_id",
            "bowling_team_id"
        ]
    ]
    .drop_duplicates()
)

# Verify that each match/innings has exactly one team pairing.
pairing_counts = (
    cross_innings_source
    .groupby(["match_id", "innings"], as_index=False)
    .size()
    .rename(columns={"size": "team_pairing_rows"})
)

assert pairing_counts["team_pairing_rows"].eq(1).all(), (
    "At least one match/innings has multiple distinct batting/bowling "
    "team pairings."
)

# Separate innings 1 and innings 2.
innings_1_teams = (
    cross_innings_source.loc[
        cross_innings_source["innings"] == 1
    ]
    .drop(columns="innings")
    .rename(columns={
        "batting_team_id": "innings_1_batting_team_id",
        "bowling_team_id": "innings_1_bowling_team_id"
    })
)

innings_2_teams = (
    cross_innings_source.loc[
        cross_innings_source["innings"] == 2
    ]
    .drop(columns="innings")
    .rename(columns={
        "batting_team_id": "innings_2_batting_team_id",
        "bowling_team_id": "innings_2_bowling_team_id"
    })
)

# Explicit one-to-one match-level reconciliation.
cross_innings_team_audit = innings_1_teams.merge(
    innings_2_teams,
    on="match_id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Check whether teams reverse their batting/bowling roles.
cross_innings_team_audit["innings_1_batting_equals_innings_2_bowling"] = (
    cross_innings_team_audit["innings_1_batting_team_id"]
    == cross_innings_team_audit["innings_2_bowling_team_id"]
)

cross_innings_team_audit["innings_1_bowling_equals_innings_2_batting"] = (
    cross_innings_team_audit["innings_1_bowling_team_id"]
    == cross_innings_team_audit["innings_2_batting_team_id"]
)

cross_innings_team_audit["team_roles_reverse_correctly"] = (
    cross_innings_team_audit[
        "innings_1_batting_equals_innings_2_bowling"
    ]
    &
    cross_innings_team_audit[
        "innings_1_bowling_equals_innings_2_batting"
    ]
)

# 1. Display sample.
print("1. CROSS-INNINGS TEAM REVERSAL — FIRST 30 MATCHES")

display(cross_innings_team_audit.head(30))


# 2. Summarise match-level reconciliation.
cross_innings_summary = (
    cross_innings_team_audit
    .groupby(
        ["_merge", "team_roles_reverse_correctly"],
        dropna=False
    )
    .size()
    .reset_index(name="match_count")
)

print("2. TEAM REVERSAL SUMMARY")
display(cross_innings_summary)


# 3. Extract exceptions for inspection.
cross_innings_exceptions = cross_innings_team_audit.loc[
    (cross_innings_team_audit["_merge"] != "both")
    | (~cross_innings_team_audit["team_roles_reverse_correctly"])
].copy()

print("3. CROSS-INNINGS TEAM REVERSAL EXCEPTIONS")
display(cross_innings_exceptions)


# 4. Validation table.
cross_innings_validation = pd.DataFrame([{
    "integrated_matches": int(df_deliveries["match_id"].nunique()),
    "matches_in_cross_innings_audit": len(cross_innings_team_audit),
    "matches_present_in_both_innings": int(
        (cross_innings_team_audit["_merge"] == "both").sum()
    ),
    "matches_missing_an_innings": int(
        (cross_innings_team_audit["_merge"] != "both").sum()
    ),
    "matches_with_correct_team_reversal": int(
        cross_innings_team_audit[
            "team_roles_reverse_correctly"
        ].sum()
    ),
    "matches_with_incorrect_team_reversal": int(
        (~cross_innings_team_audit[
            "team_roles_reverse_correctly"
        ]).sum()
    ),
    "exception_rows": len(cross_innings_exceptions)
}])

print("4. CROSS-INNINGS VALIDATION")
display(cross_innings_validation)


# 5. Enforce population and structural checks.
assert len(cross_innings_team_audit) == 813

assert (
    cross_innings_validation.loc[
        0, "matches_missing_an_innings"
    ] == 0
)

assert (
    cross_innings_validation.loc[
        0, "matches_with_incorrect_team_reversal"
    ] == 0
)

assert len(cross_innings_exceptions) == 0

print("All cross-innings team reversal validations passed.")

1. CROSS-INNINGS TEAM REVERSAL — FIRST 30 MATCHES


,match_id,innings_1_batting_team_id,innings_1_bowling_team_id,innings_2_batting_team_id,innings_2_bowling_team_id,_merge,innings_1_batting_equals_innings_2_bowling,innings_1_bowling_equals_innings_2_batting,team_roles_reverse_correctly
0,13511,22,28,28,22,both,True,True,True
1,13512,22,28,28,22,both,True,True,True
2,13513,28,22,22,28,both,True,True,True
3,13514,22,28,28,22,both,True,True,True
4,13515,28,22,22,28,both,True,True,True
5,13522,28,18,18,28,both,True,True,True
6,13523,18,28,28,18,both,True,True,True
7,13524,18,28,28,18,both,True,True,True
8,13612,20,27,27,20,both,True,True,True
9,13613,20,27,27,20,both,True,True,True


2. TEAM REVERSAL SUMMARY


/tmp/ipykernel_411/1409440548.py:89: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(


,_merge,team_roles_reverse_correctly,match_count
0,left_only,True,0
1,right_only,True,0
2,both,True,813


3. CROSS-INNINGS TEAM REVERSAL EXCEPTIONS


,match_id,innings_1_batting_team_id,innings_1_bowling_team_id,innings_2_batting_team_id,innings_2_bowling_team_id,_merge,innings_1_batting_equals_innings_2_bowling,innings_1_bowling_equals_innings_2_batting,team_roles_reverse_correctly


4. CROSS-INNINGS VALIDATION


,integrated_matches,matches_in_cross_innings_audit,matches_present_in_both_innings,matches_missing_an_innings,matches_with_correct_team_reversal,matches_with_incorrect_team_reversal,exception_rows
0,813,813,813,0,813,0,0


All cross-innings team reversal validations passed.


### Chase Target and Match-Level Context Audit

In [62]:
# 1. Identify target and innings-length related source fields.
target_context_candidates = [
    col for col in df_source_matches.columns
    if any(
        term in col.lower()
        for term in [
            "target", "revised", "over", "innings",
            "result", "win", "status"
        ]
    )
]

target_context_schema = pd.DataFrame({
    "field": target_context_candidates,
    "non_null_source_matches": [
        int(df_source_matches[col].notna().sum())
        for col in target_context_candidates
    ],
    "distinct_non_null_values": [
        int(df_source_matches[col].nunique(dropna=True))
        for col in target_context_candidates
    ],
    "data_type": [
        str(df_source_matches[col].dtype)
        for col in target_context_candidates
    ]
})

print("1. MATCH-LEVEL TARGET AND CONTEXT FIELD INVENTORY")
display(target_context_schema)


# 2. Build a unique source match metadata table.
assert df_source_matches["match_id"].is_unique, (
    "Source match metadata contains duplicate match_id values."
)

context_fields = [
    col for col in [
        "match_id",
        "comp_id",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target",
        "result",
        "win_team_id"
    ]
    if col in df_source_matches.columns
]

source_match_context = df_source_matches[context_fields].copy()

# Match-level integrated population.
integrated_match_ids = (
    df_deliveries[["match_id"]]
    .drop_duplicates()
)

integrated_match_context = integrated_match_ids.merge(
    source_match_context,
    on="match_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("2. INTEGRATED MATCH CONTEXT — FIRST 30")
display(integrated_match_context.head(30))


# 3. Summarise target and revised-innings metadata coverage.
coverage_rows = []

for field in [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]:
    if field in integrated_match_context.columns:
        coverage_rows.append({
            "field": field,
            "integrated_matches": len(integrated_match_context),
            "non_null_matches": int(
                integrated_match_context[field].notna().sum()
            ),
            "missing_matches": int(
                integrated_match_context[field].isna().sum()
            ),
            "distinct_non_null_values": int(
                integrated_match_context[field].nunique(dropna=True)
            ),
            "minimum_value": integrated_match_context[field].min(),
            "maximum_value": integrated_match_context[field].max()
        })

target_context_coverage = pd.DataFrame(coverage_rows)

print("3. INTEGRATED MATCH CONTEXT COVERAGE")
display(target_context_coverage)


# 4. Inspect all matches with a recorded revised target.
if "revised_target" in integrated_match_context.columns:
    revised_target_matches = integrated_match_context.loc[
        integrated_match_context["revised_target"].notna()
    ].copy()

    print("4. MATCHES WITH RECORDED REVISED TARGET")
    display(revised_target_matches.head(40))
else:
    revised_target_matches = pd.DataFrame()
    print("4. No revised_target field exists in source match metadata.")


# 5. Inspect result field patterns without interpreting them as scores.
if "result" in integrated_match_context.columns:
    result_distribution = (
        integrated_match_context["result"]
        .fillna("<MISSING>")
        .astype(str)
        .value_counts(dropna=False)
        .head(30)
        .rename_axis("result_value")
        .reset_index(name="match_count")
    )

    print("5. MOST COMMON MATCH RESULT VALUES")
    display(result_distribution)


# 6. Validation summary.
target_context_validation = pd.DataFrame([{
    "integrated_matches": len(integrated_match_context),
    "matches_with_source_metadata": int(
        (integrated_match_context["_merge"] == "both").sum()
    ),
    "matches_without_source_metadata": int(
        (integrated_match_context["_merge"] != "both").sum()
    ),
    "matches_with_revised_target": (
        int(integrated_match_context["revised_target"].notna().sum())
        if "revised_target" in integrated_match_context.columns
        else None
    ),
    "matches_with_first_innings_revision": (
        int(
            integrated_match_context[
                "first_inn_revised_overs"
            ].notna().sum()
        )
        if "first_inn_revised_overs" in integrated_match_context.columns
        else None
    ),
    "matches_with_second_innings_revision": (
        int(
            integrated_match_context[
                "second_inn_revised_overs"
            ].notna().sum()
        )
        if "second_inn_revised_overs" in integrated_match_context.columns
        else None
    )
}])

print("6. TARGET CONTEXT VALIDATION")
display(target_context_validation)


# 7. Preserve match population.
assert len(integrated_match_context) == 813
assert integrated_match_context["match_id"].is_unique
assert (
    integrated_match_context["_merge"] == "both"
).all()

print("All match-context population checks passed.")

1. MATCH-LEVEL TARGET AND CONTEXT FIELD INVENTORY


,field,non_null_source_matches,distinct_non_null_values,data_type
0,toss_win_team_id,868,37,float64
1,first_inn_revised_overs,25,13,float64
2,second_inn_revised_overs,25,12,float64
3,revised_target,23,23,float64
4,win_team_id,842,36,float64
5,result,883,670,object


2. INTEGRATED MATCH CONTEXT — FIRST 30


,match_id,comp_id,first_inn_revised_overs,second_inn_revised_overs,revised_target,result,win_team_id,_merge
0,13511,1331,NaN,NaN,NaN,West Indies Won by 9 Wickets,28.0,both
1,13512,1331,NaN,NaN,NaN,England Won by 1 Run,22.0,both
2,13513,1331,NaN,NaN,NaN,West Indies Won by 20 runs,28.0,both
3,13514,1331,NaN,NaN,NaN,England Won by 34 runs,22.0,both
4,13515,1331,NaN,NaN,NaN,West Indies Won by 17 runs,28.0,both
5,13522,1335,NaN,NaN,NaN,India Won by 6 Wickets,18.0,both
6,13523,1335,NaN,NaN,NaN,India Won by 8 Runs,18.0,both
7,13524,1335,NaN,NaN,NaN,India Won by 17 runs,18.0,both
8,13612,1345,NaN,114.0,143.0,Australia Won by 20 runs (D/L method),20.0,both
9,13613,1345,NaN,NaN,NaN,Australia Won the one over Eliminator,20.0,both


3. INTEGRATED MATCH CONTEXT COVERAGE


,field,integrated_matches,non_null_matches,missing_matches,distinct_non_null_values,minimum_value,maximum_value
0,first_inn_revised_overs,813,15,798,11,30.0,114.0
1,second_inn_revised_overs,813,22,791,10,30.0,114.0
2,revised_target,813,22,791,22,33.0,213.0


... (3 outputs left out)

,result_value,match_count
0,India Won by 7 wickets,8
1,Delhi Capitals Won by 6 wickets,7
2,Kolkata Knight Riders Won by 8 wickets,6
3,Punjab Kings Won by 5 wickets,5
4,Gujarat Titans Won by 7 wickets,5
5,India Won by 6 Wickets,5
6,Rajasthan Royals Won by 6 wickets,4
7,Pakistan Won by 7 wickets,4
8,West Indies Won by 5 wickets,4
9,Australia Won by 3 wickets,4


6. TARGET CONTEXT VALIDATION


,integrated_matches,matches_with_source_metadata,matches_without_source_metadata,matches_with_revised_target,matches_with_first_innings_revision,matches_with_second_innings_revision
0,813,813,0,22,15,22


All match-context population checks passed.


### Revised Overs Metadata vs Actual Delivery-Record Counts

In [64]:
# ============================================================
# PHASE 3B.14 — REVISED METADATA CROSSWALK (CORRECTED)
# ============================================================

# 1. Rebuild first- and second-innings crosswalks.
# Keep all source matches, including matches excluded from integration.

first_innings_crosswalk = revised_metadata[
    ["match_id", "first_inn_revised_overs"]
].merge(
    actual_innings_records.loc[
        actual_innings_records["innings"] == 1,
        ["match_id", "actual_delivery_records", "maximum_sequence_number"]
    ],
    on="match_id",
    how="left",
    validate="one_to_one",
    indicator="delivery_match_status"
)

first_innings_crosswalk["metadata_value"] = (
    first_innings_crosswalk["first_inn_revised_overs"]
)

first_innings_crosswalk["metadata_divided_by_six"] = (
    first_innings_crosswalk["metadata_value"] / 6
)

first_innings_crosswalk["metadata_minus_delivery_records"] = (
    first_innings_crosswalk["metadata_value"]
    - first_innings_crosswalk["actual_delivery_records"]
)

first_innings_crosswalk["metadata_equals_delivery_records"] = (
    first_innings_crosswalk["metadata_value"]
    == first_innings_crosswalk["actual_delivery_records"]
)


second_innings_crosswalk = revised_metadata[
    ["match_id", "second_inn_revised_overs", "revised_target"]
].merge(
    actual_innings_records.loc[
        actual_innings_records["innings"] == 2,
        ["match_id", "actual_delivery_records", "maximum_sequence_number"]
    ],
    on="match_id",
    how="left",
    validate="one_to_one",
    indicator="delivery_match_status"
)

second_innings_crosswalk["metadata_value"] = (
    second_innings_crosswalk["second_inn_revised_overs"]
)

second_innings_crosswalk["metadata_divided_by_six"] = (
    second_innings_crosswalk["metadata_value"] / 6
)

second_innings_crosswalk["metadata_minus_delivery_records"] = (
    second_innings_crosswalk["metadata_value"]
    - second_innings_crosswalk["actual_delivery_records"]
)

second_innings_crosswalk["metadata_equals_delivery_records"] = (
    second_innings_crosswalk["metadata_value"]
    == second_innings_crosswalk["actual_delivery_records"]
)


# 2. Show metadata-bearing matches with delivery-match status.
print("1. FIRST-INNINGS REVISED METADATA CROSSWALK")

display(
    first_innings_crosswalk.loc[
        first_innings_crosswalk["metadata_value"].notna()
    ].sort_values("match_id")
)

print("2. SECOND-INNINGS REVISED METADATA CROSSWALK")

display(
    second_innings_crosswalk.loc[
        second_innings_crosswalk["metadata_value"].notna()
    ].sort_values("match_id")
)


# 3. Build correctly aligned summary rows.
crosswalk_summary_rows = []

for innings_number, field_name, crosswalk in [
    (1, "first_inn_revised_overs", first_innings_crosswalk),
    (2, "second_inn_revised_overs", second_innings_crosswalk)
]:
    metadata_present = crosswalk["metadata_value"].notna()

    crosswalk_summary_rows.append({
        "innings": innings_number,
        "metadata_field": field_name,
        "source_matches": len(crosswalk),
        "metadata_rows": int(metadata_present.sum()),
        "metadata_rows_with_integrated_delivery_match": int(
            (
                metadata_present
                & crosswalk["delivery_match_status"].eq("both")
            ).sum()
        ),
        "metadata_rows_without_integrated_delivery_match": int(
            (
                metadata_present
                & crosswalk["delivery_match_status"].eq("left_only")
            ).sum()
        ),
        "metadata_value_min": crosswalk.loc[
            metadata_present, "metadata_value"
        ].min(),
        "metadata_value_max": crosswalk.loc[
            metadata_present, "metadata_value"
        ].max(),
        "metadata_equal_to_delivery_records": int(
            (
                metadata_present
                & crosswalk["metadata_equals_delivery_records"]
            ).sum()
        )
    })

crosswalk_summary = pd.DataFrame(crosswalk_summary_rows)

print("3. REVISED METADATA CROSSWALK SUMMARY")
display(crosswalk_summary)


# 4. Identify source-only matches carrying revised metadata.
source_only_first = first_innings_crosswalk.loc[
    first_innings_crosswalk["metadata_value"].notna()
    & first_innings_crosswalk["delivery_match_status"].eq("left_only")
].copy()

source_only_second = second_innings_crosswalk.loc[
    second_innings_crosswalk["metadata_value"].notna()
    & second_innings_crosswalk["delivery_match_status"].eq("left_only")
].copy()

source_only_metadata_exceptions = pd.concat(
    [
        source_only_first.assign(
            innings=1,
            metadata_field="first_inn_revised_overs"
        )[
            ["match_id", "innings", "metadata_field", "metadata_value"]
        ],
        source_only_second.assign(
            innings=2,
            metadata_field="second_inn_revised_overs"
        )[
            ["match_id", "innings", "metadata_field", "metadata_value"]
        ]
    ],
    ignore_index=True
)

print("4. SOURCE-ONLY MATCHES WITH REVISED METADATA")
display(source_only_metadata_exceptions)


# 5. Validation summary.
crosswalk_validation = pd.DataFrame([{
    "source_matches": len(df_source_matches),
    "integrated_matches": int(df_deliveries["match_id"].nunique()),
    "first_innings_crosswalk_rows": len(first_innings_crosswalk),
    "second_innings_crosswalk_rows": len(second_innings_crosswalk),
    "first_innings_metadata_rows": int(
        first_innings_crosswalk["metadata_value"].notna().sum()
    ),
    "first_innings_metadata_without_delivery_match": len(source_only_first),
    "second_innings_metadata_rows": int(
        second_innings_crosswalk["metadata_value"].notna().sum()
    ),
    "second_innings_metadata_without_delivery_match": len(source_only_second),
    "source_only_metadata_exception_rows": len(
        source_only_metadata_exceptions
    ),
    "crosswalk_rows_preserved": (
        len(first_innings_crosswalk) == len(df_source_matches)
        and len(second_innings_crosswalk) == len(df_source_matches)
    )
}])

print("5. CORRECTED CROSSWALK VALIDATION")
display(crosswalk_validation)


# 6. Assertions: validate the known population boundary, not
#    the false assumption that every source match has deliveries.

assert len(first_innings_crosswalk) == len(df_source_matches)
assert len(second_innings_crosswalk) == len(df_source_matches)

assert first_innings_crosswalk["match_id"].is_unique
assert second_innings_crosswalk["match_id"].is_unique

assert (
    first_innings_crosswalk["delivery_match_status"]
    .eq("both").sum()
    == 813
)

assert (
    second_innings_crosswalk["delivery_match_status"]
    .eq("both").sum()
    == 813
)

assert len(source_only_first) == 10
assert len(source_only_second) == 3

assert (
    crosswalk_validation.loc[0, "crosswalk_rows_preserved"]
)

print("All corrected revised metadata crosswalk validations passed.")

1. FIRST-INNINGS REVISED METADATA CROSSWALK


,match_id,first_inn_revised_overs,actual_delivery_records,maximum_sequence_number,delivery_match_status,metadata_value,metadata_divided_by_six,metadata_minus_delivery_records,metadata_equals_delivery_records
94,15410,114.0,NaN,NaN,left_only,114.0,19.0,NaN,False
95,15546,84.0,NaN,NaN,left_only,84.0,14.0,NaN,False
143,16753,66.0,71.0,71.0,both,66.0,11.0,-5.0,False
152,17395,48.0,49.0,49.0,both,48.0,8.0,-1.0,False
177,17765,72.0,NaN,NaN,left_only,72.0,12.0,NaN,False
263,19959,66.0,73.0,73.0,both,66.0,11.0,-7.0,False
273,21071,102.0,110.0,110.0,both,102.0,17.0,-8.0,False
420,31707,30.0,NaN,NaN,left_only,30.0,5.0,NaN,False
459,36098,78.0,79.0,79.0,both,78.0,13.0,-1.0,False
496,36208,54.0,49.0,49.0,both,54.0,9.0,5.0,False


2. SECOND-INNINGS REVISED METADATA CROSSWALK


,match_id,second_inn_revised_overs,revised_target,actual_delivery_records,maximum_sequence_number,delivery_match_status,metadata_value,metadata_divided_by_six,metadata_minus_delivery_records,metadata_equals_delivery_records
8,13612,114.0,143.0,118.0,118.0,both,114.0,19.0,-4.0,False
144,16754,42.0,56.0,40.0,40.0,both,42.0,7.0,2.0,False
152,17395,48.0,91.0,48.0,48.0,both,48.0,8.0,0.0,True
177,17765,72.0,130.0,NaN,NaN,left_only,72.0,12.0,NaN,False
272,21070,48.0,104.0,51.0,51.0,both,48.0,8.0,-3.0,False
273,21071,102.0,203.0,103.0,103.0,both,102.0,17.0,-1.0,False
262,21586,90.0,171.0,94.0,94.0,both,90.0,15.0,-4.0,False
307,28046,90.0,152.0,86.0,86.0,both,90.0,15.0,4.0,False
330,29984,60.0,126.0,63.0,63.0,both,60.0,10.0,-3.0,False
477,34428,48.0,78.0,39.0,39.0,both,48.0,8.0,9.0,False


3. REVISED METADATA CROSSWALK SUMMARY


,innings,metadata_field,source_matches,metadata_rows,metadata_rows_with_integrated_delivery_match,metadata_rows_without_integrated_delivery_match,metadata_value_min,metadata_value_max,metadata_equal_to_delivery_records
0,1,first_inn_revised_overs,883,25,15,10,30.0,114.0,2
1,2,second_inn_revised_overs,883,25,22,3,30.0,114.0,2


4. SOURCE-ONLY MATCHES WITH REVISED METADATA


,match_id,innings,metadata_field,metadata_value
0,15410,1,first_inn_revised_overs,114.0
1,15546,1,first_inn_revised_overs,84.0
2,17765,1,first_inn_revised_overs,72.0
3,31707,1,first_inn_revised_overs,30.0
4,39983,1,first_inn_revised_overs,54.0
5,39985,1,first_inn_revised_overs,108.0
6,41975,1,first_inn_revised_overs,108.0
7,46311,1,first_inn_revised_overs,54.0
8,46749,1,first_inn_revised_overs,84.0
9,50408,1,first_inn_revised_overs,54.0


5. CORRECTED CROSSWALK VALIDATION


,source_matches,integrated_matches,first_innings_crosswalk_rows,second_innings_crosswalk_rows,first_innings_metadata_rows,first_innings_metadata_without_delivery_match,second_innings_metadata_rows,second_innings_metadata_without_delivery_match,source_only_metadata_exception_rows,crosswalk_rows_preserved
0,883,813,883,883,25,10,25,3,13,True


All corrected revised metadata crosswalk validations passed.


## Competition Format and Benchmark-Stratum Audit

In [65]:
# 1. Inspect source competition metadata schema.
competition_schema = pd.DataFrame({
    "column": df_source_competitions.columns,
    "data_type": [
        str(df_source_competitions[col].dtype)
        for col in df_source_competitions.columns
    ],
    "non_null_rows": [
        int(df_source_competitions[col].notna().sum())
        for col in df_source_competitions.columns
    ],
    "distinct_non_null_values": [
        int(df_source_competitions[col].nunique(dropna=True))
        for col in df_source_competitions.columns
    ]
})

print("1. SOURCE COMPETITION METADATA SCHEMA")
display(competition_schema)


# 2. Identify likely competition identity and format fields by name.
competition_identity_candidates = [
    col for col in df_source_competitions.columns
    if any(
        term in col.lower()
        for term in [
            "comp", "format", "type", "name", "gender",
            "category", "level", "sport"
        ]
    )
]

print("2. COMPETITION IDENTITY / FORMAT CANDIDATE FIELDS")
display(
    competition_schema.loc[
        competition_schema["column"].isin(
            competition_identity_candidates
        )
    ].reset_index(drop=True)
)


# 3. Inspect integrated delivery fields relevant to competition grouping.
integrated_competition_candidates = [
    col for col in df_deliveries.columns
    if any(
        term in col.lower()
        for term in [
            "comp", "format", "type", "match_id"
        ]
    )
]

integrated_competition_schema = pd.DataFrame({
    "column": integrated_competition_candidates,
    "data_type": [
        str(df_deliveries[col].dtype)
        for col in integrated_competition_candidates
    ],
    "non_null_rows": [
        int(df_deliveries[col].notna().sum())
        for col in integrated_competition_candidates
    ],
    "distinct_non_null_values": [
        int(df_deliveries[col].nunique(dropna=True))
        for col in integrated_competition_candidates
    ]
})

print("3. INTEGRATED COMPETITION-RELATED FIELDS")
display(integrated_competition_schema)


# 4. Build a unique integrated match-to-competition mapping.
assert {"match_id", "comp_id"}.issubset(df_deliveries.columns)
assert "comp_id" in df_source_competitions.columns

integrated_match_competitions = (
    df_deliveries[["match_id", "comp_id"]]
    .drop_duplicates()
)

assert integrated_match_competitions["match_id"].is_unique

# Match each integrated competition ID to the source competition table.
competition_metadata_audit = integrated_match_competitions[
    ["comp_id"]
].drop_duplicates().merge(
    df_source_competitions,
    on="comp_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("4. INTEGRATED COMPETITION METADATA MATCH — FIRST 30")
display(competition_metadata_audit.head(30))


# 5. Summarise integrated matches and competitions.
competition_population_summary = pd.DataFrame([{
    "integrated_matches": len(integrated_match_competitions),
    "integrated_competitions": int(
        integrated_match_competitions["comp_id"].nunique()
    ),
    "competition_ids_matched_to_source": int(
        (competition_metadata_audit["_merge"] == "both").sum()
    ),
    "competition_ids_missing_from_source": int(
        (competition_metadata_audit["_merge"] != "both").sum()
    ),
    "source_competition_records": len(df_source_competitions),
    "source_competition_ids_unique": bool(
        df_source_competitions["comp_id"].is_unique
    )
}])

print("5. COMPETITION POPULATION RECONCILIATION")
display(competition_population_summary)


# 6. If a match-format field exists in the integrated data,
#    inspect its values without assuming their meaning.
format_fields = [
    col for col in df_deliveries.columns
    if col.lower() in [
        "match_format", "format", "match_type", "competition_format"
    ]
]

if format_fields:
    format_distribution_rows = []

    for field in format_fields:
        values = (
            df_deliveries[["match_id", field]]
            .drop_duplicates()
            .groupby(field, dropna=False)
            .agg(
                match_count=("match_id", "nunique")
            )
            .reset_index()
            .rename(columns={field: "observed_format_value"})
        )

        values.insert(0, "source_field", field)
        format_distribution_rows.append(values)

    format_distribution = pd.concat(
        format_distribution_rows,
        ignore_index=True
    )

    print("6. OBSERVED MATCH FORMAT VALUES")
    display(format_distribution)
else:
    print(
        "6. No explicitly named match-format field was found "
        "in the integrated delivery schema."
    )


# 7. Display competition IDs that failed source metadata matching.
competition_metadata_exceptions = competition_metadata_audit.loc[
    competition_metadata_audit["_merge"] != "both"
].copy()

print("7. COMPETITION METADATA MATCH EXCEPTIONS")
display(competition_metadata_exceptions)


# 8. Population-preservation checks.
assert len(integrated_match_competitions) == 813
assert integrated_match_competitions["match_id"].is_unique
assert df_source_competitions["comp_id"].is_unique

print("Competition schema and population audit completed.")

1. SOURCE COMPETITION METADATA SCHEMA


,column,data_type,non_null_rows,distinct_non_null_values
0,comp_id,int64,141,141
1,comp_name,object,141,141
2,match_format,object,141,2
3,season,object,141,8


2. COMPETITION IDENTITY / FORMAT CANDIDATE FIELDS


,column,data_type,non_null_rows,distinct_non_null_values
0,comp_id,int64,141,141
1,comp_name,object,141,141
2,match_format,object,141,2


3. INTEGRATED COMPETITION-RELATED FIELDS


,column,data_type,non_null_rows,distinct_non_null_values
0,match_id,int64,192235,813
1,batting_type,object,192235,2
2,bowling_type,object,192235,12
3,dismissal_type_id,float64,10392,8
4,dismissal_type,object,10392,9
5,extras_type,object,11373,6
6,dismissal_type_original,object,10392,9
7,dismissal_type_normalized,object,10392,8
8,comp_id,int64,192235,136
9,comp_name,object,192235,136


... (5 outputs left out)

,source_field,observed_format_value,match_count
0,match_format,T20,362
1,match_format,T20I,451


7. COMPETITION METADATA MATCH EXCEPTIONS


,comp_id,comp_name,match_format,season,_merge


Competition schema and population audit completed.


### Competition-Level Match Population Audit

In [67]:
# 1. Create one row per integrated match and competition.
integrated_match_competitions = (
    df_deliveries[
        ["match_id", "comp_id", "match_format"]
    ]
    .drop_duplicates()
    .copy()
)

assert integrated_match_competitions["match_id"].is_unique, (
    "An integrated match maps to multiple competition records."
)

# 2. Rename source and integrated fields explicitly before merging.
source_competition_lookup = (
    df_source_competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ]
    .rename(
        columns={
            "match_format": "source_match_format"
        }
    )
    .copy()
)

competition_match_detail = integrated_match_competitions.merge(
    source_competition_lookup,
    on="comp_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# 3. Compare integrated and source competition formats.
competition_match_detail["format_matches_source"] = (
    competition_match_detail["match_format"]
    == competition_match_detail["source_match_format"]
)

# 4. Aggregate integrated matches by competition.
competition_match_counts = (
    competition_match_detail
    .groupby(
        [
            "comp_id",
            "comp_name",
            "match_format",
            "season"
        ],
        dropna=False,
        observed=True
    )
    .agg(
        integrated_matches=("match_id", "nunique"),
        source_metadata_matched=(
            "_merge",
            lambda x: (x == "both").all()
        ),
        formats_match_source=(
            "format_matches_source",
            "all"
        )
    )
    .reset_index()
    .sort_values(
        ["match_format", "comp_name"],
        na_position="last"
    )
)

# 5. Summarise the integrated population by format.
format_population_summary = (
    competition_match_detail
    .groupby("match_format", dropna=False, observed=True)
    .agg(
        integrated_matches=("match_id", "nunique"),
        competition_count=("comp_id", "nunique")
    )
    .reset_index()
)

# 6. Validate population and metadata consistency.
validation_summary = pd.DataFrame(
    [
        {
            "check": "Integrated match count",
            "observed": competition_match_detail["match_id"].nunique(),
            "expected": 813,
            "passed": (
                competition_match_detail["match_id"].nunique() == 813
            )
        },
        {
            "check": "Unmatched competition metadata rows",
            "observed": (competition_match_detail["_merge"] != "both").sum(),
            "expected": 0,
            "passed": (
                (competition_match_detail["_merge"] == "both").all()
            )
        },
        {
            "check": "Competition format mismatches",
            "observed": (~competition_match_detail["format_matches_source"]).sum(),
            "expected": 0,
            "passed": (
                competition_match_detail["format_matches_source"].all()
            )
        },
        {
            "check": "T20 integrated matches",
            "observed": (
                competition_match_detail.loc[
                    competition_match_detail["match_format"] == "T20",
                    "match_id"
                ].nunique()
            ),
            "expected": 362,
            "passed": (
                competition_match_detail.loc[
                    competition_match_detail["match_format"] == "T20",
                    "match_id"
                ].nunique() == 362
            )
        },
        {
            "check": "T20I integrated matches",
            "observed": (
                competition_match_detail.loc[
                    competition_match_detail["match_format"] == "T20I",
                    "match_id"
                ].nunique()
            ),
            "expected": 451,
            "passed": (
                competition_match_detail.loc[
                    competition_match_detail["match_format"] == "T20I",
                    "match_id"
                ].nunique() == 451
            )
        }
    ]
)

# 7. Display readable audit tables.
display(format_population_summary)
display(competition_match_counts)
display(validation_summary)

assert validation_summary["passed"].all(), (
    "Competition-level population audit failed. "
    "Review the validation table before proceeding."
)

,match_format,integrated_matches,competition_count
0,T20,362,5
1,T20I,451,131


,comp_id,comp_name,match_format,season,integrated_matches,source_metadata_matched,formats_match_source
3,1358,IPL 2022,T20,2022,74,True,True
31,1715,IPL 2023,T20,2023,73,True,True
53,2193,IPL 2024,T20,2024,71,True,True
83,2725,IPL 2025,T20,2025,71,True,True
116,3298,IPL 2026,T20,2026,73,True,True
...,...,...,...,...,...,...,...
32,1718,West Indies in South Africa T20I Series 2023,T20I,2023,3,True,True
91,2906,West Indies in South Africa T20I Series 2026,T20I,2026,3,True,True
79,2692,West Indies in Sri Lanka T20I Series 2024,T20I,2024,3,True,True
60,2371,Zimbabwe in Bangladesh T20I Series 2024,T20I,2024,5,True,True


,check,observed,expected,passed
0,Integrated match count,813,813,True
1,Unmatched competition metadata rows,0,0,True
2,Competition format mismatches,0,0,True
3,T20 integrated matches,362,362,True
4,T20I integrated matches,451,451,True


### Approved Benchmark Population Classification

In [68]:
# 1. Create a match-level competition classification.
phase3_benchmark_population = competition_match_detail[
    [
        "match_id",
        "comp_id",
        "comp_name",
        "match_format",
        "season"
    ]
].copy()

# 2. Assign benchmark populations using explicit rules.
phase3_benchmark_population["benchmark_population"] = "UNCLASSIFIED"

phase3_benchmark_population.loc[
    phase3_benchmark_population["match_format"].eq("T20")
    & phase3_benchmark_population["comp_name"].str.startswith(
        "IPL", na=False
    ),
    "benchmark_population"
] = "IPL"

phase3_benchmark_population.loc[
    phase3_benchmark_population["match_format"].eq("T20I"),
    "benchmark_population"
] = "T20I"

# 3. Summarise the assigned populations.
benchmark_population_summary = (
    phase3_benchmark_population
    .groupby(
        "benchmark_population",
        dropna=False,
        observed=True
    )
    .agg(
        integrated_matches=("match_id", "nunique"),
        competition_count=("comp_id", "nunique"),
        first_season=("season", "min"),
        last_season=("season", "max")
    )
    .reset_index()
)

# 4. Identify any unclassified or duplicate match assignments.
unclassified_matches = phase3_benchmark_population.loc[
    phase3_benchmark_population["benchmark_population"].eq("UNCLASSIFIED")
].copy()

duplicate_match_assignments = (
    phase3_benchmark_population["match_id"].duplicated().sum()
)

# 5. Validate the two benchmark populations.
validation_summary_3b17 = pd.DataFrame(
    [
        {
            "check": "All integrated matches classified",
            "observed": len(unclassified_matches),
            "expected": 0,
            "passed": len(unclassified_matches) == 0
        },
        {
            "check": "No duplicate match assignments",
            "observed": duplicate_match_assignments,
            "expected": 0,
            "passed": duplicate_match_assignments == 0
        },
        {
            "check": "IPL match population",
            "observed": phase3_benchmark_population.loc[
                phase3_benchmark_population["benchmark_population"].eq("IPL"),
                "match_id"
            ].nunique(),
            "expected": 362,
            "passed": (
                phase3_benchmark_population.loc[
                    phase3_benchmark_population["benchmark_population"].eq("IPL"),
                    "match_id"
                ].nunique() == 362
            )
        },
        {
            "check": "T20I match population",
            "observed": phase3_benchmark_population.loc[
                phase3_benchmark_population["benchmark_population"].eq("T20I"),
                "match_id"
            ].nunique(),
            "expected": 451,
            "passed": (
                phase3_benchmark_population.loc[
                    phase3_benchmark_population["benchmark_population"].eq("T20I"),
                    "match_id"
                ].nunique() == 451
            )
        },
        {
            "check": "Total classified match population",
            "observed": phase3_benchmark_population["match_id"].nunique(),
            "expected": 813,
            "passed": (
                phase3_benchmark_population["match_id"].nunique() == 813
            )
        }
    ]
)

# 6. Display audit tables.
display(benchmark_population_summary)
display(
    phase3_benchmark_population[
        [
            "comp_id",
            "comp_name",
            "match_format",
            "season",
            "benchmark_population"
        ]
    ].drop_duplicates().sort_values(
        ["benchmark_population", "season", "comp_name"],
        na_position="last"
    )
)
display(validation_summary_3b17)

assert validation_summary_3b17["passed"].all(), (
    "Benchmark population classification failed. "
    "Review the displayed audit tables."
)

,benchmark_population,integrated_matches,competition_count,first_season,last_season
0,IPL,362,5,2022,2026
1,T20I,451,131,2022,2026


,comp_id,comp_name,match_format,season,benchmark_population
13,1358,IPL 2022,T20,2022,IPL
192,1715,IPL 2023,T20,2023,IPL
334,2193,IPL 2024,T20,2024,IPL
505,2725,IPL 2025,T20,2025,IPL
705,3298,IPL 2026,T20,2026,IPL
...,...,...,...,...,...
614,3128,Sri Lanka in England T20I Series 2026,T20I,2026,T20I
807,3765,Sri Lanka in Pakistan T20I Series 2026,T20I,2026,T20I
782,3464,Sri Lanka in West Indies T20I Series 2026,T20I,2026,T20I
778,3450,West Indies in India T20I Series 2026,T20I,2026,T20I


,check,observed,expected,passed
0,All integrated matches classified,0,0,True
1,No duplicate match assignments,0,0,True
2,IPL match population,362,362,True
3,T20I match population,451,451,True
4,Total classified match population,813,813,True


## Pressure-State Reconstruction Feasibility Gate

In [69]:
# 1. Consolidate audited evidence into a pressure-state feasibility register.
phase3B_feasibility_register = pd.DataFrame(
    [
        {
            "component": "Pre-delivery wickets fallen",
            "required_evidence": "Reliable wicket-event sequence before each delivery",
            "current_evidence": "Reconstructed for 192,235 deliveries; 10,392 wicket events reconcile",
            "status": "READY_WITH_CAVEAT",
            "limitation": "Arithmetic validated against supplied is_wicket; dismissal classification not independently verified"
        },
        {
            "component": "Pre-delivery cumulative score",
            "required_evidence": "Verified delivery scoring semantics and score aggregation",
            "current_evidence": "runs, extras1_runs and extras_2_runs audited; scoring semantics remain unresolved",
            "status": "BLOCKED",
            "limitation": "Cannot safely reconstruct cumulative innings score or scoring deviation"
        },
        {
            "component": "Legal balls remaining",
            "required_evidence": "Verified legal-ball count and scheduled innings length",
            "current_evidence": "184,002 deliveries have no recorded wide/overstep; 7,976 have non-legal evidence; 257 unresolved",
            "status": "BLOCKED",
            "limitation": "Delivery sequence is not a verified legal-ball counter; scheduled length units and semantics are unresolved"
        },
        {
            "component": "Innings stage",
            "required_evidence": "Verified legal-ball progression and scheduled innings allocation",
            "current_evidence": "Standard T20 stage definitions exist, but match-specific legal-ball progression and scheduled lengths are not established",
            "status": "BLOCKED",
            "limitation": "Cannot reliably assign shortened or revised innings to benchmark stages"
        },
        {
            "component": "First-innings scoring position",
            "required_evidence": "Verified cumulative score and comparable pre-delivery progression",
            "current_evidence": "Expected cumulative scoring requires validated score and progression inputs",
            "status": "BLOCKED",
            "limitation": "Neither actual cumulative score nor comparable legal-ball progression is currently established"
        },
        {
            "component": "Second-innings scoring demand",
            "required_evidence": "Verified target, current score and legal balls remaining",
            "current_evidence": "22 integrated matches contain revised targets; no complete target history or verified scoring reconstruction",
            "status": "BLOCKED",
            "limitation": "Required run rate cannot be reliably reconstructed across the benchmark population"
        },
        {
            "component": "Competition benchmark separation",
            "required_evidence": "Audited match-to-competition mapping",
            "current_evidence": "362 IPL matches across 5 competitions and 451 T20I matches across 131 competitions",
            "status": "READY",
            "limitation": "The approved population is limited to the audited IPL competition IDs and T20I population"
        },
        {
            "component": "Innings and team structure",
            "required_evidence": "Stable innings identity and batting/bowling team assignments",
            "current_evidence": "813 matches; exactly two innings per match; team-role reversal validated",
            "status": "READY",
            "limitation": "Structural validation does not resolve scoring or legal-ball semantics"
        },
        {
            "component": "Historical benchmark construction",
            "required_evidence": "Validated pressure-state inputs and defensible comparable-state dimensions",
            "current_evidence": "Competition populations identified; scoring, legal-ball progression and scheduled-length evidence remain incomplete",
            "status": "BLOCKED",
            "limitation": "Constructing historical expected-scoring benchmarks now would rely on unverified state variables"
        }
    ]
)

# 2. Summarise feasibility status.
phase3B_feasibility_summary = (
    phase3B_feasibility_register
    .groupby("status", observed=True)
    .size()
    .reset_index(name="component_count")
)

# 3. Export audit registers.
phase3B_feasibility_register.to_csv(
    OUTPUT_DIR / "phase3B_pressure_state_feasibility_register.csv",
    index=False
)

phase3B_feasibility_summary.to_csv(
    OUTPUT_DIR / "phase3B_pressure_state_feasibility_summary.csv",
    index=False
)

# 4. Display readable tables.
display(phase3B_feasibility_summary)
display(phase3B_feasibility_register)

assert phase3B_feasibility_register["component"].is_unique
assert phase3B_feasibility_register["status"].isin(
    ["READY", "READY_WITH_CAVEAT", "BLOCKED"]
).all()
assert len(phase3B_feasibility_register) == 9

,status,component_count
0,BLOCKED,6
1,READY,2
2,READY_WITH_CAVEAT,1


,component,required_evidence,current_evidence,status,limitation
0,Pre-delivery wickets fallen,Reliable wicket-event sequence before each delivery,"Reconstructed for 192,235 deliveries; 10,392 wicket events reconcile",READY_WITH_CAVEAT,Arithmetic validated against supplied is_wicket; dismissal classification not independently verified
1,Pre-delivery cumulative score,Verified delivery scoring semantics and score aggregation,"runs, extras1_runs and extras_2_runs audited; scoring semantics remain unresolved",BLOCKED,Cannot safely reconstruct cumulative innings score or scoring deviation
2,Legal balls remaining,Verified legal-ball count and scheduled innings length,"184,002 deliveries have no recorded wide/overstep; 7,976 have non-legal evidence; 257 unresolved",BLOCKED,Delivery sequence is not a verified legal-ball counter; scheduled length units and semantics are unresolved
3,Innings stage,Verified legal-ball progression and scheduled innings allocation,"Standard T20 stage definitions exist, but match-specific legal-ball progression and scheduled lengths are not established",BLOCKED,Cannot reliably assign shortened or revised innings to benchmark stages
4,First-innings scoring position,Verified cumulative score and comparable pre-delivery progression,Expected cumulative scoring requires validated score and progression inputs,BLOCKED,Neither actual cumulative score nor comparable legal-ball progression is currently established
5,Second-innings scoring demand,"Verified target, current score and legal balls remaining",22 integrated matches contain revised targets; no complete target history or verified scoring reconstruction,BLOCKED,Required run rate cannot be reliably reconstructed across the benchmark population
6,Competition benchmark separation,Audited match-to-competition mapping,362 IPL matches across 5 competitions and 451 T20I matches across 131 competitions,READY,The approved population is limited to the audited IPL competition IDs and T20I population
7,Innings and team structure,Stable innings identity and batting/bowling team assignments,813 matches; exactly two innings per match; team-role reversal validated,READY,Structural validation does not resolve scoring or legal-ball semantics
8,Historical benchmark construction,Validated pressure-state inputs and defensible comparable-state dimensions,"Competition populations identified; scoring, legal-ball progression and scheduled-length evidence remain incomplete",BLOCKED,Constructing historical expected-scoring benchmarks now would rely on unverified state variables


### Dismissal Event and Wicket-State Evidence Audit

In [70]:
# 1. Create a separate dismissal evidence table.
dismissal_evidence = df_deliveries[
    [
        "match_id",
        "innings",
        "innings_ball_num",
        "is_wicket",
        "dismissal_type",
        "dismissal_type_normalized",
        "dismissed_player_id"
    ]
].copy()

dismissal_evidence["is_wicket"] = (
    dismissal_evidence["is_wicket"].fillna(False).astype(bool)
)

# 2. Summarise dismissal types on recorded wicket events.
wicket_type_summary = (
    dismissal_evidence.loc[dismissal_evidence["is_wicket"]]
    .groupby(
        ["dismissal_type", "dismissal_type_normalized"],
        dropna=False,
        observed=True
    )
    .agg(
        wicket_events=("match_id", "size"),
        matches=("match_id", "nunique"),
        missing_dismissed_player_ids=(
            "dismissed_player_id",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
    .sort_values("wicket_events", ascending=False)
)

# 3. Identify record-level inconsistencies.
wicket_missing_type = dismissal_evidence.loc[
    dismissal_evidence["is_wicket"]
    & dismissal_evidence["dismissal_type_normalized"].isna()
].copy()

non_wicket_with_dismissed_id = dismissal_evidence.loc[
    ~dismissal_evidence["is_wicket"]
    & dismissal_evidence["dismissed_player_id"].notna()
].copy()

wicket_missing_player_id = dismissal_evidence.loc[
    dismissal_evidence["is_wicket"]
    & dismissal_evidence["dismissed_player_id"].isna()
].copy()

# 4. Compare wicket events with the reconstructed wicket-state table.
wicket_state_reconciliation = pd.DataFrame(
    [
        {
            "measure": "Wicket events in integrated deliveries",
            "observed": int(dismissal_evidence["is_wicket"].sum()),
            "expected": 10392
        },
        {
            "measure": "Wicket events in reconstructed state",
            "observed": int(
                phase3_wicket_state["is_wicket"].fillna(False).astype(bool).sum()
            ),
            "expected": 10392
        },
        {
            "measure": "Wicket events missing dismissal type",
            "observed": len(wicket_missing_type),
            "expected": 0
        },
        {
            "measure": "Wicket events missing dismissed player ID",
            "observed": len(wicket_missing_player_id),
            "expected": 0
        },
        {
            "measure": "Non-wicket records with dismissed player ID",
            "observed": len(non_wicket_with_dismissed_id),
            "expected": 0
        }
    ]
)

wicket_state_reconciliation["difference"] = (
    wicket_state_reconciliation["observed"]
    - wicket_state_reconciliation["expected"]
)

# 5. Export audit tables.
wicket_type_summary.to_csv(
    OUTPUT_DIR / "phase3B_dismissal_type_evidence.csv",
    index=False
)

wicket_state_reconciliation.to_csv(
    OUTPUT_DIR / "phase3B_wicket_state_reconciliation.csv",
    index=False
)

wicket_missing_player_id.to_csv(
    OUTPUT_DIR / "phase3B_wicket_missing_player_id.csv",
    index=False
)

non_wicket_with_dismissed_id.to_csv(
    OUTPUT_DIR / "phase3B_non_wicket_with_dismissed_id.csv",
    index=False
)

# 6. Display the evidence tables.
display(wicket_type_summary)
display(wicket_state_reconciliation)
display(
    wicket_missing_player_id[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "dismissal_type",
            "dismissal_type_normalized"
        ]
    ]
)
display(
    non_wicket_with_dismissed_id[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "dismissed_player_id"
        ]
    ]
)

assert dismissal_evidence["is_wicket"].sum() == 10392
assert len(wicket_missing_type) == 0
assert (
    phase3_wicket_state["is_wicket"].fillna(False).astype(bool).sum()
    == 10392
)

,dismissal_type,dismissal_type_normalized,wicket_events,matches,missing_dismissed_player_ids
1,Caught,Caught,7264,812,0
0,Bowled,Bowled,1675,694,0
3,LBW,LBW,663,419,0
7,RunOut,RunOut,567,402,0
8,Stump,Stump,198,171,0
2,Hit Wicket,Hit Wicket,17,17,0
6,Retire Out,Retire Out,6,6,1
4,Obstracting the field,Obstructing the field,1,1,0
5,Obstructing the field,Obstructing the field,1,1,0


,measure,observed,expected,difference
0,Wicket events in integrated deliveries,10392,10392,0
1,Wicket events in reconstructed state,10392,10392,0
2,Wicket events missing dismissal type,0,0,0
3,Wicket events missing dismissed player ID,1,0,1
4,Non-wicket records with dismissed player ID,11,0,11


,match_id,innings,innings_ball_num,dismissal_type,dismissal_type_normalized
123543,40709,2,117,Retire Out,Retire Out


,match_id,innings,innings_ball_num,dismissed_player_id
8245,14114,2,80,1440.0
23527,15610,1,38,6433.0
25751,16064,2,10,126.0
47546,20188,1,64,2003.0
60694,20243,1,99,4219.0
63254,20880,2,65,2533.0
76606,28964,1,65,1568.0
78912,29926,2,54,6931.0
98444,31709,1,80,3024.0
127313,40725,2,36,2258.0


### Wicket-State Progression and Innings Boundary Audit

In [71]:
# 1. Build an independent validation view of the reconstructed wicket states.
wicket_progression_audit = phase3_wicket_state[
    [
        "match_id",
        "innings",
        "innings_ball_num",
        "is_wicket",
        "pre_delivery_wickets_fallen",
        "post_delivery_wickets_fallen"
    ]
].copy()

wicket_progression_audit["is_wicket"] = (
    wicket_progression_audit["is_wicket"]
    .fillna(False)
    .astype(bool)
)

# 2. Calculate the expected post-delivery state.
wicket_progression_audit["expected_post_delivery_wickets"] = (
    wicket_progression_audit["pre_delivery_wickets_fallen"]
    + wicket_progression_audit["is_wicket"].astype(int)
)

wicket_progression_audit["state_transition_valid"] = (
    wicket_progression_audit["post_delivery_wickets_fallen"]
    == wicket_progression_audit["expected_post_delivery_wickets"]
)

# 3. Summarise each innings.
innings_wicket_progression = (
    wicket_progression_audit
    .groupby(["match_id", "innings"], observed=True)
    .agg(
        delivery_records=("innings_ball_num", "size"),
        wicket_events=("is_wicket", "sum"),
        starting_pre_delivery_wickets=(
            "pre_delivery_wickets_fallen", "first"
        ),
        ending_post_delivery_wickets=(
            "post_delivery_wickets_fallen", "last"
        ),
        maximum_pre_delivery_wickets=(
            "pre_delivery_wickets_fallen", "max"
        ),
        maximum_post_delivery_wickets=(
            "post_delivery_wickets_fallen", "max"
        ),
        all_transitions_valid=("state_transition_valid", "all")
    )
    .reset_index()
)

innings_wicket_progression["ending_state_matches_event_count"] = (
    innings_wicket_progression["ending_post_delivery_wickets"]
    == innings_wicket_progression["wicket_events"]
)

# 4. Identify any innings with more than 10 recorded wicket events.
innings_over_ten_wickets = innings_wicket_progression.loc[
    innings_wicket_progression["wicket_events"] > 10
].copy()

# 5. Produce a compact validation summary.
wicket_progression_validation = pd.DataFrame(
    [
        {
            "check": "All delivery-level state transitions valid",
            "observed": int(
                (~wicket_progression_audit["state_transition_valid"]).sum()
            ),
            "expected": 0
        },
        {
            "check": "All innings start at zero wickets",
            "observed": int(
                (
                    innings_wicket_progression[
                        "starting_pre_delivery_wickets"
                    ] != 0
                ).sum()
            ),
            "expected": 0
        },
        {
            "check": "Innings ending state matches wicket-event count",
            "observed": int(
                (
                    ~innings_wicket_progression[
                        "ending_state_matches_event_count"
                    ]
                ).sum()
            ),
            "expected": 0
        },
        {
            "check": "Innings with more than 10 wicket events",
            "observed": len(innings_over_ten_wickets),
            "expected": 0
        },
        {
            "check": "Total innings audited",
            "observed": len(innings_wicket_progression),
            "expected": 1626
        }
    ]
)

wicket_progression_validation["passed"] = (
    wicket_progression_validation["observed"]
    == wicket_progression_validation["expected"]
)

# 6. Export audit outputs.
innings_wicket_progression.to_csv(
    OUTPUT_DIR / "phase3B_innings_wicket_progression.csv",
    index=False
)

wicket_progression_validation.to_csv(
    OUTPUT_DIR / "phase3B_wicket_progression_validation.csv",
    index=False
)

innings_over_ten_wickets.to_csv(
    OUTPUT_DIR / "phase3B_innings_over_ten_wickets.csv",
    index=False
)

# 7. Display readable audit tables.
display(wicket_progression_validation)
display(
    innings_wicket_progression[
        [
            "match_id",
            "innings",
            "delivery_records",
            "wicket_events",
            "starting_pre_delivery_wickets",
            "ending_post_delivery_wickets",
            "maximum_post_delivery_wickets",
            "all_transitions_valid",
            "ending_state_matches_event_count"
        ]
    ].describe(include="all").T
)
display(innings_over_ten_wickets)

assert wicket_progression_validation["passed"].all(), (
    "Wicket progression validation failed. Review the audit tables."
)

,check,observed,expected,passed
0,All delivery-level state transitions valid,0,0,True
1,All innings start at zero wickets,0,0,True
2,Innings ending state matches wicket-event count,0,0,True
3,Innings with more than 10 wicket events,0,0,True
4,Total innings audited,1626,1626,True


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_id,1626.0,NaN,NaN,NaN,32463.105781,12942.712452,13511.0,20192.0,31287.0,44267.0,55561.0
innings,1626.0,NaN,NaN,NaN,1.5,0.500154,1.0,1.0,1.5,2.0,2.0
delivery_records,1626.0,NaN,NaN,NaN,118.225707,15.308121,30.0,118.0,123.0,126.0,136.0
wicket_events,1626.0,NaN,NaN,NaN,6.391144,2.525834,0.0,5.0,6.0,8.0,10.0
starting_pre_delivery_wickets,1626.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ending_post_delivery_wickets,1626.0,NaN,NaN,NaN,6.391144,2.525834,0.0,5.0,6.0,8.0,10.0
maximum_post_delivery_wickets,1626.0,NaN,NaN,NaN,6.391144,2.525834,0.0,5.0,6.0,8.0,10.0
all_transitions_valid,1626,1,True,1626,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ending_state_matches_event_count,1626,1,True,1626,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,match_id,innings,delivery_records,wicket_events,starting_pre_delivery_wickets,ending_post_delivery_wickets,maximum_pre_delivery_wickets,maximum_post_delivery_wickets,all_transitions_valid,ending_state_matches_event_count


### Approved Pre-Delivery Wicket-State Dataset

In [72]:
# 1. Create a unique match-to-benchmark-population lookup.
benchmark_population_lookup = (
    phase3_benchmark_population[
        ["match_id", "benchmark_population"]
    ]
    .drop_duplicates()
    .copy()
)

assert benchmark_population_lookup["match_id"].is_unique

# 2. Attach benchmark population to the reconstructed wicket states.
phase3_pre_delivery_wicket_states = phase3_wicket_state.merge(
    benchmark_population_lookup,
    on="match_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# 3. Validate delivery coverage and state completeness.
wicket_state_dataset_validation = pd.DataFrame(
    [
        {
            "check": "Total delivery rows",
            "observed": len(phase3_pre_delivery_wicket_states),
            "expected": 192235
        },
        {
            "check": "Rows assigned to benchmark population",
            "observed": (
                phase3_pre_delivery_wicket_states[
                    "benchmark_population"
                ].notna().sum()
            ),
            "expected": 192235
        },
        {
            "check": "Rows with unmatched population",
            "observed": (
                phase3_pre_delivery_wicket_states["_merge"]
                .ne("both").sum()
            ),
            "expected": 0
        },
        {
            "check": "Missing pre-delivery wicket states",
            "observed": (
                phase3_pre_delivery_wicket_states[
                    "pre_delivery_wickets_fallen"
                ].isna().sum()
            ),
            "expected": 0
        },
        {
            "check": "Pre-delivery wicket states outside 0–10",
            "observed": (
                (
                    ~phase3_pre_delivery_wicket_states[
                        "pre_delivery_wickets_fallen"
                    ].between(0, 10)
                ).sum()
            ),
            "expected": 0
        }
    ]
)

wicket_state_dataset_validation["passed"] = (
    wicket_state_dataset_validation["observed"]
    == wicket_state_dataset_validation["expected"]
)

# 4. Summarise the pre-delivery wicket-state distribution.
pre_delivery_wicket_distribution = (
    phase3_pre_delivery_wicket_states
    .groupby(
        ["benchmark_population", "pre_delivery_wickets_fallen"],
        observed=True,
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["benchmark_population", "pre_delivery_wickets_fallen"]
    )
)

# 5. Export the reusable dataset and audit tables.
phase3_pre_delivery_wicket_states.to_parquet(
    OUTPUT_DIR / "phase3B_pre_delivery_wicket_states.parquet",
    index=False
)

pre_delivery_wicket_distribution.to_csv(
    OUTPUT_DIR / "phase3B_pre_delivery_wicket_distribution.csv",
    index=False
)

wicket_state_dataset_validation.to_csv(
    OUTPUT_DIR / "phase3B_pre_delivery_wicket_validation.csv",
    index=False
)

# 6. Display validation and distribution tables.
display(wicket_state_dataset_validation)
display(pre_delivery_wicket_distribution)

assert wicket_state_dataset_validation["passed"].all(), (
    "Pre-delivery wicket-state dataset validation failed."
)

,check,observed,expected,passed
0,Total delivery rows,192235,192235,True
1,Rows assigned to benchmark population,192235,192235,True
2,Rows with unmatched population,0,0,True
3,Missing pre-delivery wicket states,0,0,True
4,Pre-delivery wicket states outside 0–10,0,0,True


,benchmark_population,pre_delivery_wickets_fallen,delivery_records,matches
0,IPL,0,17400,362
1,IPL,1,16201,362
2,IPL,2,14814,362
3,IPL,3,12690,362
4,IPL,4,9970,359
5,IPL,5,7066,343
6,IPL,6,4287,310
7,IPL,7,2336,253
8,IPL,8,1564,191
9,IPL,9,889,120


## Uploaded Documentation Search for Unresolved Field Semantics

In [73]:
# 1. Collect report text already loaded in the notebook.
phase3_document_sources = {
    "Phase 0 report": phase0_report,
    "Phase 1 report": phase1_report_text,
    "Phase 2 report": phase2_report
}

# 2. Define unresolved field terms and related concepts.
phase3_search_terms = [
    "runs",
    "extras_2_runs",
    "extras1_runs",
    "revised_overs",
    "revised_target",
    "legal ball",
    "legal delivery",
    "scheduled overs",
    "innings length",
    "innings_ball_num",
    "scoring reconstruction",
    "target calculation"
]

# 3. Search each report and extract matching context.
documentation_search_results = []

for source_name, source_text in phase3_document_sources.items():
    text_content = str(source_text)
    text_lines = text_content.splitlines()

    for line_number, line in enumerate(text_lines):
        for search_term in phase3_search_terms:
            if search_term.lower() in line.lower():
                start = max(0, line_number - 1)
                end = min(len(text_lines), line_number + 2)

                context = " ".join(
                    text_lines[start:end]
                ).strip()

                documentation_search_results.append(
                    {
                        "source": source_name,
                        "search_term": search_term,
                        "line_number": line_number + 1,
                        "matching_text": line.strip(),
                        "context": context
                    }
                )

# 4. Create a readable result table.
documentation_search_results_df = pd.DataFrame(
    documentation_search_results
)

if not documentation_search_results_df.empty:
    documentation_search_results_df = (
        documentation_search_results_df
        .drop_duplicates(
            subset=["source", "search_term", "line_number"]
        )
        .sort_values(
            ["search_term", "source", "line_number"]
        )
        .reset_index(drop=True)
    )

# 5. Summarise search coverage by source and term.
documentation_search_summary = (
    documentation_search_results_df
    .groupby(
        ["source", "search_term"],
        observed=True
    )
    .size()
    .reset_index(name="matching_lines")
)

display(documentation_search_summary)
display(documentation_search_results_df)

,source,search_term,matching_lines
0,Phase 0 report,innings length,2
1,Phase 0 report,legal ball,1
2,Phase 0 report,runs,3
3,Phase 1 report,extras1_runs,1
4,Phase 1 report,extras_2_runs,2
5,Phase 1 report,innings_ball_num,2
6,Phase 1 report,runs,7
7,Phase 2 report,extras1_runs,1
8,Phase 2 report,extras_2_runs,2
9,Phase 2 report,innings_ball_num,1


,source,search_term,line_number,matching_text,context
0,Phase 1 report,extras1_runs,47,"extras1_runs is an encoded category, not a numerical extras-run total: 1=Overstep, 2=wide, 3=legbyes, 4=byes, 5=other, 6=no extras, 7=Penalty.","8.1 Extras and runs extras1_runs is an encoded category, not a numerical extras-run total: 1=Overstep, 2=wide, 3=legbyes, 4=byes, 5=other, 6=no extras, 7=Penalty. extras_2_runs is non-zero in 35 primary regulation records. Thirty-four are associated with Overstep or other; one has no extras_type. Semantics remain unresolved."
1,Phase 2 report,extras1_runs,80,The source `extras1_runs` codes were mapped to descriptive evidence categories:,The source `extras1_runs` codes were mapped to descriptive evidence categories:
2,Phase 1 report,extras_2_runs,48,extras_2_runs is non-zero in 35 primary regulation records. Thirty-four are associated with Overstep or other; one has no extras_type. Semantics remain unresolved.,"extras1_runs is an encoded category, not a numerical extras-run total: 1=Overstep, 2=wide, 3=legbyes, 4=byes, 5=other, 6=no extras, 7=Penalty. extras_2_runs is non-zero in 35 primary regulation records. Thirty-four are associated with Overstep or other; one has no extras_type. Semantics remain unresolved. The runs field could not be confidently labelled as batter runs or total delivery runs from the evidence. Do not assume its meaning without further validation."
3,Phase 1 report,extras_2_runs,66,"1. Runs and extras_2_runs semantics remain unresolved. Metrics dependent on batter runs, total runs or extras need a defensible field interpretation.","10. Limitations and risks for later analysis 1. Runs and extras_2_runs semantics remain unresolved. Metrics dependent on batter runs, total runs or extras need a defensible field interpretation. 2. Legal-ball counts cannot be reliably inferred from the sequence fields alone. Bowling economy and balls-per-over denominators need validated legal-delivery classification."
4,Phase 2 report,extras_2_runs,98,The meaning of `extras_2_runs` remains unresolved. Non-zero values were documented in the exception register.,The meaning of `extras_2_runs` remains unresolved. Non-zero values were documented in the exception register.
5,Phase 2 report,extras_2_runs,222,4. The semantics of `extras_2_runs` remain unresolved.,3. The 147 unexplained extended overs cannot be fully explained using the available source fields. 4. The semantics of `extras_2_runs` remain unresolved. 5. Source boundary-flag mismatches were preserved rather than corrected.
6,Phase 0 report,innings length,155,- Scheduled innings length where relevant.,- Wickets remaining. - Scheduled innings length where relevant.
7,Phase 0 report,innings length,175,2. Expand to nearby scheduled innings lengths while preserving important contextual dimensions.,1. Use exact comparable states where available. 2. Expand to nearby scheduled innings lengths while preserving important contextual dimensions. 3. Relax wicket-state matching through adjacent states.
8,Phase 1 report,innings_ball_num,55,"No duplicate (match_id, innings, innings_ball_num) or (match_id, innings, overid, over_ball_num) keys were found.","8.3 Sequence and overs No duplicate (match_id, innings, innings_ball_num) or (match_id, innings, overid, over_ball_num) keys were found. innings_ball_num is continuous within every match-innings group. over_ball_num is a delivery sequence position, not a legal-ball counter."
9,Phase 1 report,innings_ball_num,56,"innings_ball_num is continuous within every match-innings group. over_ball_num is a delivery sequence position, not a legal-ball counter.","No duplicate (match_id, innings, innings_ball_num) or (match_id, innings, overid, over_ball_num) keys were found. innings_ball_num is continuous within every match-innings group. over_ball_num is a delivery sequence position, not a legal-ball counter. In the primary population, 6,450 overs have more than six delivery records; 147 of these have no recorded wide or O

### Final Candidate Field Inventory

In [74]:
# 1. Define the concepts for which alternative fields are needed.
candidate_concepts = {
    "scoring": ["run", "score", "total"],
    "legal_ball_progression": ["ball", "legal", "delivery"],
    "innings_length": ["over", "length", "scheduled", "revised"],
    "chase_target": ["target", "par", "revision"],
    "innings_context": ["innings", "resource", "wicket"]
}

# 2. Search column names across the loaded source and integrated datasets.
candidate_datasets = {
    "Raw ball-by-ball": df_source_ball,
    "Raw matches": df_source_matches,
    "Integrated deliveries": df_deliveries
}

candidate_field_rows = []

for dataset_name, dataset in candidate_datasets.items():
    for column in dataset.columns:
        column_lower = str(column).lower()

        matched_concepts = [
            concept
            for concept, keywords in candidate_concepts.items()
            if any(keyword in column_lower for keyword in keywords)
        ]

        if matched_concepts:
            candidate_field_rows.append(
                {
                    "dataset": dataset_name,
                    "column": column,
                    "matched_concepts": ", ".join(matched_concepts),
                    "dtype": str(dataset[column].dtype),
                    "non_missing_records": int(dataset[column].notna().sum()),
                    "distinct_non_missing_values": int(
                        dataset[column].nunique(dropna=True)
                    )
                }
            )

candidate_field_inventory = pd.DataFrame(candidate_field_rows)

# 3. Summarise candidate field coverage by dataset.
candidate_field_summary = (
    candidate_field_inventory
    .groupby("dataset", observed=True)
    .agg(
        candidate_field_count=("column", "size"),
        concepts_covered=("matched_concepts", "nunique")
    )
    .reset_index()
)

# 4. Display the inventory and coverage summary.
display(candidate_field_summary)
display(
    candidate_field_inventory.sort_values(
        ["dataset", "matched_concepts", "column"]
    ).reset_index(drop=True)
)

assert not candidate_field_inventory.empty

,dataset,candidate_field_count,concepts_covered
0,Integrated deliveries,17,7
1,Raw ball-by-ball,11,6
2,Raw matches,3,2


,dataset,column,matched_concepts,dtype,non_missing_records,distinct_non_missing_values
0,Integrated deliveries,innings,innings_context,int64,192235,2
1,Integrated deliveries,is_wicket,innings_context,bool,192235,2
2,Integrated deliveries,first_inn_revised_overs,innings_length,float64,2156,11
3,Integrated deliveries,over_or_around,innings_length,object,191839,2
4,Integrated deliveries,overid,innings_length,int64,192235,20
5,Integrated deliveries,second_inn_revised_overs,innings_length,float64,3627,10
6,Integrated deliveries,revised_target,"innings_length, chase_target",float64,3627,22
7,Integrated deliveries,delivery_ist_time,legal_ball_progression,object,192235,192070
8,Integrated deliveries,delivery_style,legal_ball_progression,object,191859,34
9,Integrated deliveries,legal_ball_evidence_class,legal_ball_progression,object,192235,7


### Over Structure and Delivery Position Semantics Audit

In [75]:
# 1. Build a separate over-structure audit view.
over_structure = df_deliveries[
    [
        "match_id",
        "innings",
        "innings_ball_num",
        "overid",
        "over_ball_num",
        "over_or_around",
        "legal_ball_evidence_class"
    ]
].copy()

# 2. Summarise over_or_around values.
over_or_around_summary = (
    over_structure
    .groupby("over_or_around", dropna=False, observed=True)
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique"),
        innings_count=(
            ["match_id", "innings"],
            lambda x: x.nunique()
        ) if False else ("innings_ball_num", "count")
    )
    .reset_index()
)

# 3. Summarise delivery positions within each over.
over_position_summary = (
    over_structure
    .groupby(
        ["overid", "over_ball_num"],
        dropna=False,
        observed=True
    )
    .agg(
        delivery_records=("match_id", "size"),
        innings_count=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(["overid", "over_ball_num"])
)

# 4. Summarise each innings-over combination.
innings_over_summary = (
    over_structure
    .groupby(
        ["match_id", "innings", "overid"],
        observed=True
    )
    .agg(
        delivery_records=("innings_ball_num", "size"),
        minimum_sequence=("innings_ball_num", "min"),
        maximum_sequence=("innings_ball_num", "max"),
        minimum_over_ball_num=("over_ball_num", "min"),
        maximum_over_ball_num=("over_ball_num", "max"),
        distinct_over_ball_positions=("over_ball_num", "nunique")
    )
    .reset_index()
)

innings_over_summary["more_than_six_records"] = (
    innings_over_summary["delivery_records"] > 6
)

# 5. Summarise extended-over patterns.
extended_over_summary = (
    innings_over_summary
    .groupby("more_than_six_records", observed=True)
    .agg(
        innings_over_count=("overid", "size"),
        total_delivery_records=("delivery_records", "sum"),
        affected_matches=("match_id", "nunique")
    )
    .reset_index()
)

# 6. Compare sequence and over-position structure.
over_structure_validation = pd.DataFrame(
    [
        {
            "check": "Maximum overid",
            "observed": over_structure["overid"].max(),
            "interpretation": "Observed maximum over identifier; not a legal-ball count"
        },
        {
            "check": "Maximum over_ball_num",
            "observed": over_structure["over_ball_num"].max(),
            "interpretation": "Observed maximum delivery position within an over"
        },
        {
            "check": "Distinct over_ball_num values",
            "observed": over_structure["over_ball_num"].nunique(),
            "interpretation": "Number of distinct delivery-position values"
        },
        {
            "check": "Innings-over groups with more than six records",
            "observed": int(
                innings_over_summary["more_than_six_records"].sum()
            ),
            "interpretation": "Includes recorded deliveries, not verified legal balls"
        },
        {
            "check": "Innings-over groups with duplicate over_ball_num positions",
            "observed": int(
                (
                    over_structure
                    .groupby(
                        ["match_id", "innings", "overid", "over_ball_num"],
                        observed=True
                    )
                    .size()
                    .gt(1)
                ).sum()
            ),
            "interpretation": "Tests uniqueness of the within-over delivery position"
        }
    ]
)

# 7. Display readable audit tables.
display(over_or_around_summary)
display(over_position_summary)
display(extended_over_summary)
display(over_structure_validation)

# 8. Export audit outputs.
over_or_around_summary.to_csv(
    OUTPUT_DIR / "phase3B_over_or_around_summary.csv",
    index=False
)

over_position_summary.to_csv(
    OUTPUT_DIR / "phase3B_over_delivery_position_summary.csv",
    index=False
)

innings_over_summary.to_csv(
    OUTPUT_DIR / "phase3B_innings_over_structure.csv",
    index=False
)

extended_over_summary.to_csv(
    OUTPUT_DIR / "phase3B_extended_over_structure_summary.csv",
    index=False
)

,over_or_around,delivery_records,matches,innings_count
0,Around,43219,811,43219
1,Over,148620,813,148620
2,NaN,396,126,396


,overid,over_ball_num,delivery_records,innings_count
0,1,1,1626,813
1,1,2,1626,813
2,1,3,1626,813
3,1,4,1626,813
4,1,5,1626,813
...,...,...,...,...
208,20,7,345,321
209,20,8,93,92
210,20,9,21,21
211,20,10,6,6


,more_than_six_records,innings_over_count,total_delivery_records,affected_matches
0,False,24471,145414,813
1,True,6450,46821,810


,check,observed,interpretation
0,Maximum overid,20,Observed maximum over identifier; not a legal-ball count
1,Maximum over_ball_num,13,Observed maximum delivery position within an over
2,Distinct over_ball_num values,13,Number of distinct delivery-position values
3,Innings-over groups with more than six records,6450,"Includes recorded deliveries, not verified legal balls"
4,Innings-over groups with duplicate over_ball_num positions,0,Tests uniqueness of the within-over delivery position


### over_or_around Classification and Sequence Relationship Audit

In [76]:
# 1. Count records, matches and distinct innings for each label.
over_label_records = (
    over_structure
    .groupby("over_or_around", dropna=False, observed=True)
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique")
    )
)

over_label_innings = (
    over_structure[
        ["match_id", "innings", "over_or_around"]
    ]
    .drop_duplicates()
    .groupby("over_or_around", dropna=False, observed=True)
    .size()
    .rename("distinct_innings")
)

over_label_summary = (
    over_label_records
    .join(over_label_innings)
    .reset_index()
)

# 2. Examine how labels are distributed across over identifiers.
over_label_by_over = (
    over_structure
    .groupby(
        ["over_or_around", "overid"],
        dropna=False,
        observed=True
    )
    .agg(
        delivery_records=("match_id", "size"),
        distinct_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(["overid", "over_or_around"], na_position="last")
)

# 3. Examine delivery-position distribution by label.
over_label_by_position = (
    over_structure
    .groupby(
        ["over_or_around", "over_ball_num"],
        dropna=False,
        observed=True
    )
    .agg(
        delivery_records=("match_id", "size"),
        distinct_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["over_or_around", "over_ball_num"],
        na_position="last"
    )
)

# 4. Summarise the proportion of records by label.
total_over_records = len(over_structure)

over_label_summary["record_share_pct"] = (
    over_label_summary["delivery_records"]
    / total_over_records * 100
)

# 5. Display the corrected audit tables.
display(over_label_summary)
display(over_label_by_over)
display(over_label_by_position)

# 6. Export the audit outputs.
over_label_summary.to_csv(
    OUTPUT_DIR / "phase3B_over_or_around_corrected_summary.csv",
    index=False
)

over_label_by_over.to_csv(
    OUTPUT_DIR / "phase3B_over_or_around_by_over.csv",
    index=False
)

over_label_by_position.to_csv(
    OUTPUT_DIR / "phase3B_over_or_around_by_position.csv",
    index=False
)

assert over_label_summary["delivery_records"].sum() == len(over_structure)
assert over_label_summary["distinct_innings"].sum() >= 0

,over_or_around,delivery_records,matches,distinct_innings,record_share_pct
0,Around,43219,811,1584,22.482378
1,Over,148620,813,1626,77.311624
2,NaN,396,126,161,0.205998


,over_or_around,overid,delivery_records,distinct_matches
0,Around,1,1384,286
20,Over,1,8818,805
40,NaN,1,44,12
1,Around,2,1243,286
21,Over,2,8981,807
41,NaN,2,34,20
2,Around,3,1533,344
22,Over,3,8567,803
42,NaN,3,33,26
3,Around,4,1863,400


,over_or_around,over_ball_num,delivery_records,distinct_matches
0,Around,1,6742,806
1,Around,2,6850,809
2,Around,3,6869,805
3,Around,4,6958,807
4,Around,5,7083,807
5,Around,6,7165,808
6,Around,7,1263,616
7,Around,8,227,182
8,Around,9,50,47
9,Around,10,9,9


### Over Annotation vs Legal-Ball Evidence Cross-Audit

In [77]:
# 1. Create a cross-audit dataset without modifying source data.
over_legal_crosswalk = df_deliveries[
    [
        "match_id",
        "innings",
        "innings_ball_num",
        "overid",
        "over_ball_num",
        "over_or_around",
        "legal_ball_evidence_class",
        "legal_ball_evidence_status"
    ]
].copy()

# 2. Count records by over annotation and legal-ball evidence class.
over_legal_class_counts = (
    over_legal_crosswalk
    .groupby(
        ["over_or_around", "legal_ball_evidence_class"],
        dropna=False,
        observed=True
    )
    .agg(
        delivery_records=("match_id", "size"),
        distinct_matches=("match_id", "nunique")
    )
    .reset_index()
)

# 3. Calculate the within-annotation distribution.
over_legal_class_counts["share_within_annotation_pct"] = (
    over_legal_class_counts["delivery_records"]
    / over_legal_class_counts.groupby(
        "over_or_around",
        dropna=False,
        observed=True
    )["delivery_records"].transform("sum")
    * 100
)

# 4. Examine the detailed cross-tabulation by delivery position.
over_position_legal_counts = (
    over_legal_crosswalk
    .groupby(
        [
            "over_or_around",
            "over_ball_num",
            "legal_ball_evidence_class"
        ],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(name="delivery_records")
)

# 5. Summarise legal-ball evidence status by annotation.
over_legal_status_summary = (
    over_legal_crosswalk
    .groupby(
        ["over_or_around", "legal_ball_evidence_status"],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(name="delivery_records")
)

over_legal_status_summary["share_within_annotation_pct"] = (
    over_legal_status_summary["delivery_records"]
    / over_legal_status_summary.groupby(
        "over_or_around",
        dropna=False,
        observed=True
    )["delivery_records"].transform("sum")
    * 100
)

# 6. Display the audit tables.
display(over_legal_class_counts)
display(over_legal_status_summary)
display(over_position_legal_counts)

# 7. Export audit outputs.
over_legal_class_counts.to_csv(
    OUTPUT_DIR / "phase3B_over_annotation_legal_class_crosswalk.csv",
    index=False
)

over_legal_status_summary.to_csv(
    OUTPUT_DIR / "phase3B_over_annotation_legal_status_summary.csv",
    index=False
)

over_position_legal_counts.to_csv(
    OUTPUT_DIR / "phase3B_over_position_legal_class_detail.csv",
    index=False
)

# 8. Confirm that the cross-audit retains the full integrated population.
assert len(over_legal_crosswalk) == 192235
assert over_legal_class_counts["delivery_records"].sum() == 192235
assert over_legal_status_summary["delivery_records"].sum() == 192235

,over_or_around,legal_ball_evidence_class,delivery_records,distinct_matches,share_within_annotation_pct
0,Around,RECORDED_BYES,116,106,0.268400
1,Around,RECORDED_LEGBYES,492,340,1.138388
2,Around,RECORDED_NO_EXTRAS,41173,811,95.265971
3,Around,RECORDED_OTHER,38,35,0.087924
4,Around,RECORDED_OVERSTEP,102,89,0.236007
5,Around,RECORDED_PENALTY,1,1,0.002314
6,Around,RECORDED_WIDE,1297,582,3.000995
7,Over,RECORDED_BYES,386,308,0.259723
8,Over,RECORDED_LEGBYES,2136,742,1.437222
9,Over,RECORDED_NO_EXTRAS,139331,813,93.749832


,over_or_around,legal_ball_evidence_status,delivery_records,share_within_annotation_pct
0,Around,NO_RECORDED_WIDE_OR_OVERSTEP,41781,96.672760
1,Around,RECORDED_NON_LEGAL_EVIDENCE,1399,3.237002
2,Around,UNRESOLVED,39,0.090238
3,Over,NO_RECORDED_WIDE_OR_OVERSTEP,141853,95.446777
4,Over,RECORDED_NON_LEGAL_EVIDENCE,6549,4.406540
5,Over,UNRESOLVED,218,0.146683
6,NaN,NO_RECORDED_WIDE_OR_OVERSTEP,368,92.929293
7,NaN,RECORDED_NON_LEGAL_EVIDENCE,28,7.070707


,over_or_around,over_ball_num,legal_ball_evidence_class,delivery_records
0,Around,1,RECORDED_BYES,16
1,Around,1,RECORDED_LEGBYES,68
2,Around,1,RECORDED_NO_EXTRAS,6452
3,Around,1,RECORDED_OTHER,7
4,Around,1,RECORDED_OVERSTEP,10
...,...,...,...,...
125,NaN,6,RECORDED_WIDE,2
126,NaN,7,RECORDED_NO_EXTRAS,5
127,NaN,8,RECORDED_NO_EXTRAS,1
128,NaN,8,RECORDED_WIDE,1


## Consolidated Evidence and Exception Closure

In [78]:
# 1. Consolidate Phase 3B findings into a closure register.
phase3B_closure_register = pd.DataFrame([
    {
        "area": "Competition population",
        "evidence": "362 IPL matches across 5 competitions; 451 T20I matches across 131 competitions",
        "decision": "USABLE",
        "limitation": "Use the audited IPL competition IDs and separate T20I population"
    },
    {
        "area": "Innings and team structure",
        "evidence": "813 matches; exactly two innings per match; team-role reversal validated",
        "decision": "USABLE",
        "limitation": "Structural validity does not establish scoring semantics"
    },
    {
        "area": "Pre-delivery wickets fallen",
        "evidence": "192,235 reconstructed delivery states; 1,626 innings; all progression checks passed",
        "decision": "CONDITIONAL",
        "limitation": "Based on supplied is_wicket flags; dismissal legality not independently certified"
    },
    {
        "area": "Dismissal event identity",
        "evidence": "10,392 wicket events; one missing dismissed-player ID; 11 non-wicket records with dismissed-player IDs",
        "decision": "CONDITIONAL",
        "limitation": "Preserve the 12 identity exceptions; do not infer or correct identities"
    },
    {
        "area": "Cumulative scoring",
        "evidence": "runs field audited; extras1_runs is categorical; extras_2_runs semantics unresolved",
        "decision": "UNAVAILABLE",
        "limitation": "Cannot establish authoritative cumulative innings scores"
    },
    {
        "area": "Legal-ball progression",
        "evidence": "184,002 records without recorded wide/overstep; 7,976 non-legal evidence records; 257 unresolved",
        "decision": "UNAVAILABLE",
        "limitation": "No verified legal-ball counter; delivery sequence is not a legal-ball count"
    },
    {
        "area": "Over structure",
        "evidence": "overid, over_ball_num and over_or_around audited; 6,450 innings-over groups have more than six records",
        "decision": "CONDITIONAL",
        "limitation": "Descriptive delivery-order metadata only; cannot calculate legal balls remaining"
    },
    {
        "area": "Scheduled innings length",
        "evidence": "Revised-overs fields are sparse and their units remain unconfirmed",
        "decision": "UNAVAILABLE",
        "limitation": "Cannot establish scheduled legal-ball allocation across the population"
    },
    {
        "area": "First-innings scoring position",
        "evidence": "Requires verified cumulative score and comparable progression",
        "decision": "UNAVAILABLE",
        "limitation": "Cannot construct defensible scoring deviation benchmarks"
    },
    {
        "area": "Second-innings scoring demand",
        "evidence": "Revised targets exist for only 22 integrated matches; scoring reconstruction is unresolved",
        "decision": "UNAVAILABLE",
        "limitation": "Cannot reconstruct required run rate consistently"
    },
    {
        "area": "Historical pressure benchmarks",
        "evidence": "Competition populations and wicket state available; scoring and legal-ball inputs unavailable",
        "decision": "UNAVAILABLE",
        "limitation": "Historical expected-scoring benchmarks cannot yet be constructed defensibly"
    }
])

# 2. Summarise closure decisions.
phase3B_closure_summary = (
    phase3B_closure_register
    .groupby("decision", observed=True)
    .size()
    .reset_index(name="area_count")
)

# 3. Export closure artifacts.
phase3B_closure_register.to_csv(
    OUTPUT_DIR / "phase3B_consolidated_closure_register.csv",
    index=False
)

phase3B_closure_summary.to_csv(
    OUTPUT_DIR / "phase3B_consolidated_closure_summary.csv",
    index=False
)

# 4. Display the closure tables.
display(phase3B_closure_summary)
display(phase3B_closure_register)

# 5. Validate coverage and classification.
assert len(phase3B_closure_register) == 11
assert phase3B_closure_register["area"].is_unique
assert phase3B_closure_register["decision"].isin(
    ["USABLE", "CONDITIONAL", "UNAVAILABLE"]
).all()

,decision,area_count
0,CONDITIONAL,3
1,UNAVAILABLE,6
2,USABLE,2


,area,evidence,decision,limitation
0,Competition population,362 IPL matches across 5 competitions; 451 T20I matches across 131 competitions,USABLE,Use the audited IPL competition IDs and separate T20I population
1,Innings and team structure,813 matches; exactly two innings per match; team-role reversal validated,USABLE,Structural validity does not establish scoring semantics
2,Pre-delivery wickets fallen,"192,235 reconstructed delivery states; 1,626 innings; all progression checks passed",CONDITIONAL,Based on supplied is_wicket flags; dismissal legality not independently certified
3,Dismissal event identity,"10,392 wicket events; one missing dismissed-player ID; 11 non-wicket records with dismissed-player IDs",CONDITIONAL,Preserve the 12 identity exceptions; do not infer or correct identities
4,Cumulative scoring,runs field audited; extras1_runs is categorical; extras_2_runs semantics unresolved,UNAVAILABLE,Cannot establish authoritative cumulative innings scores
5,Legal-ball progression,"184,002 records without recorded wide/overstep; 7,976 non-legal evidence records; 257 unresolved",UNAVAILABLE,No verified legal-ball counter; delivery sequence is not a legal-ball count
6,Over structure,"overid, over_ball_num and over_or_around audited; 6,450 innings-over groups have more than six records",CONDITIONAL,Descriptive delivery-order metadata only; cannot calculate legal balls remaining
7,Scheduled innings length,Revised-overs fields are sparse and their units remain unconfirmed,UNAVAILABLE,Cannot establish scheduled legal-ball allocation across the population
8,First-innings scoring position,Requires verified cumulative score and comparable progression,UNAVAILABLE,Cannot construct defensible scoring deviation benchmarks
9,Second-innings scoring demand,Revised targets exist for only 22 integrated matches; scoring reconstruction is unresolved,UNAVAILABLE,Cannot reconstruct required run rate consistently


## Phase 3B Final Report and Closure

In [79]:
# 1. Define the Phase 3B report content.
phase3B_final_report = f"""
# SupeRR Selector Season 2
## Phase 3B — Data Audit and Reconstruction Feasibility Report

### 1. Objective

Audit the available source and integrated datasets to establish whether
pre-delivery match-state variables can be reconstructed reliably for
historical pressure modelling.

### 2. Population

- Integrated matches: 813
- IPL matches: 362 across 5 competitions
- T20I matches: 451 across 131 competitions
- Regulation delivery records: 192,235
- Innings audited: 1,626

IPL and T20I populations remain separate. The two out-of-scope T20
competitions identified in Phase 3A are excluded.

### 3. Validated Components

- Competition population mapping and separation.
- Innings and batting/bowling team structure.
- Pre-delivery wicket-state arithmetic across 192,235 deliveries.
- Wicket-event totals reconcile to 10,392.
- Wicket-state progression checks passed across 1,626 innings.
- Delivery sequence and over-position structures audited.

### 4. Conditional Components

- Wicket state relies on the supplied is_wicket field.
- One wicket record lacks dismissed_player_id.
- Eleven non-wicket records contain dismissed_player_id.
- Over-related fields are descriptive sequence metadata only.

### 5. Unavailable Components

- Authoritative cumulative innings scoring.
- Verified legal-ball progression.
- Verified scheduled innings length.
- First-innings scoring-position reconstruction.
- Second-innings scoring-demand reconstruction.
- Historical expected-scoring pressure benchmarks.

### 6. Key Data Limitations

- runs cannot confidently be labelled batter runs or total delivery runs.
- extras1_runs is an encoded category, not a numerical extras total.
- extras_2_runs semantics remain unresolved.
- innings_ball_num is a delivery-record sequence, not a legal-ball counter.
- 7,976 deliveries have non-legal evidence and 257 remain unresolved.
- Revised-overs fields are sparse and their units are unconfirmed.
- Revised targets are available in only 22 integrated matches.
- 6,450 innings-over groups contain more than six delivery records.
- over_or_around does not establish legal-ball progression.

### 7. Final Feasibility Classification

{phase3B_closure_summary.to_string(index=False)}

### 8. Methodological Decision

Do not construct first-innings scoring deviation or second-innings
required-rate pressure benchmarks from the current evidence.

Do not infer legal balls remaining from innings_ball_num, overid,
over_ball_num, over_or_around, or timestamps.

Retain pre-delivery wickets fallen as a conditional contextual variable.
Preserve source values and unresolved exceptions.

### 9. Phase 3B Outcome

The data audit and reconstruction-feasibility assessment are complete.
The inability to reconstruct certain variables is a documented
methodological finding, not permission to substitute unverified values.

Phase 3B status: TECHNICALLY COMPLETE.
Formal project approval remains separate.
"""

# 2. Save the final report.
phase3B_report_path = OUTPUT_DIR / "SupeRR_Selector_S2_Phase_3B_Report.md"

phase3B_report_path.write_text(
    phase3B_final_report,
    encoding="utf-8"
)

# 3. Create the completion summary.
phase3B_completion_summary = pd.DataFrame([
    {
        "phase": "3B",
        "workstream": "Data Audit and Reconstruction Feasibility",
        "status": "TECHNICALLY COMPLETE",
        "audit_areas": len(phase3B_closure_register),
        "usable_areas": int(
            (phase3B_closure_register["decision"] == "USABLE").sum()
        ),
        "conditional_areas": int(
            (phase3B_closure_register["decision"] == "CONDITIONAL").sum()
        ),
        "unavailable_areas": int(
            (phase3B_closure_register["decision"] == "UNAVAILABLE").sum()
        ),
        "report": phase3B_report_path.name
    }
])

phase3B_completion_summary.to_csv(
    OUTPUT_DIR / "phase3B_completion_summary.csv",
    index=False
)

# 4. Final validation.
phase3B_final_validation = pd.DataFrame([
    {
        "check": "All 11 feasibility areas classified",
        "passed": len(phase3B_closure_register) == 11
    },
    {
        "check": "All decision categories valid",
        "passed": phase3B_closure_register["decision"].isin(
            ["USABLE", "CONDITIONAL", "UNAVAILABLE"]
        ).all()
    },
    {
        "check": "Final report created",
        "passed": phase3B_report_path.exists()
    },
    {
        "check": "Completion summary created",
        "passed": (
            OUTPUT_DIR / "phase3B_completion_summary.csv"
        ).exists()
    }
])

display(phase3B_completion_summary)
display(phase3B_final_validation)

assert phase3B_final_validation["passed"].all()

,phase,workstream,status,audit_areas,usable_areas,conditional_areas,unavailable_areas,report
0,3B,Data Audit and Reconstruction Feasibility,TECHNICALLY COMPLETE,11,2,3,6,SupeRR_Selector_S2_Phase_3B_Report.md


,check,passed
0,All 11 feasibility areas classified,True
1,All decision categories valid,True
2,Final report created,True
3,Completion summary created,True


# Pre-delivery State Specification

### Define State Specification Boundaries

In [80]:
phase3C_state_boundaries = pd.DataFrame([
    {
        "state_element": "Match and delivery identifiers",
        "intended_role": "Identify each delivery and its position in the match",
        "evidence_status": "USABLE",
        "phase3C_treatment": "Retain validated match and delivery keys",
        "restriction": "Do not alter source identifiers"
    },
    {
        "state_element": "Competition format",
        "intended_role": "Separate IPL/T20 and T20I populations",
        "evidence_status": "USABLE",
        "phase3C_treatment": "Retain validated competition classification",
        "restriction": "Never pool IPL and T20I benchmarks"
    },
    {
        "state_element": "Innings and batting/bowling teams",
        "intended_role": "Identify innings context and team roles",
        "evidence_status": "USABLE",
        "phase3C_treatment": "Retain validated innings and team structure",
        "restriction": "Preserve the established team-role mapping"
    },
    {
        "state_element": "Pre-delivery wickets fallen",
        "intended_role": "Represent wickets lost before the current delivery",
        "evidence_status": "CONDITIONAL",
        "phase3C_treatment": "Retain the validated pre-delivery wicket state",
        "restriction": "Depends on supplied is_wicket field and documented exceptions"
    },
    {
        "state_element": "Cumulative innings score",
        "intended_role": "Represent runs scored before the current delivery",
        "evidence_status": "UNAVAILABLE",
        "phase3C_treatment": "Do not construct or populate",
        "restriction": "runs and extras semantics are unresolved"
    },
    {
        "state_element": "Legal balls remaining",
        "intended_role": "Represent remaining legal deliveries in the innings",
        "evidence_status": "UNAVAILABLE",
        "phase3C_treatment": "Do not infer or populate",
        "restriction": "Legal-ball progression is not fully established"
    },
    {
        "state_element": "Scheduled innings length",
        "intended_role": "Establish the innings delivery allocation",
        "evidence_status": "UNAVAILABLE",
        "phase3C_treatment": "Do not infer or populate",
        "restriction": "Revised-overs metadata is sparse and units are unconfirmed"
    },
    {
        "state_element": "Innings stage",
        "intended_role": "Classify powerplay, middle and death stages",
        "evidence_status": "UNAVAILABLE",
        "phase3C_treatment": "Do not assign stage labels",
        "restriction": "Requires defensible scheduled length and progression"
    },
    {
        "state_element": "First-innings scoring position",
        "intended_role": "Compare cumulative scoring with expected scoring",
        "evidence_status": "UNAVAILABLE",
        "phase3C_treatment": "Defer to Phase 3D feasibility assessment",
        "restriction": "Requires reliable cumulative score and benchmark inputs"
    },
    {
        "state_element": "Second-innings scoring demand",
        "intended_role": "Represent chase demand relative to expected scoring capability",
        "evidence_status": "UNAVAILABLE",
        "phase3C_treatment": "Defer to Phase 3E feasibility assessment",
        "restriction": "Target, runs required and legal balls remaining are not reliably reconstructable"
    },
    {
        "state_element": "Super Over state",
        "intended_role": "Represent tie-break innings context",
        "evidence_status": "OUT OF PRIMARY SCOPE",
        "phase3C_treatment": "Keep separate from regulation state specification",
        "restriction": "26 Super Over deliveries remain separately retained"
    }
])

# Validate the boundary register.
assert phase3C_state_boundaries["state_element"].is_unique
assert phase3C_state_boundaries["state_element"].notna().all()
assert phase3C_state_boundaries["evidence_status"].isin([
    "USABLE",
    "CONDITIONAL",
    "UNAVAILABLE",
    "OUT OF PRIMARY SCOPE"
]).all()

# Save the audit artifact.
phase3C_state_boundaries_path = (
    OUTPUT_DIR / "phase3C_state_specification_boundaries.csv"
)

phase3C_state_boundaries.to_csv(
    phase3C_state_boundaries_path,
    index=False
)

# Display readable validation summaries.
display(phase3C_state_boundaries)

phase3C_boundary_summary = (
    phase3C_state_boundaries
    .groupby("evidence_status", as_index=False)
    .agg(state_elements=("state_element", "count"))
    .sort_values("evidence_status")
)

display(phase3C_boundary_summary)

phase3C_boundary_validation = pd.DataFrame([
    {
        "check": "State elements are unique",
        "passed": phase3C_state_boundaries["state_element"].is_unique
    },
    {
        "check": "Every state element has an evidence classification",
        "passed": phase3C_state_boundaries["evidence_status"].notna().all()
    },
    {
        "check": "No unsupported evidence classifications",
        "passed": phase3C_state_boundaries["evidence_status"].isin([
            "USABLE",
            "CONDITIONAL",
            "UNAVAILABLE",
            "OUT OF PRIMARY SCOPE"
        ]).all()
    },
    {
        "check": "Boundary register exported",
        "passed": phase3C_state_boundaries_path.exists()
    }
])

display(phase3C_boundary_validation)

assert phase3C_boundary_validation["passed"].all()

,state_element,intended_role,evidence_status,phase3C_treatment,restriction
0,Match and delivery identifiers,Identify each delivery and its position in the match,USABLE,Retain validated match and delivery keys,Do not alter source identifiers
1,Competition format,Separate IPL/T20 and T20I populations,USABLE,Retain validated competition classification,Never pool IPL and T20I benchmarks
2,Innings and batting/bowling teams,Identify innings context and team roles,USABLE,Retain validated innings and team structure,Preserve the established team-role mapping
3,Pre-delivery wickets fallen,Represent wickets lost before the current delivery,CONDITIONAL,Retain the validated pre-delivery wicket state,Depends on supplied is_wicket field and documented exceptions
4,Cumulative innings score,Represent runs scored before the current delivery,UNAVAILABLE,Do not construct or populate,runs and extras semantics are unresolved
5,Legal balls remaining,Represent remaining legal deliveries in the innings,UNAVAILABLE,Do not infer or populate,Legal-ball progression is not fully established
6,Scheduled innings length,Establish the innings delivery allocation,UNAVAILABLE,Do not infer or populate,Revised-overs metadata is sparse and units are unconfirmed
7,Innings stage,"Classify powerplay, middle and death stages",UNAVAILABLE,Do not assign stage labels,Requires defensible scheduled length and progression
8,First-innings scoring position,Compare cumulative scoring with expected scoring,UNAVAILABLE,Defer to Phase 3D feasibility assessment,Requires reliable cumulative score and benchmark inputs
9,Second-innings scoring demand,Represent chase demand relative to expected scoring capability,UNAVAILABLE,Defer to Phase 3E feasibility assessment,"Target, runs required and legal balls remaining are not reliably reconstructable"


,evidence_status,state_elements
0,CONDITIONAL,1
1,OUT OF PRIMARY SCOPE,1
2,UNAVAILABLE,6
3,USABLE,3


,check,passed
0,State elements are unique,True
1,Every state element has an evidence classification,True
2,No unsupported evidence classifications,True
3,Boundary register exported,True


### Source-Field Mapping and Availability

In [81]:
phase3C_candidate_fields = {
    "Match identifier": ["match_id"],
    "Innings number": ["innings"],
    "Competition identity": ["competition_id", "competition_name"],
    "Competition format": [
        "competition_format", "format", "competition_type"
    ],
    "Season": ["season"],
    "Batting team": ["batting_team_id", "batting_team_name"],
    "Bowling team": ["bowling_team_id", "bowling_team_name"],
    "Delivery sequence": ["innings_ball_num"],
    "Over sequence": ["overid", "over_ball_num"],
    "Wicket event": ["is_wicket"],
    "Pre-delivery wickets": ["pre_delivery_wickets_fallen"],
    "Post-delivery wickets": ["post_delivery_wickets_fallen"],
    "Legal-ball evidence": ["legal_ball_evidence"],
    "Scoring field": ["runs"],
    "Revised innings length": [
        "first_inn_revised_overs",
        "second_inn_revised_overs"
    ],
    "Revised target": ["revised_target"]
}

phase3C_dataset_inventory = []

phase3C_datasets_to_inspect = {
    "Integrated deliveries": df_deliveries,
    "Pre-delivery wicket states": phase3_pre_delivery_wicket_states
}

for dataset_name, dataset in phase3C_datasets_to_inspect.items():
    for state_element, candidate_names in phase3C_candidate_fields.items():
        for field_name in candidate_names:
            if field_name in dataset.columns:
                phase3C_dataset_inventory.append({
                    "dataset": dataset_name,
                    "state_element": state_element,
                    "source_field": field_name,
                    "field_present": True,
                    "data_type": str(dataset[field_name].dtype),
                    "missing_values": int(dataset[field_name].isna().sum()),
                    "row_count": len(dataset)
                })

phase3C_field_inventory = pd.DataFrame(
    phase3C_dataset_inventory
)

assert not phase3C_field_inventory.empty
assert phase3C_field_inventory[
    ["dataset", "source_field"]
].duplicated().sum() == 0

# Display the exact fields found.
display(
    phase3C_field_inventory.sort_values(
        ["state_element", "dataset", "source_field"]
    ).reset_index(drop=True)
)

# Summarize field availability by dataset.
phase3C_field_summary = (
    phase3C_field_inventory
    .groupby("dataset", as_index=False)
    .agg(
        available_fields=("source_field", "count"),
        fields_with_missing_values=(
            "missing_values",
            lambda x: int((x > 0).sum())
        )
    )
)

display(phase3C_field_summary)

# Export the source-field inventory.
phase3C_field_inventory_path = (
    OUTPUT_DIR / "phase3C_source_field_inventory.csv"
)

phase3C_field_inventory.to_csv(
    phase3C_field_inventory_path,
    index=False
)

phase3C_field_validation = pd.DataFrame([
    {
        "check": "Integrated delivery dataset inspected",
        "passed": "Integrated deliveries" in phase3C_field_inventory["dataset"].values
    },
    {
        "check": "Pre-delivery state dataset inspected",
        "passed": "Pre-delivery wicket states" in phase3C_field_inventory["dataset"].values
    },
    {
        "check": "No duplicate dataset-field entries",
        "passed": not phase3C_field_inventory[
            ["dataset", "source_field"]
        ].duplicated().any()
    },
    {
        "check": "Source-field inventory exported",
        "passed": phase3C_field_inventory_path.exists()
    }
])

display(phase3C_field_validation)

assert phase3C_field_validation["passed"].all()

,dataset,state_element,source_field,field_present,data_type,missing_values,row_count
0,Integrated deliveries,Batting team,batting_team_id,True,int64,0,192235
1,Integrated deliveries,Batting team,batting_team_name,True,object,0,192235
2,Pre-delivery wicket states,Batting team,batting_team_id,True,int64,0,192235
3,Pre-delivery wicket states,Batting team,batting_team_name,True,object,0,192235
4,Integrated deliveries,Bowling team,bowling_team_id,True,int64,0,192235
5,Integrated deliveries,Bowling team,bowling_team_name,True,object,0,192235
6,Pre-delivery wicket states,Bowling team,bowling_team_id,True,int64,0,192235
7,Pre-delivery wicket states,Bowling team,bowling_team_name,True,object,0,192235
8,Integrated deliveries,Delivery sequence,innings_ball_num,True,int64,0,192235
9,Pre-delivery wicket states,Delivery sequence,innings_ball_num,True,int64,0,192235


,dataset,available_fields,fields_with_missing_values
0,Integrated deliveries,15,3
1,Pre-delivery wicket states,17,3


,check,passed
0,Integrated delivery dataset inspected,True
1,Pre-delivery state dataset inspected,True
2,No duplicate dataset-field entries,True
3,Source-field inventory exported,True


In [82]:
phase3C_schema_rows = []

for dataset_name, dataset in {
    "Integrated deliveries": df_deliveries,
    "Pre-delivery wicket states": phase3_pre_delivery_wicket_states
}.items():

    for position, column_name in enumerate(dataset.columns):
        phase3C_schema_rows.append({
            "dataset": dataset_name,
            "column_position": position,
            "column_name": column_name,
            "data_type": str(dataset[column_name].dtype),
            "missing_values": int(dataset[column_name].isna().sum()),
            "row_count": len(dataset)
        })

phase3C_full_schema = pd.DataFrame(phase3C_schema_rows)

display(
    phase3C_full_schema.sort_values(
        ["dataset", "column_position"]
    ).reset_index(drop=True)
)

# Highlight columns potentially relevant to the state specification.
phase3C_relevant_schema = phase3C_full_schema[
    phase3C_full_schema["column_name"].str.contains(
        "comp|format|legal|ball|over|target|run|wicket|innings|match|team|season",
        case=False,
        regex=True
    )
].copy()

display(phase3C_relevant_schema.reset_index(drop=True))

# Verify that the two schemas have been inspected.
phase3C_schema_validation = pd.DataFrame([
    {
        "check": "Integrated dataset has 61 columns",
        "passed": len(df_deliveries.columns) == 61
    },
    {
        "check": "Pre-delivery dataset has columns",
        "passed": len(phase3_pre_delivery_wicket_states.columns) > 0
    },
    {
        "check": "Schema inventory covers every column",
        "passed": len(phase3C_full_schema) == (
            len(df_deliveries.columns)
            + len(phase3_pre_delivery_wicket_states.columns)
        )
    }
])

display(phase3C_schema_validation)

assert phase3C_schema_validation["passed"].all()

,dataset,column_position,column_name,data_type,missing_values,row_count
0,Integrated deliveries,0,match_id,int64,0,192235
1,Integrated deliveries,1,innings,int64,0,192235
2,Integrated deliveries,2,overid,int64,0,192235
3,Integrated deliveries,3,over_ball_num,int64,0,192235
4,Integrated deliveries,4,innings_ball_num,int64,0,192235
...,...,...,...,...,...,...
121,Pre-delivery wicket states,60,bowling_team_name,object,0,192235
122,Pre-delivery wicket states,61,pre_delivery_wickets_fallen,int64,0,192235
123,Pre-delivery wicket states,62,post_delivery_wickets_fallen,int64,0,192235
124,Pre-delivery wicket states,63,benchmark_population,object,0,192235


,dataset,column_position,column_name,data_type,missing_values,row_count
0,Integrated deliveries,0,match_id,int64,0,192235
1,Integrated deliveries,1,innings,int64,0,192235
2,Integrated deliveries,2,overid,int64,0,192235
3,Integrated deliveries,3,over_ball_num,int64,0,192235
4,Integrated deliveries,4,innings_ball_num,int64,0,192235
5,Integrated deliveries,5,batting_team_id,int64,0,192235
6,Integrated deliveries,6,bowling_team_id,int64,0,192235
7,Integrated deliveries,16,runs,int64,0,192235
8,Integrated deliveries,19,is_wicket,bool,0,192235
9,Integrated deliveries,24,extras1_runs,int64,0,192235


,check,passed
0,Integrated dataset has 61 columns,True
1,Pre-delivery dataset has columns,True
2,Schema inventory covers every column,True


### Define the pre-delivery state field mapping

In [86]:
phase3C_field_mapping_rows = [
    ("match_id", "Match identifier", "DIRECTLY OBSERVED", "Allowed",
     "Identifies the match containing the delivery."),
    ("innings", "Innings identifier", "DIRECTLY OBSERVED", "Allowed",
     "Identifies the innings containing the delivery."),
    ("comp_id", "Competition identifier", "DIRECTLY OBSERVED", "Allowed",
     "Identifies the competition."),
    ("comp_name", "Competition name", "DIRECTLY OBSERVED", "Allowed",
     "Provides competition context."),
    ("match_format", "Competition format", "DIRECTLY OBSERVED", "Allowed",
     "Provides the recorded match format."),
    ("season", "Season", "DIRECTLY OBSERVED", "Allowed",
     "Provides season context."),
    ("batting_team_id", "Batting team", "DIRECTLY OBSERVED", "Allowed",
     "Identifies the batting team."),
    ("bowling_team_id", "Bowling team", "DIRECTLY OBSERVED", "Allowed",
     "Identifies the bowling team."),
    ("overid", "Over sequence identifier", "DIRECTLY OBSERVED",
     "Allowed with limitation",
     "Provides recorded over grouping, not legal balls remaining."),
    ("over_ball_num", "Within-over sequence position",
     "DIRECTLY OBSERVED", "Allowed with limitation",
     "Sequence position only; not a legal-ball counter."),
    ("innings_ball_num", "Innings delivery sequence position",
     "DIRECTLY OBSERVED", "Allowed with limitation",
     "Sequence position only; not a legal-ball counter."),
    ("pre_delivery_wickets_fallen",
     "Wickets fallen before current delivery", "RECONSTRUCTED",
     "Conditional",
     "Derived from prior wicket events; retain documented exceptions."),
    ("post_delivery_wickets_fallen",
     "Wickets fallen after current delivery", "DELIVERY OUTCOME",
     "Prohibited",
     "Includes the current delivery's wicket event."),
    ("is_wicket", "Current delivery wicket indicator",
     "DELIVERY OUTCOME", "Prohibited",
     "Describes the current delivery outcome."),
    ("runs", "Current delivery runs field", "DELIVERY OUTCOME",
     "Prohibited",
     "Current delivery scoring semantics remain unresolved."),
    ("extras1_runs", "Current delivery extras category code",
     "DELIVERY OUTCOME", "Prohibited",
     "Categorical code, not a numeric extras total."),
    ("extras_2_runs", "Current delivery secondary extras field",
     "UNRESOLVED OUTCOME FIELD", "Prohibited",
     "Semantics remain unresolved."),
    ("legal_ball_evidence_class",
     "Legal-ball evidence classification", "DELIVERY EVIDENCE",
     "Not as remaining balls",
     "Evidence about the current delivery; does not establish cumulative legal-ball progression."),
    ("legal_ball_evidence_status",
     "Legal-ball evidence status", "DELIVERY EVIDENCE",
     "Not as remaining balls",
     "Does not establish a validated legal-ball counter."),
    ("legal_ball_final_treatment",
     "Final legal-ball treatment", "DELIVERY EVIDENCE",
     "Not as remaining balls",
     "Does not establish a validated legal-ball counter."),
    ("revised_target", "Revised innings target", "SPARSE / UNRESOLVED",
     "Conditional; exclude pending validation",
     "Sparse and not yet validated for state use."),
    ("first_inn_revised_overs", "Revised first-innings overs",
     "SPARSE / UNRESOLVED", "Exclude pending validation",
     "Sparse; units and interpretation unresolved."),
    ("second_inn_revised_overs", "Revised second-innings overs",
     "SPARSE / UNRESOLVED", "Exclude pending validation",
     "Sparse; units and interpretation unresolved."),
    ("win_team_id", "Match winner", "POST-MATCH OUTCOME",
     "Prohibited",
     "Known only after the match outcome."),
    ("scheduled_innings_length", "Scheduled innings length",
     "UNAVAILABLE", "Unavailable",
     "No validated scheduled-length field established."),
    ("cumulative_innings_score", "Score before current delivery",
     "UNAVAILABLE", "Unavailable",
     "Scoring semantics and cumulative reconstruction are unresolved."),
    ("legal_balls_remaining", "Legal balls remaining",
     "UNAVAILABLE", "Unavailable",
     "Legal-ball progression cannot be reliably reconstructed."),
    ("innings_stage", "Innings stage", "UNAVAILABLE", "Unavailable",
     "No validated stage definition or reliable progression variable.")
]

phase3C_field_mapping = pd.DataFrame(
    phase3C_field_mapping_rows,
    columns=[
        "field",
        "role",
        "classification",
        "pre_delivery_use",
        "reason"
    ]
)

# Identify the correct source dataset for each field.
phase3C_reconstructed_fields = {
    "pre_delivery_wickets_fallen",
    "post_delivery_wickets_fallen"
}

phase3C_field_mapping["source_dataset"] = (
    phase3C_field_mapping["field"].apply(
        lambda field:
            "Pre-delivery wicket states"
            if field in phase3C_reconstructed_fields
            else "Integrated deliveries"
    )
)

# Validate field availability against the correct dataset.
phase3C_field_mapping["source_field_present"] = (
    phase3C_field_mapping.apply(
        lambda row:
            row["field"] in (
                phase3_pre_delivery_wicket_states.columns
                if row["source_dataset"] == "Pre-delivery wicket states"
                else df_deliveries.columns
            ),
        axis=1
    )
)

phase3C_field_mapping["mapping_check"] = (
    phase3C_field_mapping["source_field_present"]
    | phase3C_field_mapping["classification"].eq("UNAVAILABLE")
)

# Display the complete mapping.
display(phase3C_field_mapping)

# Mapping summary.
phase3C_mapping_summary = (
    phase3C_field_mapping
    .groupby(
        ["classification", "pre_delivery_use"],
        dropna=False
    )
    .size()
    .reset_index(name="field_count")
)

display(phase3C_mapping_summary)

# Final validation.
phase3C_mapping_validation = pd.DataFrame([
    {
        "check": "Mapped field names are unique",
        "passed": phase3C_field_mapping["field"].is_unique
    },
    {
        "check": "All mapped fields are present or explicitly unavailable",
        "passed": phase3C_field_mapping["mapping_check"].all()
    },
    {
        "check": "Current delivery wicket outcome is prohibited",
        "passed": (
            phase3C_field_mapping.loc[
                phase3C_field_mapping["field"].eq("is_wicket"),
                "pre_delivery_use"
            ].iloc[0] == "Prohibited"
        )
    },
    {
        "check": "Current delivery runs are prohibited",
        "passed": (
            phase3C_field_mapping.loc[
                phase3C_field_mapping["field"].eq("runs"),
                "pre_delivery_use"
            ].iloc[0] == "Prohibited"
        )
    },
    {
        "check": "Pre-delivery wicket state is conditional",
        "passed": (
            phase3C_field_mapping.loc[
                phase3C_field_mapping["field"].eq(
                    "pre_delivery_wickets_fallen"
                ),
                "pre_delivery_use"
            ].iloc[0] == "Conditional"
        )
    }
])

display(phase3C_mapping_validation)

assert phase3C_mapping_validation["passed"].all()

phase3C_field_mapping.to_csv(
    OUTPUT_DIR / "phase3C_pre_delivery_field_mapping.csv",
    index=False
)

print("Section 3C.3 mapping and validation completed successfully.")

,field,role,classification,pre_delivery_use,reason,source_dataset,source_field_present,mapping_check
0,match_id,Match identifier,DIRECTLY OBSERVED,Allowed,Identifies the match containing the delivery.,Integrated deliveries,True,True
1,innings,Innings identifier,DIRECTLY OBSERVED,Allowed,Identifies the innings containing the delivery.,Integrated deliveries,True,True
2,comp_id,Competition identifier,DIRECTLY OBSERVED,Allowed,Identifies the competition.,Integrated deliveries,True,True
3,comp_name,Competition name,DIRECTLY OBSERVED,Allowed,Provides competition context.,Integrated deliveries,True,True
4,match_format,Competition format,DIRECTLY OBSERVED,Allowed,Provides the recorded match format.,Integrated deliveries,True,True
5,season,Season,DIRECTLY OBSERVED,Allowed,Provides season context.,Integrated deliveries,True,True
6,batting_team_id,Batting team,DIRECTLY OBSERVED,Allowed,Identifies the batting team.,Integrated deliveries,True,True
7,bowling_team_id,Bowling team,DIRECTLY OBSERVED,Allowed,Identifies the bowling team.,Integrated deliveries,True,True
8,overid,Over sequence identifier,DIRECTLY OBSERVED,Allowed with limitation,"Provides recorded over grouping, not legal balls remaining.",Integrated deliveries,True,True
9,over_ball_num,Within-over sequence position,DIRECTLY OBSERVED,Allowed with limitation,Sequence position only; not a legal-ball counter.,Integrated deliveries,True,True


,classification,pre_delivery_use,field_count
0,DELIVERY EVIDENCE,Not as remaining balls,3
1,DELIVERY OUTCOME,Prohibited,4
2,DIRECTLY OBSERVED,Allowed,8
3,DIRECTLY OBSERVED,Allowed with limitation,3
4,POST-MATCH OUTCOME,Prohibited,1
5,RECONSTRUCTED,Conditional,1
6,SPARSE / UNRESOLVED,Conditional; exclude pending validation,1
7,SPARSE / UNRESOLVED,Exclude pending validation,2
8,UNAVAILABLE,Unavailable,4
9,UNRESOLVED OUTCOME FIELD,Prohibited,1


,check,passed
0,Mapped field names are unique,True
1,All mapped fields are present or explicitly unavailable,True
2,Current delivery wicket outcome is prohibited,True
3,Current delivery runs are prohibited,True
4,Pre-delivery wicket state is conditional,True


Section 3C.3 mapping and validation completed successfully.


### Delivery-level alignment validation

In [87]:
phase3C_delivery_keys = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num"
]

# 1. Verify required keys exist in both datasets.
phase3C_key_presence = pd.DataFrame([
    {
        "dataset": dataset_name,
        "key": key,
        "present": key in dataset.columns
    }
    for dataset_name, dataset in [
        ("Integrated deliveries", df_deliveries),
        ("Pre-delivery wicket states", phase3_pre_delivery_wicket_states)
    ]
    for key in phase3C_delivery_keys
])

display(phase3C_key_presence)

assert phase3C_key_presence["present"].all(), (
    "One or more required delivery keys are missing."
)

# 2. Check duplicate and missing keys.
phase3C_alignment_rows = []

for dataset_name, dataset in [
    ("Integrated deliveries", df_deliveries),
    ("Pre-delivery wicket states", phase3_pre_delivery_wicket_states)
]:
    phase3C_alignment_rows.append({
        "dataset": dataset_name,
        "row_count": len(dataset),
        "duplicate_key_rows": int(
            dataset.duplicated(
                subset=phase3C_delivery_keys,
                keep=False
            ).sum()
        ),
        "missing_key_rows": int(
            dataset[phase3C_delivery_keys].isna().any(axis=1).sum()
        ),
        "unique_delivery_keys": int(
            dataset[phase3C_delivery_keys]
            .drop_duplicates()
            .shape[0]
        )
    })

phase3C_alignment_summary = pd.DataFrame(
    phase3C_alignment_rows
)

display(phase3C_alignment_summary)

# 3. Compare the delivery-key populations.
phase3C_integrated_keys = (
    df_deliveries[phase3C_delivery_keys]
    .drop_duplicates()
)

phase3C_wicket_keys = (
    phase3_pre_delivery_wicket_states[phase3C_delivery_keys]
    .drop_duplicates()
)

phase3C_key_comparison = phase3C_integrated_keys.merge(
    phase3C_wicket_keys,
    on=phase3C_delivery_keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)

phase3C_key_comparison_summary = (
    phase3C_key_comparison["_merge"]
    .value_counts(dropna=False)
    .rename_axis("key_match_status")
    .reset_index(name="delivery_key_count")
)

display(phase3C_key_comparison_summary)

# 4. Validate wicket-state transitions against current delivery events.
phase3C_wicket_comparison = df_deliveries[
    phase3C_delivery_keys + ["is_wicket"]
].merge(
    phase3_pre_delivery_wicket_states[
        phase3C_delivery_keys
        + [
            "pre_delivery_wickets_fallen",
            "post_delivery_wickets_fallen"
        ]
    ],
    on=phase3C_delivery_keys,
    how="inner",
    validate="one_to_one"
)

phase3C_wicket_comparison["expected_post_delivery_wickets"] = (
    phase3C_wicket_comparison["pre_delivery_wickets_fallen"]
    + phase3C_wicket_comparison["is_wicket"].astype("int64")
)

phase3C_wicket_comparison["post_state_matches_current_event"] = (
    phase3C_wicket_comparison["post_delivery_wickets_fallen"]
    == phase3C_wicket_comparison["expected_post_delivery_wickets"]
)

phase3C_wicket_transition_summary = pd.DataFrame([
    {
        "check": "Joined delivery count",
        "observed": len(phase3C_wicket_comparison),
        "expected": len(df_deliveries)
    },
    {
        "check": "Post-delivery wicket transition mismatches",
        "observed": int(
            (~phase3C_wicket_comparison[
                "post_state_matches_current_event"
            ]).sum()
        ),
        "expected": 0
    }
])

display(phase3C_wicket_transition_summary)

# 5. Consolidated validation.
phase3C_alignment_validation = pd.DataFrame([
    {
        "check": "Integrated delivery keys are unique",
        "passed": (
            phase3C_alignment_summary.loc[
                phase3C_alignment_summary["dataset"].eq(
                    "Integrated deliveries"
                ),
                "duplicate_key_rows"
            ].iloc[0] == 0
        )
    },
    {
        "check": "Pre-delivery state keys are unique",
        "passed": (
            phase3C_alignment_summary.loc[
                phase3C_alignment_summary["dataset"].eq(
                    "Pre-delivery wicket states"
                ),
                "duplicate_key_rows"
            ].iloc[0] == 0
        )
    },
    {
        "check": "No missing delivery keys",
        "passed": (
            phase3C_alignment_summary["missing_key_rows"].sum() == 0
        )
    },
    {
        "check": "Delivery-key populations match exactly",
        "passed": (
            len(phase3C_key_comparison) == len(df_deliveries)
            and phase3C_key_comparison["_merge"].eq("both").all()
        )
    },
    {
        "check": "Every delivery has a matching wicket-state record",
        "passed": (
            len(phase3C_wicket_comparison) == len(df_deliveries)
        )
    },
    {
        "check": "Post-delivery wicket transitions reconcile",
        "passed": (
            phase3C_wicket_comparison[
                "post_state_matches_current_event"
            ].all()
        )
    }
])

display(phase3C_alignment_validation)

assert phase3C_alignment_validation["passed"].all(), (
    "Delivery-level alignment validation failed. "
    "Inspect the displayed audit tables before proceeding."
)

print("Section 3C.4 delivery-level alignment validation passed.")

,dataset,key,present
0,Integrated deliveries,match_id,True
1,Integrated deliveries,innings,True
2,Integrated deliveries,overid,True
3,Integrated deliveries,over_ball_num,True
4,Integrated deliveries,innings_ball_num,True
5,Pre-delivery wicket states,match_id,True
6,Pre-delivery wicket states,innings,True
7,Pre-delivery wicket states,overid,True
8,Pre-delivery wicket states,over_ball_num,True
9,Pre-delivery wicket states,innings_ball_num,True


,dataset,row_count,duplicate_key_rows,missing_key_rows,unique_delivery_keys
0,Integrated deliveries,192235,0,0,192235
1,Pre-delivery wicket states,192235,0,0,192235


,key_match_status,delivery_key_count
0,both,192235
1,left_only,0
2,right_only,0


,check,observed,expected
0,Joined delivery count,192235,192235
1,Post-delivery wicket transition mismatches,0,0


,check,passed
0,Integrated delivery keys are unique,True
1,Pre-delivery state keys are unique,True
2,No missing delivery keys,True
3,Delivery-key populations match exactly,True
4,Every delivery has a matching wicket-state record,True
5,Post-delivery wicket transitions reconcile,True


Section 3C.4 delivery-level alignment validation passed.


### Pre-delivery state integrity and leakage checks

In [90]:
## Confirm required upstream objects exist.
assert "phase3C_wicket_comparison" in globals(), (
    "Run the successful Section 3C.4 alignment cell first."
)

assert "phase3C_field_mapping" in globals(), (
    "Section 3C.3 field mapping is not available."
)

# Create a derived validation dataset.
phase3C_integrity = phase3C_wicket_comparison.copy()

phase3C_integrity = phase3C_integrity.sort_values(
    [
        "match_id",
        "innings",
        "innings_ball_num",
        "overid",
        "over_ball_num"
    ]
).reset_index(drop=True)

# 1. Validate wicket-state bounds.
phase3C_integrity["pre_wickets_within_bounds"] = (
    phase3C_integrity["pre_delivery_wickets_fallen"].between(0, 10)
)

phase3C_integrity["post_wickets_within_bounds"] = (
    phase3C_integrity["post_delivery_wickets_fallen"].between(0, 10)
)

# 2. Validate that every innings starts at zero wickets.
phase3C_innings_starts = (
    phase3C_integrity
    .groupby(["match_id", "innings"], as_index=False)
    .first()
)

phase3C_innings_starts["starts_at_zero_wickets"] = (
    phase3C_innings_starts["pre_delivery_wickets_fallen"] == 0
)

# 3. Validate continuity between consecutive deliveries.
phase3C_integrity["next_pre_delivery_wickets"] = (
    phase3C_integrity
    .groupby(["match_id", "innings"])[
        "pre_delivery_wickets_fallen"
    ]
    .shift(-1)
)

phase3C_integrity["next_pre_matches_current_post"] = (
    phase3C_integrity["next_pre_delivery_wickets"]
    == phase3C_integrity["post_delivery_wickets_fallen"]
)

phase3C_integrity["is_last_delivery_in_innings"] = (
    phase3C_integrity["next_pre_delivery_wickets"].isna()
)

# 4. Summarise integrity results.
phase3C_integrity_summary = pd.DataFrame([
    {
        "check": "Pre-delivery wicket counts within 0–10",
        "observed_failures": int(
            (~phase3C_integrity["pre_wickets_within_bounds"]).sum()
        ),
        "expected_failures": 0
    },
    {
        "check": "Post-delivery wicket counts within 0–10",
        "observed_failures": int(
            (~phase3C_integrity["post_wickets_within_bounds"]).sum()
        ),
        "expected_failures": 0
    },
    {
        "check": "Innings beginning with zero wickets",
        "observed_failures": int(
            (~phase3C_innings_starts["starts_at_zero_wickets"]).sum()
        ),
        "expected_failures": 0
    },
    {
        "check": "Consecutive delivery wicket-state continuity",
        "observed_failures": int(
            (
                ~phase3C_integrity.loc[
                    ~phase3C_integrity["is_last_delivery_in_innings"],
                    "next_pre_matches_current_post"
                ]
            ).sum()
        ),
        "expected_failures": 0
    }
])

display(phase3C_integrity_summary)

# 5. Explicit leakage exclusion audit.
phase3C_prohibited_fields = [
    "is_wicket",
    "runs",
    "extras1_runs",
    "extras_2_runs",
    "post_delivery_wickets_fallen",
    "win_team_id"
]

phase3C_leakage_audit = phase3C_field_mapping[
    phase3C_field_mapping["field"].isin(
        phase3C_prohibited_fields
    )
][
    ["field", "classification", "pre_delivery_use"]
].copy()

phase3C_leakage_audit["correctly_prohibited"] = (
    phase3C_leakage_audit["pre_delivery_use"] == "Prohibited"
)

display(phase3C_leakage_audit)

# 6. Consolidated validation.
phase3C_integrity_validation = pd.DataFrame([
    {
        "check": "All pre-delivery wicket counts are within bounds",
        "passed": phase3C_integrity_summary.loc[
            phase3C_integrity_summary["check"].eq(
                "Pre-delivery wicket counts within 0–10"
            ),
            "observed_failures"
        ].iloc[0] == 0
    },
    {
        "check": "All post-delivery wicket counts are within bounds",
        "passed": phase3C_integrity_summary.loc[
            phase3C_integrity_summary["check"].eq(
                "Post-delivery wicket counts within 0–10"
            ),
            "observed_failures"
        ].iloc[0] == 0
    },
    {
        "check": "All innings start at zero wickets",
        "passed": phase3C_innings_starts[
            "starts_at_zero_wickets"
        ].all()
    },
    {
        "check": "Wicket state carries forward between deliveries",
        "passed": phase3C_integrity_summary.loc[
            phase3C_integrity_summary["check"].eq(
                "Consecutive delivery wicket-state continuity"
            ),
            "observed_failures"
        ].iloc[0] == 0
    },
    {
        "check": "All identified outcome fields are prohibited",
        "passed": (
            len(phase3C_leakage_audit) == len(phase3C_prohibited_fields)
            and phase3C_leakage_audit["correctly_prohibited"].all()
        )
    }
])

display(phase3C_integrity_validation)

assert phase3C_integrity_validation["passed"].all(), (
    "Pre-delivery integrity validation failed. "
    "Inspect the audit outputs before proceeding."
)

print("Section 3C.5 pre-delivery integrity validation passed.")

,check,observed_failures,expected_failures
0,Pre-delivery wicket counts within 0–10,0,0
1,Post-delivery wicket counts within 0–10,0,0
2,Innings beginning with zero wickets,0,0
3,Consecutive delivery wicket-state continuity,0,0


,field,classification,pre_delivery_use,correctly_prohibited
12,post_delivery_wickets_fallen,DELIVERY OUTCOME,Prohibited,True
13,is_wicket,DELIVERY OUTCOME,Prohibited,True
14,runs,DELIVERY OUTCOME,Prohibited,True
15,extras1_runs,DELIVERY OUTCOME,Prohibited,True
16,extras_2_runs,UNRESOLVED OUTCOME FIELD,Prohibited,True
23,win_team_id,POST-MATCH OUTCOME,Prohibited,True


,check,passed
0,All pre-delivery wicket counts are within bounds,True
1,All post-delivery wicket counts are within bounds,True
2,All innings start at zero wickets,True
3,Wicket state carries forward between deliveries,True
4,All identified outcome fields are prohibited,True


Section 3C.5 pre-delivery integrity validation passed.


### Exception and limitation register

In [91]:
phase3C_exception_rows = [
    {
        "exception_id": "3C-EX-001",
        "field": "pre_delivery_wickets_fallen",
        "status": "CONDITIONAL",
        "evidence": (
            "Reconstructed from prior wicket events. "
            "Delivery alignment and state transitions passed validation."
        ),
        "impact": (
            "Wicket count is usable as a reconstructed pre-delivery "
            "state variable, subject to documented dismissal exceptions."
        ),
        "treatment": (
            "Retain as conditional. Do not describe it as independently "
            "verified dismissal truth."
        )
    },
    {
        "exception_id": "3C-EX-002",
        "field": "cumulative_innings_score",
        "status": "UNAVAILABLE",
        "evidence": (
            "The runs field semantics remain unresolved; extras fields "
            "do not provide a validated scoring reconstruction."
        ),
        "impact": (
            "Score before the current delivery cannot be established "
            "reliably."
        ),
        "treatment": (
            "Exclude from the pre-delivery state schema."
        )
    },
    {
        "exception_id": "3C-EX-003",
        "field": "legal_balls_remaining",
        "status": "UNAVAILABLE",
        "evidence": (
            "Legal-ball progression remains unresolved. Sequence fields "
            "and delivery-level legal-ball evidence do not establish "
            "a cumulative legal-ball counter."
        ),
        "impact": (
            "Remaining legal balls cannot be calculated reliably."
        ),
        "treatment": (
            "Exclude from the pre-delivery state schema."
        )
    },
    {
        "exception_id": "3C-EX-004",
        "field": "scheduled_innings_length",
        "status": "UNAVAILABLE",
        "evidence": (
            "No validated scheduled innings-length field has been "
            "established."
        ),
        "impact": (
            "The expected innings duration cannot be established "
            "consistently."
        ),
        "treatment": (
            "Exclude from the pre-delivery state schema."
        )
    },
    {
        "exception_id": "3C-EX-005",
        "field": "innings_stage",
        "status": "UNAVAILABLE",
        "evidence": (
            "No validated stage definition or reliable progression "
            "variable is available."
        ),
        "impact": (
            "Powerplay, middle and death-stage classifications cannot "
            "be assigned reliably from the current evidence."
        ),
        "treatment": (
            "Do not infer or assign innings stages."
        )
    },
    {
        "exception_id": "3C-EX-006",
        "field": "revised_target",
        "status": "SPARSE / UNRESOLVED",
        "evidence": (
            "The field is populated for only a small subset of delivery "
            "records; its use in a complete state specification has "
            "not been validated."
        ),
        "impact": (
            "Revised target context is not consistently available."
        ),
        "treatment": (
            "Exclude from the common state schema pending separate "
            "validation."
        )
    },
    {
        "exception_id": "3C-EX-007",
        "field": "first_inn_revised_overs",
        "status": "SPARSE / UNRESOLVED",
        "evidence": (
            "The field is sparsely populated and its units and "
            "interpretation remain unresolved."
        ),
        "impact": (
            "Cannot reliably establish revised first-innings duration."
        ),
        "treatment": (
            "Exclude pending semantic and coverage validation."
        )
    },
    {
        "exception_id": "3C-EX-008",
        "field": "second_inn_revised_overs",
        "status": "SPARSE / UNRESOLVED",
        "evidence": (
            "The field is sparsely populated and its units and "
            "interpretation remain unresolved."
        ),
        "impact": (
            "Cannot reliably establish revised second-innings duration."
        ),
        "treatment": (
            "Exclude pending semantic and coverage validation."
        )
    },
    {
        "exception_id": "3C-EX-009",
        "field": "over_ball_num",
        "status": "SEQUENCE ONLY",
        "evidence": (
            "The field represents within-over delivery sequence position, "
            "not a validated legal-ball count."
        ),
        "impact": (
            "Cannot use it to infer legal balls remaining."
        ),
        "treatment": (
            "Retain only as a delivery-order descriptor."
        )
    },
    {
        "exception_id": "3C-EX-010",
        "field": "innings_ball_num",
        "status": "SEQUENCE ONLY",
        "evidence": (
            "The field represents innings delivery sequence position, "
            "not a validated legal-ball count."
        ),
        "impact": (
            "Cannot use it to infer legal balls remaining or innings stage."
        ),
        "treatment": (
            "Retain only as a delivery-order descriptor."
        )
    },
    {
        "exception_id": "3C-EX-011",
        "field": "current_delivery_outcomes",
        "status": "LEAKAGE PROHIBITION",
        "evidence": (
            "The current delivery's runs, wicket indicator, extras and "
            "post-delivery wicket state describe an event occurring "
            "after the pre-delivery state boundary."
        ),
        "impact": (
            "Including these fields would introduce future information "
            "into the pre-delivery state."
        ),
        "treatment": (
            "Prohibit current-delivery and post-delivery outcome fields "
            "from pre-delivery state inputs."
        )
    }
]

phase3C_exception_register = pd.DataFrame(
    phase3C_exception_rows
)

display(phase3C_exception_register)

# Summary by exception status.
phase3C_exception_summary = (
    phase3C_exception_register
    .groupby("status")
    .size()
    .reset_index(name="exception_count")
)

display(phase3C_exception_summary)

# Validate completeness and traceability.
phase3C_exception_validation = pd.DataFrame([
    {
        "check": "Exception IDs are unique",
        "passed": phase3C_exception_register[
            "exception_id"
        ].is_unique
    },
    {
        "check": "All exceptions have a field",
        "passed": phase3C_exception_register[
            "field"
        ].notna().all()
    },
    {
        "check": "All exceptions have evidence",
        "passed": phase3C_exception_register[
            "evidence"
        ].str.strip().ne("").all()
    },
    {
        "check": "All exceptions have impact descriptions",
        "passed": phase3C_exception_register[
            "impact"
        ].str.strip().ne("").all()
    },
    {
        "check": "All exceptions have explicit treatment",
        "passed": phase3C_exception_register[
            "treatment"
        ].str.strip().ne("").all()
    }
])

display(phase3C_exception_validation)

assert phase3C_exception_validation["passed"].all()

phase3C_exception_register.to_csv(
    OUTPUT_DIR / "phase3C_exception_limitation_register.csv",
    index=False
)

print("Section 3C.6 exception and limitation register exported successfully.")

,exception_id,field,status,evidence,impact,treatment
0,3C-EX-001,pre_delivery_wickets_fallen,CONDITIONAL,Reconstructed from prior wicket events. Delivery alignment and state transitions passed validation.,"Wicket count is usable as a reconstructed pre-delivery state variable, subject to documented dismissal exceptions.",Retain as conditional. Do not describe it as independently verified dismissal truth.
1,3C-EX-002,cumulative_innings_score,UNAVAILABLE,The runs field semantics remain unresolved; extras fields do not provide a validated scoring reconstruction.,Score before the current delivery cannot be established reliably.,Exclude from the pre-delivery state schema.
2,3C-EX-003,legal_balls_remaining,UNAVAILABLE,Legal-ball progression remains unresolved. Sequence fields and delivery-level legal-ball evidence do not establish a cumulative legal-ball counter.,Remaining legal balls cannot be calculated reliably.,Exclude from the pre-delivery state schema.
3,3C-EX-004,scheduled_innings_length,UNAVAILABLE,No validated scheduled innings-length field has been established.,The expected innings duration cannot be established consistently.,Exclude from the pre-delivery state schema.
4,3C-EX-005,innings_stage,UNAVAILABLE,No validated stage definition or reliable progression variable is available.,"Powerplay, middle and death-stage classifications cannot be assigned reliably from the current evidence.",Do not infer or assign innings stages.
5,3C-EX-006,revised_target,SPARSE / UNRESOLVED,The field is populated for only a small subset of delivery records; its use in a complete state specification has not been validated.,Revised target context is not consistently available.,Exclude from the common state schema pending separate validation.
6,3C-EX-007,first_inn_revised_overs,SPARSE / UNRESOLVED,The field is sparsely populated and its units and interpretation remain unresolved.,Cannot reliably establish revised first-innings duration.,Exclude pending semantic and coverage validation.
7,3C-EX-008,second_inn_revised_overs,SPARSE / UNRESOLVED,The field is sparsely populated and its units and interpretation remain unresolved.,Cannot reliably establish revised second-innings duration.,Exclude pending semantic and coverage validation.
8,3C-EX-009,over_ball_num,SEQUENCE ONLY,"The field represents within-over delivery sequence position, not a validated legal-ball count.",Cannot use it to infer legal balls remaining.,Retain only as a delivery-order descriptor.
9,3C-EX-010,innings_ball_num,SEQUENCE ONLY,"The field represents innings delivery sequence position, not a validated legal-ball count.",Cannot use it to infer legal balls remaining or innings stage.,Retain only as a delivery-order descriptor.


,status,exception_count
0,CONDITIONAL,1
1,LEAKAGE PROHIBITION,1
2,SEQUENCE ONLY,2
3,SPARSE / UNRESOLVED,3
4,UNAVAILABLE,4


,check,passed
0,Exception IDs are unique,True
1,All exceptions have a field,True
2,All exceptions have evidence,True
3,All exceptions have impact descriptions,True
4,All exceptions have explicit treatment,True


Section 3C.6 exception and limitation register exported successfully.


### Final pre-delivery state specification

In [92]:
# Select only fields approved for pre-delivery state use.
phase3C_approved_uses = [
    "Allowed",
    "Allowed with limitation",
    "Conditional"
]

phase3C_state_schema = phase3C_field_mapping[
    phase3C_field_mapping["pre_delivery_use"].isin(
        phase3C_approved_uses
    )
].copy()

# Assign explicit state roles.
phase3C_state_schema["state_role"] = (
    phase3C_state_schema["field"].map({
        "match_id": "Match identifier",
        "innings": "Innings context",
        "comp_id": "Competition identifier",
        "comp_name": "Competition context",
        "match_format": "Format context",
        "season": "Season context",
        "batting_team_id": "Batting team context",
        "bowling_team_id": "Bowling team context",
        "overid": "Delivery sequence context",
        "over_ball_num": "Within-over sequence position",
        "innings_ball_num": "Innings sequence position",
        "pre_delivery_wickets_fallen": "Pre-delivery wicket state"
    })
)

# Retrieve source data types from the appropriate source dataset.
phase3C_state_schema["data_type"] = (
    phase3C_state_schema.apply(
        lambda row: str(
            (
                phase3_pre_delivery_wicket_states
                if row["source_dataset"] == "Pre-delivery wicket states"
                else df_deliveries
            )[row["field"]].dtype
        ),
        axis=1
    )
)

# Define the temporal boundary explicitly.
phase3C_state_schema["temporal_boundary"] = (
    "Known immediately before current delivery"
)

# Arrange the final specification columns.
phase3C_state_schema = phase3C_state_schema[
    [
        "field",
        "state_role",
        "data_type",
        "classification",
        "pre_delivery_use",
        "source_dataset",
        "temporal_boundary",
        "reason"
    ]
].reset_index(drop=True)

display(phase3C_state_schema)

# Summarise the approved state fields.
phase3C_state_summary = (
    phase3C_state_schema
    .groupby(
        ["classification", "pre_delivery_use"],
        dropna=False
    )
    .size()
    .reset_index(name="field_count")
)

display(phase3C_state_summary)

# Validate schema completeness and leakage exclusion.
phase3C_schema_validation = pd.DataFrame([
    {
        "check": "State schema contains 12 approved fields",
        "passed": len(phase3C_state_schema) == 12
    },
    {
        "check": "State field names are unique",
        "passed": phase3C_state_schema["field"].is_unique
    },
    {
        "check": "All state fields have source data types",
        "passed": phase3C_state_schema["data_type"].notna().all()
    },
    {
        "check": "All state fields have assigned roles",
        "passed": phase3C_state_schema["state_role"].notna().all()
    },
    {
        "check": "All fields have pre-delivery boundary labels",
        "passed": (
            phase3C_state_schema["temporal_boundary"]
            .eq("Known immediately before current delivery")
            .all()
        )
    },
    {
        "check": "No prohibited outcome fields included",
        "passed": not phase3C_state_schema["field"].isin([
            "is_wicket",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "post_delivery_wickets_fallen",
            "win_team_id"
        ]).any()
    },
    {
        "check": "Conditional wicket state retained",
        "passed": (
            phase3C_state_schema.loc[
                phase3C_state_schema["field"].eq(
                    "pre_delivery_wickets_fallen"
                ),
                "pre_delivery_use"
            ].iloc[0] == "Conditional"
        )
    }
])

display(phase3C_schema_validation)

assert phase3C_schema_validation["passed"].all(), (
    "Final state schema validation failed. "
    "Inspect the schema and validation outputs."
)

phase3C_state_schema.to_csv(
    OUTPUT_DIR / "phase3C_final_pre_delivery_state_schema.csv",
    index=False
)

print("Final pre-delivery state schema exported successfully.")

,field,state_role,data_type,classification,pre_delivery_use,source_dataset,temporal_boundary,reason
0,match_id,Match identifier,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the match containing the delivery.
1,innings,Innings context,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the innings containing the delivery.
2,comp_id,Competition identifier,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the competition.
3,comp_name,Competition context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides competition context.
4,match_format,Format context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides the recorded match format.
5,season,Season context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides season context.
6,batting_team_id,Batting team context,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the batting team.
7,bowling_team_id,Bowling team context,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the bowling team.
8,overid,Delivery sequence context,int64,DIRECTLY OBSERVED,Allowed with limitation,Integrated deliveries,Known immediately before current delivery,"Provides recorded over grouping, not legal balls remaining."
9,over_ball_num,Within-over sequence position,int64,DIRECTLY OBSERVED,Allowed with limitation,Integrated deliveries,Known immediately before current delivery,Sequence position only; not a legal-ball counter.


,classification,pre_delivery_use,field_count
0,DIRECTLY OBSERVED,Allowed,8
1,DIRECTLY OBSERVED,Allowed with limitation,3
2,RECONSTRUCTED,Conditional,1


,check,passed
0,State schema contains 12 approved fields,True
1,State field names are unique,True
2,All state fields have source data types,True
3,All state fields have assigned roles,True
4,All fields have pre-delivery boundary labels,True
5,No prohibited outcome fields included,True
6,Conditional wicket state retained,True


Final pre-delivery state schema exported successfully.


## Consolidated report

In [93]:
# 1. Confirm all required Phase 3C objects exist.
phase3C_required_objects = [
    "phase3C_state_boundaries",
    "phase3C_field_mapping",
    "phase3C_alignment_validation",
    "phase3C_integrity_validation",
    "phase3C_exception_register",
    "phase3C_schema_validation",
    "phase3C_state_schema"
]

phase3C_object_check = pd.DataFrame([
    {
        "object": object_name,
        "available": object_name in globals()
    }
    for object_name in phase3C_required_objects
])

display(phase3C_object_check)

assert phase3C_object_check["available"].all(), (
    "One or more required Phase 3C objects are unavailable."
)

# 2. Consolidate validation results.
phase3C_validation_register = pd.concat(
    [
        phase3C_alignment_validation.assign(section="3C.4"),
        phase3C_integrity_validation.assign(section="3C.5"),
        phase3C_schema_validation.assign(section="3C.7")
    ],
    ignore_index=True
)

phase3C_validation_register = phase3C_validation_register[
    ["section", "check", "passed"]
]

display(phase3C_validation_register)

# 3. Create the Phase 3C completion summary.
phase3C_completion_summary = pd.DataFrame([
    {
        "subsection": "3C.1",
        "description": "State specification boundaries",
        "status": "COMPLETE"
    },
    {
        "subsection": "3C.2",
        "description": "Source-field mapping audit",
        "status": "COMPLETE"
    },
    {
        "subsection": "3C.3",
        "description": "Pre-delivery field mapping",
        "status": "COMPLETE"
    },
    {
        "subsection": "3C.4",
        "description": "Delivery-level alignment validation",
        "status": "COMPLETE"
    },
    {
        "subsection": "3C.5",
        "description": "Pre-delivery integrity and leakage checks",
        "status": "COMPLETE"
    },
    {
        "subsection": "3C.6",
        "description": "Exception and limitation register",
        "status": "COMPLETE"
    },
    {
        "subsection": "3C.7",
        "description": "Final pre-delivery state schema",
        "status": "COMPLETE"
    },
    {
        "subsection": "3C.8",
        "description": "Consolidated report and closure",
        "status": "IN PROGRESS"
    }
])

display(phase3C_completion_summary)

# 4. Build the consolidated report.
phase3C_report_text = f"""# SupeRR Selector Season 2
# Phase 3C — Pre-delivery State Specification

## 1. Objective

Phase 3C defines the information available immediately before each delivery.
It establishes a reproducible pre-delivery state specification, identifies
permitted and prohibited fields, validates reconstructed wicket-state
consistency, and documents unavailable or unresolved state dimensions.

This phase does not construct pressure scores, player metrics, rankings,
or first-innings benchmarks.

## 2. Population and data coverage

- Integrated delivery records: {len(df_deliveries):,}
- Pre-delivery wicket-state records: {len(phase3_pre_delivery_wicket_states):,}
- Final approved state fields: {len(phase3C_state_schema)}
- Documented exceptions and limitations: {len(phase3C_exception_register)}

## 3. Subsection completion

| Subsection | Description | Status |
|---|---|---|
| 3C.1 | State specification boundaries | Complete |
| 3C.2 | Source-field mapping audit | Complete |
| 3C.3 | Pre-delivery field mapping | Complete |
| 3C.4 | Delivery-level alignment validation | Complete |
| 3C.5 | Pre-delivery integrity and leakage checks | Complete |
| 3C.6 | Exception and limitation register | Complete |
| 3C.7 | Final pre-delivery state schema | Complete |
| 3C.8 | Consolidated report and closure | Complete subject to final checks |

## 4. Final pre-delivery state schema

The approved schema contains {len(phase3C_state_schema)} fields:

{phase3C_state_schema[
    ['field', 'classification', 'pre_delivery_use']
].to_markdown(index=False)}

The schema consists of directly observed match and delivery context,
sequence-position fields with restricted interpretation, and a conditional
reconstructed pre-delivery wicket count.

## 5. Delivery alignment and integrity

The integrated delivery dataset and reconstructed wicket-state dataset
contain {len(df_deliveries):,} records each.

The Phase 3C.4 checks established:
- Unique delivery keys in both datasets.
- No missing delivery keys.
- Exact delivery-key population agreement.
- One-to-one delivery-level matching.
- No post-delivery wicket transition mismatches.

The Phase 3C.5 checks established:
- Pre-delivery wicket counts within 0–10.
- Post-delivery wicket counts within 0–10.
- Every innings begins with zero wickets.
- Wicket state carries forward consistently between consecutive deliveries.
- Identified current-delivery and post-match outcome fields are prohibited
  from the pre-delivery state.

## 6. Exceptions and limitations

The exception register contains {len(phase3C_exception_register)} entries.

{phase3C_exception_register[
    ['exception_id', 'field', 'status', 'treatment']
].to_markdown(index=False)}

Key limitations:
- Cumulative innings score is unavailable.
- Legal balls remaining are unavailable.
- Scheduled innings length is unavailable.
- Innings stage is unavailable.
- Revised target and revised-overs fields remain sparse or unresolved.
- Delivery sequence positions must not be interpreted as legal-ball counts.
- Reconstructed wicket state remains conditional and subject to documented
  dismissal exceptions.

These limitations are retained explicitly. No unsupported state variables
have been inferred or manufactured.

## 7. Validation results

Total final alignment, integrity and schema checks:
{len(phase3C_validation_register)}

Passed:
{int(phase3C_validation_register['passed'].sum())}

Failed:
{int((~phase3C_validation_register['passed']).sum())}

## 8. Scope boundary

Phase 3C specifies the pre-delivery state only.

It does not:
- Calculate player performance metrics.
- Calculate player scores or rankings.
- Define player eligibility thresholds.
- Construct first-innings or second-innings benchmarks.
- Assign pressure scores.
- Resolve unavailable scoring or legal-ball variables.

Phase 3D has not been started or authorized by this closure.

## 9. Closure status

Technical completion is conditional on all final artifact and validation
checks passing.

Formal project approval remains separate from technical completion.
The documented data limitations remain active constraints for subsequent
work.
"""

# 5. Export report and audit artifacts.
phase3C_report_path = (
    OUTPUT_DIR / "SupeRR_Selector_S2_Phase_3C_Report.md"
)

phase3C_validation_path = (
    OUTPUT_DIR / "phase3C_final_validation_register.csv"
)

phase3C_completion_path = (
    OUTPUT_DIR / "phase3C_completion_summary.csv"
)

phase3C_report_path.write_text(
    phase3C_report_text,
    encoding="utf-8"
)

phase3C_validation_register.to_csv(
    phase3C_validation_path,
    index=False
)

phase3C_completion_summary.to_csv(
    phase3C_completion_path,
    index=False
)

# 6. Final artifact and validation checks.
phase3C_final_closure_validation = pd.DataFrame([
    {
        "check": "All required Phase 3C objects are available",
        "passed": phase3C_object_check["available"].all()
    },
    {
        "check": "All alignment, integrity and schema checks passed",
        "passed": phase3C_validation_register["passed"].all()
    },
    {
        "check": "Final state schema contains 12 fields",
        "passed": len(phase3C_state_schema) == 12
    },
    {
        "check": "Exception register contains 11 entries",
        "passed": len(phase3C_exception_register) == 11
    },
    {
        "check": "Consolidated report exists",
        "passed": phase3C_report_path.exists()
    },
    {
        "check": "Validation register export exists",
        "passed": phase3C_validation_path.exists()
    },
    {
        "check": "Completion summary export exists",
        "passed": phase3C_completion_path.exists()
    }
])

display(phase3C_final_closure_validation)

assert phase3C_final_closure_validation["passed"].all(), (
    "Phase 3C final closure validation failed."
)

# Update the final subsection status after successful checks.
phase3C_completion_summary.loc[
    phase3C_completion_summary["subsection"].eq("3C.8"),
    "status"
] = "COMPLETE"

phase3C_completion_summary.to_csv(
    phase3C_completion_path,
    index=False
)

print("Phase 3C technical closure checks passed.")
print("Report:", phase3C_report_path)
print("Validation register:", phase3C_validation_path)
print("Completion summary:", phase3C_completion_path)

,object,available
0,phase3C_state_boundaries,True
1,phase3C_field_mapping,True
2,phase3C_alignment_validation,True
3,phase3C_integrity_validation,True
4,phase3C_exception_register,True
5,phase3C_schema_validation,True
6,phase3C_state_schema,True


,section,check,passed
0,3C.4,Integrated delivery keys are unique,True
1,3C.4,Pre-delivery state keys are unique,True
2,3C.4,No missing delivery keys,True
3,3C.4,Delivery-key populations match exactly,True
4,3C.4,Every delivery has a matching wicket-state record,True
5,3C.4,Post-delivery wicket transitions reconcile,True
6,3C.5,All pre-delivery wicket counts are within bounds,True
7,3C.5,All post-delivery wicket counts are within bounds,True
8,3C.5,All innings start at zero wickets,True
9,3C.5,Wicket state carries forward between deliveries,True


,subsection,description,status
0,3C.1,State specification boundaries,COMPLETE
1,3C.2,Source-field mapping audit,COMPLETE
2,3C.3,Pre-delivery field mapping,COMPLETE
3,3C.4,Delivery-level alignment validation,COMPLETE
4,3C.5,Pre-delivery integrity and leakage checks,COMPLETE
5,3C.6,Exception and limitation register,COMPLETE
6,3C.7,Final pre-delivery state schema,COMPLETE
7,3C.8,Consolidated report and closure,IN PROGRESS


,check,passed
0,All required Phase 3C objects are available,True
1,"All alignment, integrity and schema checks passed",True
2,Final state schema contains 12 fields,True
3,Exception register contains 11 entries,True
4,Consolidated report exists,True
5,Validation register export exists,True
6,Completion summary export exists,True


Phase 3C technical closure checks passed.
Report: /content/phase3_outputs/SupeRR_Selector_S2_Phase_3C_Report.md
Validation register: /content/phase3_outputs/phase3C_final_validation_register.csv
Completion summary: /content/phase3_outputs/phase3C_completion_summary.csv


# First-innings benchmarks

### Benchmark scope and boundaries

In [94]:
# 1. Confirm required Phase 3 objects exist
required_objects = [
    "df_deliveries",
    "phase3_pre_delivery_wicket_states",
    "phase3C_state_schema",
    "phase3C_exception_register",
]

object_check = pd.DataFrame([
    {
        "object": name,
        "available": name in globals(),
        "type": type(globals()[name]).__name__ if name in globals() else "NOT FOUND",
        "rows": len(globals()[name])
                 if name in globals() and hasattr(globals()[name], "__len__")
                 else None
    }
    for name in required_objects
])

print("1. Required Phase 3 objects")
display(object_check)

# 2. Inspect the delivery and pre-delivery state schemas
schema_checks = []

for object_name in ["df_deliveries", "phase3_pre_delivery_wicket_states"]:
    if object_name in globals() and isinstance(globals()[object_name], pd.DataFrame):
        current_df = globals()[object_name]
        schema_checks.append({
            "object": object_name,
            "rows": len(current_df),
            "columns": len(current_df.columns),
            "has_match_id": "match_id" in current_df.columns,
            "has_innings": "innings" in current_df.columns,
            "has_comp_id": "comp_id" in current_df.columns,
            "has_match_format": "match_format" in current_df.columns,
            "has_pre_delivery_wickets_fallen":
                "pre_delivery_wickets_fallen" in current_df.columns
        })

schema_check = pd.DataFrame(schema_checks)

print("\n2. Current table structure")
display(schema_check)

# 3. Show the approved Phase 3C state schema
print("\n3. Phase 3C state schema")
if "phase3C_state_schema" in globals():
    display(phase3C_state_schema)
else:
    print("BLOCKED: phase3C_state_schema is not available.")

# 4. Show the Phase 3C exception register
print("\n4. Phase 3C exception register")
if "phase3C_exception_register" in globals():
    display(phase3C_exception_register)
else:
    print("BLOCKED: phase3C_exception_register is not available.")

# 5. Define the Phase 3D scope and input boundaries
phase3D_scope = pd.DataFrame([
    {
        "area": "Benchmark population",
        "scope": "First innings only",
        "status": "IN SCOPE",
        "boundary": "Use the approved Phase 3 target population; do not silently add out-of-scope matches."
    },
    {
        "area": "Competition separation",
        "scope": "IPL and T20I separately",
        "status": "IN SCOPE",
        "boundary": "Do not use cross-competition fallback."
    },
    {
        "area": "Match and innings identifiers",
        "scope": "Match, competition and innings context",
        "status": "USABLE",
        "boundary": "Use observed identifiers and metadata."
    },
    {
        "area": "Batting and bowling teams",
        "scope": "First-innings team context",
        "status": "USABLE",
        "boundary": "Use observed team identifiers."
    },
    {
        "area": "Pre-delivery wickets fallen",
        "scope": "Wickets in hand / wicket state",
        "status": "CONDITIONAL",
        "boundary": "Use only with the Phase 3C reconstruction limitation documented."
    },
    {
        "area": "Cumulative innings score",
        "scope": "Actual score at each delivery before it",
        "status": "UNAVAILABLE",
        "boundary": "Do not derive until delivery-run and extras semantics are resolved."
    },
    {
        "area": "Legal balls remaining",
        "scope": "Remaining legal deliveries",
        "status": "UNAVAILABLE",
        "boundary": "Delivery sequence fields are not legal-ball counters."
    },
    {
        "area": "Scheduled innings length",
        "scope": "Scheduled legal-ball allocation",
        "status": "UNAVAILABLE",
        "boundary": "Sparse revised-overs fields are not a validated common schedule field."
    },
    {
        "area": "Innings stage",
        "scope": "Powerplay, middle and death phases",
        "status": "UNAVAILABLE",
        "boundary": "Do not infer stages from over sequence without validated schedule and legal-ball progression."
    },
    {
        "area": "Expected cumulative scoring benchmark",
        "scope": "Historical first-innings scoring expectation",
        "status": "NOT YET CONSTRUCTED",
        "boundary": "Requires validated score, schedule, stage and comparable-state inputs."
    },
    {
        "area": "Player-level outputs",
        "scope": "Player scores, rankings and selection",
        "status": "OUT OF SCOPE",
        "boundary": "No player metrics, rankings or selections in Phase 3D."
    }
])

print("\n5. Phase 3D scope register")
display(phase3D_scope)

# 6. Basic readiness checks
readiness_checks = {
    "deliveries_dataframe_available":
        "df_deliveries" in globals()
        and isinstance(globals().get("df_deliveries"), pd.DataFrame),

    "pre_delivery_states_dataframe_available":
        "phase3_pre_delivery_wicket_states" in globals()
        and isinstance(globals().get("phase3_pre_delivery_wicket_states"), pd.DataFrame),

    "phase3C_state_schema_available":
        "phase3C_state_schema" in globals()
        and isinstance(globals().get("phase3C_state_schema"), pd.DataFrame),

    "phase3C_exception_register_available":
        "phase3C_exception_register" in globals()
        and isinstance(globals().get("phase3C_exception_register"), pd.DataFrame),
}

phase3D_1_validation = pd.DataFrame([
    {"check": check, "passed": bool(result)}
    for check, result in readiness_checks.items()
])

print("\n6. 3D.1 validation")
display(phase3D_1_validation)

print(
    "\n3D.1 readiness:",
    "READY FOR NEXT AUDIT STEP"
    if phase3D_1_validation["passed"].all()
    else "BLOCKED — inspect missing or invalid objects"
)

1. Required Phase 3 objects


,object,available,type,rows
0,df_deliveries,True,DataFrame,192235
1,phase3_pre_delivery_wicket_states,True,DataFrame,192235
2,phase3C_state_schema,True,DataFrame,12
3,phase3C_exception_register,True,DataFrame,11



2. Current table structure


,object,rows,columns,has_match_id,has_innings,has_comp_id,has_match_format,has_pre_delivery_wickets_fallen
0,df_deliveries,192235,61,True,True,True,True,False
1,phase3_pre_delivery_wicket_states,192235,65,True,True,True,True,True



3. Phase 3C state schema


,field,state_role,data_type,classification,pre_delivery_use,source_dataset,temporal_boundary,reason
0,match_id,Match identifier,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the match containing the delivery.
1,innings,Innings context,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the innings containing the delivery.
2,comp_id,Competition identifier,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the competition.
3,comp_name,Competition context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides competition context.
4,match_format,Format context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides the recorded match format.
5,season,Season context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides season context.
6,batting_team_id,Batting team context,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the batting team.
7,bowling_team_id,Bowling team context,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the bowling team.
8,overid,Delivery sequence context,int64,DIRECTLY OBSERVED,Allowed with limitation,Integrated deliveries,Known immediately before current delivery,"Provides recorded over grouping, not legal balls remaining."
9,over_ball_num,Within-over sequence position,int64,DIRECTLY OBSERVED,Allowed with limitation,Integrated deliveries,Known immediately before current delivery,Sequence position only; not a legal-ball counter.


... (3 outputs left out)

,area,scope,status,boundary
0,Benchmark population,First innings only,IN SCOPE,Use the approved Phase 3 target population; do not silently add out-of-scope matches.
1,Competition separation,IPL and T20I separately,IN SCOPE,Do not use cross-competition fallback.
2,Match and innings identifiers,"Match, competition and innings context",USABLE,Use observed identifiers and metadata.
3,Batting and bowling teams,First-innings team context,USABLE,Use observed team identifiers.
4,Pre-delivery wickets fallen,Wickets in hand / wicket state,CONDITIONAL,Use only with the Phase 3C reconstruction limitation documented.
5,Cumulative innings score,Actual score at each delivery before it,UNAVAILABLE,Do not derive until delivery-run and extras semantics are resolved.
6,Legal balls remaining,Remaining legal deliveries,UNAVAILABLE,Delivery sequence fields are not legal-ball counters.
7,Scheduled innings length,Scheduled legal-ball allocation,UNAVAILABLE,Sparse revised-overs fields are not a validated common schedule field.
8,Innings stage,"Powerplay, middle and death phases",UNAVAILABLE,Do not infer stages from over sequence without validated schedule and legal-ball progression.
9,Expected cumulative scoring benchmark,Historical first-innings scoring expectation,NOT YET CONSTRUCTED,"Requires validated score, schedule, stage and comparable-state inputs."



6. 3D.1 validation


,check,passed
0,deliveries_dataframe_available,True
1,pre_delivery_states_dataframe_available,True
2,phase3C_state_schema_available,True
3,phase3C_exception_register_available,True



3D.1 readiness: READY FOR NEXT AUDIT STEP


### First-innings scoring reconstruction feasibility

In [95]:
# 1. Identify scoring-related columns without assuming their names
candidate_terms = [
    "run", "score", "extra", "wide", "bye", "leg", "penalty",
    "boundary", "four", "six"
]

source_frames = {
    "integrated_deliveries": df_deliveries
}

if "df_source_ball" in globals() and isinstance(df_source_ball, pd.DataFrame):
    source_frames["source_ball_by_ball"] = df_source_ball

scoring_columns = []

for source_name, source_df in source_frames.items():
    for col in source_df.columns:
        if any(term in str(col).lower() for term in candidate_terms):
            scoring_columns.append({
                "source": source_name,
                "column": col,
                "dtype": str(source_df[col].dtype),
                "non_null": int(source_df[col].notna().sum()),
                "null": int(source_df[col].isna().sum()),
                "unique_values": int(source_df[col].nunique(dropna=True))
            })

scoring_schema_audit = pd.DataFrame(scoring_columns)

print("1. Scoring-related fields found")
display(scoring_schema_audit)

# 2. Inspect distributions of the known ambiguous fields
distribution_rows = []

fields_to_inspect = ["runs", "extras1_runs", "extras_2_runs"]

for source_name, source_df in source_frames.items():
    for col in fields_to_inspect:
        if col in source_df.columns:
            series = source_df[col]
            distribution_rows.append({
                "source": source_name,
                "column": col,
                "dtype": str(series.dtype),
                "non_null": int(series.notna().sum()),
                "null": int(series.isna().sum()),
                "minimum": series.min() if pd.api.types.is_numeric_dtype(series) else None,
                "maximum": series.max() if pd.api.types.is_numeric_dtype(series) else None,
                "distinct_values": int(series.nunique(dropna=True))
            })

scoring_field_summary = pd.DataFrame(distribution_rows)

print("\n2. Known scoring-field summaries")
display(scoring_field_summary)

# 3. Inspect common values for ambiguous fields
value_count_tables = {}

for source_name, source_df in source_frames.items():
    for col in fields_to_inspect:
        if col in source_df.columns:
            counts = (
                source_df[col]
                .value_counts(dropna=False)
                .head(20)
                .rename_axis("value")
                .reset_index(name="records")
            )
            value_count_tables[f"{source_name}__{col}"] = counts

for table_name, table in value_count_tables.items():
    print(f"\n3. Value counts — {table_name}")
    display(table)

# 4. Compare matching fields between source and integrated data, if both exist
comparison_rows = []

if "source_ball_by_ball" in source_frames:
    integrated = source_frames["integrated_deliveries"]
    original = source_frames["source_ball_by_ball"]

    for col in fields_to_inspect:
        if col in integrated.columns and col in original.columns:
            comparison_rows.append({
                "column": col,
                "integrated_dtype": str(integrated[col].dtype),
                "source_dtype": str(original[col].dtype),
                "integrated_non_null": int(integrated[col].notna().sum()),
                "source_non_null": int(original[col].notna().sum()),
                "integrated_unique": int(integrated[col].nunique(dropna=True)),
                "source_unique": int(original[col].nunique(dropna=True))
            })

scoring_source_comparison = pd.DataFrame(comparison_rows)

print("\n4. Source versus integrated field comparison")
if not scoring_source_comparison.empty:
    display(scoring_source_comparison)
else:
    print("No directly comparable scoring fields found, or source ball-by-ball table unavailable.")

# 5. First-innings coverage of candidate scoring fields
first_innings_coverage = []

for source_name, source_df in source_frames.items():
    if "innings" not in source_df.columns:
        continue

    first_innings = source_df.loc[source_df["innings"].eq(1)]

    candidate_cols = [
        col for col in source_df.columns
        if any(term in str(col).lower() for term in candidate_terms)
    ]

    for col in candidate_cols:
        first_innings_coverage.append({
            "source": source_name,
            "column": col,
            "first_innings_rows": len(first_innings),
            "non_null": int(first_innings[col].notna().sum()),
            "null": int(first_innings[col].isna().sum()),
            "coverage_pct": round(
                first_innings[col].notna().mean() * 100, 4
            ) if len(first_innings) else None
        })

first_innings_scoring_coverage = pd.DataFrame(first_innings_coverage)

print("\n5. First-innings scoring-field coverage")
display(first_innings_scoring_coverage)

# 6. Preliminary feasibility decision — deliberately conservative
known_ambiguous_fields_present = (
    "runs" in df_deliveries.columns
    and "extras1_runs" in df_deliveries.columns
    and "extras_2_runs" in df_deliveries.columns
)

alternative_total_fields = [
    col for col in df_deliveries.columns
    if any(term in str(col).lower() for term in [
        "total_runs", "innings_total", "delivery_total", "team_runs"
    ])
]

phase3D_2_preliminary = pd.DataFrame([
    {
        "check": "Known ambiguous fields present",
        "result": bool(known_ambiguous_fields_present),
        "interpretation": "Presence does not resolve field semantics."
    },
    {
        "check": "Alternative explicit total-run fields found",
        "result": bool(alternative_total_fields),
        "interpretation": (
            ", ".join(alternative_total_fields)
            if alternative_total_fields
            else "No obvious alternative total-run field found."
        )
    },
    {
        "check": "Cumulative score reconstruction approved",
        "result": False,
        "interpretation": "Not approved by this descriptive audit; semantic validation is still required."
    }
])

print("\n6. Preliminary feasibility status")
display(phase3D_2_preliminary)

1. Scoring-related fields found


,source,column,dtype,non_null,null,unique_values
0,integrated_deliveries,runs,int64,192235,0,7
1,integrated_deliveries,is_four,bool,192235,0,2
2,integrated_deliveries,is_six,bool,192235,0,2
3,integrated_deliveries,extras_type,object,11373,180862,6
4,integrated_deliveries,extras1_runs,int64,192235,0,7
5,integrated_deliveries,extras_2_runs,int64,192235,0,3
6,integrated_deliveries,legal_ball_evidence_class,object,192235,0,7
7,integrated_deliveries,legal_ball_evidence_status,object,192235,0,3
8,integrated_deliveries,legal_ball_final_treatment,object,192235,0,3
9,source_ball_by_ball,runs,int64,200999,0,7



2. Known scoring-field summaries


,source,column,dtype,non_null,null,minimum,maximum,distinct_values
0,integrated_deliveries,runs,int64,192235,0,0,6,7
1,integrated_deliveries,extras1_runs,int64,192235,0,1,7,7
2,integrated_deliveries,extras_2_runs,int64,192235,0,0,4,3
3,source_ball_by_ball,runs,int64,200999,0,0,6,7
4,source_ball_by_ball,extras1_runs,int64,200999,0,1,7,7
5,source_ball_by_ball,extras_2_runs,int64,200999,0,0,4,3



3. Value counts — integrated_deliveries__runs


,value,records
0,0,75779
1,1,68986
2,4,22490
3,6,12296
4,2,12036
5,3,626
6,5,22


... (12 outputs left out)


5. First-innings scoring-field coverage


,source,column,first_innings_rows,non_null,null,coverage_pct
0,integrated_deliveries,runs,100333,100333,0,100.0000
1,integrated_deliveries,is_four,100333,100333,0,100.0000
2,integrated_deliveries,is_six,100333,100333,0,100.0000
3,integrated_deliveries,extras_type,100333,6035,94298,6.0150
4,integrated_deliveries,extras1_runs,100333,100333,0,100.0000
5,integrated_deliveries,extras_2_runs,100333,100333,0,100.0000
6,integrated_deliveries,legal_ball_evidence_class,100333,100333,0,100.0000
7,integrated_deliveries,legal_ball_evidence_status,100333,100333,0,100.0000
8,integrated_deliveries,legal_ball_final_treatment,100333,100333,0,100.0000
9,source_ball_by_ball,runs,105727,105727,0,100.0000



6. Preliminary feasibility status


,check,result,interpretation
0,Known ambiguous fields present,True,Presence does not resolve field semantics.
1,Alternative explicit total-run fields found,False,No obvious alternative total-run field found.
2,Cumulative score reconstruction approved,False,Not approved by this descriptive audit; semantic validation is still required.


#### Inspect scoring fields jointly

In [96]:
required_scoring_fields = [
    "match_id", "innings", "overid", "over_ball_num",
    "runs", "extras1_runs", "extras_2_runs",
    "extras_type", "is_four", "is_six"
]

missing_fields = [
    col for col in required_scoring_fields
    if col not in df_deliveries.columns
]

print("1. Required fields missing from integrated deliveries")
display(pd.DataFrame({"missing_field": missing_fields}))

if missing_fields:
    print("STOP: Do not interpret scoring fields until the missing columns are resolved.")
else:
    # 2. Cross-tabulate the extras code against runs and extras_2_runs.
    extras_joint_summary = (
        df_deliveries
        .groupby(
            ["extras1_runs", "extras_type", "runs", "extras_2_runs"],
            dropna=False
        )
        .size()
        .reset_index(name="record_count")
        .sort_values(
            ["extras1_runs", "extras_type", "runs", "extras_2_runs"],
            na_position="first"
        )
    )

    print("\n2. Joint distribution: extras code, extras type, runs and extras_2_runs")
    display(extras_joint_summary)

    # 3. Summarise runs and extras_2_runs by extras code.
    extras_code_summary = (
        df_deliveries
        .groupby("extras1_runs", dropna=False)
        .agg(
            records=("match_id", "size"),
            distinct_runs=("runs", "nunique"),
            runs_min=("runs", "min"),
            runs_max=("runs", "max"),
            runs_sum=("runs", "sum"),
            distinct_extras2=("extras_2_runs", "nunique"),
            extras2_min=("extras_2_runs", "min"),
            extras2_max=("extras_2_runs", "max"),
            extras2_sum=("extras_2_runs", "sum"),
            extras_type_values=(
                "extras_type",
                lambda s: " | ".join(
                    sorted(s.dropna().astype(str).unique())
                )
            )
        )
        .reset_index()
        .sort_values("extras1_runs")
    )

    print("\n3. Summary by extras code")
    display(extras_code_summary)

    # 4. Inspect representative records for each extras code.
    sample_frames = []

    for code in sorted(df_deliveries["extras1_runs"].dropna().unique()):
        subset = df_deliveries.loc[
            df_deliveries["extras1_runs"].eq(code)
        ].head(5)

        sample_frames.append(subset[required_scoring_fields].assign(
            sample_extras_code=code
        ))

    scoring_examples = pd.concat(sample_frames, ignore_index=True)

    print("\n4. Representative records by extras code (up to 5 each)")
    display(scoring_examples)

    # 5. Check boundary flags against the runs field.
    boundary_audit = pd.DataFrame([
        {
            "check": "is_four=True but runs != 4",
            "records": int(
                (df_deliveries["is_four"] & df_deliveries["runs"].ne(4)).sum()
            )
        },
        {
            "check": "runs=4 but is_four=False",
            "records": int(
                (df_deliveries["runs"].eq(4) & ~df_deliveries["is_four"]).sum()
            )
        },
        {
            "check": "is_six=True but runs != 6",
            "records": int(
                (df_deliveries["is_six"] & df_deliveries["runs"].ne(6)).sum()
            )
        },
        {
            "check": "runs=6 but is_six=False",
            "records": int(
                (df_deliveries["runs"].eq(6) & ~df_deliveries["is_six"]).sum()
            )
        },
        {
            "check": "is_four and is_six both True",
            "records": int(
                (df_deliveries["is_four"] & df_deliveries["is_six"]).sum()
            )
        }
    ])

    print("\n5. Boundary flag consistency checks")
    display(boundary_audit)

    # 6. Keep the first-innings view separate.
    first_innings_extras_summary = (
        df_deliveries.loc[df_deliveries["innings"].eq(1)]
        .groupby("extras1_runs", dropna=False)
        .agg(
            records=("match_id", "size"),
            runs_values=("runs", "nunique"),
            extras2_values=("extras_2_runs", "nunique"),
            extras2_nonzero=("extras_2_runs", lambda s: int(s.ne(0).sum()))
        )
        .reset_index()
        .sort_values("extras1_runs")
    )

    print("\n6. First-innings summary by extras code")
    display(first_innings_extras_summary)

1. Required fields missing from integrated deliveries


,missing_field



2. Joint distribution: extras code, extras type, runs and extras_2_runs


,extras1_runs,extras_type,runs,extras_2_runs,record_count
0,1,Overstep,0,0,211
1,1,Overstep,0,1,12
2,1,Overstep,0,4,9
3,1,Overstep,1,0,166
4,1,Overstep,2,0,33
5,1,Overstep,3,0,2
6,1,Overstep,4,0,79
7,1,Overstep,6,0,45
8,2,wide,0,0,7419
9,3,legbyes,0,0,2635



3. Summary by extras code


,extras1_runs,records,distinct_runs,runs_min,runs_max,runs_sum,distinct_extras2,extras2_min,extras2_max,extras2_sum,extras_type_values
0,1,557,6,0,6,824,3,0,4,48,Overstep
1,2,7419,1,0,0,0,1,0,0,0,wide
2,3,2635,1,0,0,0,1,0,0,0,legbyes
3,4,505,1,0,0,0,1,0,0,0,byes
4,5,256,6,0,6,403,3,0,4,34,other
5,6,180862,7,0,6,257555,2,0,1,1,
6,7,1,1,0,0,0,1,0,0,0,Penalty


... (2 outputs left out)


5. Boundary flag consistency checks


,check,records
0,is_four=True but runs != 4,0
1,runs=4 but is_four=False,20
2,is_six=True but runs != 6,0
3,runs=6 but is_six=False,3
4,is_four and is_six both True,0



6. First-innings summary by extras code


,extras1_runs,records,runs_values,extras2_values,extras2_nonzero
0,1,296,6,3,11
1,2,3925,1,1,0
2,3,1392,1,1,0
3,4,286,1,1,0
4,5,136,6,3,8
5,6,94298,7,2,1


#### Check for independent match-level score evidence

In [97]:
if "df_source_matches" not in globals() or not isinstance(df_source_matches, pd.DataFrame):
    print("BLOCKED: df_source_matches is not available as a DataFrame.")

else:
    # 1. Inspect all match-level columns
    match_schema = pd.DataFrame({
        "column": df_source_matches.columns,
        "dtype": [str(df_source_matches[col].dtype) for col in df_source_matches.columns],
        "non_null": [int(df_source_matches[col].notna().sum()) for col in df_source_matches.columns],
        "null": [int(df_source_matches[col].isna().sum()) for col in df_source_matches.columns],
        "unique_values": [int(df_source_matches[col].nunique(dropna=True)) for col in df_source_matches.columns]
    })

    print("1. Match-level source schema")
    display(match_schema)

    # 2. Identify potentially relevant score and innings fields
    terms = [
        "run", "score", "total", "innings", "target",
        "over", "result", "win", "match_id"
    ]

    relevant_match_columns = [
        col for col in df_source_matches.columns
        if any(term in str(col).lower() for term in terms)
    ]

    print("\n2. Potentially relevant match-level columns")
    display(match_schema.loc[match_schema["column"].isin(relevant_match_columns)])

    # 3. Show sample values for relevant fields
    sample_columns = [
        col for col in relevant_match_columns
        if col in df_source_matches.columns
    ]

    print("\n3. Sample match-level records")
    display(df_source_matches[sample_columns].head(15))

    # 4. Check whether match_id is present and unique
    if "match_id" in df_source_matches.columns:
        match_id_audit = pd.DataFrame([{
            "rows": len(df_source_matches),
            "non_null_match_ids": int(df_source_matches["match_id"].notna().sum()),
            "unique_match_ids": int(df_source_matches["match_id"].nunique(dropna=True)),
            "duplicate_match_id_rows": int(
                df_source_matches["match_id"].duplicated(keep=False).sum()
            )
        }])

        print("\n4. Match identifier integrity")
        display(match_id_audit)
    else:
        print("\n4. No match_id field found in match-level source.")

1. Match-level source schema


,column,dtype,non_null,null,unique_values
0,match_id,int64,883,0,883
1,comp_id,int64,883,0,138
2,match_number,int64,883,0,75
3,match_date,object,883,0,686
4,stage,object,883,0,7
5,venue_id,int64,883,0,98
6,venue_country,object,883,0,17
7,home_team_id,int64,883,0,36
8,home_team,object,883,0,36
9,away_team_id,int64,883,0,36



2. Potentially relevant match-level columns


,column,dtype,non_null,null,unique_values
0,match_id,int64,883,0,883
13,toss_win_team_id,float64,868,15,37
16,first_inn_revised_overs,float64,25,858,13
17,second_inn_revised_overs,float64,25,858,12
18,revised_target,float64,23,860,23
20,win_team_id,float64,842,41,36
21,result,object,883,0,670



3. Sample match-level records


,match_id,toss_win_team_id,first_inn_revised_overs,second_inn_revised_overs,revised_target,win_team_id,result
0,13511,28.0,NaN,NaN,NaN,28.0,West Indies Won by 9 Wickets
1,13512,28.0,NaN,NaN,NaN,22.0,England Won by 1 Run
2,13513,22.0,NaN,NaN,NaN,28.0,West Indies Won by 20 runs
3,13514,28.0,NaN,NaN,NaN,22.0,England Won by 34 runs
4,13515,28.0,NaN,NaN,NaN,28.0,West Indies Won by 17 runs
5,13522,18.0,NaN,NaN,NaN,18.0,India Won by 6 Wickets
6,13523,28.0,NaN,NaN,NaN,18.0,India Won by 8 Runs
7,13524,28.0,NaN,NaN,NaN,18.0,India Won by 17 runs
8,13612,27.0,NaN,114.0,143.0,20.0,Australia Won by 20 runs (D/L method)
9,13613,27.0,NaN,NaN,NaN,20.0,Australia Won the one over Eliminator



4. Match identifier integrity


,rows,non_null_match_ids,unique_match_ids,duplicate_match_id_rows
0,883,883,883,0


### Scheduled innings length and stage feasibility

In [98]:
# 1. Approved match population from the existing Phase 3 benchmark population
print("1. Phase 3 benchmark population structure")
display(pd.DataFrame({
    "column": phase3_benchmark_population.columns,
    "dtype": [str(phase3_benchmark_population[c].dtype)
              for c in phase3_benchmark_population.columns]
}))

display(phase3_benchmark_population.head())

# 2. Revised-overs field coverage in the approved population
population_ids = phase3_benchmark_population["match_id"].drop_duplicates()

revised_fields = [
    "match_id",
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

available_revised_fields = [
    c for c in revised_fields if c in df_source_matches.columns
]

population_match_metadata = (
    population_ids.to_frame()
    .merge(
        df_source_matches[available_revised_fields],
        on="match_id",
        how="left",
        validate="one_to_one",
        indicator=True
    )
)

print("\n2. Approved population match-metadata join")
display(population_match_metadata["_merge"].value_counts().rename_axis("merge_status").reset_index(name="matches"))

revised_coverage = pd.DataFrame([
    {
        "field": c,
        "population_matches": len(population_match_metadata),
        "non_null_matches": int(population_match_metadata[c].notna().sum()),
        "missing_matches": int(population_match_metadata[c].isna().sum()),
        "coverage_pct": round(population_match_metadata[c].notna().mean() * 100, 3)
    }
    for c in available_revised_fields if c != "match_id"
])

print("\n3. Revised field coverage in approved population")
display(revised_coverage)

# 3. Distribution of populated revised-overs values
for col in ["first_inn_revised_overs", "second_inn_revised_overs"]:
    if col in population_match_metadata.columns:
        print(f"\n4. Populated values — {col}")
        display(
            population_match_metadata[col]
            .dropna()
            .value_counts()
            .sort_index()
            .rename_axis("recorded_value")
            .reset_index(name="matches")
        )

# 4. Delivery sequence range by first innings and competition format
first_innings_deliveries = df_deliveries.loc[
    df_deliveries["innings"].eq(1)
].copy()

sequence_summary = (
    first_innings_deliveries
    .groupby(["match_id", "comp_id", "match_format"], dropna=False)
    .agg(
        delivery_records=("innings_ball_num", "size"),
        max_innings_sequence=("innings_ball_num", "max"),
        max_overid=("overid", "max"),
        distinct_overids=("overid", "nunique")
    )
    .reset_index()
)

print("\n5. First-innings delivery-sequence summary")
display(sequence_summary[
    ["delivery_records", "max_innings_sequence", "max_overid", "distinct_overids"]
].describe().T)

# 5. Inspect sequence patterns without treating them as legal-ball counts
print("\n6. First-innings sequence examples")
display(
    sequence_summary
    .sort_values("max_overid")
    .head(10)
)

print("\n7. Largest recorded overid examples")
display(
    sequence_summary
    .sort_values("max_overid", ascending=False)
    .head(10)
)

# 6. Check whether revised-overs metadata is internally present on delivery rows
delivery_revised_fields = [
    c for c in [
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target"
    ]
    if c in df_deliveries.columns
]

print("\n8. Revised fields available directly in delivery table")
display(pd.DataFrame({
    "field": delivery_revised_fields
}))

1. Phase 3 benchmark population structure


,column,dtype
0,match_id,int64
1,comp_id,int64
2,comp_name,object
3,match_format,object
4,season,object
5,benchmark_population,object


,match_id,comp_id,comp_name,match_format,season,benchmark_population
0,13511,1331,England in West Indies T20I Series 2022,T20I,2022,T20I
1,13512,1331,England in West Indies T20I Series 2022,T20I,2022,T20I
2,13513,1331,England in West Indies T20I Series 2022,T20I,2022,T20I
3,13514,1331,England in West Indies T20I Series 2022,T20I,2022,T20I
4,13515,1331,England in West Indies T20I Series 2022,T20I,2022,T20I



2. Approved population match-metadata join


,merge_status,matches
0,both,813
1,left_only,0
2,right_only,0



3. Revised field coverage in approved population


... (9 outputs left out)


7. Largest recorded overid examples


,match_id,comp_id,match_format,delivery_records,max_innings_sequence,max_overid,distinct_overids
811,55560,3787,T20I,126,126,20,20
0,13511,1331,T20I,121,121,20,20
1,13512,1331,T20I,128,128,20,20
2,13513,1331,T20I,125,125,20,20
3,13514,1331,T20I,128,128,20,20
4,13515,1331,T20I,127,127,20,20
5,13522,1335,T20I,131,131,20,20
6,13523,1335,T20I,130,130,20,20
7,13524,1335,T20I,124,124,20,20
8,13612,1345,T20I,124,124,20,20



8. Revised fields available directly in delivery table


,field
0,first_inn_revised_overs
1,second_inn_revised_overs
2,revised_target


#### Validate revised-overs metadata consistency

In [99]:
revised_fields = [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

# 1. Confirm fields exist in both datasets
field_presence = pd.DataFrame([
    {
        "field": field,
        "match_source_present": field in df_source_matches.columns,
        "delivery_source_present": field in df_deliveries.columns
    }
    for field in revised_fields
])

print("1. Revised-field presence")
display(field_presence)

# 2. Summarise delivery-level coverage
delivery_coverage_rows = []

for field in revised_fields:
    if field in df_deliveries.columns:
        values = df_deliveries[field]
        delivery_coverage_rows.append({
            "field": field,
            "delivery_rows": len(df_deliveries),
            "non_null_rows": int(values.notna().sum()),
            "null_rows": int(values.isna().sum()),
            "coverage_pct": round(values.notna().mean() * 100, 4),
            "distinct_non_null_values": int(values.nunique(dropna=True))
        })

delivery_revised_coverage = pd.DataFrame(delivery_coverage_rows)

print("\n2. Delivery-level coverage")
display(delivery_revised_coverage)

# 3. Check within-match consistency in delivery data
within_match_rows = []

for field in revised_fields:
    if field in df_deliveries.columns:
        per_match = (
            df_deliveries.groupby("match_id")[field]
            .agg(
                non_null_rows="count",
                distinct_values=lambda s: s.nunique(dropna=True)
            )
            .reset_index()
        )

        within_match_rows.append({
            "field": field,
            "matches": len(per_match),
            "matches_with_any_value": int(per_match["non_null_rows"].gt(0).sum()),
            "matches_with_multiple_values": int(per_match["distinct_values"].gt(1).sum())
        })

within_match_consistency = pd.DataFrame(within_match_rows)

print("\n3. Within-match consistency")
display(within_match_consistency)

# 4. Compare one match-level value against delivery-level values
comparison_results = []

for field in revised_fields:
    if field not in df_source_matches.columns or field not in df_deliveries.columns:
        continue

    match_values = df_source_matches[["match_id", field]].copy()
    delivery_values = (
        df_deliveries.groupby("match_id")[field]
        .agg(
            delivery_non_null="count",
            delivery_distinct=lambda s: s.nunique(dropna=True),
            delivery_value=lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA
        )
        .reset_index()
    )

    joined = match_values.merge(
        delivery_values,
        on="match_id",
        how="left",
        validate="one_to_one",
        indicator=True
    )

    both_populated = joined[field].notna() & joined["delivery_value"].notna()

    comparison_results.append({
        "field": field,
        "matches": len(joined),
        "both_populated": int(both_populated.sum()),
        "same_value": int(
            (joined.loc[both_populated, field] ==
             joined.loc[both_populated, "delivery_value"]).sum()
        ),
        "different_value": int(
            (joined.loc[both_populated, field] !=
             joined.loc[both_populated, "delivery_value"]).sum()
        ),
        "match_only_populated": int(
            (joined[field].notna() & joined["delivery_value"].isna()).sum()
        ),
        "delivery_only_populated": int(
            (joined[field].isna() & joined["delivery_value"].notna()).sum()
        )
    })

revised_field_comparison = pd.DataFrame(comparison_results)

print("\n4. Match-level versus delivery-level comparison")
display(revised_field_comparison)

# 5. Show examples where the metadata is populated
for field in revised_fields:
    if field in df_source_matches.columns:
        populated = df_source_matches.loc[
            df_source_matches[field].notna(),
            ["match_id", field]
        ]

        print(f"\n5. Populated match-level examples — {field}")
        display(populated.head(15))

# 6. Explicitly record what this audit can and cannot establish
phase3D_3_1_scope = pd.DataFrame([
    {
        "question": "Are revised fields present?",
        "answer": "Checked",
        "limitation": "Presence does not establish field meaning or units."
    },
    {
        "question": "Are values repeated consistently within matches?",
        "answer": "Checked",
        "limitation": "Consistency does not establish correctness."
    },
    {
        "question": "Do match-level and delivery-level values agree?",
        "answer": "Checked",
        "limitation": "Agreement does not establish scheduled legal-ball allocation."
    },
    {
        "question": "Can scheduled innings length be established?",
        "answer": "No",
        "limitation": "Requires authoritative field semantics and reliable population coverage."
    },
    {
        "question": "Can innings stages be assigned?",
        "answer": "No",
        "limitation": "Requires validated legal-ball progression and scheduled length."
    }
])

print("\n6. Audit interpretation boundaries")
display(phase3D_3_1_scope)

1. Revised-field presence


,field,match_source_present,delivery_source_present
0,first_inn_revised_overs,True,True
1,second_inn_revised_overs,True,True
2,revised_target,True,True



2. Delivery-level coverage


,field,delivery_rows,non_null_rows,null_rows,coverage_pct,distinct_non_null_values
0,first_inn_revised_overs,192235,2156,190079,1.1215,11
1,second_inn_revised_overs,192235,3627,188608,1.8868,10
2,revised_target,192235,3627,188608,1.8868,22



3. Within-match consistency


,field,matches,matches_with_any_value,matches_with_multiple_values
0,first_inn_revised_overs,813,15,0
1,second_inn_revised_overs,813,22,0
2,revised_target,813,22,0


... (6 outputs left out)


5. Populated match-level examples — revised_target


,match_id,revised_target
8,13612,143.0
144,16754,56.0
152,17395,91.0
177,17765,130.0
262,21586,171.0
272,21070,104.0
273,21071,203.0
307,28046,152.0
330,29984,126.0
439,37354,94.0



6. Audit interpretation boundaries


,question,answer,limitation
0,Are revised fields present?,Checked,Presence does not establish field meaning or units.
1,Are values repeated consistently within matches?,Checked,Consistency does not establish correctness.
2,Do match-level and delivery-level values agree?,Checked,Agreement does not establish scheduled legal-ball allocation.
3,Can scheduled innings length be established?,No,Requires authoritative field semantics and reliable population coverage.
4,Can innings stages be assigned?,No,Requires validated legal-ball progression and scheduled length.


#### Record schedule and stage feasibility findings

In [100]:
phase3D_3_exception_register = pd.DataFrame([
    {
        "exception_id": "3D-EX-001",
        "field": "scheduled_innings_length",
        "status": "UNAVAILABLE",
        "evidence": "No validated common scheduled-length field. Revised first-innings values exist for only 15 of 813 approved matches.",
        "impact": "Cannot establish a comparable scheduled innings length across the benchmark population.",
        "treatment": "Exclude from benchmark construction until field semantics, units and coverage are validated."
    },
    {
        "exception_id": "3D-EX-002",
        "field": "second_innings_revised_overs",
        "status": "SPARSE / UNRESOLVED",
        "evidence": "Populated for 22 of 813 approved matches. Match-level and delivery-level values agree, but units and interpretation are unresolved.",
        "impact": "Cannot establish a common scheduled-length definition.",
        "treatment": "Retain as audit evidence only; do not convert or use as a common schedule."
    },
    {
        "exception_id": "3D-EX-003",
        "field": "first_inn_revised_overs",
        "status": "SPARSE / UNRESOLVED",
        "evidence": "Populated for 15 of 813 approved matches. Match-level and delivery-level values agree, but units and interpretation are unresolved.",
        "impact": "Cannot establish a common first-innings scheduled length.",
        "treatment": "Retain as audit evidence only; do not convert or use as a common schedule."
    },
    {
        "exception_id": "3D-EX-004",
        "field": "innings_stage",
        "status": "UNAVAILABLE",
        "evidence": "Legal-ball progression and scheduled innings length remain unvalidated.",
        "impact": "Powerplay, middle and death stages cannot be assigned reliably.",
        "treatment": "Do not infer or assign innings stages."
    },
    {
        "exception_id": "3D-EX-005",
        "field": "overid",
        "status": "SEQUENCE ONLY",
        "evidence": "Recorded over grouping; not a validated legal-ball counter or schedule field.",
        "impact": "Cannot independently establish balls remaining or innings stage.",
        "treatment": "Retain only as a sequence descriptor."
    },
    {
        "exception_id": "3D-EX-006",
        "field": "innings_ball_num",
        "status": "SEQUENCE ONLY",
        "evidence": "Delivery sequence position; not a validated legal-ball count.",
        "impact": "Cannot independently establish balls remaining or innings stage.",
        "treatment": "Retain only as a sequence descriptor."
    }
])

phase3D_3_validation = pd.DataFrame([
    {
        "check": "Approved population contains 813 unique matches",
        "passed": (
            phase3_benchmark_population["match_id"].nunique() == 813
            and len(phase3_benchmark_population) == 813
        )
    },
    {
        "check": "All approved matches matched source metadata",
        "passed": (
            population_match_metadata["_merge"].eq("both").all()
        )
    },
    {
        "check": "No within-match revised-field conflicts",
        "passed": (
            within_match_consistency["matches_with_multiple_values"].eq(0).all()
        )
    },
    {
        "check": "Match and delivery revised values agree where both populated",
        "passed": (
            revised_field_comparison["different_value"].eq(0).all()
        )
    },
    {
        "check": "Schedule and stage limitations recorded",
        "passed": (
            len(phase3D_3_exception_register) == 6
            and phase3D_3_exception_register["exception_id"].is_unique
        )
    }
])

print("1. Phase 3D.3 exception register")
display(phase3D_3_exception_register)

print("\n2. Phase 3D.3 validation")
display(phase3D_3_validation)

print(
    "\n3D.3 audit closure:",
    "PASSED" if phase3D_3_validation["passed"].all() else "FAILED"
)

1. Phase 3D.3 exception register


,exception_id,field,status,evidence,impact,treatment
0,3D-EX-001,scheduled_innings_length,UNAVAILABLE,No validated common scheduled-length field. Revised first-innings values exist for only 15 of 813 approved matches.,Cannot establish a comparable scheduled innings length across the benchmark population.,"Exclude from benchmark construction until field semantics, units and coverage are validated."
1,3D-EX-002,second_innings_revised_overs,SPARSE / UNRESOLVED,"Populated for 22 of 813 approved matches. Match-level and delivery-level values agree, but units and interpretation are unresolved.",Cannot establish a common scheduled-length definition.,Retain as audit evidence only; do not convert or use as a common schedule.
2,3D-EX-003,first_inn_revised_overs,SPARSE / UNRESOLVED,"Populated for 15 of 813 approved matches. Match-level and delivery-level values agree, but units and interpretation are unresolved.",Cannot establish a common first-innings scheduled length.,Retain as audit evidence only; do not convert or use as a common schedule.
3,3D-EX-004,innings_stage,UNAVAILABLE,Legal-ball progression and scheduled innings length remain unvalidated.,"Powerplay, middle and death stages cannot be assigned reliably.",Do not infer or assign innings stages.
4,3D-EX-005,overid,SEQUENCE ONLY,Recorded over grouping; not a validated legal-ball counter or schedule field.,Cannot independently establish balls remaining or innings stage.,Retain only as a sequence descriptor.
5,3D-EX-006,innings_ball_num,SEQUENCE ONLY,Delivery sequence position; not a validated legal-ball count.,Cannot independently establish balls remaining or innings stage.,Retain only as a sequence descriptor.



2. Phase 3D.3 validation


,check,passed
0,Approved population contains 813 unique matches,True
1,All approved matches matched source metadata,True
2,No within-match revised-field conflicts,True
3,Match and delivery revised values agree where both populated,True
4,Schedule and stage limitations recorded,True



3D.3 audit closure: PASSED


### First-innings benchmark population preparation

#### Build and validate the first-innings population

In [102]:
# Phase 3D.4.1 — Corrected population merge
# Fixes only the indicator-column collision.

# 1. Inspect existing merge-related columns
existing_merge_columns = [
    col for col in phase3D_first_innings_states.columns
    if col == "_merge" or str(col).startswith("_merge")
]

print("1. Existing merge-related columns")
display(pd.DataFrame({"column": existing_merge_columns}))

# 2. Merge using a unique indicator name
phase3D_first_innings_population = (
    phase3D_first_innings_states.merge(
        phase3_benchmark_population[
            required_population_columns
        ],
        on="match_id",
        how="inner",
        validate="many_to_one",
        suffixes=("", "_population"),
        indicator="_population_merge_status"
    )
)

# 3. Verify merge outcome
print("\n2. Population merge result")
display(
    phase3D_first_innings_population[
        "_population_merge_status"
    ].value_counts(dropna=False)
    .rename_axis("merge_status")
    .reset_index(name="rows")
)

print("\n3. Merged population dimensions")
display(pd.DataFrame([{
    "rows": len(phase3D_first_innings_population),
    "unique_matches": phase3D_first_innings_population["match_id"].nunique(),
    "columns": len(phase3D_first_innings_population.columns)
}]))

# 4. Verify metadata agreement
metadata_checks = pd.DataFrame([
    {
        "field": field,
        "mismatches": int(
            phase3D_first_innings_population[field].ne(
                phase3D_first_innings_population[f"{field}_population"]
            ).sum()
        )
    }
    for field in ["comp_id", "match_format"]
])

print("\n4. Population metadata agreement")
display(metadata_checks)

# 5. Re-run delivery-key and team-context audit
delivery_key = [
    "match_id", "innings", "overid",
    "over_ball_num", "innings_ball_num"
]

key_audit = pd.DataFrame([
    {
        "check": "Rows with missing delivery keys",
        "count": int(
            phase3D_first_innings_population[delivery_key]
            .isna().any(axis=1).sum()
        )
    },
    {
        "check": "Duplicate delivery key rows",
        "count": int(
            phase3D_first_innings_population.duplicated(
                delivery_key, keep=False
            ).sum()
        )
    },
    {
        "check": "Rows missing batting team",
        "count": int(
            phase3D_first_innings_population["batting_team_id"].isna().sum()
        )
    },
    {
        "check": "Rows missing bowling team",
        "count": int(
            phase3D_first_innings_population["bowling_team_id"].isna().sum()
        )
    },
    {
        "check": "Rows missing pre-delivery wickets",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].isna().sum()
        )
    }
])

print("\n5. Delivery and team-context audit")
display(key_audit)

# 6. Competition-format summary
format_summary = (
    phase3D_first_innings_population
    .groupby(
        ["benchmark_population", "match_format", "comp_id"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        delivery_records=("match_id", "size")
    )
    .reset_index()
    .sort_values(
        ["benchmark_population", "match_format", "comp_id"]
    )
)

print("\n6. Competition-format summary")
display(format_summary)

overall_format_summary = (
    phase3D_first_innings_population
    .groupby(
        ["benchmark_population", "match_format"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        delivery_records=("match_id", "size")
    )
    .reset_index()
)

print("\n7. Overall population split")
display(overall_format_summary)

# 7. Validation
phase3D_4_validation = pd.DataFrame([
    {
        "check": "First-innings population has 813 matches",
        "passed": phase3D_first_innings_population["match_id"].nunique() == 813
    },
    {
        "check": "All approved population matches represented",
        "passed": (
            phase3D_first_innings_population["match_id"].nunique()
            == phase3_benchmark_population["match_id"].nunique()
        )
    },
    {
        "check": "No competition metadata mismatches",
        "passed": metadata_checks["mismatches"].eq(0).all()
    },
    {
        "check": "No missing or duplicate delivery keys",
        "passed": key_audit.loc[
            key_audit["check"].isin([
                "Rows with missing delivery keys",
                "Duplicate delivery key rows"
            ]),
            "count"
        ].eq(0).all()
    },
    {
        "check": "Batting and bowling team context complete",
        "passed": key_audit.loc[
            key_audit["check"].isin([
                "Rows missing batting team",
                "Rows missing bowling team"
            ]),
            "count"
        ].eq(0).all()
    },
    {
        "check": "Pre-delivery wicket state complete",
        "passed": key_audit.loc[
            key_audit["check"].eq("Rows missing pre-delivery wickets"),
            "count"
        ].eq(0).all()
    }
])

print("\n8. Phase 3D.4.1 validation")
display(phase3D_4_validation)

print(
    "\n3D.4.1 status:",
    "PASSED" if phase3D_4_validation["passed"].all()
    else "FAILED — inspect failed checks"
)

1. Existing merge-related columns


,column
0,_merge



2. Population merge result


,merge_status,rows
0,both,100333
1,left_only,0
2,right_only,0



3. Merged population dimensions


,rows,unique_matches,columns
0,100333,813,69


... (7 outputs left out)

,benchmark_population,match_format,matches,delivery_records
0,IPL,T20,362,45139
1,T20I,T20I,451,55194



8. Phase 3D.4.1 validation


,check,passed
0,First-innings population has 813 matches,True
1,All approved population matches represented,True
2,No competition metadata mismatches,True
3,No missing or duplicate delivery keys,True
4,Batting and bowling team context complete,True
5,Pre-delivery wicket state complete,True



3D.4.1 status: PASSED


#### Population coverage and wicket-state distribution

In [103]:
# 1. Competition-level population coverage
phase3D_competition_coverage = (
    phase3D_first_innings_population
    .groupby(
        ["benchmark_population", "match_format", "comp_id", "comp_name"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        delivery_records=("match_id", "size"),
        seasons=("season", "nunique")
    )
    .reset_index()
    .sort_values(
        ["benchmark_population", "matches", "comp_id"],
        ascending=[True, False, True]
    )
)

print("1. Competition-level coverage")
display(phase3D_competition_coverage)

# 2. Wicket-state distribution by population
phase3D_wicket_distribution = (
    phase3D_first_innings_population
    .groupby(
        ["benchmark_population", "pre_delivery_wickets_fallen"],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["benchmark_population", "pre_delivery_wickets_fallen"]
    )
)

print("\n2. Wicket-state distribution by population")
display(phase3D_wicket_distribution)

# 3. Competition x wicket-state coverage
phase3D_comp_wicket_coverage = (
    phase3D_first_innings_population
    .groupby(
        [
            "benchmark_population",
            "comp_id",
            "comp_name",
            "pre_delivery_wickets_fallen"
        ],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

print("\n3. Competition x wicket-state coverage")
display(phase3D_comp_wicket_coverage)

# 4. Explicitly identify absent competition/wicket-state combinations
expected_wickets = list(range(11))

competition_keys = (
    phase3D_first_innings_population[
        ["benchmark_population", "comp_id", "comp_name"]
    ]
    .drop_duplicates()
)

expected_grid = competition_keys.merge(
    pd.DataFrame({"pre_delivery_wickets_fallen": expected_wickets}),
    how="cross"
)

observed_grid = phase3D_comp_wicket_coverage[
    [
        "benchmark_population",
        "comp_id",
        "comp_name",
        "pre_delivery_wickets_fallen"
    ]
].drop_duplicates()

phase3D_missing_comp_wicket_states = expected_grid.merge(
    observed_grid,
    on=[
        "benchmark_population",
        "comp_id",
        "comp_name",
        "pre_delivery_wickets_fallen"
    ],
    how="left",
    indicator=True,
    validate="one_to_one"
)

phase3D_missing_comp_wicket_states = (
    phase3D_missing_comp_wicket_states.loc[
        phase3D_missing_comp_wicket_states["_merge"].eq("left_only")
    ]
    .drop(columns="_merge")
)

print("\n4. Missing competition/wicket-state combinations")
display(phase3D_missing_comp_wicket_states)

# 5. Wicket-state integrity
phase3D_wicket_integrity = pd.DataFrame([
    {
        "check": "Missing pre-delivery wicket states",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].isna().sum()
        )
    },
    {
        "check": "Wicket states below zero",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].lt(0).sum()
        )
    },
    {
        "check": "Wicket states above ten",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].gt(10).sum()
        )
    },
    {
        "check": "Distinct observed wicket states",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].nunique()
        )
    }
])

print("\n5. Wicket-state integrity")
display(phase3D_wicket_integrity)

# 6. Overall validation
phase3D_4_2_validation = pd.DataFrame([
    {
        "check": "Competition coverage accounts for all 813 matches",
        "passed": (
            phase3D_competition_coverage["matches"].sum() == 813
        )
    },
    {
        "check": "Competition coverage accounts for all 100333 deliveries",
        "passed": (
            phase3D_competition_coverage["delivery_records"].sum() == 100333
        )
    },
    {
        "check": "Wicket states are complete and within 0–10",
        "passed": (
            phase3D_wicket_integrity.loc[
                phase3D_wicket_integrity["check"].isin([
                    "Missing pre-delivery wicket states",
                    "Wicket states below zero",
                    "Wicket states above ten"
                ]),
                "count"
            ].eq(0).all()
        )
    },
    {
        "check": "All observed wicket states are in expected domain",
        "passed": (
            set(
                phase3D_first_innings_population[
                    "pre_delivery_wickets_fallen"
                ].dropna().unique()
            ).issubset(set(expected_wickets))
        )
    }
])

print("\n6. Phase 3D.4.2 validation")
display(phase3D_4_2_validation)

print(
    "\n3D.4.2 status:",
    "PASSED" if phase3D_4_2_validation["passed"].all()
    else "FAILED — inspect failed checks"
)

1. Competition-level coverage


,benchmark_population,match_format,comp_id,comp_name,matches,delivery_records,seasons
0,IPL,T20,1358,IPL 2022,74,9217,1
1,IPL,T20,1715,IPL 2023,73,9124,1
4,IPL,T20,3298,IPL 2026,73,9101,1
2,IPL,T20,2193,IPL 2024,71,8893,1
3,IPL,T20,2725,IPL 2025,71,8804,1
...,...,...,...,...,...,...,...
125,T20I,T20I,3446,India in Ireland T20I Series 2026,2,250,1
88,T20I,T20I,2801,Ireland in Zimbabwe T20I Series 2025,1,121,1
89,T20I,T20I,2873,West Indies in Ireland T20I Series 2025,1,129,1
110,T20I,T20I,3226,England in New Zealand T20I Series 2025,1,125,1



2. Wicket-state distribution by population


,benchmark_population,pre_delivery_wickets_fallen,delivery_records,matches,competitions
0,IPL,0,9187,362,5
1,IPL,1,8393,361,5
2,IPL,2,7602,360,5
3,IPL,3,6502,347,5
4,IPL,4,5500,315,5
5,IPL,5,3802,277,5
6,IPL,6,2080,214,5
7,IPL,7,974,145,5
8,IPL,8,702,101,5
9,IPL,9,397,52,5



3. Competition x wicket-state coverage


,benchmark_population,comp_id,comp_name,pre_delivery_wickets_fallen,delivery_records,matches
0,IPL,1358,IPL 2022,0,1759,74
1,IPL,1358,IPL 2022,1,1975,73
2,IPL,1358,IPL 2022,2,1561,73
3,IPL,1358,IPL 2022,3,1278,70
4,IPL,1358,IPL 2022,4,1222,64
...,...,...,...,...,...,...
1236,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,5,93,3
1237,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,6,20,3
1238,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,7,26,3
1239,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,8,8,2


... (3 outputs left out)

,check,count
0,Missing pre-delivery wicket states,0
1,Wicket states below zero,0
2,Wicket states above ten,0
3,Distinct observed wicket states,10



6. Phase 3D.4.2 validation


,check,passed
0,Competition coverage accounts for all 813 matches,True
1,Competition coverage accounts for all 100333 deliveries,True
2,Wicket states are complete and within 0–10,True
3,All observed wicket states are in expected domain,True



3D.4.2 status: PASSED


#### Population coverage and wicket-state distribution

In [104]:
# 1. Competition-level population coverage
phase3D_competition_coverage = (
    phase3D_first_innings_population
    .groupby(
        ["benchmark_population", "match_format", "comp_id", "comp_name"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        delivery_records=("match_id", "size"),
        seasons=("season", "nunique")
    )
    .reset_index()
    .sort_values(
        ["benchmark_population", "matches", "comp_id"],
        ascending=[True, False, True]
    )
)

print("1. Competition-level coverage")
display(phase3D_competition_coverage)

# 2. Wicket-state distribution by population
phase3D_wicket_distribution = (
    phase3D_first_innings_population
    .groupby(
        ["benchmark_population", "pre_delivery_wickets_fallen"],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["benchmark_population", "pre_delivery_wickets_fallen"]
    )
)

print("\n2. Wicket-state distribution by population")
display(phase3D_wicket_distribution)

# 3. Competition x wicket-state coverage
phase3D_comp_wicket_coverage = (
    phase3D_first_innings_population
    .groupby(
        [
            "benchmark_population",
            "comp_id",
            "comp_name",
            "pre_delivery_wickets_fallen"
        ],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

print("\n3. Competition x wicket-state coverage")
display(phase3D_comp_wicket_coverage)

# 4. Explicitly identify absent competition/wicket-state combinations
expected_wickets = list(range(11))

competition_keys = (
    phase3D_first_innings_population[
        ["benchmark_population", "comp_id", "comp_name"]
    ]
    .drop_duplicates()
)

expected_grid = competition_keys.merge(
    pd.DataFrame({"pre_delivery_wickets_fallen": expected_wickets}),
    how="cross"
)

observed_grid = phase3D_comp_wicket_coverage[
    [
        "benchmark_population",
        "comp_id",
        "comp_name",
        "pre_delivery_wickets_fallen"
    ]
].drop_duplicates()

phase3D_missing_comp_wicket_states = expected_grid.merge(
    observed_grid,
    on=[
        "benchmark_population",
        "comp_id",
        "comp_name",
        "pre_delivery_wickets_fallen"
    ],
    how="left",
    indicator=True,
    validate="one_to_one"
)

phase3D_missing_comp_wicket_states = (
    phase3D_missing_comp_wicket_states.loc[
        phase3D_missing_comp_wicket_states["_merge"].eq("left_only")
    ]
    .drop(columns="_merge")
)

print("\n4. Missing competition/wicket-state combinations")
display(phase3D_missing_comp_wicket_states)

# 5. Wicket-state integrity
phase3D_wicket_integrity = pd.DataFrame([
    {
        "check": "Missing pre-delivery wicket states",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].isna().sum()
        )
    },
    {
        "check": "Wicket states below zero",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].lt(0).sum()
        )
    },
    {
        "check": "Wicket states above ten",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].gt(10).sum()
        )
    },
    {
        "check": "Distinct observed wicket states",
        "count": int(
            phase3D_first_innings_population[
                "pre_delivery_wickets_fallen"
            ].nunique()
        )
    }
])

print("\n5. Wicket-state integrity")
display(phase3D_wicket_integrity)

# 6. Overall validation
phase3D_4_2_validation = pd.DataFrame([
    {
        "check": "Competition coverage accounts for all 813 matches",
        "passed": (
            phase3D_competition_coverage["matches"].sum() == 813
        )
    },
    {
        "check": "Competition coverage accounts for all 100333 deliveries",
        "passed": (
            phase3D_competition_coverage["delivery_records"].sum() == 100333
        )
    },
    {
        "check": "Wicket states are complete and within 0–10",
        "passed": (
            phase3D_wicket_integrity.loc[
                phase3D_wicket_integrity["check"].isin([
                    "Missing pre-delivery wicket states",
                    "Wicket states below zero",
                    "Wicket states above ten"
                ]),
                "count"
            ].eq(0).all()
        )
    },
    {
        "check": "All observed wicket states are in expected domain",
        "passed": (
            set(
                phase3D_first_innings_population[
                    "pre_delivery_wickets_fallen"
                ].dropna().unique()
            ).issubset(set(expected_wickets))
        )
    }
])

print("\n6. Phase 3D.4.2 validation")
display(phase3D_4_2_validation)

print(
    "\n3D.4.2 status:",
    "PASSED" if phase3D_4_2_validation["passed"].all()
    else "FAILED — inspect failed checks"
)

1. Competition-level coverage


,benchmark_population,match_format,comp_id,comp_name,matches,delivery_records,seasons
0,IPL,T20,1358,IPL 2022,74,9217,1
1,IPL,T20,1715,IPL 2023,73,9124,1
4,IPL,T20,3298,IPL 2026,73,9101,1
2,IPL,T20,2193,IPL 2024,71,8893,1
3,IPL,T20,2725,IPL 2025,71,8804,1
...,...,...,...,...,...,...,...
125,T20I,T20I,3446,India in Ireland T20I Series 2026,2,250,1
88,T20I,T20I,2801,Ireland in Zimbabwe T20I Series 2025,1,121,1
89,T20I,T20I,2873,West Indies in Ireland T20I Series 2025,1,129,1
110,T20I,T20I,3226,England in New Zealand T20I Series 2025,1,125,1



2. Wicket-state distribution by population


,benchmark_population,pre_delivery_wickets_fallen,delivery_records,matches,competitions
0,IPL,0,9187,362,5
1,IPL,1,8393,361,5
2,IPL,2,7602,360,5
3,IPL,3,6502,347,5
4,IPL,4,5500,315,5
5,IPL,5,3802,277,5
6,IPL,6,2080,214,5
7,IPL,7,974,145,5
8,IPL,8,702,101,5
9,IPL,9,397,52,5



3. Competition x wicket-state coverage


,benchmark_population,comp_id,comp_name,pre_delivery_wickets_fallen,delivery_records,matches
0,IPL,1358,IPL 2022,0,1759,74
1,IPL,1358,IPL 2022,1,1975,73
2,IPL,1358,IPL 2022,2,1561,73
3,IPL,1358,IPL 2022,3,1278,70
4,IPL,1358,IPL 2022,4,1222,64
...,...,...,...,...,...,...
1236,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,5,93,3
1237,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,6,20,3
1238,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,7,26,3
1239,T20I,3787,Afghanistan v Zimbabwe T20I Series 2026,8,8,2


... (3 outputs left out)

,check,count
0,Missing pre-delivery wicket states,0
1,Wicket states below zero,0
2,Wicket states above ten,0
3,Distinct observed wicket states,10



6. Phase 3D.4.2 validation


,check,passed
0,Competition coverage accounts for all 813 matches,True
1,Competition coverage accounts for all 100333 deliveries,True
2,Wicket states are complete and within 0–10,True
3,All observed wicket states are in expected domain,True



3D.4.2 status: PASSED


#### Match-level population consistency

In [109]:
match_audit_source = phase3D_first_innings_population.copy()

match_level_audit = (
    match_audit_source
    .groupby("match_id", dropna=False)
    .agg(
        competition_count=("comp_id", "nunique"),
        competition_name_count=("comp_name", "nunique"),
        format_count=("match_format", "nunique"),
        season_count=("season", "nunique"),
        population_count=("benchmark_population", "nunique"),
        delivery_records=("match_id", "size"),
        comp_id=("comp_id", "first"),
        comp_name=("comp_name", "first"),
        match_format=("match_format", "first"),
        season=("season", "first"),
        benchmark_population=("benchmark_population", "first")
    )
    .reset_index()
)

inconsistent_matches = match_level_audit[
    (match_level_audit["competition_count"] != 1) |
    (match_level_audit["competition_name_count"] != 1) |
    (match_level_audit["format_count"] != 1) |
    (match_level_audit["season_count"] != 1) |
    (match_level_audit["population_count"] != 1)
].copy()

match_reconciliation = (
    match_level_audit
    .groupby("benchmark_population", dropna=False)
    .agg(
        matches=("match_id", "nunique"),
        delivery_records=("delivery_records", "sum"),
        competitions=("comp_id", "nunique"),
        seasons=("season", "nunique")
    )
    .reset_index()
)

population_match_counts = (
    phase3_benchmark_population
    .groupby("benchmark_population", dropna=False)
    .agg(population_matches=("match_id", "nunique"))
    .reset_index()
)

match_reconciliation = match_reconciliation.merge(
    population_match_counts,
    on="benchmark_population",
    how="outer",
    validate="one_to_one",
    indicator="_population_reconciliation"
)

match_reconciliation["match_count_difference"] = (
    match_reconciliation["matches"] -
    match_reconciliation["population_matches"]
)

phase3D_4_3_validation = pd.DataFrame([
    {
        "check": "Exactly 813 unique first-innings matches",
        "passed": match_level_audit["match_id"].nunique() == 813
    },
    {
        "check": "Every match has one competition, name, format, season and population",
        "passed": len(inconsistent_matches) == 0
    },
    {
        "check": "No missing match-level grouping fields",
        "passed": not match_level_audit[
            ["comp_id", "comp_name", "match_format", "season", "benchmark_population"]
        ].isna().any().any()
    },
    {
        "check": "Match counts reconcile to the approved population",
        "passed": (
            (match_reconciliation["_population_reconciliation"] == "both").all()
            and (match_reconciliation["match_count_difference"] == 0).all()
        )
    },
    {
        "check": "Reconciled delivery records total 100,333",
        "passed": match_reconciliation["delivery_records"].sum() == 100333
    }
])

print("MATCH-LEVEL AUDIT")
print(f"Unique matches: {match_level_audit['match_id'].nunique():,}")
print(f"Inconsistent matches: {len(inconsistent_matches):,}")
print(f"Total delivery records: {match_level_audit['delivery_records'].sum():,}")

print("\nPOPULATION RECONCILIATION")
display(match_reconciliation)

print("\nINCONSISTENT MATCHES")
display(inconsistent_matches.head(25))

print("\n3D.4.3 VALIDATION")
display(phase3D_4_3_validation)

assert phase3D_4_3_validation["passed"].all(), (
    "3D.4.3 validation failed. Inspect the audit tables before proceeding."
)

print("\n3D.4.3 status: PASSED")

MATCH-LEVEL AUDIT
Unique matches: 813
Inconsistent matches: 0
Total delivery records: 100,333

POPULATION RECONCILIATION


,benchmark_population,matches,delivery_records,competitions,seasons,population_matches,_population_reconciliation,match_count_difference
0,IPL,362,45139,5,5,362,both,0
1,T20I,451,55194,131,8,451,both,0



INCONSISTENT MATCHES


,match_id,competition_count,competition_name_count,format_count,season_count,population_count,delivery_records,comp_id,comp_name,match_format,season,benchmark_population



3D.4.3 VALIDATION


,check,passed
0,Exactly 813 unique first-innings matches,True
1,"Every match has one competition, name, format, season and population",True
2,No missing match-level grouping fields,True
3,Match counts reconcile to the approved population,True
4,"Reconciled delivery records total 100,333",True



3D.4.3 status: PASSED


#### Consolidated validation and export

In [110]:
validation_register = pd.concat(
    [
        phase3D_4_validation.assign(subsection="3D.4.1"),
        phase3D_4_2_validation.assign(subsection="3D.4.2"),
        phase3D_4_3_validation.assign(subsection="3D.4.3")
    ],
    ignore_index=True
)

validation_register = validation_register[
    ["subsection", "check", "passed"]
]

phase3D_4_summary = pd.DataFrame([
    {
        "metric": "First-innings matches",
        "value": phase3D_first_innings_population["match_id"].nunique()
    },
    {
        "metric": "First-innings delivery records",
        "value": len(phase3D_first_innings_population)
    },
    {
        "metric": "Competition groups",
        "value": phase3D_first_innings_population["comp_id"].nunique()
    },
    {
        "metric": "IPL matches",
        "value": int((match_level_audit["benchmark_population"] == "IPL").sum())
    },
    {
        "metric": "T20I matches",
        "value": int((match_level_audit["benchmark_population"] == "T20I").sum())
    },
    {
        "metric": "Missing competition/wicket-state combinations",
        "value": len(phase3D_missing_comp_wicket_states)
    },
    {
        "metric": "Inconsistent match-level assignments",
        "value": len(inconsistent_matches)
    },
    {
        "metric": "Passed validation checks",
        "value": int(validation_register["passed"].sum())
    },
    {
        "metric": "Failed validation checks",
        "value": int((~validation_register["passed"]).sum())
    }
])

# Export audit and population-preparation outputs
phase3D_first_innings_population.to_parquet(
    OUTPUT_DIR / "phase3D_first_innings_population.parquet",
    index=False
)

phase3D_competition_coverage.to_csv(
    OUTPUT_DIR / "phase3D_competition_coverage.csv",
    index=False
)

phase3D_wicket_distribution.to_csv(
    OUTPUT_DIR / "phase3D_wicket_distribution.csv",
    index=False
)

phase3D_comp_wicket_coverage.to_csv(
    OUTPUT_DIR / "phase3D_competition_wicket_coverage.csv",
    index=False
)

phase3D_missing_comp_wicket_states.to_csv(
    OUTPUT_DIR / "phase3D_missing_competition_wicket_states.csv",
    index=False
)

validation_register.to_csv(
    OUTPUT_DIR / "phase3D_4_validation_register.csv",
    index=False
)

phase3D_4_summary.to_csv(
    OUTPUT_DIR / "phase3D_4_population_summary.csv",
    index=False
)

print("CONSOLIDATED VALIDATION REGISTER")
display(validation_register)

print("\nPOPULATION PREPARATION SUMMARY")
display(phase3D_4_summary)

print("\nVALIDATION TOTALS")
print(f"Passed: {(validation_register['passed']).sum()}")
print(f"Failed: {(~validation_register['passed']).sum()}")

assert validation_register["passed"].all(), (
    "3D.4 closure blocked: one or more validation checks failed."
)

assert len(phase3D_first_innings_population) == 100333
assert phase3D_first_innings_population["match_id"].nunique() == 813

print("\n3D.4.4 status: PASSED")
print(f"Outputs saved under: {OUTPUT_DIR}")

CONSOLIDATED VALIDATION REGISTER


,subsection,check,passed
0,3D.4.1,First-innings population has 813 matches,True
1,3D.4.1,All approved population matches represented,True
2,3D.4.1,No competition metadata mismatches,True
3,3D.4.1,No missing or duplicate delivery keys,True
4,3D.4.1,Batting and bowling team context complete,True
5,3D.4.1,Pre-delivery wicket state complete,True
6,3D.4.2,Competition coverage accounts for all 813 matches,True
7,3D.4.2,Competition coverage accounts for all 100333 deliveries,True
8,3D.4.2,Wicket states are complete and within 0–10,True
9,3D.4.2,All observed wicket states are in expected domain,True



POPULATION PREPARATION SUMMARY


,metric,value
0,First-innings matches,813
1,First-innings delivery records,100333
2,Competition groups,136
3,IPL matches,362
4,T20I matches,451
5,Missing competition/wicket-state combinations,255
6,Inconsistent match-level assignments,0
7,Passed validation checks,15
8,Failed validation checks,0



VALIDATION TOTALS
Passed: 15
Failed: 0

3D.4.4 status: PASSED
Outputs saved under: /content/phase3_outputs


### Define comparable benchmark states

#### Phase 3D.2

In [111]:
# Phase 3D.2.2 — Inspect original source scoring-related fields

source_columns = pd.DataFrame({
    "source_field": df_source_ball.columns,
    "dtype": [str(df_source_ball[col].dtype) for col in df_source_ball.columns],
    "non_null": [int(df_source_ball[col].notna().sum()) for col in df_source_ball.columns],
    "null": [int(df_source_ball[col].isna().sum()) for col in df_source_ball.columns],
    "distinct_values": [int(df_source_ball[col].nunique(dropna=True)) for col in df_source_ball.columns]
})

scoring_terms = (
    "run", "score", "extra", "four", "six", "boundary", "total", "bat"
)

scoring_field_inventory = source_columns[
    source_columns["source_field"].str.lower().apply(
        lambda name: any(term in name for term in scoring_terms)
    )
].copy()

print("ORIGINAL SOURCE DATASET")
print(f"Rows: {len(df_source_ball):,}")
print(f"Columns: {len(df_source_ball.columns)}")

print("\nALL ORIGINAL SOURCE COLUMNS")
display(source_columns)

print("\nSCORING-RELATED SOURCE FIELD INVENTORY")
display(scoring_field_inventory)

print("\nRAW VALUES FOR CANDIDATE SCORING FIELDS")
for col in scoring_field_inventory["source_field"]:
    print(f"\n{col}")
    print(df_source_ball[col].value_counts(dropna=False).head(20))

ORIGINAL SOURCE DATASET
Rows: 200,999
Columns: 33

ALL ORIGINAL SOURCE COLUMNS


,source_field,dtype,non_null,null,distinct_values
0,match_id,int64,200999,0,867
1,innings,int64,200999,0,4
2,overid,int64,200999,0,20
3,over_ball_num,int64,200999,0,13
4,innings_ball_num,int64,200999,0,136
5,batting_team_id,int64,200999,0,38
6,bowling_team_id,int64,200999,0,38
7,striker_id,int64,200999,0,888
8,striker_name,object,200999,0,885
9,non_striker_id,int64,200999,0,867



SCORING-RELATED SOURCE FIELD INVENTORY


,source_field,dtype,non_null,null,distinct_values
5,batting_team_id,int64,200999,0,38
13,batting_type,object,200999,0,2
15,batting_order,int64,200999,0,13
16,runs,int64,200999,0,7
17,is_four,bool,200999,0,2
18,is_six,bool,200999,0,2
23,extras_type,object,11832,189167,6
24,extras1_runs,int64,200999,0,7
25,extras_2_runs,int64,200999,0,3



RAW VALUES FOR CANDIDATE SCORING FIELDS

batting_team_id
batting_team_id
18      13369
25      10482
28      10407
17       9423
3508     9379
3        9107
5        9025
21       8982
2        8854
3509     8853
4        8778
242      8710
1        8692
27       8434
7        8384
8        7905
22       7451
78       6266
20       6248
72       6231
Name: count, dtype: int64

batting_type
batting_type
RHB    131006
LHB     69993
Name: count, dtype: int64

batting_order
batting_order
2     34488
1     34126
3     31065
4     30068
5     24125
6     18297
7     13091
8      7578
9      4448
10     2554
11     1133
12       17
13        9
Name: count, dtype: int64

runs
runs
0    79387
1    72267
4    23303
6    12814
2    12547
3      656
5       25
Name: count, dtype: int64

is_four
is_four
False    177716
True      23283
Name: count, dtype: int64

is_six
is_six
False    188188
True      12811
Name: count, dtype: int64

extras_type
extras_type
NaN         189167
wide          7714
leg

In [112]:
# Phase 3D.2.3 — Formal scoring reconstruction boundary

phase3D_scoring_boundary_register = pd.DataFrame([
    {
        "field": "runs",
        "evidence": "Integer values 0–6; no independent score field in raw source",
        "limitation": "Cannot establish batter-runs versus total-delivery-runs semantics",
        "approved_treatment": "Preserve raw; do not sum as cumulative innings score",
        "status": "UNRESOLVED"
    },
    {
        "field": "extras1_runs",
        "evidence": "Encoded categories 1–7; category 6 means no extras",
        "limitation": "Field is not a numerical extras-run quantity",
        "approved_treatment": "Use only as categorical evidence; never add as runs",
        "status": "SEMANTICS ESTABLISHED — NOT A RUN QUANTITY"
    },
    {
        "field": "extras_2_runs",
        "evidence": "Values 0, 1 and 4; non-zero semantics undocumented",
        "limitation": "Cannot establish whether or how values contribute to delivery totals",
        "approved_treatment": "Preserve raw; do not add to runs",
        "status": "UNRESOLVED"
    },
    {
        "field": "extras_type",
        "evidence": "Labels include wide, legbyes, Overstep, byes, other and Penalty",
        "limitation": "Category labels do not establish the numeric allocation of runs",
        "approved_treatment": "Retain as categorical evidence only",
        "status": "PARTIALLY INFORMATIVE"
    },
    {
        "field": "is_four / is_six",
        "evidence": "23 boundary-flag mismatches in the primary population",
        "limitation": "Flags cannot independently reconstruct total delivery runs",
        "approved_treatment": "Preserve flags independently; do not use to repair runs",
        "status": "LIMITED"
    },
    {
        "field": "independent innings score",
        "evidence": "No such field found in the original ball-by-ball schema",
        "limitation": "No independent delivery-to-innings score reconciliation available",
        "approved_treatment": "Require authoritative documentation or independent score source",
        "status": "UNAVAILABLE"
    }
])

phase3D_scoring_boundary_validation = pd.DataFrame([
    {
        "check": "Raw source has no separate total-runs or batter-runs field",
        "passed": not any(
            col.lower() in {
                "total_runs", "total_run", "batter_runs",
                "batsman_runs", "innings_score", "cumulative_score"
            }
            for col in df_source_ball.columns
        )
    },
    {
        "check": "All identified scoring-related source fields are documented",
        "passed": set(scoring_field_inventory["source_field"]).issubset(
            set(phase3D_scoring_boundary_register["field"].str.split(" / ").explode())
            | {"batting_team_id", "batting_type", "batting_order"}
        )
    },
    {
        "check": "No score reconstruction is approved",
        "passed": not any(
            term in phase3D_scoring_boundary_register["approved_treatment"].str.lower().to_string()
            for term in ["sum runs", "calculate cumulative score", "reconstruct score"]
        )
    }
])

phase3D_scoring_boundary_register.to_csv(
    OUTPUT_DIR / "phase3D_scoring_reconstruction_boundary_register.csv",
    index=False
)

phase3D_scoring_boundary_validation.to_csv(
    OUTPUT_DIR / "phase3D_scoring_reconstruction_boundary_validation.csv",
    index=False
)

print("SCORING RECONSTRUCTION BOUNDARY REGISTER")
display(phase3D_scoring_boundary_register)

print("\nVALIDATION")
display(phase3D_scoring_boundary_validation)

assert phase3D_scoring_boundary_validation["passed"].all(), (
    "Scoring boundary validation failed; inspect the register."
)

print("\n3D.2.3 status: PASSED — reconstruction limitation formally recorded")

SCORING RECONSTRUCTION BOUNDARY REGISTER


,field,evidence,limitation,approved_treatment,status
0,runs,Integer values 0–6; no independent score field in raw source,Cannot establish batter-runs versus total-delivery-runs semantics,Preserve raw; do not sum as cumulative innings score,UNRESOLVED
1,extras1_runs,Encoded categories 1–7; category 6 means no extras,Field is not a numerical extras-run quantity,Use only as categorical evidence; never add as runs,SEMANTICS ESTABLISHED — NOT A RUN QUANTITY
2,extras_2_runs,"Values 0, 1 and 4; non-zero semantics undocumented",Cannot establish whether or how values contribute to delivery totals,Preserve raw; do not add to runs,UNRESOLVED
3,extras_type,"Labels include wide, legbyes, Overstep, byes, other and Penalty",Category labels do not establish the numeric allocation of runs,Retain as categorical evidence only,PARTIALLY INFORMATIVE
4,is_four / is_six,23 boundary-flag mismatches in the primary population,Flags cannot independently reconstruct total delivery runs,Preserve flags independently; do not use to repair runs,LIMITED
5,independent innings score,No such field found in the original ball-by-ball schema,No independent delivery-to-innings score reconciliation available,Require authoritative documentation or independent score source,UNAVAILABLE



VALIDATION


,check,passed
0,Raw source has no separate total-runs or batter-runs field,True
1,All identified scoring-related source fields are documented,True
2,No score reconstruction is approved,True



3D.2.3 status: PASSED — reconstruction limitation formally recorded


#### Benchmark-input feasibility and impact register

In [113]:
phase3D_benchmark_input_register = pd.DataFrame([
    {
        "input": "Match and competition identity",
        "evidence": "813 approved matches; 136 competition groups; match assignments reconciled",
        "status": "AVAILABLE",
        "benchmark_use": "Allowed",
        "impact": "IPL and T20I can remain separate benchmark populations"
    },
    {
        "input": "Pre-delivery wickets fallen",
        "evidence": "Complete for 100,333 first-innings records; values within 0–10",
        "status": "CONDITIONAL",
        "benchmark_use": "Allowed with provenance caveat",
        "impact": "Can describe wicket state, but it is reconstructed from prior wicket events"
    },
    {
        "input": "Cumulative innings score",
        "evidence": "Raw runs semantics unresolved; extras_2_runs semantics unresolved; no independent innings score field",
        "status": "UNAVAILABLE",
        "benchmark_use": "Prohibited",
        "impact": "Cannot calculate actual cumulative scoring position or scoring deviation"
    },
    {
        "input": "Legal balls remaining",
        "evidence": "Legal-ball progression unresolved; sequence fields are not legal-ball counters",
        "status": "UNAVAILABLE",
        "benchmark_use": "Prohibited",
        "impact": "Cannot establish remaining-ball state or normalized innings progression"
    },
    {
        "input": "Scheduled innings length",
        "evidence": "Revised-overs fields populated for only 15 first innings; units unresolved",
        "status": "UNAVAILABLE",
        "benchmark_use": "Prohibited",
        "impact": "Cannot establish a validated common scheduled-length denominator"
    },
    {
        "input": "Innings stage",
        "evidence": "Depends on validated schedule and legal-ball progression, neither available",
        "status": "UNAVAILABLE",
        "benchmark_use": "Prohibited",
        "impact": "Cannot assign defensible powerplay, middle or death-stage benchmark states"
    },
    {
        "input": "Competition-specific wicket-state sample coverage",
        "evidence": "255 competition/wicket-state combinations unobserved; high-wicket states are sparse",
        "status": "SPARSE",
        "benchmark_use": "Coverage audit only",
        "impact": "Exact competition-level benchmark cells may be unsupported"
    },
    {
        "input": "Expected cumulative scoring benchmark",
        "evidence": "Actual cumulative score and required schedule/progression dimensions unavailable",
        "status": "NOT CONSTRUCTIBLE",
        "benchmark_use": "Prohibited",
        "impact": "No expected-score curve, scoring deviation or standardized deviation can be approved"
    }
])

phase3D_benchmark_input_validation = pd.DataFrame([
    {
        "check": "Cumulative score is explicitly unavailable",
        "passed": (
            phase3D_benchmark_input_register.loc[
                phase3D_benchmark_input_register["input"] == "Cumulative innings score",
                "status"
            ].iloc[0] == "UNAVAILABLE"
        )
    },
    {
        "check": "Legal balls remaining is explicitly unavailable",
        "passed": (
            phase3D_benchmark_input_register.loc[
                phase3D_benchmark_input_register["input"] == "Legal balls remaining",
                "status"
            ].iloc[0] == "UNAVAILABLE"
        )
    },
    {
        "check": "Scheduled length and innings stage are explicitly unavailable",
        "passed": (
            phase3D_benchmark_input_register.loc[
                phase3D_benchmark_input_register["input"].isin(
                    ["Scheduled innings length", "Innings stage"]
                ),
                "status"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check": "Expected cumulative scoring benchmark is not approved",
        "passed": (
            phase3D_benchmark_input_register.loc[
                phase3D_benchmark_input_register["input"] == "Expected cumulative scoring benchmark",
                "status"
            ].iloc[0] == "NOT CONSTRUCTIBLE"
        )
    },
    {
        "check": "Wicket-state reconstruction remains conditional",
        "passed": (
            phase3D_benchmark_input_register.loc[
                phase3D_benchmark_input_register["input"] == "Pre-delivery wickets fallen",
                "status"
            ].iloc[0] == "CONDITIONAL"
        )
    }
])

phase3D_benchmark_input_register.to_csv(
    OUTPUT_DIR / "phase3D_benchmark_input_feasibility_register.csv",
    index=False
)

phase3D_benchmark_input_validation.to_csv(
    OUTPUT_DIR / "phase3D_benchmark_input_feasibility_validation.csv",
    index=False
)

print("BENCHMARK INPUT FEASIBILITY REGISTER")
display(phase3D_benchmark_input_register)

print("\nVALIDATION")
display(phase3D_benchmark_input_validation)

assert phase3D_benchmark_input_validation["passed"].all(), (
    "Benchmark-input feasibility validation failed."
)

print("\n3D.5.1 status: PASSED — feasibility and impact recorded")

BENCHMARK INPUT FEASIBILITY REGISTER


,input,evidence,status,benchmark_use,impact
0,Match and competition identity,813 approved matches; 136 competition groups; match assignments reconciled,AVAILABLE,Allowed,IPL and T20I can remain separate benchmark populations
1,Pre-delivery wickets fallen,"Complete for 100,333 first-innings records; values within 0–10",CONDITIONAL,Allowed with provenance caveat,"Can describe wicket state, but it is reconstructed from prior wicket events"
2,Cumulative innings score,Raw runs semantics unresolved; extras_2_runs semantics unresolved; no independent innings score field,UNAVAILABLE,Prohibited,Cannot calculate actual cumulative scoring position or scoring deviation
3,Legal balls remaining,Legal-ball progression unresolved; sequence fields are not legal-ball counters,UNAVAILABLE,Prohibited,Cannot establish remaining-ball state or normalized innings progression
4,Scheduled innings length,Revised-overs fields populated for only 15 first innings; units unresolved,UNAVAILABLE,Prohibited,Cannot establish a validated common scheduled-length denominator
5,Innings stage,"Depends on validated schedule and legal-ball progression, neither available",UNAVAILABLE,Prohibited,"Cannot assign defensible powerplay, middle or death-stage benchmark states"
6,Competition-specific wicket-state sample coverage,255 competition/wicket-state combinations unobserved; high-wicket states are sparse,SPARSE,Coverage audit only,Exact competition-level benchmark cells may be unsupported
7,Expected cumulative scoring benchmark,Actual cumulative score and required schedule/progression dimensions unavailable,NOT CONSTRUCTIBLE,Prohibited,"No expected-score curve, scoring deviation or standardized deviation can be approved"



VALIDATION


,check,passed
0,Cumulative score is explicitly unavailable,True
1,Legal balls remaining is explicitly unavailable,True
2,Scheduled length and innings stage are explicitly unavailable,True
3,Expected cumulative scoring benchmark is not approved,True
4,Wicket-state reconstruction remains conditional,True



3D.5.1 status: PASSED — feasibility and impact recorded


#### Benchmark construction decision and blocker register

In [115]:
phase3D_construction_decision_register = pd.DataFrame([
    {
        "item": "First-innings benchmark construction",
        "decision": "BLOCKED",
        "reason": "Cumulative innings score, legal balls remaining and validated scheduled innings length are unavailable",
        "required_evidence": "Authoritative source definitions and/or additional validated data supporting these inputs",
        "permitted_action": "Retain the prepared first-innings population and feasibility audit",
        "prohibited_action": "Construct expected cumulative scoring benchmarks"
    },
    {
        "item": "Cumulative innings score",
        "decision": "BLOCKED",
        "reason": "runs and extras_2_runs semantics remain unresolved",
        "required_evidence": "Authoritative field-level scoring definitions or independently verified innings totals with a defensible delivery-level reconciliation",
        "permitted_action": "Preserve raw scoring fields and existing audit evidence",
        "prohibited_action": "Sum or reinterpret raw fields to manufacture cumulative scores"
    },
    {
        "item": "Legal balls remaining",
        "decision": "BLOCKED",
        "reason": "Legal-ball progression is unresolved",
        "required_evidence": "Validated legal-delivery classification and innings-level legal-ball progression",
        "permitted_action": "Retain delivery sequence and legal-ball exception audits as evidence",
        "prohibited_action": "Treat over_ball_num or innings_ball_num as legal-ball counters"
    },
    {
        "item": "Scheduled innings length",
        "decision": "BLOCKED",
        "reason": "Revised-overs fields are sparse and their units are unresolved",
        "required_evidence": "Authoritative scheduled-innings-length definitions and match-level schedule data",
        "permitted_action": "Retain the revised-field coverage audit",
        "prohibited_action": "Assume revised-overs values are literal overs or convert their units without evidence"
    },
    {
        "item": "Innings stage",
        "decision": "BLOCKED",
        "reason": "Stage assignment depends on validated schedule and legal-ball progression",
        "required_evidence": "Validated scheduled length and legal-ball progression, including applicable shortened-innings treatment",
        "permitted_action": "Retain stage as unavailable in the state specification",
        "prohibited_action": "Assign powerplay, middle or death stages from sequence fields alone"
    },
    {
        "item": "Pre-delivery wickets fallen",
        "decision": "CONDITIONAL",
        "reason": "State is reconstructed from prior wicket events",
        "required_evidence": "Retain reconstruction provenance and existing integrity checks",
        "permitted_action": "Use as a conditional state descriptor with provenance caveat",
        "prohibited_action": "Describe it as independently observed dismissal truth"
    },
    {
        "item": "Competition-specific wicket-state coverage",
        "decision": "AUDIT_ONLY",
        "reason": "255 competition/wicket-state combinations are unobserved and high-wicket states are sparse",
        "required_evidence": "Future benchmark-cell sample coverage and reliability assessment after required scoring inputs become available",
        "permitted_action": "Use for coverage diagnostics only",
        "prohibited_action": "Treat observed coverage as proof that benchmark estimates are reliable"
    },
    {
        "item": "Scoring deviation and standardized deviation",
        "decision": "BLOCKED",
        "reason": "Actual cumulative score and approved expected-score benchmark are unavailable",
        "required_evidence": "Validated cumulative score plus a constructed and validated expected-scoring benchmark",
        "permitted_action": "Document as unavailable outputs",
        "prohibited_action": "Calculate or publish either deviation"
    }
])

phase3D_construction_decision_validation = pd.DataFrame([
    {
        "check": "Benchmark construction is explicitly blocked",
        "passed": (
            phase3D_construction_decision_register.loc[
                phase3D_construction_decision_register["item"] == "First-innings benchmark construction",
                "decision"
            ].iloc[0] == "BLOCKED"
        )
    },
    {
        "check": "Cumulative scoring reconstruction remains blocked",
        "passed": (
            phase3D_construction_decision_register.loc[
                phase3D_construction_decision_register["item"] == "Cumulative innings score",
                "decision"
            ].iloc[0] == "BLOCKED"
        )
    },
    {
        "check": "Legal-ball and schedule inputs remain blocked",
        "passed": all(
            phase3D_construction_decision_register.loc[
                phase3D_construction_decision_register["item"].isin(
                    ["Legal balls remaining", "Scheduled innings length"]
                ),
                "decision"
            ] == "BLOCKED"
        )
    },
    {
        "check": "Wicket-state use remains conditional",
        "passed": (
            phase3D_construction_decision_register.loc[
                phase3D_construction_decision_register["item"] == "Pre-delivery wickets fallen",
                "decision"
            ].iloc[0] == "CONDITIONAL"
        )
    },
    {
        "check": "No scoring deviation output is approved",
        "passed": (
            phase3D_construction_decision_register.loc[
                phase3D_construction_decision_register["item"] == "Scoring deviation and standardized deviation",
                "decision"
            ].iloc[0] == "BLOCKED"
        )
    },
    {
        "check": "Every decision has a reason and evidence requirement",
        "passed": (
            phase3D_construction_decision_register["reason"].notna().all()
            and phase3D_construction_decision_register["required_evidence"].notna().all()
        )
    }
])

phase3D_construction_decision_register.to_csv(
    OUTPUT_DIR / "phase3D_benchmark_construction_decision_register.csv",
    index=False
)

phase3D_construction_decision_validation.to_csv(
    OUTPUT_DIR / "phase3D_benchmark_construction_decision_validation.csv",
    index=False
)

display(phase3D_construction_decision_register)
display(phase3D_construction_decision_validation)

print("All construction-decision checks passed:",
      phase3D_construction_decision_validation["passed"].all())
print("Decision register rows:", len(phase3D_construction_decision_register))
print("Validation checks:", len(phase3D_construction_decision_validation))
print("Passed:", int(phase3D_construction_decision_validation["passed"].sum()))
print("Failed:", int((~phase3D_construction_decision_validation["passed"]).sum()))
print("Saved:", OUTPUT_DIR / "phase3D_benchmark_construction_decision_register.csv")
print("Saved:", OUTPUT_DIR / "phase3D_benchmark_construction_decision_validation.csv")

,item,decision,reason,required_evidence,permitted_action,prohibited_action
0,First-innings benchmark construction,BLOCKED,"Cumulative innings score, legal balls remaining and validated scheduled innings length are unavailable",Authoritative source definitions and/or additional validated data supporting these inputs,Retain the prepared first-innings population and feasibility audit,Construct expected cumulative scoring benchmarks
1,Cumulative innings score,BLOCKED,runs and extras_2_runs semantics remain unresolved,Authoritative field-level scoring definitions or independently verified innings totals with a defensible delivery-level reconciliation,Preserve raw scoring fields and existing audit evidence,Sum or reinterpret raw fields to manufacture cumulative scores
2,Legal balls remaining,BLOCKED,Legal-ball progression is unresolved,Validated legal-delivery classification and innings-level legal-ball progression,Retain delivery sequence and legal-ball exception audits as evidence,Treat over_ball_num or innings_ball_num as legal-ball counters
3,Scheduled innings length,BLOCKED,Revised-overs fields are sparse and their units are unresolved,Authoritative scheduled-innings-length definitions and match-level schedule data,Retain the revised-field coverage audit,Assume revised-overs values are literal overs or convert their units without evidence
4,Innings stage,BLOCKED,Stage assignment depends on validated schedule and legal-ball progression,"Validated scheduled length and legal-ball progression, including applicable shortened-innings treatment",Retain stage as unavailable in the state specification,"Assign powerplay, middle or death stages from sequence fields alone"
5,Pre-delivery wickets fallen,CONDITIONAL,State is reconstructed from prior wicket events,Retain reconstruction provenance and existing integrity checks,Use as a conditional state descriptor with provenance caveat,Describe it as independently observed dismissal truth
6,Competition-specific wicket-state coverage,AUDIT_ONLY,255 competition/wicket-state combinations are unobserved and high-wicket states are sparse,Future benchmark-cell sample coverage and reliability assessment after required scoring inputs become available,Use for coverage diagnostics only,Treat observed coverage as proof that benchmark estimates are reliable
7,Scoring deviation and standardized deviation,BLOCKED,Actual cumulative score and approved expected-score benchmark are unavailable,Validated cumulative score plus a constructed and validated expected-scoring benchmark,Document as unavailable outputs,Calculate or publish either deviation


,check,passed
0,Benchmark construction is explicitly blocked,True
1,Cumulative scoring reconstruction remains blocked,True
2,Legal-ball and schedule inputs remain blocked,True
3,Wicket-state use remains conditional,True
4,No scoring deviation output is approved,True
5,Every decision has a reason and evidence requirement,True


All construction-decision checks passed: True
Decision register rows: 8
Validation checks: 6
Passed: 6
Failed: 0
Saved: /content/phase3_outputs/phase3D_benchmark_construction_decision_register.csv
Saved: /content/phase3_outputs/phase3D_benchmark_construction_decision_validation.csv


#### Comparable benchmark-state specification

In [116]:
phase3D_comparable_state_specification = pd.DataFrame([
    {
        "dimension": "Competition format",
        "intended_definition": "Separate IPL and T20I benchmark populations",
        "source_basis": "Approved competition and match-format mapping",
        "status": "AVAILABLE",
        "benchmark_role": "Required grouping dimension",
        "current_use": "Allowed",
        "limitation": "No cross-format fallback"
    },
    {
        "dimension": "Pre-delivery wickets fallen",
        "intended_definition": "Wickets fallen before the current delivery",
        "source_basis": "Reconstructed pre-delivery wicket state",
        "status": "CONDITIONAL",
        "benchmark_role": "Potential state dimension",
        "current_use": "Conditional; retain provenance",
        "limitation": "Reconstructed from prior wicket events"
    },
    {
        "dimension": "Legal balls remaining",
        "intended_definition": "Scheduled legal balls remaining before the current delivery",
        "source_basis": "Requires validated legal-ball progression and scheduled innings length",
        "status": "UNAVAILABLE",
        "benchmark_role": "Required progression dimension",
        "current_use": "Prohibited",
        "limitation": "Sequence fields are not legal-ball counters"
    },
    {
        "dimension": "Scheduled innings length",
        "intended_definition": "Validated scheduled legal-ball allocation for the innings",
        "source_basis": "Requires authoritative match schedule and shortened-innings treatment",
        "status": "UNAVAILABLE",
        "benchmark_role": "Required denominator and context",
        "current_use": "Prohibited",
        "limitation": "Revised-overs fields are sparse and units unresolved"
    },
    {
        "dimension": "Innings stage",
        "intended_definition": "Stage derived from validated innings progression and applicable schedule",
        "source_basis": "Requires legal-ball progression and scheduled innings length",
        "status": "UNAVAILABLE",
        "benchmark_role": "Potential grouping dimension",
        "current_use": "Prohibited",
        "limitation": "Cannot defensibly assign powerplay, middle or death stages"
    },
    {
        "dimension": "Cumulative innings score",
        "intended_definition": "Validated total score immediately before the current delivery",
        "source_basis": "Requires resolved delivery-run semantics and reconciled scoring reconstruction",
        "status": "UNAVAILABLE",
        "benchmark_role": "Observed outcome for benchmark comparison",
        "current_use": "Prohibited",
        "limitation": "Raw runs and extras_2_runs semantics unresolved"
    },
    {
        "dimension": "Expected cumulative scoring benchmark",
        "intended_definition": "Historical expected cumulative score for comparable first-innings states",
        "source_basis": "Requires validated cumulative score and progression dimensions",
        "status": "NOT_CONSTRUCTIBLE",
        "benchmark_role": "Benchmark output",
        "current_use": "Prohibited",
        "limitation": "Required input dimensions are unavailable"
    }
])

phase3D_comparable_state_validation = pd.DataFrame([
    {
        "check": "IPL and T20I remain separate",
        "passed": (
            phase3D_comparable_state_specification.loc[
                phase3D_comparable_state_specification["dimension"] == "Competition format",
                "current_use"
            ].iloc[0] == "Allowed"
        )
    },
    {
        "check": "Wicket state retains conditional status",
        "passed": (
            phase3D_comparable_state_specification.loc[
                phase3D_comparable_state_specification["dimension"] == "Pre-delivery wickets fallen",
                "status"
            ].iloc[0] == "CONDITIONAL"
        )
    },
    {
        "check": "Legal-ball and schedule dimensions remain unavailable",
        "passed": (
            phase3D_comparable_state_specification.loc[
                phase3D_comparable_state_specification["dimension"].isin(
                    ["Legal balls remaining", "Scheduled innings length"]
                ),
                "status"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check": "Innings stage and cumulative score remain unavailable",
        "passed": (
            phase3D_comparable_state_specification.loc[
                phase3D_comparable_state_specification["dimension"].isin(
                    ["Innings stage", "Cumulative innings score"]
                ),
                "status"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check": "Expected benchmark is not constructible",
        "passed": (
            phase3D_comparable_state_specification.loc[
                phase3D_comparable_state_specification["dimension"] == "Expected cumulative scoring benchmark",
                "status"
            ].iloc[0] == "NOT_CONSTRUCTIBLE"
        )
    },
    {
        "check": "Every dimension has a limitation recorded",
        "passed": phase3D_comparable_state_specification["limitation"].notna().all()
    }
])

phase3D_comparable_state_specification.to_csv(
    OUTPUT_DIR / "phase3D_comparable_benchmark_state_specification.csv",
    index=False
)

phase3D_comparable_state_validation.to_csv(
    OUTPUT_DIR / "phase3D_comparable_benchmark_state_validation.csv",
    index=False
)

display(phase3D_comparable_state_specification)
display(phase3D_comparable_state_validation)

print("All state-specification checks passed:",
      phase3D_comparable_state_validation["passed"].all())
print("State dimensions:", len(phase3D_comparable_state_specification))
print("Validation checks:", len(phase3D_comparable_state_validation))
print("Passed:", int(phase3D_comparable_state_validation["passed"].sum()))
print("Failed:", int((~phase3D_comparable_state_validation["passed"]).sum()))
print("Saved:", OUTPUT_DIR / "phase3D_comparable_benchmark_state_specification.csv")
print("Saved:", OUTPUT_DIR / "phase3D_comparable_benchmark_state_validation.csv")

,dimension,intended_definition,source_basis,status,benchmark_role,current_use,limitation
0,Competition format,Separate IPL and T20I benchmark populations,Approved competition and match-format mapping,AVAILABLE,Required grouping dimension,Allowed,No cross-format fallback
1,Pre-delivery wickets fallen,Wickets fallen before the current delivery,Reconstructed pre-delivery wicket state,CONDITIONAL,Potential state dimension,Conditional; retain provenance,Reconstructed from prior wicket events
2,Legal balls remaining,Scheduled legal balls remaining before the current delivery,Requires validated legal-ball progression and scheduled innings length,UNAVAILABLE,Required progression dimension,Prohibited,Sequence fields are not legal-ball counters
3,Scheduled innings length,Validated scheduled legal-ball allocation for the innings,Requires authoritative match schedule and shortened-innings treatment,UNAVAILABLE,Required denominator and context,Prohibited,Revised-overs fields are sparse and units unresolved
4,Innings stage,Stage derived from validated innings progression and applicable schedule,Requires legal-ball progression and scheduled innings length,UNAVAILABLE,Potential grouping dimension,Prohibited,"Cannot defensibly assign powerplay, middle or death stages"
5,Cumulative innings score,Validated total score immediately before the current delivery,Requires resolved delivery-run semantics and reconciled scoring reconstruction,UNAVAILABLE,Observed outcome for benchmark comparison,Prohibited,Raw runs and extras_2_runs semantics unresolved
6,Expected cumulative scoring benchmark,Historical expected cumulative score for comparable first-innings states,Requires validated cumulative score and progression dimensions,NOT_CONSTRUCTIBLE,Benchmark output,Prohibited,Required input dimensions are unavailable


,check,passed
0,IPL and T20I remain separate,True
1,Wicket state retains conditional status,True
2,Legal-ball and schedule dimensions remain unavailable,True
3,Innings stage and cumulative score remain unavailable,True
4,Expected benchmark is not constructible,True
5,Every dimension has a limitation recorded,True


All state-specification checks passed: True
State dimensions: 7
Validation checks: 6
Passed: 6
Failed: 0
Saved: /content/phase3_outputs/phase3D_comparable_benchmark_state_specification.csv
Saved: /content/phase3_outputs/phase3D_comparable_benchmark_state_validation.csv


#### Benchmark matching and fallback policy

In [117]:
# Phase 3D.5.4 — Benchmark matching and fallback policy register

phase3D_fallback_policy = pd.DataFrame([
    {
        "priority": 1,
        "fallback_level": "Exact comparable state",
        "policy": "Use observations matching the same competition format and all validated benchmark-state dimensions",
        "conditions": "All required state dimensions and scoring outcomes must be validated",
        "status": "DEFINED_NOT_EXECUTABLE",
        "restriction": "No estimate where required dimensions are unavailable"
    },
    {
        "priority": 2,
        "fallback_level": "Nearby scheduled-innings lengths",
        "policy": "Pool nearby scheduled lengths only where a defensible distance rule is documented",
        "conditions": "Scheduled innings length must be validated and units consistent",
        "status": "DEFINED_NOT_EXECUTABLE",
        "restriction": "Do not interpret unresolved revised-overs values"
    },
    {
        "priority": 3,
        "fallback_level": "Adjacent wicket states",
        "policy": "Pool adjacent wicket states using observation-weighted aggregation",
        "conditions": "Wicket states must retain reconstruction provenance and scoring inputs must be valid",
        "status": "DEFINED_NOT_EXECUTABLE",
        "restriction": "Do not weight wicket-state groups equally regardless of sample size"
    },
    {
        "priority": 4,
        "fallback_level": "Symmetric state expansion",
        "policy": "Expand around the target state symmetrically and record included states",
        "conditions": "A defensible expansion rule and sample coverage assessment must be specified",
        "status": "DEFINED_NOT_EXECUTABLE",
        "restriction": "Do not silently broaden the comparison population"
    },
    {
        "priority": 5,
        "fallback_level": "Broader innings-stage grouping",
        "policy": "Use a broader stage grouping only if stage boundaries and progression are validated",
        "conditions": "Validated innings stage and schedule required",
        "status": "DEFINED_NOT_EXECUTABLE",
        "restriction": "Do not infer stage from delivery sequence alone"
    },
    {
        "priority": 6,
        "fallback_level": "Unavailable",
        "policy": "Return no benchmark when no defensible comparable state can be formed",
        "conditions": "Required inputs unavailable or fallback evidence insufficient",
        "status": "APPROVED_AS_TERMINAL_RULE",
        "restriction": "Do not manufacture or publish an unsupported estimate"
    }
])

phase3D_fallback_policy_validation = pd.DataFrame([
    {
        "check": "Fallback priorities are sequential and unique",
        "passed": (
            phase3D_fallback_policy["priority"].tolist() == [1, 2, 3, 4, 5, 6]
            and phase3D_fallback_policy["priority"].is_unique
        )
    },
    {
        "check": "Exact matching requires validated dimensions",
        "passed": "all required state dimensions" in phase3D_fallback_policy.loc[0, "conditions"]
    },
    {
        "check": "Scheduled-length fallback requires validated units",
        "passed": "units consistent" in phase3D_fallback_policy.loc[1, "conditions"]
    },
    {
        "check": "Adjacent wicket pooling is observation-weighted",
        "passed": "observation-weighted" in phase3D_fallback_policy.loc[2, "policy"]
    },
    {
        "check": "No cross-format fallback is permitted",
        "passed": phase3D_fallback_policy["restriction"].notna().all()
    },
    {
        "check": "Unavailable is the terminal fallback",
        "passed": (
            phase3D_fallback_policy.iloc[-1]["fallback_level"] == "Unavailable"
            and phase3D_fallback_policy.iloc[-1]["status"] == "APPROVED_AS_TERMINAL_RULE"
        )
    },
    {
        "check": "No fallback is incorrectly marked executable",
        "passed": (
            (phase3D_fallback_policy.iloc[:5]["status"] == "DEFINED_NOT_EXECUTABLE").all()
        )
    }
])

phase3D_fallback_policy.to_csv(
    OUTPUT_DIR / "phase3D_benchmark_fallback_policy.csv",
    index=False
)

phase3D_fallback_policy_validation.to_csv(
    OUTPUT_DIR / "phase3D_benchmark_fallback_policy_validation.csv",
    index=False
)

display(phase3D_fallback_policy)
display(phase3D_fallback_policy_validation)

print("All fallback-policy checks passed:",
      phase3D_fallback_policy_validation["passed"].all())
print("Fallback levels:", len(phase3D_fallback_policy))
print("Validation checks:", len(phase3D_fallback_policy_validation))
print("Passed:", int(phase3D_fallback_policy_validation["passed"].sum()))
print("Failed:", int((~phase3D_fallback_policy_validation["passed"]).sum()))
print("Saved:", OUTPUT_DIR / "phase3D_benchmark_fallback_policy.csv")
print("Saved:", OUTPUT_DIR / "phase3D_benchmark_fallback_policy_validation.csv")

,priority,fallback_level,policy,conditions,status,restriction
0,1,Exact comparable state,Use observations matching the same competition format and all validated benchmark-state dimensions,All required state dimensions and scoring outcomes must be validated,DEFINED_NOT_EXECUTABLE,No estimate where required dimensions are unavailable
1,2,Nearby scheduled-innings lengths,Pool nearby scheduled lengths only where a defensible distance rule is documented,Scheduled innings length must be validated and units consistent,DEFINED_NOT_EXECUTABLE,Do not interpret unresolved revised-overs values
2,3,Adjacent wicket states,Pool adjacent wicket states using observation-weighted aggregation,Wicket states must retain reconstruction provenance and scoring inputs must be valid,DEFINED_NOT_EXECUTABLE,Do not weight wicket-state groups equally regardless of sample size
3,4,Symmetric state expansion,Expand around the target state symmetrically and record included states,A defensible expansion rule and sample coverage assessment must be specified,DEFINED_NOT_EXECUTABLE,Do not silently broaden the comparison population
4,5,Broader innings-stage grouping,Use a broader stage grouping only if stage boundaries and progression are validated,Validated innings stage and schedule required,DEFINED_NOT_EXECUTABLE,Do not infer stage from delivery sequence alone
5,6,Unavailable,Return no benchmark when no defensible comparable state can be formed,Required inputs unavailable or fallback evidence insufficient,APPROVED_AS_TERMINAL_RULE,Do not manufacture or publish an unsupported estimate


,check,passed
0,Fallback priorities are sequential and unique,True
1,Exact matching requires validated dimensions,False
2,Scheduled-length fallback requires validated units,True
3,Adjacent wicket pooling is observation-weighted,True
4,No cross-format fallback is permitted,True
5,Unavailable is the terminal fallback,True
6,No fallback is incorrectly marked executable,True


All fallback-policy checks passed: False
Fallback levels: 6
Validation checks: 7
Passed: 6
Failed: 1
Saved: /content/phase3_outputs/phase3D_benchmark_fallback_policy.csv
Saved: /content/phase3_outputs/phase3D_benchmark_fallback_policy_validation.csv


In [118]:
print("Stored condition:")
print(repr(phase3D_fallback_policy.loc[0, "conditions"]))

print("\nExact substring test:")
print("all required state dimensions" in phase3D_fallback_policy.loc[0, "conditions"])

print("\nStored first policy row:")
display(phase3D_fallback_policy.iloc[[0]])

print("\nFailed validation rows:")
display(
    phase3D_fallback_policy_validation.loc[
        ~phase3D_fallback_policy_validation["passed"]
    ]
)

Stored condition:
'All required state dimensions and scoring outcomes must be validated'

Exact substring test:
False

Stored first policy row:


,priority,fallback_level,policy,conditions,status,restriction
0,1,Exact comparable state,Use observations matching the same competition format and all validated benchmark-state dimensions,All required state dimensions and scoring outcomes must be validated,DEFINED_NOT_EXECUTABLE,No estimate where required dimensions are unavailable



Failed validation rows:


,check,passed
1,Exact matching requires validated dimensions,False


In [119]:
# Phase 3D.5.4 — Correct the case-sensitive validation check

condition_text = str(
    phase3D_fallback_policy.loc[0, "conditions"]
)

phase3D_fallback_policy_validation.loc[
    phase3D_fallback_policy_validation["check"]
    == "Exact matching requires validated dimensions",
    "passed"
] = "all required state dimensions" in condition_text.lower()

display(phase3D_fallback_policy_validation)

print(
    "All fallback-policy checks passed:",
    phase3D_fallback_policy_validation["passed"].all()
)
print(
    "Passed:",
    int(phase3D_fallback_policy_validation["passed"].sum())
)
print(
    "Failed:",
    int((~phase3D_fallback_policy_validation["passed"]).sum())
)

phase3D_fallback_policy_validation.to_csv(
    OUTPUT_DIR / "phase3D_benchmark_fallback_policy_validation.csv",
    index=False
)

print(
    "Updated validation saved:",
    OUTPUT_DIR / "phase3D_benchmark_fallback_policy_validation.csv"
)

,check,passed
0,Fallback priorities are sequential and unique,True
1,Exact matching requires validated dimensions,True
2,Scheduled-length fallback requires validated units,True
3,Adjacent wicket pooling is observation-weighted,True
4,No cross-format fallback is permitted,True
5,Unavailable is the terminal fallback,True
6,No fallback is incorrectly marked executable,True


All fallback-policy checks passed: True
Passed: 7
Failed: 0
Updated validation saved: /content/phase3_outputs/phase3D_benchmark_fallback_policy_validation.csv


### Phase 3D — Remaining roadmap status register

In [120]:
# Phase 3D — Remaining roadmap status register

phase3D_remaining_roadmap = pd.DataFrame([
    {
        "subphase": "3D.6",
        "name": "Expected cumulative scoring benchmark construction",
        "status": "BLOCKED",
        "work_completed": "Feasibility audit and comparable-state specification",
        "reason": "Cumulative score, legal balls remaining and scheduled innings length are unavailable",
        "required_to_unblock": "Validated scoring reconstruction, legal-ball progression and scheduled innings length",
        "deliverable_status": "Not constructed"
    },
    {
        "subphase": "3D.7",
        "name": "Adaptive fallback hierarchy implementation",
        "status": "BLOCKED",
        "work_completed": "Six-level fallback policy documented and validated",
        "reason": "No valid benchmark estimates or comparable scoring states exist to apply the hierarchy to",
        "required_to_unblock": "Validated benchmark population and initial benchmark estimates",
        "deliverable_status": "Policy documented; implementation not executed"
    },
    {
        "subphase": "3D.8",
        "name": "Benchmark coverage and reliability validation",
        "status": "PARTIALLY_COMPLETED",
        "work_completed": "Population coverage, competition coverage and wicket-state distribution audits",
        "reason": "Benchmark-cell reliability and fallback performance cannot be tested without benchmark estimates",
        "required_to_unblock": "Constructed benchmarks and defined reliability assessment criteria",
        "deliverable_status": "Coverage diagnostics completed; benchmark reliability not assessed"
    },
    {
        "subphase": "3D.9",
        "name": "Consolidated report and phase closure",
        "status": "PENDING",
        "work_completed": "Subphase reports, decision registers and validation evidence prepared",
        "reason": "Final report must explicitly distinguish completed, blocked and partially completed work",
        "required_to_unblock": "Final roadmap status and output audit",
        "deliverable_status": "Final consolidated report pending"
    }
])

phase3D_remaining_roadmap_validation = pd.DataFrame([
    {
        "check": "3D.6 is explicitly blocked",
        "passed": phase3D_remaining_roadmap.loc[
            phase3D_remaining_roadmap["subphase"] == "3D.6", "status"
        ].iloc[0] == "BLOCKED"
    },
    {
        "check": "3D.7 policy is distinguished from implementation",
        "passed": "implementation not executed" in phase3D_remaining_roadmap.loc[
            phase3D_remaining_roadmap["subphase"] == "3D.7", "deliverable_status"
        ].iloc[0]
    },
    {
        "check": "3D.8 separates coverage from reliability",
        "passed": (
            "Coverage diagnostics completed" in phase3D_remaining_roadmap.loc[
                phase3D_remaining_roadmap["subphase"] == "3D.8", "deliverable_status"
            ].iloc[0]
            and "not assessed" in phase3D_remaining_roadmap.loc[
                phase3D_remaining_roadmap["subphase"] == "3D.8", "deliverable_status"
            ].iloc[0]
        )
    },
    {
        "check": "3D.9 remains pending until final report",
        "passed": phase3D_remaining_roadmap.loc[
            phase3D_remaining_roadmap["subphase"] == "3D.9", "status"
        ].iloc[0] == "PENDING"
    },
    {
        "check": "Every remaining subphase has an unblock requirement",
        "passed": phase3D_remaining_roadmap["required_to_unblock"].notna().all()
    }
])

phase3D_remaining_roadmap.to_csv(
    OUTPUT_DIR / "phase3D_remaining_roadmap_status.csv",
    index=False
)

phase3D_remaining_roadmap_validation.to_csv(
    OUTPUT_DIR / "phase3D_remaining_roadmap_validation.csv",
    index=False
)

display(phase3D_remaining_roadmap)
display(phase3D_remaining_roadmap_validation)

print("All roadmap checks passed:",
      phase3D_remaining_roadmap_validation["passed"].all())
print("Roadmap entries:", len(phase3D_remaining_roadmap))
print("Passed:", int(phase3D_remaining_roadmap_validation["passed"].sum()))
print("Failed:", int((~phase3D_remaining_roadmap_validation["passed"]).sum()))
print("Saved:", OUTPUT_DIR / "phase3D_remaining_roadmap_status.csv")
print("Saved:", OUTPUT_DIR / "phase3D_remaining_roadmap_validation.csv")

,subphase,name,status,work_completed,reason,required_to_unblock,deliverable_status
0,3D.6,Expected cumulative scoring benchmark construction,BLOCKED,Feasibility audit and comparable-state specification,"Cumulative score, legal balls remaining and scheduled innings length are unavailable","Validated scoring reconstruction, legal-ball progression and scheduled innings length",Not constructed
1,3D.7,Adaptive fallback hierarchy implementation,BLOCKED,Six-level fallback policy documented and validated,No valid benchmark estimates or comparable scoring states exist to apply the hierarchy to,Validated benchmark population and initial benchmark estimates,Policy documented; implementation not executed
2,3D.8,Benchmark coverage and reliability validation,PARTIALLY_COMPLETED,"Population coverage, competition coverage and wicket-state distribution audits",Benchmark-cell reliability and fallback performance cannot be tested without benchmark estimates,Constructed benchmarks and defined reliability assessment criteria,Coverage diagnostics completed; benchmark reliability not assessed
3,3D.9,Consolidated report and phase closure,PENDING,"Subphase reports, decision registers and validation evidence prepared","Final report must explicitly distinguish completed, blocked and partially completed work",Final roadmap status and output audit,Final consolidated report pending


,check,passed
0,3D.6 is explicitly blocked,True
1,3D.7 policy is distinguished from implementation,True
2,3D.8 separates coverage from reliability,True
3,3D.9 remains pending until final report,True
4,Every remaining subphase has an unblock requirement,True


All roadmap checks passed: True
Roadmap entries: 4
Passed: 5
Failed: 0
Saved: /content/phase3_outputs/phase3D_remaining_roadmap_status.csv
Saved: /content/phase3_outputs/phase3D_remaining_roadmap_validation.csv


### Final report and closure

In [121]:
# Phase 3D.9 — Consolidated report and final closure

from pathlib import Path
import pandas as pd

# 1. Consolidate validation evidence from 3D.1–3D.5
validation_groups = {
    "3D.1 Scope and boundaries": ["phase3D_1_validation"],
    "3D.2 Scoring reconstruction feasibility": [
        "phase3D_scoring_boundary_validation"
    ],
    "3D.3 Schedule and innings-stage feasibility": [
        "phase3D_3_validation"
    ],
    "3D.4 Population preparation": [
        "phase3D_4_validation",
        "phase3D_4_2_validation",
        "phase3D_4_3_validation"
    ],
    "3D.5 Benchmark-state and fallback specification": [
        "phase3D_benchmark_input_validation",
        "phase3D_construction_decision_validation",
        "phase3D_comparable_state_validation",
        "phase3D_fallback_policy_validation"
    ]
}

validation_rows = []

for component, object_names in validation_groups.items():
    found_checks = []
    missing_objects = []

    for object_name in object_names:
        obj = globals().get(object_name)

        if not isinstance(obj, pd.DataFrame) or "passed" not in obj.columns:
            missing_objects.append(object_name)
        else:
            found_checks.extend(
                obj["passed"].fillna(False).astype(bool).tolist()
            )

    validation_rows.append({
        "component": component,
        "checks_found": len(found_checks),
        "passed": sum(found_checks),
        "failed": len(found_checks) - sum(found_checks),
        "missing_objects": "; ".join(missing_objects),
        "component_passed": (
            len(found_checks) > 0
            and not missing_objects
            and all(found_checks)
        )
    })

phase3D_final_validation_register = pd.DataFrame(validation_rows)

# 2. Include roadmap status for 3D.6–3D.9
phase3D_final_roadmap = phase3D_remaining_roadmap.copy()

# 3. Required project outputs
required_outputs = [
    "phase3D_first_innings_population.parquet",
    "phase3D_competition_coverage.csv",
    "phase3D_wicket_distribution.csv",
    "phase3D_competition_wicket_coverage.csv",
    "phase3D_missing_competition_wicket_states.csv",
    "phase3D_4_validation_register.csv",
    "phase3D_4_population_summary.csv",
    "phase3D_scoring_reconstruction_boundary_register.csv",
    "phase3D_scoring_reconstruction_boundary_validation.csv",
    "phase3D_benchmark_input_feasibility_register.csv",
    "phase3D_benchmark_input_feasibility_validation.csv",
    "phase3D_benchmark_construction_decision_register.csv",
    "phase3D_benchmark_construction_decision_validation.csv",
    "phase3D_comparable_benchmark_state_specification.csv",
    "phase3D_comparable_benchmark_state_validation.csv",
    "phase3D_benchmark_fallback_policy.csv",
    "phase3D_benchmark_fallback_policy_validation.csv",
    "phase3D_remaining_roadmap_status.csv",
    "phase3D_remaining_roadmap_validation.csv"
]

phase3D_final_output_audit = pd.DataFrame([
    {
        "filename": filename,
        "exists": (OUTPUT_DIR / filename).exists(),
        "size_bytes": (
            (OUTPUT_DIR / filename).stat().st_size
            if (OUTPUT_DIR / filename).exists() else 0
        )
    }
    for filename in required_outputs
])

# 4. Closure conditions
early_work_passed = bool(
    phase3D_final_validation_register["component_passed"].all()
)

roadmap_checks_passed = bool(
    phase3D_remaining_roadmap_validation["passed"].all()
)

roadmap_status_correct = (
    phase3D_final_roadmap.set_index("subphase")["status"].to_dict()
    == {
        "3D.6": "BLOCKED",
        "3D.7": "BLOCKED",
        "3D.8": "PARTIALLY_COMPLETED",
        "3D.9": "PENDING"
    }
)

all_required_outputs_exist = bool(
    phase3D_final_output_audit["exists"].all()
)

phase3D_final_closure_passed = bool(
    early_work_passed
    and roadmap_checks_passed
    and roadmap_status_correct
    and all_required_outputs_exist
)

# 5. Save final validation and output audits
phase3D_final_validation_register.to_csv(
    OUTPUT_DIR / "phase3D_final_validation_register.csv",
    index=False
)

phase3D_final_output_audit.to_csv(
    OUTPUT_DIR / "phase3D_final_output_audit.csv",
    index=False
)

# 6. Generate consolidated Phase 3D report
total_checks = int(phase3D_final_validation_register["checks_found"].sum())
passed_checks = int(phase3D_final_validation_register["passed"].sum())
failed_checks = int(phase3D_final_validation_register["failed"].sum())

report = [
    "# SupeRR Selector Season 2 — Phase 3D Report",
    "",
    "## Phase objective",
    "",
    "Assess first-innings benchmark feasibility, prepare the first-innings "
    "population, define comparable benchmark states and document the "
    "benchmark construction and fallback decisions.",
    "",
    "## Subphase status",
    "",
    "| Subphase | Work | Status |",
    "|---|---|---|",
    "| 3D.1 | Scope and boundaries | COMPLETED |",
    "| 3D.2 | Scoring reconstruction feasibility | COMPLETED — INFEASIBLE WITH CURRENT DATA |",
    "| 3D.3 | Scheduled innings length and stage feasibility | COMPLETED — BLOCKED FOR BENCHMARK USE |",
    "| 3D.4 | First-innings population preparation | COMPLETED |",
    "| 3D.5 | Comparable states and fallback policy | COMPLETED |",
    "| 3D.6 | Expected cumulative scoring benchmark construction | BLOCKED |",
    "| 3D.7 | Adaptive fallback hierarchy implementation | BLOCKED — POLICY DOCUMENTED ONLY |",
    "| 3D.8 | Benchmark coverage and reliability validation | PARTIALLY COMPLETED |",
    "| 3D.9 | Consolidated report and closure | COMPLETED IF CLOSURE CHECKS PASS |",
    "",
    "## First-innings population",
    "",
    "- Approved matches: 813.",
    "- First-innings delivery records: 100,333.",
    "- IPL: 362 matches and 45,139 delivery records.",
    "- T20I: 451 matches and 55,194 delivery records.",
    "- Competition groups: 136.",
    "- Unobserved competition/wicket-state combinations: 255.",
    "- Inconsistent match assignments: 0.",
    "",
    "## Benchmark construction decision",
    "",
    "**No expected cumulative first-innings scoring benchmark was constructed.** "
    "The supplied data does not currently support defensible reconstruction "
    "of cumulative innings score, legal balls remaining or scheduled innings length. "
    "Innings stage is consequently unavailable.",
    "",
    "Pre-delivery wickets fallen remains conditional because it is reconstructed "
    "from prior wicket events. Competition-specific wicket-state coverage is "
    "available for diagnostics only.",
    "",
    "No expected-score curves, scoring deviations or standardized deviations "
    "are approved. No player-level scores or rankings were produced.",
    "",
    "## Fallback policy",
    "",
    "The six-level fallback hierarchy is documented and validated. Its first "
    "five levels are defined but not executable. Returning Unavailable is the "
    "approved terminal rule. Cross-format fallback is prohibited.",
    "",
    "## 3D.8 coverage and reliability distinction",
    "",
    "Population coverage, competition coverage and wicket-state distribution "
    "audits were completed. Benchmark-cell reliability, estimate stability "
    "and fallback performance were not assessed because no benchmark estimates "
    "could be constructed.",
    "",
    "## Outstanding evidence required",
    "",
    "1. Authoritative definitions for `runs` and `extras_2_runs`, or independently "
    "verified innings totals with defensible delivery-level reconciliation.",
    "2. Validated legal-delivery classification and innings-level legal-ball progression.",
    "3. Authoritative scheduled innings length and shortened-innings treatment.",
    "4. Validated innings-stage derivation.",
    "5. After benchmark construction becomes possible: benchmark-cell coverage, "
    "reliability and fallback-performance assessment.",
    "",
    "## Final validation",
    "",
    f"- Validation checks assessed: {total_checks}.",
    f"- Passed: {passed_checks}.",
    f"- Failed: {failed_checks}.",
    f"- Required output files present: "
    f"{int(phase3D_final_output_audit['exists'].sum())}/{len(required_outputs)}.",
    f"- Phase 3D closure checks passed: {phase3D_final_closure_passed}.",
    "",
    "Phase 3D closure records completion of the feasible audit, population "
    "preparation and specification work. It does not mean that expected-score "
    "benchmark construction or benchmark reliability validation was completed.",
    "Formal project approval remains separate."
]

phase3D_final_report_text = "\n".join(report)

if phase3D_final_closure_passed:
    (OUTPUT_DIR / "SupeRR_Selector_S2_Phase_3D_Report.md").write_text(
        phase3D_final_report_text,
        encoding="utf-8"
    )

    phase3D_final_completion_summary = pd.DataFrame([{
        "phase": "3D",
        "status": "CLOSED — AUDITS COMPLETE; BENCHMARK CONSTRUCTION BLOCKED",
        "subphases": "3D.1–3D.9",
        "validation_checks": total_checks,
        "passed_checks": passed_checks,
        "failed_checks": failed_checks,
        "approved_matches": 813,
        "first_innings_delivery_records": 100333,
        "competition_groups": 136,
        "3D_6_status": "BLOCKED",
        "3D_7_status": "BLOCKED",
        "3D_8_status": "PARTIALLY_COMPLETED",
        "expected_score_benchmark_constructed": False,
        "report": "SupeRR_Selector_S2_Phase_3D_Report.md"
    }])

    phase3D_final_completion_summary.to_csv(
        OUTPUT_DIR / "phase3D_completion_summary.csv",
        index=False
    )

display(phase3D_final_validation_register)
display(phase3D_final_roadmap)
display(phase3D_final_output_audit)

print("\nPHASE 3D FINAL CLOSURE")
print("Validation checks:", total_checks)
print("Passed:", passed_checks)
print("Failed:", failed_checks)
print("Roadmap checks passed:", roadmap_checks_passed)
print("Roadmap statuses correct:", roadmap_status_correct)
print(
    "Required output files present:",
    int(phase3D_final_output_audit["exists"].sum()),
    "/",
    len(required_outputs)
)
print("Phase 3D closure passed:", phase3D_final_closure_passed)

if phase3D_final_closure_passed:
    display(phase3D_final_completion_summary)
    print("Final report:", OUTPUT_DIR / "SupeRR_Selector_S2_Phase_3D_Report.md")
    print("Completion summary:", OUTPUT_DIR / "phase3D_completion_summary.csv")
    print("Final validation:", OUTPUT_DIR / "phase3D_final_validation_register.csv")
    print("Output audit:", OUTPUT_DIR / "phase3D_final_output_audit.csv")
else:
    print("CLOSURE NOT GRANTED — inspect failed checks and missing files.")

,component,checks_found,passed,failed,missing_objects,component_passed
0,3D.1 Scope and boundaries,4,4,0,,True
1,3D.2 Scoring reconstruction feasibility,3,3,0,,True
2,3D.3 Schedule and innings-stage feasibility,5,5,0,,True
3,3D.4 Population preparation,15,15,0,,True
4,3D.5 Benchmark-state and fallback specification,24,24,0,,True


,subphase,name,status,work_completed,reason,required_to_unblock,deliverable_status
0,3D.6,Expected cumulative scoring benchmark construction,BLOCKED,Feasibility audit and comparable-state specification,"Cumulative score, legal balls remaining and scheduled innings length are unavailable","Validated scoring reconstruction, legal-ball progression and scheduled innings length",Not constructed
1,3D.7,Adaptive fallback hierarchy implementation,BLOCKED,Six-level fallback policy documented and validated,No valid benchmark estimates or comparable scoring states exist to apply the hierarchy to,Validated benchmark population and initial benchmark estimates,Policy documented; implementation not executed
2,3D.8,Benchmark coverage and reliability validation,PARTIALLY_COMPLETED,"Population coverage, competition coverage and wicket-state distribution audits",Benchmark-cell reliability and fallback performance cannot be tested without benchmark estimates,Constructed benchmarks and defined reliability assessment criteria,Coverage diagnostics completed; benchmark reliability not assessed
3,3D.9,Consolidated report and phase closure,PENDING,"Subphase reports, decision registers and validation evidence prepared","Final report must explicitly distinguish completed, blocked and partially completed work",Final roadmap status and output audit,Final consolidated report pending


,filename,exists,size_bytes
0,phase3D_first_innings_population.parquet,True,1942312
1,phase3D_competition_coverage.csv,True,8489
2,phase3D_wicket_distribution.csv,True,449
3,phase3D_competition_wicket_coverage.csv,True,69080
4,phase3D_missing_competition_wicket_states.csv,True,13335
5,phase3D_4_validation_register.csv,True,885
6,phase3D_4_population_summary.csv,True,277
7,phase3D_scoring_reconstruction_boundary_register.csv,True,1256
8,phase3D_scoring_reconstruction_boundary_validation.csv,True,183
9,phase3D_benchmark_input_feasibility_register.csv,True,1692



PHASE 3D FINAL CLOSURE
Validation checks: 51
Passed: 51
Failed: 0
Roadmap checks passed: True
Roadmap statuses correct: True
Required output files present: 19 / 19
Phase 3D closure passed: True


,phase,status,subphases,validation_checks,passed_checks,failed_checks,approved_matches,first_innings_delivery_records,competition_groups,3D_6_status,3D_7_status,3D_8_status,expected_score_benchmark_constructed,report
0,3D,CLOSED — AUDITS COMPLETE; BENCHMARK CONSTRUCTION BLOCKED,3D.1–3D.9,51,51,0,813,100333,136,BLOCKED,BLOCKED,PARTIALLY_COMPLETED,False,SupeRR_Selector_S2_Phase_3D_Report.md


Final report: /content/phase3_outputs/SupeRR_Selector_S2_Phase_3D_Report.md
Completion summary: /content/phase3_outputs/phase3D_completion_summary.csv
Final validation: /content/phase3_outputs/phase3D_final_validation_register.csv
Output audit: /content/phase3_outputs/phase3D_final_output_audit.csv


# Second-innings benchmark

### Scope and input boundaries

In [122]:
# Phase 3E.1 — Second-innings scope and input boundaries

phase3E_scope = pd.DataFrame([
    {
        "input": "Match and competition identity",
        "required_definition": "Match ID, competition format, competition and season",
        "initial_status": "AVAILABLE_FROM_PHASE_3D",
        "intended_use": "Separate IPL and T20I analysis populations"
    },
    {
        "input": "Batting and bowling team identity",
        "required_definition": "Second-innings batting and bowling teams",
        "initial_status": "TO_VERIFY",
        "intended_use": "Identify chasing and defending sides"
    },
    {
        "input": "Target score",
        "required_definition": "Validated second-innings target",
        "initial_status": "UNVERIFIED",
        "intended_use": "Calculate runs required"
    },
    {
        "input": "Current cumulative innings score",
        "required_definition": "Validated score immediately before each delivery",
        "initial_status": "KNOWN_BLOCKER_FROM_PHASE_3D",
        "intended_use": "Calculate runs required"
    },
    {
        "input": "Runs required",
        "required_definition": "Target score minus current cumulative score",
        "initial_status": "NOT_CONSTRUCTIBLE_UNTIL_INPUTS_VALIDATED",
        "intended_use": "Second-innings match demand"
    },
    {
        "input": "Legal balls remaining",
        "required_definition": "Validated scheduled legal balls remaining",
        "initial_status": "KNOWN_BLOCKER_FROM_PHASE_3D",
        "intended_use": "Calculate required run rate and progression"
    },
    {
        "input": "Required run rate",
        "required_definition": "Runs required relative to validated legal balls remaining",
        "initial_status": "NOT_CONSTRUCTIBLE_UNTIL_INPUTS_VALIDATED",
        "intended_use": "Second-innings pressure indicator"
    },
    {
        "input": "Pre-delivery wickets fallen",
        "required_definition": "Wickets fallen before the current delivery",
        "initial_status": "CONDITIONAL_FROM_PHASE_3D",
        "intended_use": "Wickets remaining"
    },
    {
        "input": "Scheduled innings length",
        "required_definition": "Validated scheduled legal-ball allocation",
        "initial_status": "KNOWN_BLOCKER_FROM_PHASE_3D",
        "intended_use": "Remaining resources and innings progression"
    },
    {
        "input": "Innings stage",
        "required_definition": "Stage derived from validated schedule and progression",
        "initial_status": "UNAVAILABLE_FROM_PHASE_3D",
        "intended_use": "Stage-specific benchmark grouping"
    },
    {
        "input": "Second-innings expected scoring-rate benchmark",
        "required_definition": "Historical expected scoring rate for comparable chase states",
        "initial_status": "NOT_CONSTRUCTED",
        "intended_use": "Contextual comparison of required scoring demand"
    }
])

phase3E_scope_validation = pd.DataFrame([
    {
        "check": "Second-innings target is explicitly unverified",
        "passed": phase3E_scope.loc[
            phase3E_scope["input"] == "Target score", "initial_status"
        ].iloc[0] == "UNVERIFIED"
    },
    {
        "check": "Cumulative score remains blocked",
        "passed": phase3E_scope.loc[
            phase3E_scope["input"] == "Current cumulative innings score",
            "initial_status"
        ].iloc[0] == "KNOWN_BLOCKER_FROM_PHASE_3D"
    },
    {
        "check": "Legal balls remaining remains blocked",
        "passed": phase3E_scope.loc[
            phase3E_scope["input"] == "Legal balls remaining", "initial_status"
        ].iloc[0] == "KNOWN_BLOCKER_FROM_PHASE_3D"
    },
    {
        "check": "Wicket state retains conditional status",
        "passed": phase3E_scope.loc[
            phase3E_scope["input"] == "Pre-delivery wickets fallen",
            "initial_status"
        ].iloc[0] == "CONDITIONAL_FROM_PHASE_3D"
    },
    {
        "check": "No second-innings benchmark is assumed to exist",
        "passed": phase3E_scope.loc[
            phase3E_scope["input"] == "Second-innings expected scoring-rate benchmark",
            "initial_status"
        ].iloc[0] == "NOT_CONSTRUCTED"
    }
])

# Verify inherited Phase 3D objects without modifying them
phase3E_inherited_objects = [
    "phase3D_first_innings_population",
    "phase3D_first_innings_states",
    "phase3_benchmark_population",
    "phase3C_state_schema",
    "phase3C_exception_register"
]

phase3E_object_audit = pd.DataFrame([
    {
        "object": name,
        "available": name in globals(),
        "type": type(globals()[name]).__name__ if name in globals() else "MISSING",
        "rows": len(globals()[name])
        if name in globals() and hasattr(globals()[name], "__len__")
        else None
    }
    for name in phase3E_inherited_objects
])

display(phase3E_scope)
display(phase3E_scope_validation)
display(phase3E_object_audit)

phase3E_scope.to_csv(
    OUTPUT_DIR / "phase3E_scope_and_input_boundaries.csv",
    index=False
)

phase3E_scope_validation.to_csv(
    OUTPUT_DIR / "phase3E_scope_validation.csv",
    index=False
)

phase3E_object_audit.to_csv(
    OUTPUT_DIR / "phase3E_inherited_object_audit.csv",
    index=False
)

print("Scope checks passed:", phase3E_scope_validation["passed"].all())
print("Scope checks:", len(phase3E_scope_validation))
print("Passed:", int(phase3E_scope_validation["passed"].sum()))
print("Failed:", int((~phase3E_scope_validation["passed"]).sum()))
print("Inherited objects available:", int(phase3E_object_audit["available"].sum()),
      "/", len(phase3E_object_audit))

,input,required_definition,initial_status,intended_use
0,Match and competition identity,"Match ID, competition format, competition and season",AVAILABLE_FROM_PHASE_3D,Separate IPL and T20I analysis populations
1,Batting and bowling team identity,Second-innings batting and bowling teams,TO_VERIFY,Identify chasing and defending sides
2,Target score,Validated second-innings target,UNVERIFIED,Calculate runs required
3,Current cumulative innings score,Validated score immediately before each delivery,KNOWN_BLOCKER_FROM_PHASE_3D,Calculate runs required
4,Runs required,Target score minus current cumulative score,NOT_CONSTRUCTIBLE_UNTIL_INPUTS_VALIDATED,Second-innings match demand
5,Legal balls remaining,Validated scheduled legal balls remaining,KNOWN_BLOCKER_FROM_PHASE_3D,Calculate required run rate and progression
6,Required run rate,Runs required relative to validated legal balls remaining,NOT_CONSTRUCTIBLE_UNTIL_INPUTS_VALIDATED,Second-innings pressure indicator
7,Pre-delivery wickets fallen,Wickets fallen before the current delivery,CONDITIONAL_FROM_PHASE_3D,Wickets remaining
8,Scheduled innings length,Validated scheduled legal-ball allocation,KNOWN_BLOCKER_FROM_PHASE_3D,Remaining resources and innings progression
9,Innings stage,Stage derived from validated schedule and progression,UNAVAILABLE_FROM_PHASE_3D,Stage-specific benchmark grouping


,check,passed
0,Second-innings target is explicitly unverified,True
1,Cumulative score remains blocked,True
2,Legal balls remaining remains blocked,True
3,Wicket state retains conditional status,True
4,No second-innings benchmark is assumed to exist,True


,object,available,type,rows
0,phase3D_first_innings_population,True,DataFrame,100333
1,phase3D_first_innings_states,True,DataFrame,100333
2,phase3_benchmark_population,True,DataFrame,813
3,phase3C_state_schema,True,DataFrame,12
4,phase3C_exception_register,True,DataFrame,11


Scope checks passed: True
Scope checks: 5
Passed: 5
Failed: 0
Inherited objects available: 5 / 5


### Target score and runs-required feasibility

In [123]:
# Phase 3E.2 — Second-innings target score and runs-required feasibility

import pandas as pd
from IPython.display import display

required_objects = ["df_source_matches", "df_deliveries", "phase3D_first_innings_population"]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(f"Required objects are missing: {missing_objects}")

# 1. Inspect source schemas without modifying source DataFrames.
schema_rows = []

for object_name in ["df_source_matches", "df_deliveries", "phase3D_first_innings_population"]:
    obj = globals()[object_name]
    for column in obj.columns:
        if any(term in column.lower() for term in [
            "target", "run", "score", "innings", "match", "team",
            "revised", "result", "winner"
        ]):
            schema_rows.append({
                "object": object_name,
                "field": column,
                "dtype": str(obj[column].dtype),
                "non_null": int(obj[column].notna().sum()),
                "null": int(obj[column].isna().sum()),
                "distinct_non_null": int(obj[column].nunique(dropna=True))
            })

phase3E_target_schema_audit = pd.DataFrame(schema_rows)

# 2. Inspect match-level revised-target coverage.
matches = df_source_matches.copy()

if "match_id" not in matches.columns:
    raise KeyError("df_source_matches has no match_id field.")

target_field = "revised_target" if "revised_target" in matches.columns else None

if target_field:
    target_values = pd.to_numeric(matches[target_field], errors="coerce")
    phase3E_target_coverage = pd.DataFrame([{
        "source_field": target_field,
        "source_match_rows": len(matches),
        "non_null_values": int(matches[target_field].notna().sum()),
        "numeric_values": int(target_values.notna().sum()),
        "non_numeric_non_null_values": int(
            (matches[target_field].notna() & target_values.isna()).sum()
        ),
        "distinct_numeric_values": int(target_values.nunique(dropna=True)),
        "minimum_numeric_value": target_values.min(),
        "maximum_numeric_value": target_values.max(),
        "validated_as_official_target": False,
        "reason": "Field semantics and units have not been independently validated."
    }])
else:
    phase3E_target_coverage = pd.DataFrame([{
        "source_field": None,
        "source_match_rows": len(matches),
        "non_null_values": 0,
        "numeric_values": 0,
        "non_numeric_non_null_values": 0,
        "distinct_numeric_values": 0,
        "minimum_numeric_value": None,
        "maximum_numeric_value": None,
        "validated_as_official_target": False,
        "reason": "No revised_target field exists in the match-level source."
    }])

# 3. Check whether match-level target values are unique per match.
if target_field:
    target_per_match = (
        matches.groupby("match_id", dropna=False)[target_field]
        .agg(
            source_rows="size",
            non_null_values="count",
            distinct_non_null_values="nunique"
        )
        .reset_index()
    )
    target_per_match["multiple_distinct_targets"] = (
        target_per_match["distinct_non_null_values"] > 1
    )
else:
    target_per_match = pd.DataFrame()

# 4. Inspect second-innings delivery coverage and target-field repetition.
deliveries = df_deliveries.copy()

if not {"match_id", "innings"}.issubset(deliveries.columns):
    raise KeyError("df_deliveries must contain match_id and innings.")

second_innings = deliveries.loc[deliveries["innings"] == 2].copy()

if target_field and target_field in second_innings.columns:
    delivery_target = second_innings[target_field]
    phase3E_delivery_target_audit = pd.DataFrame([{
        "second_innings_delivery_rows": len(second_innings),
        "target_field": target_field,
        "non_null_delivery_values": int(delivery_target.notna().sum()),
        "matches_with_target_value": int(
            second_innings.loc[delivery_target.notna(), "match_id"].nunique()
        ),
        "distinct_target_values": int(delivery_target.nunique(dropna=True)),
        "multiple_target_values_within_match": int(
            second_innings.loc[delivery_target.notna()]
            .groupby("match_id")[target_field]
            .nunique()
            .gt(1)
            .sum()
        )
    }])
else:
    phase3E_delivery_target_audit = pd.DataFrame([{
        "second_innings_delivery_rows": len(second_innings),
        "target_field": target_field,
        "non_null_delivery_values": 0,
        "matches_with_target_value": 0,
        "distinct_target_values": 0,
        "multiple_target_values_within_match": 0
    }])

# 5. Determine whether a validated current cumulative score exists.
score_candidates = [
    column for column in deliveries.columns
    if any(term in column.lower() for term in [
        "total_runs", "innings_score", "cumulative_score",
        "current_score", "score_before"
    ])
]

runs_field_audit = pd.DataFrame([{
    "candidate_score_fields": ", ".join(score_candidates) if score_candidates else None,
    "validated_cumulative_score_available": False,
    "reason": (
        "No candidate field is approved as a validated pre-delivery cumulative score. "
        "The existing runs/extras semantics remain unresolved."
    )
}])

# 6. Explicit feasibility decisions.
phase3E_target_decisions = pd.DataFrame([
    {
        "requirement": "Official second-innings target score",
        "evidence": target_field if target_field else "No revised_target field",
        "decision": "UNVERIFIED",
        "permitted_use": "Coverage and consistency audit only"
    },
    {
        "requirement": "Current cumulative innings score",
        "evidence": "No approved pre-delivery cumulative score field",
        "decision": "UNAVAILABLE",
        "permitted_use": "Do not calculate runs required"
    },
    {
        "requirement": "Runs required",
        "evidence": "Target and current score are not both validated",
        "decision": "NOT_CONSTRUCTIBLE",
        "permitted_use": "No derived runs-required values"
    }
])

# 7. Validation checks.
phase3E_target_validation = pd.DataFrame([
    {
        "check": "Source match table was inspected without modifying it",
        "passed": len(matches) == len(df_source_matches)
    },
    {
        "check": "Second-innings delivery subset contains only innings 2",
        "passed": bool(second_innings["innings"].eq(2).all())
    },
    {
        "check": "Revised target is not automatically treated as official target",
        "passed": not bool(
            phase3E_target_coverage["validated_as_official_target"].any()
        )
    },
    {
        "check": "Runs required is not calculated without validated inputs",
        "passed": bool(
            phase3E_target_decisions.loc[
                phase3E_target_decisions["requirement"] == "Runs required",
                "decision"
            ].eq("NOT_CONSTRUCTIBLE").all()
        )
    }
])

# 8. Export audit outputs.
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

phase3E_target_schema_audit.to_csv(
    output_dir / "phase3E_target_schema_audit.csv", index=False
)
phase3E_target_coverage.to_csv(
    output_dir / "phase3E_target_coverage.csv", index=False
)
phase3E_target_per_match = target_per_match
if not phase3E_target_per_match.empty:
    phase3E_target_per_match.to_csv(
        output_dir / "phase3E_target_per_match_audit.csv", index=False
    )
phase3E_delivery_target_audit.to_csv(
    output_dir / "phase3E_delivery_target_audit.csv", index=False
)
runs_field_audit.to_csv(
    output_dir / "phase3E_cumulative_score_field_audit.csv", index=False
)
phase3E_target_decisions.to_csv(
    output_dir / "phase3E_target_feasibility_decisions.csv", index=False
)
phase3E_target_validation.to_csv(
    output_dir / "phase3E_target_validation.csv", index=False
)

# 9. Display compact audit results.
print("TARGET-RELATED SOURCE FIELDS")
display(phase3E_target_schema_audit)

print("MATCH-LEVEL TARGET COVERAGE")
display(phase3E_target_coverage)

if not phase3E_target_per_match.empty:
    print("TARGET CONSISTENCY BY MATCH")
    display(phase3E_target_per_match.head(20))

print("SECOND-INNINGS DELIVERY TARGET AUDIT")
display(phase3E_delivery_target_audit)

print("CUMULATIVE SCORE FIELD AUDIT")
display(runs_field_audit)

print("FEASIBILITY DECISIONS")
display(phase3E_target_decisions)

print("VALIDATION")
display(phase3E_target_validation)

print(
    "Target audit checks passed:",
    int(phase3E_target_validation["passed"].sum()),
    "/",
    len(phase3E_target_validation)
)

TARGET-RELATED SOURCE FIELDS


,object,field,dtype,non_null,null,distinct_non_null
0,df_source_matches,match_id,int64,883,0,883
1,df_source_matches,match_number,int64,883,0,75
2,df_source_matches,match_date,object,883,0,686
3,df_source_matches,home_team_id,int64,883,0,36
4,df_source_matches,home_team,object,883,0,36
5,df_source_matches,away_team_id,int64,883,0,36
6,df_source_matches,away_team,object,883,0,36
7,df_source_matches,home_team_captain_id,float64,839,44,89
8,df_source_matches,away_team_captain_id,float64,838,45,98
9,df_source_matches,toss_win_team_id,float64,868,15,37


MATCH-LEVEL TARGET COVERAGE


,source_field,source_match_rows,non_null_values,numeric_values,non_numeric_non_null_values,distinct_numeric_values,minimum_numeric_value,maximum_numeric_value,validated_as_official_target,reason
0,revised_target,883,23,23,0,23,33.0,213.0,False,Field semantics and units have not been independently validated.


TARGET CONSISTENCY BY MATCH


,match_id,source_rows,non_null_values,distinct_non_null_values,multiple_distinct_targets
0,13511,1,0,0,False
1,13512,1,0,0,False
2,13513,1,0,0,False
3,13514,1,0,0,False
4,13515,1,0,0,False
5,13522,1,0,0,False
6,13523,1,0,0,False
7,13524,1,0,0,False
8,13612,1,1,1,False
9,13613,1,0,0,False


... (5 outputs left out)

,requirement,evidence,decision,permitted_use
0,Official second-innings target score,revised_target,UNVERIFIED,Coverage and consistency audit only
1,Current cumulative innings score,No approved pre-delivery cumulative score field,UNAVAILABLE,Do not calculate runs required
2,Runs required,Target and current score are not both validated,NOT_CONSTRUCTIBLE,No derived runs-required values


VALIDATION


,check,passed
0,Source match table was inspected without modifying it,True
1,Second-innings delivery subset contains only innings 2,True
2,Revised target is not automatically treated as official target,True
3,Runs required is not calculated without validated inputs,True


Target audit checks passed: 4 / 4


### Legal balls remaining and scheduled innings length

In [124]:
# Phase 3E.3 — Second-innings legal-ball and schedule feasibility

import pandas as pd
from IPython.display import display

required_objects = ["df_deliveries", "df_legal_evidence"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(f"Required objects are missing: {missing_objects}")

# Work from copies; preserve source objects.
second_innings = df_deliveries.loc[
    df_deliveries["innings"].eq(2)
].copy()

print("SECOND-INNINGS POPULATION")
print({
    "delivery_rows": len(second_innings),
    "matches": second_innings["match_id"].nunique(),
    "innings_values": sorted(second_innings["innings"].dropna().unique().tolist())
})

# 1. Inspect legal-ball evidence object and its available fields.
print("\nLEGAL-BALL EVIDENCE OBJECT")
print("Type:", type(df_legal_evidence).__name__)
print("Shape:", df_legal_evidence.shape)
print("Columns:", df_legal_evidence.columns.tolist())

# 2. Identify relevant legal-ball evidence fields in the delivery table.
legal_fields = [
    c for c in second_innings.columns
    if any(term in c.lower() for term in [
        "legal_ball", "wide", "overstep", "over_ball_num",
        "innings_ball_num", "overid"
    ])
]

legal_schema_rows = []
for column in legal_fields:
    legal_schema_rows.append({
        "field": column,
        "dtype": str(second_innings[column].dtype),
        "non_null": int(second_innings[column].notna().sum()),
        "null": int(second_innings[column].isna().sum()),
        "distinct_non_null": int(second_innings[column].nunique(dropna=True))
    })

phase3E_legal_schema_audit = pd.DataFrame(legal_schema_rows)

# 3. Summarise legal-ball evidence classes/statuses in second innings.
evidence_summaries = []

for column in [
    "legal_ball_evidence_class",
    "legal_ball_evidence_status",
    "legal_ball_final_treatment"
]:
    if column in second_innings.columns:
        counts = (
            second_innings[column]
            .value_counts(dropna=False)
            .rename_axis("evidence_value")
            .reset_index(name="delivery_rows")
        )
        counts.insert(0, "field", column)
        evidence_summaries.append(counts)

phase3E_legal_evidence_distribution = (
    pd.concat(evidence_summaries, ignore_index=True)
    if evidence_summaries else pd.DataFrame()
)

# 4. Audit revised second-innings schedule values at match level.
schedule_field = (
    "second_inn_revised_overs"
    if "second_inn_revised_overs" in second_innings.columns
    else None
)

if schedule_field:
    schedule_by_match = (
        second_innings.groupby("match_id")[schedule_field]
        .agg(
            delivery_rows="size",
            non_null_values="count",
            distinct_values="nunique",
            observed_value="first"
        )
        .reset_index()
    )
    schedule_by_match["multiple_schedule_values"] = (
        schedule_by_match["distinct_values"] > 1
    )

    schedule_numeric = pd.to_numeric(
        schedule_by_match["observed_value"], errors="coerce"
    )

    phase3E_schedule_coverage = pd.DataFrame([{
        "schedule_field": schedule_field,
        "second_innings_matches": len(schedule_by_match),
        "matches_with_schedule_value": int(
            schedule_by_match["non_null_values"].gt(0).sum()
        ),
        "matches_without_schedule_value": int(
            schedule_by_match["non_null_values"].eq(0).sum()
        ),
        "matches_with_multiple_values": int(
            schedule_by_match["multiple_schedule_values"].sum()
        ),
        "distinct_numeric_values": int(schedule_numeric.nunique(dropna=True)),
        "minimum_observed_value": schedule_numeric.min(),
        "maximum_observed_value": schedule_numeric.max(),
        "units_validated": False,
        "decision": "SPARSE_AND_UNVERIFIED"
    }])
else:
    schedule_by_match = pd.DataFrame()
    phase3E_schedule_coverage = pd.DataFrame([{
        "schedule_field": None,
        "second_innings_matches": second_innings["match_id"].nunique(),
        "matches_with_schedule_value": 0,
        "matches_without_schedule_value": second_innings["match_id"].nunique(),
        "matches_with_multiple_values": 0,
        "distinct_numeric_values": 0,
        "minimum_observed_value": None,
        "maximum_observed_value": None,
        "units_validated": False,
        "decision": "FIELD_NOT_AVAILABLE"
    }])

# 5. Audit sequence fields without treating them as legal-ball counters.
sequence_rows = []

for column in ["innings_ball_num", "overid", "over_ball_num"]:
    if column in second_innings.columns:
        sequence_rows.append({
            "field": column,
            "non_null": int(second_innings[column].notna().sum()),
            "distinct_values": int(second_innings[column].nunique(dropna=True)),
            "minimum": second_innings[column].min(),
            "maximum": second_innings[column].max(),
            "approved_as_legal_ball_counter": False
        })

phase3E_sequence_audit = pd.DataFrame(sequence_rows)

# 6. Retain explicit feasibility decisions.
phase3E_resource_decisions = pd.DataFrame([
    {
        "requirement": "Legal balls remaining",
        "evidence": "Legal-ball evidence is not a validated remaining-ball counter",
        "decision": "UNAVAILABLE",
        "permitted_use": "Evidence classification audit only"
    },
    {
        "requirement": "Scheduled innings length",
        "evidence": "second_inn_revised_overs is sparse and units are unverified",
        "decision": "UNAVAILABLE",
        "permitted_use": "Coverage and consistency audit only"
    },
    {
        "requirement": "Innings progression / stage",
        "evidence": "Sequence fields do not establish legal-ball progression",
        "decision": "UNAVAILABLE",
        "permitted_use": "Do not assign innings stages"
    },
    {
        "requirement": "Required run rate",
        "evidence": "Runs required and legal balls remaining are not validated",
        "decision": "NOT_CONSTRUCTIBLE",
        "permitted_use": "No required-run-rate values"
    }
])

# 7. Validation.
phase3E_resource_validation = pd.DataFrame([
    {
        "check": "Second-innings subset contains only innings 2",
        "passed": bool(second_innings["innings"].eq(2).all())
    },
    {
        "check": "Legal-ball sequence fields are not approved as legal-ball counters",
        "passed": bool(
            phase3E_sequence_audit.empty
            or not phase3E_sequence_audit[
                "approved_as_legal_ball_counter"
            ].any()
        )
    },
    {
        "check": "Revised schedule units remain explicitly unverified",
        "passed": not bool(
            phase3E_schedule_coverage["units_validated"].any()
        )
    },
    {
        "check": "Required run rate is not calculated",
        "passed": bool(
            phase3E_resource_decisions.loc[
                phase3E_resource_decisions["requirement"] == "Required run rate",
                "decision"
            ].eq("NOT_CONSTRUCTIBLE").all()
        )
    }
])

# 8. Export audit outputs.
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

phase3E_legal_schema_audit.to_csv(
    output_dir / "phase3E_legal_ball_schema_audit.csv", index=False
)
phase3E_legal_evidence_distribution.to_csv(
    output_dir / "phase3E_legal_ball_evidence_distribution.csv", index=False
)
phase3E_schedule_coverage.to_csv(
    output_dir / "phase3E_schedule_coverage.csv", index=False
)
if not schedule_by_match.empty:
    schedule_by_match.to_csv(
        output_dir / "phase3E_schedule_by_match_audit.csv", index=False
    )
phase3E_sequence_audit.to_csv(
    output_dir / "phase3E_sequence_field_audit.csv", index=False
)
phase3E_resource_decisions.to_csv(
    output_dir / "phase3E_resource_feasibility_decisions.csv", index=False
)
phase3E_resource_validation.to_csv(
    output_dir / "phase3E_resource_validation.csv", index=False
)

# 9. Display results.
print("\nLEGAL-BALL FIELD SCHEMA")
display(phase3E_legal_schema_audit)

print("\nSECOND-INNINGS LEGAL-BALL EVIDENCE DISTRIBUTION")
display(phase3E_legal_evidence_distribution)

print("\nSECOND-INNINGS SCHEDULE COVERAGE")
display(phase3E_schedule_coverage)

print("\nSEQUENCE FIELD AUDIT")
display(phase3E_sequence_audit)

print("\nRESOURCE FEASIBILITY DECISIONS")
display(phase3E_resource_decisions)

print("\nVALIDATION")
display(phase3E_resource_validation)

print(
    "\nResource audit checks passed:",
    int(phase3E_resource_validation["passed"].sum()),
    "/",
    len(phase3E_resource_validation)
)

SECOND-INNINGS POPULATION
{'delivery_rows': 91902, 'matches': 813, 'innings_values': [2]}

LEGAL-BALL EVIDENCE OBJECT
Type: DataFrame
Shape: (192235, 36)
Columns: ['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time', 'legal_ball_evidence_class', 'legal_ball_evidence_status', 'legal_ball_final_treatment']

LEGAL-BALL FIELD SCHEMA


,field,dtype,non_null,null,distinct_non_null
0,overid,int64,91902,0,20
1,over_ball_num,int64,91902,0,13
2,innings_ball_num,int64,91902,0,136
3,legal_ball_evidence_class,object,91902,0,7
4,legal_ball_evidence_status,object,91902,0,3
5,legal_ball_final_treatment,object,91902,0,3



SECOND-INNINGS LEGAL-BALL EVIDENCE DISTRIBUTION


,field,evidence_value,delivery_rows
0,legal_ball_evidence_class,RECORDED_NO_EXTRAS,86564
1,legal_ball_evidence_class,RECORDED_WIDE,3494
2,legal_ball_evidence_class,RECORDED_LEGBYES,1243
3,legal_ball_evidence_class,RECORDED_OVERSTEP,261
4,legal_ball_evidence_class,RECORDED_BYES,219
5,legal_ball_evidence_class,RECORDED_OTHER,120
6,legal_ball_evidence_class,RECORDED_PENALTY,1
7,legal_ball_evidence_status,NO_RECORDED_WIDE_OR_OVERSTEP,88026
8,legal_ball_evidence_status,RECORDED_NON_LEGAL_EVIDENCE,3755
9,legal_ball_evidence_status,UNRESOLVED,121



SECOND-INNINGS SCHEDULE COVERAGE


,schedule_field,second_innings_matches,matches_with_schedule_value,matches_without_schedule_value,matches_with_multiple_values,distinct_numeric_values,minimum_observed_value,maximum_observed_value,units_validated,decision
0,second_inn_revised_overs,813,22,791,0,10,30.0,114.0,False,SPARSE_AND_UNVERIFIED


... (3 outputs left out)

,requirement,evidence,decision,permitted_use
0,Legal balls remaining,Legal-ball evidence is not a validated remaining-ball counter,UNAVAILABLE,Evidence classification audit only
1,Scheduled innings length,second_inn_revised_overs is sparse and units are unverified,UNAVAILABLE,Coverage and consistency audit only
2,Innings progression / stage,Sequence fields do not establish legal-ball progression,UNAVAILABLE,Do not assign innings stages
3,Required run rate,Runs required and legal balls remaining are not validated,NOT_CONSTRUCTIBLE,No required-run-rate values



VALIDATION


,check,passed
0,Second-innings subset contains only innings 2,True
1,Legal-ball sequence fields are not approved as legal-ball counters,True
2,Revised schedule units remain explicitly unverified,True
3,Required run rate is not calculated,True



Resource audit checks passed: 4 / 4


### Wickets remaining and innings-stage feasibility

In [125]:
# Phase 3E.4 — Second-innings wickets remaining and stage feasibility

import pandas as pd
from IPython.display import display

required_objects = [
    "df_deliveries",
    "phase3_pre_delivery_wicket_states"
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(f"Required objects are missing: {missing_objects}")

delivery_keys = [
    "match_id", "innings", "overid", "over_ball_num", "innings_ball_num"
]

if not set(delivery_keys).issubset(df_deliveries.columns):
    raise KeyError("df_deliveries is missing one or more delivery keys.")

if not set(delivery_keys + ["pre_delivery_wickets_fallen"]).issubset(
    phase3_pre_delivery_wicket_states.columns
):
    raise KeyError(
        "Pre-delivery wicket-state object is missing required keys or state field."
    )

# 1. Isolate second innings and preserve only the required state fields.
second_innings = df_deliveries.loc[
    df_deliveries["innings"].eq(2)
].copy()

wicket_states = phase3_pre_delivery_wicket_states[
    delivery_keys + ["pre_delivery_wickets_fallen"]
].copy()

# Check key uniqueness before joining.
duplicate_delivery_keys = int(
    second_innings.duplicated(delivery_keys).sum()
)
duplicate_state_keys = int(
    wicket_states.duplicated(delivery_keys).sum()
)

if duplicate_delivery_keys > 0 or duplicate_state_keys > 0:
    raise ValueError(
        "Duplicate delivery keys detected. Resolve key integrity before merging. "
        f"Deliveries={duplicate_delivery_keys}; states={duplicate_state_keys}"
    )

# 2. One-to-one key alignment; do not silently drop unmatched records.
phase3E_second_innings_wicket_alignment = second_innings.merge(
    wicket_states,
    on=delivery_keys,
    how="left",
    validate="one_to_one",
    indicator="_phase3E_wicket_merge"
)

merge_counts = (
    phase3E_second_innings_wicket_alignment["_phase3E_wicket_merge"]
    .value_counts(dropna=False)
)

both_count = int(merge_counts.get("both", 0))
left_only_count = int(merge_counts.get("left_only", 0))
right_only_count = int(merge_counts.get("right_only", 0))

# 3. Derive wickets remaining only as a conditional state.
phase3E_second_innings_wicket_alignment[
    "wickets_remaining_conditional"
] = (
    10 - phase3E_second_innings_wicket_alignment[
        "pre_delivery_wickets_fallen"
    ]
)

# 4. Audit state validity.
fallen = phase3E_second_innings_wicket_alignment[
    "pre_delivery_wickets_fallen"
]
remaining = phase3E_second_innings_wicket_alignment[
    "wickets_remaining_conditional"
]

state_integrity = pd.DataFrame([{
    "second_innings_delivery_rows": len(
        phase3E_second_innings_wicket_alignment
    ),
    "missing_pre_delivery_wicket_states": int(fallen.isna().sum()),
    "fallen_below_zero": int((fallen.dropna() < 0).sum()),
    "fallen_above_ten": int((fallen.dropna() > 10).sum()),
    "remaining_below_zero": int((remaining.dropna() < 0).sum()),
    "remaining_above_ten": int((remaining.dropna() > 10).sum()),
    "distinct_wickets_fallen": int(fallen.nunique(dropna=True)),
    "distinct_wickets_remaining": int(remaining.nunique(dropna=True))
}])

# 5. Summarise conditional wickets remaining by format.
if "match_format" not in phase3E_second_innings_wicket_alignment.columns:
    raise KeyError("match_format is unavailable in the second-innings data.")

phase3E_wickets_remaining_distribution = (
    phase3E_second_innings_wicket_alignment
    .groupby(
        ["match_format", "wickets_remaining_conditional"],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(["match_format", "wickets_remaining_conditional"])
)

# 6. Match-level format consistency.
phase3E_wicket_match_consistency = (
    phase3E_second_innings_wicket_alignment
    .groupby("match_id")
    .agg(
        distinct_formats=("match_format", "nunique"),
        distinct_batting_teams=("batting_team_id", "nunique"),
        distinct_bowling_teams=("bowling_team_id", "nunique")
    )
    .reset_index()
)

# 7. Explicit stage feasibility decision.
phase3E_stage_decisions = pd.DataFrame([
    {
        "dimension": "Pre-delivery wickets fallen",
        "status": "CONDITIONAL",
        "basis": "Reconstructed prior wicket-event state from Phase 3D",
        "permitted_use": "Conditional state analysis with provenance retained"
    },
    {
        "dimension": "Wickets remaining",
        "status": "CONDITIONAL",
        "basis": "10 minus reconstructed pre-delivery wickets fallen",
        "permitted_use": "Conditional resource indicator only"
    },
    {
        "dimension": "Innings stage",
        "status": "UNAVAILABLE",
        "basis": "Validated legal-ball progression and schedule are unavailable",
        "permitted_use": "Do not assign powerplay, middle, or death stages"
    }
])

# 8. Validation.
phase3E_wicket_validation = pd.DataFrame([
    {
        "check": "Second-innings delivery keys are unique",
        "passed": duplicate_delivery_keys == 0
    },
    {
        "check": "Pre-delivery wicket-state keys are unique",
        "passed": duplicate_state_keys == 0
    },
    {
        "check": "All second-innings delivery keys matched a wicket state",
        "passed": (
            both_count == len(second_innings)
            and left_only_count == 0
            and right_only_count == 0
        )
    },
    {
        "check": "No missing pre-delivery wicket states after alignment",
        "passed": bool(fallen.isna().sum() == 0)
    },
    {
        "check": "Conditional wickets remaining stays within 0–10",
        "passed": bool(
            (remaining.dropna().between(0, 10)).all()
        )
    },
    {
        "check": "Innings stage remains unavailable",
        "passed": bool(
            phase3E_stage_decisions.loc[
                phase3E_stage_decisions["dimension"] == "Innings stage",
                "status"
            ].eq("UNAVAILABLE").all()
        )
    }
])

# 9. Export audit outputs.
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

phase3E_second_innings_wicket_alignment.to_parquet(
    output_dir / "phase3E_second_innings_wicket_states.parquet",
    index=False
)
phase3E_wickets_remaining_distribution.to_csv(
    output_dir / "phase3E_wickets_remaining_distribution.csv",
    index=False
)
state_integrity.to_csv(
    output_dir / "phase3E_wicket_state_integrity.csv",
    index=False
)
phase3E_wicket_match_consistency.to_csv(
    output_dir / "phase3E_wicket_match_consistency.csv",
    index=False
)
phase3E_stage_decisions.to_csv(
    output_dir / "phase3E_wicket_and_stage_decisions.csv",
    index=False
)
phase3E_wicket_validation.to_csv(
    output_dir / "phase3E_wicket_validation.csv",
    index=False
)

# 10. Display results.
print("DELIVERY-KEY ALIGNMENT")
display(pd.DataFrame([{
    "second_innings_delivery_rows": len(second_innings),
    "both": both_count,
    "left_only": left_only_count,
    "right_only": right_only_count,
    "duplicate_delivery_keys": duplicate_delivery_keys,
    "duplicate_state_keys": duplicate_state_keys
}]))

print("\nWICKET STATE INTEGRITY")
display(state_integrity)

print("\nCONDITIONAL WICKETS REMAINING BY FORMAT")
display(phase3E_wickets_remaining_distribution)

print("\nMATCH-LEVEL FORMAT AND TEAM CONSISTENCY")
display(phase3E_wicket_match_consistency.head(20))

print("\nWICKET AND STAGE DECISIONS")
display(phase3E_stage_decisions)

print("\nVALIDATION")
display(phase3E_wicket_validation)

print(
    "\nWicket/stage checks passed:",
    int(phase3E_wicket_validation["passed"].sum()),
    "/",
    len(phase3E_wicket_validation)
)

DELIVERY-KEY ALIGNMENT


,second_innings_delivery_rows,both,left_only,right_only,duplicate_delivery_keys,duplicate_state_keys
0,91902,91902,0,0,0,0



WICKET STATE INTEGRITY


,second_innings_delivery_rows,missing_pre_delivery_wicket_states,fallen_below_zero,fallen_above_ten,remaining_below_zero,remaining_above_ten,distinct_wickets_fallen,distinct_wickets_remaining
0,91902,0,0,0,0,0,10,10



CONDITIONAL WICKETS REMAINING BY FORMAT


,match_format,wickets_remaining_conditional,delivery_records,matches
0,T20,1,492,75
1,T20,2,862,111
2,T20,3,1362,154
3,T20,4,2207,196
4,T20,5,3264,241
5,T20,6,4470,285
6,T20,7,6188,323
7,T20,8,7212,351
8,T20,9,7808,360
9,T20,10,8213,362


... (3 outputs left out)

,dimension,status,basis,permitted_use
0,Pre-delivery wickets fallen,CONDITIONAL,Reconstructed prior wicket-event state from Phase 3D,Conditional state analysis with provenance retained
1,Wickets remaining,CONDITIONAL,10 minus reconstructed pre-delivery wickets fallen,Conditional resource indicator only
2,Innings stage,UNAVAILABLE,Validated legal-ball progression and schedule are unavailable,"Do not assign powerplay, middle, or death stages"



VALIDATION


,check,passed
0,Second-innings delivery keys are unique,True
1,Pre-delivery wicket-state keys are unique,True
2,All second-innings delivery keys matched a wicket state,True
3,No missing pre-delivery wicket states after alignment,True
4,Conditional wickets remaining stays within 0–10,True
5,Innings stage remains unavailable,True



Wicket/stage checks passed: 6 / 6


### Second-innings benchmark-state specification

In [126]:
# Phase 3E.5 — Second-innings benchmark-state specification

import pandas as pd
from IPython.display import display

# 1. Define the proposed benchmark dimensions and their evidence status.
phase3E_benchmark_state_specification = pd.DataFrame([
    {
        "dimension": "Competition format",
        "role": "Benchmark population separation",
        "required_definition": "IPL and T20I kept as separate populations",
        "evidence_status": "AVAILABLE",
        "current_basis": "Approved Phase 1/3D benchmark population",
        "benchmark_use": "Separate benchmark construction by format",
        "fallback_or_restriction": "No cross-format fallback"
    },
    {
        "dimension": "Competition identity",
        "role": "Benchmark grouping",
        "required_definition": "Competition identity within the approved format",
        "evidence_status": "AVAILABLE",
        "current_basis": "Competition identifiers and metadata in Phase 3D population",
        "benchmark_use": "Retain competition provenance and coverage",
        "fallback_or_restriction": "Do not merge unrelated competitions silently"
    },
    {
        "dimension": "Innings stage",
        "role": "Benchmark state dimension",
        "required_definition": "Stage derived from validated scheduled legal balls and progression",
        "evidence_status": "UNAVAILABLE",
        "current_basis": "No validated legal-ball progression or common schedule",
        "benchmark_use": "Not executable",
        "fallback_or_restriction": "Do not infer powerplay, middle, or death stages"
    },
    {
        "dimension": "Wickets remaining",
        "role": "Benchmark state dimension",
        "required_definition": "10 minus pre-delivery wickets fallen",
        "evidence_status": "CONDITIONAL",
        "current_basis": "Reconstructed pre-delivery wicket state from prior wicket events",
        "benchmark_use": "Conditional state grouping only",
        "fallback_or_restriction": "Retain reconstruction provenance; do not call independently verified"
    },
    {
        "dimension": "Remaining legal-ball resources",
        "role": "Benchmark state dimension",
        "required_definition": "Validated scheduled legal balls remaining before each delivery",
        "evidence_status": "UNAVAILABLE",
        "current_basis": "Sequence fields are not legal-ball counters; schedule is sparse/unverified",
        "benchmark_use": "Not executable",
        "fallback_or_restriction": "Do not substitute innings_ball_num or overid"
    },
    {
        "dimension": "Target score",
        "role": "Chase-demand input",
        "required_definition": "Validated official second-innings target",
        "evidence_status": "UNVERIFIED",
        "current_basis": "revised_target present in only 22 second-innings matches; semantics unresolved",
        "benchmark_use": "Not executable",
        "fallback_or_restriction": "Do not treat revised_target as official target"
    },
    {
        "dimension": "Current cumulative score",
        "role": "Chase-demand input",
        "required_definition": "Validated cumulative innings score immediately before each delivery",
        "evidence_status": "UNAVAILABLE",
        "current_basis": "Source runs/extras semantics unresolved",
        "benchmark_use": "Not executable",
        "fallback_or_restriction": "Do not sum unresolved runs and extras fields"
    },
    {
        "dimension": "Runs required",
        "role": "Chase-demand measure",
        "required_definition": "Validated target score minus validated current cumulative score",
        "evidence_status": "NOT_CONSTRUCTIBLE",
        "current_basis": "Target and cumulative score are not both validated",
        "benchmark_use": "Not executable",
        "fallback_or_restriction": "No derived runs-required values"
    },
    {
        "dimension": "Required run rate",
        "role": "Separate chase-demand indicator",
        "required_definition": "Runs required relative to validated legal balls remaining",
        "evidence_status": "NOT_CONSTRUCTIBLE",
        "current_basis": "Runs required and legal balls remaining are unavailable",
        "benchmark_use": "Not executable",
        "fallback_or_restriction": "Do not use as a benchmark conditioning variable initially"
    },
    {
        "dimension": "Expected scoring-rate benchmark",
        "role": "Historical comparison baseline",
        "required_definition": "Expected scoring rate for comparable second-innings states",
        "evidence_status": "NOT_CONSTRUCTED",
        "current_basis": "Required state dimensions cannot currently be assembled",
        "benchmark_use": "Future benchmark target only",
        "fallback_or_restriction": "Do not produce scores or rankings"
    }
])

# 2. Record the initial conditioning policy.
phase3E_conditioning_policy = pd.DataFrame([
    {
        "variable": "Competition format",
        "initial_benchmark_conditioning": "YES",
        "reason": "Keep IPL and T20I populations separate"
    },
    {
        "variable": "Competition identity",
        "initial_benchmark_conditioning": "RETAIN_AS_GROUPING_CONTEXT",
        "reason": "Preserve competition-specific provenance and coverage"
    },
    {
        "variable": "Innings stage",
        "initial_benchmark_conditioning": "YES_IF_VALIDATED",
        "reason": "Phase 0 benchmark design; currently unavailable"
    },
    {
        "variable": "Wickets remaining",
        "initial_benchmark_conditioning": "YES_IF_CONDITIONALLY_ACCEPTED",
        "reason": "Phase 0 benchmark design; reconstructed state is conditional"
    },
    {
        "variable": "Remaining resources",
        "initial_benchmark_conditioning": "YES_IF_VALIDATED",
        "reason": "Phase 0 benchmark design; legal balls remaining unavailable"
    },
    {
        "variable": "Required run rate",
        "initial_benchmark_conditioning": "NO_INITIAL CONDITIONING",
        "reason": "Keep required-rate difficulty separate from expected-rate baseline"
    }
])

# 3. Record implementation constraints.
phase3E_benchmark_constraints = pd.DataFrame([
    {
        "constraint": "Format separation",
        "rule": "No IPL-to-T20I fallback",
        "status": "APPROVED"
    },
    {
        "constraint": "Wicket-state provenance",
        "rule": "Any use must retain conditional reconstruction status",
        "status": "APPROVED"
    },
    {
        "constraint": "Remaining-resource representation",
        "rule": "Do not invent a composite resource index",
        "status": "APPROVED"
    },
    {
        "constraint": "Required run rate",
        "rule": "Do not condition the initial expected scoring-rate benchmark on required run rate",
        "status": "APPROVED"
    },
    {
        "constraint": "Unavailable dimensions",
        "rule": "Do not impute unsupported stage, schedule, score, or legal-ball values",
        "status": "APPROVED"
    },
    {
        "constraint": "Fallback execution",
        "rule": "Document policy only; no benchmark fallback execution in this substep",
        "status": "NOT_EXECUTED"
    }
])

# 4. Validate the specification.
phase3E_benchmark_spec_validation = pd.DataFrame([
    {
        "check": "IPL and T20I are explicitly kept separate",
        "passed": bool(
            phase3E_benchmark_state_specification.loc[
                phase3E_benchmark_state_specification["dimension"] == "Competition format",
                "fallback_or_restriction"
            ].str.contains("No cross-format fallback").all()
        )
    },
    {
        "check": "Wickets remaining retains conditional status",
        "passed": bool(
            phase3E_benchmark_state_specification.loc[
                phase3E_benchmark_state_specification["dimension"] == "Wickets remaining",
                "evidence_status"
            ].eq("CONDITIONAL").all()
        )
    },
    {
        "check": "Innings stage remains unavailable",
        "passed": bool(
            phase3E_benchmark_state_specification.loc[
                phase3E_benchmark_state_specification["dimension"] == "Innings stage",
                "evidence_status"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check": "Remaining legal-ball resources remain unavailable",
        "passed": bool(
            phase3E_benchmark_state_specification.loc[
                phase3E_benchmark_state_specification["dimension"] == "Remaining legal-ball resources",
                "evidence_status"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check": "Required run rate is excluded from initial benchmark conditioning",
        "passed": bool(
            phase3E_conditioning_policy.loc[
                phase3E_conditioning_policy["variable"] == "Required run rate",
                "initial_benchmark_conditioning"
            ].eq("NO_INITIAL CONDITIONING").all()
        )
    },
    {
        "check": "No expected scoring-rate benchmark is claimed as constructed",
        "passed": bool(
            phase3E_benchmark_state_specification.loc[
                phase3E_benchmark_state_specification["dimension"] == "Expected scoring-rate benchmark",
                "evidence_status"
            ].eq("NOT_CONSTRUCTED").all()
        )
    },
    {
        "check": "No unsupported resource composite is authorized",
        "passed": bool(
            phase3E_benchmark_constraints.loc[
                phase3E_benchmark_constraints["constraint"] == "Remaining-resource representation",
                "rule"
            ].str.contains("Do not invent a composite resource index").all()
        )
    }
])

# 5. Export specification and validation.
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

phase3E_benchmark_state_specification.to_csv(
    output_dir / "phase3E_benchmark_state_specification.csv",
    index=False
)
phase3E_conditioning_policy.to_csv(
    output_dir / "phase3E_conditioning_policy.csv",
    index=False
)
phase3E_benchmark_constraints.to_csv(
    output_dir / "phase3E_benchmark_constraints.csv",
    index=False
)
phase3E_benchmark_spec_validation.to_csv(
    output_dir / "phase3E_benchmark_spec_validation.csv",
    index=False
)

# 6. Display outputs.
print("SECOND-INNINGS BENCHMARK-STATE SPECIFICATION")
display(phase3E_benchmark_state_specification)

print("\nINITIAL CONDITIONING POLICY")
display(phase3E_conditioning_policy)

print("\nBENCHMARK CONSTRUCTION CONSTRAINTS")
display(phase3E_benchmark_constraints)

print("\nVALIDATION")
display(phase3E_benchmark_spec_validation)

print(
    "\nBenchmark specification checks passed:",
    int(phase3E_benchmark_spec_validation["passed"].sum()),
    "/",
    len(phase3E_benchmark_spec_validation)
)

SECOND-INNINGS BENCHMARK-STATE SPECIFICATION


,dimension,role,required_definition,evidence_status,current_basis,benchmark_use,fallback_or_restriction
0,Competition format,Benchmark population separation,IPL and T20I kept as separate populations,AVAILABLE,Approved Phase 1/3D benchmark population,Separate benchmark construction by format,No cross-format fallback
1,Competition identity,Benchmark grouping,Competition identity within the approved format,AVAILABLE,Competition identifiers and metadata in Phase 3D population,Retain competition provenance and coverage,Do not merge unrelated competitions silently
2,Innings stage,Benchmark state dimension,Stage derived from validated scheduled legal balls and progression,UNAVAILABLE,No validated legal-ball progression or common schedule,Not executable,"Do not infer powerplay, middle, or death stages"
3,Wickets remaining,Benchmark state dimension,10 minus pre-delivery wickets fallen,CONDITIONAL,Reconstructed pre-delivery wicket state from prior wicket events,Conditional state grouping only,Retain reconstruction provenance; do not call independently verified
4,Remaining legal-ball resources,Benchmark state dimension,Validated scheduled legal balls remaining before each delivery,UNAVAILABLE,Sequence fields are not legal-ball counters; schedule is sparse/unverified,Not executable,Do not substitute innings_ball_num or overid
5,Target score,Chase-demand input,Validated official second-innings target,UNVERIFIED,revised_target present in only 22 second-innings matches; semantics unresolved,Not executable,Do not treat revised_target as official target
6,Current cumulative score,Chase-demand input,Validated cumulative innings score immediately before each delivery,UNAVAILABLE,Source runs/extras semantics unresolved,Not executable,Do not sum unresolved runs and extras fields
7,Runs required,Chase-demand measure,Validated target score minus validated current cumulative score,NOT_CONSTRUCTIBLE,Target and cumulative score are not both validated,Not executable,No derived runs-required values
8,Required run rate,Separate chase-demand indicator,Runs required relative to validated legal balls remaining,NOT_CONSTRUCTIBLE,Runs required and legal balls remaining are unavailable,Not executable,Do not use as a benchmark conditioning variable initially
9,Expected scoring-rate benchmark,Historical comparison baseline,Expected scoring rate for comparable second-innings states,NOT_CONSTRUCTED,Required state dimensions cannot currently be assembled,Future benchmark target only,Do not produce scores or rankings



INITIAL CONDITIONING POLICY


,variable,initial_benchmark_conditioning,reason
0,Competition format,YES,Keep IPL and T20I populations separate
1,Competition identity,RETAIN_AS_GROUPING_CONTEXT,Preserve competition-specific provenance and coverage
2,Innings stage,YES_IF_VALIDATED,Phase 0 benchmark design; currently unavailable
3,Wickets remaining,YES_IF_CONDITIONALLY_ACCEPTED,Phase 0 benchmark design; reconstructed state is conditional
4,Remaining resources,YES_IF_VALIDATED,Phase 0 benchmark design; legal balls remaining unavailable
5,Required run rate,NO_INITIAL CONDITIONING,Keep required-rate difficulty separate from expected-rate baseline



BENCHMARK CONSTRUCTION CONSTRAINTS


,constraint,rule,status
0,Format separation,No IPL-to-T20I fallback,APPROVED
1,Wicket-state provenance,Any use must retain conditional reconstruction status,APPROVED
2,Remaining-resource representation,Do not invent a composite resource index,APPROVED
3,Required run rate,Do not condition the initial expected scoring-rate benchmark on required run rate,APPROVED
4,Unavailable dimensions,"Do not impute unsupported stage, schedule, score, or legal-ball values",APPROVED
5,Fallback execution,Document policy only; no benchmark fallback execution in this substep,NOT_EXECUTED



VALIDATION


,check,passed
0,IPL and T20I are explicitly kept separate,True
1,Wickets remaining retains conditional status,True
2,Innings stage remains unavailable,True
3,Remaining legal-ball resources remain unavailable,True
4,Required run rate is excluded from initial benchmark conditioning,True
5,No expected scoring-rate benchmark is claimed as constructed,True
6,No unsupported resource composite is authorized,True



Benchmark specification checks passed: 7 / 7


### Feasibility blockers and fallback rules

In [128]:
# 1. Explicit blocker register.
phase3E_blocker_register = pd.DataFrame([
    {
        "blocker_id": "3E-B01",
        "dimension": "Target score",
        "evidence": "revised_target is populated in 22 of 813 second-innings matches",
        "status": "UNVERIFIED",
        "effect": "Runs required cannot be calculated",
        "resolution_required": "Authoritative target definition, units, and match-level validation"
    },
    {
        "blocker_id": "3E-B02",
        "dimension": "Current cumulative score",
        "evidence": "No approved pre-delivery cumulative score; runs/extras semantics unresolved",
        "status": "UNAVAILABLE",
        "effect": "Runs required cannot be calculated",
        "resolution_required": "Validated batter runs, extras, total delivery runs, and cumulative score reconstruction"
    },
    {
        "blocker_id": "3E-B03",
        "dimension": "Legal balls remaining",
        "evidence": "121 second-innings legal-ball evidence records unresolved; sequence fields are not legal-ball counters",
        "status": "UNAVAILABLE",
        "effect": "Required run rate and remaining resources cannot be calculated",
        "resolution_required": "Validated legal-ball classification and progression"
    },
    {
        "blocker_id": "3E-B04",
        "dimension": "Scheduled innings length",
        "evidence": "second_inn_revised_overs populated in only 22 of 813 matches; units unresolved",
        "status": "UNAVAILABLE",
        "effect": "Cannot establish scheduled legal-ball allocation",
        "resolution_required": "Authoritative schedule metadata and unit definition"
    },
    {
        "blocker_id": "3E-B05",
        "dimension": "Innings stage",
        "evidence": "Depends on validated schedule and legal-ball progression",
        "status": "UNAVAILABLE",
        "effect": "Stage-specific benchmark grouping cannot be implemented",
        "resolution_required": "Validated schedule and legal-ball progression"
    },
    {
        "blocker_id": "3E-B06",
        "dimension": "Wickets remaining",
        "evidence": "Derived from reconstructed pre-delivery wicket events",
        "status": "CONDITIONAL",
        "effect": "Wicket-state grouping has reconstruction uncertainty",
        "resolution_required": "Retain provenance; independently validate dismissal reconstruction if possible"
    },
    {
        "blocker_id": "3E-B07",
        "dimension": "Competition/state coverage",
        "evidence": "Competition-specific state coverage may be sparse once valid chase-state dimensions exist",
        "status": "NOT_YET_ASSESSABLE",
        "effect": "Benchmark reliability and fallback need cannot be measured",
        "resolution_required": "Reassess coverage only after valid state variables are available"
    }
])

# 2. Fallback hierarchy: policy only, not execution.
phase3E_fallback_policy = pd.DataFrame([
    {
        "priority": 1,
        "fallback_level": "Exact comparable state",
        "rule": "Use observations matching format, competition context, innings stage, wickets remaining, and remaining resources",
        "execution_status": "NOT_EXECUTABLE",
        "reason": "Stage and remaining resources are unavailable"
    },
    {
        "priority": 2,
        "fallback_level": "Nearby scheduled innings lengths",
        "rule": "Use nearby validated scheduled lengths when exact schedule state is sparse",
        "execution_status": "NOT_EXECUTABLE",
        "reason": "Scheduled innings length and units are unverified"
    },
    {
        "priority": 3,
        "fallback_level": "Adjacent wicket states",
        "rule": "Use observation-weighted adjacent wicket-state estimates when exact wicket-state coverage is sparse",
        "execution_status": "POLICY_DEFINED_ONLY",
        "reason": "Wickets remaining is conditional; other required state dimensions remain unavailable"
    },
    {
        "priority": 4,
        "fallback_level": "Symmetric state expansion",
        "rule": "Expand comparable states symmetrically and record the exact expansion",
        "execution_status": "NOT_EXECUTABLE",
        "reason": "Valid schedule, stage, and remaining-resource coordinates are unavailable"
    },
    {
        "priority": 5,
        "fallback_level": "Broader stage grouping",
        "rule": "Use broader stage groups only if stage definitions are validated and grouping remains defensible",
        "execution_status": "NOT_EXECUTABLE",
        "reason": "Innings stage is unavailable"
    },
    {
        "priority": 6,
        "fallback_level": "Unavailable terminal rule",
        "rule": "Mark the benchmark unavailable when valid comparable states cannot be established or estimates remain materially unstable",
        "execution_status": "APPROVED_TERMINAL_RULE",
        "reason": "Prevents unsupported benchmark estimates"
    }
])

# 3. Global restrictions for any future implementation.
phase3E_fallback_constraints = pd.DataFrame([
    {
        "constraint": "Format separation",
        "rule": "Keep IPL and T20I separate at every fallback level",
        "status": "APPROVED"
    },
    {
        "constraint": "No cross-competition substitution",
        "rule": "Do not silently substitute unrelated competitions",
        "status": "APPROVED"
    },
    {
        "constraint": "Wicket-state provenance",
        "rule": "Retain conditional reconstruction status in all derived states",
        "status": "APPROVED"
    },
    {
        "constraint": "Observation weighting",
        "rule": "Use observation-weighted estimates for adjacent wicket-state fallback",
        "status": "POLICY_DEFINED"
    },
    {
        "constraint": "Uncertainty and instability",
        "rule": "If exact and pooled estimates materially disagree relative to uncertainty or practical significance, mark unavailable",
        "status": "POLICY_DEFINED"
    },
    {
        "constraint": "No invented thresholds",
        "rule": "Do not invent minimum sample sizes or arbitrary expansion widths",
        "status": "APPROVED"
    },
    {
        "constraint": "No unsupported imputation",
        "rule": "Do not infer target, score, legal balls, schedule, or stage from unrelated fields",
        "status": "APPROVED"
    },
    {
        "constraint": "No benchmark execution",
        "rule": "Do not calculate expected scoring rates, pressure scores, or player rankings in this subphase",
        "status": "APPROVED"
    }
])

# 4. Overall feasibility decision.
phase3E_feasibility_decision = pd.DataFrame([
    {
        "item": "Second-innings expected scoring-rate benchmark",
        "decision": "BLOCKED",
        "reason": "Required target, score, legal-ball, schedule, and stage inputs are not validated"
    },
    {
        "item": "Wickets remaining",
        "decision": "CONDITIONAL",
        "reason": "Reconstructed pre-delivery wicket state aligns to all second-innings deliveries"
    },
    {
        "item": "Required run rate",
        "decision": "NOT_CONSTRUCTIBLE",
        "reason": "Runs required and legal balls remaining are unavailable"
    },
    {
        "item": "Fallback implementation",
        "decision": "POLICY_ONLY",
        "reason": "Fallback coordinates cannot be assembled from current validated fields"
    }
])

# 5. Validation.
phase3E_blocker_validation = pd.DataFrame([
    {
        "check": "Target score remains unverified",
        "passed": bool(
            phase3E_blocker_register.loc[
                phase3E_blocker_register["dimension"] == "Target score",
                "status"
            ].eq("UNVERIFIED").all()
        )
    },
    {
        "check": "Cumulative score remains unavailable",
        "passed": bool(
            phase3E_blocker_register.loc[
                phase3E_blocker_register["dimension"] == "Current cumulative score",
                "status"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check": "Legal balls and schedule remain unavailable",
        "passed": bool(
            phase3E_blocker_register.loc[
                phase3E_blocker_register["dimension"].isin(
                    ["Legal balls remaining", "Scheduled innings length"]
                ),
                "status"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check": "Wickets remaining retains conditional status",
        "passed": bool(
            phase3E_blocker_register.loc[
                phase3E_blocker_register["dimension"] == "Wickets remaining",
                "status"
            ].eq("CONDITIONAL").all()
        )
    },
    {
        "check": "Fallback hierarchy has six ordered levels",
        "passed": (
            phase3E_fallback_policy["priority"].tolist() == [1, 2, 3, 4, 5, 6]
        )
    },
    {
        "check": "Unavailable terminal rule is approved",
        "passed": bool(
            phase3E_fallback_policy.loc[
                phase3E_fallback_policy["priority"] == 6,
                "execution_status"
            ].eq("APPROVED_TERMINAL_RULE").all()
        )
    },
    {
        "check": "No cross-format fallback is allowed",
        "passed": bool(
            phase3E_fallback_constraints.loc[
                phase3E_fallback_constraints["constraint"] == "Format separation",
                "rule"
            ].str.contains("Keep IPL and T20I separate").all()
        )
    },
    {
        "check": "Fallback implementation is not falsely marked complete",
        "passed": bool(
            phase3E_feasibility_decision.loc[
                phase3E_feasibility_decision["item"] == "Fallback implementation",
                "decision"
            ].eq("POLICY_ONLY").all()
        )
    }
])

# 6. Export.
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

phase3E_blocker_register.to_csv(
    output_dir / "phase3E_feasibility_blocker_register.csv", index=False
)
phase3E_fallback_policy.to_csv(
    output_dir / "phase3E_fallback_policy.csv", index=False
)
phase3E_fallback_constraints.to_csv(
    output_dir / "phase3E_fallback_constraints.csv", index=False
)
phase3E_feasibility_decision.to_csv(
    output_dir / "phase3E_feasibility_decision.csv", index=False
)
phase3E_blocker_validation.to_csv(
    output_dir / "phase3E_blocker_validation.csv", index=False
)

# 7. Display.
print("FEASIBILITY BLOCKER REGISTER")
display(phase3E_blocker_register)

print("\nFALLBACK HIERARCHY")
display(phase3E_fallback_policy)

print("\nFALLBACK CONSTRAINTS")
display(phase3E_fallback_constraints)

print("\nOVERALL FEASIBILITY DECISION")
display(phase3E_feasibility_decision)

print("\nVALIDATION")
display(phase3E_blocker_validation)

print(
    "\nBlocker/fallback checks passed:",
    int(phase3E_blocker_validation["passed"].sum()),
    "/",
    len(phase3E_blocker_validation)
)

FEASIBILITY BLOCKER REGISTER


,blocker_id,dimension,evidence,status,effect,resolution_required
0,3E-B01,Target score,revised_target is populated in 22 of 813 second-innings matches,UNVERIFIED,Runs required cannot be calculated,"Authoritative target definition, units, and match-level validation"
1,3E-B02,Current cumulative score,No approved pre-delivery cumulative score; runs/extras semantics unresolved,UNAVAILABLE,Runs required cannot be calculated,"Validated batter runs, extras, total delivery runs, and cumulative score reconstruction"
2,3E-B03,Legal balls remaining,121 second-innings legal-ball evidence records unresolved; sequence fields are not legal-ball counters,UNAVAILABLE,Required run rate and remaining resources cannot be calculated,Validated legal-ball classification and progression
3,3E-B04,Scheduled innings length,second_inn_revised_overs populated in only 22 of 813 matches; units unresolved,UNAVAILABLE,Cannot establish scheduled legal-ball allocation,Authoritative schedule metadata and unit definition
4,3E-B05,Innings stage,Depends on validated schedule and legal-ball progression,UNAVAILABLE,Stage-specific benchmark grouping cannot be implemented,Validated schedule and legal-ball progression
5,3E-B06,Wickets remaining,Derived from reconstructed pre-delivery wicket events,CONDITIONAL,Wicket-state grouping has reconstruction uncertainty,Retain provenance; independently validate dismissal reconstruction if possible
6,3E-B07,Competition/state coverage,Competition-specific state coverage may be sparse once valid chase-state dimensions exist,NOT_YET_ASSESSABLE,Benchmark reliability and fallback need cannot be measured,Reassess coverage only after valid state variables are available



FALLBACK HIERARCHY


,priority,fallback_level,rule,execution_status,reason
0,1,Exact comparable state,"Use observations matching format, competition context, innings stage, wickets remaining, and remaining resources",NOT_EXECUTABLE,Stage and remaining resources are unavailable
1,2,Nearby scheduled innings lengths,Use nearby validated scheduled lengths when exact schedule state is sparse,NOT_EXECUTABLE,Scheduled innings length and units are unverified
2,3,Adjacent wicket states,Use observation-weighted adjacent wicket-state estimates when exact wicket-state coverage is sparse,POLICY_DEFINED_ONLY,Wickets remaining is conditional; other required state dimensions remain unavailable
3,4,Symmetric state expansion,Expand comparable states symmetrically and record the exact expansion,NOT_EXECUTABLE,"Valid schedule, stage, and remaining-resource coordinates are unavailable"
4,5,Broader stage grouping,Use broader stage groups only if stage definitions are validated and grouping remains defensible,NOT_EXECUTABLE,Innings stage is unavailable
5,6,Unavailable terminal rule,Mark the benchmark unavailable when valid comparable states cannot be established or estimates remain materially unstable,APPROVED_TERMINAL_RULE,Prevents unsupported benchmark estimates



FALLBACK CONSTRAINTS


,constraint,rule,status
0,Format separation,Keep IPL and T20I separate at every fallback level,APPROVED
1,No cross-competition substitution,Do not silently substitute unrelated competitions,APPROVED
2,Wicket-state provenance,Retain conditional reconstruction status in all derived states,APPROVED
3,Observation weighting,Use observation-weighted estimates for adjacent wicket-state fallback,POLICY_DEFINED
4,Uncertainty and instability,"If exact and pooled estimates materially disagree relative to uncertainty or practical significance, mark unavailable",POLICY_DEFINED
5,No invented thresholds,Do not invent minimum sample sizes or arbitrary expansion widths,APPROVED
6,No unsupported imputation,"Do not infer target, score, legal balls, schedule, or stage from unrelated fields",APPROVED
7,No benchmark execution,"Do not calculate expected scoring rates, pressure scores, or player rankings in this subphase",APPROVED



OVERALL FEASIBILITY DECISION


,item,decision,reason
0,Second-innings expected scoring-rate benchmark,BLOCKED,"Required target, score, legal-ball, schedule, and stage inputs are not validated"
1,Wickets remaining,CONDITIONAL,Reconstructed pre-delivery wicket state aligns to all second-innings deliveries
2,Required run rate,NOT_CONSTRUCTIBLE,Runs required and legal balls remaining are unavailable
3,Fallback implementation,POLICY_ONLY,Fallback coordinates cannot be assembled from current validated fields



VALIDATION


,check,passed
0,Target score remains unverified,True
1,Cumulative score remains unavailable,True
2,Legal balls and schedule remain unavailable,True
3,Wickets remaining retains conditional status,True
4,Fallback hierarchy has six ordered levels,True
5,Unavailable terminal rule is approved,True
6,No cross-format fallback is allowed,True
7,Fallback implementation is not falsely marked complete,True



Blocker/fallback checks passed: 8 / 8


### Consolidated report and closure

In [129]:
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Collect all subphase validation registers.
validation_sources = {
    "3E.1_scope": phase3E_scope_validation,
    "3E.2_target": phase3E_target_validation,
    "3E.3_resources": phase3E_resource_validation,
    "3E.4_wickets_stage": phase3E_wicket_validation,
    "3E.5_benchmark_specification": phase3E_benchmark_spec_validation,
    "3E.6_blockers_fallback": phase3E_blocker_validation
}

validation_frames = []
for section, frame in validation_sources.items():
    required_columns = {"check", "passed"}
    if not required_columns.issubset(frame.columns):
        raise ValueError(
            f"Validation register {section} is missing required columns."
        )

    temp = frame[["check", "passed"]].copy()
    temp.insert(0, "section", section)
    temp["passed"] = temp["passed"].astype(bool)
    validation_frames.append(temp)

phase3E_final_validation_register = pd.concat(
    validation_frames, ignore_index=True
)

# 2. Summarise validation totals.
total_checks = len(phase3E_final_validation_register)
passed_checks = int(
    phase3E_final_validation_register["passed"].sum()
)
failed_checks = total_checks - passed_checks

# 3. Create a consolidated status summary.
phase3E_completion_summary = pd.DataFrame([
    {
        "area": "Scope and inherited objects",
        "status": "COMPLETE",
        "finding": "Scope boundaries validated; all five inherited objects available"
    },
    {
        "area": "Target score and runs required",
        "status": "BLOCKED",
        "finding": "revised_target unverified in 22 of 813 second-innings matches; cumulative score unavailable"
    },
    {
        "area": "Legal balls and scheduled length",
        "status": "BLOCKED",
        "finding": "Legal-ball progression and common schedule unavailable; revised schedule values sparse and units unresolved"
    },
    {
        "area": "Wickets remaining",
        "status": "CONDITIONAL",
        "finding": "91,902 second-innings deliveries aligned; state reconstructed from prior wicket events"
    },
    {
        "area": "Innings stage",
        "status": "UNAVAILABLE",
        "finding": "Cannot derive stages without validated schedule and legal-ball progression"
    },
    {
        "area": "Benchmark-state specification",
        "status": "COMPLETE",
        "finding": "Dimensions, conditioning policy, and restrictions documented"
    },
    {
        "area": "Fallback hierarchy",
        "status": "POLICY_ONLY",
        "finding": "Six-level hierarchy documented; execution blocked by unavailable state coordinates"
    },
    {
        "area": "Second-innings expected scoring-rate benchmark",
        "status": "BLOCKED",
        "finding": "Required target, score, resource, and stage inputs are not validated"
    }
])

# 4. Audit expected output files from the 3E substeps.
expected_files = [
    "phase3E_scope_and_input_boundaries.csv",
    "phase3E_scope_validation.csv",
    "phase3E_inherited_object_audit.csv",
    "phase3E_target_schema_audit.csv",
    "phase3E_target_coverage.csv",
    "phase3E_delivery_target_audit.csv",
    "phase3E_cumulative_score_field_audit.csv",
    "phase3E_target_feasibility_decisions.csv",
    "phase3E_target_validation.csv",
    "phase3E_legal_ball_schema_audit.csv",
    "phase3E_legal_ball_evidence_distribution.csv",
    "phase3E_schedule_coverage.csv",
    "phase3E_sequence_field_audit.csv",
    "phase3E_resource_feasibility_decisions.csv",
    "phase3E_resource_validation.csv",
    "phase3E_second_innings_wicket_states.parquet",
    "phase3E_wickets_remaining_distribution.csv",
    "phase3E_wicket_state_integrity.csv",
    "phase3E_wicket_match_consistency.csv",
    "phase3E_wicket_and_stage_decisions.csv",
    "phase3E_wicket_validation.csv",
    "phase3E_benchmark_state_specification.csv",
    "phase3E_conditioning_policy.csv",
    "phase3E_benchmark_constraints.csv",
    "phase3E_benchmark_spec_validation.csv",
    "phase3E_feasibility_blocker_register.csv",
    "phase3E_fallback_policy.csv",
    "phase3E_fallback_constraints.csv",
    "phase3E_feasibility_decision.csv",
    "phase3E_blocker_validation.csv"
]

phase3E_output_audit = pd.DataFrame([
    {
        "file": filename,
        "exists": (output_dir / filename).is_file(),
        "size_bytes": (
            (output_dir / filename).stat().st_size
            if (output_dir / filename).is_file()
            else None
        )
    }
    for filename in expected_files
])

# 5. Final closure checks.
phase3E_closure_validation = pd.DataFrame([
    {
        "check": "All subphase validation registers were collected",
        "passed": len(validation_sources) == 6
    },
    {
        "check": "All collected subphase validation checks passed",
        "passed": failed_checks == 0
    },
    {
        "check": "Second-innings benchmark remains blocked",
        "passed": bool(
            phase3E_completion_summary.loc[
                phase3E_completion_summary["area"]
                == "Second-innings expected scoring-rate benchmark",
                "status"
            ].eq("BLOCKED").all()
        )
    },
    {
        "check": "Wickets remaining retains conditional status",
        "passed": bool(
            phase3E_completion_summary.loc[
                phase3E_completion_summary["area"] == "Wickets remaining",
                "status"
            ].eq("CONDITIONAL").all()
        )
    },
    {
        "check": "Fallback execution is not falsely marked complete",
        "passed": bool(
            phase3E_completion_summary.loc[
                phase3E_completion_summary["area"] == "Fallback hierarchy",
                "status"
            ].eq("POLICY_ONLY").all()
        )
    },
    {
        "check": "All expected subphase output files exist",
        "passed": bool(phase3E_output_audit["exists"].all())
    }
])

# 6. Write consolidated Markdown report.
report_lines = [
    "# SupeRR Selector Season 2 — Phase 3E Report",
    "",
    "## 1. Scope",
    "",
    "Phase 3E assessed the feasibility and specification of second-innings "
    "pressure states. It did not construct a pressure score, expected "
    "scoring-rate benchmark, or player ranking.",
    "",
    "## 2. Population and structural alignment",
    "",
    "- Approved matches assessed: 813.",
    "- Second-innings delivery records: 91,902.",
    "- Second-innings delivery keys matched to reconstructed wicket states: 91,902.",
    "- Duplicate delivery keys and unmatched records: zero.",
    "",
    "## 3. Target and scoring feasibility",
    "",
    "- `revised_target` is populated in 22 of 813 second-innings matches.",
    "- Its semantics and units have not been independently validated.",
    "- No approved pre-delivery cumulative innings score is available.",
    "- Runs required is therefore not constructible.",
    "",
    "## 4. Legal-ball and schedule feasibility",
    "",
    "- Second-innings legal-ball evidence: 88,026 records with no recorded "
    "wide or overstep, 3,755 with recorded non-legal evidence, and 121 unresolved.",
    "- The unresolved category is an evidence classification, not proof that "
    "all such records are illegal deliveries.",
    "- `second_inn_revised_overs` is populated in 22 of 813 matches; units "
    "remain unverified.",
    "- Sequence fields are not approved legal-ball counters.",
    "- Legal balls remaining, scheduled innings length, and innings stage "
    "remain unavailable.",
    "",
    "## 5. Wickets remaining",
    "",
    "- All 91,902 second-innings deliveries matched the reconstructed "
    "pre-delivery wicket-state table.",
    "- No duplicate keys, unmatched rows, or missing wicket states were found.",
    "- Wickets remaining is derived as 10 minus reconstructed wickets fallen.",
    "- This remains conditional because the underlying wicket state is "
    "reconstructed from prior wicket events.",
    "",
    "## 6. Benchmark-state specification",
    "",
    "- IPL and T20I remain separate benchmark populations.",
    "- Competition identity is retained as grouping context.",
    "- Wickets remaining may be used conditionally with provenance retained.",
    "- Innings stage and remaining legal-ball resources are unavailable.",
    "- Target score and cumulative score are not validated.",
    "- Required run rate is excluded from initial expected-scoring-rate "
    "benchmark conditioning, consistent with the Phase 0 design.",
    "- No expected scoring-rate benchmark has been constructed.",
    "",
    "## 7. Fallback policy",
    "",
    "The six-level fallback hierarchy is documented: exact comparable state; "
    "nearby scheduled innings lengths; adjacent wicket states; symmetric "
    "state expansion; broader stage grouping where defensible; and an "
    "unavailable terminal rule.",
    "",
    "Fallback execution was not performed. The required state coordinates "
    "are unavailable or unverified. No cross-format fallback, arbitrary "
    "minimum sample threshold, or unsupported imputation is permitted.",
    "",
    "## 8. Final feasibility decision",
    "",
    "**Second-innings expected scoring-rate benchmark: BLOCKED.**",
    "",
    "Wickets remaining is conditional. Required run rate is not constructible. "
    "Fallback policy is documented but not executable.",
    "",
    "## 9. Validation",
    "",
    f"- Subphase validation checks: {total_checks}.",
    f"- Passed: {passed_checks}.",
    f"- Failed: {failed_checks}.",
    "",
    "## 10. Scope exclusions",
    "",
    "No pressure scores, expected scoring-rate estimates, player metrics, "
    "player rankings, top-10 selections, or Phase 4 work were performed.",
    "",
    "## 11. Closure status",
    "",
    "Phase 3E is technically complete as a feasibility and specification "
    "subphase, subject to the final closure checks and separate formal approval."
]

phase3E_report_path = output_dir / "SupeRR_Selector_S2_Phase_3E_Report.md"
phase3E_report_path.write_text("\n".join(report_lines), encoding="utf-8")

# 7. Export final registers and summaries.
phase3E_final_validation_register.to_csv(
    output_dir / "phase3E_final_validation_register.csv",
    index=False
)
phase3E_completion_summary.to_csv(
    output_dir / "phase3E_completion_summary.csv",
    index=False
)
phase3E_output_audit.to_csv(
    output_dir / "phase3E_output_audit.csv",
    index=False
)
phase3E_closure_validation.to_csv(
    output_dir / "phase3E_closure_validation.csv",
    index=False
)

# 8. Display closure results.
print("SUBPHASE COMPLETION SUMMARY")
display(phase3E_completion_summary)

print("\nCONSOLIDATED VALIDATION TOTALS")
display(pd.DataFrame([{
    "total_checks": total_checks,
    "passed": passed_checks,
    "failed": failed_checks
}]))

print("\nOUTPUT FILE AUDIT")
display(phase3E_output_audit)

print("\nCLOSURE VALIDATION")
display(phase3E_closure_validation)

print("\nReport path:", phase3E_report_path)
print(
    "Phase 3E closure checks passed:",
    int(phase3E_closure_validation["passed"].sum()),
    "/",
    len(phase3E_closure_validation)
)

SUBPHASE COMPLETION SUMMARY


,area,status,finding
0,Scope and inherited objects,COMPLETE,Scope boundaries validated; all five inherited objects available
1,Target score and runs required,BLOCKED,revised_target unverified in 22 of 813 second-innings matches; cumulative score unavailable
2,Legal balls and scheduled length,BLOCKED,Legal-ball progression and common schedule unavailable; revised schedule values sparse and units unresolved
3,Wickets remaining,CONDITIONAL,"91,902 second-innings deliveries aligned; state reconstructed from prior wicket events"
4,Innings stage,UNAVAILABLE,Cannot derive stages without validated schedule and legal-ball progression
5,Benchmark-state specification,COMPLETE,"Dimensions, conditioning policy, and restrictions documented"
6,Fallback hierarchy,POLICY_ONLY,Six-level hierarchy documented; execution blocked by unavailable state coordinates
7,Second-innings expected scoring-rate benchmark,BLOCKED,"Required target, score, resource, and stage inputs are not validated"



CONSOLIDATED VALIDATION TOTALS


,total_checks,passed,failed
0,34,34,0



OUTPUT FILE AUDIT


,file,exists,size_bytes
0,phase3E_scope_and_input_boundaries.csv,True,1512
1,phase3E_scope_validation.csv,True,244
2,phase3E_inherited_object_audit.csv,True,264
3,phase3E_target_schema_audit.csv,True,3515
4,phase3E_target_coverage.csv,True,306
5,phase3E_delivery_target_audit.csv,True,186
6,phase3E_cumulative_score_field_audit.csv,True,205
7,phase3E_target_feasibility_decisions.csv,True,379
8,phase3E_target_validation.csv,True,262
9,phase3E_legal_ball_schema_audit.csv,True,265



CLOSURE VALIDATION


,check,passed
0,All subphase validation registers were collected,True
1,All collected subphase validation checks passed,True
2,Second-innings benchmark remains blocked,True
3,Wickets remaining retains conditional status,True
4,Fallback execution is not falsely marked complete,True
5,All expected subphase output files exist,True



Report path: /content/phase3_outputs/SupeRR_Selector_S2_Phase_3E_Report.md
Phase 3E closure checks passed: 6 / 6


# Pressure scale validation

### Pressure definition and scale-input audit

In [130]:
# Read-only audit: does not modify any source DataFrame.

audit_specs = [
    {
        "innings_context": "First innings",
        "pressure_component": "Scoring position relative to expectation",
        "required_inputs": "Cumulative innings score; expected cumulative score for comparable state",
        "candidate_fields": "runs; cumulative score (not validated); expected score (not constructed)",
        "phase_status": "BLOCKED",
        "reason": "Phase 3D could not validate cumulative scoring or construct the expected-score benchmark."
    },
    {
        "innings_context": "First innings",
        "pressure_component": "Standardized scoring deviation",
        "required_inputs": "Actual cumulative score; expected cumulative score; historical standard deviation for comparable states",
        "candidate_fields": "Scoring deviation inputs and historical benchmark",
        "phase_status": "BLOCKED",
        "reason": "Depends on the unavailable scoring and benchmark inputs; no standardized value can currently be calculated."
    },
    {
        "innings_context": "First innings",
        "pressure_component": "Wickets remaining",
        "required_inputs": "Pre-delivery wickets fallen",
        "candidate_fields": "pre_delivery_wickets_fallen in reconstructed state table",
        "phase_status": "CONDITIONAL",
        "reason": "Reconstructed from prior wicket events; not independently verified dismissal truth."
    },
    {
        "innings_context": "First innings",
        "pressure_component": "Balls remaining",
        "required_inputs": "Validated legal balls remaining and scheduled innings length",
        "candidate_fields": "Legal-ball evidence; overid; over_ball_num; innings_ball_num; first_inn_revised_overs",
        "phase_status": "UNAVAILABLE",
        "reason": "Sequence fields are not legal-ball counters; scheduled-length field is sparse and its units are unresolved."
    },
    {
        "innings_context": "Second innings",
        "pressure_component": "Required-rate difficulty",
        "required_inputs": "Verified target; validated current cumulative score; validated legal balls remaining",
        "candidate_fields": "revised_target; runs; legal-ball evidence; second_inn_revised_overs",
        "phase_status": "BLOCKED",
        "reason": "Target is unverified and sparse; cumulative score and legal balls remaining are unavailable."
    },
    {
        "innings_context": "Second innings",
        "pressure_component": "Expected scoring-rate benchmark",
        "required_inputs": "Competition; innings stage; wickets remaining; remaining resources",
        "candidate_fields": "comp_id; match_format; pre-delivery_wickets_fallen; legal-ball and schedule inputs",
        "phase_status": "BLOCKED",
        "reason": "Competition is available and wickets are conditional, but stage and remaining-resource inputs are unavailable."
    },
    {
        "innings_context": "Second innings",
        "pressure_component": "Wickets remaining",
        "required_inputs": "Pre-delivery wickets fallen",
        "candidate_fields": "pre_delivery_wickets_fallen in reconstructed state table",
        "phase_status": "CONDITIONAL",
        "reason": "Derived as 10 minus reconstructed pre-delivery wickets fallen; retains the same reconstruction limitation."
    },
    {
        "innings_context": "Second innings",
        "pressure_component": "Balls/resources remaining",
        "required_inputs": "Validated legal balls remaining and scheduled innings length",
        "candidate_fields": "Legal-ball evidence; overid; over_ball_num; innings_ball_num; second_inn_revised_overs",
        "phase_status": "UNAVAILABLE",
        "reason": "Legal-ball progression and scheduled innings length are not validated."
    },
    {
        "innings_context": "Both innings",
        "pressure_component": "Competition context",
        "required_inputs": "Competition format and competition identity",
        "candidate_fields": "match_format; comp_id; comp_name",
        "phase_status": "AVAILABLE",
        "reason": "Competition identity and format are available; IPL and T20I remain separate."
    },
    {
        "innings_context": "Both innings",
        "pressure_component": "Innings stage",
        "required_inputs": "Validated innings progress relative to scheduled innings length",
        "candidate_fields": "overid; innings_ball_num; scheduled innings length",
        "phase_status": "UNAVAILABLE",
        "reason": "Sequence positions do not establish legal-ball progress or innings stage."
    }
]

phase3F1_pressure_input_audit = pd.DataFrame(audit_specs)

# Inspect relevant objects and their actual columns without changing them.
objects_to_inspect = [
    "df_deliveries",
    "phase3_pre_delivery_wicket_states",
    "phase3D_first_innings_population",
    "phase3E_second_innings_wicket_alignment",
    "phase3E_benchmark_state_specification",
    "phase3C_state_schema",
    "phase3C_exception_register"
]

object_inventory = []

for object_name in objects_to_inspect:
    obj = globals().get(object_name)
    exists = obj is not None
    is_dataframe = exists and isinstance(obj, pd.DataFrame)

    object_inventory.append({
        "object_name": object_name,
        "exists": exists,
        "is_dataframe": is_dataframe,
        "rows": len(obj) if is_dataframe else None,
        "columns": len(obj.columns) if is_dataframe else None,
        "relevant_columns_present": (
            ", ".join(
                col for col in [
                    "match_id", "innings", "comp_id", "comp_name",
                    "match_format", "runs", "revised_target",
                    "pre_delivery_wickets_fallen",
                    "legal_ball_evidence_class",
                    "legal_ball_evidence_status",
                    "overid", "over_ball_num", "innings_ball_num",
                    "first_inn_revised_overs", "second_inn_revised_overs"
                ]
                if col in obj.columns
            ) if is_dataframe else "NOT AVAILABLE"
        )
    })

phase3F1_object_inventory = pd.DataFrame(object_inventory)

# Basic audit checks.
assert len(phase3F1_pressure_input_audit) == 10
assert phase3F1_pressure_input_audit["pressure_component"].notna().all()
assert phase3F1_pressure_input_audit["phase_status"].isin(
    ["AVAILABLE", "CONDITIONAL", "UNAVAILABLE", "BLOCKED"]
).all()

print("PHASE 3F.1 — PHASE 0 PRESSURE COMPONENT AUDIT")
display(phase3F1_pressure_input_audit)

print("\nEXISTING NOTEBOOK OBJECT INVENTORY")
display(phase3F1_object_inventory)

print("\nSTATUS COUNTS")
display(
    phase3F1_pressure_input_audit["phase_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="component_count")
)

print("\nAUDIT ASSERTIONS: PASSED")

PHASE 3F.1 — PHASE 0 PRESSURE COMPONENT AUDIT


,innings_context,pressure_component,required_inputs,candidate_fields,phase_status,reason
0,First innings,Scoring position relative to expectation,Cumulative innings score; expected cumulative score for comparable state,runs; cumulative score (not validated); expected score (not constructed),BLOCKED,Phase 3D could not validate cumulative scoring or construct the expected-score benchmark.
1,First innings,Standardized scoring deviation,Actual cumulative score; expected cumulative score; historical standard deviation for comparable states,Scoring deviation inputs and historical benchmark,BLOCKED,Depends on the unavailable scoring and benchmark inputs; no standardized value can currently be calculated.
2,First innings,Wickets remaining,Pre-delivery wickets fallen,pre_delivery_wickets_fallen in reconstructed state table,CONDITIONAL,Reconstructed from prior wicket events; not independently verified dismissal truth.
3,First innings,Balls remaining,Validated legal balls remaining and scheduled innings length,Legal-ball evidence; overid; over_ball_num; innings_ball_num; first_inn_revised_overs,UNAVAILABLE,Sequence fields are not legal-ball counters; scheduled-length field is sparse and its units are unresolved.
4,Second innings,Required-rate difficulty,Verified target; validated current cumulative score; validated legal balls remaining,revised_target; runs; legal-ball evidence; second_inn_revised_overs,BLOCKED,Target is unverified and sparse; cumulative score and legal balls remaining are unavailable.
5,Second innings,Expected scoring-rate benchmark,Competition; innings stage; wickets remaining; remaining resources,comp_id; match_format; pre-delivery_wickets_fallen; legal-ball and schedule inputs,BLOCKED,"Competition is available and wickets are conditional, but stage and remaining-resource inputs are unavailable."
6,Second innings,Wickets remaining,Pre-delivery wickets fallen,pre_delivery_wickets_fallen in reconstructed state table,CONDITIONAL,Derived as 10 minus reconstructed pre-delivery wickets fallen; retains the same reconstruction limitation.
7,Second innings,Balls/resources remaining,Validated legal balls remaining and scheduled innings length,Legal-ball evidence; overid; over_ball_num; innings_ball_num; second_inn_revised_overs,UNAVAILABLE,Legal-ball progression and scheduled innings length are not validated.
8,Both innings,Competition context,Competition format and competition identity,match_format; comp_id; comp_name,AVAILABLE,Competition identity and format are available; IPL and T20I remain separate.
9,Both innings,Innings stage,Validated innings progress relative to scheduled innings length,overid; innings_ball_num; scheduled innings length,UNAVAILABLE,Sequence positions do not establish legal-ball progress or innings stage.



EXISTING NOTEBOOK OBJECT INVENTORY


,object_name,exists,is_dataframe,rows,columns,relevant_columns_present
0,df_deliveries,True,True,192235,61,"match_id, innings, comp_id, comp_name, match_format, runs, revised_target, legal_ball_evidence_class, legal_ball_evidence_status, overid, over_ball_num, innings_ball_num, first_inn_revised_overs, second_inn_revised_overs"
1,phase3_pre_delivery_wicket_states,True,True,192235,65,"match_id, innings, comp_id, comp_name, match_format, runs, revised_target, pre_delivery_wickets_fallen, legal_ball_evidence_class, legal_ball_evidence_status, overid, over_ball_num, innings_ball_num, first_inn_revised_overs, second_inn_revised_overs"
2,phase3D_first_innings_population,True,True,100333,69,"match_id, innings, comp_id, comp_name, match_format, runs, revised_target, pre_delivery_wickets_fallen, legal_ball_evidence_class, legal_ball_evidence_status, overid, over_ball_num, innings_ball_num, first_inn_revised_overs, second_inn_revised_overs"
3,phase3E_second_innings_wicket_alignment,True,True,91902,64,"match_id, innings, comp_id, comp_name, match_format, runs, revised_target, pre_delivery_wickets_fallen, legal_ball_evidence_class, legal_ball_evidence_status, overid, over_ball_num, innings_ball_num, first_inn_revised_overs, second_inn_revised_overs"
4,phase3E_benchmark_state_specification,True,True,10,7,
5,phase3C_state_schema,True,True,12,8,
6,phase3C_exception_register,True,True,11,6,



STATUS COUNTS


,status,component_count
0,BLOCKED,4
1,UNAVAILABLE,3
2,CONDITIONAL,2
3,AVAILABLE,1



AUDIT ASSERTIONS: PASSED


In [131]:
# Phase 3F.1 — Formal input decision register

phase3F1_decisions = {
    "AVAILABLE": {
        "decision": "CONTEXT_ONLY",
        "numerical_scale_eligible": False,
        "explanation": "Competition context supports grouping and stratification, not pressure measurement by itself."
    },
    "CONDITIONAL": {
        "decision": "CONDITIONAL_INPUT",
        "numerical_scale_eligible": False,
        "explanation": "Wicket-state input may be used only with its reconstruction limitation explicitly retained."
    },
    "UNAVAILABLE": {
        "decision": "BLOCKED_INPUT",
        "numerical_scale_eligible": False,
        "explanation": "Required resource or stage measurement is not validated in the supplied data."
    },
    "BLOCKED": {
        "decision": "BLOCKED_COMPONENT",
        "numerical_scale_eligible": False,
        "explanation": "The component depends on unavailable or unverified inputs."
    }
}

phase3F1_decision_register = phase3F1_pressure_input_audit.copy()

phase3F1_decision_register["decision"] = (
    phase3F1_decision_register["phase_status"]
    .map(lambda x: phase3F1_decisions[x]["decision"])
)

phase3F1_decision_register["numerical_scale_eligible"] = (
    phase3F1_decision_register["phase_status"]
    .map(lambda x: phase3F1_decisions[x]["numerical_scale_eligible"])
)

phase3F1_decision_register["decision_explanation"] = (
    phase3F1_decision_register["phase_status"]
    .map(lambda x: phase3F1_decisions[x]["explanation"])
)

# Validation
assert len(phase3F1_decision_register) == 10
assert phase3F1_decision_register["decision"].notna().all()
assert phase3F1_decision_register["numerical_scale_eligible"].eq(False).all()
assert (
    phase3F1_decision_register.loc[
        phase3F1_decision_register["phase_status"] == "AVAILABLE",
        "decision"
    ].eq("CONTEXT_ONLY").all()
)
assert (
    phase3F1_decision_register.loc[
        phase3F1_decision_register["phase_status"] == "CONDITIONAL",
        "decision"
    ].eq("CONDITIONAL_INPUT").all()
)

print("PHASE 3F.1 — FORMAL INPUT DECISION REGISTER")
display(
    phase3F1_decision_register[
        [
            "innings_context",
            "pressure_component",
            "phase_status",
            "decision",
            "numerical_scale_eligible",
            "decision_explanation"
        ]
    ]
)

print("\nDECISION COUNTS")
display(
    phase3F1_decision_register["decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="component_count")
)

print("\nVALIDATION: 5/5 PASSED")

PHASE 3F.1 — FORMAL INPUT DECISION REGISTER


,innings_context,pressure_component,phase_status,decision,numerical_scale_eligible,decision_explanation
0,First innings,Scoring position relative to expectation,BLOCKED,BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
1,First innings,Standardized scoring deviation,BLOCKED,BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
2,First innings,Wickets remaining,CONDITIONAL,CONDITIONAL_INPUT,False,Wicket-state input may be used only with its reconstruction limitation explicitly retained.
3,First innings,Balls remaining,UNAVAILABLE,BLOCKED_INPUT,False,Required resource or stage measurement is not validated in the supplied data.
4,Second innings,Required-rate difficulty,BLOCKED,BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
5,Second innings,Expected scoring-rate benchmark,BLOCKED,BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
6,Second innings,Wickets remaining,CONDITIONAL,CONDITIONAL_INPUT,False,Wicket-state input may be used only with its reconstruction limitation explicitly retained.
7,Second innings,Balls/resources remaining,UNAVAILABLE,BLOCKED_INPUT,False,Required resource or stage measurement is not validated in the supplied data.
8,Both innings,Competition context,AVAILABLE,CONTEXT_ONLY,False,"Competition context supports grouping and stratification, not pressure measurement by itself."
9,Both innings,Innings stage,UNAVAILABLE,BLOCKED_INPUT,False,Required resource or stage measurement is not validated in the supplied data.



DECISION COUNTS


,decision,component_count
0,BLOCKED_COMPONENT,4
1,BLOCKED_INPUT,3
2,CONDITIONAL_INPUT,2
3,CONTEXT_ONLY,1



VALIDATION: 5/5 PASSED


### Numerical scale feasibility

#### Feasibility decision framework

In [132]:
# Phase 3F.2.1 — Numerical scale construction feasibility

phase3F2_feasibility = pd.DataFrame([
    {
        "approach": "First-innings numerical pressure scale",
        "conceptual_basis": "Scoring position relative to expected scoring; wickets remaining; balls remaining",
        "input_readiness": "Scoring and expected-score inputs blocked; wickets conditional; balls remaining unavailable",
        "current_constructibility": "BLOCKED",
        "reason": "The central scoring-position measure cannot be calculated, and the resource input is unavailable."
    },
    {
        "approach": "Second-innings numerical pressure scale",
        "conceptual_basis": "Required-rate difficulty; expected scoring-rate benchmark; wickets remaining; remaining resources",
        "input_readiness": "Target unverified; cumulative score unavailable; legal balls and schedule unavailable; wickets conditional",
        "current_constructibility": "BLOCKED",
        "reason": "Required-rate difficulty and its expected-rate benchmark cannot be calculated from validated inputs."
    },
    {
        "approach": "One unified numerical scale across both innings",
        "conceptual_basis": "A common numerical measure of match-state difficulty across both innings",
        "input_readiness": "Both innings-specific central measures blocked; shared wicket input conditional",
        "current_constructibility": "NOT_JUSTIFIED",
        "reason": "The innings frameworks use different central measurements, and no common numerical mapping or calibration rule was defined in Phase 0."
    },
    {
        "approach": "Shared pressure-state framework with separate innings-specific measures",
        "conceptual_basis": "Common pre-delivery state principles, with first- and second-innings measures retained separately",
        "input_readiness": "Competition context available; wickets conditional; innings-specific core inputs blocked",
        "current_constructibility": "CONCEPTUALLY_PREFERRED_BUT_BLOCKED",
        "reason": "This preserves the Phase 0 distinction between innings frameworks without inventing a conversion between them; numerical measures remain blocked."
    }
])

phase3F2_criteria = pd.DataFrame([
    {
        "criterion": "Conceptual definition exists",
        "first_innings": "YES",
        "second_innings": "YES",
        "unified_scale": "NOT_DEFINED",
        "evidence": "Phase 0 defines distinct innings-specific pressure components."
    },
    {
        "criterion": "Central numerical input validated",
        "first_innings": "NO",
        "second_innings": "NO",
        "unified_scale": "NO",
        "evidence": "Cumulative scoring and expected scoring are blocked; target and required-rate inputs are unverified or unavailable."
    },
    {
        "criterion": "Resource input validated",
        "first_innings": "NO",
        "second_innings": "NO",
        "unified_scale": "NO",
        "evidence": "Legal-ball progression and scheduled innings length are not validated."
    },
    {
        "criterion": "Wicket-state input usable",
        "first_innings": "CONDITIONAL",
        "second_innings": "CONDITIONAL",
        "unified_scale": "CONDITIONAL",
        "evidence": "Pre-delivery wickets are reconstructed and retain documented limitations."
    },
    {
        "criterion": "Numerical combination rule defined",
        "first_innings": "NO",
        "second_innings": "NO",
        "unified_scale": "NO",
        "evidence": "Phase 0 does not define weights or a formula combining the pressure components."
    },
    {
        "criterion": "Cross-innings calibration defined",
        "first_innings": "NOT_APPLICABLE",
        "second_innings": "NOT_APPLICABLE",
        "unified_scale": "NO",
        "evidence": "Phase 0 does not define a common numerical mapping between first- and second-innings pressure."
    }
])

# Validation
assert len(phase3F2_feasibility) == 4
assert phase3F2_feasibility["approach"].is_unique
assert phase3F2_feasibility["current_constructibility"].notna().all()
assert len(phase3F2_criteria) == 6
assert phase3F2_criteria["criterion"].is_unique
assert phase3F2_criteria.notna().all().all()

print("PHASE 3F.2.1 — CONSTRUCTION APPROACH FEASIBILITY")
display(phase3F2_feasibility)

print("\nPHASE 3F.2.1 — REQUIRED CRITERIA")
display(phase3F2_criteria)

print("\nCONSTRUCTIBILITY COUNTS")
display(
    phase3F2_feasibility["current_constructibility"]
    .value_counts()
    .rename_axis("constructibility_status")
    .reset_index(name="approach_count")
)

print("\nVALIDATION: 6/6 PASSED")

PHASE 3F.2.1 — CONSTRUCTION APPROACH FEASIBILITY


,approach,conceptual_basis,input_readiness,current_constructibility,reason
0,First-innings numerical pressure scale,Scoring position relative to expected scoring; wickets remaining; balls remaining,Scoring and expected-score inputs blocked; wickets conditional; balls remaining unavailable,BLOCKED,"The central scoring-position measure cannot be calculated, and the resource input is unavailable."
1,Second-innings numerical pressure scale,Required-rate difficulty; expected scoring-rate benchmark; wickets remaining; remaining resources,Target unverified; cumulative score unavailable; legal balls and schedule unavailable; wickets conditional,BLOCKED,Required-rate difficulty and its expected-rate benchmark cannot be calculated from validated inputs.
2,One unified numerical scale across both innings,A common numerical measure of match-state difficulty across both innings,Both innings-specific central measures blocked; shared wicket input conditional,NOT_JUSTIFIED,"The innings frameworks use different central measurements, and no common numerical mapping or calibration rule was defined in Phase 0."
3,Shared pressure-state framework with separate innings-specific measures,"Common pre-delivery state principles, with first- and second-innings measures retained separately",Competition context available; wickets conditional; innings-specific core inputs blocked,CONCEPTUALLY_PREFERRED_BUT_BLOCKED,This preserves the Phase 0 distinction between innings frameworks without inventing a conversion between them; numerical measures remain blocked.



PHASE 3F.2.1 — REQUIRED CRITERIA


,criterion,first_innings,second_innings,unified_scale,evidence
0,Conceptual definition exists,YES,YES,NOT_DEFINED,Phase 0 defines distinct innings-specific pressure components.
1,Central numerical input validated,NO,NO,NO,Cumulative scoring and expected scoring are blocked; target and required-rate inputs are unverified or unavailable.
2,Resource input validated,NO,NO,NO,Legal-ball progression and scheduled innings length are not validated.
3,Wicket-state input usable,CONDITIONAL,CONDITIONAL,CONDITIONAL,Pre-delivery wickets are reconstructed and retain documented limitations.
4,Numerical combination rule defined,NO,NO,NO,Phase 0 does not define weights or a formula combining the pressure components.
5,Cross-innings calibration defined,NOT_APPLICABLE,NOT_APPLICABLE,NO,Phase 0 does not define a common numerical mapping between first- and second-innings pressure.



CONSTRUCTIBILITY COUNTS


,constructibility_status,approach_count
0,BLOCKED,2
1,NOT_JUSTIFIED,1
2,CONCEPTUALLY_PREFERRED_BUT_BLOCKED,1



VALIDATION: 6/6 PASSED


#### Numerical representation and calibration requirements

In [133]:
# Phase 3F.2.2 — Numerical representation and calibration requirements

phase3F2_requirements = pd.DataFrame([
    {
        "requirement": "Pressure construct definition",
        "first_innings": "DEFINED",
        "second_innings": "DEFINED",
        "unified_scale": "NOT_DEFINED",
        "phase0_basis": "Phase 0 defines separate first- and second-innings pressure components.",
        "current_limitation": "No common cross-innings numerical construct is specified."
    },
    {
        "requirement": "Central situation-difficulty measure",
        "first_innings": "DEFINED_BUT_INPUTS_BLOCKED",
        "second_innings": "DEFINED_BUT_INPUTS_BLOCKED",
        "unified_scale": "NOT_DEFINED",
        "phase0_basis": "First innings uses scoring position relative to expectation; second innings uses required-rate difficulty.",
        "current_limitation": "Neither central measure can currently be calculated from validated inputs."
    },
    {
        "requirement": "Wicket-state representation",
        "first_innings": "DEFINED_CONDITIONALLY_AVAILABLE",
        "second_innings": "DEFINED_CONDITIONALLY_AVAILABLE",
        "unified_scale": "CONDITIONAL",
        "phase0_basis": "Wickets remaining are a separate pressure indicator.",
        "current_limitation": "Reconstructed pre-delivery wicket states are not independently verified."
    },
    {
        "requirement": "Remaining-resource representation",
        "first_innings": "DEFINED_BUT_UNAVAILABLE",
        "second_innings": "DEFINED_BUT_UNAVAILABLE",
        "unified_scale": "UNAVAILABLE",
        "phase0_basis": "Balls or remaining resources are pressure components.",
        "current_limitation": "Legal-ball progression and scheduled innings length are not validated."
    },
    {
        "requirement": "Benchmark reference population",
        "first_innings": "PARTIALLY_SPECIFIED",
        "second_innings": "PARTIALLY_SPECIFIED",
        "unified_scale": "NOT_SPECIFIED",
        "phase0_basis": "Comparable historical states and competition-specific benchmarks are required.",
        "current_limitation": "Expected scoring benchmarks cannot currently be constructed."
    },
    {
        "requirement": "Component normalization",
        "first_innings": "NOT_DEFINED",
        "second_innings": "NOT_DEFINED",
        "unified_scale": "NOT_DEFINED",
        "phase0_basis": "Standardized scoring deviation is described for first innings.",
        "current_limitation": "No complete normalization method across all pressure components is specified."
    },
    {
        "requirement": "Component combination rule",
        "first_innings": "NOT_DEFINED",
        "second_innings": "NOT_DEFINED",
        "unified_scale": "NOT_DEFINED",
        "phase0_basis": "Phase 0 initially keeps wickets and balls separate and rejects an arbitrary combined resource index.",
        "current_limitation": "No defensible weights or combination formula have been established."
    },
    {
        "requirement": "Direction and interpretation",
        "first_innings": "PARTIALLY_DEFINED",
        "second_innings": "NOT_FULLY_DEFINED",
        "unified_scale": "NOT_DEFINED",
        "phase0_basis": "Scoring deviation direction is defined; negative means below expectation and positive means above expectation.",
        "current_limitation": "A consistent direction for a complete pressure score is not specified."
    },
    {
        "requirement": "Cross-competition treatment",
        "first_innings": "DEFINED",
        "second_innings": "DEFINED",
        "unified_scale": "DEFINED_IN_PRINCIPLE",
        "phase0_basis": "IPL and T20I benchmarks remain separate; no cross-competition fallback.",
        "current_limitation": "Separate benchmark construction remains blocked by missing state inputs."
    },
    {
        "requirement": "Cross-innings calibration",
        "first_innings": "NOT_APPLICABLE",
        "second_innings": "NOT_APPLICABLE",
        "unified_scale": "NOT_DEFINED",
        "phase0_basis": "No cross-innings numerical conversion rule is specified.",
        "current_limitation": "No evidence supports interpreting equal numerical values as equal pressure across innings."
    },
    {
        "requirement": "Empirical validation",
        "first_innings": "NOT_SPECIFIED",
        "second_innings": "NOT_SPECIFIED",
        "unified_scale": "NOT_SPECIFIED",
        "phase0_basis": "Phase 0 requires evidence for additional benchmark conditioning and stability.",
        "current_limitation": "No complete acceptance criteria for a pressure scale are established."
    },
    {
        "requirement": "Pressure bands or thresholds",
        "first_innings": "NOT_DEFINED",
        "second_innings": "NOT_DEFINED",
        "unified_scale": "NOT_DEFINED",
        "phase0_basis": "No numerical pressure bands are specified.",
        "current_limitation": "No low, medium or high pressure thresholds can be justified from Phase 0."
    }
])

# Validation
assert len(phase3F2_requirements) == 12
assert phase3F2_requirements["requirement"].is_unique
assert phase3F2_requirements.notna().all().all()
assert phase3F2_requirements["phase0_basis"].str.len().gt(0).all()
assert phase3F2_requirements["current_limitation"].str.len().gt(0).all()

print("PHASE 3F.2.2 — NUMERICAL SCALE REQUIREMENTS REGISTER")
display(phase3F2_requirements)

print("\nREQUIREMENT STATUS COUNTS — FIRST INNINGS")
display(
    phase3F2_requirements["first_innings"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="requirement_count")
)

print("\nREQUIREMENT STATUS COUNTS — SECOND INNINGS")
display(
    phase3F2_requirements["second_innings"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="requirement_count")
)

print("\nREQUIREMENT STATUS COUNTS — UNIFIED SCALE")
display(
    phase3F2_requirements["unified_scale"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="requirement_count")
)

print("\nVALIDATION: 5/5 PASSED")

PHASE 3F.2.2 — NUMERICAL SCALE REQUIREMENTS REGISTER


,requirement,first_innings,second_innings,unified_scale,phase0_basis,current_limitation
0,Pressure construct definition,DEFINED,DEFINED,NOT_DEFINED,Phase 0 defines separate first- and second-innings pressure components.,No common cross-innings numerical construct is specified.
1,Central situation-difficulty measure,DEFINED_BUT_INPUTS_BLOCKED,DEFINED_BUT_INPUTS_BLOCKED,NOT_DEFINED,First innings uses scoring position relative to expectation; second innings uses required-rate difficulty.,Neither central measure can currently be calculated from validated inputs.
2,Wicket-state representation,DEFINED_CONDITIONALLY_AVAILABLE,DEFINED_CONDITIONALLY_AVAILABLE,CONDITIONAL,Wickets remaining are a separate pressure indicator.,Reconstructed pre-delivery wicket states are not independently verified.
3,Remaining-resource representation,DEFINED_BUT_UNAVAILABLE,DEFINED_BUT_UNAVAILABLE,UNAVAILABLE,Balls or remaining resources are pressure components.,Legal-ball progression and scheduled innings length are not validated.
4,Benchmark reference population,PARTIALLY_SPECIFIED,PARTIALLY_SPECIFIED,NOT_SPECIFIED,Comparable historical states and competition-specific benchmarks are required.,Expected scoring benchmarks cannot currently be constructed.
5,Component normalization,NOT_DEFINED,NOT_DEFINED,NOT_DEFINED,Standardized scoring deviation is described for first innings.,No complete normalization method across all pressure components is specified.
6,Component combination rule,NOT_DEFINED,NOT_DEFINED,NOT_DEFINED,Phase 0 initially keeps wickets and balls separate and rejects an arbitrary combined resource index.,No defensible weights or combination formula have been established.
7,Direction and interpretation,PARTIALLY_DEFINED,NOT_FULLY_DEFINED,NOT_DEFINED,Scoring deviation direction is defined; negative means below expectation and positive means above expectation.,A consistent direction for a complete pressure score is not specified.
8,Cross-competition treatment,DEFINED,DEFINED,DEFINED_IN_PRINCIPLE,IPL and T20I benchmarks remain separate; no cross-competition fallback.,Separate benchmark construction remains blocked by missing state inputs.
9,Cross-innings calibration,NOT_APPLICABLE,NOT_APPLICABLE,NOT_DEFINED,No cross-innings numerical conversion rule is specified.,No evidence supports interpreting equal numerical values as equal pressure across innings.



REQUIREMENT STATUS COUNTS — FIRST INNINGS


,status,requirement_count
0,NOT_DEFINED,3
1,DEFINED,2
2,DEFINED_BUT_INPUTS_BLOCKED,1
3,DEFINED_BUT_UNAVAILABLE,1
4,DEFINED_CONDITIONALLY_AVAILABLE,1
5,PARTIALLY_SPECIFIED,1
6,PARTIALLY_DEFINED,1
7,NOT_APPLICABLE,1
8,NOT_SPECIFIED,1



REQUIREMENT STATUS COUNTS — SECOND INNINGS


,status,requirement_count
0,NOT_DEFINED,3
1,DEFINED,2
2,DEFINED_BUT_INPUTS_BLOCKED,1
3,DEFINED_BUT_UNAVAILABLE,1
4,DEFINED_CONDITIONALLY_AVAILABLE,1
5,PARTIALLY_SPECIFIED,1
6,NOT_FULLY_DEFINED,1
7,NOT_APPLICABLE,1
8,NOT_SPECIFIED,1



REQUIREMENT STATUS COUNTS — UNIFIED SCALE


,status,requirement_count
0,NOT_DEFINED,7
1,NOT_SPECIFIED,2
2,CONDITIONAL,1
3,UNAVAILABLE,1
4,DEFINED_IN_PRINCIPLE,1



VALIDATION: 5/5 PASSED


#### Construction feasibility decision

In [134]:
phase3F2_architecture_decisions = pd.DataFrame([
    {
        "architecture": "First-innings numerical scale",
        "decision": "BLOCKED",
        "may_proceed_to_numerical_construction": False,
        "reason": "Cumulative scoring, expected scoring benchmark and validated remaining resources are unavailable.",
        "reconsideration_requirements": "Validated cumulative score; comparable-state expected score; legal balls remaining; defensible normalization and combination policy."
    },
    {
        "architecture": "Second-innings numerical scale",
        "decision": "BLOCKED",
        "may_proceed_to_numerical_construction": False,
        "reason": "Verified target, cumulative score, legal balls remaining and expected scoring-rate benchmark are unavailable.",
        "reconsideration_requirements": "Verified target semantics; validated cumulative score; legal balls remaining; scheduled innings length; expected-rate benchmark; defensible normalization and combination policy."
    },
    {
        "architecture": "Unified numerical scale",
        "decision": "NOT_JUSTIFIED",
        "may_proceed_to_numerical_construction": False,
        "reason": "No common numerical construct, cross-innings mapping or calibration evidence is defined.",
        "reconsideration_requirements": "Validated innings-specific measures; common construct definition; cross-innings calibration method; empirical evidence that equal values represent comparable pressure."
    },
    {
        "architecture": "Shared framework with separate innings-specific measures",
        "decision": "PREFERRED_CONCEPTUAL_ARCHITECTURE",
        "may_proceed_to_numerical_construction": False,
        "reason": "Preserves Phase 0's distinct innings frameworks without assuming a common numerical scale; both numerical measures remain blocked.",
        "reconsideration_requirements": "Resolve each innings-specific input blocker; define normalization, direction, combination and empirical validation criteria separately."
    }
])

phase3F2_decision_summary = pd.DataFrame([
    {
        "decision_area": "Conceptual architecture",
        "approved_position": "Shared pressure-state principles with separate innings-specific measures",
        "status": "PREFERRED"
    },
    {
        "decision_area": "First-innings numerical construction",
        "approved_position": "Do not construct with current inputs",
        "status": "BLOCKED"
    },
    {
        "decision_area": "Second-innings numerical construction",
        "approved_position": "Do not construct with current inputs",
        "status": "BLOCKED"
    },
    {
        "decision_area": "Unified numerical construction",
        "approved_position": "Do not construct without a common construct and calibration evidence",
        "status": "NOT_JUSTIFIED"
    },
    {
        "decision_area": "Artificial weights or thresholds",
        "approved_position": "Prohibited in the current feasibility stage",
        "status": "NOT_APPROVED"
    }
])

# Validation
assert len(phase3F2_architecture_decisions) == 4
assert phase3F2_architecture_decisions["architecture"].is_unique
assert phase3F2_architecture_decisions["decision"].notna().all()
assert not phase3F2_architecture_decisions[
    "may_proceed_to_numerical_construction"
].any()
assert len(phase3F2_decision_summary) == 5
assert phase3F2_decision_summary["decision_area"].is_unique
assert phase3F2_decision_summary.notna().all().all()

preferred = phase3F2_architecture_decisions.loc[
    phase3F2_architecture_decisions["architecture"]
    == "Shared framework with separate innings-specific measures"
].iloc[0]

assert preferred["decision"] == "PREFERRED_CONCEPTUAL_ARCHITECTURE"
assert preferred["may_proceed_to_numerical_construction"] == False

print("PHASE 3F.2.3 — FORMAL ARCHITECTURE DECISIONS")
display(phase3F2_architecture_decisions)

print("\nPHASE 3F.2.3 — DECISION SUMMARY")
display(phase3F2_decision_summary)

print("\nDECISION COUNTS")
display(
    phase3F2_architecture_decisions["decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="architecture_count")
)

print("\nVALIDATION: 6/6 PASSED")

PHASE 3F.2.3 — FORMAL ARCHITECTURE DECISIONS


,architecture,decision,may_proceed_to_numerical_construction,reason,reconsideration_requirements
0,First-innings numerical scale,BLOCKED,False,"Cumulative scoring, expected scoring benchmark and validated remaining resources are unavailable.",Validated cumulative score; comparable-state expected score; legal balls remaining; defensible normalization and combination policy.
1,Second-innings numerical scale,BLOCKED,False,"Verified target, cumulative score, legal balls remaining and expected scoring-rate benchmark are unavailable.",Verified target semantics; validated cumulative score; legal balls remaining; scheduled innings length; expected-rate benchmark; defensible normalization and combination policy.
2,Unified numerical scale,NOT_JUSTIFIED,False,"No common numerical construct, cross-innings mapping or calibration evidence is defined.",Validated innings-specific measures; common construct definition; cross-innings calibration method; empirical evidence that equal values represent comparable pressure.
3,Shared framework with separate innings-specific measures,PREFERRED_CONCEPTUAL_ARCHITECTURE,False,Preserves Phase 0's distinct innings frameworks without assuming a common numerical scale; both numerical measures remain blocked.,"Resolve each innings-specific input blocker; define normalization, direction, combination and empirical validation criteria separately."



PHASE 3F.2.3 — DECISION SUMMARY


,decision_area,approved_position,status
0,Conceptual architecture,Shared pressure-state principles with separate innings-specific measures,PREFERRED
1,First-innings numerical construction,Do not construct with current inputs,BLOCKED
2,Second-innings numerical construction,Do not construct with current inputs,BLOCKED
3,Unified numerical construction,Do not construct without a common construct and calibration evidence,NOT_JUSTIFIED
4,Artificial weights or thresholds,Prohibited in the current feasibility stage,NOT_APPROVED



DECISION COUNTS


,decision,architecture_count
0,BLOCKED,2
1,NOT_JUSTIFIED,1
2,PREFERRED_CONCEPTUAL_ARCHITECTURE,1



VALIDATION: 6/6 PASSED


### Directionality and logical consistency

#### Directionality audit

In [136]:
# Phase 3F.3.1 — Corrected validation

# Duplicate component names are permitted across innings.
duplicate_component_audit = (
    phase3F3_directionality
    .groupby("component", dropna=False)
    .agg(
        record_count=("component", "size"),
        innings_contexts=("innings_context", lambda x: " | ".join(x))
    )
    .reset_index()
)

assert len(phase3F3_directionality) == 8
assert phase3F3_directionality["component"].notna().all()
assert phase3F3_directionality["innings_context"].notna().all()
assert phase3F3_directionality.notna().all().all()

# Confirm the expected repeated component is represented in both innings.
wicket_rows = phase3F3_directionality[
    phase3F3_directionality["component"] == "Wickets remaining"
]

assert len(wicket_rows) == 2
assert set(wicket_rows["innings_context"]) == {
    "First innings", "Second innings"
}

assert len(phase3F3_consistency_rules) == 7
assert phase3F3_consistency_rules["rule"].is_unique
assert phase3F3_consistency_rules.notna().all().all()

assert (
    phase3F3_directionality.loc[
        phase3F3_directionality["component"] == "Scoring deviation",
        "relationship_status"
    ].iloc[0] == "DEFINED_WITH_LIMITATION"
)

assert (
    phase3F3_consistency_rules.loc[
        phase3F3_consistency_rules["rule"] ==
        "Keep wickets and remaining balls/resources separate initially",
        "status"
    ].iloc[0] == "REQUIRED"
)

assert (
    phase3F3_consistency_rules.loc[
        phase3F3_consistency_rules["rule"] ==
        "Do not assume equal pressure values across innings",
        "status"
    ].iloc[0] == "NOT_ESTABLISHED"
)

print("DUPLICATE COMPONENT AUDIT")
display(duplicate_component_audit)

print("\nDIRECTIONALITY VALIDATION")
display(phase3F3_directionality)

print("\nCONSISTENCY RULE VALIDATION")
display(phase3F3_consistency_rules)

print("\nVALIDATION: 6/6 PASSED")

DUPLICATE COMPONENT AUDIT


,component,record_count,innings_contexts
0,Balls remaining,1,First innings
1,Competition context,1,Both innings
2,Expected scoring-rate benchmark,1,Second innings
3,Remaining resources,1,Second innings
4,Required run rate,1,Second innings
5,Scoring deviation,1,First innings
6,Wickets remaining,2,First innings | Second innings



DIRECTIONALITY VALIDATION


,innings_context,component,measurement_direction,expected_difficulty_relationship,relationship_status,limitation
0,First innings,Scoring deviation,Negative = below expectation; positive = above expectation,Negative deviation may indicate an adverse scoring position,DEFINED_WITH_LIMITATION,Phase 0 explicitly says scoring deviation is not a standalone pressure measure.
1,First innings,Wickets remaining,Lower wickets remaining = fewer batting resources,Fewer wickets may increase batting-state difficulty,CONCEPTUALLY_EXPECTED,Wicket states are reconstructed and conditional; no numerical pressure effect is defined.
2,First innings,Balls remaining,Fewer balls remaining = less scoring opportunity,Less remaining opportunity may constrain scoring,CONCEPTUALLY_EXPECTED,Legal balls remaining and scheduled innings length are unavailable.
3,Second innings,Required run rate,Higher required rate = greater required scoring pace,A higher required rate may represent a more demanding chase,CONCEPTUALLY_EXPECTED,"Required run rate cannot be calculated from validated target, score and legal-ball inputs."
4,Second innings,Wickets remaining,Lower wickets remaining = fewer batting resources,Fewer wickets may increase chase difficulty,CONCEPTUALLY_EXPECTED,Wicket states are reconstructed and conditional; no numerical pressure effect is defined.
5,Second innings,Remaining resources,Fewer remaining resources = less opportunity to complete the chase,Less remaining opportunity may increase chase difficulty,CONCEPTUALLY_EXPECTED,Legal-ball progression and scheduled innings length are unavailable.
6,Second innings,Expected scoring-rate benchmark,Required rate relative to expected scoring capability,Required rate exceeding expected capability may indicate greater difficulty,DEFINED_IN_PRINCIPLE,The expected scoring-rate benchmark has not been constructed.
7,Both innings,Competition context,No intrinsic high-to-low pressure direction,Provides competition-specific benchmark context,CONTEXT_ONLY,Competition identity does not itself measure pressure.



CONSISTENCY RULE VALIDATION


,rule,status,reason
0,Pre-delivery information only,REQUIRED,Pressure describes the match situation immediately before the delivery.
1,Do not treat scoring deviation as standalone pressure,REQUIRED,Explicit Phase 0 limitation.
2,Keep wickets and remaining balls/resources separate initially,REQUIRED,Phase 0 does not approve an arbitrary combined resource index.
3,Keep IPL and T20I benchmarks separate,REQUIRED,No cross-competition fallback is permitted.
4,Do not condition the initial expected-rate benchmark on required rate,REQUIRED,Additional conditioning requires evidence of improved out-of-sample performance and stability.
5,Do not assume equal pressure values across innings,NOT_ESTABLISHED,No common numerical scale or cross-innings calibration is defined.
6,Do not assign numerical weights or thresholds,NOT_APPROVED,Phase 0 does not define a complete numerical combination method or pressure bands.



VALIDATION: 6/6 PASSED


#### Empirical testability audit

In [137]:
# Phase 3F.3.2 — Empirical testability audit

phase3F3_testability = pd.DataFrame([
    {
        "relationship": "Scoring deviation direction",
        "required_evidence": "Validated cumulative score and expected cumulative score",
        "current_data_support": "UNAVAILABLE",
        "testability": "BLOCKED",
        "reason": "Neither cumulative scoring nor the expected-score benchmark is validated."
    },
    {
        "relationship": "Wickets remaining and batting-state difficulty",
        "required_evidence": "Reliable pre-delivery wicket states and a validated difficulty reference",
        "current_data_support": "CONDITIONAL_WICKET_STATE_ONLY",
        "testability": "PARTIAL_DESCRIPTIVE_ONLY",
        "reason": "Wicket-state distributions can be inspected, but no validated pressure outcome or numerical effect is available."
    },
    {
        "relationship": "Balls remaining and first-innings difficulty",
        "required_evidence": "Legal balls remaining and scheduled innings length",
        "current_data_support": "UNAVAILABLE",
        "testability": "BLOCKED",
        "reason": "Sequence positions do not establish legal balls remaining."
    },
    {
        "relationship": "Required run rate and chase difficulty",
        "required_evidence": "Verified target, validated cumulative score and legal balls remaining",
        "current_data_support": "UNAVAILABLE_OR_UNVERIFIED",
        "testability": "BLOCKED",
        "reason": "The required run rate cannot be calculated reliably."
    },
    {
        "relationship": "Wickets remaining and second-innings difficulty",
        "required_evidence": "Reliable pre-delivery wicket states and a validated difficulty reference",
        "current_data_support": "CONDITIONAL_WICKET_STATE_ONLY",
        "testability": "PARTIAL_DESCRIPTIVE_ONLY",
        "reason": "Wicket states are reconstructed; no validated pressure outcome or numerical effect is available."
    },
    {
        "relationship": "Remaining resources and chase difficulty",
        "required_evidence": "Legal balls remaining and scheduled innings length",
        "current_data_support": "UNAVAILABLE",
        "testability": "BLOCKED",
        "reason": "Legal-ball progression and schedule length are not validated."
    },
    {
        "relationship": "Required rate relative to expected scoring capability",
        "required_evidence": "Validated required rate and competition/state-specific expected scoring-rate benchmark",
        "current_data_support": "UNAVAILABLE",
        "testability": "BLOCKED",
        "reason": "The expected scoring-rate benchmark has not been constructed."
    },
    {
        "relationship": "Cross-innings pressure comparability",
        "required_evidence": "Validated innings-specific measures and cross-innings calibration evidence",
        "current_data_support": "NOT_DEFINED",
        "testability": "NOT_TESTABLE",
        "reason": "No common numerical mapping or calibration method exists."
    }
])

# Validation
assert len(phase3F3_testability) == 8
assert phase3F3_testability["relationship"].is_unique
assert phase3F3_testability.notna().all().all()
assert set(phase3F3_testability["testability"]).issubset({
    "BLOCKED",
    "PARTIAL_DESCRIPTIVE_ONLY",
    "NOT_TESTABLE"
})

print("PHASE 3F.3.2 — EMPIRICAL TESTABILITY REGISTER")
display(phase3F3_testability)

print("\nTESTABILITY COUNTS")
display(
    phase3F3_testability["testability"]
    .value_counts()
    .rename_axis("testability")
    .reset_index(name="relationship_count")
)

print("\nVALIDATION: 4/4 PASSED")

PHASE 3F.3.2 — EMPIRICAL TESTABILITY REGISTER


,relationship,required_evidence,current_data_support,testability,reason
0,Scoring deviation direction,Validated cumulative score and expected cumulative score,UNAVAILABLE,BLOCKED,Neither cumulative scoring nor the expected-score benchmark is validated.
1,Wickets remaining and batting-state difficulty,Reliable pre-delivery wicket states and a validated difficulty reference,CONDITIONAL_WICKET_STATE_ONLY,PARTIAL_DESCRIPTIVE_ONLY,"Wicket-state distributions can be inspected, but no validated pressure outcome or numerical effect is available."
2,Balls remaining and first-innings difficulty,Legal balls remaining and scheduled innings length,UNAVAILABLE,BLOCKED,Sequence positions do not establish legal balls remaining.
3,Required run rate and chase difficulty,"Verified target, validated cumulative score and legal balls remaining",UNAVAILABLE_OR_UNVERIFIED,BLOCKED,The required run rate cannot be calculated reliably.
4,Wickets remaining and second-innings difficulty,Reliable pre-delivery wicket states and a validated difficulty reference,CONDITIONAL_WICKET_STATE_ONLY,PARTIAL_DESCRIPTIVE_ONLY,Wicket states are reconstructed; no validated pressure outcome or numerical effect is available.
5,Remaining resources and chase difficulty,Legal balls remaining and scheduled innings length,UNAVAILABLE,BLOCKED,Legal-ball progression and schedule length are not validated.
6,Required rate relative to expected scoring capability,Validated required rate and competition/state-specific expected scoring-rate benchmark,UNAVAILABLE,BLOCKED,The expected scoring-rate benchmark has not been constructed.
7,Cross-innings pressure comparability,Validated innings-specific measures and cross-innings calibration evidence,NOT_DEFINED,NOT_TESTABLE,No common numerical mapping or calibration method exists.



TESTABILITY COUNTS


,testability,relationship_count
0,BLOCKED,5
1,PARTIAL_DESCRIPTIVE_ONLY,2
2,NOT_TESTABLE,1



VALIDATION: 4/4 PASSED


#### Directionality and logical consistency closure

In [139]:
# Phase 3F.3.3 — Directionality and logical consistency closure

# Verify source objects and their required structure.
required_objects = {
    "phase3F3_directionality": [
        "innings_context",
        "component",
        "relationship_status"
    ],
    "phase3F3_consistency_rules": [
        "rule",
        "status",
        "reason"
    ],
    "phase3F3_testability": [
        "relationship",
        "testability",
        "reason"
    ]
}

for object_name, required_columns in required_objects.items():
    obj = globals().get(object_name)

    assert isinstance(obj, pd.DataFrame), (
        f"{object_name} is missing or is not a DataFrame."
    )

    missing_columns = set(required_columns) - set(obj.columns)

    assert not missing_columns, (
        f"{object_name} is missing columns: {missing_columns}"
    )

# Create closure register from the verified objects.
phase3F3_closure_register = pd.DataFrame([
    {
        "closure_area": "Component directionality",
        "source_records": len(phase3F3_directionality),
        "result": "DOCUMENTED",
        "conclusion": "Expected directions are documented, but most relationships remain conceptual."
    },
    {
        "closure_area": "Logical consistency rules",
        "source_records": len(phase3F3_consistency_rules),
        "result": "DOCUMENTED",
        "conclusion": "Seven methodological constraints are recorded for pressure-scale design."
    },
    {
        "closure_area": "Empirical testability",
        "source_records": len(phase3F3_testability),
        "result": "AUDITED",
        "conclusion": "Five relationships blocked, two partially descriptively inspectable, and one not testable."
    },
    {
        "closure_area": "Numerical pressure validation",
        "source_records": 0,
        "result": "NOT_PERFORMED",
        "conclusion": "No numerical pressure scale exists to validate against observed data."
    }
])

phase3F3_closure_summary = pd.DataFrame([
    {
        "measure": "Directionality components documented",
        "count": len(phase3F3_directionality),
        "status": "COMPLETE"
    },
    {
        "measure": "Logical consistency rules documented",
        "count": len(phase3F3_consistency_rules),
        "status": "COMPLETE"
    },
    {
        "measure": "Empirical relationships assessed",
        "count": len(phase3F3_testability),
        "status": "COMPLETE"
    },
    {
        "measure": "Relationships blocked",
        "count": int((phase3F3_testability["testability"] == "BLOCKED").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Relationships partially descriptively inspectable",
        "count": int((phase3F3_testability["testability"] == "PARTIAL_DESCRIPTIVE_ONLY").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Relationships not testable",
        "count": int((phase3F3_testability["testability"] == "NOT_TESTABLE").sum()),
        "status": "LIMITATION_RECORDED"
    }
])

# Validation without hard-coded record-count assertions.
assert phase3F3_closure_register["closure_area"].is_unique
assert phase3F3_closure_register.notna().all().all()

assert phase3F3_closure_summary["measure"].is_unique
assert phase3F3_closure_summary.notna().all().all()

assert (
    phase3F3_testability["testability"].value_counts().to_dict()
    == {
        "BLOCKED": 5,
        "PARTIAL_DESCRIPTIVE_ONLY": 2,
        "NOT_TESTABLE": 1
    }
)

assert (
    phase3F3_closure_register.loc[
        phase3F3_closure_register["closure_area"]
        == "Numerical pressure validation",
        "result"
    ].iloc[0] == "NOT_PERFORMED"
)

print("PHASE 3F.3.3 — CLOSURE REGISTER")
display(phase3F3_closure_register)

print("\nPHASE 3F.3.3 — CLOSURE SUMMARY")
display(phase3F3_closure_summary)

print("\nVALIDATION: PASSED")

PHASE 3F.3.3 — CLOSURE REGISTER


,closure_area,source_records,result,conclusion
0,Component directionality,8,DOCUMENTED,"Expected directions are documented, but most relationships remain conceptual."
1,Logical consistency rules,7,DOCUMENTED,Seven methodological constraints are recorded for pressure-scale design.
2,Empirical testability,8,AUDITED,"Five relationships blocked, two partially descriptively inspectable, and one not testable."
3,Numerical pressure validation,0,NOT_PERFORMED,No numerical pressure scale exists to validate against observed data.



PHASE 3F.3.3 — CLOSURE SUMMARY


,measure,count,status
0,Directionality components documented,8,COMPLETE
1,Logical consistency rules documented,7,COMPLETE
2,Empirical relationships assessed,8,COMPLETE
3,Relationships blocked,5,LIMITATION_RECORDED
4,Relationships partially descriptively inspectable,2,LIMITATION_RECORDED
5,Relationships not testable,1,LIMITATION_RECORDED



VALIDATION: PASSED


### Coverage and sensitivity feasibility

#### Coverage and sensitivity requirements register

In [140]:
# Phase 3F.4.1 — Coverage and sensitivity requirements

phase3F4_requirements = pd.DataFrame([
    {
        "analysis_area": "First-innings scoring input coverage",
        "required_evidence": "Validated cumulative score and expected cumulative score",
        "current_readiness": "BLOCKED",
        "reason": "Scoring reconstruction and expected benchmark construction were blocked in Phase 3D."
    },
    {
        "analysis_area": "First-innings wicket-state coverage",
        "required_evidence": "Competition-specific pre-delivery wicket-state observations",
        "current_readiness": "CONDITIONAL_AUDIT_AVAILABLE",
        "reason": "Competition and wicket-state distributions were audited in Phase 3D; wicket states remain reconstructed."
    },
    {
        "analysis_area": "First-innings remaining-resource coverage",
        "required_evidence": "Validated legal balls remaining and scheduled innings length",
        "current_readiness": "BLOCKED",
        "reason": "Legal-ball progression and scheduled innings length remain unavailable."
    },
    {
        "analysis_area": "Second-innings target and score coverage",
        "required_evidence": "Verified target and validated cumulative score",
        "current_readiness": "BLOCKED",
        "reason": "Target semantics are unverified and cumulative scoring is unavailable."
    },
    {
        "analysis_area": "Second-innings wicket-state coverage",
        "required_evidence": "Competition-specific pre-delivery wicket-state observations",
        "current_readiness": "CONDITIONAL_AUDIT_AVAILABLE",
        "reason": "Wicket-state distributions were audited in Phase 3E; reconstructed wicket states remain conditional."
    },
    {
        "analysis_area": "Second-innings remaining-resource coverage",
        "required_evidence": "Validated legal balls remaining and scheduled innings length",
        "current_readiness": "BLOCKED",
        "reason": "Legal-ball progression and schedule length remain unavailable."
    },
    {
        "analysis_area": "Expected scoring benchmark coverage",
        "required_evidence": "Validated comparable states and scoring observations",
        "current_readiness": "BLOCKED",
        "reason": "Required state dimensions and cumulative scoring inputs are unavailable."
    },
    {
        "analysis_area": "Pressure classification sensitivity",
        "required_evidence": "Constructed pressure measure and defensible alternative specifications",
        "current_readiness": "NOT_TESTABLE",
        "reason": "No numerical pressure measure or approved alternative specifications exist."
    },
    {
        "analysis_area": "Cross-competition stability",
        "required_evidence": "Constructed competition-specific pressure measures and comparable validation results",
        "current_readiness": "NOT_TESTABLE",
        "reason": "Competition identity is available, but pressure measures and benchmarks are not constructed."
    },
    {
        "analysis_area": "Cross-innings stability",
        "required_evidence": "Validated innings-specific measures and a defined comparison framework",
        "current_readiness": "NOT_TESTABLE",
        "reason": "No common cross-innings calibration or numerical measures exist."
    }
])

# Validation
assert len(phase3F4_requirements) == 10
assert phase3F4_requirements["analysis_area"].is_unique
assert phase3F4_requirements.notna().all().all()

print("PHASE 3F.4.1 — COVERAGE AND SENSITIVITY REQUIREMENTS")
display(phase3F4_requirements)

print("\nREADINESS COUNTS")
display(
    phase3F4_requirements["current_readiness"]
    .value_counts()
    .rename_axis("readiness")
    .reset_index(name="analysis_area_count")
)

print("\nVALIDATION: 3/3 PASSED")

PHASE 3F.4.1 — COVERAGE AND SENSITIVITY REQUIREMENTS


,analysis_area,required_evidence,current_readiness,reason
0,First-innings scoring input coverage,Validated cumulative score and expected cumulative score,BLOCKED,Scoring reconstruction and expected benchmark construction were blocked in Phase 3D.
1,First-innings wicket-state coverage,Competition-specific pre-delivery wicket-state observations,CONDITIONAL_AUDIT_AVAILABLE,Competition and wicket-state distributions were audited in Phase 3D; wicket states remain reconstructed.
2,First-innings remaining-resource coverage,Validated legal balls remaining and scheduled innings length,BLOCKED,Legal-ball progression and scheduled innings length remain unavailable.
3,Second-innings target and score coverage,Verified target and validated cumulative score,BLOCKED,Target semantics are unverified and cumulative scoring is unavailable.
4,Second-innings wicket-state coverage,Competition-specific pre-delivery wicket-state observations,CONDITIONAL_AUDIT_AVAILABLE,Wicket-state distributions were audited in Phase 3E; reconstructed wicket states remain conditional.
5,Second-innings remaining-resource coverage,Validated legal balls remaining and scheduled innings length,BLOCKED,Legal-ball progression and schedule length remain unavailable.
6,Expected scoring benchmark coverage,Validated comparable states and scoring observations,BLOCKED,Required state dimensions and cumulative scoring inputs are unavailable.
7,Pressure classification sensitivity,Constructed pressure measure and defensible alternative specifications,NOT_TESTABLE,No numerical pressure measure or approved alternative specifications exist.
8,Cross-competition stability,Constructed competition-specific pressure measures and comparable validation results,NOT_TESTABLE,"Competition identity is available, but pressure measures and benchmarks are not constructed."
9,Cross-innings stability,Validated innings-specific measures and a defined comparison framework,NOT_TESTABLE,No common cross-innings calibration or numerical measures exist.



READINESS COUNTS


,readiness,analysis_area_count
0,BLOCKED,5
1,NOT_TESTABLE,3
2,CONDITIONAL_AUDIT_AVAILABLE,2



VALIDATION: 3/3 PASSED


#### Existing wicket-state coverage evidence

In [142]:
# Phase 3F.4.2 — Corrected inventory validation

# Validate inventory structure.
assert len(phase3F4_coverage_inventory) == len(coverage_objects)
assert phase3F4_coverage_inventory["object_name"].is_unique

# Validate that existing objects have valid dimensions.
available_objects = phase3F4_coverage_inventory[
    phase3F4_coverage_inventory["exists"] == True
]

assert available_objects["rows"].notna().all()
assert available_objects["columns"].notna().all()
assert available_objects["column_names"].ne("NOT AVAILABLE").all()

# Validate that the unavailable object is explicitly recorded.
unavailable_objects = phase3F4_coverage_inventory[
    phase3F4_coverage_inventory["exists"] == False
]

assert unavailable_objects["object_name"].tolist() == [
    "phase3D_competition_wicket_coverage"
]

assert unavailable_objects["column_names"].eq("NOT AVAILABLE").all()

# Validate actual available source DataFrames.
assert len(phase3D_wicket_distribution) == 20
assert len(phase3D_missing_comp_wicket_states) == 255
assert len(phase3E_wickets_remaining_distribution) == 20
assert len(phase3E_wicket_match_consistency) == 813

print("AVAILABLE COVERAGE OBJECTS")
display(available_objects)

print("\nUNAVAILABLE COVERAGE OBJECTS")
display(unavailable_objects)

print("\nINVENTORY VALIDATION: PASSED")
print("Available objects:", len(available_objects))
print("Unavailable objects:", len(unavailable_objects))
print("Source audit row counts verified.")

AVAILABLE COVERAGE OBJECTS


,object_name,exists,rows,columns,column_names
1,phase3D_wicket_distribution,True,20.0,5.0,"benchmark_population, pre_delivery_wickets_fallen, delivery_records, matches, competitions"
2,phase3D_missing_comp_wicket_states,True,255.0,4.0,"benchmark_population, comp_id, comp_name, pre_delivery_wickets_fallen"
3,phase3E_wickets_remaining_distribution,True,20.0,4.0,"match_format, wickets_remaining_conditional, delivery_records, matches"
4,phase3E_wicket_match_consistency,True,813.0,4.0,"match_id, distinct_formats, distinct_batting_teams, distinct_bowling_teams"



UNAVAILABLE COVERAGE OBJECTS


,object_name,exists,rows,columns,column_names
0,phase3D_competition_wicket_coverage,False,NaN,NaN,NOT AVAILABLE



INVENTORY VALIDATION: PASSED
Available objects: 4
Unavailable objects: 1
Source audit row counts verified.


#### Descriptive wicket-state coverage

In [143]:
# Phase 3F.4.3 — Descriptive wicket-state coverage

# First-innings coverage, by benchmark population and wickets fallen.
first_innings_coverage = (
    phase3D_wicket_distribution
    .groupby("benchmark_population", dropna=False)
    .agg(
        observed_wicket_states=("pre_delivery_wickets_fallen", "nunique"),
        delivery_records=("delivery_records", "sum"),
        minimum_state_deliveries=("delivery_records", "min"),
        maximum_state_deliveries=("delivery_records", "max"),
        maximum_state_match_coverage=("matches", "max"),
        competition_count=("competitions", "max")
    )
    .reset_index()
)

# Missing first-innings competition/wicket-state combinations.
first_innings_missing = (
    phase3D_missing_comp_wicket_states
    .groupby("benchmark_population", dropna=False)
    .agg(
        missing_competition_state_combinations=("pre_delivery_wickets_fallen", "size"),
        affected_competitions=("comp_id", "nunique"),
        missing_wicket_states=("pre_delivery_wickets_fallen", "nunique")
    )
    .reset_index()
)

# Second-innings coverage, by match format and wickets remaining.
second_innings_coverage = (
    phase3E_wickets_remaining_distribution
    .groupby("match_format", dropna=False)
    .agg(
        observed_wicket_states=("wickets_remaining_conditional", "nunique"),
        delivery_records=("delivery_records", "sum"),
        minimum_state_deliveries=("delivery_records", "min"),
        maximum_state_deliveries=("delivery_records", "max"),
        maximum_state_match_coverage=("matches", "max")
    )
    .reset_index()
)

# Match-level consistency audit.
second_innings_consistency = pd.DataFrame([{
    "matches_audited": len(phase3E_wicket_match_consistency),
    "matches_single_format": int(
        phase3E_wicket_match_consistency["distinct_formats"].eq(1).sum()
    ),
    "matches_single_batting_team": int(
        phase3E_wicket_match_consistency["distinct_batting_teams"].eq(1).sum()
    ),
    "matches_single_bowling_team": int(
        phase3E_wicket_match_consistency["distinct_bowling_teams"].eq(1).sum()
    ),
    "matches_with_format_or_team_inconsistency": int(
        (
            phase3E_wicket_match_consistency[
                ["distinct_formats", "distinct_batting_teams", "distinct_bowling_teams"]
            ].ne(1).any(axis=1)
        ).sum()
    )
}])

# Validation: confirm the underlying records and summary structure.
assert len(first_innings_coverage) == 2
assert len(first_innings_missing) == 2
assert len(second_innings_coverage) == 2
assert len(second_innings_consistency) == 1

assert first_innings_coverage["delivery_records"].sum() == (
    phase3D_wicket_distribution["delivery_records"].sum()
)
assert second_innings_coverage["delivery_records"].sum() == (
    phase3E_wickets_remaining_distribution["delivery_records"].sum()
)

assert second_innings_consistency.loc[
    0, "matches_with_format_or_team_inconsistency"
] == 0

print("FIRST-INNINGS WICKET-STATE COVERAGE")
display(first_innings_coverage)

print("\nFIRST-INNINGS MISSING COMPETITION/STATE COMBINATIONS")
display(first_innings_missing)

print("\nSECOND-INNINGS WICKETS-REMAINING COVERAGE")
display(second_innings_coverage)

print("\nSECOND-INNINGS MATCH-LEVEL CONSISTENCY")
display(second_innings_consistency)

print("\nVALIDATION: PASSED")

FIRST-INNINGS WICKET-STATE COVERAGE


,benchmark_population,observed_wicket_states,delivery_records,minimum_state_deliveries,maximum_state_deliveries,maximum_state_match_coverage,competition_count
0,IPL,10,45139,397,9187,362,5
1,T20I,10,55194,568,9392,451,131



FIRST-INNINGS MISSING COMPETITION/STATE COMBINATIONS


,benchmark_population,missing_competition_state_combinations,affected_competitions,missing_wicket_states
0,IPL,5,5,1
1,T20I,250,131,6



SECOND-INNINGS WICKETS-REMAINING COVERAGE


,match_format,observed_wicket_states,delivery_records,minimum_state_deliveries,maximum_state_deliveries,maximum_state_match_coverage
0,T20,10,42078,492,8213,362
1,T20I,10,49824,1006,9212,451



SECOND-INNINGS MATCH-LEVEL CONSISTENCY


,matches_audited,matches_single_format,matches_single_batting_team,matches_single_bowling_team,matches_with_format_or_team_inconsistency
0,813,813,813,813,0



VALIDATION: PASSED


#### Sensitivity and stability feasibility

In [144]:
# Phase 3F.4.4 — Sensitivity and stability feasibility

phase3F4_sensitivity_register = pd.DataFrame([
    {
        "sensitivity_test": "First-innings exact competition/state coverage",
        "evidence_available": "Aggregate wicket distributions and missing competition/state register",
        "feasibility": "PARTIAL_DESCRIPTIVE",
        "limitation": "255 competition/state combinations are unobserved; detailed coverage table is unavailable."
    },
    {
        "sensitivity_test": "Second-innings wicket-state coverage by format",
        "evidence_available": "Wickets-remaining distributions by T20 and T20I",
        "feasibility": "DESCRIPTIVE_ONLY",
        "limitation": "Wicket states are conditional reconstructions; no pressure outcome is available."
    },
    {
        "sensitivity_test": "Alternative wicket-state grouping",
        "evidence_available": "Observed wicket-state distributions",
        "feasibility": "POLICY_DESIGN_ONLY",
        "limitation": "No grouping threshold or pooling rule has been empirically approved."
    },
    {
        "sensitivity_test": "Alternative pressure component weights",
        "evidence_available": "No constructed pressure score or approved weights",
        "feasibility": "BLOCKED",
        "limitation": "No numerical scale exists on which to test weight sensitivity."
    },
    {
        "sensitivity_test": "Alternative normalization methods",
        "evidence_available": "No complete component normalization specification",
        "feasibility": "BLOCKED",
        "limitation": "Normalization alternatives cannot be compared without validated component measurements."
    },
    {
        "sensitivity_test": "Alternative pressure thresholds or bands",
        "evidence_available": "No numerical pressure values or approved thresholds",
        "feasibility": "BLOCKED",
        "limitation": "No pressure classifications exist to compare."
    },
    {
        "sensitivity_test": "First- versus second-innings stability",
        "evidence_available": "Separate conceptual frameworks and conditional wicket-state coverage",
        "feasibility": "NOT_TESTABLE",
        "limitation": "No validated numerical measures or cross-innings calibration."
    },
    {
        "sensitivity_test": "IPL versus T20I pressure stability",
        "evidence_available": "Competition-specific aggregate wicket-state coverage",
        "feasibility": "PARTIAL_DESCRIPTIVE",
        "limitation": "Benchmark measures are not constructed and exact-state coverage is incomplete."
    }
])

phase3F4_coverage_findings = pd.DataFrame([
    {
        "finding": "First-innings aggregate wicket-state representation",
        "observed_evidence": "10 states in IPL and T20I",
        "interpretation": "Aggregate representation exists; competition-level completeness is not established."
    },
    {
        "finding": "First-innings missing competition/state combinations",
        "observed_evidence": "255 total: 5 IPL and 250 T20I",
        "interpretation": "Exact competition/state benchmark coverage is incomplete."
    },
    {
        "finding": "Second-innings aggregate wicket-state representation",
        "observed_evidence": "10 wickets-remaining states in T20 and T20I",
        "interpretation": "Aggregate representation exists; wicket states remain conditional."
    },
    {
        "finding": "Second-innings match identity consistency",
        "observed_evidence": "813 of 813 matches consistent",
        "interpretation": "Format and batting/bowling team consistency passed; this does not independently validate wicket events."
    }
])

# Validation
assert len(phase3F4_sensitivity_register) == 8
assert phase3F4_sensitivity_register["sensitivity_test"].is_unique
assert phase3F4_sensitivity_register.notna().all().all()

assert len(phase3F4_coverage_findings) == 4
assert phase3F4_coverage_findings["finding"].is_unique
assert phase3F4_coverage_findings.notna().all().all()

assert (
    phase3F4_sensitivity_register["feasibility"]
    .isin([
        "PARTIAL_DESCRIPTIVE",
        "DESCRIPTIVE_ONLY",
        "POLICY_DESIGN_ONLY",
        "BLOCKED",
        "NOT_TESTABLE",
        "PARTIAL_DESCRIPTIVE"
    ]).all()
)

assert int(
    phase3D_missing_comp_wicket_states.shape[0]
) == 255

assert int(
    phase3E_wicket_match_consistency.shape[0]
) == 813

print("PHASE 3F.4.4 — SENSITIVITY FEASIBILITY")
display(phase3F4_sensitivity_register)

print("\nPHASE 3F.4.4 — COVERAGE FINDINGS")
display(phase3F4_coverage_findings)

print("\nSENSITIVITY FEASIBILITY COUNTS")
display(
    phase3F4_sensitivity_register["feasibility"]
    .value_counts()
    .rename_axis("feasibility")
    .reset_index(name="test_count")
)

print("\nVALIDATION: PASSED")

PHASE 3F.4.4 — SENSITIVITY FEASIBILITY


,sensitivity_test,evidence_available,feasibility,limitation
0,First-innings exact competition/state coverage,Aggregate wicket distributions and missing competition/state register,PARTIAL_DESCRIPTIVE,255 competition/state combinations are unobserved; detailed coverage table is unavailable.
1,Second-innings wicket-state coverage by format,Wickets-remaining distributions by T20 and T20I,DESCRIPTIVE_ONLY,Wicket states are conditional reconstructions; no pressure outcome is available.
2,Alternative wicket-state grouping,Observed wicket-state distributions,POLICY_DESIGN_ONLY,No grouping threshold or pooling rule has been empirically approved.
3,Alternative pressure component weights,No constructed pressure score or approved weights,BLOCKED,No numerical scale exists on which to test weight sensitivity.
4,Alternative normalization methods,No complete component normalization specification,BLOCKED,Normalization alternatives cannot be compared without validated component measurements.
5,Alternative pressure thresholds or bands,No numerical pressure values or approved thresholds,BLOCKED,No pressure classifications exist to compare.
6,First- versus second-innings stability,Separate conceptual frameworks and conditional wicket-state coverage,NOT_TESTABLE,No validated numerical measures or cross-innings calibration.
7,IPL versus T20I pressure stability,Competition-specific aggregate wicket-state coverage,PARTIAL_DESCRIPTIVE,Benchmark measures are not constructed and exact-state coverage is incomplete.



PHASE 3F.4.4 — COVERAGE FINDINGS


,finding,observed_evidence,interpretation
0,First-innings aggregate wicket-state representation,10 states in IPL and T20I,Aggregate representation exists; competition-level completeness is not established.
1,First-innings missing competition/state combinations,255 total: 5 IPL and 250 T20I,Exact competition/state benchmark coverage is incomplete.
2,Second-innings aggregate wicket-state representation,10 wickets-remaining states in T20 and T20I,Aggregate representation exists; wicket states remain conditional.
3,Second-innings match identity consistency,813 of 813 matches consistent,Format and batting/bowling team consistency passed; this does not independently validate wicket events.



SENSITIVITY FEASIBILITY COUNTS


,feasibility,test_count
0,BLOCKED,3
1,PARTIAL_DESCRIPTIVE,2
2,DESCRIPTIVE_ONLY,1
3,POLICY_DESIGN_ONLY,1
4,NOT_TESTABLE,1



VALIDATION: PASSED


#### Coverage and sensitivity closure

In [145]:
# Phase 3F.4.5 — Coverage and sensitivity closure

phase3F4_closure_register = pd.DataFrame([
    {
        "closure_area": "First-innings aggregate coverage",
        "status": "DESCRIPTIVE_COMPLETE",
        "evidence_records": 2,
        "conclusion": "Ten wicket states observed in IPL and T20I; 255 competition/state combinations are unobserved."
    },
    {
        "closure_area": "Second-innings aggregate coverage",
        "status": "DESCRIPTIVE_COMPLETE",
        "evidence_records": 2,
        "conclusion": "Ten wickets-remaining states observed in T20 and T20I; reconstructed states remain conditional."
    },
    {
        "closure_area": "Competition/state completeness",
        "status": "INCOMPLETE",
        "evidence_records": 255,
        "conclusion": "Exact first-innings competition/state coverage is incomplete."
    },
    {
        "closure_area": "Pressure-scale sensitivity",
        "status": "BLOCKED",
        "evidence_records": 8,
        "conclusion": "No numerical pressure scale exists for weight, normalization, threshold or calibration sensitivity testing."
    },
    {
        "closure_area": "Cross-innings stability",
        "status": "NOT_TESTABLE",
        "evidence_records": 0,
        "conclusion": "No validated numerical measures or cross-innings calibration exist."
    }
])

phase3F4_closure_summary = pd.DataFrame([
    {
        "measure": "Coverage findings documented",
        "count": len(phase3F4_coverage_findings),
        "status": "COMPLETE"
    },
    {
        "measure": "Sensitivity tests assessed",
        "count": len(phase3F4_sensitivity_register),
        "status": "COMPLETE"
    },
    {
        "measure": "Blocked sensitivity tests",
        "count": int((phase3F4_sensitivity_register["feasibility"] == "BLOCKED").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Partially descriptive sensitivity tests",
        "count": int((phase3F4_sensitivity_register["feasibility"].isin(
            ["PARTIAL_DESCRIPTIVE", "PARTIAL_DESCRIPTIVE_ONLY"]
        )).sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Descriptive-only sensitivity tests",
        "count": int((phase3F4_sensitivity_register["feasibility"] == "DESCRIPTIVE_ONLY").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Policy-design-only sensitivity tests",
        "count": int((phase3F4_sensitivity_register["feasibility"] == "POLICY_DESIGN_ONLY").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Not-testable sensitivity tests",
        "count": int((phase3F4_sensitivity_register["feasibility"] == "NOT_TESTABLE").sum()),
        "status": "LIMITATION_RECORDED"
    }
])

# Validation
assert phase3F4_closure_register["closure_area"].is_unique
assert phase3F4_closure_register.notna().all().all()
assert phase3F4_closure_summary["measure"].is_unique
assert phase3F4_closure_summary.notna().all().all()

assert len(phase3F4_coverage_findings) == 4
assert len(phase3F4_sensitivity_register) == 8
assert int(phase3D_missing_comp_wicket_states.shape[0]) == 255

assert (
    phase3F4_sensitivity_register["feasibility"]
    .value_counts().to_dict()
    == {
        "BLOCKED": 3,
        "PARTIAL_DESCRIPTIVE": 2,
        "DESCRIPTIVE_ONLY": 1,
        "POLICY_DESIGN_ONLY": 1,
        "NOT_TESTABLE": 1
    }
)

print("PHASE 3F.4.5 — CLOSURE REGISTER")
display(phase3F4_closure_register)

print("\nPHASE 3F.4.5 — CLOSURE SUMMARY")
display(phase3F4_closure_summary)

print("\nVALIDATION: PASSED")

PHASE 3F.4.5 — CLOSURE REGISTER


,closure_area,status,evidence_records,conclusion
0,First-innings aggregate coverage,DESCRIPTIVE_COMPLETE,2,Ten wicket states observed in IPL and T20I; 255 competition/state combinations are unobserved.
1,Second-innings aggregate coverage,DESCRIPTIVE_COMPLETE,2,Ten wickets-remaining states observed in T20 and T20I; reconstructed states remain conditional.
2,Competition/state completeness,INCOMPLETE,255,Exact first-innings competition/state coverage is incomplete.
3,Pressure-scale sensitivity,BLOCKED,8,"No numerical pressure scale exists for weight, normalization, threshold or calibration sensitivity testing."
4,Cross-innings stability,NOT_TESTABLE,0,No validated numerical measures or cross-innings calibration exist.



PHASE 3F.4.5 — CLOSURE SUMMARY


,measure,count,status
0,Coverage findings documented,4,COMPLETE
1,Sensitivity tests assessed,8,COMPLETE
2,Blocked sensitivity tests,3,LIMITATION_RECORDED
3,Partially descriptive sensitivity tests,2,LIMITATION_RECORDED
4,Descriptive-only sensitivity tests,1,LIMITATION_RECORDED
5,Policy-design-only sensitivity tests,1,LIMITATION_RECORDED
6,Not-testable sensitivity tests,1,LIMITATION_RECORDED



VALIDATION: PASSED


### Validation limitations and acceptance criteria

#### Acceptance criteria register

In [146]:
# Phase 3F.5.1 — Validation limitations and acceptance criteria

phase3F5_acceptance_criteria = pd.DataFrame([
    {
        "criterion": "Pre-delivery information integrity",
        "acceptance_requirement": "Every pressure input must be available before the delivery; delivery outcomes must not leak into the state.",
        "basis": "PHASE_0_DEFINED",
        "current_status": "PARTIALLY_AUDITED",
        "evidence_or_limitation": "Phase 3C leakage controls passed; reconstructed wicket state remains conditional."
    },
    {
        "criterion": "Cumulative scoring validity",
        "acceptance_requirement": "Cumulative innings scoring must be reconstructed from validated run semantics and reconciled independently.",
        "basis": "REQUIRED_FOR_CONSTRUCTION",
        "current_status": "BLOCKED",
        "evidence_or_limitation": "Phase 3D could not validate runs and extras semantics or construct cumulative score."
    },
    {
        "criterion": "Legal-ball and remaining-resource validity",
        "acceptance_requirement": "Legal-ball progression and scheduled innings length must be validated before remaining resources are calculated.",
        "basis": "REQUIRED_FOR_CONSTRUCTION",
        "current_status": "BLOCKED",
        "evidence_or_limitation": "Phase 2 and Phase 3D found sequence fields insufficient to establish legal-ball progression."
    },
    {
        "criterion": "Target and chase-state validity",
        "acceptance_requirement": "Second-innings target, cumulative score and remaining resources must have verified semantics.",
        "basis": "REQUIRED_FOR_CONSTRUCTION",
        "current_status": "BLOCKED",
        "evidence_or_limitation": "Phase 3E found revised_target sparse and unverified; cumulative score and legal balls unavailable."
    },
    {
        "criterion": "Wicket-state reliability",
        "acceptance_requirement": "Reconstructed wicket states must be reconciled against reliable dismissal evidence, with unresolved events documented.",
        "basis": "CONDITIONAL_INPUT",
        "current_status": "CONDITIONAL",
        "evidence_or_limitation": "Wicket states are reconstructed from prior events and are not independently verified dismissal truth."
    },
    {
        "criterion": "Comparable benchmark construction",
        "acceptance_requirement": "Expected scoring benchmarks must use comparable competition-specific states and preserve sample size and provenance.",
        "basis": "PHASE_0_DEFINED",
        "current_status": "BLOCKED",
        "evidence_or_limitation": "Expected scoring benchmarks were not constructible in Phases 3D and 3E."
    },
    {
        "criterion": "Fallback and unavailable-state treatment",
        "acceptance_requirement": "Use the documented fallback hierarchy; mark unsupported or unstable states unavailable rather than inventing estimates.",
        "basis": "PHASE_0_DEFINED",
        "current_status": "POLICY_DEFINED_NOT_EXECUTABLE",
        "evidence_or_limitation": "Fallback policies exist, but required state dimensions and benchmarks are unavailable."
    },
    {
        "criterion": "Component directionality",
        "acceptance_requirement": "Each component must have a defensible direction and interpretation; scoring deviation must not be treated as standalone pressure.",
        "basis": "PARTIALLY_DEFINED",
        "current_status": "CONCEPTUAL_ONLY",
        "evidence_or_limitation": "Phase 3F.3 documented conceptual directions but could not empirically test them."
    },
    {
        "criterion": "Normalization and combination",
        "acceptance_requirement": "Normalization and component combination must be specified, justified and validated before a numerical score is produced.",
        "basis": "NOT_DEFINED_IN_PHASE_0",
        "current_status": "NOT_ESTABLISHED",
        "evidence_or_limitation": "No complete normalization method, weights or combination formula is defined."
    },
    {
        "criterion": "Sensitivity and stability",
        "acceptance_requirement": "Alternative defensible specifications must be tested and conclusions must remain sufficiently stable.",
        "basis": "VALIDATION_REQUIREMENT",
        "current_status": "NOT_TESTABLE",
        "evidence_or_limitation": "No numerical pressure scale or approved alternative specifications exist."
    },
    {
        "criterion": "Cross-competition treatment",
        "acceptance_requirement": "IPL and T20I benchmarks must remain separate; no unsupported cross-competition fallback.",
        "basis": "PHASE_0_DEFINED",
        "current_status": "POLICY_DEFINED",
        "evidence_or_limitation": "Competition separation is defined, but competition-specific benchmarks are not constructed."
    },
    {
        "criterion": "Cross-innings comparability",
        "acceptance_requirement": "Any common scale must demonstrate a defensible mapping and calibration across innings.",
        "basis": "NOT_DEFINED_IN_PHASE_0",
        "current_status": "NOT_ESTABLISHED",
        "evidence_or_limitation": "No common numerical construct or cross-innings calibration is specified."
    },
    {
        "criterion": "Pressure bands and thresholds",
        "acceptance_requirement": "Any pressure bands must have empirically justified thresholds and documented interpretation.",
        "basis": "NOT_DEFINED_IN_PHASE_0",
        "current_status": "NOT_ESTABLISHED",
        "evidence_or_limitation": "No low, medium or high pressure thresholds are defined."
    }
])

# Validation
assert len(phase3F5_acceptance_criteria) == 13
assert phase3F5_acceptance_criteria["criterion"].is_unique
assert phase3F5_acceptance_criteria.notna().all().all()
assert phase3F5_acceptance_criteria["acceptance_requirement"].str.len().gt(0).all()
assert phase3F5_acceptance_criteria["evidence_or_limitation"].str.len().gt(0).all()

print("PHASE 3F.5.1 — ACCEPTANCE CRITERIA REGISTER")
display(phase3F5_acceptance_criteria)

print("\nCURRENT STATUS COUNTS")
display(
    phase3F5_acceptance_criteria["current_status"]
    .value_counts()
    .rename_axis("current_status")
    .reset_index(name="criterion_count")
)

print("\nVALIDATION: 5/5 PASSED")

PHASE 3F.5.1 — ACCEPTANCE CRITERIA REGISTER


,criterion,acceptance_requirement,basis,current_status,evidence_or_limitation
0,Pre-delivery information integrity,Every pressure input must be available before the delivery; delivery outcomes must not leak into the state.,PHASE_0_DEFINED,PARTIALLY_AUDITED,Phase 3C leakage controls passed; reconstructed wicket state remains conditional.
1,Cumulative scoring validity,Cumulative innings scoring must be reconstructed from validated run semantics and reconciled independently.,REQUIRED_FOR_CONSTRUCTION,BLOCKED,Phase 3D could not validate runs and extras semantics or construct cumulative score.
2,Legal-ball and remaining-resource validity,Legal-ball progression and scheduled innings length must be validated before remaining resources are calculated.,REQUIRED_FOR_CONSTRUCTION,BLOCKED,Phase 2 and Phase 3D found sequence fields insufficient to establish legal-ball progression.
3,Target and chase-state validity,"Second-innings target, cumulative score and remaining resources must have verified semantics.",REQUIRED_FOR_CONSTRUCTION,BLOCKED,Phase 3E found revised_target sparse and unverified; cumulative score and legal balls unavailable.
4,Wicket-state reliability,"Reconstructed wicket states must be reconciled against reliable dismissal evidence, with unresolved events documented.",CONDITIONAL_INPUT,CONDITIONAL,Wicket states are reconstructed from prior events and are not independently verified dismissal truth.
5,Comparable benchmark construction,Expected scoring benchmarks must use comparable competition-specific states and preserve sample size and provenance.,PHASE_0_DEFINED,BLOCKED,Expected scoring benchmarks were not constructible in Phases 3D and 3E.
6,Fallback and unavailable-state treatment,Use the documented fallback hierarchy; mark unsupported or unstable states unavailable rather than inventing estimates.,PHASE_0_DEFINED,POLICY_DEFINED_NOT_EXECUTABLE,"Fallback policies exist, but required state dimensions and benchmarks are unavailable."
7,Component directionality,Each component must have a defensible direction and interpretation; scoring deviation must not be treated as standalone pressure.,PARTIALLY_DEFINED,CONCEPTUAL_ONLY,Phase 3F.3 documented conceptual directions but could not empirically test them.
8,Normalization and combination,"Normalization and component combination must be specified, justified and validated before a numerical score is produced.",NOT_DEFINED_IN_PHASE_0,NOT_ESTABLISHED,"No complete normalization method, weights or combination formula is defined."
9,Sensitivity and stability,Alternative defensible specifications must be tested and conclusions must remain sufficiently stable.,VALIDATION_REQUIREMENT,NOT_TESTABLE,No numerical pressure scale or approved alternative specifications exist.



CURRENT STATUS COUNTS


,current_status,criterion_count
0,BLOCKED,4
1,NOT_ESTABLISHED,3
2,CONDITIONAL,1
3,PARTIALLY_AUDITED,1
4,POLICY_DEFINED_NOT_EXECUTABLE,1
5,CONCEPTUAL_ONLY,1
6,NOT_TESTABLE,1
7,POLICY_DEFINED,1



VALIDATION: 5/5 PASSED


#### Acceptance gate and decision rules

In [147]:
# Phase 3F.5.2 — Acceptance gate and decision rules

def classify_phase3F5_status(status):
    mapping = {
        "BLOCKED": "BLOCKED_PREREQUISITE",
        "NOT_ESTABLISHED": "DESIGN_GAP",
        "CONDITIONAL": "CONDITIONAL_ACCEPTANCE_ONLY",
        "PARTIALLY_AUDITED": "INCOMPLETE_EVIDENCE",
        "POLICY_DEFINED_NOT_EXECUTABLE": "POLICY_ONLY",
        "CONCEPTUAL_ONLY": "CONCEPTUAL_ONLY",
        "NOT_TESTABLE": "VALIDATION_NOT_TESTABLE",
        "POLICY_DEFINED": "POLICY_DEFINED_NOT_EMPIRICALLY_VALIDATED"
    }
    return mapping.get(status, "REQUIRES_REVIEW")


phase3F5_gate_register = phase3F5_acceptance_criteria.copy()

phase3F5_gate_register["acceptance_gate"] = (
    phase3F5_gate_register["current_status"]
    .map(classify_phase3F5_status)
)

phase3F5_gate_register["scale_acceptance_effect"] = (
    phase3F5_gate_register["acceptance_gate"].map({
        "BLOCKED_PREREQUISITE": "PREVENTS_NUMERICAL_SCALE_ACCEPTANCE",
        "DESIGN_GAP": "PREVENTS_NUMERICAL_SCALE_ACCEPTANCE",
        "CONDITIONAL_ACCEPTANCE_ONLY": "REQUIRES_EVIDENCE_BEFORE_ACCEPTANCE",
        "INCOMPLETE_EVIDENCE": "REQUIRES_FURTHER_AUDIT",
        "POLICY_ONLY": "POLICY_DOCUMENTATION_ONLY",
        "CONCEPTUAL_ONLY": "REQUIRES_EMPIRICAL_VALIDATION",
        "VALIDATION_NOT_TESTABLE": "ROBUSTNESS_ACCEPTANCE_UNAVAILABLE",
        "POLICY_DEFINED_NOT_EMPIRICALLY_VALIDATED": "POLICY_DOCUMENTATION_ONLY"
    })
)

phase3F5_decision_rules = pd.DataFrame([
    {
        "decision": "ACCEPT",
        "rule": "All mandatory inputs and design requirements are validated; benchmark construction, leakage controls, directionality, normalization, combination and required stability tests have supporting evidence.",
        "current_phase3F_status": "NOT_MET",
        "permitted_conclusion": "A numerical pressure scale may be described as validated only within its demonstrated population and scope."
    },
    {
        "decision": "CONDITIONAL_ACCEPT",
        "rule": "A limited component or population has sufficient evidence, while clearly identified conditions remain unresolved and do not invalidate that limited use.",
        "current_phase3F_status": "NOT_ESTABLISHED",
        "permitted_conclusion": "Only the explicitly validated component or restricted scope may be reported; no general scale-validity claim."
    },
    {
        "decision": "REJECT",
        "rule": "Evidence demonstrates leakage, invalid input semantics, logically contradictory construction, or materially unreliable results that cannot be resolved by the documented method.",
        "current_phase3F_status": "NOT_ASSESSED",
        "permitted_conclusion": "Reject the affected construction or specification; document the evidence and affected scope."
    },
    {
        "decision": "UNAVAILABLE",
        "rule": "Required inputs, benchmark evidence, calibration or validation are missing or cannot be established from the available data.",
        "current_phase3F_status": "CURRENT_DECISION",
        "permitted_conclusion": "Do not construct or report a numerical pressure scale for the unsupported scope."
    },
    {
        "decision": "POLICY_ONLY",
        "rule": "A methodological rule is documented but its required data, implementation or empirical validation is not available.",
        "current_phase3F_status": "APPLIES_TO_FALLBACK_AND_COMPETITION_POLICIES",
        "permitted_conclusion": "Report the policy as specified, not as executed or empirically validated."
    }
])

# Validation
assert len(phase3F5_gate_register) == len(phase3F5_acceptance_criteria)
assert phase3F5_gate_register["criterion"].equals(
    phase3F5_acceptance_criteria["criterion"]
)
assert phase3F5_gate_register["acceptance_gate"].notna().all()
assert phase3F5_gate_register["scale_acceptance_effect"].notna().all()
assert phase3F5_decision_rules["decision"].is_unique
assert phase3F5_decision_rules.notna().all().all()

assert (
    (phase3F5_gate_register["acceptance_gate"] == "BLOCKED_PREREQUISITE").sum()
    == 4
)
assert (
    (phase3F5_gate_register["acceptance_gate"] == "DESIGN_GAP").sum()
    == 3
)

print("PHASE 3F.5.2 — ACCEPTANCE GATE REGISTER")
display(
    phase3F5_gate_register[
        [
            "criterion",
            "current_status",
            "acceptance_gate",
            "scale_acceptance_effect"
        ]
    ]
)

print("\nPHASE 3F.5.2 — DECISION RULES")
display(phase3F5_decision_rules)

print("\nVALIDATION: 6/6 PASSED")

PHASE 3F.5.2 — ACCEPTANCE GATE REGISTER


,criterion,current_status,acceptance_gate,scale_acceptance_effect
0,Pre-delivery information integrity,PARTIALLY_AUDITED,INCOMPLETE_EVIDENCE,REQUIRES_FURTHER_AUDIT
1,Cumulative scoring validity,BLOCKED,BLOCKED_PREREQUISITE,PREVENTS_NUMERICAL_SCALE_ACCEPTANCE
2,Legal-ball and remaining-resource validity,BLOCKED,BLOCKED_PREREQUISITE,PREVENTS_NUMERICAL_SCALE_ACCEPTANCE
3,Target and chase-state validity,BLOCKED,BLOCKED_PREREQUISITE,PREVENTS_NUMERICAL_SCALE_ACCEPTANCE
4,Wicket-state reliability,CONDITIONAL,CONDITIONAL_ACCEPTANCE_ONLY,REQUIRES_EVIDENCE_BEFORE_ACCEPTANCE
5,Comparable benchmark construction,BLOCKED,BLOCKED_PREREQUISITE,PREVENTS_NUMERICAL_SCALE_ACCEPTANCE
6,Fallback and unavailable-state treatment,POLICY_DEFINED_NOT_EXECUTABLE,POLICY_ONLY,POLICY_DOCUMENTATION_ONLY
7,Component directionality,CONCEPTUAL_ONLY,CONCEPTUAL_ONLY,REQUIRES_EMPIRICAL_VALIDATION
8,Normalization and combination,NOT_ESTABLISHED,DESIGN_GAP,PREVENTS_NUMERICAL_SCALE_ACCEPTANCE
9,Sensitivity and stability,NOT_TESTABLE,VALIDATION_NOT_TESTABLE,ROBUSTNESS_ACCEPTANCE_UNAVAILABLE



PHASE 3F.5.2 — DECISION RULES


,decision,rule,current_phase3F_status,permitted_conclusion
0,ACCEPT,"All mandatory inputs and design requirements are validated; benchmark construction, leakage controls, directionality, normalization, combination and required stability tests have supporting evidence.",NOT_MET,A numerical pressure scale may be described as validated only within its demonstrated population and scope.
1,CONDITIONAL_ACCEPT,"A limited component or population has sufficient evidence, while clearly identified conditions remain unresolved and do not invalidate that limited use.",NOT_ESTABLISHED,Only the explicitly validated component or restricted scope may be reported; no general scale-validity claim.
2,REJECT,"Evidence demonstrates leakage, invalid input semantics, logically contradictory construction, or materially unreliable results that cannot be resolved by the documented method.",NOT_ASSESSED,Reject the affected construction or specification; document the evidence and affected scope.
3,UNAVAILABLE,"Required inputs, benchmark evidence, calibration or validation are missing or cannot be established from the available data.",CURRENT_DECISION,Do not construct or report a numerical pressure scale for the unsupported scope.
4,POLICY_ONLY,"A methodological rule is documented but its required data, implementation or empirical validation is not available.",APPLIES_TO_FALLBACK_AND_COMPETITION_POLICIES,"Report the policy as specified, not as executed or empirically validated."



VALIDATION: 6/6 PASSED


#### Consolidated limitations and current validation decision

In [148]:
# Phase 3F.5.3 — Consolidated limitations and validation decision

phase3F5_limitation_register = phase3F5_gate_register[
    [
        "criterion",
        "current_status",
        "acceptance_gate",
        "evidence_or_limitation"
    ]
].copy()

phase3F5_limitation_register["required_resolution"] = [
    "Complete audit of all pressure inputs and preserve pre-delivery boundaries.",
    "Obtain validated run semantics and independently reconciled cumulative scores.",
    "Validate legal-ball progression and scheduled innings length.",
    "Verify target semantics and establish cumulative chase-state inputs.",
    "Reconcile reconstructed wicket states against reliable dismissal evidence.",
    "Construct competition-specific benchmarks with coverage and provenance diagnostics.",
    "Implement the fallback hierarchy only when its required inputs are available.",
    "Empirically test component directionality against defensible outcomes.",
    "Specify and validate normalization and component-combination methods.",
    "Run sensitivity and stability tests after a numerical scale exists.",
    "Retain separate IPL and T20I treatment and validate resulting benchmarks.",
    "Define and test any proposed cross-innings mapping and calibration.",
    "Define pressure bands only after empirical threshold validation."
]

phase3F5_current_decision = pd.DataFrame([
    {
        "decision_area": "First-innings numerical pressure scale",
        "decision": "UNAVAILABLE",
        "reason": "Cumulative scoring, remaining resources and expected scoring benchmarks are unavailable."
    },
    {
        "decision_area": "Second-innings numerical pressure scale",
        "decision": "UNAVAILABLE",
        "reason": "Target semantics, cumulative score, remaining resources and expected scoring benchmarks are not validated."
    },
    {
        "decision_area": "Unified cross-innings numerical scale",
        "decision": "UNAVAILABLE",
        "reason": "Neither innings-specific numerical scale is constructible and no cross-innings calibration exists."
    },
    {
        "decision_area": "Shared conceptual framework",
        "decision": "RETAIN_AS_CONCEPTUAL_FRAMEWORK",
        "reason": "The common pre-delivery state principles and separate innings-specific measures remain conceptually useful, but are not numerically validated."
    },
    {
        "decision_area": "Wicket-state component",
        "decision": "CONDITIONAL",
        "reason": "Reconstructed pre-delivery wicket states are available, subject to their documented reliability limitation."
    },
    {
        "decision_area": "Pressure bands and thresholds",
        "decision": "NOT_ESTABLISHED",
        "reason": "No numerical scale or empirically justified thresholds exist."
    }
])

# Validation
assert len(phase3F5_limitation_register) == 13
assert phase3F5_limitation_register["criterion"].is_unique
assert phase3F5_limitation_register.notna().all().all()
assert phase3F5_limitation_register["required_resolution"].str.len().gt(0).all()

assert len(phase3F5_current_decision) == 6
assert phase3F5_current_decision["decision_area"].is_unique
assert phase3F5_current_decision.notna().all().all()

assert (
    phase3F5_current_decision.loc[
        phase3F5_current_decision["decision_area"].str.contains(
            "numerical pressure scale|Unified cross-innings",
            case=False,
            regex=True
        ),
        "decision"
    ] == "UNAVAILABLE"
).all()

assert (
    phase3F5_current_decision.loc[
        phase3F5_current_decision["decision_area"] == "Wicket-state component",
        "decision"
    ].iloc[0] == "CONDITIONAL"
)

assert (
    phase3F5_current_decision.loc[
        phase3F5_current_decision["decision_area"] == "Shared conceptual framework",
        "decision"
    ].iloc[0] == "RETAIN_AS_CONCEPTUAL_FRAMEWORK"
)

print("PHASE 3F.5.3 — CONSOLIDATED LIMITATION REGISTER")
display(phase3F5_limitation_register)

print("\nPHASE 3F.5.3 — CURRENT VALIDATION DECISION")
display(phase3F5_current_decision)

print("\nVALIDATION: 7/7 PASSED")

PHASE 3F.5.3 — CONSOLIDATED LIMITATION REGISTER


,criterion,current_status,acceptance_gate,evidence_or_limitation,required_resolution
0,Pre-delivery information integrity,PARTIALLY_AUDITED,INCOMPLETE_EVIDENCE,Phase 3C leakage controls passed; reconstructed wicket state remains conditional.,Complete audit of all pressure inputs and preserve pre-delivery boundaries.
1,Cumulative scoring validity,BLOCKED,BLOCKED_PREREQUISITE,Phase 3D could not validate runs and extras semantics or construct cumulative score.,Obtain validated run semantics and independently reconciled cumulative scores.
2,Legal-ball and remaining-resource validity,BLOCKED,BLOCKED_PREREQUISITE,Phase 2 and Phase 3D found sequence fields insufficient to establish legal-ball progression.,Validate legal-ball progression and scheduled innings length.
3,Target and chase-state validity,BLOCKED,BLOCKED_PREREQUISITE,Phase 3E found revised_target sparse and unverified; cumulative score and legal balls unavailable.,Verify target semantics and establish cumulative chase-state inputs.
4,Wicket-state reliability,CONDITIONAL,CONDITIONAL_ACCEPTANCE_ONLY,Wicket states are reconstructed from prior events and are not independently verified dismissal truth.,Reconcile reconstructed wicket states against reliable dismissal evidence.
5,Comparable benchmark construction,BLOCKED,BLOCKED_PREREQUISITE,Expected scoring benchmarks were not constructible in Phases 3D and 3E.,Construct competition-specific benchmarks with coverage and provenance diagnostics.
6,Fallback and unavailable-state treatment,POLICY_DEFINED_NOT_EXECUTABLE,POLICY_ONLY,"Fallback policies exist, but required state dimensions and benchmarks are unavailable.",Implement the fallback hierarchy only when its required inputs are available.
7,Component directionality,CONCEPTUAL_ONLY,CONCEPTUAL_ONLY,Phase 3F.3 documented conceptual directions but could not empirically test them.,Empirically test component directionality against defensible outcomes.
8,Normalization and combination,NOT_ESTABLISHED,DESIGN_GAP,"No complete normalization method, weights or combination formula is defined.",Specify and validate normalization and component-combination methods.
9,Sensitivity and stability,NOT_TESTABLE,VALIDATION_NOT_TESTABLE,No numerical pressure scale or approved alternative specifications exist.,Run sensitivity and stability tests after a numerical scale exists.



PHASE 3F.5.3 — CURRENT VALIDATION DECISION


,decision_area,decision,reason
0,First-innings numerical pressure scale,UNAVAILABLE,"Cumulative scoring, remaining resources and expected scoring benchmarks are unavailable."
1,Second-innings numerical pressure scale,UNAVAILABLE,"Target semantics, cumulative score, remaining resources and expected scoring benchmarks are not validated."
2,Unified cross-innings numerical scale,UNAVAILABLE,Neither innings-specific numerical scale is constructible and no cross-innings calibration exists.
3,Shared conceptual framework,RETAIN_AS_CONCEPTUAL_FRAMEWORK,"The common pre-delivery state principles and separate innings-specific measures remain conceptually useful, but are not numerically validated."
4,Wicket-state component,CONDITIONAL,"Reconstructed pre-delivery wicket states are available, subject to their documented reliability limitation."
5,Pressure bands and thresholds,NOT_ESTABLISHED,No numerical scale or empirically justified thresholds exist.



VALIDATION: 7/7 PASSED


#### Acceptance criteria closure

In [149]:
# Phase 3F.5.4 — Acceptance criteria closure

phase3F5_closure_register = pd.DataFrame([
    {
        "closure_area": "Acceptance criteria",
        "status": "COMPLETE",
        "evidence_count": len(phase3F5_acceptance_criteria),
        "conclusion": "Thirteen criteria documented with explicit requirements and current evidence status."
    },
    {
        "closure_area": "Acceptance gate classification",
        "status": "COMPLETE",
        "evidence_count": len(phase3F5_gate_register),
        "conclusion": "All criteria classified by their effect on numerical scale acceptance."
    },
    {
        "closure_area": "Limitations and required resolutions",
        "status": "COMPLETE",
        "evidence_count": len(phase3F5_limitation_register),
        "conclusion": "All identified limitations have a corresponding required resolution."
    },
    {
        "closure_area": "Current pressure-scale decision",
        "status": "UNAVAILABLE",
        "evidence_count": len(phase3F5_current_decision),
        "conclusion": "No first-innings, second-innings or unified numerical pressure scale can be accepted using current evidence."
    },
    {
        "closure_area": "Conceptual framework",
        "status": "RETAINED_WITH_LIMITATIONS",
        "evidence_count": 1,
        "conclusion": "Shared pre-delivery framework retained; numerical validity is not established."
    }
])

phase3F5_closure_summary = pd.DataFrame([
    {
        "measure": "Acceptance criteria",
        "count": len(phase3F5_acceptance_criteria),
        "status": "COMPLETE"
    },
    {
        "measure": "Blocked prerequisites",
        "count": int((phase3F5_gate_register["acceptance_gate"] == "BLOCKED_PREREQUISITE").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Design gaps",
        "count": int((phase3F5_gate_register["acceptance_gate"] == "DESIGN_GAP").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Conditional acceptance criteria",
        "count": int((phase3F5_gate_register["acceptance_gate"] == "CONDITIONAL_ACCEPTANCE_ONLY").sum()),
        "status": "LIMITATION_RECORDED"
    },
    {
        "measure": "Numerical scales accepted",
        "count": int((phase3F5_current_decision["decision"] == "ACCEPT").sum()),
        "status": "NONE"
    },
    {
        "measure": "Numerical scales unavailable",
        "count": int((phase3F5_current_decision["decision"] == "UNAVAILABLE").sum()),
        "status": "CURRENT_DECISION"
    }
])

# Validation
assert len(phase3F5_closure_register) == 5
assert phase3F5_closure_register["closure_area"].is_unique
assert phase3F5_closure_register.notna().all().all()

assert len(phase3F5_closure_summary) == 6
assert phase3F5_closure_summary["measure"].is_unique
assert phase3F5_closure_summary.notna().all().all()

assert len(phase3F5_acceptance_criteria) == 13
assert len(phase3F5_gate_register) == 13
assert len(phase3F5_limitation_register) == 13
assert len(phase3F5_decision_rules) == 5
assert len(phase3F5_current_decision) == 6

assert (
    phase3F5_current_decision["decision"]
    .value_counts().get("UNAVAILABLE", 0)
    == 3
)

assert (
    phase3F5_current_decision.loc[
        phase3F5_current_decision["decision_area"] == "Shared conceptual framework",
        "decision"
    ].iloc[0]
    == "RETAIN_AS_CONCEPTUAL_FRAMEWORK"
)

# Export audit artifacts
phase3F5_acceptance_criteria.to_csv(
    OUTPUT_DIR / "phase3F5_acceptance_criteria.csv", index=False
)
phase3F5_gate_register.to_csv(
    OUTPUT_DIR / "phase3F5_acceptance_gate_register.csv", index=False
)
phase3F5_decision_rules.to_csv(
    OUTPUT_DIR / "phase3F5_decision_rules.csv", index=False
)
phase3F5_limitation_register.to_csv(
    OUTPUT_DIR / "phase3F5_limitation_register.csv", index=False
)
phase3F5_current_decision.to_csv(
    OUTPUT_DIR / "phase3F5_current_validation_decision.csv", index=False
)
phase3F5_closure_register.to_csv(
    OUTPUT_DIR / "phase3F5_closure_register.csv", index=False
)
phase3F5_closure_summary.to_csv(
    OUTPUT_DIR / "phase3F5_closure_summary.csv", index=False
)

phase3F5_export_audit = pd.DataFrame([
    {"file": name, "exists": (OUTPUT_DIR / name).exists()}
    for name in [
        "phase3F5_acceptance_criteria.csv",
        "phase3F5_acceptance_gate_register.csv",
        "phase3F5_decision_rules.csv",
        "phase3F5_limitation_register.csv",
        "phase3F5_current_validation_decision.csv",
        "phase3F5_closure_register.csv",
        "phase3F5_closure_summary.csv"
    ]
])

assert phase3F5_export_audit["exists"].all()

print("PHASE 3F.5.4 — CLOSURE REGISTER")
display(phase3F5_closure_register)

print("\nPHASE 3F.5.4 — CLOSURE SUMMARY")
display(phase3F5_closure_summary)

print("\nPHASE 3F.5.4 — EXPORT AUDIT")
display(phase3F5_export_audit)

print("\nVALIDATION: PASSED")

PHASE 3F.5.4 — CLOSURE REGISTER


,closure_area,status,evidence_count,conclusion
0,Acceptance criteria,COMPLETE,13,Thirteen criteria documented with explicit requirements and current evidence status.
1,Acceptance gate classification,COMPLETE,13,All criteria classified by their effect on numerical scale acceptance.
2,Limitations and required resolutions,COMPLETE,13,All identified limitations have a corresponding required resolution.
3,Current pressure-scale decision,UNAVAILABLE,6,"No first-innings, second-innings or unified numerical pressure scale can be accepted using current evidence."
4,Conceptual framework,RETAINED_WITH_LIMITATIONS,1,Shared pre-delivery framework retained; numerical validity is not established.



PHASE 3F.5.4 — CLOSURE SUMMARY


,measure,count,status
0,Acceptance criteria,13,COMPLETE
1,Blocked prerequisites,4,LIMITATION_RECORDED
2,Design gaps,3,LIMITATION_RECORDED
3,Conditional acceptance criteria,1,LIMITATION_RECORDED
4,Numerical scales accepted,0,NONE
5,Numerical scales unavailable,3,CURRENT_DECISION



PHASE 3F.5.4 — EXPORT AUDIT


,file,exists
0,phase3F5_acceptance_criteria.csv,True
1,phase3F5_acceptance_gate_register.csv,True
2,phase3F5_decision_rules.csv,True
3,phase3F5_limitation_register.csv,True
4,phase3F5_current_validation_decision.csv,True
5,phase3F5_closure_register.csv,True
6,phase3F5_closure_summary.csv,True



VALIDATION: PASSED


### Consolidated report and final closure

#### Report and closure cell

In [150]:
# Phase 3F.6.1 — Consolidated report and final closure

phase3F6_report = """# SupeRR Selector Season 2
# Phase 3F — Pressure Scale Validation

## 1. Executive conclusion

Phase 3F audited whether the pressure framework established in Phase 0
can support a numerical pressure scale using the supplied data and the
feasibility findings from Phases 2, 3B, 3C, 3D and 3E.

**Final decision: NUMERICAL PRESSURE SCALE UNAVAILABLE.**

The shared conceptual framework is retained with limitations. Separate
first-innings and second-innings measures remain conceptually preferable
to an unsupported unified numerical scale. However, neither innings-specific
numerical scale can be constructed and validated with the current inputs.

This phase did not construct pressure scores, weights, normalization
functions, thresholds, player metrics or rankings.

## 2. Scope

Phase 3F covered:

- 3F.1 — Pressure definition and scale-input audit
- 3F.2 — Numerical scale feasibility and architecture
- 3F.3 — Directionality and logical consistency
- 3F.4 — Coverage and sensitivity feasibility
- 3F.5 — Validation limitations and acceptance criteria
- 3F.6 — Consolidated report and closure

The scope was limited to pressure-scale validation feasibility.
Player-level performance scoring and subsequent phases were not performed.

## 3. Pressure definition and input audit

The Phase 0 framework distinguishes:

- First-innings scoring position relative to expected scoring.
- First-innings scoring deviation and its standardized form.
- Wickets remaining and balls remaining as separate state dimensions.
- Second-innings required-rate difficulty relative to expected scoring capability.
- Competition-specific context.
- Innings stage, where valid data exists.

The audit found:

- First-innings cumulative scoring and scoring deviation are blocked.
- First-innings remaining balls/resources are unavailable.
- Second-innings target and cumulative chase state are not validated.
- Second-innings required-rate difficulty and expected scoring benchmark are blocked.
- Wicket-state information is conditionally available.
- Competition context is available.
- Innings stage is unavailable.

The first-innings scoring deviation is not a standalone pressure measure.
The second-innings required-rate difficulty is a distinct construct.
Neither should be converted into a common numerical scale without
defensible calibration evidence.

## 4. Numerical scale architecture

The architecture assessment concluded:

| Architecture | Decision |
|---|---|
| First-innings numerical scale | BLOCKED |
| Second-innings numerical scale | BLOCKED |
| Unified numerical scale | NOT JUSTIFIED |
| Shared conceptual framework with separate innings-specific measures | CONCEPTUALLY PREFERRED, BUT BLOCKED |

A unified numerical scale cannot be justified merely because both innings
are described using the word pressure. Cross-innings mapping and calibration
would require evidence that is not currently available.

No arbitrary weights, normalization methods or thresholds were approved.

## 5. Directionality and logical consistency

The directionality audit documented eight component relationships.

Key methodological constraints:

1. Use pre-delivery information only.
2. Do not treat first-innings scoring deviation as standalone pressure.
3. Keep wickets and balls/resources separate initially.
4. Keep IPL and T20I benchmarks separate.
5. Do not condition the initial expected-rate benchmark on required rate.
6. Do not assume equal pressure values across innings.
7. Do not assign numerical weights or thresholds without evidence.

Conceptual directions were documented, but empirical directionality tests
were blocked or only descriptively feasible.

## 6. Coverage and sensitivity feasibility

Aggregate wicket-state coverage was documented for both innings.

First innings:
- Ten wicket states observed in IPL and T20I.
- 255 competition/state combinations are unobserved.
- Exact competition/state completeness is therefore not established.

Second innings:
- Ten wickets-remaining states observed in T20 and T20I.
- Reconstructed wicket states remain conditional.

Sensitivity assessment:
- Three tests blocked.
- Two partially descriptive.
- One descriptive-only.
- One policy-design-only.
- One not testable.

No numerical pressure scale exists against which alternative weights,
normalization methods, thresholds or calibration choices could be tested.
Cross-innings stability is not testable.

## 7. Acceptance criteria and limitations

Thirteen acceptance criteria were documented.

The principal limitations are:

- Cumulative scoring semantics are unresolved.
- Legal-ball progression and scheduled innings length are unvalidated.
- Target semantics and chase-state inputs are not validated.
- Reconstructed wicket states remain conditional.
- Expected scoring benchmarks were not constructible.
- Fallback policy is documented but not executable.
- Component directionality remains conceptual.
- Normalization and combination are not established.
- Sensitivity and stability are not testable.
- Cross-competition policy is defined but not empirically validated.
- Cross-innings calibration is not established.
- Pressure bands and thresholds are not established.

Acceptance gate counts:
- Four blocked prerequisites.
- Three design gaps.
- One conditional acceptance criterion.
- Remaining criteria are incomplete, conceptual, policy-only or not testable.

## 8. Final decision

| Pressure framework element | Final decision |
|---|---|
| First-innings numerical pressure scale | UNAVAILABLE |
| Second-innings numerical pressure scale | UNAVAILABLE |
| Unified cross-innings numerical scale | UNAVAILABLE |
| Shared conceptual framework | RETAINED WITH LIMITATIONS |
| Wicket-state component | CONDITIONAL |
| Pressure bands and thresholds | NOT ESTABLISHED |

The evidence does not justify accepting a numerical pressure scale.
It also does not establish that a future numerical scale is inherently
invalid. The current decision is unavailable because essential inputs,
benchmarks, design specifications and validation evidence are missing.

## 9. Requirements before future numerical construction

Before numerical construction can be reconsidered:

1. Validate run semantics and independently reconcile cumulative scores.
2. Validate legal-ball progression and scheduled innings length.
3. Verify target semantics and second-innings chase-state inputs.
4. Reconcile reconstructed wicket states against reliable dismissal evidence.
5. Construct competition-specific benchmarks with sample sizes and provenance.
6. Implement fallback rules only where their prerequisites are satisfied.
7. Define and justify component normalization and combination.
8. Empirically test directionality and sensitivity.
9. Establish cross-innings calibration before claiming a unified scale.
10. Establish empirical support before assigning pressure bands or thresholds.

Unsupported states must remain unavailable rather than being filled through
arbitrary assumptions or undocumented imputation.

## 10. Phase closure

Phase 3F is closed as a pressure-scale validation and feasibility audit.

It is not a completed numerical pressure model.
No pressure scores, weights, thresholds, player rankings or Phase 4
deliverables were produced.

**Closure status: CLOSED — CONCEPTUAL FRAMEWORK RETAINED;
NUMERICAL PRESSURE SCALE UNAVAILABLE.**
"""

phase3F6_report_path = OUTPUT_DIR / "SupeRR_Selector_S2_Phase_3F_Report.md"

with open(phase3F6_report_path, "w", encoding="utf-8") as f:
    f.write(phase3F6_report)

phase3F6_completion_summary = pd.DataFrame([
    {
        "subphase": "3F.1",
        "name": "Pressure definition and scale-input audit",
        "status": "COMPLETE"
    },
    {
        "subphase": "3F.2",
        "name": "Numerical scale feasibility",
        "status": "COMPLETE"
    },
    {
        "subphase": "3F.3",
        "name": "Directionality and logical consistency",
        "status": "COMPLETE"
    },
    {
        "subphase": "3F.4",
        "name": "Coverage and sensitivity feasibility",
        "status": "COMPLETE"
    },
    {
        "subphase": "3F.5",
        "name": "Validation limitations and acceptance criteria",
        "status": "COMPLETE"
    },
    {
        "subphase": "3F.6",
        "name": "Consolidated report and final closure",
        "status": "COMPLETE"
    }
])

phase3F6_final_decision = pd.DataFrame([
    {
        "scope": "First-innings numerical pressure scale",
        "decision": "UNAVAILABLE"
    },
    {
        "scope": "Second-innings numerical pressure scale",
        "decision": "UNAVAILABLE"
    },
    {
        "scope": "Unified cross-innings numerical scale",
        "decision": "UNAVAILABLE"
    },
    {
        "scope": "Shared conceptual framework",
        "decision": "RETAINED_WITH_LIMITATIONS"
    },
    {
        "scope": "Wicket-state component",
        "decision": "CONDITIONAL"
    },
    {
        "scope": "Pressure bands and thresholds",
        "decision": "NOT_ESTABLISHED"
    }
])

phase3F6_validation = pd.DataFrame([
    {
        "check": "All six Phase 3F subphases listed",
        "passed": len(phase3F6_completion_summary) == 6
    },
    {
        "check": "All subphases marked complete",
        "passed": phase3F6_completion_summary["status"].eq("COMPLETE").all()
    },
    {
        "check": "Three numerical scale decisions unavailable",
        "passed": (
            phase3F6_final_decision["decision"].eq("UNAVAILABLE").sum() == 3
        )
    },
    {
        "check": "Conceptual framework retained with limitations",
        "passed": (
            phase3F6_final_decision.loc[
                phase3F6_final_decision["scope"] == "Shared conceptual framework",
                "decision"
            ].iloc[0] == "RETAINED_WITH_LIMITATIONS"
        )
    },
    {
        "check": "Report file exists",
        "passed": phase3F6_report_path.exists()
    },
    {
        "check": "Report contains final closure statement",
        "passed": (
            "NUMERICAL PRESSURE SCALE UNAVAILABLE"
            in phase3F6_report_path.read_text(encoding="utf-8")
        )
    }
])

assert phase3F6_validation["passed"].all()

phase3F6_completion_summary.to_csv(
    OUTPUT_DIR / "phase3F_completion_summary.csv", index=False
)
phase3F6_final_decision.to_csv(
    OUTPUT_DIR / "phase3F_final_decision.csv", index=False
)
phase3F6_validation.to_csv(
    OUTPUT_DIR / "phase3F_final_validation_register.csv", index=False
)

phase3F6_export_audit = pd.DataFrame([
    {
        "file": name,
        "exists": (OUTPUT_DIR / name).exists()
    }
    for name in [
        "SupeRR_Selector_S2_Phase_3F_Report.md",
        "phase3F_completion_summary.csv",
        "phase3F_final_decision.csv",
        "phase3F_final_validation_register.csv"
    ]
])

assert phase3F6_export_audit["exists"].all()

print("PHASE 3F.6.1 — SUBPHASE COMPLETION")
display(phase3F6_completion_summary)

print("\nPHASE 3F.6.1 — FINAL DECISION")
display(phase3F6_final_decision)

print("\nPHASE 3F.6.1 — VALIDATION")
display(phase3F6_validation)

print("\nPHASE 3F.6.1 — EXPORT AUDIT")
display(phase3F6_export_audit)

print("\nREPORT PATH:")
print(phase3F6_report_path)

print("\nVALIDATION: PASSED")

PHASE 3F.6.1 — SUBPHASE COMPLETION


,subphase,name,status
0,3F.1,Pressure definition and scale-input audit,COMPLETE
1,3F.2,Numerical scale feasibility,COMPLETE
2,3F.3,Directionality and logical consistency,COMPLETE
3,3F.4,Coverage and sensitivity feasibility,COMPLETE
4,3F.5,Validation limitations and acceptance criteria,COMPLETE
5,3F.6,Consolidated report and final closure,COMPLETE



PHASE 3F.6.1 — FINAL DECISION


,scope,decision
0,First-innings numerical pressure scale,UNAVAILABLE
1,Second-innings numerical pressure scale,UNAVAILABLE
2,Unified cross-innings numerical scale,UNAVAILABLE
3,Shared conceptual framework,RETAINED_WITH_LIMITATIONS
4,Wicket-state component,CONDITIONAL
5,Pressure bands and thresholds,NOT_ESTABLISHED



PHASE 3F.6.1 — VALIDATION


,check,passed
0,All six Phase 3F subphases listed,True
1,All subphases marked complete,True
2,Three numerical scale decisions unavailable,True
3,Conceptual framework retained with limitations,True
4,Report file exists,True
5,Report contains final closure statement,True



PHASE 3F.6.1 — EXPORT AUDIT


,file,exists
0,SupeRR_Selector_S2_Phase_3F_Report.md,True
1,phase3F_completion_summary.csv,True
2,phase3F_final_decision.csv,True
3,phase3F_final_validation_register.csv,True



REPORT PATH:
/content/phase3_outputs/SupeRR_Selector_S2_Phase_3F_Report.md

VALIDATION: PASSED


# Robustness and validation

### Scope and validation-readiness audit

In [151]:
# Phase 3G.1 — Scope and validation-readiness audit

# 1. Define the robustness and validation scope
phase3G_scope = pd.DataFrame([
    {
        "validation_area": "Approved match population",
        "objective": "Check consistency of the approved match population and its format composition.",
        "proposed_test": "Population reconciliation and competition-format consistency",
        "initial_feasibility": "EXECUTABLE"
    },
    {
        "validation_area": "Delivery structure",
        "objective": "Check delivery-key uniqueness, completeness and match/innings structure.",
        "proposed_test": "Key integrity, duplicate detection and structural reconciliation",
        "initial_feasibility": "EXECUTABLE"
    },
    {
        "validation_area": "Pre-delivery wicket states",
        "objective": "Check robustness of reconstructed wicket-state alignment and continuity.",
        "proposed_test": "Independent key alignment, state bounds and transition consistency",
        "initial_feasibility": "CONDITIONAL"
    },
    {
        "validation_area": "Competition separation",
        "objective": "Check whether IPL and T20I evidence remains separately identified.",
        "proposed_test": "Format and competition stratification audit",
        "initial_feasibility": "EXECUTABLE"
    },
    {
        "validation_area": "State coverage",
        "objective": "Assess the effect of sparse and unobserved competition/state combinations.",
        "proposed_test": "Coverage comparison and missing-state diagnostics",
        "initial_feasibility": "DESCRIPTIVE_ONLY"
    },
    {
        "validation_area": "Fallback decisions",
        "objective": "Check consistency between fallback policy and the documented unavailable-state rules.",
        "proposed_test": "Policy and limitation consistency audit",
        "initial_feasibility": "POLICY_AUDIT_ONLY"
    },
    {
        "validation_area": "Pressure-scale numerical robustness",
        "objective": "Test alternative scores, weights, normalization and thresholds.",
        "proposed_test": "Numerical sensitivity and calibration testing",
        "initial_feasibility": "BLOCKED"
    },
    {
        "validation_area": "Cross-innings robustness",
        "objective": "Test stability and comparability of first- and second-innings pressure measures.",
        "proposed_test": "Cross-innings calibration and stability comparison",
        "initial_feasibility": "BLOCKED"
    },
    {
        "validation_area": "Player-level robustness",
        "objective": "Test player metrics, scores and rankings under alternative specifications.",
        "proposed_test": "Player-level sensitivity and ranking stability",
        "initial_feasibility": "OUT_OF_SCOPE"
    }
])

# 2. Inventory the key existing project objects
phase3G_required_objects = [
    "df_deliveries",
    "df_source_matches",
    "df_source_ball",
    "phase3_benchmark_population",
    "phase3_pre_delivery_wicket_states",
    "phase3D_first_innings_states",
    "phase3D_first_innings_population",
    "phase3D_wicket_distribution",
    "phase3D_missing_comp_wicket_states",
    "phase3E_second_innings_wicket_alignment",
    "phase3E_wickets_remaining_distribution",
    "phase3E_wicket_match_consistency",
    "phase3C_state_schema",
    "phase3C_exception_register",
    "phase3D_fallback_policy",
    "phase3E_fallback_policy",
    "phase3F1_decision_register",
    "phase3F2_architecture_decisions",
    "phase3F3_directionality",
    "phase3F4_sensitivity_register",
    "phase3F5_current_decision",
    "phase3F6_validation"
]

phase3G_object_inventory_rows = []

for object_name in phase3G_required_objects:
    obj = globals().get(object_name)
    exists = obj is not None

    if isinstance(obj, pd.DataFrame):
        object_type = "DataFrame"
        row_count, column_count = obj.shape
    elif exists:
        object_type = type(obj).__name__
        row_count, column_count = None, None
    else:
        object_type = "MISSING"
        row_count, column_count = None, None

    phase3G_object_inventory_rows.append({
        "object_name": object_name,
        "exists": exists,
        "object_type": object_type,
        "rows": row_count,
        "columns": column_count
    })

phase3G_object_inventory = pd.DataFrame(phase3G_object_inventory_rows)

# 3. Summarise initial readiness
phase3G_readiness_summary = (
    phase3G_scope.groupby("initial_feasibility")
    .size()
    .reset_index(name="validation_area_count")
)

phase3G_missing_objects = phase3G_object_inventory.loc[
    ~phase3G_object_inventory["exists"],
    ["object_name"]
].copy()

# 4. Validation
assert phase3G_scope["validation_area"].is_unique
assert phase3G_scope.notna().all().all()
assert len(phase3G_scope) == 9

assert phase3G_object_inventory["object_name"].is_unique
assert len(phase3G_object_inventory) == len(phase3G_required_objects)
assert phase3G_object_inventory["exists"].isin([True, False]).all()

assert phase3G_readiness_summary["validation_area_count"].sum() == 9
assert (
    phase3G_scope.loc[
        phase3G_scope["validation_area"] == "Pressure-scale numerical robustness",
        "initial_feasibility"
    ].iloc[0] == "BLOCKED"
)

# 5. Export audit artifacts
phase3G_scope.to_csv(
    OUTPUT_DIR / "phase3G_scope_register.csv", index=False
)
phase3G_object_inventory.to_csv(
    OUTPUT_DIR / "phase3G_object_inventory.csv", index=False
)
phase3G_readiness_summary.to_csv(
    OUTPUT_DIR / "phase3G_readiness_summary.csv", index=False
)

print("PHASE 3G.1 — SCOPE REGISTER")
display(phase3G_scope)

print("\nPHASE 3G.1 — OBJECT INVENTORY")
display(phase3G_object_inventory)

print("\nPHASE 3G.1 — READINESS SUMMARY")
display(phase3G_readiness_summary)

print("\nMISSING OBJECTS — IF ANY")
display(phase3G_missing_objects)

print("\nVALIDATION: PASSED")

PHASE 3G.1 — SCOPE REGISTER


,validation_area,objective,proposed_test,initial_feasibility
0,Approved match population,Check consistency of the approved match population and its format composition.,Population reconciliation and competition-format consistency,EXECUTABLE
1,Delivery structure,"Check delivery-key uniqueness, completeness and match/innings structure.","Key integrity, duplicate detection and structural reconciliation",EXECUTABLE
2,Pre-delivery wicket states,Check robustness of reconstructed wicket-state alignment and continuity.,"Independent key alignment, state bounds and transition consistency",CONDITIONAL
3,Competition separation,Check whether IPL and T20I evidence remains separately identified.,Format and competition stratification audit,EXECUTABLE
4,State coverage,Assess the effect of sparse and unobserved competition/state combinations.,Coverage comparison and missing-state diagnostics,DESCRIPTIVE_ONLY
5,Fallback decisions,Check consistency between fallback policy and the documented unavailable-state rules.,Policy and limitation consistency audit,POLICY_AUDIT_ONLY
6,Pressure-scale numerical robustness,"Test alternative scores, weights, normalization and thresholds.",Numerical sensitivity and calibration testing,BLOCKED
7,Cross-innings robustness,Test stability and comparability of first- and second-innings pressure measures.,Cross-innings calibration and stability comparison,BLOCKED
8,Player-level robustness,"Test player metrics, scores and rankings under alternative specifications.",Player-level sensitivity and ranking stability,OUT_OF_SCOPE



PHASE 3G.1 — OBJECT INVENTORY


,object_name,exists,object_type,rows,columns
0,df_deliveries,True,DataFrame,192235,61
1,df_source_matches,True,DataFrame,883,22
2,df_source_ball,True,DataFrame,200999,33
3,phase3_benchmark_population,True,DataFrame,813,6
4,phase3_pre_delivery_wicket_states,True,DataFrame,192235,65
5,phase3D_first_innings_states,True,DataFrame,100333,65
6,phase3D_first_innings_population,True,DataFrame,100333,69
7,phase3D_wicket_distribution,True,DataFrame,20,5
8,phase3D_missing_comp_wicket_states,True,DataFrame,255,4
9,phase3E_second_innings_wicket_alignment,True,DataFrame,91902,64



PHASE 3G.1 — READINESS SUMMARY


,initial_feasibility,validation_area_count
0,BLOCKED,2
1,CONDITIONAL,1
2,DESCRIPTIVE_ONLY,1
3,EXECUTABLE,3
4,OUT_OF_SCOPE,1
5,POLICY_AUDIT_ONLY,1



MISSING OBJECTS — IF ANY


,object_name



VALIDATION: PASSED


### Source population and structural robustness

#### Match population reconciliation

In [152]:
# Phase 3G.2.1 — Match population reconciliation

# 1. Confirm required identifiers
required_match_key = "match_id"

for object_name, obj in [
    ("df_source_matches", df_source_matches),
    ("phase3_benchmark_population", phase3_benchmark_population),
    ("df_deliveries", df_deliveries)
]:
    assert required_match_key in obj.columns, (
        f"{object_name} is missing required key: {required_match_key}"
    )

# 2. Identify match-level populations
source_match_ids = (
    df_source_matches[[required_match_key]]
    .drop_duplicates()
    .copy()
)

approved_match_ids = (
    phase3_benchmark_population[[required_match_key]]
    .drop_duplicates()
    .copy()
)

delivery_match_ids = (
    df_deliveries[[required_match_key]]
    .drop_duplicates()
    .copy()
)

# 3. Check duplicate match IDs in match-level tables
phase3G_population_key_audit = pd.DataFrame([
    {
        "dataset": "df_source_matches",
        "rows": len(df_source_matches),
        "unique_match_ids": df_source_matches[required_match_key].nunique(),
        "duplicate_key_rows": int(
            df_source_matches[required_match_key].duplicated().sum()
        )
    },
    {
        "dataset": "phase3_benchmark_population",
        "rows": len(phase3_benchmark_population),
        "unique_match_ids": phase3_benchmark_population[required_match_key].nunique(),
        "duplicate_key_rows": int(
            phase3_benchmark_population[required_match_key].duplicated().sum()
        )
    },
    {
        "dataset": "df_deliveries",
        "rows": len(df_deliveries),
        "unique_match_ids": df_deliveries[required_match_key].nunique(),
        "duplicate_key_rows": None
    }
])

# 4. Compare approved population with source and delivery match IDs
source_reconciliation = approved_match_ids.merge(
    source_match_ids.assign(source_match_present=True),
    on=required_match_key,
    how="outer",
    indicator="source_reconciliation",
    validate="one_to_one"
)

delivery_reconciliation = approved_match_ids.merge(
    delivery_match_ids.assign(delivery_match_present=True),
    on=required_match_key,
    how="outer",
    indicator="delivery_reconciliation",
    validate="one_to_one"
)

phase3G_source_reconciliation_summary = (
    source_reconciliation["source_reconciliation"]
    .value_counts()
    .rename_axis("reconciliation_status")
    .reset_index(name="match_count")
)

phase3G_delivery_reconciliation_summary = (
    delivery_reconciliation["delivery_reconciliation"]
    .value_counts()
    .rename_axis("reconciliation_status")
    .reset_index(name="match_count")
)

# 5. Compare shared match-level metadata
metadata_fields = [
    field for field in ["comp_id", "match_format", "season"]
    if field in phase3_benchmark_population.columns
    and field in df_source_matches.columns
]

phase3G_metadata_comparison = []

if metadata_fields:
    approved_metadata = phase3_benchmark_population[
        [required_match_key] + metadata_fields
    ].copy()

    source_metadata = df_source_matches[
        [required_match_key] + metadata_fields
    ].copy()

    metadata_join = approved_metadata.merge(
        source_metadata,
        on=required_match_key,
        how="inner",
        suffixes=("_approved", "_source"),
        validate="one_to_one"
    )

    for field in metadata_fields:
        approved_col = f"{field}_approved"
        source_col = f"{field}_source"

        mismatch = (
            metadata_join[approved_col].notna()
            & metadata_join[source_col].notna()
            & metadata_join[approved_col].ne(metadata_join[source_col])
        )

        phase3G_metadata_comparison.append({
            "metadata_field": field,
            "compared_matches": len(metadata_join),
            "mismatched_matches": int(mismatch.sum()),
            "missing_approved_values": int(
                metadata_join[approved_col].isna().sum()
            ),
            "missing_source_values": int(
                metadata_join[source_col].isna().sum()
            )
        })

phase3G_metadata_comparison = pd.DataFrame(
    phase3G_metadata_comparison
)

# 6. Preserve discrepancy records for investigation
phase3G_source_population_exceptions = source_reconciliation.loc[
    source_reconciliation["source_reconciliation"] != "both"
].copy()

phase3G_delivery_population_exceptions = delivery_reconciliation.loc[
    delivery_reconciliation["delivery_reconciliation"] != "both"
].copy()

# 7. Validation
assert phase3G_population_key_audit["unique_match_ids"].notna().all()
assert phase3G_source_reconciliation_summary["match_count"].sum() == (
    len(source_match_ids) + len(approved_match_ids)
    - source_reconciliation["source_reconciliation"].eq("both").sum()
)
assert phase3G_delivery_reconciliation_summary["match_count"].sum() == (
    len(delivery_match_ids) + len(approved_match_ids)
    - delivery_reconciliation["delivery_reconciliation"].eq("both").sum()
)

assert (
    phase3G_population_key_audit.loc[
        phase3G_population_key_audit["dataset"].isin(
            ["df_source_matches", "phase3_benchmark_population"]
        ),
        "duplicate_key_rows"
    ] == 0
).all()

# 8. Export
phase3G_population_key_audit.to_csv(
    OUTPUT_DIR / "phase3G_population_key_audit.csv", index=False
)
phase3G_source_reconciliation_summary.to_csv(
    OUTPUT_DIR / "phase3G_source_population_reconciliation.csv", index=False
)
phase3G_delivery_reconciliation_summary.to_csv(
    OUTPUT_DIR / "phase3G_delivery_population_reconciliation.csv", index=False
)
phase3G_metadata_comparison.to_csv(
    OUTPUT_DIR / "phase3G_population_metadata_comparison.csv", index=False
)
phase3G_source_population_exceptions.to_csv(
    OUTPUT_DIR / "phase3G_source_population_exceptions.csv", index=False
)
phase3G_delivery_population_exceptions.to_csv(
    OUTPUT_DIR / "phase3G_delivery_population_exceptions.csv", index=False
)

print("PHASE 3G.2.1 — POPULATION KEY AUDIT")
display(phase3G_population_key_audit)

print("\nSOURCE MATCH POPULATION RECONCILIATION")
display(phase3G_source_reconciliation_summary)

print("\nINTEGRATED DELIVERY POPULATION RECONCILIATION")
display(phase3G_delivery_reconciliation_summary)

print("\nMATCH METADATA COMPARISON")
display(phase3G_metadata_comparison)

print("\nSOURCE POPULATION EXCEPTIONS")
display(phase3G_source_population_exceptions.head(20))

print("\nDELIVERY POPULATION EXCEPTIONS")
display(phase3G_delivery_population_exceptions.head(20))

print("\nVALIDATION: PASSED")

PHASE 3G.2.1 — POPULATION KEY AUDIT


,dataset,rows,unique_match_ids,duplicate_key_rows
0,df_source_matches,883,883,0.0
1,phase3_benchmark_population,813,813,0.0
2,df_deliveries,192235,813,NaN



SOURCE MATCH POPULATION RECONCILIATION


,reconciliation_status,match_count
0,both,813
1,right_only,70
2,left_only,0



INTEGRATED DELIVERY POPULATION RECONCILIATION


,reconciliation_status,match_count
0,both,813
1,left_only,0
2,right_only,0


... (3 outputs left out)

,match_id,source_match_present,source_reconciliation
91,15410,True,right_only
95,15546,True,right_only
177,17765,True,right_only
240,20226,True,right_only
277,21406,True,right_only
318,28333,True,right_only
408,31281,True,right_only
411,31284,True,right_only
415,31288,True,right_only
420,31707,True,right_only



DELIVERY POPULATION EXCEPTIONS


,match_id,delivery_match_present,delivery_reconciliation



VALIDATION: PASSED


#### Delivery-key and innings-structure robustness

In [153]:
# Phase 3G.2.2 — Delivery-key and innings-structure robustness

# 1. Define the approved delivery key and structural fields
phase3G_delivery_key = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num"
]

phase3G_team_fields = [
    "match_id",
    "innings",
    "batting_team_id",
    "bowling_team_id"
]

required_fields = phase3G_delivery_key + [
    "batting_team_id",
    "bowling_team_id"
]

missing_fields = [
    col for col in required_fields
    if col not in df_deliveries.columns
]

assert not missing_fields, (
    f"Required delivery fields missing: {missing_fields}"
)

# 2. Delivery key integrity
phase3G_delivery_key_audit = pd.DataFrame([
    {
        "check": "Total integrated delivery records",
        "value": len(df_deliveries)
    },
    {
        "check": "Unique delivery keys",
        "value": int(
            df_deliveries[phase3G_delivery_key]
            .drop_duplicates()
            .shape[0]
        )
    },
    {
        "check": "Duplicate delivery-key rows",
        "value": int(
            df_deliveries.duplicated(
                subset=phase3G_delivery_key,
                keep=False
            ).sum()
        )
    },
    {
        "check": "Rows with missing delivery-key fields",
        "value": int(
            df_deliveries[phase3G_delivery_key]
            .isna().any(axis=1).sum()
        )
    },
    {
        "check": "Rows with missing batting team",
        "value": int(df_deliveries["batting_team_id"].isna().sum())
    },
    {
        "check": "Rows with missing bowling team",
        "value": int(df_deliveries["bowling_team_id"].isna().sum())
    }
])

# 3. Preserve duplicate and missing-key exception rows
phase3G_duplicate_delivery_keys = df_deliveries.loc[
    df_deliveries.duplicated(
        subset=phase3G_delivery_key,
        keep=False
    ),
    phase3G_delivery_key
].copy()

phase3G_missing_delivery_keys = df_deliveries.loc[
    df_deliveries[phase3G_delivery_key].isna().any(axis=1),
    phase3G_delivery_key
].copy()

# 4. Match/innings structure
phase3G_innings_counts = (
    df_deliveries.groupby("match_id")["innings"]
    .nunique()
    .rename("distinct_innings")
    .reset_index()
)

phase3G_innings_record_counts = (
    df_deliveries.groupby(["match_id", "innings"])
    .size()
    .rename("delivery_records")
    .reset_index()
)

phase3G_match_innings_summary = (
    phase3G_innings_counts["distinct_innings"]
    .value_counts()
    .sort_index()
    .rename_axis("distinct_innings")
    .reset_index(name="match_count")
)

phase3G_incomplete_innings_matches = phase3G_innings_counts.loc[
    phase3G_innings_counts["distinct_innings"] != 2
].copy()

# 5. Team consistency within each innings
phase3G_innings_team_summary = (
    df_deliveries.groupby(["match_id", "innings"])
    .agg(
        distinct_batting_teams=("batting_team_id", "nunique"),
        distinct_bowling_teams=("bowling_team_id", "nunique"),
        missing_batting_team=("batting_team_id", lambda x: x.isna().any()),
        missing_bowling_team=("bowling_team_id", lambda x: x.isna().any())
    )
    .reset_index()
)

phase3G_innings_team_exceptions = phase3G_innings_team_summary.loc[
    (phase3G_innings_team_summary["distinct_batting_teams"] != 1)
    | (phase3G_innings_team_summary["distinct_bowling_teams"] != 1)
    | phase3G_innings_team_summary["missing_batting_team"]
    | phase3G_innings_team_summary["missing_bowling_team"]
].copy()

# 6. Verify team reversal across the two innings
phase3G_team_pairs = phase3G_innings_team_summary.merge(
    df_deliveries[
        ["match_id", "innings", "batting_team_id", "bowling_team_id"]
    ].drop_duplicates(),
    on=["match_id", "innings"],
    how="left",
    validate="one_to_one"
)

phase3G_team_wide = phase3G_team_pairs.pivot(
    index="match_id",
    columns="innings",
    values=["batting_team_id", "bowling_team_id"]
)

phase3G_team_wide.columns = [
    f"{field}_innings_{innings}"
    for field, innings in phase3G_team_wide.columns
]
phase3G_team_wide = phase3G_team_wide.reset_index()

expected_team_columns = [
    "batting_team_id_innings_1",
    "bowling_team_id_innings_1",
    "batting_team_id_innings_2",
    "bowling_team_id_innings_2"
]

if all(col in phase3G_team_wide.columns for col in expected_team_columns):
    phase3G_team_reversal_exceptions = phase3G_team_wide.loc[
        (
            phase3G_team_wide["batting_team_id_innings_1"]
            != phase3G_team_wide["bowling_team_id_innings_2"]
        )
        | (
            phase3G_team_wide["bowling_team_id_innings_1"]
            != phase3G_team_wide["batting_team_id_innings_2"]
        )
    ].copy()
else:
    phase3G_team_reversal_exceptions = pd.DataFrame()

# 7. Match-level structural summary
phase3G_structure_summary = pd.DataFrame([
    {
        "measure": "Matches represented in deliveries",
        "value": int(df_deliveries["match_id"].nunique())
    },
    {
        "measure": "Matches with exactly two innings",
        "value": int(
            phase3G_innings_counts["distinct_innings"].eq(2).sum()
        )
    },
    {
        "measure": "Matches with other than two innings",
        "value": len(phase3G_incomplete_innings_matches)
    },
    {
        "measure": "Innings with inconsistent team identity",
        "value": len(phase3G_innings_team_exceptions)
    },
    {
        "measure": "Matches with team-reversal exceptions",
        "value": len(phase3G_team_reversal_exceptions)
    }
])

# 8. Validation — report results without assuming all checks pass
assert len(phase3G_delivery_key_audit) == 6
assert phase3G_delivery_key_audit["value"].notna().all()
assert phase3G_match_innings_summary["match_count"].sum() == (
    df_deliveries["match_id"].nunique()
)
assert phase3G_structure_summary["value"].notna().all()

# 9. Export
phase3G_delivery_key_audit.to_csv(
    OUTPUT_DIR / "phase3G_delivery_key_audit.csv", index=False
)
phase3G_duplicate_delivery_keys.to_csv(
    OUTPUT_DIR / "phase3G_duplicate_delivery_keys.csv", index=False
)
phase3G_missing_delivery_keys.to_csv(
    OUTPUT_DIR / "phase3G_missing_delivery_keys.csv", index=False
)
phase3G_match_innings_summary.to_csv(
    OUTPUT_DIR / "phase3G_match_innings_summary.csv", index=False
)
phase3G_incomplete_innings_matches.to_csv(
    OUTPUT_DIR / "phase3G_incomplete_innings_matches.csv", index=False
)
phase3G_innings_team_exceptions.to_csv(
    OUTPUT_DIR / "phase3G_innings_team_exceptions.csv", index=False
)
phase3G_team_reversal_exceptions.to_csv(
    OUTPUT_DIR / "phase3G_team_reversal_exceptions.csv", index=False
)
phase3G_structure_summary.to_csv(
    OUTPUT_DIR / "phase3G_structure_summary.csv", index=False
)

print("PHASE 3G.2.2 — DELIVERY KEY AUDIT")
display(phase3G_delivery_key_audit)

print("\nMATCH INNINGS DISTRIBUTION")
display(phase3G_match_innings_summary)

print("\nSTRUCTURAL SUMMARY")
display(phase3G_structure_summary)

print("\nDUPLICATE DELIVERY KEYS")
display(phase3G_duplicate_delivery_keys.head(20))

print("\nMISSING DELIVERY KEYS")
display(phase3G_missing_delivery_keys.head(20))

print("\nINCOMPLETE INNINGS MATCHES")
display(phase3G_incomplete_innings_matches.head(20))

print("\nINNINGS TEAM EXCEPTIONS")
display(phase3G_innings_team_exceptions.head(20))

print("\nTEAM REVERSAL EXCEPTIONS")
display(phase3G_team_reversal_exceptions.head(20))

print("\nSTRUCTURAL AUDIT CELL: COMPLETED")

PHASE 3G.2.2 — DELIVERY KEY AUDIT


,check,value
0,Total integrated delivery records,192235
1,Unique delivery keys,192235
2,Duplicate delivery-key rows,0
3,Rows with missing delivery-key fields,0
4,Rows with missing batting team,0
5,Rows with missing bowling team,0



MATCH INNINGS DISTRIBUTION


,distinct_innings,match_count
0,2,813



STRUCTURAL SUMMARY


,measure,value
0,Matches represented in deliveries,813
1,Matches with exactly two innings,813
2,Matches with other than two innings,0
3,Innings with inconsistent team identity,0
4,Matches with team-reversal exceptions,0


... (7 outputs left out)

,match_id,innings,distinct_batting_teams,distinct_bowling_teams,missing_batting_team,missing_bowling_team



TEAM REVERSAL EXCEPTIONS


,match_id,batting_team_id_innings_1,batting_team_id_innings_2,bowling_team_id_innings_1,bowling_team_id_innings_2



STRUCTURAL AUDIT CELL: COMPLETED


#### Delivery sequence robustness

In [154]:
# Phase 3G.2.3 — Delivery sequence robustness

# 1. Validate required columns
phase3G_sequence_fields = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num"
]

missing_sequence_fields = [
    col for col in phase3G_sequence_fields
    if col not in df_deliveries.columns
]

assert not missing_sequence_fields, (
    f"Missing sequence fields: {missing_sequence_fields}"
)

# 2. Create a derived audit table; raw source remains unchanged
phase3G_sequence_data = df_deliveries[
    phase3G_sequence_fields
].copy()

phase3G_sequence_data["innings_ball_num_numeric"] = pd.to_numeric(
    phase3G_sequence_data["innings_ball_num"],
    errors="coerce"
)

phase3G_sequence_data["overid_numeric"] = pd.to_numeric(
    phase3G_sequence_data["overid"],
    errors="coerce"
)

phase3G_sequence_data["over_ball_num_numeric"] = pd.to_numeric(
    phase3G_sequence_data["over_ball_num"],
    errors="coerce"
)

# 3. Overall sequence-field quality
phase3G_sequence_quality = pd.DataFrame([
    {
        "field": field,
        "missing_values": int(phase3G_sequence_data[field].isna().sum()),
        "non_numeric_values": int(
            pd.to_numeric(
                phase3G_sequence_data[field],
                errors="coerce"
            ).isna().sum()
            - phase3G_sequence_data[field].isna().sum()
        ),
        "distinct_values": int(
            phase3G_sequence_data[field].nunique(dropna=True)
        ),
        "minimum": pd.to_numeric(
            phase3G_sequence_data[field],
            errors="coerce"
        ).min(),
        "maximum": pd.to_numeric(
            phase3G_sequence_data[field],
            errors="coerce"
        ).max()
    }
    for field in ["innings_ball_num", "overid", "over_ball_num"]
])

# 4. Innings-level sequence diagnostics
phase3G_sequence_data = phase3G_sequence_data.sort_values(
    ["match_id", "innings", "innings_ball_num_numeric"],
    kind="stable"
)

phase3G_innings_sequence_rows = []

for (match_id, innings), group in phase3G_sequence_data.groupby(
    ["match_id", "innings"],
    dropna=False
):
    seq = group["innings_ball_num_numeric"].dropna()

    integer_valued = bool(
        seq.mod(1).eq(0).all()
    ) if len(seq) else False

    duplicate_positions = int(seq.duplicated(keep=False).sum())

    if len(seq):
        expected_positions = set(range(int(seq.min()), int(seq.max()) + 1))
        observed_positions = set(seq.astype(int).tolist())
        missing_positions = sorted(expected_positions - observed_positions)
        sequence_contiguous = len(missing_positions) == 0
    else:
        missing_positions = []
        sequence_contiguous = False

    over_seq = group["overid_numeric"]
    paired = group[
        ["innings_ball_num_numeric", "overid_numeric"]
    ].dropna()

    over_non_decreasing = (
        bool(paired["overid_numeric"].is_monotonic_increasing)
        if len(paired) else False
    )

    phase3G_innings_sequence_rows.append({
        "match_id": match_id,
        "innings": innings,
        "delivery_records": len(group),
        "sequence_values": int(seq.nunique()),
        "sequence_min": seq.min() if len(seq) else None,
        "sequence_max": seq.max() if len(seq) else None,
        "integer_valued": integer_valued,
        "duplicate_sequence_positions": duplicate_positions,
        "missing_sequence_positions": len(missing_positions),
        "sequence_contiguous": sequence_contiguous,
        "overid_non_decreasing": over_non_decreasing,
        "distinct_overid": int(over_seq.nunique(dropna=True))
    })

phase3G_innings_sequence_audit = pd.DataFrame(
    phase3G_innings_sequence_rows
)

# 5. Summarise anomalies
phase3G_sequence_anomaly_summary = pd.DataFrame([
    {
        "diagnostic": "Innings with non-integer sequence positions",
        "count": int((~phase3G_innings_sequence_audit["integer_valued"]).sum())
    },
    {
        "diagnostic": "Innings with duplicate sequence positions",
        "count": int(
            (phase3G_innings_sequence_audit["duplicate_sequence_positions"] > 0).sum()
        )
    },
    {
        "diagnostic": "Innings with missing sequence positions",
        "count": int(
            (phase3G_innings_sequence_audit["missing_sequence_positions"] > 0).sum()
        )
    },
    {
        "diagnostic": "Innings with non-contiguous sequence positions",
        "count": int((~phase3G_innings_sequence_audit["sequence_contiguous"]).sum())
    },
    {
        "diagnostic": "Innings with decreasing overid in sequence order",
        "count": int((~phase3G_innings_sequence_audit["overid_non_decreasing"]).sum())
    }
])

# 6. Preserve exception rows
phase3G_sequence_exceptions = phase3G_innings_sequence_audit.loc[
    (~phase3G_innings_sequence_audit["integer_valued"])
    | (phase3G_innings_sequence_audit["duplicate_sequence_positions"] > 0)
    | (phase3G_innings_sequence_audit["missing_sequence_positions"] > 0)
    | (~phase3G_innings_sequence_audit["sequence_contiguous"])
    | (~phase3G_innings_sequence_audit["overid_non_decreasing"])
].copy()

# 7. Validation of audit construction
assert len(phase3G_innings_sequence_audit) == (
    df_deliveries[["match_id", "innings"]].drop_duplicates().shape[0]
)
assert phase3G_innings_sequence_audit["delivery_records"].sum() == len(df_deliveries)
assert phase3G_sequence_anomaly_summary["count"].ge(0).all()
assert phase3G_innings_sequence_audit["match_id"].notna().all()
assert phase3G_innings_sequence_audit["innings"].notna().all()

# 8. Export
phase3G_sequence_quality.to_csv(
    OUTPUT_DIR / "phase3G_sequence_field_quality.csv", index=False
)
phase3G_innings_sequence_audit.to_csv(
    OUTPUT_DIR / "phase3G_innings_sequence_audit.csv", index=False
)
phase3G_sequence_anomaly_summary.to_csv(
    OUTPUT_DIR / "phase3G_sequence_anomaly_summary.csv", index=False
)
phase3G_sequence_exceptions.to_csv(
    OUTPUT_DIR / "phase3G_sequence_exceptions.csv", index=False
)

print("PHASE 3G.2.3 — SEQUENCE FIELD QUALITY")
display(phase3G_sequence_quality)

print("\nPHASE 3G.2.3 — ANOMALY SUMMARY")
display(phase3G_sequence_anomaly_summary)

print("\nPHASE 3G.2.3 — INNINGS SEQUENCE AUDIT SAMPLE")
display(phase3G_innings_sequence_audit.head(15))

print("\nPHASE 3G.2.3 — SEQUENCE EXCEPTIONS")
display(phase3G_sequence_exceptions.head(30))

print("\nAUDIT CONSTRUCTION: COMPLETED")

PHASE 3G.2.3 — SEQUENCE FIELD QUALITY


,field,missing_values,non_numeric_values,distinct_values,minimum,maximum
0,innings_ball_num,0,0,136,1,136
1,overid,0,0,20,1,20
2,over_ball_num,0,0,13,1,13



PHASE 3G.2.3 — ANOMALY SUMMARY


,diagnostic,count
0,Innings with non-integer sequence positions,0
1,Innings with duplicate sequence positions,0
2,Innings with missing sequence positions,0
3,Innings with non-contiguous sequence positions,0
4,Innings with decreasing overid in sequence order,0



PHASE 3G.2.3 — INNINGS SEQUENCE AUDIT SAMPLE


,match_id,innings,delivery_records,sequence_values,sequence_min,sequence_max,integer_valued,duplicate_sequence_positions,missing_sequence_positions,sequence_contiguous,overid_non_decreasing,distinct_overid
0,13511,1,121,121,1,121,True,0,0,True,True,20
1,13511,2,103,103,1,103,True,0,0,True,True,18
2,13512,1,128,128,1,128,True,0,0,True,True,20
3,13512,2,124,124,1,124,True,0,0,True,True,20
4,13513,1,125,125,1,125,True,0,0,True,True,20
5,13513,2,127,127,1,127,True,0,0,True,True,20
6,13514,1,128,128,1,128,True,0,0,True,True,20
7,13514,2,125,125,1,125,True,0,0,True,True,20
8,13515,1,127,127,1,127,True,0,0,True,True,20
9,13515,2,127,127,1,127,True,0,0,True,True,20



PHASE 3G.2.3 — SEQUENCE EXCEPTIONS


,match_id,innings,delivery_records,sequence_values,sequence_min,sequence_max,integer_valued,duplicate_sequence_positions,missing_sequence_positions,sequence_contiguous,overid_non_decreasing,distinct_overid



AUDIT CONSTRUCTION: COMPLETED


#### Raw-source to integrated-delivery reconciliation

In [156]:
# Phase 3G.2.4 — Raw-source to integrated-delivery reconciliation
# Corrected: match-level population is used only to filter approved matches.

phase3G_source_delivery_key = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num"
]

# 1. Validate delivery keys only on delivery-level datasets
for dataset_name, dataset in [
    ("df_source_ball", df_source_ball),
    ("df_deliveries", df_deliveries)
]:
    missing = [
        col for col in phase3G_source_delivery_key
        if col not in dataset.columns
    ]
    assert not missing, (
        f"{dataset_name} missing delivery-key fields: {missing}"
    )

# 2. Validate match identifier on the approved match-level population
assert "match_id" in phase3_benchmark_population.columns

# 3. Identify approved match IDs
phase3G_approved_ids = set(
    phase3_benchmark_population["match_id"].dropna().unique()
)

assert len(phase3G_approved_ids) == len(
    phase3_benchmark_population
)

# 4. Restrict raw source to approved matches
phase3G_raw_approved = df_source_ball.loc[
    df_source_ball["match_id"].isin(phase3G_approved_ids),
    phase3G_source_delivery_key
].copy()

phase3G_integrated_keys = df_deliveries[
    phase3G_source_delivery_key
].copy()

# 5. Audit key completeness and duplication
phase3G_source_key_quality = pd.DataFrame([
    {
        "dataset": "Raw source — approved matches",
        "rows": len(phase3G_raw_approved),
        "unique_keys": len(
            phase3G_raw_approved.drop_duplicates(
                subset=phase3G_source_delivery_key
            )
        ),
        "duplicate_key_rows": int(
            phase3G_raw_approved.duplicated(
                subset=phase3G_source_delivery_key,
                keep=False
            ).sum()
        ),
        "missing_key_rows": int(
            phase3G_raw_approved[phase3G_source_delivery_key]
            .isna().any(axis=1).sum()
        )
    },
    {
        "dataset": "Integrated deliveries",
        "rows": len(phase3G_integrated_keys),
        "unique_keys": len(
            phase3G_integrated_keys.drop_duplicates(
                subset=phase3G_source_delivery_key
            )
        ),
        "duplicate_key_rows": int(
            phase3G_integrated_keys.duplicated(
                subset=phase3G_source_delivery_key,
                keep=False
            ).sum()
        ),
        "missing_key_rows": int(
            phase3G_integrated_keys[phase3G_source_delivery_key]
            .isna().any(axis=1).sum()
        )
    }
])

# 6. Reconcile unique delivery keys
phase3G_raw_unique_keys = phase3G_raw_approved.drop_duplicates(
    subset=phase3G_source_delivery_key
)

phase3G_integrated_unique_keys = phase3G_integrated_keys.drop_duplicates(
    subset=phase3G_source_delivery_key
)

phase3G_source_delivery_reconciliation = (
    phase3G_raw_unique_keys.merge(
        phase3G_integrated_unique_keys,
        on=phase3G_source_delivery_key,
        how="outer",
        indicator="reconciliation_status",
        validate="one_to_one"
    )
)

phase3G_source_delivery_summary = (
    phase3G_source_delivery_reconciliation[
        "reconciliation_status"
    ]
    .value_counts()
    .rename_axis("reconciliation_status")
    .reset_index(name="unique_delivery_keys")
)

# 7. Preserve exceptions
phase3G_raw_only_delivery_keys = (
    phase3G_source_delivery_reconciliation.loc[
        phase3G_source_delivery_reconciliation[
            "reconciliation_status"
        ] == "left_only"
    ].copy()
)

phase3G_integrated_only_delivery_keys = (
    phase3G_source_delivery_reconciliation.loc[
        phase3G_source_delivery_reconciliation[
            "reconciliation_status"
        ] == "right_only"
    ].copy()
)

# 8. Record record-count differences
phase3G_source_delivery_counts = pd.DataFrame([
    {
        "dataset": "Raw source — all matches",
        "delivery_records": len(df_source_ball)
    },
    {
        "dataset": "Raw source — approved matches",
        "delivery_records": len(phase3G_raw_approved)
    },
    {
        "dataset": "Integrated deliveries",
        "delivery_records": len(df_deliveries)
    }
])

# 9. Audit construction checks
assert phase3G_source_key_quality["rows"].notna().all()
assert phase3G_source_delivery_summary[
    "unique_delivery_keys"
].sum() == len(phase3G_source_delivery_reconciliation)

assert phase3G_source_delivery_reconciliation[
    "reconciliation_status"
].isin(["both", "left_only", "right_only"]).all()

# 10. Export
phase3G_source_key_quality.to_csv(
    OUTPUT_DIR / "phase3G_source_delivery_key_quality.csv",
    index=False
)
phase3G_source_delivery_summary.to_csv(
    OUTPUT_DIR / "phase3G_source_delivery_reconciliation.csv",
    index=False
)
phase3G_raw_only_delivery_keys.to_csv(
    OUTPUT_DIR / "phase3G_raw_only_delivery_keys.csv",
    index=False
)
phase3G_integrated_only_delivery_keys.to_csv(
    OUTPUT_DIR / "phase3G_integrated_only_delivery_keys.csv",
    index=False
)
phase3G_source_delivery_counts.to_csv(
    OUTPUT_DIR / "phase3G_source_delivery_record_counts.csv",
    index=False
)

print("PHASE 3G.2.4 — SOURCE KEY QUALITY")
display(phase3G_source_key_quality)

print("\nSOURCE-TO-INTEGRATED DELIVERY RECONCILIATION")
display(phase3G_source_delivery_summary)

print("\nRAW AND INTEGRATED RECORD COUNTS")
display(phase3G_source_delivery_counts)

print("\nRAW-ONLY DELIVERY KEYS — FIRST 20")
display(phase3G_raw_only_delivery_keys.head(20))

print("\nINTEGRATED-ONLY DELIVERY KEYS — FIRST 20")
display(phase3G_integrated_only_delivery_keys.head(20))

print("\nAUDIT CONSTRUCTION: COMPLETED")

PHASE 3G.2.4 — SOURCE KEY QUALITY


,dataset,rows,unique_keys,duplicate_key_rows,missing_key_rows
0,Raw source — approved matches,192261,192261,0,0
1,Integrated deliveries,192235,192235,0,0



SOURCE-TO-INTEGRATED DELIVERY RECONCILIATION


,reconciliation_status,unique_delivery_keys
0,both,192235
1,left_only,26
2,right_only,0



RAW AND INTEGRATED RECORD COUNTS


,dataset,delivery_records
0,Raw source — all matches,200999
1,Raw source — approved matches,192261
2,Integrated deliveries,192235



RAW-ONLY DELIVERY KEYS — FIRST 20


,match_id,innings,overid,over_ball_num,innings_ball_num,reconciliation_status
105254,34429,3,1,1,1,left_only
105255,34429,3,1,2,2,left_only
105256,34429,3,1,3,3,left_only
105257,34429,3,1,4,4,left_only
105258,34429,4,1,1,1,left_only
127407,40725,3,1,1,1,left_only
127408,40725,3,1,2,2,left_only
127409,40725,3,1,3,3,left_only
127410,40725,3,1,4,4,left_only
127411,40725,3,1,5,5,left_only



INTEGRATED-ONLY DELIVERY KEYS — FIRST 20


,match_id,innings,overid,over_ball_num,innings_ball_num,reconciliation_status



AUDIT CONSTRUCTION: COMPLETED


#### Investigate the 26 unmatched records

In [157]:
# Phase 3G.2.5 — Investigate raw-only delivery records

# 1. Retrieve full original source rows for the 26 unmatched keys
phase3G_raw_only_full = phase3G_raw_only_delivery_keys[
    phase3G_source_delivery_key
].merge(
    df_source_ball,
    on=phase3G_source_delivery_key,
    how="left",
    validate="one_to_one"
)

# 2. Summarise unmatched records by match and innings
phase3G_raw_only_by_innings = (
    phase3G_raw_only_full
    .groupby(["match_id", "innings"], dropna=False)
    .size()
    .rename("raw_only_records")
    .reset_index()
    .sort_values(["match_id", "innings"])
)

phase3G_raw_only_by_match = (
    phase3G_raw_only_full
    .groupby("match_id")
    .agg(
        raw_only_records=("innings", "size"),
        innings_present=("innings", lambda x: sorted(x.dropna().unique().tolist())),
        minimum_innings=("innings", "min"),
        maximum_innings=("innings", "max")
    )
    .reset_index()
)

# 3. Join available match metadata
phase3G_match_metadata_fields = [
    col for col in [
        "match_id", "comp_id", "comp_name",
        "match_format", "season"
    ]
    if col in df_source_matches.columns
]

phase3G_raw_only_match_metadata = (
    phase3G_raw_only_by_match.merge(
        df_source_matches[phase3G_match_metadata_fields].drop_duplicates(
            subset=["match_id"]
        ),
        on="match_id",
        how="left",
        validate="one_to_one",
        indicator=True
    )
)

# 4. Compare unmatched records with primary-innings scope
phase3G_raw_only_scope_audit = pd.DataFrame([
    {
        "measure": "Total unmatched raw-only records",
        "count": len(phase3G_raw_only_full)
    },
    {
        "measure": "Records in innings 1 or 2",
        "count": int(
            phase3G_raw_only_full["innings"].isin([1, 2]).sum()
        )
    },
    {
        "measure": "Records in innings 3 or 4",
        "count": int(
            phase3G_raw_only_full["innings"].isin([3, 4]).sum()
        )
    },
    {
        "measure": "Records outside innings 1 to 4",
        "count": int(
            (~phase3G_raw_only_full["innings"].isin([1, 2, 3, 4])).sum()
        )
    },
    {
        "measure": "Distinct affected matches",
        "count": int(phase3G_raw_only_full["match_id"].nunique())
    },
    {
        "measure": "Affected matches with source metadata",
        "count": int(
            phase3G_raw_only_match_metadata["_merge"].eq("both").sum()
        )
    }
])

# 5. Select available source fields for direct inspection
phase3G_raw_only_detail_fields = [
    col for col in [
        "match_id", "innings", "overid", "over_ball_num",
        "innings_ball_num", "batting_team_id", "bowling_team_id",
        "runs", "is_four", "is_six", "extras_type",
        "extras1_runs", "extras_2_runs", "over_or_around"
    ]
    if col in phase3G_raw_only_full.columns
]

phase3G_raw_only_detail = phase3G_raw_only_full[
    phase3G_raw_only_detail_fields
].copy()

# 6. Validation
assert len(phase3G_raw_only_full) == len(phase3G_raw_only_delivery_keys)
assert len(phase3G_raw_only_by_match) == (
    phase3G_raw_only_full["match_id"].nunique()
)
assert phase3G_raw_only_by_innings["raw_only_records"].sum() == (
    len(phase3G_raw_only_full)
)
assert phase3G_raw_only_match_metadata["match_id"].is_unique

# 7. Export investigation evidence
phase3G_raw_only_by_innings.to_csv(
    OUTPUT_DIR / "phase3G_raw_only_by_innings.csv", index=False
)
phase3G_raw_only_by_match.to_csv(
    OUTPUT_DIR / "phase3G_raw_only_by_match.csv", index=False
)
phase3G_raw_only_match_metadata.to_csv(
    OUTPUT_DIR / "phase3G_raw_only_match_metadata.csv", index=False
)
phase3G_raw_only_scope_audit.to_csv(
    OUTPUT_DIR / "phase3G_raw_only_scope_audit.csv", index=False
)
phase3G_raw_only_detail.to_csv(
    OUTPUT_DIR / "phase3G_raw_only_source_detail.csv", index=False
)

print("PHASE 3G.2.5 — UNMATCHED RECORDS BY MATCH AND INNINGS")
display(phase3G_raw_only_by_innings)

print("\nUNMATCHED RECORDS BY MATCH")
display(phase3G_raw_only_by_match)

print("\nSOURCE MATCH METADATA")
display(phase3G_raw_only_match_metadata)

print("\nSCOPE AUDIT")
display(phase3G_raw_only_scope_audit)

print("\nORIGINAL SOURCE DETAILS")
display(phase3G_raw_only_detail)

print("\nINVESTIGATION CONSTRUCTION: COMPLETED")

PHASE 3G.2.5 — UNMATCHED RECORDS BY MATCH AND INNINGS


,match_id,innings,raw_only_records
0,34429,3,4
1,34429,4,1
2,40725,3,6
3,40725,4,4
4,45512,3,6
5,45512,4,1
6,50359,3,3
7,50359,4,1



UNMATCHED RECORDS BY MATCH


,match_id,raw_only_records,innings_present,minimum_innings,maximum_innings
0,34429,5,"[3, 4]",3,4
1,40725,10,"[3, 4]",3,4
2,45512,7,"[3, 4]",3,4
3,50359,4,"[3, 4]",3,4



SOURCE MATCH METADATA


,match_id,raw_only_records,innings_present,minimum_innings,maximum_innings,comp_id,_merge
0,34429,5,"[3, 4]",3,4,2542,both
1,40725,10,"[3, 4]",3,4,2725,both
2,45512,7,"[3, 4]",3,4,3133,both
3,50359,4,"[3, 4]",3,4,3298,both



SCOPE AUDIT


,measure,count
0,Total unmatched raw-only records,26
1,Records in innings 1 or 2,0
2,Records in innings 3 or 4,26
3,Records outside innings 1 to 4,0
4,Distinct affected matches,4
5,Affected matches with source metadata,4



ORIGINAL SOURCE DETAILS


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,runs,is_four,is_six,extras_type,extras1_runs,extras_2_runs,over_or_around
0,34429,3,1,1,1,27,18,0,False,False,wide,2,0,Over
1,34429,3,1,2,2,27,18,1,False,False,NaN,6,0,Over
2,34429,3,1,3,3,27,18,0,False,False,NaN,6,0,Around
3,34429,3,1,4,4,27,18,0,False,False,NaN,6,0,Over
4,34429,4,1,1,1,18,27,4,True,False,NaN,6,0,Over
5,40725,3,1,1,1,3,7,0,False,False,NaN,6,0,Over
6,40725,3,1,2,2,3,7,4,True,False,NaN,6,0,Over
7,40725,3,1,3,3,3,7,1,False,False,NaN,6,0,Over
8,40725,3,1,4,4,3,7,4,True,False,other,5,0,Around
9,40725,3,1,5,5,3,7,0,False,False,NaN,6,0,Around



INVESTIGATION CONSTRUCTION: COMPLETED


#### Verify match and competition context

In [158]:
# PHASE 3G.2.6 — VERIFY MATCH AND COMPETITION CONTEXT
# Purpose:
# Determine whether the 26 unmatched records can be classified
# using available match metadata and innings/team structure.
# No source DataFrames are modified.

phase3G_affected_match_ids = (
    phase3G_raw_only_by_match["match_id"]
    .drop_duplicates()
    .tolist()
)

# 1. Complete available source match metadata for affected matches
phase3G_match_metadata_columns = [
    col for col in df_source_matches.columns
    if col in [
        "match_id", "comp_id", "comp_name", "match_format",
        "season", "match_date", "start_date", "end_date",
        "venue", "city", "country", "team1_id", "team2_id",
        "team1", "team2", "winner_id", "win_team_id",
        "result", "match_result", "status"
    ]
]

phase3G_affected_match_metadata = (
    df_source_matches.loc[
        df_source_matches["match_id"].isin(phase3G_affected_match_ids),
        phase3G_match_metadata_columns
    ]
    .sort_values("match_id")
    .reset_index(drop=True)
)

print("AFFECTED MATCH METADATA")
display(phase3G_affected_match_metadata)

# 2. Raw-source innings/team structure for affected matches
phase3G_raw_affected = df_source_ball.loc[
    df_source_ball["match_id"].isin(phase3G_affected_match_ids)
].copy()

phase3G_raw_innings_team_structure = (
    phase3G_raw_affected
    .groupby(
        ["match_id", "innings", "batting_team_id", "bowling_team_id"],
        dropna=False
    )
    .agg(
        source_delivery_records=("innings_ball_num", "size"),
        first_sequence=("innings_ball_num", "min"),
        last_sequence=("innings_ball_num", "max"),
        distinct_sequence_positions=("innings_ball_num", "nunique")
    )
    .reset_index()
    .sort_values(["match_id", "innings"])
)

print("\nRAW-SOURCE INNINGS AND TEAM STRUCTURE")
display(phase3G_raw_innings_team_structure)

# 3. Integrated innings/team structure for the same matches
phase3G_integrated_affected = df_deliveries.loc[
    df_deliveries["match_id"].isin(phase3G_affected_match_ids)
].copy()

phase3G_integrated_innings_team_structure = (
    phase3G_integrated_affected
    .groupby(
        ["match_id", "innings", "batting_team_id", "bowling_team_id"],
        dropna=False
    )
    .agg(
        integrated_delivery_records=("innings_ball_num", "size"),
        first_sequence=("innings_ball_num", "min"),
        last_sequence=("innings_ball_num", "max"),
        distinct_sequence_positions=("innings_ball_num", "nunique")
    )
    .reset_index()
    .sort_values(["match_id", "innings"])
)

print("\nINTEGRATED INNINGS AND TEAM STRUCTURE")
display(phase3G_integrated_innings_team_structure)

# 4. Compare source and integrated innings coverage
phase3G_raw_innings_coverage = (
    phase3G_raw_affected.groupby(["match_id", "innings"])
    .size()
    .rename("raw_source_records")
    .reset_index()
)

phase3G_integrated_innings_coverage = (
    phase3G_integrated_affected.groupby(["match_id", "innings"])
    .size()
    .rename("integrated_records")
    .reset_index()
)

phase3G_innings_coverage_comparison = phase3G_raw_innings_coverage.merge(
    phase3G_integrated_innings_coverage,
    on=["match_id", "innings"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

phase3G_innings_coverage_comparison["raw_source_records"] = (
    phase3G_innings_coverage_comparison["raw_source_records"].fillna(0).astype(int)
)
phase3G_innings_coverage_comparison["integrated_records"] = (
    phase3G_innings_coverage_comparison["integrated_records"].fillna(0).astype(int)
)

phase3G_innings_coverage_comparison["record_difference"] = (
    phase3G_innings_coverage_comparison["raw_source_records"]
    - phase3G_innings_coverage_comparison["integrated_records"]
)

phase3G_innings_coverage_comparison = (
    phase3G_innings_coverage_comparison
    .sort_values(["match_id", "innings"])
    .reset_index(drop=True)
)

print("\nRAW VS INTEGRATED INNINGS COVERAGE")
display(phase3G_innings_coverage_comparison)

# 5. Compact investigation summary
phase3G_context_summary = pd.DataFrame([
    {
        "measure": "Affected matches",
        "count": len(phase3G_affected_match_ids)
    },
    {
        "measure": "Raw-source records in affected matches",
        "count": len(phase3G_raw_affected)
    },
    {
        "measure": "Integrated records in affected matches",
        "count": len(phase3G_integrated_affected)
    },
    {
        "measure": "Raw-source innings groups",
        "count": len(phase3G_raw_innings_team_structure)
    },
    {
        "measure": "Integrated innings groups",
        "count": len(phase3G_integrated_innings_team_structure)
    },
    {
        "measure": "Raw-source innings 3/4 records",
        "count": int(
            phase3G_raw_affected["innings"].isin([3, 4]).sum()
        )
    },
    {
        "measure": "Integrated innings 3/4 records",
        "count": int(
            phase3G_integrated_affected["innings"].isin([3, 4]).sum()
        )
    }
])

print("\nCONTEXT SUMMARY")
display(phase3G_context_summary)

# 6. Export audit outputs
phase3G_affected_match_metadata.to_csv(
    "/content/phase3_outputs/phase3G_affected_match_metadata.csv",
    index=False
)
phase3G_raw_innings_team_structure.to_csv(
    "/content/phase3_outputs/phase3G_raw_innings_team_structure.csv",
    index=False
)
phase3G_integrated_innings_team_structure.to_csv(
    "/content/phase3_outputs/phase3G_integrated_innings_team_structure.csv",
    index=False
)
phase3G_innings_coverage_comparison.to_csv(
    "/content/phase3_outputs/phase3G_innings_coverage_comparison.csv",
    index=False
)
phase3G_context_summary.to_csv(
    "/content/phase3_outputs/phase3G_context_summary.csv",
    index=False
)

# 7. Structural validation
assert set(phase3G_affected_match_metadata["match_id"]) == set(
    phase3G_affected_match_ids
)
assert len(phase3G_raw_only_delivery_keys) == 26
assert int(
    phase3G_raw_affected["innings"].isin([1, 2]).sum()
) >= 0
assert (
    phase3G_innings_coverage_comparison["record_difference"].sum()
    == 26
)

print("\nVALIDATION")
print("Affected matches represented:", len(phase3G_affected_match_metadata))
print("Unmatched raw-only records retained:", len(phase3G_raw_only_delivery_keys))
print("Net raw-source minus integrated record difference:",
      int(phase3G_innings_coverage_comparison["record_difference"].sum()))
print("Investigation construction: COMPLETED")

AFFECTED MATCH METADATA


,match_id,comp_id,match_date,win_team_id,result
0,34429,2542,2024-07-30,18.0,India Tied with Srilanka (India Won the Super Over by 2 wickets)
1,40725,2725,2025-04-16,7.0,Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over)
2,45512,3133,2025-09-26,18.0,India won the Super Over(Match Tied)
3,50359,3298,2026-04-26,8.0,Kolkata Knight Riders Won by Super over (Match Tied)



RAW-SOURCE INNINGS AND TEAM STRUCTURE


,match_id,innings,batting_team_id,bowling_team_id,source_delivery_records,first_sequence,last_sequence,distinct_sequence_positions
0,34429,1,18,27,125,1,125,125
1,34429,2,27,18,127,1,127,127
2,34429,3,27,18,4,1,4,4
3,34429,4,18,27,1,1,1,1
4,40725,1,7,3,128,1,128,128
5,40725,2,3,7,124,1,124,124
6,40725,3,3,7,6,1,6,6
7,40725,4,7,3,4,1,4,4
8,45512,1,18,27,129,1,129,129
9,45512,2,27,18,124,1,124,124



INTEGRATED INNINGS AND TEAM STRUCTURE


,match_id,innings,batting_team_id,bowling_team_id,integrated_delivery_records,first_sequence,last_sequence,distinct_sequence_positions
0,34429,1,18,27,125,1,125,125
1,34429,2,27,18,127,1,127,127
2,40725,1,7,3,128,1,128,128
3,40725,2,3,7,124,1,124,124
4,45512,1,18,27,129,1,129,129
5,45512,2,27,18,124,1,124,124
6,50359,1,8,3509,127,1,127,127
7,50359,2,3509,8,125,1,125,125



RAW VS INTEGRATED INNINGS COVERAGE


,match_id,innings,raw_source_records,integrated_records,_merge,record_difference
0,34429,1,125,125,both,0
1,34429,2,127,127,both,0
2,34429,3,4,0,left_only,4
3,34429,4,1,0,left_only,1
4,40725,1,128,128,both,0
5,40725,2,124,124,both,0
6,40725,3,6,0,left_only,6
7,40725,4,4,0,left_only,4
8,45512,1,129,129,both,0
9,45512,2,124,124,both,0



CONTEXT SUMMARY


,measure,count
0,Affected matches,4
1,Raw-source records in affected matches,1035
2,Integrated records in affected matches,1009
3,Raw-source innings groups,16
4,Integrated innings groups,8
5,Raw-source innings 3/4 records,26
6,Integrated innings 3/4 records,0



VALIDATION
Affected matches represented: 4
Unmatched raw-only records retained: 26
Net raw-source minus integrated record difference: 26
Investigation construction: COMPLETED


#### Formalize and export the reconciliation decision

In [160]:
# PHASE 3G.2.7 — FORMAL SUPER OVER EXCEPTION REGISTER
# Corrected: reconstruct source detail from existing raw data and keys.

# 1. Recover the 26 original source rows using the unmatched delivery keys
phase3G_super_over_exceptions = (
    df_source_ball.merge(
        phase3G_raw_only_delivery_keys[
            [
                "match_id", "innings", "overid",
                "over_ball_num", "innings_ball_num"
            ]
        ].drop_duplicates(),
        on=[
            "match_id", "innings", "overid",
            "over_ball_num", "innings_ball_num"
        ],
        how="inner",
        validate="one_to_one"
    )
)

# 2. Attach verified match metadata
phase3G_super_over_metadata = (
    phase3G_affected_match_metadata[
        ["match_id", "comp_id", "match_date", "win_team_id", "result"]
    ]
    .drop_duplicates(subset=["match_id"])
)

phase3G_super_over_exceptions = phase3G_super_over_exceptions.merge(
    phase3G_super_over_metadata,
    on="match_id",
    how="left",
    validate="many_to_one",
    indicator="metadata_merge"
)

# 3. Record classification and treatment
phase3G_super_over_exceptions["reconciliation_classification"] = (
    "EXPECTED_EXCLUSION"
)
phase3G_super_over_exceptions["exclusion_reason"] = (
    "SUPER_OVER_OUT_OF_PRIMARY_SCOPE"
)
phase3G_super_over_exceptions["evidence_basis"] = (
    "Match result explicitly identifies Super Over; raw source contains "
    "innings 3 and 4; integrated dataset retains regulation innings 1 and 2 only."
)
phase3G_super_over_exceptions["analytical_treatment"] = (
    "RETAIN_IN_AUDIT_TRAIL_EXCLUDE_FROM_PRIMARY_REGULATION_POPULATION"
)

phase3G_super_over_exceptions = (
    phase3G_super_over_exceptions
    .sort_values(["match_id", "innings", "innings_ball_num"])
    .reset_index(drop=True)
)

# 4. Match-level summary
phase3G_super_over_match_summary = (
    phase3G_super_over_exceptions
    .groupby(
        ["match_id", "comp_id", "match_date", "result",
         "reconciliation_classification", "exclusion_reason"],
        dropna=False
    )
    .agg(
        excluded_source_records=("innings_ball_num", "size"),
        innings_present=("innings", lambda x: sorted(x.unique().tolist())),
        minimum_innings=("innings", "min"),
        maximum_innings=("innings", "max")
    )
    .reset_index()
    .sort_values("match_id")
)

# 5. Overall reconciliation decision
phase3G_reconciliation_decision = pd.DataFrame([
    {
        "measure": "Raw-only delivery records",
        "count": len(phase3G_super_over_exceptions),
        "decision": "All classified"
    },
    {
        "measure": "Expected Super Over exclusions",
        "count": int(
            phase3G_super_over_exceptions["exclusion_reason"]
            .eq("SUPER_OVER_OUT_OF_PRIMARY_SCOPE").sum()
        ),
        "decision": "Exclude from primary regulation population"
    },
    {
        "measure": "Unclassified raw-only records",
        "count": int(
            phase3G_super_over_exceptions[
                "reconciliation_classification"
            ].isna().sum()
        ),
        "decision": "Must equal zero"
    },
    {
        "measure": "Affected matches",
        "count": phase3G_super_over_exceptions["match_id"].nunique(),
        "decision": "All four match results explicitly mention Super Over"
    },
    {
        "measure": "Raw-only innings outside 3 and 4",
        "count": int(
            (~phase3G_super_over_exceptions["innings"].isin([3, 4])).sum()
        ),
        "decision": "Must equal zero"
    }
])

# 6. Validation
assert len(phase3G_super_over_exceptions) == 26
assert phase3G_super_over_exceptions["match_id"].nunique() == 4
assert phase3G_super_over_exceptions["metadata_merge"].eq("both").all()
assert phase3G_super_over_exceptions["result"].str.contains(
    "Super Over", case=False, na=False
).all()
assert phase3G_super_over_exceptions["innings"].isin([3, 4]).all()
assert phase3G_super_over_exceptions[
    "reconciliation_classification"
].eq("EXPECTED_EXCLUSION").all()
assert phase3G_super_over_exceptions["exclusion_reason"].eq(
    "SUPER_OVER_OUT_OF_PRIMARY_SCOPE"
).all()
assert len(phase3G_integrated_only_delivery_keys) == 0

# 7. Export audit outputs
phase3G_super_over_exceptions.to_csv(
    "/content/phase3_outputs/phase3G_super_over_exception_register.csv",
    index=False
)
phase3G_super_over_match_summary.to_csv(
    "/content/phase3_outputs/phase3G_super_over_match_summary.csv",
    index=False
)
phase3G_reconciliation_decision.to_csv(
    "/content/phase3_outputs/phase3G_reconciliation_decision.csv",
    index=False
)

# 8. Display results
print("SUPER OVER EXCEPTION REGISTER")
display(
    phase3G_super_over_exceptions[
        [
            "match_id", "innings", "innings_ball_num",
            "comp_id", "result",
            "reconciliation_classification",
            "exclusion_reason"
        ]
    ]
)

print("\nMATCH-LEVEL SUMMARY")
display(phase3G_super_over_match_summary)

print("\nRECONCILIATION DECISION")
display(phase3G_reconciliation_decision)

print("\nVALIDATION")
print("Raw-only records classified:", len(phase3G_super_over_exceptions))
print("Affected matches:", phase3G_super_over_exceptions["match_id"].nunique())
print("Unclassified records:",
      phase3G_super_over_exceptions[
          "reconciliation_classification"
      ].isna().sum())
print("Integrated-only records:", len(phase3G_integrated_only_delivery_keys))
print("All Super Over exception checks: PASSED")

SUPER OVER EXCEPTION REGISTER


,match_id,innings,innings_ball_num,comp_id,result,reconciliation_classification,exclusion_reason
0,34429,3,1,2542,India Tied with Srilanka (India Won the Super Over by 2 wickets),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
1,34429,3,2,2542,India Tied with Srilanka (India Won the Super Over by 2 wickets),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
2,34429,3,3,2542,India Tied with Srilanka (India Won the Super Over by 2 wickets),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
3,34429,3,4,2542,India Tied with Srilanka (India Won the Super Over by 2 wickets),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
4,34429,4,1,2542,India Tied with Srilanka (India Won the Super Over by 2 wickets),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
5,40725,3,1,2725,Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
6,40725,3,2,2725,Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
7,40725,3,3,2725,Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
8,40725,3,4,2725,Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE
9,40725,3,5,2725,Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE



MATCH-LEVEL SUMMARY


,match_id,comp_id,match_date,result,reconciliation_classification,exclusion_reason,excluded_source_records,innings_present,minimum_innings,maximum_innings
0,34429,2542,2024-07-30,India Tied with Srilanka (India Won the Super Over by 2 wickets),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE,5,"[3, 4]",3,4
1,40725,2725,2025-04-16,Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE,10,"[3, 4]",3,4
2,45512,3133,2025-09-26,India won the Super Over(Match Tied),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE,7,"[3, 4]",3,4
3,50359,3298,2026-04-26,Kolkata Knight Riders Won by Super over (Match Tied),EXPECTED_EXCLUSION,SUPER_OVER_OUT_OF_PRIMARY_SCOPE,4,"[3, 4]",3,4



RECONCILIATION DECISION


,measure,count,decision
0,Raw-only delivery records,26,All classified
1,Expected Super Over exclusions,26,Exclude from primary regulation population
2,Unclassified raw-only records,0,Must equal zero
3,Affected matches,4,All four match results explicitly mention Super Over
4,Raw-only innings outside 3 and 4,0,Must equal zero



VALIDATION
Raw-only records classified: 26
Affected matches: 4
Unclassified records: 0
Integrated-only records: 0
All Super Over exception checks: PASSED


#### Consolidated source-population robustness closure

In [162]:
# PHASE 3G.2.8 — CONSOLIDATED SOURCE-POPULATION ROBUSTNESS CLOSURE
# Corrected: calculate validation directly from underlying DataFrames.

# 1. Recompute population and delivery counts from source objects
phase3G_source_match_rows = len(df_source_matches)
phase3G_source_unique_matches = df_source_matches["match_id"].nunique()

phase3G_approved_match_rows = len(phase3_benchmark_population)
phase3G_approved_unique_matches = (
    phase3_benchmark_population["match_id"].nunique()
)

phase3G_integrated_delivery_rows = len(df_deliveries)
phase3G_integrated_unique_matches = df_deliveries["match_id"].nunique()

phase3G_raw_approved_delivery_rows = len(
    df_source_ball[
        df_source_ball["match_id"].isin(
            phase3_benchmark_population["match_id"].unique()
        )
    ]
)

# 2. Consolidated audit summary
phase3G_2_closure_summary = pd.DataFrame([
    {
        "audit_area": "Source match population",
        "observed_result": (
            f"{phase3G_source_match_rows} source rows; "
            f"{phase3G_approved_match_rows} approved rows"
        ),
        "exception_count": (
            phase3G_source_match_rows - phase3G_approved_match_rows
        ),
        "exception_treatment": "Source matches outside approved population",
        "status": "RECONCILED"
    },
    {
        "audit_area": "Integrated delivery match population",
        "observed_result": (
            f"{phase3G_integrated_delivery_rows} delivery rows; "
            f"{phase3G_integrated_unique_matches} unique matches"
        ),
        "exception_count": len(phase3G_delivery_population_exceptions),
        "exception_treatment": "No unexpected delivery matches",
        "status": "PASSED"
    },
    {
        "audit_area": "Integrated delivery keys",
        "observed_result": (
            f"{phase3G_integrated_delivery_rows} records; "
            "unique and complete delivery keys"
        ),
        "exception_count": (
            len(phase3G_duplicate_delivery_keys)
            + len(phase3G_missing_delivery_keys)
        ),
        "exception_treatment": "Duplicate and missing keys audited",
        "status": "PASSED"
    },
    {
        "audit_area": "Innings and team structure",
        "observed_result": (
            f"{phase3G_integrated_unique_matches} matches; "
            "two integrated innings per match"
        ),
        "exception_count": (
            len(phase3G_incomplete_innings_matches)
            + len(phase3G_innings_team_exceptions)
            + len(phase3G_team_reversal_exceptions)
        ),
        "exception_treatment": "No innings or team-structure exceptions",
        "status": "PASSED"
    },
    {
        "audit_area": "Delivery sequence continuity",
        "observed_result": (
            f"{len(phase3G_innings_sequence_audit)} innings audited"
        ),
        "exception_count": len(phase3G_sequence_exceptions),
        "exception_treatment": (
            "Sequence positions only; not legal-ball counts"
        ),
        "status": "PASSED_WITH_LIMITATION"
    },
    {
        "audit_area": "Raw-to-integrated delivery reconciliation",
        "observed_result": (
            f"{phase3G_raw_approved_delivery_rows} approved-match "
            f"raw records; {phase3G_integrated_delivery_rows} integrated"
        ),
        "exception_count": len(phase3G_super_over_exceptions),
        "exception_treatment": "All 26 confirmed Super Over exclusions",
        "status": "RECONCILED"
    }
])

# 3. Rebuild validation register without relying on guessed audit columns
phase3G_2_validation_register = pd.DataFrame([
    {
        "check_id": "3G2_V01",
        "check": "Source and approved match IDs are unique",
        "passed": (
            phase3G_source_match_rows == phase3G_source_unique_matches
            and phase3G_approved_match_rows == phase3G_approved_unique_matches
        )
    },
    {
        "check_id": "3G2_V02",
        "check": "Integrated delivery matches belong to approved population",
        "passed": (
            len(phase3G_delivery_population_exceptions) == 0
            and phase3G_integrated_unique_matches
                == phase3G_approved_unique_matches
        )
    },
    {
        "check_id": "3G2_V03",
        "check": "Integrated delivery keys are unique and complete",
        "passed": (
            len(phase3G_duplicate_delivery_keys) == 0
            and len(phase3G_missing_delivery_keys) == 0
        )
    },
    {
        "check_id": "3G2_V04",
        "check": "All integrated matches have exactly two innings",
        "passed": len(phase3G_incomplete_innings_matches) == 0
    },
    {
        "check_id": "3G2_V05",
        "check": "No innings team identity or reversal exceptions",
        "passed": (
            len(phase3G_innings_team_exceptions) == 0
            and len(phase3G_team_reversal_exceptions) == 0
        )
    },
    {
        "check_id": "3G2_V06",
        "check": "No delivery sequence anomalies",
        "passed": len(phase3G_sequence_exceptions) == 0
    },
    {
        "check_id": "3G2_V07",
        "check": "All raw-only records are classified as Super Over exclusions",
        "passed": (
            len(phase3G_raw_only_delivery_keys) == 26
            and len(phase3G_super_over_exceptions) == 26
            and phase3G_super_over_exceptions[
                "exclusion_reason"
            ].eq("SUPER_OVER_OUT_OF_PRIMARY_SCOPE").all()
        )
    },
    {
        "check_id": "3G2_V08",
        "check": "No integrated-only delivery keys",
        "passed": len(phase3G_integrated_only_delivery_keys) == 0
    },
    {
        "check_id": "3G2_V09",
        "check": "Raw-to-integrated record difference reconciles",
        "passed": (
            phase3G_raw_approved_delivery_rows
            - phase3G_integrated_delivery_rows
            == len(phase3G_super_over_exceptions)
        )
    }
])

phase3G_2_validation_register["status"] = (
    phase3G_2_validation_register["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# 4. Closure metrics
phase3G_2_closure_metrics = pd.DataFrame([
    {"metric": "Validation checks", "value": len(phase3G_2_validation_register)},
    {"metric": "Passed checks", "value": int(phase3G_2_validation_register["passed"].sum())},
    {"metric": "Failed checks", "value": int((~phase3G_2_validation_register["passed"]).sum())},
    {"metric": "Source matches", "value": phase3G_source_match_rows},
    {"metric": "Approved matches", "value": phase3G_approved_match_rows},
    {"metric": "Integrated delivery records", "value": phase3G_integrated_delivery_rows},
    {"metric": "Raw approved-match delivery records", "value": phase3G_raw_approved_delivery_rows},
    {"metric": "Confirmed Super Over exclusions", "value": len(phase3G_super_over_exceptions)},
    {"metric": "Unexplained raw-only regulation records", "value": 0}
])

# 5. Closure decision
phase3G_2_status = (
    "COMPLETE"
    if phase3G_2_validation_register["passed"].all()
    else "NOT_CLOSED_VALIDATION_FAILURE"
)

phase3G_2_closure_decision = pd.DataFrame([{
    "subphase": "Phase 3G.2 — Source population and structural robustness",
    "status": phase3G_2_status,
    "validation_checks": len(phase3G_2_validation_register),
    "passed": int(phase3G_2_validation_register["passed"].sum()),
    "failed": int((~phase3G_2_validation_register["passed"]).sum()),
    "super_over_exclusions": len(phase3G_super_over_exceptions),
    "unexplained_raw_only_records": 0,
    "scope_note": (
        "Structural and source-population reconciliation only. "
        "Does not validate scoring semantics, legal-ball progression, "
        "or numerical pressure measures."
    )
}])

# 6. Export
phase3G_2_closure_summary.to_csv(
    "/content/phase3_outputs/phase3G_2_closure_summary.csv",
    index=False
)
phase3G_2_validation_register.to_csv(
    "/content/phase3_outputs/phase3G_2_validation_register.csv",
    index=False
)
phase3G_2_closure_metrics.to_csv(
    "/content/phase3_outputs/phase3G_2_closure_metrics.csv",
    index=False
)
phase3G_2_closure_decision.to_csv(
    "/content/phase3_outputs/phase3G_2_closure_decision.csv",
    index=False
)

# 7. Display
print("PHASE 3G.2 CONSOLIDATED AUDIT SUMMARY")
display(phase3G_2_closure_summary)

print("\nVALIDATION REGISTER")
display(phase3G_2_validation_register)

print("\nCLOSURE METRICS")
display(phase3G_2_closure_metrics)

print("\nCLOSURE DECISION")
display(phase3G_2_closure_decision)

assert phase3G_2_validation_register["passed"].all()
assert phase3G_2_status == "COMPLETE"

print("\nPHASE 3G.2 CLOSURE VALIDATION: PASSED")

PHASE 3G.2 CONSOLIDATED AUDIT SUMMARY


,audit_area,observed_result,exception_count,exception_treatment,status
0,Source match population,883 source rows; 813 approved rows,70,Source matches outside approved population,RECONCILED
1,Integrated delivery match population,192235 delivery rows; 813 unique matches,0,No unexpected delivery matches,PASSED
2,Integrated delivery keys,192235 records; unique and complete delivery keys,0,Duplicate and missing keys audited,PASSED
3,Innings and team structure,813 matches; two integrated innings per match,0,No innings or team-structure exceptions,PASSED
4,Delivery sequence continuity,1626 innings audited,0,Sequence positions only; not legal-ball counts,PASSED_WITH_LIMITATION
5,Raw-to-integrated delivery reconciliation,192261 approved-match raw records; 192235 integrated,26,All 26 confirmed Super Over exclusions,RECONCILED



VALIDATION REGISTER


,check_id,check,passed,status
0,3G2_V01,Source and approved match IDs are unique,True,PASSED
1,3G2_V02,Integrated delivery matches belong to approved population,True,PASSED
2,3G2_V03,Integrated delivery keys are unique and complete,True,PASSED
3,3G2_V04,All integrated matches have exactly two innings,True,PASSED
4,3G2_V05,No innings team identity or reversal exceptions,True,PASSED
5,3G2_V06,No delivery sequence anomalies,True,PASSED
6,3G2_V07,All raw-only records are classified as Super Over exclusions,True,PASSED
7,3G2_V08,No integrated-only delivery keys,True,PASSED
8,3G2_V09,Raw-to-integrated record difference reconciles,True,PASSED



CLOSURE METRICS


,metric,value
0,Validation checks,9
1,Passed checks,9
2,Failed checks,0
3,Source matches,883
4,Approved matches,813
5,Integrated delivery records,192235
6,Raw approved-match delivery records,192261
7,Confirmed Super Over exclusions,26
8,Unexplained raw-only regulation records,0



CLOSURE DECISION


,subphase,status,validation_checks,passed,failed,super_over_exclusions,unexplained_raw_only_records,scope_note
0,Phase 3G.2 — Source population and structural robustness,COMPLETE,9,9,0,26,0,"Structural and source-population reconciliation only. Does not validate scoring semantics, legal-ball progression, or numerical pressure measures."



PHASE 3G.2 CLOSURE VALIDATION: PASSED


### State alignment audit

#### Reconstructed-state alignment audit

In [163]:
# PHASE 3G.3.1 — PRE-DELIVERY STATE ALIGNMENT AUDIT

phase3G_state_delivery_key = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num"
]

# 1. Check key completeness and uniqueness in both datasets
phase3G_delivery_key_profile = pd.DataFrame([
    {
        "dataset": "Integrated deliveries",
        "rows": len(df_deliveries),
        "unique_keys": len(
            df_deliveries[phase3G_state_delivery_key].drop_duplicates()
        ),
        "missing_key_rows": int(
            df_deliveries[phase3G_state_delivery_key].isna().any(axis=1).sum()
        )
    },
    {
        "dataset": "Pre-delivery wicket states",
        "rows": len(phase3_pre_delivery_wicket_states),
        "unique_keys": len(
            phase3_pre_delivery_wicket_states[
                phase3G_state_delivery_key
            ].drop_duplicates()
        ),
        "missing_key_rows": int(
            phase3_pre_delivery_wicket_states[
                phase3G_state_delivery_key
            ].isna().any(axis=1).sum()
        )
    }
])

phase3G_delivery_key_profile["duplicate_key_rows"] = (
    phase3G_delivery_key_profile["rows"]
    - phase3G_delivery_key_profile["unique_keys"]
)

print("DELIVERY AND STATE KEY PROFILE")
display(phase3G_delivery_key_profile)

# 2. Full outer reconciliation
phase3G_state_delivery_reconciliation = (
    df_deliveries[phase3G_state_delivery_key]
    .merge(
        phase3_pre_delivery_wicket_states[
            phase3G_state_delivery_key
        ],
        on=phase3G_state_delivery_key,
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)

phase3G_state_alignment_summary = (
    phase3G_state_delivery_reconciliation["_merge"]
    .value_counts()
    .reindex(["both", "left_only", "right_only"], fill_value=0)
    .rename_axis("alignment_status")
    .reset_index(name="record_count")
)

print("\nDELIVERY-TO-STATE RECONCILIATION")
display(phase3G_state_alignment_summary)

# 3. Isolate exceptions
phase3G_state_alignment_exceptions = (
    phase3G_state_delivery_reconciliation.loc[
        phase3G_state_delivery_reconciliation["_merge"] != "both"
    ].copy()
)

print("\nALIGNMENT EXCEPTIONS")
display(phase3G_state_alignment_exceptions.head(20))

# 4. Compare match and innings coverage
phase3G_state_match_coverage = pd.DataFrame([
    {
        "dataset": "Integrated deliveries",
        "matches": df_deliveries["match_id"].nunique(),
        "innings": df_deliveries[
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    },
    {
        "dataset": "Pre-delivery wicket states",
        "matches": phase3_pre_delivery_wicket_states["match_id"].nunique(),
        "innings": phase3_pre_delivery_wicket_states[
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    }
])

print("\nMATCH AND INNINGS COVERAGE")
display(phase3G_state_match_coverage)

# 5. Export
phase3G_delivery_key_profile.to_csv(
    "/content/phase3_outputs/phase3G_state_delivery_key_profile.csv",
    index=False
)
phase3G_state_alignment_summary.to_csv(
    "/content/phase3_outputs/phase3G_state_alignment_summary.csv",
    index=False
)
phase3G_state_alignment_exceptions.to_csv(
    "/content/phase3_outputs/phase3G_state_alignment_exceptions.csv",
    index=False
)
phase3G_state_match_coverage.to_csv(
    "/content/phase3_outputs/phase3G_state_match_coverage.csv",
    index=False
)

# 6. Validation
assert phase3G_delivery_key_profile["missing_key_rows"].eq(0).all()
assert phase3G_delivery_key_profile["duplicate_key_rows"].eq(0).all()
assert phase3G_state_alignment_summary.loc[
    phase3G_state_alignment_summary["alignment_status"] != "both",
    "record_count"
].sum() == 0

print("\nVALIDATION")
print("Integrated delivery rows:", len(df_deliveries))
print("Pre-delivery state rows:", len(phase3_pre_delivery_wicket_states))
print("Unmatched state/delivery records:",
      len(phase3G_state_alignment_exceptions))
print("Alignment audit construction: COMPLETED")

DELIVERY AND STATE KEY PROFILE


,dataset,rows,unique_keys,missing_key_rows,duplicate_key_rows
0,Integrated deliveries,192235,192235,0,0
1,Pre-delivery wicket states,192235,192235,0,0



DELIVERY-TO-STATE RECONCILIATION


,alignment_status,record_count
0,both,192235
1,left_only,0
2,right_only,0



ALIGNMENT EXCEPTIONS


,match_id,innings,overid,over_ball_num,innings_ball_num,_merge



MATCH AND INNINGS COVERAGE


,dataset,matches,innings
0,Integrated deliveries,813,1626
1,Pre-delivery wicket states,813,1626



VALIDATION
Integrated delivery rows: 192235
Pre-delivery state rows: 192235
Unmatched state/delivery records: 0
Alignment audit construction: COMPLETED


#### Reconstructed wicket-state continuity

In [164]:
# PHASE 3G.3.2 — RECONSTRUCTED WICKET-STATE CONTINUITY

phase3G_state_key = [
    "match_id", "innings", "overid",
    "over_ball_num", "innings_ball_num"
]

# 1. Build an independent audit table from integrated delivery outcomes
phase3G_wicket_continuity = df_deliveries[
    phase3G_state_key + ["is_wicket"]
].merge(
    phase3_pre_delivery_wicket_states[
        phase3G_state_key + ["pre_delivery_wickets_fallen"]
    ],
    on=phase3G_state_key,
    how="inner",
    validate="one_to_one"
)

phase3G_wicket_continuity["wicket_event_numeric"] = pd.to_numeric(
    phase3G_wicket_continuity["is_wicket"],
    errors="coerce"
)

# 2. Validate the recorded wicket-event indicator
phase3G_wicket_event_quality = pd.DataFrame([
    {
        "measure": "Missing or nonnumeric wicket-event values",
        "count": int(
            phase3G_wicket_continuity["wicket_event_numeric"].isna().sum()
        )
    },
    {
        "measure": "Wicket-event values outside 0 and 1",
        "count": int(
            (~phase3G_wicket_continuity["wicket_event_numeric"].isin([0, 1])).sum()
        )
    }
])

# 3. Sort deliveries and calculate expected pre-delivery wickets
phase3G_wicket_continuity = (
    phase3G_wicket_continuity
    .sort_values(phase3G_state_key)
    .reset_index(drop=True)
)

phase3G_wicket_continuity["expected_pre_delivery_wickets"] = (
    phase3G_wicket_continuity
    .groupby(["match_id", "innings"])["wicket_event_numeric"]
    .transform(lambda s: s.cumsum().shift(1).fillna(0))
)

phase3G_wicket_continuity["state_difference"] = (
    phase3G_wicket_continuity["pre_delivery_wickets_fallen"]
    - phase3G_wicket_continuity["expected_pre_delivery_wickets"]
)

# 4. Identify continuity exceptions
phase3G_wicket_continuity_exceptions = (
    phase3G_wicket_continuity.loc[
        phase3G_wicket_continuity["state_difference"] != 0
    ].copy()
)

# 5. Check innings-opening wicket states
phase3G_innings_opening_states = (
    phase3G_wicket_continuity
    .groupby(["match_id", "innings"], as_index=False)
    .first()
)

phase3G_opening_state_exceptions = (
    phase3G_innings_opening_states.loc[
        phase3G_innings_opening_states["pre_delivery_wickets_fallen"] != 0
    ].copy()
)

# 6. Check reconstructed state range and missingness
phase3G_state_range_summary = pd.DataFrame([
    {
        "measure": "Missing pre-delivery wicket states",
        "count": int(
            phase3G_wicket_continuity[
                "pre_delivery_wickets_fallen"
            ].isna().sum()
        )
    },
    {
        "measure": "States below zero",
        "count": int(
            (phase3G_wicket_continuity[
                "pre_delivery_wickets_fallen"
            ] < 0).sum()
        )
    },
    {
        "measure": "States above ten",
        "count": int(
            (phase3G_wicket_continuity[
                "pre_delivery_wickets_fallen"
            ] > 10).sum()
        )
    }
])

# 7. Overall continuity summary
phase3G_wicket_continuity_summary = pd.DataFrame([
    {
        "measure": "Aligned delivery-state records",
        "count": len(phase3G_wicket_continuity)
    },
    {
        "measure": "Distinct innings audited",
        "count": phase3G_wicket_continuity[
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    },
    {
        "measure": "Continuity mismatches",
        "count": len(phase3G_wicket_continuity_exceptions)
    },
    {
        "measure": "Opening-state exceptions",
        "count": len(phase3G_opening_state_exceptions)
    },
    {
        "measure": "Missing or invalid wicket-event values",
        "count": int(
            phase3G_wicket_event_quality["count"].sum()
        )
    },
    {
        "measure": "Out-of-range or missing reconstructed states",
        "count": int(phase3G_state_range_summary["count"].sum())
    }
])

# 8. Export audit outputs
phase3G_wicket_continuity_summary.to_csv(
    "/content/phase3_outputs/phase3G_wicket_continuity_summary.csv",
    index=False
)

phase3G_wicket_continuity_exceptions.to_csv(
    "/content/phase3_outputs/phase3G_wicket_continuity_exceptions.csv",
    index=False
)

phase3G_opening_state_exceptions.to_csv(
    "/content/phase3_outputs/phase3G_opening_state_exceptions.csv",
    index=False
)

phase3G_wicket_event_quality.to_csv(
    "/content/phase3_outputs/phase3G_wicket_event_quality.csv",
    index=False
)

phase3G_state_range_summary.to_csv(
    "/content/phase3_outputs/phase3G_state_range_summary.csv",
    index=False
)

# 9. Display
print("WICKET EVENT QUALITY")
display(phase3G_wicket_event_quality)

print("\nWICKET-STATE CONTINUITY SUMMARY")
display(phase3G_wicket_continuity_summary)

print("\nOPENING-STATE EXCEPTIONS")
display(phase3G_opening_state_exceptions.head(20))

print("\nCONTINUITY EXCEPTIONS")
display(phase3G_wicket_continuity_exceptions.head(20))

print("\nSTATE RANGE SUMMARY")
display(phase3G_state_range_summary)

# 10. Assertions for structural validity
assert phase3G_wicket_event_quality["count"].sum() == 0
assert len(phase3G_wicket_continuity_exceptions) == 0
assert len(phase3G_opening_state_exceptions) == 0
assert phase3G_state_range_summary["count"].sum() == 0

print("\nPHASE 3G.3.2 AUDIT CONSTRUCTION: COMPLETED")

WICKET EVENT QUALITY


,measure,count
0,Missing or nonnumeric wicket-event values,0
1,Wicket-event values outside 0 and 1,0



WICKET-STATE CONTINUITY SUMMARY


,measure,count
0,Aligned delivery-state records,192235
1,Distinct innings audited,1626
2,Continuity mismatches,0
3,Opening-state exceptions,0
4,Missing or invalid wicket-event values,0
5,Out-of-range or missing reconstructed states,0



OPENING-STATE EXCEPTIONS


,match_id,innings,overid,over_ball_num,innings_ball_num,is_wicket,pre_delivery_wickets_fallen,wicket_event_numeric,expected_pre_delivery_wickets,state_difference



CONTINUITY EXCEPTIONS


,match_id,innings,overid,over_ball_num,innings_ball_num,is_wicket,pre_delivery_wickets_fallen,wicket_event_numeric,expected_pre_delivery_wickets,state_difference



STATE RANGE SUMMARY


,measure,count
0,Missing pre-delivery wicket states,0
1,States below zero,0
2,States above ten,0



PHASE 3G.3.2 AUDIT CONSTRUCTION: COMPLETED


#### Second-innings state consistency

In [165]:
# PHASE 3G.3.3 — SECOND-INNINGS STATE CONSISTENCY

phase3G_second_innings_key = [
    "match_id", "innings", "overid",
    "over_ball_num", "innings_ball_num"
]

# 1. Confirm required columns exist before proceeding
phase3G_required_state_columns = (
    phase3G_second_innings_key + ["pre_delivery_wickets_fallen"]
)

phase3G_missing_columns = {
    "full_state_dataset": [
        c for c in phase3G_required_state_columns
        if c not in phase3_pre_delivery_wicket_states.columns
    ],
    "second_innings_dataset": [
        c for c in phase3G_required_state_columns
        if c not in phase3E_second_innings_wicket_alignment.columns
    ]
}

print("REQUIRED COLUMN CHECK")
display(pd.DataFrame([
    {"dataset": dataset, "missing_columns": missing}
    for dataset, missing in phase3G_missing_columns.items()
]))

assert all(len(v) == 0 for v in phase3G_missing_columns.values()), (
    "Required columns are missing. Inspect the displayed schema before continuing."
)

# 2. Extract second-innings records from the full state dataset
phase3G_full_second_innings_states = (
    phase3_pre_delivery_wicket_states.loc[
        phase3_pre_delivery_wicket_states["innings"] == 2,
        phase3G_required_state_columns
    ].copy()
)

phase3G_phase3E_second_innings_states = (
    phase3E_second_innings_wicket_alignment[
        phase3G_required_state_columns
    ].copy()
)

# 3. Key uniqueness and completeness
phase3G_second_innings_key_profile = pd.DataFrame([
    {
        "dataset": "Full reconstructed states, innings 2",
        "rows": len(phase3G_full_second_innings_states),
        "unique_keys": len(
            phase3G_full_second_innings_states[
                phase3G_second_innings_key
            ].drop_duplicates()
        ),
        "missing_key_rows": int(
            phase3G_full_second_innings_states[
                phase3G_second_innings_key
            ].isna().any(axis=1).sum()
        )
    },
    {
        "dataset": "Phase 3E second-innings alignment",
        "rows": len(phase3G_phase3E_second_innings_states),
        "unique_keys": len(
            phase3G_phase3E_second_innings_states[
                phase3G_second_innings_key
            ].drop_duplicates()
        ),
        "missing_key_rows": int(
            phase3G_phase3E_second_innings_states[
                phase3G_second_innings_key
            ].isna().any(axis=1).sum()
        )
    }
])

phase3G_second_innings_key_profile["duplicate_key_rows"] = (
    phase3G_second_innings_key_profile["rows"]
    - phase3G_second_innings_key_profile["unique_keys"]
)

print("\nSECOND-INNINGS KEY PROFILE")
display(phase3G_second_innings_key_profile)

# 4. Compare keys and reconstructed wicket states
phase3G_second_innings_state_comparison = (
    phase3G_full_second_innings_states.merge(
        phase3G_phase3E_second_innings_states,
        on=phase3G_second_innings_key,
        how="outer",
        suffixes=("_full", "_phase3E"),
        validate="one_to_one",
        indicator=True
    )
)

phase3G_second_innings_state_comparison["state_match"] = (
    phase3G_second_innings_state_comparison[
        "pre_delivery_wickets_fallen_full"
    ].eq(
        phase3G_second_innings_state_comparison[
            "pre_delivery_wickets_fallen_phase3E"
        ]
    )
)

phase3G_second_innings_alignment_summary = (
    phase3G_second_innings_state_comparison["_merge"]
    .value_counts()
    .reindex(["both", "left_only", "right_only"], fill_value=0)
    .rename_axis("alignment_status")
    .reset_index(name="record_count")
)

phase3G_second_innings_state_mismatches = (
    phase3G_second_innings_state_comparison.loc[
        (phase3G_second_innings_state_comparison["_merge"] == "both")
        & (~phase3G_second_innings_state_comparison["state_match"])
    ].copy()
)

# 5. Match and innings coverage
phase3G_second_innings_coverage = pd.DataFrame([
    {
        "dataset": "Full reconstructed states, innings 2",
        "matches": phase3G_full_second_innings_states["match_id"].nunique(),
        "innings_groups": phase3G_full_second_innings_states[
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    },
    {
        "dataset": "Phase 3E second-innings alignment",
        "matches": phase3G_phase3E_second_innings_states["match_id"].nunique(),
        "innings_groups": phase3G_phase3E_second_innings_states[
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    }
])

# 6. Summary and exports
phase3G_second_innings_state_summary = pd.DataFrame([
    {
        "measure": "Full reconstructed second-innings records",
        "count": len(phase3G_full_second_innings_states)
    },
    {
        "measure": "Phase 3E second-innings records",
        "count": len(phase3G_phase3E_second_innings_states)
    },
    {
        "measure": "Matched delivery keys",
        "count": int(
            (phase3G_second_innings_state_comparison["_merge"] == "both").sum()
        )
    },
    {
        "measure": "Unmatched keys",
        "count": int(
            (phase3G_second_innings_state_comparison["_merge"] != "both").sum()
        )
    },
    {
        "measure": "Wicket-state value mismatches",
        "count": len(phase3G_second_innings_state_mismatches)
    }
])

phase3G_second_innings_key_profile.to_csv(
    "/content/phase3_outputs/phase3G_second_innings_key_profile.csv",
    index=False
)
phase3G_second_innings_alignment_summary.to_csv(
    "/content/phase3_outputs/phase3G_second_innings_alignment_summary.csv",
    index=False
)
phase3G_second_innings_state_mismatches.to_csv(
    "/content/phase3_outputs/phase3G_second_innings_state_mismatches.csv",
    index=False
)
phase3G_second_innings_coverage.to_csv(
    "/content/phase3_outputs/phase3G_second_innings_coverage.csv",
    index=False
)
phase3G_second_innings_state_summary.to_csv(
    "/content/phase3_outputs/phase3G_second_innings_state_summary.csv",
    index=False
)

print("\nSECOND-INNINGS ALIGNMENT SUMMARY")
display(phase3G_second_innings_alignment_summary)

print("\nSECOND-INNINGS COVERAGE")
display(phase3G_second_innings_coverage)

print("\nSTATE COMPARISON SUMMARY")
display(phase3G_second_innings_state_summary)

print("\nSTATE MISMATCH EXCEPTIONS")
display(phase3G_second_innings_state_mismatches.head(20))

# 7. Validation
assert phase3G_second_innings_key_profile["missing_key_rows"].eq(0).all()
assert phase3G_second_innings_key_profile["duplicate_key_rows"].eq(0).all()
assert (
    phase3G_second_innings_alignment_summary.loc[
        phase3G_second_innings_alignment_summary["alignment_status"] != "both",
        "record_count"
    ].sum() == 0
)
assert len(phase3G_second_innings_state_mismatches) == 0
assert phase3G_second_innings_coverage["matches"].eq(813).all()
assert phase3G_second_innings_coverage["innings_groups"].eq(813).all()

print("\nPHASE 3G.3.3 AUDIT CONSTRUCTION: COMPLETED")

REQUIRED COLUMN CHECK


,dataset,missing_columns
0,full_state_dataset,[]
1,second_innings_dataset,[]



SECOND-INNINGS KEY PROFILE


,dataset,rows,unique_keys,missing_key_rows,duplicate_key_rows
0,"Full reconstructed states, innings 2",91902,91902,0,0
1,Phase 3E second-innings alignment,91902,91902,0,0



SECOND-INNINGS ALIGNMENT SUMMARY


,alignment_status,record_count
0,both,91902
1,left_only,0
2,right_only,0


... (3 outputs left out)

,measure,count
0,Full reconstructed second-innings records,91902
1,Phase 3E second-innings records,91902
2,Matched delivery keys,91902
3,Unmatched keys,0
4,Wicket-state value mismatches,0



STATE MISMATCH EXCEPTIONS


,match_id,innings,overid,over_ball_num,innings_ball_num,pre_delivery_wickets_fallen_full,pre_delivery_wickets_fallen_phase3E,_merge,state_match



PHASE 3G.3.3 AUDIT CONSTRUCTION: COMPLETED


#### Audit second-innings wickets-remaining derivation

In [166]:
# PHASE 3G.3.4 — INSPECT SECOND-INNINGS WICKET FIELDS

phase3G_wicket_related_columns = [
    col for col in phase3E_second_innings_wicket_alignment.columns
    if any(
        term in col.lower()
        for term in ["wicket", "dismiss", "out"]
    )
]

phase3G_wicket_field_inventory = pd.DataFrame([
    {
        "column": col,
        "dtype": str(
            phase3E_second_innings_wicket_alignment[col].dtype
        ),
        "missing_values": int(
            phase3E_second_innings_wicket_alignment[col].isna().sum()
        ),
        "distinct_values": int(
            phase3E_second_innings_wicket_alignment[col].nunique(
                dropna=True
            )
        ),
        "sample_values": (
            phase3E_second_innings_wicket_alignment[col]
            .dropna()
            .unique()[:12]
            .tolist()
        )
    }
    for col in phase3G_wicket_related_columns
])

print("PHASE 3E WICKET-RELATED FIELD INVENTORY")
display(phase3G_wicket_field_inventory)

print("\nPHASE 3E DATASET DIMENSIONS")
print(phase3E_second_innings_wicket_alignment.shape)

print("\nWICKET-REMAINING DISTRIBUTION OBJECT")
display(phase3E_wickets_remaining_distribution.head(20))

print("\nMATCH CONSISTENCY OBJECT")
display(phase3E_wicket_match_consistency.head(10))

PHASE 3E WICKET-RELATED FIELD INVENTORY


,column,dtype,missing_values,distinct_values,sample_values
0,is_wicket,bool,0,2,"[False, True]"
1,dismissal_type_id,float64,86841,8,"[5.0, 3.0, 4.0, 2.0, 1.0, 6.0, 9.0, 10.0]"
2,dismissal_type,object,86841,8,"[Stump, LBW, RunOut, Caught, Bowled, Hit Wicket, Retire Out, Obstracting the field]"
3,dismissed_player_id,float64,86836,685,"[4369.0, 3639.0, 3125.0, 494.0, 778.0, 2633.0, 4840.0, 3272.0, 1187.0, 1210.0, 1596.0, 4391.0]"
4,dismissal_type_original,object,86841,8,"[Stump, LBW, RunOut, Caught, Bowled, Hit Wicket, Retire Out, Obstracting the field]"
5,dismissal_type_normalized,object,86841,8,"[Stump, LBW, RunOut, Caught, Bowled, Hit Wicket, Retire Out, Obstructing the field]"
6,knock_out,int64,0,2,"[0, 1]"
7,pre_delivery_wickets_fallen,int64,0,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
8,_phase3E_wicket_merge,category,0,1,[both]
9,wickets_remaining_conditional,int64,0,10,"[10, 9, 8, 7, 6, 5, 4, 3, 2, 1]"



PHASE 3E DATASET DIMENSIONS
(91902, 64)

WICKET-REMAINING DISTRIBUTION OBJECT


,match_format,wickets_remaining_conditional,delivery_records,matches
0,T20,1,492,75
1,T20,2,862,111
2,T20,3,1362,154
3,T20,4,2207,196
4,T20,5,3264,241
5,T20,6,4470,285
6,T20,7,6188,323
7,T20,8,7212,351
8,T20,9,7808,360
9,T20,10,8213,362



MATCH CONSISTENCY OBJECT


,match_id,distinct_formats,distinct_batting_teams,distinct_bowling_teams
0,13511,1,1,1
1,13512,1,1,1
2,13513,1,1,1
3,13514,1,1,1
4,13515,1,1,1
5,13522,1,1,1
6,13523,1,1,1
7,13524,1,1,1
8,13612,1,1,1
9,13613,1,1,1


#### Wickets-remaining derivation audit

In [167]:
# PHASE 3G.3.4 — WICKETS-REMAINING DERIVATION AUDIT

phase3G_wickets_remaining_audit = (
    phase3E_second_innings_wicket_alignment[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "pre_delivery_wickets_fallen",
            "wickets_remaining_conditional",
            "match_format"
        ]
    ].copy()
)

# 1. Independently calculate expected wickets remaining
phase3G_wickets_remaining_audit["expected_wickets_remaining"] = (
    10 - phase3G_wickets_remaining_audit["pre_delivery_wickets_fallen"]
)

# 2. Compare stored and independently calculated values
phase3G_wickets_remaining_audit["derivation_match"] = (
    phase3G_wickets_remaining_audit[
        "wickets_remaining_conditional"
    ].eq(
        phase3G_wickets_remaining_audit[
            "expected_wickets_remaining"
        ]
    )
)

phase3G_wickets_remaining_mismatches = (
    phase3G_wickets_remaining_audit.loc[
        ~phase3G_wickets_remaining_audit["derivation_match"]
    ].copy()
)

# 3. Check missingness and valid ranges
phase3G_wickets_remaining_quality = pd.DataFrame([
    {
        "measure": "Missing pre-delivery wickets fallen",
        "count": int(
            phase3G_wickets_remaining_audit[
                "pre_delivery_wickets_fallen"
            ].isna().sum()
        )
    },
    {
        "measure": "Missing conditional wickets remaining",
        "count": int(
            phase3G_wickets_remaining_audit[
                "wickets_remaining_conditional"
            ].isna().sum()
        )
    },
    {
        "measure": "Pre-delivery wickets below zero",
        "count": int(
            (phase3G_wickets_remaining_audit[
                "pre_delivery_wickets_fallen"
            ] < 0).sum()
        )
    },
    {
        "measure": "Pre-delivery wickets above ten",
        "count": int(
            (phase3G_wickets_remaining_audit[
                "pre_delivery_wickets_fallen"
            ] > 10).sum()
        )
    },
    {
        "measure": "Conditional wickets remaining below zero",
        "count": int(
            (phase3G_wickets_remaining_audit[
                "wickets_remaining_conditional"
            ] < 0).sum()
        )
    },
    {
        "measure": "Conditional wickets remaining above ten",
        "count": int(
            (phase3G_wickets_remaining_audit[
                "wickets_remaining_conditional"
            ] > 10).sum()
        )
    }
])

# 4. Rebuild format-by-wickets-remaining distribution
phase3G_rebuilt_wickets_remaining_distribution = (
    phase3G_wickets_remaining_audit
    .groupby(
        ["match_format", "wickets_remaining_conditional"],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(["match_format", "wickets_remaining_conditional"])
)

# 5. Compare with the existing Phase 3E distribution
phase3G_distribution_comparison = (
    phase3E_wickets_remaining_distribution.merge(
        phase3G_rebuilt_wickets_remaining_distribution,
        on=["match_format", "wickets_remaining_conditional"],
        how="outer",
        suffixes=("_phase3E", "_recomputed"),
        validate="one_to_one",
        indicator=True
    )
)

for metric in ["delivery_records", "matches"]:
    phase3G_distribution_comparison[f"{metric}_difference"] = (
        phase3G_distribution_comparison[f"{metric}_phase3E"]
        - phase3G_distribution_comparison[f"{metric}_recomputed"]
    )

# 6. Summarize
phase3G_wickets_remaining_summary = pd.DataFrame([
    {
        "measure": "Second-innings records audited",
        "count": len(phase3G_wickets_remaining_audit)
    },
    {
        "measure": "Arithmetic derivation mismatches",
        "count": len(phase3G_wickets_remaining_mismatches)
    },
    {
        "measure": "Distribution comparison mismatches",
        "count": int(
            (
                (phase3G_distribution_comparison["_merge"] != "both")
                | (phase3G_distribution_comparison["delivery_records_difference"] != 0)
                | (phase3G_distribution_comparison["matches_difference"] != 0)
            ).sum()
        )
    },
    {
        "measure": "Distinct matches",
        "count": phase3G_wickets_remaining_audit["match_id"].nunique()
    }
])

# 7. Export
phase3G_wickets_remaining_quality.to_csv(
    "/content/phase3_outputs/phase3G_wickets_remaining_quality.csv",
    index=False
)

phase3G_wickets_remaining_mismatches.to_csv(
    "/content/phase3_outputs/phase3G_wickets_remaining_mismatches.csv",
    index=False
)

phase3G_rebuilt_wickets_remaining_distribution.to_csv(
    "/content/phase3_outputs/phase3G_rebuilt_wickets_remaining_distribution.csv",
    index=False
)

phase3G_distribution_comparison.to_csv(
    "/content/phase3_outputs/phase3G_wickets_remaining_distribution_comparison.csv",
    index=False
)

phase3G_wickets_remaining_summary.to_csv(
    "/content/phase3_outputs/phase3G_wickets_remaining_summary.csv",
    index=False
)

# 8. Display
print("WICKETS-REMAINING DATA QUALITY")
display(phase3G_wickets_remaining_quality)

print("\nDERIVATION SUMMARY")
display(phase3G_wickets_remaining_summary)

print("\nDERIVATION MISMATCHES")
display(phase3G_wickets_remaining_mismatches.head(20))

print("\nREBUILT DISTRIBUTION")
display(phase3G_rebuilt_wickets_remaining_distribution)

print("\nPHASE 3E VS RECOMPUTED DISTRIBUTION")
display(phase3G_distribution_comparison)

# 9. Validation
assert phase3G_wickets_remaining_quality["count"].sum() == 0
assert len(phase3G_wickets_remaining_mismatches) == 0
assert (
    phase3G_distribution_comparison["_merge"].eq("both").all()
)
assert phase3G_distribution_comparison[
    "delivery_records_difference"
].eq(0).all()
assert phase3G_distribution_comparison[
    "matches_difference"
].eq(0).all()

print("\nPHASE 3G.3.4 AUDIT CONSTRUCTION: COMPLETED")

WICKETS-REMAINING DATA QUALITY


,measure,count
0,Missing pre-delivery wickets fallen,0
1,Missing conditional wickets remaining,0
2,Pre-delivery wickets below zero,0
3,Pre-delivery wickets above ten,0
4,Conditional wickets remaining below zero,0
5,Conditional wickets remaining above ten,0



DERIVATION SUMMARY


,measure,count
0,Second-innings records audited,91902
1,Arithmetic derivation mismatches,0
2,Distribution comparison mismatches,0
3,Distinct matches,813



DERIVATION MISMATCHES


,match_id,innings,overid,over_ball_num,innings_ball_num,pre_delivery_wickets_fallen,wickets_remaining_conditional,match_format,expected_wickets_remaining,derivation_match



REBUILT DISTRIBUTION


,match_format,wickets_remaining_conditional,delivery_records,matches
0,T20,1,492,75
1,T20,2,862,111
2,T20,3,1362,154
3,T20,4,2207,196
4,T20,5,3264,241
5,T20,6,4470,285
6,T20,7,6188,323
7,T20,8,7212,351
8,T20,9,7808,360
9,T20,10,8213,362



PHASE 3E VS RECOMPUTED DISTRIBUTION


,match_format,wickets_remaining_conditional,delivery_records_phase3E,matches_phase3E,delivery_records_recomputed,matches_recomputed,_merge,delivery_records_difference,matches_difference
0,T20,1,492,75,492,75,both,0,0
1,T20,2,862,111,862,111,both,0,0
2,T20,3,1362,154,1362,154,both,0,0
3,T20,4,2207,196,2207,196,both,0,0
4,T20,5,3264,241,3264,241,both,0,0
5,T20,6,4470,285,4470,285,both,0,0
6,T20,7,6188,323,6188,323,both,0,0
7,T20,8,7212,351,7212,351,both,0,0
8,T20,9,7808,360,7808,360,both,0,0
9,T20,10,8213,362,8213,362,both,0,0



PHASE 3G.3.4 AUDIT CONSTRUCTION: COMPLETED


#### First-innings state consistency

In [168]:
# PHASE 3G.3.5 — FIRST-INNINGS STATE CONSISTENCY

phase3G_first_innings_key = [
    "match_id", "innings", "overid",
    "over_ball_num", "innings_ball_num"
]

phase3G_first_innings_columns = (
    phase3G_first_innings_key + ["pre_delivery_wickets_fallen"]
)

# 1. Verify required columns
phase3G_first_missing_columns = {
    "full_state_dataset": [
        c for c in phase3G_first_innings_columns
        if c not in phase3_pre_delivery_wicket_states.columns
    ],
    "phase3D_first_innings_states": [
        c for c in phase3G_first_innings_columns
        if c not in phase3D_first_innings_states.columns
    ]
}

print("REQUIRED COLUMN CHECK")
display(pd.DataFrame([
    {"dataset": name, "missing_columns": missing}
    for name, missing in phase3G_first_missing_columns.items()
]))

assert all(len(v) == 0 for v in phase3G_first_missing_columns.values())

# 2. Extract first innings from the full reconstructed-state dataset
phase3G_full_first_innings_states = (
    phase3_pre_delivery_wicket_states.loc[
        phase3_pre_delivery_wicket_states["innings"] == 1,
        phase3G_first_innings_columns
    ].copy()
)

phase3G_phase3D_first_innings_states = (
    phase3D_first_innings_states[
        phase3G_first_innings_columns
    ].copy()
)

# 3. Key profile
phase3G_first_innings_key_profile = pd.DataFrame([
    {
        "dataset": "Full reconstructed states, innings 1",
        "rows": len(phase3G_full_first_innings_states),
        "unique_keys": len(
            phase3G_full_first_innings_states[
                phase3G_first_innings_key
            ].drop_duplicates()
        ),
        "missing_key_rows": int(
            phase3G_full_first_innings_states[
                phase3G_first_innings_key
            ].isna().any(axis=1).sum()
        )
    },
    {
        "dataset": "Phase 3D first-innings states",
        "rows": len(phase3G_phase3D_first_innings_states),
        "unique_keys": len(
            phase3G_phase3D_first_innings_states[
                phase3G_first_innings_key
            ].drop_duplicates()
        ),
        "missing_key_rows": int(
            phase3G_phase3D_first_innings_states[
                phase3G_first_innings_key
            ].isna().any(axis=1).sum()
        )
    }
])

phase3G_first_innings_key_profile["duplicate_key_rows"] = (
    phase3G_first_innings_key_profile["rows"]
    - phase3G_first_innings_key_profile["unique_keys"]
)

print("\nFIRST-INNINGS KEY PROFILE")
display(phase3G_first_innings_key_profile)

# 4. Full key and state comparison
phase3G_first_innings_state_comparison = (
    phase3G_full_first_innings_states.merge(
        phase3G_phase3D_first_innings_states,
        on=phase3G_first_innings_key,
        how="outer",
        suffixes=("_full", "_phase3D"),
        validate="one_to_one",
        indicator=True
    )
)

phase3G_first_innings_state_comparison["state_match"] = (
    phase3G_first_innings_state_comparison[
        "pre_delivery_wickets_fallen_full"
    ].eq(
        phase3G_first_innings_state_comparison[
            "pre_delivery_wickets_fallen_phase3D"
        ]
    )
)

phase3G_first_innings_state_mismatches = (
    phase3G_first_innings_state_comparison.loc[
        (phase3G_first_innings_state_comparison["_merge"] == "both")
        & (~phase3G_first_innings_state_comparison["state_match"])
    ].copy()
)

phase3G_first_innings_alignment_summary = (
    phase3G_first_innings_state_comparison["_merge"]
    .value_counts()
    .reindex(["both", "left_only", "right_only"], fill_value=0)
    .rename_axis("alignment_status")
    .reset_index(name="record_count")
)

# 5. Match coverage
phase3G_first_innings_coverage = pd.DataFrame([
    {
        "dataset": "Full reconstructed states, innings 1",
        "matches": phase3G_full_first_innings_states["match_id"].nunique(),
        "innings_groups": phase3G_full_first_innings_states[
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    },
    {
        "dataset": "Phase 3D first-innings states",
        "matches": phase3G_phase3D_first_innings_states["match_id"].nunique(),
        "innings_groups": phase3G_phase3D_first_innings_states[
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    }
])

# 6. Summary
phase3G_first_innings_state_summary = pd.DataFrame([
    {
        "measure": "Full reconstructed first-innings records",
        "count": len(phase3G_full_first_innings_states)
    },
    {
        "measure": "Phase 3D first-innings records",
        "count": len(phase3G_phase3D_first_innings_states)
    },
    {
        "measure": "Matched delivery keys",
        "count": int(
            (phase3G_first_innings_state_comparison["_merge"] == "both").sum()
        )
    },
    {
        "measure": "Unmatched keys",
        "count": int(
            (phase3G_first_innings_state_comparison["_merge"] != "both").sum()
        )
    },
    {
        "measure": "Wicket-state value mismatches",
        "count": len(phase3G_first_innings_state_mismatches)
    }
])

# 7. Export
phase3G_first_innings_key_profile.to_csv(
    "/content/phase3_outputs/phase3G_first_innings_key_profile.csv",
    index=False
)
phase3G_first_innings_alignment_summary.to_csv(
    "/content/phase3_outputs/phase3G_first_innings_alignment_summary.csv",
    index=False
)
phase3G_first_innings_state_mismatches.to_csv(
    "/content/phase3_outputs/phase3G_first_innings_state_mismatches.csv",
    index=False
)
phase3G_first_innings_coverage.to_csv(
    "/content/phase3_outputs/phase3G_first_innings_coverage.csv",
    index=False
)
phase3G_first_innings_state_summary.to_csv(
    "/content/phase3_outputs/phase3G_first_innings_state_summary.csv",
    index=False
)

# 8. Display
print("\nFIRST-INNINGS ALIGNMENT SUMMARY")
display(phase3G_first_innings_alignment_summary)

print("\nFIRST-INNINGS COVERAGE")
display(phase3G_first_innings_coverage)

print("\nFIRST-INNINGS STATE SUMMARY")
display(phase3G_first_innings_state_summary)

print("\nFIRST-INNINGS STATE MISMATCHES")
display(phase3G_first_innings_state_mismatches.head(20))

# 9. Validation
assert phase3G_first_innings_key_profile["missing_key_rows"].eq(0).all()
assert phase3G_first_innings_key_profile["duplicate_key_rows"].eq(0).all()
assert phase3G_first_innings_alignment_summary.loc[
    phase3G_first_innings_alignment_summary["alignment_status"] != "both",
    "record_count"
].sum() == 0
assert len(phase3G_first_innings_state_mismatches) == 0
assert phase3G_first_innings_coverage["matches"].eq(813).all()
assert phase3G_first_innings_coverage["innings_groups"].eq(813).all()

print("\nPHASE 3G.3.5 AUDIT CONSTRUCTION: COMPLETED")

REQUIRED COLUMN CHECK


,dataset,missing_columns
0,full_state_dataset,[]
1,phase3D_first_innings_states,[]



FIRST-INNINGS KEY PROFILE


,dataset,rows,unique_keys,missing_key_rows,duplicate_key_rows
0,"Full reconstructed states, innings 1",100333,100333,0,0
1,Phase 3D first-innings states,100333,100333,0,0



FIRST-INNINGS ALIGNMENT SUMMARY


,alignment_status,record_count
0,both,100333
1,left_only,0
2,right_only,0


... (3 outputs left out)

,measure,count
0,Full reconstructed first-innings records,100333
1,Phase 3D first-innings records,100333
2,Matched delivery keys,100333
3,Unmatched keys,0
4,Wicket-state value mismatches,0



FIRST-INNINGS STATE MISMATCHES


,match_id,innings,overid,over_ball_num,innings_ball_num,pre_delivery_wickets_fallen_full,pre_delivery_wickets_fallen_phase3D,_merge,state_match



PHASE 3G.3.5 AUDIT CONSTRUCTION: COMPLETED


#### Competition and team-context consistency

In [169]:
# PHASE 3G.3.6 — COMPETITION AND TEAM-CONTEXT CONSISTENCY

phase3G_context_key = [
    "match_id", "innings", "overid",
    "over_ball_num", "innings_ball_num"
]

phase3G_context_fields = [
    "comp_id",
    "match_format",
    "batting_team_id",
    "bowling_team_id"
]

# 1. Verify required columns
phase3G_context_datasets = {
    "integrated_deliveries": df_deliveries,
    "first_innings_states": phase3D_first_innings_states,
    "second_innings_states": phase3E_second_innings_wicket_alignment
}

phase3G_context_missing_columns = pd.DataFrame([
    {
        "dataset": name,
        "missing_columns": [
            col for col in phase3G_context_key + phase3G_context_fields
            if col not in dataset.columns
        ]
    }
    for name, dataset in phase3G_context_datasets.items()
])

print("REQUIRED COLUMN CHECK")
display(phase3G_context_missing_columns)

assert phase3G_context_missing_columns["missing_columns"].map(len).eq(0).all()

# 2. Compare each innings-specific population with integrated deliveries
def phase3G_compare_context(reference, comparison, innings_number, label):
    reference_subset = reference.loc[
        reference["innings"] == innings_number,
        phase3G_context_key + phase3G_context_fields
    ].copy()

    comparison_subset = comparison.loc[
        comparison["innings"] == innings_number,
        phase3G_context_key + phase3G_context_fields
    ].copy()

    merged = reference_subset.merge(
        comparison_subset,
        on=phase3G_context_key,
        how="outer",
        suffixes=("_integrated", "_phase"),
        validate="one_to_one",
        indicator=True
    )

    for field in phase3G_context_fields:
        merged[f"{field}_match"] = (
            merged[f"{field}_integrated"]
            .eq(merged[f"{field}_phase"])
            | (
                merged[f"{field}_integrated"].isna()
                & merged[f"{field}_phase"].isna()
            )
        )

    merged["comparison_dataset"] = label
    merged["innings_audited"] = innings_number

    return merged

phase3G_first_context_comparison = phase3G_compare_context(
    df_deliveries,
    phase3D_first_innings_states,
    1,
    "Phase 3D first-innings states"
)

phase3G_second_context_comparison = phase3G_compare_context(
    df_deliveries,
    phase3E_second_innings_wicket_alignment,
    2,
    "Phase 3E second-innings states"
)

phase3G_context_comparison = pd.concat(
    [
        phase3G_first_context_comparison,
        phase3G_second_context_comparison
    ],
    ignore_index=True
)

# 3. Summarize key alignment and context mismatches
phase3G_context_comparison_summary = pd.DataFrame([
    {
        "comparison": label,
        "records_compared": len(group),
        "matched_keys": int(group["_merge"].eq("both").sum()),
        "unmatched_keys": int(group["_merge"].ne("both").sum()),
        "comp_id_mismatches": int((~group["comp_id_match"]).sum()),
        "format_mismatches": int((~group["match_format_match"]).sum()),
        "batting_team_mismatches": int((~group["batting_team_id_match"]).sum()),
        "bowling_team_mismatches": int((~group["bowling_team_id_match"]).sum())
    }
    for label, group in phase3G_context_comparison.groupby(
        "comparison_dataset", sort=False
    )
])

# 4. Isolate all key or context exceptions
phase3G_context_exception_mask = (
    phase3G_context_comparison["_merge"].ne("both")
    | ~phase3G_context_comparison["comp_id_match"]
    | ~phase3G_context_comparison["match_format_match"]
    | ~phase3G_context_comparison["batting_team_id_match"]
    | ~phase3G_context_comparison["bowling_team_id_match"]
)

phase3G_context_exceptions = (
    phase3G_context_comparison.loc[
        phase3G_context_exception_mask
    ].copy()
)

# 5. Match coverage
phase3G_context_coverage = pd.DataFrame([
    {
        "dataset": "Integrated deliveries, innings 1",
        "matches": df_deliveries.loc[
            df_deliveries["innings"] == 1, "match_id"
        ].nunique()
    },
    {
        "dataset": "Phase 3D first-innings states",
        "matches": phase3D_first_innings_states["match_id"].nunique()
    },
    {
        "dataset": "Integrated deliveries, innings 2",
        "matches": df_deliveries.loc[
            df_deliveries["innings"] == 2, "match_id"
        ].nunique()
    },
    {
        "dataset": "Phase 3E second-innings states",
        "matches": phase3E_second_innings_wicket_alignment[
            "match_id"
        ].nunique()
    }
])

# 6. Export
phase3G_context_comparison_summary.to_csv(
    "/content/phase3_outputs/phase3G_context_comparison_summary.csv",
    index=False
)

phase3G_context_exceptions.to_csv(
    "/content/phase3_outputs/phase3G_context_exceptions.csv",
    index=False
)

phase3G_context_coverage.to_csv(
    "/content/phase3_outputs/phase3G_context_coverage.csv",
    index=False
)

# 7. Display
print("\nCONTEXT COMPARISON SUMMARY")
display(phase3G_context_comparison_summary)

print("\nCONTEXT COVERAGE")
display(phase3G_context_coverage)

print("\nCONTEXT EXCEPTIONS")
display(phase3G_context_exceptions.head(20))

# 8. Validation
assert phase3G_context_comparison_summary["unmatched_keys"].eq(0).all()
assert phase3G_context_comparison_summary["comp_id_mismatches"].eq(0).all()
assert phase3G_context_comparison_summary["format_mismatches"].eq(0).all()
assert phase3G_context_comparison_summary["batting_team_mismatches"].eq(0).all()
assert phase3G_context_comparison_summary["bowling_team_mismatches"].eq(0).all()
assert len(phase3G_context_exceptions) == 0
assert phase3G_context_coverage["matches"].eq(813).all()

print("\nPHASE 3G.3.6 AUDIT CONSTRUCTION: COMPLETED")

REQUIRED COLUMN CHECK


,dataset,missing_columns
0,integrated_deliveries,[]
1,first_innings_states,[]
2,second_innings_states,[]



CONTEXT COMPARISON SUMMARY


,comparison,records_compared,matched_keys,unmatched_keys,comp_id_mismatches,format_mismatches,batting_team_mismatches,bowling_team_mismatches
0,Phase 3D first-innings states,100333,100333,0,0,0,0,0
1,Phase 3E second-innings states,91902,91902,0,0,0,0,0



CONTEXT COVERAGE


,dataset,matches
0,"Integrated deliveries, innings 1",813
1,Phase 3D first-innings states,813
2,"Integrated deliveries, innings 2",813
3,Phase 3E second-innings states,813



CONTEXT EXCEPTIONS


,match_id,innings,overid,over_ball_num,innings_ball_num,comp_id_integrated,match_format_integrated,batting_team_id_integrated,bowling_team_id_integrated,comp_id_phase,match_format_phase,batting_team_id_phase,bowling_team_id_phase,_merge,comp_id_match,match_format_match,batting_team_id_match,bowling_team_id_match,comparison_dataset,innings_audited



PHASE 3G.3.6 AUDIT CONSTRUCTION: COMPLETED


#### Consolidated delivery and reconstructed-state closure

In [170]:
# PHASE 3G.3.7 — DELIVERY ALIGNMENT AND RECONSTRUCTED-STATE CLOSURE

# 1. Consolidated findings
phase3G_3_closure_summary = pd.DataFrame([
    {
        "audit_area": "Full delivery-to-state alignment",
        "records_audited": 192235,
        "exceptions": len(phase3G_state_alignment_exceptions),
        "finding": "Exact one-to-one key alignment",
        "status": "PASSED"
    },
    {
        "audit_area": "Pre-delivery wicket-state continuity",
        "records_audited": len(phase3G_wicket_continuity),
        "exceptions": (
            len(phase3G_wicket_continuity_exceptions)
            + len(phase3G_opening_state_exceptions)
        ),
        "finding": "State sequence consistent with recorded wicket events",
        "status": "PASSED_WITH_PROVENANCE_LIMITATION"
    },
    {
        "audit_area": "Second-innings Phase 3E consistency",
        "records_audited": len(phase3G_second_innings_state_comparison),
        "exceptions": (
            int(
                (phase3G_second_innings_state_comparison["_merge"] != "both").sum()
            )
            + len(phase3G_second_innings_state_mismatches)
        ),
        "finding": "Exact key and wicket-state value agreement",
        "status": "PASSED"
    },
    {
        "audit_area": "Conditional wickets-remaining derivation",
        "records_audited": len(phase3G_wickets_remaining_audit),
        "exceptions": (
            len(phase3G_wickets_remaining_mismatches)
            + int(phase3G_distribution_comparison[
                "delivery_records_difference"
            ].ne(0).sum())
            + int(phase3G_distribution_comparison[
                "matches_difference"
            ].ne(0).sum())
        ),
        "finding": "Arithmetic and distribution fully reconciled",
        "status": "PASSED_WITH_PROVENANCE_LIMITATION"
    },
    {
        "audit_area": "First-innings Phase 3D consistency",
        "records_audited": len(phase3G_first_innings_state_comparison),
        "exceptions": (
            int(
                (phase3G_first_innings_state_comparison["_merge"] != "both").sum()
            )
            + len(phase3G_first_innings_state_mismatches)
        ),
        "finding": "Exact key and wicket-state value agreement",
        "status": "PASSED"
    },
    {
        "audit_area": "Competition and team context",
        "records_audited": len(phase3G_context_comparison),
        "exceptions": len(phase3G_context_exceptions),
        "finding": "Competition, format, and team identifiers agree",
        "status": "PASSED"
    }
])

# 2. Explicit limitation register
phase3G_3_limitation_register = pd.DataFrame([
    {
        "limitation": "Pre-delivery wickets are reconstructed",
        "classification": "CONDITIONAL",
        "implication": (
            "Continuity checks confirm consistency with recorded is_wicket "
            "events, not independent dismissal-event truth."
        ),
        "treatment": "Retain conditional status"
    },
    {
        "limitation": "Wickets remaining is derived",
        "classification": "CONDITIONAL",
        "implication": (
            "The 10 minus wickets-fallen arithmetic is exact, but its input "
            "inherits the reconstructed wicket-state limitation."
        ),
        "treatment": "Retain wickets_remaining_conditional naming"
    },
    {
        "limitation": "Scoring semantics remain unresolved",
        "classification": "BLOCKED",
        "implication": (
            "This subphase does not validate cumulative score or run totals."
        ),
        "treatment": "No scoring reconstruction"
    },
    {
        "limitation": "Legal-ball progression remains unresolved",
        "classification": "BLOCKED",
        "implication": (
            "Delivery sequence continuity does not establish legal balls "
            "remaining."
        ),
        "treatment": "No balls-remaining reconstruction"
    }
])

# 3. Validation register
phase3G_3_validation_register = pd.DataFrame([
    {
        "check_id": "3G3_V01",
        "check": "Full delivery-state alignment has no unmatched keys",
        "passed": len(phase3G_state_alignment_exceptions) == 0
    },
    {
        "check_id": "3G3_V02",
        "check": "Wicket-state continuity and opening checks have no exceptions",
        "passed": (
            len(phase3G_wicket_continuity_exceptions) == 0
            and len(phase3G_opening_state_exceptions) == 0
        )
    },
    {
        "check_id": "3G3_V03",
        "check": "Second-innings Phase 3E alignment and values agree",
        "passed": (
            (
                phase3G_second_innings_state_comparison["_merge"] == "both"
            ).all()
            and len(phase3G_second_innings_state_mismatches) == 0
        )
    },
    {
        "check_id": "3G3_V04",
        "check": "Wickets-remaining arithmetic and distributions reconcile",
        "passed": (
            len(phase3G_wickets_remaining_mismatches) == 0
            and phase3G_distribution_comparison["_merge"].eq("both").all()
            and phase3G_distribution_comparison[
                "delivery_records_difference"
            ].eq(0).all()
            and phase3G_distribution_comparison[
                "matches_difference"
            ].eq(0).all()
        )
    },
    {
        "check_id": "3G3_V05",
        "check": "First-innings Phase 3D alignment and values agree",
        "passed": (
            (
                phase3G_first_innings_state_comparison["_merge"] == "both"
            ).all()
            and len(phase3G_first_innings_state_mismatches) == 0
        )
    },
    {
        "check_id": "3G3_V06",
        "check": "Competition and team context agrees",
        "passed": len(phase3G_context_exceptions) == 0
    }
])

phase3G_3_validation_register["status"] = (
    phase3G_3_validation_register["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# 4. Closure decision
phase3G_3_status = (
    "COMPLETE"
    if phase3G_3_validation_register["passed"].all()
    else "NOT_CLOSED_VALIDATION_FAILURE"
)

phase3G_3_closure_decision = pd.DataFrame([{
    "subphase": "Phase 3G.3 — Delivery alignment and reconstructed-state robustness",
    "status": phase3G_3_status,
    "validation_checks": len(phase3G_3_validation_register),
    "passed": int(phase3G_3_validation_register["passed"].sum()),
    "failed": int((~phase3G_3_validation_register["passed"]).sum()),
    "records_aligned": 192235,
    "first_innings_records": 100333,
    "second_innings_records": 91902,
    "wicket_state_provenance": "CONDITIONAL",
    "scope_note": (
        "Internal structural and derivation consistency only; "
        "not independent verification of dismissal truth, scoring, "
        "legal-ball progression, or pressure-scale validity."
    )
}])

# 5. Export
phase3G_3_closure_summary.to_csv(
    "/content/phase3_outputs/phase3G_3_closure_summary.csv",
    index=False
)
phase3G_3_limitation_register.to_csv(
    "/content/phase3_outputs/phase3G_3_limitation_register.csv",
    index=False
)
phase3G_3_validation_register.to_csv(
    "/content/phase3_outputs/phase3G_3_validation_register.csv",
    index=False
)
phase3G_3_closure_decision.to_csv(
    "/content/phase3_outputs/phase3G_3_closure_decision.csv",
    index=False
)

# 6. Display
print("PHASE 3G.3 CONSOLIDATED CLOSURE SUMMARY")
display(phase3G_3_closure_summary)

print("\nLIMITATION REGISTER")
display(phase3G_3_limitation_register)

print("\nVALIDATION REGISTER")
display(phase3G_3_validation_register)

print("\nCLOSURE DECISION")
display(phase3G_3_closure_decision)

# 7. Final assertions
assert phase3G_3_validation_register["passed"].all()
assert phase3G_3_status == "COMPLETE"
assert len(phase3G_3_limitation_register) == 4

print("\nPHASE 3G.3 CLOSURE VALIDATION: PASSED")

PHASE 3G.3 CONSOLIDATED CLOSURE SUMMARY


,audit_area,records_audited,exceptions,finding,status
0,Full delivery-to-state alignment,192235,0,Exact one-to-one key alignment,PASSED
1,Pre-delivery wicket-state continuity,192235,0,State sequence consistent with recorded wicket events,PASSED_WITH_PROVENANCE_LIMITATION
2,Second-innings Phase 3E consistency,91902,0,Exact key and wicket-state value agreement,PASSED
3,Conditional wickets-remaining derivation,91902,0,Arithmetic and distribution fully reconciled,PASSED_WITH_PROVENANCE_LIMITATION
4,First-innings Phase 3D consistency,100333,0,Exact key and wicket-state value agreement,PASSED
5,Competition and team context,192235,0,"Competition, format, and team identifiers agree",PASSED



LIMITATION REGISTER


,limitation,classification,implication,treatment
0,Pre-delivery wickets are reconstructed,CONDITIONAL,"Continuity checks confirm consistency with recorded is_wicket events, not independent dismissal-event truth.",Retain conditional status
1,Wickets remaining is derived,CONDITIONAL,"The 10 minus wickets-fallen arithmetic is exact, but its input inherits the reconstructed wicket-state limitation.",Retain wickets_remaining_conditional naming
2,Scoring semantics remain unresolved,BLOCKED,This subphase does not validate cumulative score or run totals.,No scoring reconstruction
3,Legal-ball progression remains unresolved,BLOCKED,Delivery sequence continuity does not establish legal balls remaining.,No balls-remaining reconstruction



VALIDATION REGISTER


,check_id,check,passed,status
0,3G3_V01,Full delivery-state alignment has no unmatched keys,True,PASSED
1,3G3_V02,Wicket-state continuity and opening checks have no exceptions,True,PASSED
2,3G3_V03,Second-innings Phase 3E alignment and values agree,True,PASSED
3,3G3_V04,Wickets-remaining arithmetic and distributions reconcile,True,PASSED
4,3G3_V05,First-innings Phase 3D alignment and values agree,True,PASSED
5,3G3_V06,Competition and team context agrees,True,PASSED



CLOSURE DECISION


,subphase,status,validation_checks,passed,failed,records_aligned,first_innings_records,second_innings_records,wicket_state_provenance,scope_note
0,Phase 3G.3 — Delivery alignment and reconstructed-state robustness,COMPLETE,6,6,0,192235,100333,91902,CONDITIONAL,"Internal structural and derivation consistency only; not independent verification of dismissal truth, scoring, legal-ball progression, or pressure-scale validity."



PHASE 3G.3 CLOSURE VALIDATION: PASSED


### Competition and coverage robustness

#### Competition separation and population coverage

In [171]:
# PHASE 3G.4.1 — COMPETITION SEPARATION AND POPULATION COVERAGE

# 1. Select only the fields needed for this audit
phase3G_4_first = phase3D_first_innings_population[
    ["match_id", "comp_id", "match_format", "pre_delivery_wickets_fallen"]
].copy()

phase3G_4_second = phase3E_second_innings_wicket_alignment[
    ["match_id", "comp_id", "match_format", "wickets_remaining_conditional"]
].copy()

phase3G_4_first["innings_population"] = "First innings"
phase3G_4_second["innings_population"] = "Second innings"

phase3G_4_first = phase3G_4_first.rename(
    columns={"pre_delivery_wickets_fallen": "wicket_state"}
)
phase3G_4_second = phase3G_4_second.rename(
    columns={"wickets_remaining_conditional": "wicket_state"}
)

phase3G_4_combined = pd.concat(
    [
        phase3G_4_first,
        phase3G_4_second
    ],
    ignore_index=True
)

# 2. Competition-to-format consistency
phase3G_4_competition_format = (
    phase3G_4_combined.groupby("comp_id", dropna=False)
    .agg(
        formats=("match_format", "nunique"),
        format_values=("match_format", lambda x: tuple(sorted(x.dropna().unique()))),
        matches=("match_id", "nunique"),
        delivery_records=("match_id", "size")
    )
    .reset_index()
)

phase3G_4_competition_format_exceptions = (
    phase3G_4_competition_format.loc[
        phase3G_4_competition_format["formats"] != 1
    ].copy()
)

# 3. Match-to-format consistency
phase3G_4_match_format = (
    phase3G_4_combined.groupby("match_id", dropna=False)
    .agg(
        formats=("match_format", "nunique"),
        competitions=("comp_id", "nunique"),
        innings_populations=("innings_population", "nunique")
    )
    .reset_index()
)

phase3G_4_match_format_exceptions = (
    phase3G_4_match_format.loc[
        (phase3G_4_match_format["formats"] != 1)
        | (phase3G_4_match_format["competitions"] != 1)
        | (phase3G_4_match_format["innings_populations"] != 2)
    ].copy()
)

# 4. Format and innings coverage
phase3G_4_format_coverage = (
    phase3G_4_combined.groupby(
        ["match_format", "innings_population"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique"),
        delivery_records=("match_id", "size"),
        wicket_states_observed=("wicket_state", "nunique"),
        minimum_wicket_state=("wicket_state", "min"),
        maximum_wicket_state=("wicket_state", "max")
    )
    .reset_index()
)

# 5. Match-level format coverage
phase3G_4_match_coverage = (
    phase3G_4_combined[
        ["match_id", "match_format", "comp_id", "innings_population"]
    ]
    .drop_duplicates()
    .groupby(
        ["match_format", "innings_population"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
)

# 6. Competition counts by format
phase3G_4_competition_counts = (
    phase3G_4_competition_format.groupby("format_values")
    .agg(
        competition_groups=("comp_id", "nunique"),
        matches=("matches", "sum"),
        delivery_records=("delivery_records", "sum")
    )
    .reset_index()
)

# 7. Export audit tables
phase3G_4_competition_format.to_csv(
    "/content/phase3_outputs/phase3G_4_competition_format_audit.csv",
    index=False
)

phase3G_4_competition_format_exceptions.to_csv(
    "/content/phase3_outputs/phase3G_4_competition_format_exceptions.csv",
    index=False
)

phase3G_4_match_format_exceptions.to_csv(
    "/content/phase3_outputs/phase3G_4_match_format_exceptions.csv",
    index=False
)

phase3G_4_format_coverage.to_csv(
    "/content/phase3_outputs/phase3G_4_format_coverage.csv",
    index=False
)

phase3G_4_match_coverage.to_csv(
    "/content/phase3_outputs/phase3G_4_match_coverage.csv",
    index=False
)

# 8. Display results
print("COMPETITION-TO-FORMAT AUDIT")
display(phase3G_4_competition_format.head(20))

print("\nCOMPETITION-TO-FORMAT EXCEPTIONS")
display(phase3G_4_competition_format_exceptions)

print("\nMATCH-TO-FORMAT EXCEPTIONS")
display(phase3G_4_match_format_exceptions)

print("\nFORMAT AND INNINGS COVERAGE")
display(phase3G_4_format_coverage)

print("\nMATCH-LEVEL FORMAT COVERAGE")
display(phase3G_4_match_coverage)

# 9. Structural validation
assert phase3G_4_combined["match_id"].nunique() == 813
assert phase3G_4_competition_format_exceptions.empty
assert phase3G_4_match_format_exceptions.empty
assert phase3G_4_format_coverage["matches"].sum() == 1626
assert phase3G_4_format_coverage["delivery_records"].sum() == 192235
assert phase3G_4_format_coverage["wicket_states_observed"].between(1, 10).all()

print("\nPHASE 3G.4.1 AUDIT CONSTRUCTION: COMPLETED")

COMPETITION-TO-FORMAT AUDIT


,comp_id,formats,format_values,matches,delivery_records
0,1331,1,"(T20I,)",5,1235
1,1335,1,"(T20I,)",3,755
2,1345,1,"(T20I,)",5,1201
3,1358,1,"(T20,)",74,17909
4,1435,1,"(T20I,)",3,684
5,1439,1,"(T20I,)",4,950
6,1452,1,"(T20I,)",2,505
7,1458,1,"(T20I,)",3,756
8,1474,1,"(T20I,)",2,399
9,1494,1,"(T20I,)",3,733



COMPETITION-TO-FORMAT EXCEPTIONS


,comp_id,formats,format_values,matches,delivery_records



MATCH-TO-FORMAT EXCEPTIONS


,match_id,formats,competitions,innings_populations



FORMAT AND INNINGS COVERAGE


,match_format,innings_population,matches,competitions,delivery_records,wicket_states_observed,minimum_wicket_state,maximum_wicket_state
0,T20,First innings,362,5,45139,10,0,9
1,T20,Second innings,362,5,42078,10,1,10
2,T20I,First innings,451,131,55194,10,0,9
3,T20I,Second innings,451,131,49824,10,1,10



MATCH-LEVEL FORMAT COVERAGE


,match_format,innings_population,matches,competitions
0,T20,First innings,362,5
1,T20,Second innings,362,5
2,T20I,First innings,451,131
3,T20I,Second innings,451,131



PHASE 3G.4.1 AUDIT CONSTRUCTION: COMPLETED


#### Competition-by-wicket-state coverage

In [172]:
# PHASE 3G.4.2 — COMPETITION-BY-WICKET-STATE COVERAGE

# 1. Prepare observed coverage by competition, format, innings, and state
phase3G_4_state_observed = (
    phase3G_4_combined.groupby(
        [
            "comp_id",
            "match_format",
            "innings_population",
            "wicket_state"
        ],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

# 2. Build expected state combinations within each observed competition
phase3G_4_expected_rows = []

for _, comp_row in (
    phase3G_4_combined[
        ["comp_id", "match_format", "innings_population"]
    ]
    .drop_duplicates()
    .iterrows()
):
    if comp_row["innings_population"] == "First innings":
        expected_states = range(0, 10)
    else:
        expected_states = range(1, 11)

    for state in expected_states:
        phase3G_4_expected_rows.append({
            "comp_id": comp_row["comp_id"],
            "match_format": comp_row["match_format"],
            "innings_population": comp_row["innings_population"],
            "wicket_state": state
        })

phase3G_4_expected_coverage = pd.DataFrame(
    phase3G_4_expected_rows
)

# 3. Reconcile expected and observed combinations
phase3G_4_competition_state_coverage = (
    phase3G_4_expected_coverage.merge(
        phase3G_4_state_observed,
        on=[
            "comp_id",
            "match_format",
            "innings_population",
            "wicket_state"
        ],
        how="left",
        validate="one_to_one",
        indicator=True
    )
)

phase3G_4_competition_state_coverage[
    ["delivery_records", "matches"]
] = (
    phase3G_4_competition_state_coverage[
        ["delivery_records", "matches"]
    ].fillna(0).astype(int)
)

phase3G_4_competition_state_coverage["coverage_status"] = (
    phase3G_4_competition_state_coverage["_merge"]
    .map({
        "both": "OBSERVED",
        "left_only": "UNOBSERVED"
    })
)

# 4. Isolate unobserved combinations
phase3G_4_missing_competition_states = (
    phase3G_4_competition_state_coverage.loc[
        phase3G_4_competition_state_coverage["coverage_status"]
        == "UNOBSERVED"
    ].copy()
)

# 5. Summarize coverage by format and innings
phase3G_4_coverage_summary = (
    phase3G_4_competition_state_coverage.groupby(
        ["match_format", "innings_population"],
        dropna=False
    )
    .agg(
        competition_groups=("comp_id", "nunique"),
        expected_combinations=("wicket_state", "size"),
        observed_combinations=(
            "coverage_status",
            lambda x: x.eq("OBSERVED").sum()
        ),
        unobserved_combinations=(
            "coverage_status",
            lambda x: x.eq("UNOBSERVED").sum()
        ),
        observed_delivery_records=("delivery_records", "sum"),
        competitions_with_all_states=(
            "comp_id",
            lambda x: x[
                phase3G_4_competition_state_coverage.loc[
                    x.index, "coverage_status"
                ].eq("OBSERVED")
            ].nunique()
        )
    )
    .reset_index()
)

phase3G_4_coverage_summary["observed_state_coverage_pct"] = (
    100
    * phase3G_4_coverage_summary["observed_combinations"]
    / phase3G_4_coverage_summary["expected_combinations"]
)

# 6. Competition-level coverage profile
phase3G_4_competition_coverage_profile = (
    phase3G_4_competition_state_coverage.groupby(
        ["comp_id", "match_format", "innings_population"],
        dropna=False
    )
    .agg(
        expected_states=("wicket_state", "size"),
        observed_states=(
            "coverage_status",
            lambda x: x.eq("OBSERVED").sum()
        ),
        unobserved_states=(
            "coverage_status",
            lambda x: x.eq("UNOBSERVED").sum()
        ),
        delivery_records=("delivery_records", "sum"),
        matches=("matches", "max")
    )
    .reset_index()
)

phase3G_4_competition_coverage_profile["coverage_pct"] = (
    100
    * phase3G_4_competition_coverage_profile["observed_states"]
    / phase3G_4_competition_coverage_profile["expected_states"]
)

# 7. Export
phase3G_4_competition_state_coverage.to_csv(
    "/content/phase3_outputs/phase3G_4_competition_state_coverage.csv",
    index=False
)

phase3G_4_missing_competition_states.to_csv(
    "/content/phase3_outputs/phase3G_4_missing_competition_states.csv",
    index=False
)

phase3G_4_coverage_summary.to_csv(
    "/content/phase3_outputs/phase3G_4_coverage_summary.csv",
    index=False
)

phase3G_4_competition_coverage_profile.to_csv(
    "/content/phase3_outputs/phase3G_4_competition_coverage_profile.csv",
    index=False
)

# 8. Display
print("FORMAT / INNINGS COVERAGE SUMMARY")
display(phase3G_4_coverage_summary)

print("\nCOMPETITION COVERAGE PROFILE — FIRST 25 ROWS")
display(phase3G_4_competition_coverage_profile.head(25))

print("\nUNOBSERVED COMPETITION-STATE COMBINATIONS — FIRST 30 ROWS")
display(phase3G_4_missing_competition_states.head(30))

print("\nTOTAL UNOBSERVED COMBINATIONS")
print(len(phase3G_4_missing_competition_states))

# 9. Validation
assert len(phase3G_4_competition_state_coverage) == 136 * 20
assert (
    phase3G_4_competition_state_coverage["_merge"]
    .isin(["both", "left_only"])
    .all()
)
assert (
    phase3G_4_competition_state_coverage.loc[
        phase3G_4_competition_state_coverage["coverage_status"] == "OBSERVED",
        "delivery_records"
    ].gt(0).all()
)
assert (
    phase3G_4_competition_state_coverage.loc[
        phase3G_4_competition_state_coverage["coverage_status"] == "UNOBSERVED",
        "delivery_records"
    ].eq(0).all()
)
assert phase3G_4_coverage_summary["expected_combinations"].sum() == 2720

print("\nPHASE 3G.4.2 AUDIT CONSTRUCTION: COMPLETED")

FORMAT / INNINGS COVERAGE SUMMARY


,match_format,innings_population,competition_groups,expected_combinations,observed_combinations,unobserved_combinations,observed_delivery_records,competitions_with_all_states,observed_state_coverage_pct
0,T20,First innings,5,50,50,0,45139,5,100.000000
1,T20,Second innings,5,50,50,0,42078,5,100.000000
2,T20I,First innings,131,1310,1191,119,55194,131,90.916031
3,T20I,Second innings,131,1310,1221,89,49824,131,93.206107



COMPETITION COVERAGE PROFILE — FIRST 25 ROWS


,comp_id,match_format,innings_population,expected_states,observed_states,unobserved_states,delivery_records,matches,coverage_pct
0,1331,T20I,First innings,10,10,0,629,5,100.0
1,1331,T20I,Second innings,10,10,0,606,5,100.0
2,1335,T20I,First innings,10,7,3,385,3,70.0
3,1335,T20I,Second innings,10,10,0,370,3,100.0
4,1345,T20I,First innings,10,10,0,620,5,100.0
5,1345,T20I,Second innings,10,9,1,581,5,90.0
6,1358,T20,First innings,10,10,0,9217,74,100.0
7,1358,T20,Second innings,10,10,0,8692,74,100.0
8,1435,T20I,First innings,10,10,0,364,3,100.0
9,1435,T20I,Second innings,10,8,2,320,3,80.0



UNOBSERVED COMPETITION-STATE COMBINATIONS — FIRST 30 ROWS


,comp_id,match_format,innings_population,wicket_state,delivery_records,matches,_merge,coverage_status
17,1335,T20I,First innings,7,0,0,left_only,UNOBSERVED
18,1335,T20I,First innings,8,0,0,left_only,UNOBSERVED
19,1335,T20I,First innings,9,0,0,left_only,UNOBSERVED
47,1439,T20I,First innings,7,0,0,left_only,UNOBSERVED
48,1439,T20I,First innings,8,0,0,left_only,UNOBSERVED
49,1439,T20I,First innings,9,0,0,left_only,UNOBSERVED
66,1452,T20I,First innings,6,0,0,left_only,UNOBSERVED
67,1452,T20I,First innings,7,0,0,left_only,UNOBSERVED
68,1452,T20I,First innings,8,0,0,left_only,UNOBSERVED
69,1452,T20I,First innings,9,0,0,left_only,UNOBSERVED



TOTAL UNOBSERVED COMBINATIONS
208

PHASE 3G.4.2 AUDIT CONSTRUCTION: COMPLETED


#### State-level coverage depth

In [173]:
# PHASE 3G.4.3 — STATE-LEVEL COVERAGE DEPTH

# 1. Summarize competition-level sample depth for each state
phase3G_4_state_depth = (
    phase3G_4_competition_state_coverage.groupby(
        ["match_format", "innings_population", "wicket_state"],
        dropna=False
    )
    .agg(
        competition_groups=("comp_id", "nunique"),
        competitions_observed=(
            "coverage_status",
            lambda x: x.eq("OBSERVED").sum()
        ),
        competitions_unobserved=(
            "coverage_status",
            lambda x: x.eq("UNOBSERVED").sum()
        ),
        total_delivery_records=("delivery_records", "sum"),
        total_competition_match_counts=("matches", "sum"),
        min_positive_competition_deliveries=(
            "delivery_records",
            lambda x: x[x.gt(0)].min() if x.gt(0).any() else 0
        ),
        median_competition_deliveries=(
            "delivery_records",
            lambda x: x[x.gt(0)].median() if x.gt(0).any() else 0
        ),
        max_competition_deliveries=("delivery_records", "max"),
        min_positive_competition_matches=(
            "matches",
            lambda x: x[x.gt(0)].min() if x.gt(0).any() else 0
        ),
        median_competition_matches=(
            "matches",
            lambda x: x[x.gt(0)].median() if x.gt(0).any() else 0
        ),
        max_competition_matches=("matches", "max")
    )
    .reset_index()
)

# 2. Calculate coverage percentages
phase3G_4_state_depth["competition_coverage_pct"] = (
    100
    * phase3G_4_state_depth["competitions_observed"]
    / phase3G_4_state_depth["competition_groups"]
)

# 3. Summarize competition-level coverage depth by format and innings
phase3G_4_depth_overview = (
    phase3G_4_competition_coverage_profile.groupby(
        ["match_format", "innings_population"],
        dropna=False
    )
    .agg(
        competition_groups=("comp_id", "nunique"),
        fully_covered_competitions=(
            "coverage_pct",
            lambda x: x.eq(100).sum()
        ),
        partially_covered_competitions=(
            "coverage_pct",
            lambda x: x.between(0, 100, inclusive="neither").sum()
        ),
        competitions_with_zero_states=(
            "observed_states",
            lambda x: x.eq(0).sum()
        ),
        minimum_states_observed=("observed_states", "min"),
        median_states_observed=("observed_states", "median"),
        maximum_states_observed=("observed_states", "max"),
        minimum_delivery_records=("delivery_records", "min"),
        median_delivery_records=("delivery_records", "median"),
        maximum_delivery_records=("delivery_records", "max")
    )
    .reset_index()
)

# 4. Export
phase3G_4_state_depth.to_csv(
    "/content/phase3_outputs/phase3G_4_state_depth.csv",
    index=False
)

phase3G_4_depth_overview.to_csv(
    "/content/phase3_outputs/phase3G_4_depth_overview.csv",
    index=False
)

# 5. Display
print("WICKET-STATE COVERAGE DEPTH")
display(phase3G_4_state_depth)

print("\nCOMPETITION-LEVEL COVERAGE DEPTH OVERVIEW")
display(phase3G_4_depth_overview)

# 6. Validation
assert len(phase3G_4_state_depth) == 40
assert phase3G_4_state_depth["competition_groups"].gt(0).all()
assert (
    phase3G_4_state_depth["competitions_observed"]
    + phase3G_4_state_depth["competitions_unobserved"]
).eq(phase3G_4_state_depth["competition_groups"]).all()
assert phase3G_4_state_depth["total_delivery_records"].ge(0).all()
assert phase3G_4_state_depth["total_competition_match_counts"].ge(0).all()
assert len(phase3G_4_depth_overview) == 4

print("\nPHASE 3G.4.3 AUDIT CONSTRUCTION: COMPLETED")

WICKET-STATE COVERAGE DEPTH


,match_format,innings_population,wicket_state,competition_groups,competitions_observed,competitions_unobserved,total_delivery_records,total_competition_match_counts,min_positive_competition_deliveries,median_competition_deliveries,max_competition_deliveries,min_positive_competition_matches,median_competition_matches,max_competition_matches,competition_coverage_pct
0,T20,First innings,0,5,5,0,9187,362,1655,1784.0,2184,71,73.0,74,100.000000
1,T20,First innings,1,5,5,0,8393,361,1370,1690.0,1975,71,73.0,73,100.000000
2,T20,First innings,2,5,5,0,7602,360,1370,1561.0,1673,71,72.0,73,100.000000
3,T20,First innings,3,5,5,0,6502,347,1228,1278.0,1395,69,69.0,70,100.000000
4,T20,First innings,4,5,5,0,5500,315,989,1117.0,1222,60,64.0,64,100.000000
5,T20,First innings,5,5,5,0,3802,277,647,743.0,891,53,55.0,58,100.000000
6,T20,First innings,6,5,5,0,2080,214,356,391.0,489,41,43.0,44,100.000000
7,T20,First innings,7,5,5,0,974,145,146,206.0,260,26,29.0,31,100.000000
8,T20,First innings,8,5,5,0,702,101,84,142.0,178,17,20.0,23,100.000000
9,T20,First innings,9,5,5,0,397,52,67,83.0,90,9,9.0,13,100.000000



COMPETITION-LEVEL COVERAGE DEPTH OVERVIEW


,match_format,innings_population,competition_groups,fully_covered_competitions,partially_covered_competitions,competitions_with_zero_states,minimum_states_observed,median_states_observed,maximum_states_observed,minimum_delivery_records,median_delivery_records,maximum_delivery_records
0,T20,First innings,5,5,0,0,10,10.0,10,8804,9101.0,9217
1,T20,Second innings,5,5,0,0,10,10.0,10,8155,8400.0,8692
2,T20I,First innings,131,68,63,0,5,10.0,10,32,374.0,2324
3,T20I,Second innings,131,93,38,0,3,10.0,10,32,341.0,2106



PHASE 3G.4.3 AUDIT CONSTRUCTION: COMPLETED


#### Single-match dependence audit

In [175]:
# PHASE 3G.4.4 — CORRECTED VALIDATION

assert phase3G_4_match_support_overview[
    "observed_combinations"
].sum() == 2512

assert (
    phase3G_4_match_support_overview["one_match_combinations"]
    + phase3G_4_match_support_overview["multiple_match_combinations"]
).eq(
    phase3G_4_match_support_overview["observed_combinations"]
).all()

assert phase3G_4_match_support_overview["minimum_matches"].ge(1).all()
assert phase3G_4_match_support_overview["minimum_deliveries"].ge(1).all()

assert (
    phase3G_4_match_support_overview.loc[
        phase3G_4_match_support_overview["match_format"] == "T20",
        "one_match_combinations"
    ].sum() == 0
)

assert (
    phase3G_4_match_support_overview.loc[
        phase3G_4_match_support_overview["match_format"] == "T20I",
        "one_match_combinations"
    ].sum() == 461
)

assert len(phase3G_4_thinnest_observed_combinations) == 40

print("OBSERVED COMPETITION-STATE COMBINATIONS:",
      phase3G_4_match_support_overview["observed_combinations"].sum())

print("ONE-MATCH SUPPORTED COMBINATIONS:",
      phase3G_4_match_support_overview["one_match_combinations"].sum())

print("MULTI-MATCH SUPPORTED COMBINATIONS:",
      phase3G_4_match_support_overview["multiple_match_combinations"].sum())

print("\nPHASE 3G.4.4 VALIDATION: PASSED")

OBSERVED COMPETITION-STATE COMBINATIONS: 2512
ONE-MATCH SUPPORTED COMBINATIONS: 461
MULTI-MATCH SUPPORTED COMBINATIONS: 2051

PHASE 3G.4.4 VALIDATION: PASSED


#### Competition coverage robustness register

In [176]:
# PHASE 3G.4.5 — COMPETITION COVERAGE ROBUSTNESS REGISTER

phase3G_4_robustness_register = pd.DataFrame([
    {
        "finding_id": "3G4_R01",
        "area": "Competition-format separation",
        "evidence": "136 competition groups; no competition maps to multiple formats",
        "status": "PASSED",
        "modelling_implication": "Keep T20 and T20I benchmark populations separate"
    },
    {
        "finding_id": "3G4_R02",
        "area": "Match-format consistency",
        "evidence": "813 matches; no format or competition assignment exceptions",
        "status": "PASSED",
        "modelling_implication": "Format and competition identity are structurally consistent"
    },
    {
        "finding_id": "3G4_R03",
        "area": "T20 competition-state coverage",
        "evidence": "100 of 100 combinations observed across five competition groups in both innings",
        "status": "DESCRIPTIVE_COMPLETE",
        "modelling_implication": "Coverage is complete, but state-level sample depth still varies"
    },
    {
        "finding_id": "3G4_R04",
        "area": "T20I competition-state coverage",
        "evidence": "208 of 2620 combinations unobserved across first and second innings",
        "status": "PARTIALLY_UNOBSERVED",
        "modelling_implication": "Do not impute unobserved competition-state combinations"
    },
    {
        "finding_id": "3G4_R05",
        "area": "Single-match dependence",
        "evidence": "461 of 2412 observed T20I combinations are supported by one match",
        "status": "THIN_SUPPORT",
        "modelling_implication": "Do not treat single-match observations as robust benchmarks"
    },
    {
        "finding_id": "3G4_R06",
        "area": "Competition sample-size variation",
        "evidence": "T20I competition groups vary substantially in delivery and match counts",
        "status": "UNEQUAL_COVERAGE",
        "modelling_implication": "Retain sample counts and provenance in future benchmark diagnostics"
    },
    {
        "finding_id": "3G4_R07",
        "area": "Cross-format pooling",
        "evidence": "Phase 0 and prior specifications require separate IPL/T20I treatment",
        "status": "PROHIBITED_BY_DESIGN",
        "modelling_implication": "No cross-format fallback or pooling"
    },
    {
        "finding_id": "3G4_R08",
        "area": "Benchmark reliability",
        "evidence": "No validated sample-size threshold or benchmark outcome test exists",
        "status": "NOT_ESTABLISHED",
        "modelling_implication": "Coverage is not evidence of benchmark reliability"
    }
])

# Validation register
phase3G_4_robustness_validation = pd.DataFrame([
    {
        "check": "Competition-format exception table is empty",
        "passed": phase3G_4_competition_format_exceptions.empty
    },
    {
        "check": "Match-format exception table is empty",
        "passed": phase3G_4_match_format_exceptions.empty
    },
    {
        "check": "Unobserved combinations total 208",
        "passed": len(phase3G_4_missing_competition_states) == 208
    },
    {
        "check": "Observed combinations total 2512",
        "passed": (
            phase3G_4_match_support_overview[
                "observed_combinations"
            ].sum() == 2512
        )
    },
    {
        "check": "Single-match combinations total 461",
        "passed": (
            phase3G_4_match_support_overview[
                "one_match_combinations"
            ].sum() == 461
        )
    },
    {
        "check": "Multiple-match combinations total 2051",
        "passed": (
            phase3G_4_match_support_overview[
                "multiple_match_combinations"
            ].sum() == 2051
        )
    },
    {
        "check": "No cross-format pooling recommendation",
        "passed": phase3G_4_robustness_register.loc[
            phase3G_4_robustness_register["finding_id"] == "3G4_R07",
            "status"
        ].eq("PROHIBITED_BY_DESIGN").all()
    }
])

phase3G_4_robustness_validation["status"] = (
    phase3G_4_robustness_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# Export
phase3G_4_robustness_register.to_csv(
    "/content/phase3_outputs/phase3G_4_robustness_register.csv",
    index=False
)

phase3G_4_robustness_validation.to_csv(
    "/content/phase3_outputs/phase3G_4_robustness_validation.csv",
    index=False
)

# Display
print("COMPETITION COVERAGE ROBUSTNESS REGISTER")
display(phase3G_4_robustness_register)

print("\nVALIDATION REGISTER")
display(phase3G_4_robustness_validation)

assert phase3G_4_robustness_validation["passed"].all()

print("\nPHASE 3G.4.5 VALIDATION: PASSED")

COMPETITION COVERAGE ROBUSTNESS REGISTER


,finding_id,area,evidence,status,modelling_implication
0,3G4_R01,Competition-format separation,136 competition groups; no competition maps to multiple formats,PASSED,Keep T20 and T20I benchmark populations separate
1,3G4_R02,Match-format consistency,813 matches; no format or competition assignment exceptions,PASSED,Format and competition identity are structurally consistent
2,3G4_R03,T20 competition-state coverage,100 of 100 combinations observed across five competition groups in both innings,DESCRIPTIVE_COMPLETE,"Coverage is complete, but state-level sample depth still varies"
3,3G4_R04,T20I competition-state coverage,208 of 2620 combinations unobserved across first and second innings,PARTIALLY_UNOBSERVED,Do not impute unobserved competition-state combinations
4,3G4_R05,Single-match dependence,461 of 2412 observed T20I combinations are supported by one match,THIN_SUPPORT,Do not treat single-match observations as robust benchmarks
5,3G4_R06,Competition sample-size variation,T20I competition groups vary substantially in delivery and match counts,UNEQUAL_COVERAGE,Retain sample counts and provenance in future benchmark diagnostics
6,3G4_R07,Cross-format pooling,Phase 0 and prior specifications require separate IPL/T20I treatment,PROHIBITED_BY_DESIGN,No cross-format fallback or pooling
7,3G4_R08,Benchmark reliability,No validated sample-size threshold or benchmark outcome test exists,NOT_ESTABLISHED,Coverage is not evidence of benchmark reliability



VALIDATION REGISTER


,check,passed,status
0,Competition-format exception table is empty,True,PASSED
1,Match-format exception table is empty,True,PASSED
2,Unobserved combinations total 208,True,PASSED
3,Observed combinations total 2512,True,PASSED
4,Single-match combinations total 461,True,PASSED
5,Multiple-match combinations total 2051,True,PASSED
6,No cross-format pooling recommendation,True,PASSED



PHASE 3G.4.5 VALIDATION: PASSED


#### Competition and coverage robustness closure

In [178]:
# PHASE 3G.4.6 — COMPETITION AND COVERAGE ROBUSTNESS CLOSURE

# 1. Recalculate support counts directly from the coverage table
phase3G_4_observed_for_closure = (
    phase3G_4_competition_state_coverage.loc[
        phase3G_4_competition_state_coverage["coverage_status"] == "OBSERVED"
    ].copy()
)

phase3G_4_observed_combinations_count = len(
    phase3G_4_observed_for_closure
)

phase3G_4_single_match_count = int(
    phase3G_4_observed_for_closure["matches"].eq(1).sum()
)

phase3G_4_multiple_match_count = int(
    phase3G_4_observed_for_closure["matches"].gt(1).sum()
)

phase3G_4_unobserved_count = int(
    phase3G_4_competition_state_coverage[
        "coverage_status"
    ].eq("UNOBSERVED").sum()
)

# 2. Consolidated findings
phase3G_4_closure_summary = pd.DataFrame([
    {
        "audit_area": "Competition-format separation",
        "result": "136 competition groups; zero cross-format assignments",
        "status": "PASSED"
    },
    {
        "audit_area": "Match-format and competition consistency",
        "result": "813 matches; zero assignment exceptions",
        "status": "PASSED"
    },
    {
        "audit_area": "T20 competition-state coverage",
        "result": "100/100 combinations observed",
        "status": "DESCRIPTIVE_COMPLETE"
    },
    {
        "audit_area": "T20I competition-state coverage",
        "result": "208/2620 combinations unobserved; 2412 observed",
        "status": "PARTIALLY_UNOBSERVED"
    },
    {
        "audit_area": "Single-match dependence",
        "result": "461/2412 observed T20I combinations supported by one match",
        "status": "THIN_SUPPORT"
    },
    {
        "audit_area": "Multiple-match support",
        "result": "2051/2412 observed combinations supported by multiple matches",
        "status": "DESCRIPTIVE_ONLY"
    },
    {
        "audit_area": "Benchmark reliability",
        "result": "No validated sample-size threshold or benchmark outcome test",
        "status": "NOT_ESTABLISHED"
    },
    {
        "audit_area": "Cross-format fallback",
        "result": "T20 and T20I must remain separate",
        "status": "PROHIBITED"
    }
])

# 3. Explicit limitations
phase3G_4_limitation_register = pd.DataFrame([
    {
        "limitation_id": "3G4_L01",
        "limitation": "Unobserved T20I competition-state combinations",
        "evidence": "208 combinations have no observed deliveries",
        "classification": "COVERAGE_GAP",
        "treatment": "Do not impute or silently pool"
    },
    {
        "limitation_id": "3G4_L02",
        "limitation": "Single-match T20I support",
        "evidence": "461 observed combinations depend on one match",
        "classification": "THIN_SUPPORT",
        "treatment": "Do not label as robust benchmarks"
    },
    {
        "limitation_id": "3G4_L03",
        "limitation": "Unequal competition sample sizes",
        "evidence": "Competition delivery and match counts vary substantially",
        "classification": "UNEQUAL_COVERAGE",
        "treatment": "Retain competition-level sample counts and provenance"
    },
    {
        "limitation_id": "3G4_L04",
        "limitation": "No validated reliability threshold",
        "evidence": "No empirical benchmark reliability test completed",
        "classification": "NOT_ESTABLISHED",
        "treatment": "Do not invent minimum sample thresholds"
    },
    {
        "limitation_id": "3G4_L05",
        "limitation": "No cross-format fallback",
        "evidence": "Separate T20 and T20I benchmark policy",
        "classification": "METHODOLOGICAL_CONSTRAINT",
        "treatment": "Keep formats separate"
    }
])

# 4. Closure validation
phase3G_4_closure_validation = pd.DataFrame([
    {
        "check_id": "3G4_C01",
        "check": "All 3G.4.5 robustness validations passed",
        "passed": phase3G_4_robustness_validation["passed"].all()
    },
    {
        "check_id": "3G4_C02",
        "check": "Competition-format exceptions remain empty",
        "passed": phase3G_4_competition_format_exceptions.empty
    },
    {
        "check_id": "3G4_C03",
        "check": "Match-format exceptions remain empty",
        "passed": phase3G_4_match_format_exceptions.empty
    },
    {
        "check_id": "3G4_C04",
        "check": "T20I unobserved combinations total 208",
        "passed": phase3G_4_unobserved_count == 208
    },
    {
        "check_id": "3G4_C05",
        "check": "Observed combinations total 2512",
        "passed": phase3G_4_observed_combinations_count == 2512
    },
    {
        "check_id": "3G4_C06",
        "check": "Single-match combinations total 461",
        "passed": phase3G_4_single_match_count == 461
    },
    {
        "check_id": "3G4_C07",
        "check": "Multiple-match combinations total 2051",
        "passed": phase3G_4_multiple_match_count == 2051
    },
    {
        "check_id": "3G4_C08",
        "check": "No cross-format fallback is authorized",
        "passed": phase3G_4_robustness_register.loc[
            phase3G_4_robustness_register["finding_id"] == "3G4_R07",
            "status"
        ].eq("PROHIBITED_BY_DESIGN").all()
    },
    {
        "check_id": "3G4_C09",
        "check": "Benchmark reliability remains unestablished",
        "passed": phase3G_4_robustness_register.loc[
            phase3G_4_robustness_register["finding_id"] == "3G4_R08",
            "status"
        ].eq("NOT_ESTABLISHED").all()
    }
])

phase3G_4_closure_validation["status"] = (
    phase3G_4_closure_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# 5. Closure decision
phase3G_4_closure_status = (
    "COMPLETE"
    if phase3G_4_closure_validation["passed"].all()
    else "NOT_CLOSED_VALIDATION_FAILURE"
)

phase3G_4_closure_decision = pd.DataFrame([{
    "subphase": "Phase 3G.4 — Competition and coverage robustness",
    "status": phase3G_4_closure_status,
    "validation_checks": len(phase3G_4_closure_validation),
    "passed": int(phase3G_4_closure_validation["passed"].sum()),
    "failed": int((~phase3G_4_closure_validation["passed"]).sum()),
    "competition_groups": 136,
    "matches": 813,
    "observed_competition_state_combinations": phase3G_4_observed_combinations_count,
    "unobserved_combinations": phase3G_4_unobserved_count,
    "single_match_combinations": phase3G_4_single_match_count,
    "multiple_match_combinations": phase3G_4_multiple_match_count,
    "benchmark_reliability": "NOT_ESTABLISHED",
    "cross_format_fallback": "PROHIBITED"
}])

# 6. Export
phase3G_4_closure_summary.to_csv(
    "/content/phase3_outputs/phase3G_4_closure_summary.csv",
    index=False
)

phase3G_4_limitation_register.to_csv(
    "/content/phase3_outputs/phase3G_4_limitation_register.csv",
    index=False
)

phase3G_4_closure_validation.to_csv(
    "/content/phase3_outputs/phase3G_4_closure_validation.csv",
    index=False
)

phase3G_4_closure_decision.to_csv(
    "/content/phase3_outputs/phase3G_4_closure_decision.csv",
    index=False
)

# 7. Display
print("PHASE 3G.4 CLOSURE SUMMARY")
display(phase3G_4_closure_summary)

print("\nPHASE 3G.4 LIMITATION REGISTER")
display(phase3G_4_limitation_register)

print("\nPHASE 3G.4 CLOSURE VALIDATION")
display(phase3G_4_closure_validation)

print("\nPHASE 3G.4 CLOSURE DECISION")
display(phase3G_4_closure_decision)

# 8. Final assertions
assert phase3G_4_closure_validation["passed"].all()
assert phase3G_4_closure_status == "COMPLETE"
assert len(phase3G_4_limitation_register) == 5

print("\nPHASE 3G.4 CLOSURE VALIDATION: PASSED")

PHASE 3G.4 CLOSURE SUMMARY


,audit_area,result,status
0,Competition-format separation,136 competition groups; zero cross-format assignments,PASSED
1,Match-format and competition consistency,813 matches; zero assignment exceptions,PASSED
2,T20 competition-state coverage,100/100 combinations observed,DESCRIPTIVE_COMPLETE
3,T20I competition-state coverage,208/2620 combinations unobserved; 2412 observed,PARTIALLY_UNOBSERVED
4,Single-match dependence,461/2412 observed T20I combinations supported by one match,THIN_SUPPORT
5,Multiple-match support,2051/2412 observed combinations supported by multiple matches,DESCRIPTIVE_ONLY
6,Benchmark reliability,No validated sample-size threshold or benchmark outcome test,NOT_ESTABLISHED
7,Cross-format fallback,T20 and T20I must remain separate,PROHIBITED



PHASE 3G.4 LIMITATION REGISTER


,limitation_id,limitation,evidence,classification,treatment
0,3G4_L01,Unobserved T20I competition-state combinations,208 combinations have no observed deliveries,COVERAGE_GAP,Do not impute or silently pool
1,3G4_L02,Single-match T20I support,461 observed combinations depend on one match,THIN_SUPPORT,Do not label as robust benchmarks
2,3G4_L03,Unequal competition sample sizes,Competition delivery and match counts vary substantially,UNEQUAL_COVERAGE,Retain competition-level sample counts and provenance
3,3G4_L04,No validated reliability threshold,No empirical benchmark reliability test completed,NOT_ESTABLISHED,Do not invent minimum sample thresholds
4,3G4_L05,No cross-format fallback,Separate T20 and T20I benchmark policy,METHODOLOGICAL_CONSTRAINT,Keep formats separate



PHASE 3G.4 CLOSURE VALIDATION


,check_id,check,passed,status
0,3G4_C01,All 3G.4.5 robustness validations passed,True,PASSED
1,3G4_C02,Competition-format exceptions remain empty,True,PASSED
2,3G4_C03,Match-format exceptions remain empty,True,PASSED
3,3G4_C04,T20I unobserved combinations total 208,True,PASSED
4,3G4_C05,Observed combinations total 2512,True,PASSED
5,3G4_C06,Single-match combinations total 461,True,PASSED
6,3G4_C07,Multiple-match combinations total 2051,True,PASSED
7,3G4_C08,No cross-format fallback is authorized,True,PASSED
8,3G4_C09,Benchmark reliability remains unestablished,True,PASSED



PHASE 3G.4 CLOSURE DECISION


,subphase,status,validation_checks,passed,failed,competition_groups,matches,observed_competition_state_combinations,unobserved_combinations,single_match_combinations,multiple_match_combinations,benchmark_reliability,cross_format_fallback
0,Phase 3G.4 — Competition and coverage robustness,COMPLETE,9,9,0,136,813,2512,208,461,2051,NOT_ESTABLISHED,PROHIBITED



PHASE 3G.4 CLOSURE VALIDATION: PASSED


### Methodological decision and limitation robustness

#### METHODOLOGICAL REGISTER INVENTORY

In [179]:
# PHASE 3G.5.1 — METHODOLOGICAL REGISTER INVENTORY

phase3G_5_register_objects = {
    "Phase 3C state schema": phase3C_state_schema,
    "Phase 3C exception register": phase3C_exception_register,
    "Phase 3D fallback policy": phase3D_fallback_policy,
    "Phase 3E fallback policy": phase3E_fallback_policy,
    "Phase 3F.1 decision register": phase3F1_decision_register,
    "Phase 3F.2 architecture decisions": phase3F2_architecture_decisions,
    "Phase 3F.3 directionality": phase3F3_directionality,
    "Phase 3F.4 sensitivity register": phase3F4_sensitivity_register,
    "Phase 3F.5 current decision": phase3F5_current_decision,
    "Phase 3F.6 validation": phase3F6_validation
}

# 1. Check object types and availability
phase3G_5_object_checks = pd.DataFrame([
    {
        "register": name,
        "is_dataframe": isinstance(obj, pd.DataFrame),
        "rows": len(obj) if isinstance(obj, pd.DataFrame) else None,
        "columns": len(obj.columns) if isinstance(obj, pd.DataFrame) else None,
        "empty": obj.empty if isinstance(obj, pd.DataFrame) else None
    }
    for name, obj in phase3G_5_register_objects.items()
])

print("REGISTER OBJECT CHECK")
display(phase3G_5_object_checks)

assert phase3G_5_object_checks["is_dataframe"].all()
assert (~phase3G_5_object_checks["empty"]).all()

# 2. Inspect schemas
phase3G_5_register_inventory = pd.DataFrame([
    {
        "register": name,
        "rows": len(obj),
        "columns": len(obj.columns),
        "column_names": ", ".join(map(str, obj.columns)),
        "empty": obj.empty
    }
    for name, obj in phase3G_5_register_objects.items()
])

print("\nMETHODOLOGICAL REGISTER INVENTORY")
display(phase3G_5_register_inventory)

# 3. Preview each register
for name, obj in phase3G_5_register_objects.items():
    print(f"\n{name}")
    display(obj.head(5))

# 4. Export
phase3G_5_register_inventory.to_csv(
    "/content/phase3_outputs/phase3G_5_register_inventory.csv",
    index=False
)

phase3G_5_object_checks.to_csv(
    "/content/phase3_outputs/phase3G_5_object_checks.csv",
    index=False
)

# 5. Validation
assert len(phase3G_5_register_inventory) == 10
assert phase3G_5_register_inventory["empty"].eq(False).all()
assert phase3G_5_register_inventory["rows"].gt(0).all()
assert phase3G_5_register_inventory["columns"].gt(0).all()

print("\nPHASE 3G.5.1 INVENTORY: COMPLETED")

REGISTER OBJECT CHECK


,register,is_dataframe,rows,columns,empty
0,Phase 3C state schema,True,12,8,False
1,Phase 3C exception register,True,11,6,False
2,Phase 3D fallback policy,True,6,6,False
3,Phase 3E fallback policy,True,6,5,False
4,Phase 3F.1 decision register,True,10,9,False
5,Phase 3F.2 architecture decisions,True,4,5,False
6,Phase 3F.3 directionality,True,8,6,False
7,Phase 3F.4 sensitivity register,True,8,4,False
8,Phase 3F.5 current decision,True,6,3,False
9,Phase 3F.6 validation,True,6,2,False



METHODOLOGICAL REGISTER INVENTORY


,register,rows,columns,column_names,empty
0,Phase 3C state schema,12,8,"field, state_role, data_type, classification, pre_delivery_use, source_dataset, temporal_boundary, reason",False
1,Phase 3C exception register,11,6,"exception_id, field, status, evidence, impact, treatment",False
2,Phase 3D fallback policy,6,6,"priority, fallback_level, policy, conditions, status, restriction",False
3,Phase 3E fallback policy,6,5,"priority, fallback_level, rule, execution_status, reason",False
4,Phase 3F.1 decision register,10,9,"innings_context, pressure_component, required_inputs, candidate_fields, phase_status, reason, decision, numerical_scale_eligible, decision_explanation",False
5,Phase 3F.2 architecture decisions,4,5,"architecture, decision, may_proceed_to_numerical_construction, reason, reconsideration_requirements",False
6,Phase 3F.3 directionality,8,6,"innings_context, component, measurement_direction, expected_difficulty_relationship, relationship_status, limitation",False
7,Phase 3F.4 sensitivity register,8,4,"sensitivity_test, evidence_available, feasibility, limitation",False
8,Phase 3F.5 current decision,6,3,"decision_area, decision, reason",False
9,Phase 3F.6 validation,6,2,"check, passed",False



Phase 3C state schema


,field,state_role,data_type,classification,pre_delivery_use,source_dataset,temporal_boundary,reason
0,match_id,Match identifier,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the match containing the delivery.
1,innings,Innings context,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the innings containing the delivery.
2,comp_id,Competition identifier,int64,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Identifies the competition.
3,comp_name,Competition context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides competition context.
4,match_format,Format context,object,DIRECTLY OBSERVED,Allowed,Integrated deliveries,Known immediately before current delivery,Provides the recorded match format.


... (15 outputs left out)

,decision_area,decision,reason
0,First-innings numerical pressure scale,UNAVAILABLE,"Cumulative scoring, remaining resources and expected scoring benchmarks are unavailable."
1,Second-innings numerical pressure scale,UNAVAILABLE,"Target semantics, cumulative score, remaining resources and expected scoring benchmarks are not validated."
2,Unified cross-innings numerical scale,UNAVAILABLE,Neither innings-specific numerical scale is constructible and no cross-innings calibration exists.
3,Shared conceptual framework,RETAIN_AS_CONCEPTUAL_FRAMEWORK,"The common pre-delivery state principles and separate innings-specific measures remain conceptually useful, but are not numerically validated."
4,Wicket-state component,CONDITIONAL,"Reconstructed pre-delivery wicket states are available, subject to their documented reliability limitation."



Phase 3F.6 validation


,check,passed
0,All six Phase 3F subphases listed,True
1,All subphases marked complete,True
2,Three numerical scale decisions unavailable,True
3,Conceptual framework retained with limitations,True
4,Report file exists,True



PHASE 3G.5.1 INVENTORY: COMPLETED


#### Full policy and decision inspection

In [180]:
# PHASE 3G.5.2 — FULL METHODOLOGICAL POLICY AND DECISION INSPECTION

# 1. Full fallback policies
print("PHASE 3D — COMPLETE FALLBACK POLICY")
display(phase3D_fallback_policy)

print("\nPHASE 3E — COMPLETE FALLBACK POLICY")
display(phase3E_fallback_policy)

# 2. Full Phase 3F.1 component decisions
print("\nPHASE 3F.1 — COMPLETE COMPONENT DECISION REGISTER")
display(phase3F1_decision_register)

# 3. Full Phase 3F.2 architecture decisions
print("\nPHASE 3F.2 — COMPLETE ARCHITECTURE DECISIONS")
display(phase3F2_architecture_decisions)

# 4. Full Phase 3F.3 directionality
print("\nPHASE 3F.3 — COMPLETE DIRECTIONALITY REGISTER")
display(phase3F3_directionality)

# 5. Full Phase 3F.4 sensitivity register
print("\nPHASE 3F.4 — COMPLETE SENSITIVITY REGISTER")
display(phase3F4_sensitivity_register)

# 6. Full Phase 3F.5 final decisions
print("\nPHASE 3F.5 — COMPLETE CURRENT DECISIONS")
display(phase3F5_current_decision)

# 7. Export a combined policy inspection record
phase3G_5_policy_inspection = pd.concat(
    [
        phase3D_fallback_policy.assign(source_phase="Phase 3D"),
        phase3E_fallback_policy.assign(source_phase="Phase 3E")
    ],
    ignore_index=True,
    sort=False
)

phase3G_5_policy_inspection.to_csv(
    "/content/phase3_outputs/phase3G_5_policy_inspection.csv",
    index=False
)

print("\nPHASE 3G.5.2 INSPECTION: COMPLETED")

PHASE 3D — COMPLETE FALLBACK POLICY


,priority,fallback_level,policy,conditions,status,restriction
0,1,Exact comparable state,Use observations matching the same competition format and all validated benchmark-state dimensions,All required state dimensions and scoring outcomes must be validated,DEFINED_NOT_EXECUTABLE,No estimate where required dimensions are unavailable
1,2,Nearby scheduled-innings lengths,Pool nearby scheduled lengths only where a defensible distance rule is documented,Scheduled innings length must be validated and units consistent,DEFINED_NOT_EXECUTABLE,Do not interpret unresolved revised-overs values
2,3,Adjacent wicket states,Pool adjacent wicket states using observation-weighted aggregation,Wicket states must retain reconstruction provenance and scoring inputs must be valid,DEFINED_NOT_EXECUTABLE,Do not weight wicket-state groups equally regardless of sample size
3,4,Symmetric state expansion,Expand around the target state symmetrically and record included states,A defensible expansion rule and sample coverage assessment must be specified,DEFINED_NOT_EXECUTABLE,Do not silently broaden the comparison population
4,5,Broader innings-stage grouping,Use a broader stage grouping only if stage boundaries and progression are validated,Validated innings stage and schedule required,DEFINED_NOT_EXECUTABLE,Do not infer stage from delivery sequence alone
5,6,Unavailable,Return no benchmark when no defensible comparable state can be formed,Required inputs unavailable or fallback evidence insufficient,APPROVED_AS_TERMINAL_RULE,Do not manufacture or publish an unsupported estimate



PHASE 3E — COMPLETE FALLBACK POLICY


,priority,fallback_level,rule,execution_status,reason
0,1,Exact comparable state,"Use observations matching format, competition context, innings stage, wickets remaining, and remaining resources",NOT_EXECUTABLE,Stage and remaining resources are unavailable
1,2,Nearby scheduled innings lengths,Use nearby validated scheduled lengths when exact schedule state is sparse,NOT_EXECUTABLE,Scheduled innings length and units are unverified
2,3,Adjacent wicket states,Use observation-weighted adjacent wicket-state estimates when exact wicket-state coverage is sparse,POLICY_DEFINED_ONLY,Wickets remaining is conditional; other required state dimensions remain unavailable
3,4,Symmetric state expansion,Expand comparable states symmetrically and record the exact expansion,NOT_EXECUTABLE,"Valid schedule, stage, and remaining-resource coordinates are unavailable"
4,5,Broader stage grouping,Use broader stage groups only if stage definitions are validated and grouping remains defensible,NOT_EXECUTABLE,Innings stage is unavailable
5,6,Unavailable terminal rule,Mark the benchmark unavailable when valid comparable states cannot be established or estimates remain materially unstable,APPROVED_TERMINAL_RULE,Prevents unsupported benchmark estimates



PHASE 3F.1 — COMPLETE COMPONENT DECISION REGISTER


,innings_context,pressure_component,required_inputs,candidate_fields,phase_status,reason,decision,numerical_scale_eligible,decision_explanation
0,First innings,Scoring position relative to expectation,Cumulative innings score; expected cumulative score for comparable state,runs; cumulative score (not validated); expected score (not constructed),BLOCKED,Phase 3D could not validate cumulative scoring or construct the expected-score benchmark.,BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
1,First innings,Standardized scoring deviation,Actual cumulative score; expected cumulative score; historical standard deviation for comparable states,Scoring deviation inputs and historical benchmark,BLOCKED,Depends on the unavailable scoring and benchmark inputs; no standardized value can currently be calculated.,BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
2,First innings,Wickets remaining,Pre-delivery wickets fallen,pre_delivery_wickets_fallen in reconstructed state table,CONDITIONAL,Reconstructed from prior wicket events; not independently verified dismissal truth.,CONDITIONAL_INPUT,False,Wicket-state input may be used only with its reconstruction limitation explicitly retained.
3,First innings,Balls remaining,Validated legal balls remaining and scheduled innings length,Legal-ball evidence; overid; over_ball_num; innings_ball_num; first_inn_revised_overs,UNAVAILABLE,Sequence fields are not legal-ball counters; scheduled-length field is sparse and its units are unresolved.,BLOCKED_INPUT,False,Required resource or stage measurement is not validated in the supplied data.
4,Second innings,Required-rate difficulty,Verified target; validated current cumulative score; validated legal balls remaining,revised_target; runs; legal-ball evidence; second_inn_revised_overs,BLOCKED,Target is unverified and sparse; cumulative score and legal balls remaining are unavailable.,BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
5,Second innings,Expected scoring-rate benchmark,Competition; innings stage; wickets remaining; remaining resources,comp_id; match_format; pre-delivery_wickets_fallen; legal-ball and schedule inputs,BLOCKED,"Competition is available and wickets are conditional, but stage and remaining-resource inputs are unavailable.",BLOCKED_COMPONENT,False,The component depends on unavailable or unverified inputs.
6,Second innings,Wickets remaining,Pre-delivery wickets fallen,pre_delivery_wickets_fallen in reconstructed state table,CONDITIONAL,Derived as 10 minus reconstructed pre-delivery wickets fallen; retains the same reconstruction limitation.,CONDITIONAL_INPUT,False,Wicket-state input may be used only with its reconstruction limitation explicitly retained.
7,Second innings,Balls/resources remaining,Validated legal balls remaining and scheduled innings length,Legal-ball evidence; overid; over_ball_num; innings_ball_num; second_inn_revised_overs,UNAVAILABLE,Legal-ball progression and scheduled innings length are not validated.,BLOCKED_INPUT,False,Required resource or stage measurement is not validated in the supplied data.
8,Both innings,Competition context,Competition format and competition identity,match_format; comp_id; comp_name,AVAILABLE,Competition identity and format are available; IPL and T20I remain separate.,CONTEXT_ONLY,False,"Competition context supports grouping and stratification, not pressure measurement by itself."
9,Both innings,Innings stage,Validated innings progress relative to scheduled innings length,overid; innings_ball_num; scheduled innings length,UNAVAILABLE,Sequence positions do not establish legal-ball progress or innings stage.,BLOCKED_INPUT,False,Required resource or stage measurement is not validated in the supplied data.


... (5 outputs left out)

,sensitivity_test,evidence_available,feasibility,limitation
0,First-innings exact competition/state coverage,Aggregate wicket distributions and missing competition/state register,PARTIAL_DESCRIPTIVE,255 competition/state combinations are unobserved; detailed coverage table is unavailable.
1,Second-innings wicket-state coverage by format,Wickets-remaining distributions by T20 and T20I,DESCRIPTIVE_ONLY,Wicket states are conditional reconstructions; no pressure outcome is available.
2,Alternative wicket-state grouping,Observed wicket-state distributions,POLICY_DESIGN_ONLY,No grouping threshold or pooling rule has been empirically approved.
3,Alternative pressure component weights,No constructed pressure score or approved weights,BLOCKED,No numerical scale exists on which to test weight sensitivity.
4,Alternative normalization methods,No complete component normalization specification,BLOCKED,Normalization alternatives cannot be compared without validated component measurements.
5,Alternative pressure thresholds or bands,No numerical pressure values or approved thresholds,BLOCKED,No pressure classifications exist to compare.
6,First- versus second-innings stability,Separate conceptual frameworks and conditional wicket-state coverage,NOT_TESTABLE,No validated numerical measures or cross-innings calibration.
7,IPL versus T20I pressure stability,Competition-specific aggregate wicket-state coverage,PARTIAL_DESCRIPTIVE,Benchmark measures are not constructed and exact-state coverage is incomplete.



PHASE 3F.5 — COMPLETE CURRENT DECISIONS


,decision_area,decision,reason
0,First-innings numerical pressure scale,UNAVAILABLE,"Cumulative scoring, remaining resources and expected scoring benchmarks are unavailable."
1,Second-innings numerical pressure scale,UNAVAILABLE,"Target semantics, cumulative score, remaining resources and expected scoring benchmarks are not validated."
2,Unified cross-innings numerical scale,UNAVAILABLE,Neither innings-specific numerical scale is constructible and no cross-innings calibration exists.
3,Shared conceptual framework,RETAIN_AS_CONCEPTUAL_FRAMEWORK,"The common pre-delivery state principles and separate innings-specific measures remain conceptually useful, but are not numerically validated."
4,Wicket-state component,CONDITIONAL,"Reconstructed pre-delivery wicket states are available, subject to their documented reliability limitation."
5,Pressure bands and thresholds,NOT_ESTABLISHED,No numerical scale or empirically justified thresholds exist.



PHASE 3G.5.2 INSPECTION: COMPLETED


#### Methodological consistency audit

In [183]:
# PHASE 3G.5.3 — METHODOLOGICAL CONSISTENCY AUDIT
# Self-contained: uses established source DataFrames only.

# 1. Compare fallback policies by priority
fallback_comparison = phase3D_fallback_policy[
    ["priority", "fallback_level", "status"]
].merge(
    phase3E_fallback_policy[
        ["priority", "fallback_level", "execution_status"]
    ],
    on="priority",
    how="outer",
    validate="one_to_one",
    suffixes=("_3D", "_3E"),
    indicator=True
)

fallback_comparison["priority_aligned"] = (
    fallback_comparison["_merge"].eq("both")
)

fallback_comparison["wording_difference"] = (
    fallback_comparison["fallback_level_3D"]
    != fallback_comparison["fallback_level_3E"]
)

# 2. Evaluate the methodological decisions directly
wicket_rows = phase3F1_decision_register[
    phase3F1_decision_register["pressure_component"]
    .str.contains("Wickets remaining", case=False, na=False)
]

scale_rows = phase3F5_current_decision[
    phase3F5_current_decision["decision_area"].isin([
        "First-innings numerical pressure scale",
        "Second-innings numerical pressure scale",
        "Unified cross-innings numerical scale"
    ])
]

shared_architecture = phase3F2_architecture_decisions[
    phase3F2_architecture_decisions["architecture"]
    .str.contains("Shared framework", case=False, na=False)
]

threshold_rows = phase3F5_current_decision[
    phase3F5_current_decision["decision_area"]
    .str.contains("bands|thresholds", case=False, na=False)
]

unavailable_rows = phase3C_exception_register[
    phase3C_exception_register["status"].eq("UNAVAILABLE")
]

# 3. Build consistency validation register
consistency_checks = pd.DataFrame([
    {
        "check_id": "3G5_C01",
        "check": "Phase 3D and 3E fallback priorities align",
        "passed": (
            len(fallback_comparison) == 6
            and fallback_comparison["_merge"].eq("both").all()
            and fallback_comparison["priority"].tolist()
                == [1, 2, 3, 4, 5, 6]
        )
    },
    {
        "check_id": "3G5_C02",
        "check": "Both policies approve unavailable terminal rule",
        "passed": (
            phase3D_fallback_policy.loc[
                phase3D_fallback_policy["priority"].eq(6), "status"
            ].eq("APPROVED_AS_TERMINAL_RULE").all()
            and phase3E_fallback_policy.loc[
                phase3E_fallback_policy["priority"].eq(6),
                "execution_status"
            ].eq("APPROVED_TERMINAL_RULE").all()
        )
    },
    {
        "check_id": "3G5_C03",
        "check": "No numerical component is eligible",
        "passed": (
            phase3F1_decision_register[
                "numerical_scale_eligible"
            ].eq(False).all()
        )
    },
    {
        "check_id": "3G5_C04",
        "check": "All three numerical scale decisions remain unavailable",
        "passed": (
            len(scale_rows) == 3
            and scale_rows["decision"].eq("UNAVAILABLE").all()
        )
    },
    {
        "check_id": "3G5_C05",
        "check": "Wicket components retain conditional status",
        "passed": (
            len(wicket_rows) == 2
            and wicket_rows["phase_status"].eq("CONDITIONAL").all()
            and wicket_rows["decision"].eq("CONDITIONAL_INPUT").all()
        )
    },
    {
        "check_id": "3G5_C06",
        "check": "Unavailable state inputs remain documented",
        "passed": (
            len(unavailable_rows) >= 4
        )
    },
    {
        "check_id": "3G5_C07",
        "check": "Shared conceptual architecture cannot proceed numerically",
        "passed": (
            len(shared_architecture) == 1
            and shared_architecture[
                "may_proceed_to_numerical_construction"
            ].eq(False).all()
        )
    },
    {
        "check_id": "3G5_C08",
        "check": "Pressure bands and thresholds remain unestablished",
        "passed": (
            len(threshold_rows) == 1
            and threshold_rows["decision"].eq("NOT_ESTABLISHED").all()
        )
    }
])

consistency_checks["status"] = consistency_checks["passed"].map({
    True: "PASSED",
    False: "FAILED"
})

# 4. Record wording differences without treating them as policy conflicts
fallback_wording_notes = fallback_comparison.loc[
    fallback_comparison["wording_difference"],
    ["priority", "fallback_level_3D", "fallback_level_3E"]
].copy()

fallback_wording_notes["assessment"] = (
    "Wording difference; same priority in both policies"
)

# 5. Export
fallback_comparison.to_csv(
    "/content/phase3_outputs/phase3G_5_fallback_comparison.csv",
    index=False
)

consistency_checks.to_csv(
    "/content/phase3_outputs/phase3G_5_consistency_checks.csv",
    index=False
)

fallback_wording_notes.to_csv(
    "/content/phase3_outputs/phase3G_5_fallback_wording_notes.csv",
    index=False
)

# 6. Display results
print("FALLBACK POLICY COMPARISON")
display(fallback_comparison)

print("\nFALLBACK WORDING DIFFERENCES")
display(fallback_wording_notes)

print("\nMETHODOLOGICAL CONSISTENCY CHECKS")
display(consistency_checks)

print("\nPASSED:", int(consistency_checks["passed"].sum()))
print("FAILED:", int((~consistency_checks["passed"]).sum()))

assert consistency_checks["passed"].all()

print("\nPHASE 3G.5.3 CONSISTENCY AUDIT: PASSED")

FALLBACK POLICY COMPARISON


,priority,fallback_level_3D,status,fallback_level_3E,execution_status,_merge,priority_aligned,wording_difference
0,1,Exact comparable state,DEFINED_NOT_EXECUTABLE,Exact comparable state,NOT_EXECUTABLE,both,True,False
1,2,Nearby scheduled-innings lengths,DEFINED_NOT_EXECUTABLE,Nearby scheduled innings lengths,NOT_EXECUTABLE,both,True,True
2,3,Adjacent wicket states,DEFINED_NOT_EXECUTABLE,Adjacent wicket states,POLICY_DEFINED_ONLY,both,True,False
3,4,Symmetric state expansion,DEFINED_NOT_EXECUTABLE,Symmetric state expansion,NOT_EXECUTABLE,both,True,False
4,5,Broader innings-stage grouping,DEFINED_NOT_EXECUTABLE,Broader stage grouping,NOT_EXECUTABLE,both,True,True
5,6,Unavailable,APPROVED_AS_TERMINAL_RULE,Unavailable terminal rule,APPROVED_TERMINAL_RULE,both,True,True



FALLBACK WORDING DIFFERENCES


,priority,fallback_level_3D,fallback_level_3E,assessment
1,2,Nearby scheduled-innings lengths,Nearby scheduled innings lengths,Wording difference; same priority in both policies
4,5,Broader innings-stage grouping,Broader stage grouping,Wording difference; same priority in both policies
5,6,Unavailable,Unavailable terminal rule,Wording difference; same priority in both policies



METHODOLOGICAL CONSISTENCY CHECKS


,check_id,check,passed,status
0,3G5_C01,Phase 3D and 3E fallback priorities align,True,PASSED
1,3G5_C02,Both policies approve unavailable terminal rule,True,PASSED
2,3G5_C03,No numerical component is eligible,True,PASSED
3,3G5_C04,All three numerical scale decisions remain unavailable,True,PASSED
4,3G5_C05,Wicket components retain conditional status,True,PASSED
5,3G5_C06,Unavailable state inputs remain documented,True,PASSED
6,3G5_C07,Shared conceptual architecture cannot proceed numerically,True,PASSED
7,3G5_C08,Pressure bands and thresholds remain unestablished,True,PASSED



PASSED: 8
FAILED: 0

PHASE 3G.5.3 CONSISTENCY AUDIT: PASSED


#### Inspect the coverage discrepancy

In [184]:
# PHASE 3G.5.4 — COVERAGE DISCREPANCY INSPECTION

print("PHASE 3D MISSING-STATE REGISTER")
print("Shape:", phase3D_missing_comp_wicket_states.shape)
print("Columns:", phase3D_missing_comp_wicket_states.columns.tolist())
display(phase3D_missing_comp_wicket_states.head(15))

print("\nPHASE 3G.4 MISSING-STATE REGISTER")
print("Shape:", phase3G_4_missing_competition_states.shape)
print("Columns:", phase3G_4_missing_competition_states.columns.tolist())
display(phase3G_4_missing_competition_states.head(15))

print("\nPHASE 3G.4 FULL COVERAGE TABLE")
print("Shape:", phase3G_4_competition_state_coverage.shape)
print("Columns:", phase3G_4_competition_state_coverage.columns.tolist())
display(phase3G_4_competition_state_coverage.head(15))

print("\nROW COUNTS")
print("Phase 3D missing-state rows:",
      len(phase3D_missing_comp_wicket_states))
print("Phase 3G.4 missing-state rows:",
      len(phase3G_4_missing_competition_states))

print("\nDISTINCT COMPETITION COUNTS")
if "comp_id" in phase3D_missing_comp_wicket_states.columns:
    print("Phase 3D:",
          phase3D_missing_comp_wicket_states["comp_id"].nunique())

if "comp_id" in phase3G_4_missing_competition_states.columns:
    print("Phase 3G.4:",
          phase3G_4_missing_competition_states["comp_id"].nunique())

print("\nDISTINCT WICKET-STATE COUNTS")
if "wicket_state" in phase3D_missing_comp_wicket_states.columns:
    print("Phase 3D:",
          phase3D_missing_comp_wicket_states["wicket_state"].nunique())

if "wicket_state" in phase3G_4_missing_competition_states.columns:
    print("Phase 3G.4:",
          phase3G_4_missing_competition_states["wicket_state"].nunique())

PHASE 3D MISSING-STATE REGISTER
Shape: (255, 4)
Columns: ['benchmark_population', 'comp_id', 'comp_name', 'pre_delivery_wickets_fallen']


,benchmark_population,comp_id,comp_name,pre_delivery_wickets_fallen
10,T20I,1331,England in West Indies T20I Series 2022,10
18,T20I,1335,West Indies in India T20I Series 2022,7
19,T20I,1335,West Indies in India T20I Series 2022,8
20,T20I,1335,West Indies in India T20I Series 2022,9
21,T20I,1335,West Indies in India T20I Series 2022,10
32,T20I,1345,Sri Lanka in Australia T20I Series 2022,10
43,IPL,1358,IPL 2022,10
51,T20I,1439,South Africa in India T20I Series 2022,7
52,T20I,1439,South Africa in India T20I Series 2022,8
53,T20I,1439,South Africa in India T20I Series 2022,9



PHASE 3G.4 MISSING-STATE REGISTER
Shape: (208, 8)
Columns: ['comp_id', 'match_format', 'innings_population', 'wicket_state', 'delivery_records', 'matches', '_merge', 'coverage_status']


,comp_id,match_format,innings_population,wicket_state,delivery_records,matches,_merge,coverage_status
17,1335,T20I,First innings,7,0,0,left_only,UNOBSERVED
18,1335,T20I,First innings,8,0,0,left_only,UNOBSERVED
19,1335,T20I,First innings,9,0,0,left_only,UNOBSERVED
47,1439,T20I,First innings,7,0,0,left_only,UNOBSERVED
48,1439,T20I,First innings,8,0,0,left_only,UNOBSERVED
49,1439,T20I,First innings,9,0,0,left_only,UNOBSERVED
66,1452,T20I,First innings,6,0,0,left_only,UNOBSERVED
67,1452,T20I,First innings,7,0,0,left_only,UNOBSERVED
68,1452,T20I,First innings,8,0,0,left_only,UNOBSERVED
69,1452,T20I,First innings,9,0,0,left_only,UNOBSERVED



PHASE 3G.4 FULL COVERAGE TABLE
Shape: (2720, 8)
Columns: ['comp_id', 'match_format', 'innings_population', 'wicket_state', 'delivery_records', 'matches', '_merge', 'coverage_status']


,comp_id,match_format,innings_population,wicket_state,delivery_records,matches,_merge,coverage_status
0,1331,T20I,First innings,0,101,5,both,OBSERVED
1,1331,T20I,First innings,1,98,5,both,OBSERVED
2,1331,T20I,First innings,2,135,5,both,OBSERVED
3,1331,T20I,First innings,3,107,5,both,OBSERVED
4,1331,T20I,First innings,4,63,5,both,OBSERVED
5,1331,T20I,First innings,5,49,4,both,OBSERVED
6,1331,T20I,First innings,6,20,3,both,OBSERVED
7,1331,T20I,First innings,7,33,2,both,OBSERVED
8,1331,T20I,First innings,8,22,2,both,OBSERVED
9,1331,T20I,First innings,9,1,1,both,OBSERVED



ROW COUNTS
Phase 3D missing-state rows: 255
Phase 3G.4 missing-state rows: 208

DISTINCT COMPETITION COUNTS
Phase 3D: 136
Phase 3G.4: 82

DISTINCT WICKET-STATE COUNTS
Phase 3G.4: 9


In [185]:
# PHASE 3G.5.4 — EXACT FIRST-INNINGS COVERAGE RECONCILIATION

# 1. Prepare Phase 3D missing-state keys
coverage_3D = phase3D_missing_comp_wicket_states[
    ["benchmark_population", "comp_id", "pre_delivery_wickets_fallen"]
].copy()

coverage_3D = coverage_3D.rename(columns={
    "benchmark_population": "match_format",
    "pre_delivery_wickets_fallen": "wicket_state"
})

# 2. Keep only first-innings missing states from Phase 3G.4
coverage_3G_first = phase3G_4_missing_competition_states.loc[
    phase3G_4_missing_competition_states["innings_population"]
    .eq("First innings"),
    ["comp_id", "match_format", "wicket_state"]
].copy()

# 3. Check duplicate keys before reconciliation
coverage_3D_duplicates = coverage_3D.duplicated(
    subset=["comp_id", "match_format", "wicket_state"]
).sum()

coverage_3G_duplicates = coverage_3G_first.duplicated(
    subset=["comp_id", "match_format", "wicket_state"]
).sum()

# 4. Full outer join on the common keys
coverage_reconciliation = coverage_3D.merge(
    coverage_3G_first,
    on=["comp_id", "match_format", "wicket_state"],
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_3D", "_3G")
)

coverage_reconciliation["reconciliation_status"] = (
    coverage_reconciliation["_merge"].map({
        "both": "PRESENT_IN_BOTH",
        "left_only": "PHASE_3D_ONLY",
        "right_only": "PHASE_3G4_ONLY"
    })
)

# 5. Summarize reconciliation
coverage_reconciliation_summary = (
    coverage_reconciliation["reconciliation_status"]
    .value_counts()
    .rename_axis("reconciliation_status")
    .reset_index(name="combinations")
)

print("RECONCILIATION SUMMARY")
display(coverage_reconciliation_summary)

print("\nDUPLICATE KEY CHECK")
display(pd.DataFrame([
    {"register": "Phase 3D", "duplicate_keys": int(coverage_3D_duplicates)},
    {"register": "Phase 3G.4 first innings", "duplicate_keys": int(coverage_3G_duplicates)}
]))

print("\nSTATE DOMAIN CHECK")
display(pd.DataFrame([
    {
        "register": "Phase 3D",
        "rows": len(coverage_3D),
        "minimum_state": coverage_3D["wicket_state"].min(),
        "maximum_state": coverage_3D["wicket_state"].max(),
        "distinct_states": coverage_3D["wicket_state"].nunique()
    },
    {
        "register": "Phase 3G.4 first innings",
        "rows": len(coverage_3G_first),
        "minimum_state": coverage_3G_first["wicket_state"].min(),
        "maximum_state": coverage_3G_first["wicket_state"].max(),
        "distinct_states": coverage_3G_first["wicket_state"].nunique()
    }
]))

print("\nPHASE 3D ONLY — FIRST 25")
display(
    coverage_reconciliation.loc[
        coverage_reconciliation["reconciliation_status"].eq("PHASE_3D_ONLY")
    ].head(25)
)

print("\nPHASE 3G.4 ONLY — FIRST 25")
display(
    coverage_reconciliation.loc[
        coverage_reconciliation["reconciliation_status"].eq("PHASE_3G4_ONLY")
    ].head(25)
)

print("\nFORMAT CONSISTENCY")
format_check = (
    coverage_3D[["comp_id", "match_format"]]
    .drop_duplicates()
    .merge(
        coverage_3G_first[["comp_id", "match_format"]].drop_duplicates(),
        on="comp_id",
        how="outer",
        suffixes=("_3D", "_3G"),
        indicator=True
    )
)
display(format_check["_merge"].value_counts().rename("competition_counts"))

# 6. Export reconciliation evidence
coverage_reconciliation.to_csv(
    "/content/phase3_outputs/phase3G_5_coverage_key_reconciliation.csv",
    index=False
)

coverage_reconciliation_summary.to_csv(
    "/content/phase3_outputs/phase3G_5_coverage_reconciliation_summary.csv",
    index=False
)

print("\nReconciliation files exported.")

RECONCILIATION SUMMARY


,reconciliation_status,combinations
0,PHASE_3D_ONLY,136
1,PRESENT_IN_BOTH,119
2,PHASE_3G4_ONLY,0



DUPLICATE KEY CHECK


,register,duplicate_keys
0,Phase 3D,0
1,Phase 3G.4 first innings,0



STATE DOMAIN CHECK


,register,rows,minimum_state,maximum_state,distinct_states
0,Phase 3D,255,5,10,6
1,Phase 3G.4 first innings,119,5,9,5


... (3 outputs left out)

,match_format,comp_id,wicket_state,_merge,reconciliation_status



FORMAT CONSISTENCY


,competition_counts
_merge,
left_only,73
both,63
right_only,0



Reconciliation files exported.


#### Formal reconciliation and audit record

In [187]:
# PHASE 3G.5.4 — FORMAL COVERAGE DISCREPANCY RESOLUTION

# Rebuild the common-key comparison from source registers.
audit_3D = phase3D_missing_comp_wicket_states[
    ["benchmark_population", "comp_id", "pre_delivery_wickets_fallen"]
].copy()

audit_3D = audit_3D.rename(columns={
    "benchmark_population": "match_format",
    "pre_delivery_wickets_fallen": "wicket_state"
})

audit_3G_first = phase3G_4_missing_competition_states.loc[
    phase3G_4_missing_competition_states["innings_population"]
    .eq("First innings"),
    ["comp_id", "match_format", "wicket_state"]
].copy()

audit_3D_keys = audit_3D[
    ["comp_id", "match_format", "wicket_state"]
].drop_duplicates()

audit_3G_keys = audit_3G_first[
    ["comp_id", "match_format", "wicket_state"]
].drop_duplicates()

audit_key_comparison = audit_3D_keys.merge(
    audit_3G_keys,
    on=["comp_id", "match_format", "wicket_state"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Partition Phase 3D-only rows by wicket state.
phase3D_only = audit_key_comparison.loc[
    audit_key_comparison["_merge"].eq("left_only")
].copy()

phase3G_only = audit_key_comparison.loc[
    audit_key_comparison["_merge"].eq("right_only")
].copy()

common_keys = audit_key_comparison.loc[
    audit_key_comparison["_merge"].eq("both")
].copy()

state10_only_count = int(
    phase3D_only["wicket_state"].eq(10).sum()
)

non10_only_count = int(
    (~phase3D_only["wicket_state"].eq(10)).sum()
)

# Resolution summary.
coverage_resolution = pd.DataFrame([
    {
        "measure": "Phase 3D original missing-state combinations",
        "count": len(audit_3D_keys),
        "interpretation": "Historical register; includes wicket state 10."
    },
    {
        "measure": "Phase 3G.4 first-innings missing-state combinations",
        "count": len(audit_3G_keys),
        "interpretation": "First-innings domain uses wicket states 0–9."
    },
    {
        "measure": "Matching combinations",
        "count": len(common_keys),
        "interpretation": "Same competition, format, and wicket-state keys."
    },
    {
        "measure": "Phase 3D-only state-10 combinations",
        "count": state10_only_count,
        "interpretation": "Outside the Phase 3G.4 first-innings state domain."
    },
    {
        "measure": "Phase 3D-only combinations at states other than 10",
        "count": non10_only_count,
        "interpretation": "Would indicate an unresolved mismatch if nonzero."
    },
    {
        "measure": "Phase 3G.4-only combinations",
        "count": len(phase3G_only),
        "interpretation": "Would indicate an unresolved mismatch if nonzero."
    }
])

# Formal discrepancy register.
coverage_discrepancy_register = pd.DataFrame([
    {
        "issue_id": "3G5_LIMIT_01",
        "issue": "Phase 3D versus Phase 3G.4 missing-state count",
        "historical_phase3D_count": len(audit_3D_keys),
        "phase3G4_first_innings_count": len(audit_3G_keys),
        "resolved_difference": len(phase3D_only),
        "root_cause": (
            "Phase 3D includes 136 competition/state combinations at "
            "wicket state 10; Phase 3G.4 first-innings coverage uses "
            "the observed state domain 0–9."
        ),
        "resolution": (
            "119 combinations match exactly. The 136 state-10 entries "
            "are outside the Phase 3G.4 first-innings coverage domain."
        ),
        "treatment": (
            "Retain both historical registers. Use 119 as the reconciled "
            "first-innings missing count for the 0–9 state domain. "
            "Do not describe 255 and 119 as directly comparable."
        ),
        "status": "RESOLVED_WITH_DOCUMENTED_DOMAIN_DIFFERENCE"
    }
])

# Explicit validation checks.
coverage_resolution_checks = pd.DataFrame([
    {
        "check_id": "3G5_R01",
        "check": "Phase 3D count reconciles to common plus Phase 3D-only",
        "passed": len(audit_3D_keys) == len(common_keys) + len(phase3D_only)
    },
    {
        "check_id": "3G5_R02",
        "check": "Phase 3G.4 count reconciles to common plus Phase 3G.4-only",
        "passed": len(audit_3G_keys) == len(common_keys) + len(phase3G_only)
    },
    {
        "check_id": "3G5_R03",
        "check": "All Phase 3D-only combinations are state 10",
        "passed": state10_only_count == len(phase3D_only)
    },
    {
        "check_id": "3G5_R04",
        "check": "No Phase 3D-only combinations remain at states 0–9",
        "passed": non10_only_count == 0
    },
    {
        "check_id": "3G5_R05",
        "check": "No Phase 3G.4-only combinations remain",
        "passed": len(phase3G_only) == 0
    },
    {
        "check_id": "3G5_R06",
        "check": "The common first-innings missing-state count is 119",
        "passed": len(common_keys) == 119
    },
    {
        "check_id": "3G5_R07",
        "check": "The Phase 3D state-10 difference is 136",
        "passed": state10_only_count == 136
    }
])

coverage_resolution_checks["status"] = (
    coverage_resolution_checks["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# Export without modifying source registers.
coverage_resolution.to_csv(
    "/content/phase3_outputs/phase3G_5_coverage_resolution_summary.csv",
    index=False
)

coverage_discrepancy_register.to_csv(
    "/content/phase3_outputs/phase3G_5_coverage_discrepancy_register.csv",
    index=False
)

coverage_resolution_checks.to_csv(
    "/content/phase3_outputs/phase3G_5_coverage_resolution_validation.csv",
    index=False
)

print("COVERAGE RESOLUTION SUMMARY")
display(coverage_resolution)

print("\nFORMAL DISCREPANCY REGISTER")
display(coverage_discrepancy_register)

print("\nRESOLUTION VALIDATION")
display(coverage_resolution_checks)

print("\nPASSED:", int(coverage_resolution_checks["passed"].sum()))
print("FAILED:", int((~coverage_resolution_checks["passed"]).sum()))

assert coverage_resolution_checks["passed"].all()

print("\nPHASE 3G.5.4 COVERAGE RECONCILIATION: PASSED")

COVERAGE RESOLUTION SUMMARY


,measure,count,interpretation
0,Phase 3D original missing-state combinations,255,Historical register; includes wicket state 10.
1,Phase 3G.4 first-innings missing-state combinations,119,First-innings domain uses wicket states 0–9.
2,Matching combinations,119,"Same competition, format, and wicket-state keys."
3,Phase 3D-only state-10 combinations,136,Outside the Phase 3G.4 first-innings state domain.
4,Phase 3D-only combinations at states other than 10,0,Would indicate an unresolved mismatch if nonzero.
5,Phase 3G.4-only combinations,0,Would indicate an unresolved mismatch if nonzero.



FORMAL DISCREPANCY REGISTER


,issue_id,issue,historical_phase3D_count,phase3G4_first_innings_count,resolved_difference,root_cause,resolution,treatment,status
0,3G5_LIMIT_01,Phase 3D versus Phase 3G.4 missing-state count,255,119,136,Phase 3D includes 136 competition/state combinations at wicket state 10; Phase 3G.4 first-innings coverage uses the observed state domain 0–9.,119 combinations match exactly. The 136 state-10 entries are outside the Phase 3G.4 first-innings coverage domain.,Retain both historical registers. Use 119 as the reconciled first-innings missing count for the 0–9 state domain. Do not describe 255 and 119 as directly comparable.,RESOLVED_WITH_DOCUMENTED_DOMAIN_DIFFERENCE



RESOLUTION VALIDATION


,check_id,check,passed,status
0,3G5_R01,Phase 3D count reconciles to common plus Phase 3D-only,True,PASSED
1,3G5_R02,Phase 3G.4 count reconciles to common plus Phase 3G.4-only,True,PASSED
2,3G5_R03,All Phase 3D-only combinations are state 10,True,PASSED
3,3G5_R04,No Phase 3D-only combinations remain at states 0–9,True,PASSED
4,3G5_R05,No Phase 3G.4-only combinations remain,True,PASSED
5,3G5_R06,The common first-innings missing-state count is 119,True,PASSED
6,3G5_R07,The Phase 3D state-10 difference is 136,True,PASSED



PASSED: 7
FAILED: 0

PHASE 3G.5.4 COVERAGE RECONCILIATION: PASSED


#### Consolidated limitation and decision audit

In [188]:
# PHASE 3G.5.5 — CONSOLIDATED LIMITATION AND DECISION AUDIT

phase3G_5_final_limitations = pd.DataFrame([
    {
        "limitation_id": "3G5_L01",
        "area": "Cumulative scoring",
        "status": "UNAVAILABLE",
        "evidence": "Runs and extras semantics remain unresolved.",
        "decision": "Do not construct cumulative innings score or scoring deviation."
    },
    {
        "limitation_id": "3G5_L02",
        "area": "Legal-ball progression",
        "status": "UNAVAILABLE",
        "evidence": "Delivery sequence fields are not validated legal-ball counters.",
        "decision": "Do not calculate balls remaining or resource progression."
    },
    {
        "limitation_id": "3G5_L03",
        "area": "Scheduled innings length",
        "status": "UNAVAILABLE",
        "evidence": "Revised-overs fields are sparse and units unresolved.",
        "decision": "Do not infer scheduled length from revised fields."
    },
    {
        "limitation_id": "3G5_L04",
        "area": "Innings stage",
        "status": "UNAVAILABLE",
        "evidence": "No validated stage boundaries or legal-ball progression.",
        "decision": "Do not infer powerplay, middle, or death stages."
    },
    {
        "limitation_id": "3G5_L05",
        "area": "Wicket states",
        "status": "CONDITIONAL",
        "evidence": "Reconstructed from recorded prior wicket events; continuity checks passed.",
        "decision": "Retain reconstruction provenance; not independently verified dismissal truth."
    },
    {
        "limitation_id": "3G5_L06",
        "area": "First-innings competition/state coverage",
        "status": "DESCRIPTIVE_ONLY",
        "evidence": "119 missing combinations in the reconciled 0–9 state domain.",
        "decision": "Retain competition-specific coverage limitations; do not invent minimum sample thresholds."
    },
    {
        "limitation_id": "3G5_L07",
        "area": "Phase 3D historical coverage register",
        "status": "RESOLVED_WITH_DOMAIN_DIFFERENCE",
        "evidence": "255 historical entries include 136 state-10 combinations outside the Phase 3G.4 first-innings domain.",
        "decision": "Preserve both registers and report their domains explicitly."
    },
    {
        "limitation_id": "3G5_L08",
        "area": "Second-innings target and score",
        "status": "UNVERIFIED_OR_UNAVAILABLE",
        "evidence": "Target field is sparse and unverified; cumulative score is unavailable.",
        "decision": "Do not calculate runs required or required run rate."
    },
    {
        "limitation_id": "3G5_L09",
        "area": "Fallback execution",
        "status": "POLICY_ONLY",
        "evidence": "Required benchmark inputs are unavailable.",
        "decision": "Do not execute pooling or publish unsupported estimates."
    },
    {
        "limitation_id": "3G5_L10",
        "area": "Numerical pressure scales",
        "status": "UNAVAILABLE",
        "evidence": "Neither innings-specific scale has validated component inputs.",
        "decision": "No numerical scale or unified cross-innings scale."
    },
    {
        "limitation_id": "3G5_L11",
        "area": "Weights, normalization, and thresholds",
        "status": "NOT_ESTABLISHED",
        "evidence": "No constructed score or empirical sensitivity evidence.",
        "decision": "Do not assign weights, normalization, bands, or thresholds."
    },
    {
        "limitation_id": "3G5_L12",
        "area": "Cross-format pooling",
        "status": "PROHIBITED_BY_DESIGN",
        "evidence": "IPL and T20I benchmark contexts must remain separate.",
        "decision": "No cross-format fallback or pooled benchmark."
    },
    {
        "limitation_id": "3G5_L13",
        "area": "Cross-innings comparability",
        "status": "NOT_ESTABLISHED",
        "evidence": "No validated common numerical construct or calibration.",
        "decision": "Retain separate innings-specific conceptual frameworks."
    }
])

phase3G_5_final_decision_checks = pd.DataFrame([
    {
        "check_id": "3G5_F01",
        "check": "All numerical scales remain unavailable",
        "passed": (
            phase3F5_current_decision.loc[
                phase3F5_current_decision["decision_area"].isin([
                    "First-innings numerical pressure scale",
                    "Second-innings numerical pressure scale",
                    "Unified cross-innings numerical scale"
                ]),
                "decision"
            ].eq("UNAVAILABLE").all()
        )
    },
    {
        "check_id": "3G5_F02",
        "check": "Wicket-state input remains conditional",
        "passed": (
            phase3F5_current_decision.loc[
                phase3F5_current_decision["decision_area"].eq(
                    "Wicket-state component"
                ),
                "decision"
            ].eq("CONDITIONAL").all()
        )
    },
    {
        "check_id": "3G5_F03",
        "check": "Pressure bands and thresholds remain unestablished",
        "passed": (
            phase3F5_current_decision.loc[
                phase3F5_current_decision["decision_area"].eq(
                    "Pressure bands and thresholds"
                ),
                "decision"
            ].eq("NOT_ESTABLISHED").all()
        )
    },
    {
        "check_id": "3G5_F04",
        "check": "First-innings missing coverage is reconciled",
        "passed": (
            len(common_keys) == 119
            and len(phase3D_only) == 136
            and non10_only_count == 0
            and len(phase3G_only) == 0
        )
    },
    {
        "check_id": "3G5_F05",
        "check": "No cross-format pooling is introduced",
        "passed": (
            phase3F5_current_decision["reason"].notna().all()
            and phase3G_5_final_limitations.loc[
                phase3G_5_final_limitations["limitation_id"].eq("3G5_L12"),
                "status"
            ].eq("PROHIBITED_BY_DESIGN").all()
        )
    },
    {
        "check_id": "3G5_F06",
        "check": "Fallback remains non-executable",
        "passed": (
            phase3D_fallback_policy.loc[
                phase3D_fallback_policy["priority"].lt(6), "status"
            ].eq("DEFINED_NOT_EXECUTABLE").all()
            and phase3E_fallback_policy.loc[
                phase3E_fallback_policy["priority"].lt(6),
                "execution_status"
            ].isin(["NOT_EXECUTABLE", "POLICY_DEFINED_ONLY"]).all()
        )
    }
])

phase3G_5_final_decision_checks["status"] = (
    phase3G_5_final_decision_checks["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

phase3G_5_final_limitations.to_csv(
    "/content/phase3_outputs/phase3G_5_final_limitation_register.csv",
    index=False
)

phase3G_5_final_decision_checks.to_csv(
    "/content/phase3_outputs/phase3G_5_final_decision_validation.csv",
    index=False
)

print("CONSOLIDATED LIMITATION REGISTER")
display(phase3G_5_final_limitations)

print("\nFINAL DECISION VALIDATION")
display(phase3G_5_final_decision_checks)

print("\nPASSED:", int(phase3G_5_final_decision_checks["passed"].sum()))
print("FAILED:", int((~phase3G_5_final_decision_checks["passed"]).sum()))

assert phase3G_5_final_decision_checks["passed"].all()

print("\nPHASE 3G.5.5 LIMITATION AUDIT: PASSED")

CONSOLIDATED LIMITATION REGISTER


,limitation_id,area,status,evidence,decision
0,3G5_L01,Cumulative scoring,UNAVAILABLE,Runs and extras semantics remain unresolved.,Do not construct cumulative innings score or scoring deviation.
1,3G5_L02,Legal-ball progression,UNAVAILABLE,Delivery sequence fields are not validated legal-ball counters.,Do not calculate balls remaining or resource progression.
2,3G5_L03,Scheduled innings length,UNAVAILABLE,Revised-overs fields are sparse and units unresolved.,Do not infer scheduled length from revised fields.
3,3G5_L04,Innings stage,UNAVAILABLE,No validated stage boundaries or legal-ball progression.,"Do not infer powerplay, middle, or death stages."
4,3G5_L05,Wicket states,CONDITIONAL,Reconstructed from recorded prior wicket events; continuity checks passed.,Retain reconstruction provenance; not independently verified dismissal truth.
5,3G5_L06,First-innings competition/state coverage,DESCRIPTIVE_ONLY,119 missing combinations in the reconciled 0–9 state domain.,Retain competition-specific coverage limitations; do not invent minimum sample thresholds.
6,3G5_L07,Phase 3D historical coverage register,RESOLVED_WITH_DOMAIN_DIFFERENCE,255 historical entries include 136 state-10 combinations outside the Phase 3G.4 first-innings domain.,Preserve both registers and report their domains explicitly.
7,3G5_L08,Second-innings target and score,UNVERIFIED_OR_UNAVAILABLE,Target field is sparse and unverified; cumulative score is unavailable.,Do not calculate runs required or required run rate.
8,3G5_L09,Fallback execution,POLICY_ONLY,Required benchmark inputs are unavailable.,Do not execute pooling or publish unsupported estimates.
9,3G5_L10,Numerical pressure scales,UNAVAILABLE,Neither innings-specific scale has validated component inputs.,No numerical scale or unified cross-innings scale.



FINAL DECISION VALIDATION


,check_id,check,passed,status
0,3G5_F01,All numerical scales remain unavailable,True,PASSED
1,3G5_F02,Wicket-state input remains conditional,True,PASSED
2,3G5_F03,Pressure bands and thresholds remain unestablished,True,PASSED
3,3G5_F04,First-innings missing coverage is reconciled,True,PASSED
4,3G5_F05,No cross-format pooling is introduced,True,PASSED
5,3G5_F06,Fallback remains non-executable,True,PASSED



PASSED: 6
FAILED: 0

PHASE 3G.5.5 LIMITATION AUDIT: PASSED


#### Subphase closure

In [189]:
# PHASE 3G.5.6 — SUBPHASE CLOSURE

from pathlib import Path

# Combine the three completed validation registers.
validation_1 = phase3G_5_consistency_checks.copy()
validation_2 = coverage_resolution_checks.copy()
validation_3 = phase3G_5_final_decision_checks.copy()

validation_1["audit_group"] = "Methodological consistency"
validation_2["audit_group"] = "Coverage discrepancy reconciliation"
validation_3["audit_group"] = "Final limitation and decision audit"

phase3G_5_final_validation = pd.concat(
    [validation_1, validation_2, validation_3],
    ignore_index=True,
    sort=False
)

phase3G_5_final_validation["status"] = (
    phase3G_5_final_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# Required output inventory.
required_outputs = [
    "phase3G_5_register_inventory.csv",
    "phase3G_5_object_checks.csv",
    "phase3G_5_policy_inspection.csv",
    "phase3G_5_fallback_comparison.csv",
    "phase3G_5_consistency_checks.csv",
    "phase3G_5_fallback_wording_notes.csv",
    "phase3G_5_coverage_key_reconciliation.csv",
    "phase3G_5_coverage_reconciliation_summary.csv",
    "phase3G_5_coverage_resolution_summary.csv",
    "phase3G_5_coverage_discrepancy_register.csv",
    "phase3G_5_coverage_resolution_validation.csv",
    "phase3G_5_final_limitation_register.csv",
    "phase3G_5_final_decision_validation.csv"
]

output_directory = Path("/content/phase3_outputs")

phase3G_5_output_audit = pd.DataFrame([
    {
        "file_name": file_name,
        "exists": (output_directory / file_name).is_file(),
        "file_size_bytes": (
            (output_directory / file_name).stat().st_size
            if (output_directory / file_name).is_file()
            else 0
        )
    }
    for file_name in required_outputs
])

# Summary.
total_checks = len(phase3G_5_final_validation)
passed_checks = int(
    phase3G_5_final_validation["passed"].sum()
)
failed_checks = total_checks - passed_checks

phase3G_5_completion_summary = pd.DataFrame([{
    "subphase": "Phase 3G.5 — Methodological decision and limitation robustness",
    "methodological_consistency_checks": len(validation_1),
    "coverage_reconciliation_checks": len(validation_2),
    "final_decision_checks": len(validation_3),
    "total_validation_checks": total_checks,
    "passed_checks": passed_checks,
    "failed_checks": failed_checks,
    "first_innings_missing_states_0_to_9": len(common_keys),
    "historical_phase3D_state10_difference": state10_only_count,
    "numerical_pressure_scales_available": False,
    "wicket_state_status": "CONDITIONAL",
    "fallback_execution_status": "POLICY_ONLY",
    "formal_status": (
        "COMPLETE"
        if failed_checks == 0
        and phase3G_5_output_audit["exists"].all()
        else "INCOMPLETE"
    )
}])

# Closure validation.
phase3G_5_closure_validation = pd.DataFrame([
    {
        "check_id": "3G5_CL01",
        "check": "All 21 methodological and limitation checks passed",
        "passed": passed_checks == 21 and failed_checks == 0
    },
    {
        "check_id": "3G5_CL02",
        "check": "All required output files exist",
        "passed": phase3G_5_output_audit["exists"].all()
    },
    {
        "check_id": "3G5_CL03",
        "check": "Coverage discrepancy has documented resolution",
        "passed": (
            len(coverage_discrepancy_register) == 1
            and coverage_discrepancy_register["status"].eq(
                "RESOLVED_WITH_DOCUMENTED_DOMAIN_DIFFERENCE"
            ).all()
        )
    },
    {
        "check_id": "3G5_CL04",
        "check": "No numerical pressure scale is approved",
        "passed": not phase3G_5_completion_summary[
            "numerical_pressure_scales_available"
        ].iloc[0]
    },
    {
        "check_id": "3G5_CL05",
        "check": "Wicket-state limitation remains conditional",
        "passed": (
            phase3G_5_completion_summary["wicket_state_status"].iloc[0]
            == "CONDITIONAL"
        )
    },
    {
        "check_id": "3G5_CL06",
        "check": "Fallback execution remains policy-only",
        "passed": (
            phase3G_5_completion_summary["fallback_execution_status"].iloc[0]
            == "POLICY_ONLY"
        )
    }
])

phase3G_5_closure_validation["status"] = (
    phase3G_5_closure_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

phase3G_5_completion_summary["formal_status"] = (
    "COMPLETE"
    if phase3G_5_closure_validation["passed"].all()
    else "INCOMPLETE"
)

# Export closure artifacts.
phase3G_5_final_validation.to_csv(
    output_directory / "phase3G_5_final_validation_register.csv",
    index=False
)

phase3G_5_output_audit.to_csv(
    output_directory / "phase3G_5_output_audit.csv",
    index=False
)

phase3G_5_completion_summary.to_csv(
    output_directory / "phase3G_5_completion_summary.csv",
    index=False
)

phase3G_5_closure_validation.to_csv(
    output_directory / "phase3G_5_closure_validation.csv",
    index=False
)

# Display.
print("COMBINED VALIDATION REGISTER")
display(phase3G_5_final_validation)

print("\nOUTPUT FILE AUDIT")
display(phase3G_5_output_audit)

print("\nCOMPLETION SUMMARY")
display(phase3G_5_completion_summary)

print("\nCLOSURE VALIDATION")
display(phase3G_5_closure_validation)

print("\nVALIDATION CHECKS:", total_checks)
print("PASSED:", passed_checks)
print("FAILED:", failed_checks)

assert phase3G_5_final_validation["passed"].all()
assert phase3G_5_output_audit["exists"].all()
assert phase3G_5_closure_validation["passed"].all()

print("\nPHASE 3G.5 CLOSURE: PASSED")

COMBINED VALIDATION REGISTER


,check_id,check,passed,interpretation,status,audit_group
0,3G5_C01,Phase 3D and 3E fallback priorities align,True,Both policies define the same six-level hierarchy.,PASSED,Methodological consistency
1,3G5_C02,Both fallback policies approve an unavailable terminal rule,True,Both policies terminate rather than manufacture unsupported estimates.,PASSED,Methodological consistency
2,3G5_C03,No numerical component is marked eligible,True,No Phase 3F.1 component is eligible for numerical scale construction.,PASSED,Methodological consistency
3,3G5_C04,All three numerical scale decisions remain unavailable,True,"First innings, second innings, and unified numerical scales remain unavailable.",PASSED,Methodological consistency
4,3G5_C05,Wicket components retain conditional status,True,Both innings retain reconstructed wicket-state provenance limitations.,PASSED,Methodological consistency
5,3G5_C06,Unavailable state inputs remain excluded,True,"Unavailable scoring, resource, schedule, and stage inputs are not silently promoted.",PASSED,Methodological consistency
6,3G5_C07,Shared conceptual architecture is not numerically eligible,True,Conceptual preference does not authorize numerical construction.,PASSED,Methodological consistency
7,3G5_C08,Pressure bands and thresholds remain unestablished,True,No unsupported pressure classifications are authorized.,PASSED,Methodological consistency
8,3G5_R01,Phase 3D count reconciles to common plus Phase 3D-only,True,NaN,PASSED,Coverage discrepancy reconciliation
9,3G5_R02,Phase 3G.4 count reconciles to common plus Phase 3G.4-only,True,NaN,PASSED,Coverage discrepancy reconciliation



OUTPUT FILE AUDIT


,file_name,exists,file_size_bytes
0,phase3G_5_register_inventory.csv,True,1197
1,phase3G_5_object_checks.csv,True,465
2,phase3G_5_policy_inspection.csv,True,2929
3,phase3G_5_fallback_comparison.csv,True,763
4,phase3G_5_consistency_checks.csv,True,565
5,phase3G_5_fallback_wording_notes.csv,True,373
6,phase3G_5_coverage_key_reconciliation.csv,True,9017
7,phase3G_5_coverage_reconciliation_summary.csv,True,90
8,phase3G_5_coverage_resolution_summary.csv,True,580
9,phase3G_5_coverage_discrepancy_register.csv,True,671



COMPLETION SUMMARY


,subphase,methodological_consistency_checks,coverage_reconciliation_checks,final_decision_checks,total_validation_checks,passed_checks,failed_checks,first_innings_missing_states_0_to_9,historical_phase3D_state10_difference,numerical_pressure_scales_available,wicket_state_status,fallback_execution_status,formal_status
0,Phase 3G.5 — Methodological decision and limitation robustness,8,7,6,21,21,0,119,136,False,CONDITIONAL,POLICY_ONLY,COMPLETE



CLOSURE VALIDATION


,check_id,check,passed,status
0,3G5_CL01,All 21 methodological and limitation checks passed,True,PASSED
1,3G5_CL02,All required output files exist,True,PASSED
2,3G5_CL03,Coverage discrepancy has documented resolution,True,PASSED
3,3G5_CL04,No numerical pressure scale is approved,True,PASSED
4,3G5_CL05,Wicket-state limitation remains conditional,True,PASSED
5,3G5_CL06,Fallback execution remains policy-only,True,PASSED



VALIDATION CHECKS: 21
PASSED: 21
FAILED: 0

PHASE 3G.5 CLOSURE: PASSED


### Validation limitations and acceptance criteria

#### Validation evidence inventory

In [190]:
# PHASE 3G.6.1 — VALIDATION EVIDENCE INVENTORY

phase3G_6_1_evidence_inventory = pd.DataFrame([
    {
        "evidence_id": "3G6_E01",
        "area": "Approved match population",
        "evidence": "813 approved matches reconcile to the integrated delivery population.",
        "result": "PASSED",
        "evidence_class": "STRUCTURAL_VERIFICATION",
        "limitation": "Confirms population and key reconciliation, not match-result truth."
    },
    {
        "evidence_id": "3G6_E02",
        "area": "Delivery-key integrity",
        "evidence": "192,235 integrated delivery records; no missing or duplicate delivery keys.",
        "result": "PASSED",
        "evidence_class": "STRUCTURAL_VERIFICATION",
        "limitation": "Does not verify the semantic correctness of delivery values."
    },
    {
        "evidence_id": "3G6_E03",
        "area": "Innings structure",
        "evidence": "813 matches have exactly two integrated regulation innings and consistent team reversal.",
        "result": "PASSED",
        "evidence_class": "STRUCTURAL_VERIFICATION",
        "limitation": "Super Over records are excluded from the primary integrated population."
    },
    {
        "evidence_id": "3G6_E04",
        "area": "Raw-to-integrated delivery reconciliation",
        "evidence": "26 raw-only delivery records were identified and classified as Super Over exclusions.",
        "result": "PASSED",
        "evidence_class": "RECONCILED_WITH_SOURCE_CONTEXT",
        "limitation": "Classification is supported by match-result metadata; these records remain outside the primary population."
    },
    {
        "evidence_id": "3G6_E05",
        "area": "Delivery sequence integrity",
        "evidence": "Sequence positions are continuous within the integrated innings.",
        "result": "PASSED",
        "evidence_class": "STRUCTURAL_VERIFICATION",
        "limitation": "Sequence positions are not validated legal-ball counters."
    },
    {
        "evidence_id": "3G6_E06",
        "area": "Pre-delivery wicket continuity",
        "evidence": "Reconstructed pre-delivery wicket states match cumulative prior recorded wicket indicators.",
        "result": "PASSED",
        "evidence_class": "CONDITIONAL_VERIFICATION",
        "limitation": "Recorded wicket indicators and dismissal truth were not independently verified."
    },
    {
        "evidence_id": "3G6_E07",
        "area": "Wickets remaining derivation",
        "evidence": "Second-innings wickets remaining equals 10 minus reconstructed wickets fallen; distribution reconciles.",
        "result": "PASSED",
        "evidence_class": "CONDITIONAL_ARITHMETIC_VERIFICATION",
        "limitation": "Inherits the conditional reliability of reconstructed wicket states."
    },
    {
        "evidence_id": "3G6_E08",
        "area": "First-innings Phase 3D alignment",
        "evidence": "Phase 3D first-innings states align with the full reconstructed first-innings state table.",
        "result": "PASSED",
        "evidence_class": "CROSS_TABLE_CONSISTENCY",
        "limitation": "Both tables rely on the same underlying reconstructed wicket evidence."
    },
    {
        "evidence_id": "3G6_E09",
        "area": "Second-innings Phase 3E alignment",
        "evidence": "Phase 3E second-innings wicket states align with the full reconstructed state table.",
        "result": "PASSED",
        "evidence_class": "CROSS_TABLE_CONSISTENCY",
        "limitation": "Agreement does not independently validate dismissal events."
    },
    {
        "evidence_id": "3G6_E10",
        "area": "Competition separation",
        "evidence": "T20 and T20I competition identities and formats were audited separately.",
        "result": "PASSED",
        "evidence_class": "STRUCTURAL_AND_COVERAGE_AUDIT",
        "limitation": "Does not establish benchmark reliability or justify cross-format pooling."
    },
    {
        "evidence_id": "3G6_E11",
        "area": "Competition/state coverage",
        "evidence": "First-innings missing coverage reconciled to 119 combinations in states 0–9.",
        "result": "PASSED",
        "evidence_class": "DESCRIPTIVE_COVERAGE_VERIFICATION",
        "limitation": "Coverage is uneven; 119 missing combinations remain."
    },
    {
        "evidence_id": "3G6_E12",
        "area": "Historical coverage discrepancy",
        "evidence": "255 Phase 3D entries reconcile to 119 shared entries and 136 state-10 entries.",
        "result": "PASSED",
        "evidence_class": "KEY_LEVEL_RECONCILIATION",
        "limitation": "The historical and newer registers use different state domains."
    },
    {
        "evidence_id": "3G6_E13",
        "area": "Cumulative scoring",
        "evidence": "Scoring field semantics and independent innings totals remain unresolved.",
        "result": "NOT_TESTABLE",
        "evidence_class": "BLOCKED_BY_INPUTS",
        "limitation": "Cumulative score and scoring deviation cannot be validated."
    },
    {
        "evidence_id": "3G6_E14",
        "area": "Legal-ball progression and resources",
        "evidence": "Legal-ball progression, scheduled innings length, and innings stage remain unvalidated.",
        "result": "NOT_TESTABLE",
        "evidence_class": "BLOCKED_BY_INPUTS",
        "limitation": "Balls remaining and resource-based pressure components cannot be constructed."
    },
    {
        "evidence_id": "3G6_E15",
        "area": "Second-innings chase state",
        "evidence": "Target semantics are unverified; cumulative score and legal balls remaining are unavailable.",
        "result": "NOT_TESTABLE",
        "evidence_class": "BLOCKED_BY_INPUTS",
        "limitation": "Runs required and required run rate cannot be validated."
    },
    {
        "evidence_id": "3G6_E16",
        "area": "Expected scoring benchmarks",
        "evidence": "Required benchmark-state dimensions and scoring outcomes are not available together.",
        "result": "NOT_TESTABLE",
        "evidence_class": "BLOCKED_BY_INPUTS",
        "limitation": "Neither innings-specific expected scoring benchmark can be constructed."
    },
    {
        "evidence_id": "3G6_E17",
        "area": "Fallback implementation",
        "evidence": "Fallback hierarchies are documented but their required benchmark inputs are unavailable.",
        "result": "POLICY_ONLY",
        "evidence_class": "DESIGN_AUDIT",
        "limitation": "No fallback estimates were executed."
    },
    {
        "evidence_id": "3G6_E18",
        "area": "Numerical pressure scale",
        "evidence": "Phase 3F decisions and Phase 3G.5 consistency checks retain all numerical scales as unavailable.",
        "result": "NOT_ESTABLISHED",
        "evidence_class": "METHODOLOGICAL_DECISION",
        "limitation": "No weights, normalization, numerical pressure bands, or thresholds are validated."
    },
    {
        "evidence_id": "3G6_E19",
        "area": "Cross-innings comparability",
        "evidence": "No common numerical construct or cross-innings calibration has been established.",
        "result": "NOT_TESTABLE",
        "evidence_class": "BLOCKED_BY_METHOD_AND_INPUTS",
        "limitation": "Equal numerical pressure values across innings cannot be interpreted."
    }
])

# Validate inventory structure and classifications.
allowed_classes = {
    "STRUCTURAL_VERIFICATION",
    "RECONCILED_WITH_SOURCE_CONTEXT",
    "CONDITIONAL_VERIFICATION",
    "CONDITIONAL_ARITHMETIC_VERIFICATION",
    "CROSS_TABLE_CONSISTENCY",
    "STRUCTURAL_AND_COVERAGE_AUDIT",
    "DESCRIPTIVE_COVERAGE_VERIFICATION",
    "KEY_LEVEL_RECONCILIATION",
    "BLOCKED_BY_INPUTS",
    "DESIGN_AUDIT",
    "METHODOLOGICAL_DECISION",
    "BLOCKED_BY_METHOD_AND_INPUTS"
}

phase3G_6_1_validation = pd.DataFrame([
    {
        "check_id": "3G6_1_V01",
        "check": "Evidence inventory contains 19 documented areas",
        "passed": len(phase3G_6_1_evidence_inventory) == 19
    },
    {
        "check_id": "3G6_1_V02",
        "check": "Evidence IDs are unique",
        "passed": phase3G_6_1_evidence_inventory["evidence_id"].is_unique
    },
    {
        "check_id": "3G6_1_V03",
        "check": "All evidence classes are recognized",
        "passed": phase3G_6_1_evidence_inventory["evidence_class"]
        .isin(allowed_classes).all()
    },
    {
        "check_id": "3G6_1_V04",
        "check": "Every evidence item has a limitation statement",
        "passed": (
            phase3G_6_1_evidence_inventory["limitation"]
            .notna().all()
            and phase3G_6_1_evidence_inventory["limitation"]
            .str.strip().ne("").all()
        )
    },
    {
        "check_id": "3G6_1_V05",
        "check": "Blocked and untestable areas are not marked as passed",
        "passed": (
            ~phase3G_6_1_evidence_inventory.loc[
                phase3G_6_1_evidence_inventory["evidence_class"].isin([
                    "BLOCKED_BY_INPUTS",
                    "BLOCKED_BY_METHOD_AND_INPUTS"
                ]),
                "result"
            ].eq("PASSED")
        ).any()
    }
])

phase3G_6_1_validation["status"] = (
    phase3G_6_1_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# Export.
phase3G_6_1_evidence_inventory.to_csv(
    "/content/phase3_outputs/phase3G_6_1_evidence_inventory.csv",
    index=False
)

phase3G_6_1_validation.to_csv(
    "/content/phase3_outputs/phase3G_6_1_validation.csv",
    index=False
)

# Display.
print("VALIDATION EVIDENCE INVENTORY")
display(phase3G_6_1_evidence_inventory)

print("\nEVIDENCE CLASS COUNTS")
display(
    phase3G_6_1_evidence_inventory
    .groupby(["evidence_class", "result"])
    .size()
    .reset_index(name="evidence_items")
)

print("\nINVENTORY VALIDATION")
display(phase3G_6_1_validation)

print("\nPASSED:", int(phase3G_6_1_validation["passed"].sum()))
print("FAILED:", int((~phase3G_6_1_validation["passed"]).sum()))

assert phase3G_6_1_validation["passed"].all()

print("\nPHASE 3G.6.1 EVIDENCE INVENTORY: PASSED")

VALIDATION EVIDENCE INVENTORY


,evidence_id,area,evidence,result,evidence_class,limitation
0,3G6_E01,Approved match population,813 approved matches reconcile to the integrated delivery population.,PASSED,STRUCTURAL_VERIFICATION,"Confirms population and key reconciliation, not match-result truth."
1,3G6_E02,Delivery-key integrity,"192,235 integrated delivery records; no missing or duplicate delivery keys.",PASSED,STRUCTURAL_VERIFICATION,Does not verify the semantic correctness of delivery values.
2,3G6_E03,Innings structure,813 matches have exactly two integrated regulation innings and consistent team reversal.,PASSED,STRUCTURAL_VERIFICATION,Super Over records are excluded from the primary integrated population.
3,3G6_E04,Raw-to-integrated delivery reconciliation,26 raw-only delivery records were identified and classified as Super Over exclusions.,PASSED,RECONCILED_WITH_SOURCE_CONTEXT,Classification is supported by match-result metadata; these records remain outside the primary population.
4,3G6_E05,Delivery sequence integrity,Sequence positions are continuous within the integrated innings.,PASSED,STRUCTURAL_VERIFICATION,Sequence positions are not validated legal-ball counters.
5,3G6_E06,Pre-delivery wicket continuity,Reconstructed pre-delivery wicket states match cumulative prior recorded wicket indicators.,PASSED,CONDITIONAL_VERIFICATION,Recorded wicket indicators and dismissal truth were not independently verified.
6,3G6_E07,Wickets remaining derivation,Second-innings wickets remaining equals 10 minus reconstructed wickets fallen; distribution reconciles.,PASSED,CONDITIONAL_ARITHMETIC_VERIFICATION,Inherits the conditional reliability of reconstructed wicket states.
7,3G6_E08,First-innings Phase 3D alignment,Phase 3D first-innings states align with the full reconstructed first-innings state table.,PASSED,CROSS_TABLE_CONSISTENCY,Both tables rely on the same underlying reconstructed wicket evidence.
8,3G6_E09,Second-innings Phase 3E alignment,Phase 3E second-innings wicket states align with the full reconstructed state table.,PASSED,CROSS_TABLE_CONSISTENCY,Agreement does not independently validate dismissal events.
9,3G6_E10,Competition separation,T20 and T20I competition identities and formats were audited separately.,PASSED,STRUCTURAL_AND_COVERAGE_AUDIT,Does not establish benchmark reliability or justify cross-format pooling.



EVIDENCE CLASS COUNTS


,evidence_class,result,evidence_items
0,BLOCKED_BY_INPUTS,NOT_TESTABLE,4
1,BLOCKED_BY_METHOD_AND_INPUTS,NOT_TESTABLE,1
2,CONDITIONAL_ARITHMETIC_VERIFICATION,PASSED,1
3,CONDITIONAL_VERIFICATION,PASSED,1
4,CROSS_TABLE_CONSISTENCY,PASSED,2
5,DESCRIPTIVE_COVERAGE_VERIFICATION,PASSED,1
6,DESIGN_AUDIT,POLICY_ONLY,1
7,KEY_LEVEL_RECONCILIATION,PASSED,1
8,METHODOLOGICAL_DECISION,NOT_ESTABLISHED,1
9,RECONCILED_WITH_SOURCE_CONTEXT,PASSED,1



INVENTORY VALIDATION


,check_id,check,passed,status
0,3G6_1_V01,Evidence inventory contains 19 documented areas,True,PASSED
1,3G6_1_V02,Evidence IDs are unique,True,PASSED
2,3G6_1_V03,All evidence classes are recognized,True,PASSED
3,3G6_1_V04,Every evidence item has a limitation statement,True,PASSED
4,3G6_1_V05,Blocked and untestable areas are not marked as passed,True,PASSED



PASSED: 5
FAILED: 0

PHASE 3G.6.1 EVIDENCE INVENTORY: PASSED


#### Verified, conditional, and untestable claims

In [191]:
# PHASE 3G.6.2 — VERIFIED / CONDITIONAL / UNTESTABLE CLAIMS

phase3G_6_2_claims = phase3G_6_1_evidence_inventory.copy()

claim_classification = {
    "3G6_E01": "VERIFIED_WITHIN_SCOPE",
    "3G6_E02": "VERIFIED_WITHIN_SCOPE",
    "3G6_E03": "VERIFIED_WITHIN_SCOPE",
    "3G6_E04": "VERIFIED_WITH_SOURCE_CONTEXT",
    "3G6_E05": "VERIFIED_WITHIN_SCOPE",
    "3G6_E06": "CONDITIONAL",
    "3G6_E07": "CONDITIONAL",
    "3G6_E08": "CROSS_TABLE_CONSISTENCY_ONLY",
    "3G6_E09": "CROSS_TABLE_CONSISTENCY_ONLY",
    "3G6_E10": "DESCRIPTIVE_ONLY",
    "3G6_E11": "DESCRIPTIVE_ONLY",
    "3G6_E12": "RECONCILED_WITH_DOMAIN_LIMITATION",
    "3G6_E13": "UNTESTABLE",
    "3G6_E14": "UNTESTABLE",
    "3G6_E15": "UNTESTABLE",
    "3G6_E16": "UNTESTABLE",
    "3G6_E17": "POLICY_ONLY",
    "3G6_E18": "NOT_ESTABLISHED",
    "3G6_E19": "UNTESTABLE"
}

phase3G_6_2_claims["claim_classification"] = (
    phase3G_6_2_claims["evidence_id"].map(claim_classification)
)

phase3G_6_2_claims["acceptance_interpretation"] = (
    phase3G_6_2_claims["claim_classification"].map({
        "VERIFIED_WITHIN_SCOPE":
            "Accept only the stated structural or arithmetic claim.",
        "VERIFIED_WITH_SOURCE_CONTEXT":
            "Accept the reconciliation with the documented source context.",
        "CONDITIONAL":
            "Accept only while retaining the reconstruction limitation.",
        "CROSS_TABLE_CONSISTENCY_ONLY":
            "Accept agreement between tables, not independent source truth.",
        "DESCRIPTIVE_ONLY":
            "Accept as coverage description, not benchmark reliability.",
        "RECONCILED_WITH_DOMAIN_LIMITATION":
            "Accept the reconciliation only within the documented state domain.",
        "UNTESTABLE":
            "Do not accept until the missing inputs or independent evidence are supplied.",
        "POLICY_ONLY":
            "Accept the documented policy, not its execution.",
        "NOT_ESTABLISHED":
            "Do not claim that the numerical or methodological construct is validated."
    })
)

# Count classifications.
phase3G_6_2_classification_summary = (
    phase3G_6_2_claims
    .groupby("claim_classification")
    .size()
    .reset_index(name="evidence_items")
    .sort_values("claim_classification")
    .reset_index(drop=True)
)

# Validate classification completeness and consistency.
phase3G_6_2_validation = pd.DataFrame([
    {
        "check_id": "3G6_2_V01",
        "check": "All 19 evidence items have a classification",
        "passed": (
            len(phase3G_6_2_claims) == 19
            and phase3G_6_2_claims["claim_classification"].notna().all()
        )
    },
    {
        "check_id": "3G6_2_V02",
        "check": "Evidence IDs remain unique",
        "passed": phase3G_6_2_claims["evidence_id"].is_unique
    },
    {
        "check_id": "3G6_2_V03",
        "check": "All classifications have acceptance interpretations",
        "passed": phase3G_6_2_claims["acceptance_interpretation"].notna().all()
    },
    {
        "check_id": "3G6_2_V04",
        "check": "All input-blocked items remain untestable",
        "passed": (
            phase3G_6_2_claims.loc[
                phase3G_6_2_claims["evidence_class"].isin([
                    "BLOCKED_BY_INPUTS",
                    "BLOCKED_BY_METHOD_AND_INPUTS"
                ]),
                "claim_classification"
            ].eq("UNTESTABLE").all()
        )
    },
    {
        "check_id": "3G6_2_V05",
        "check": "Conditional wicket evidence is not promoted to independent verification",
        "passed": (
            phase3G_6_2_claims.loc[
                phase3G_6_2_claims["evidence_id"].isin([
                    "3G6_E06", "3G6_E07"
                ]),
                "claim_classification"
            ].eq("CONDITIONAL").all()
        )
    },
    {
        "check_id": "3G6_2_V06",
        "check": "No numerical pressure scale is classified as verified",
        "passed": (
            phase3G_6_2_claims.loc[
                phase3G_6_2_claims["evidence_id"].eq("3G6_E18"),
                "claim_classification"
            ].eq("NOT_ESTABLISHED").all()
        )
    }
])

phase3G_6_2_validation["status"] = (
    phase3G_6_2_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# Export.
phase3G_6_2_claims.to_csv(
    "/content/phase3_outputs/phase3G_6_2_claim_classification.csv",
    index=False
)

phase3G_6_2_classification_summary.to_csv(
    "/content/phase3_outputs/phase3G_6_2_classification_summary.csv",
    index=False
)

phase3G_6_2_validation.to_csv(
    "/content/phase3_outputs/phase3G_6_2_validation.csv",
    index=False
)

print("CLAIM CLASSIFICATION")
display(
    phase3G_6_2_claims[
        [
            "evidence_id",
            "area",
            "claim_classification",
            "acceptance_interpretation"
        ]
    ]
)

print("\nCLASSIFICATION SUMMARY")
display(phase3G_6_2_classification_summary)

print("\nVALIDATION")
display(phase3G_6_2_validation)

print("\nPASSED:", int(phase3G_6_2_validation["passed"].sum()))
print("FAILED:", int((~phase3G_6_2_validation["passed"]).sum()))

assert phase3G_6_2_validation["passed"].all()

print("\nPHASE 3G.6.2 CLAIM CLASSIFICATION: PASSED")

CLAIM CLASSIFICATION


,evidence_id,area,claim_classification,acceptance_interpretation
0,3G6_E01,Approved match population,VERIFIED_WITHIN_SCOPE,Accept only the stated structural or arithmetic claim.
1,3G6_E02,Delivery-key integrity,VERIFIED_WITHIN_SCOPE,Accept only the stated structural or arithmetic claim.
2,3G6_E03,Innings structure,VERIFIED_WITHIN_SCOPE,Accept only the stated structural or arithmetic claim.
3,3G6_E04,Raw-to-integrated delivery reconciliation,VERIFIED_WITH_SOURCE_CONTEXT,Accept the reconciliation with the documented source context.
4,3G6_E05,Delivery sequence integrity,VERIFIED_WITHIN_SCOPE,Accept only the stated structural or arithmetic claim.
5,3G6_E06,Pre-delivery wicket continuity,CONDITIONAL,Accept only while retaining the reconstruction limitation.
6,3G6_E07,Wickets remaining derivation,CONDITIONAL,Accept only while retaining the reconstruction limitation.
7,3G6_E08,First-innings Phase 3D alignment,CROSS_TABLE_CONSISTENCY_ONLY,"Accept agreement between tables, not independent source truth."
8,3G6_E09,Second-innings Phase 3E alignment,CROSS_TABLE_CONSISTENCY_ONLY,"Accept agreement between tables, not independent source truth."
9,3G6_E10,Competition separation,DESCRIPTIVE_ONLY,"Accept as coverage description, not benchmark reliability."



CLASSIFICATION SUMMARY


,claim_classification,evidence_items
0,CONDITIONAL,2
1,CROSS_TABLE_CONSISTENCY_ONLY,2
2,DESCRIPTIVE_ONLY,2
3,NOT_ESTABLISHED,1
4,POLICY_ONLY,1
5,RECONCILED_WITH_DOMAIN_LIMITATION,1
6,UNTESTABLE,5
7,VERIFIED_WITHIN_SCOPE,4
8,VERIFIED_WITH_SOURCE_CONTEXT,1



VALIDATION


,check_id,check,passed,status
0,3G6_2_V01,All 19 evidence items have a classification,True,PASSED
1,3G6_2_V02,Evidence IDs remain unique,True,PASSED
2,3G6_2_V03,All classifications have acceptance interpretations,True,PASSED
3,3G6_2_V04,All input-blocked items remain untestable,True,PASSED
4,3G6_2_V05,Conditional wicket evidence is not promoted to independent verification,True,PASSED
5,3G6_2_V06,No numerical pressure scale is classified as verified,True,PASSED



PASSED: 6
FAILED: 0

PHASE 3G.6.2 CLAIM CLASSIFICATION: PASSED


#### Acceptance criteria and rejection conditions

In [192]:
# PHASE 3G.6.3 — ACCEPTANCE CRITERIA AND REJECTION CONDITIONS

phase3G_6_3_acceptance_criteria = pd.DataFrame([
    {
        "criterion_id": "3G6_A01",
        "acceptance_area": "Source population",
        "required_evidence": "Approved match population and delivery keys reconcile without unexplained exclusions.",
        "current_status": "PASSED_WITHIN_SCOPE",
        "acceptance_rule": "Retain population reconciliation and documented Super Over exclusions."
    },
    {
        "criterion_id": "3G6_A02",
        "acceptance_area": "Cumulative scoring",
        "required_evidence": "Validated batter runs, extras semantics, cumulative score, and independent innings-total reconciliation.",
        "current_status": "BLOCKED",
        "acceptance_rule": "Reject any scoring benchmark built from unresolved runs or extras fields."
    },
    {
        "criterion_id": "3G6_A03",
        "acceptance_area": "Legal-ball progression",
        "required_evidence": "Validated legal-delivery classification and independently reconciled legal-ball counts.",
        "current_status": "BLOCKED",
        "acceptance_rule": "Reject any use of delivery sequence positions as legal-ball counts."
    },
    {
        "criterion_id": "3G6_A04",
        "acceptance_area": "Scheduled innings length",
        "required_evidence": "Validated scheduled length, units, and treatment of revised innings.",
        "current_status": "BLOCKED",
        "acceptance_rule": "Reject inferred schedules from sparse revised-overs fields."
    },
    {
        "criterion_id": "3G6_A05",
        "acceptance_area": "Innings stage",
        "required_evidence": "Validated innings progression and defensible stage boundaries.",
        "current_status": "BLOCKED",
        "acceptance_rule": "Reject stage classification inferred only from over or delivery sequence."
    },
    {
        "criterion_id": "3G6_A06",
        "acceptance_area": "Wicket-state reliability",
        "required_evidence": "Independent dismissal-event reconciliation and documented treatment of exceptional dismissals.",
        "current_status": "CONDITIONAL",
        "acceptance_rule": "Do not describe reconstructed wicket states as independently verified until source-event validation is completed."
    },
    {
        "criterion_id": "3G6_A07",
        "acceptance_area": "Competition-specific coverage",
        "required_evidence": "Coverage and sample-depth reporting for each competition and comparable state.",
        "current_status": "DESCRIPTIVE_ONLY",
        "acceptance_rule": "Do not treat observed state presence as proof of benchmark reliability."
    },
    {
        "criterion_id": "3G6_A08",
        "acceptance_area": "Expected scoring benchmarks",
        "required_evidence": "Validated scoring outcomes and all required comparable-state dimensions.",
        "current_status": "BLOCKED",
        "acceptance_rule": "Reject benchmark estimates when required state dimensions or scoring outcomes are unavailable."
    },
    {
        "criterion_id": "3G6_A09",
        "acceptance_area": "Adaptive fallback",
        "required_evidence": "Executable fallback rules, documented provenance, sample support, and stability assessment.",
        "current_status": "POLICY_ONLY",
        "acceptance_rule": "Reject silent pooling, unsupported expansion, or estimates that fail the unavailable terminal rule."
    },
    {
        "criterion_id": "3G6_A10",
        "acceptance_area": "Pressure direction and normalization",
        "required_evidence": "Validated component measurements, justified directionality, and defensible normalization.",
        "current_status": "NOT_ESTABLISHED",
        "acceptance_rule": "Reject arbitrary weights, normalization, or a composite pressure score without empirical support."
    },
    {
        "criterion_id": "3G6_A11",
        "acceptance_area": "Cross-format stability",
        "required_evidence": "Separate T20 and T20I evaluation and evidence supporting any proposed comparison.",
        "current_status": "POLICY_DEFINED",
        "acceptance_rule": "Do not pool T20 and T20I benchmarks by default."
    },
    {
        "criterion_id": "3G6_A12",
        "acceptance_area": "Cross-innings comparability",
        "required_evidence": "Validated innings-specific measures and empirical cross-innings calibration.",
        "current_status": "NOT_ESTABLISHED",
        "acceptance_rule": "Reject claims that equal values represent equal pressure across innings without calibration."
    },
    {
        "criterion_id": "3G6_A13",
        "acceptance_area": "Pressure bands and thresholds",
        "required_evidence": "Validated numerical measure and empirically justified classification boundaries.",
        "current_status": "NOT_ESTABLISHED",
        "acceptance_rule": "Reject invented pressure bands, rankings, or thresholds."
    },
    {
        "criterion_id": "3G6_A14",
        "acceptance_area": "Outcome leakage",
        "required_evidence": "Pre-delivery feature boundary and explicit exclusion of current-delivery and post-match outcomes.",
        "current_status": "SCHEMA_RULE_ESTABLISHED",
        "acceptance_rule": "Reject any pressure input that uses information unavailable immediately before the delivery."
    }
])

phase3G_6_3_rejection_conditions = pd.DataFrame([
    {
        "rejection_id": "3G6_RJ01",
        "rejection_condition": "Unresolved runs or extras fields are summed to create cumulative score.",
        "severity": "CRITICAL",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ02",
        "rejection_condition": "Delivery sequence fields are treated as validated legal-ball counters.",
        "severity": "CRITICAL",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ03",
        "rejection_condition": "Sparse revised-overs fields are treated as validated scheduled innings lengths.",
        "severity": "CRITICAL",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ04",
        "rejection_condition": "Reconstructed wicket states are presented as independently verified dismissal truth.",
        "severity": "HIGH",
        "decision": "REJECT_OR_QUALIFY"
    },
    {
        "rejection_id": "3G6_RJ05",
        "rejection_condition": "Missing competition/state observations are silently imputed or pooled.",
        "severity": "HIGH",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ06",
        "rejection_condition": "T20 and T20I benchmarks are pooled without justified validation.",
        "severity": "HIGH",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ07",
        "rejection_condition": "Fallback estimates are published without valid inputs and stability checks.",
        "severity": "CRITICAL",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ08",
        "rejection_condition": "Arbitrary weights, normalization, pressure bands, or thresholds are introduced.",
        "severity": "CRITICAL",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ09",
        "rejection_condition": "Current-delivery or post-match outcomes enter pre-delivery pressure inputs.",
        "severity": "CRITICAL",
        "decision": "REJECT"
    },
    {
        "rejection_id": "3G6_RJ10",
        "rejection_condition": "Equal numerical values across innings are interpreted as equal pressure without calibration.",
        "severity": "HIGH",
        "decision": "REJECT"
    }
])

# Validation.
phase3G_6_3_validation = pd.DataFrame([
    {
        "check_id": "3G6_3_V01",
        "check": "All 14 acceptance areas are documented",
        "passed": len(phase3G_6_3_acceptance_criteria) == 14
    },
    {
        "check_id": "3G6_3_V02",
        "check": "Acceptance criterion IDs are unique",
        "passed": phase3G_6_3_acceptance_criteria["criterion_id"].is_unique
    },
    {
        "check_id": "3G6_3_V03",
        "check": "All 10 rejection conditions are documented",
        "passed": len(phase3G_6_3_rejection_conditions) == 10
    },
    {
        "check_id": "3G6_3_V04",
        "check": "Rejection condition IDs are unique",
        "passed": phase3G_6_3_rejection_conditions["rejection_id"].is_unique
    },
    {
        "check_id": "3G6_3_V05",
        "check": "Blocked scoring and resource criteria remain blocked",
        "passed": (
            phase3G_6_3_acceptance_criteria.loc[
                phase3G_6_3_acceptance_criteria["criterion_id"].isin([
                    "3G6_A02", "3G6_A03", "3G6_A04", "3G6_A05", "3G6_A08"
                ]),
                "current_status"
            ].eq("BLOCKED").all()
        )
    },
    {
        "check_id": "3G6_3_V06",
        "check": "No numerical pressure scale is treated as accepted",
        "passed": (
            phase3G_6_3_acceptance_criteria.loc[
                phase3G_6_3_acceptance_criteria["criterion_id"].isin([
                    "3G6_A10", "3G6_A12", "3G6_A13"
                ]),
                "current_status"
            ].eq("NOT_ESTABLISHED").all()
        )
    },
    {
        "check_id": "3G6_3_V07",
        "check": "Pre-delivery leakage rejection is explicit",
        "passed": (
            phase3G_6_3_rejection_conditions.loc[
                phase3G_6_3_rejection_conditions["rejection_id"].eq("3G6_RJ09"),
                "decision"
            ].eq("REJECT").all()
        )
    }
])

phase3G_6_3_validation["status"] = (
    phase3G_6_3_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# Export.
phase3G_6_3_acceptance_criteria.to_csv(
    "/content/phase3_outputs/phase3G_6_3_acceptance_criteria.csv",
    index=False
)

phase3G_6_3_rejection_conditions.to_csv(
    "/content/phase3_outputs/phase3G_6_3_rejection_conditions.csv",
    index=False
)

phase3G_6_3_validation.to_csv(
    "/content/phase3_outputs/phase3G_6_3_validation.csv",
    index=False
)

print("ACCEPTANCE CRITERIA")
display(phase3G_6_3_acceptance_criteria)

print("\nREJECTION CONDITIONS")
display(phase3G_6_3_rejection_conditions)

print("\nVALIDATION")
display(phase3G_6_3_validation)

print("\nPASSED:", int(phase3G_6_3_validation["passed"].sum()))
print("FAILED:", int((~phase3G_6_3_validation["passed"]).sum()))

assert phase3G_6_3_validation["passed"].all()

print("\nPHASE 3G.6.3 ACCEPTANCE CRITERIA: PASSED")

ACCEPTANCE CRITERIA


,criterion_id,acceptance_area,required_evidence,current_status,acceptance_rule
0,3G6_A01,Source population,Approved match population and delivery keys reconcile without unexplained exclusions.,PASSED_WITHIN_SCOPE,Retain population reconciliation and documented Super Over exclusions.
1,3G6_A02,Cumulative scoring,"Validated batter runs, extras semantics, cumulative score, and independent innings-total reconciliation.",BLOCKED,Reject any scoring benchmark built from unresolved runs or extras fields.
2,3G6_A03,Legal-ball progression,Validated legal-delivery classification and independently reconciled legal-ball counts.,BLOCKED,Reject any use of delivery sequence positions as legal-ball counts.
3,3G6_A04,Scheduled innings length,"Validated scheduled length, units, and treatment of revised innings.",BLOCKED,Reject inferred schedules from sparse revised-overs fields.
4,3G6_A05,Innings stage,Validated innings progression and defensible stage boundaries.,BLOCKED,Reject stage classification inferred only from over or delivery sequence.
5,3G6_A06,Wicket-state reliability,Independent dismissal-event reconciliation and documented treatment of exceptional dismissals.,CONDITIONAL,Do not describe reconstructed wicket states as independently verified until source-event validation is completed.
6,3G6_A07,Competition-specific coverage,Coverage and sample-depth reporting for each competition and comparable state.,DESCRIPTIVE_ONLY,Do not treat observed state presence as proof of benchmark reliability.
7,3G6_A08,Expected scoring benchmarks,Validated scoring outcomes and all required comparable-state dimensions.,BLOCKED,Reject benchmark estimates when required state dimensions or scoring outcomes are unavailable.
8,3G6_A09,Adaptive fallback,"Executable fallback rules, documented provenance, sample support, and stability assessment.",POLICY_ONLY,"Reject silent pooling, unsupported expansion, or estimates that fail the unavailable terminal rule."
9,3G6_A10,Pressure direction and normalization,"Validated component measurements, justified directionality, and defensible normalization.",NOT_ESTABLISHED,"Reject arbitrary weights, normalization, or a composite pressure score without empirical support."



REJECTION CONDITIONS


,rejection_id,rejection_condition,severity,decision
0,3G6_RJ01,Unresolved runs or extras fields are summed to create cumulative score.,CRITICAL,REJECT
1,3G6_RJ02,Delivery sequence fields are treated as validated legal-ball counters.,CRITICAL,REJECT
2,3G6_RJ03,Sparse revised-overs fields are treated as validated scheduled innings lengths.,CRITICAL,REJECT
3,3G6_RJ04,Reconstructed wicket states are presented as independently verified dismissal truth.,HIGH,REJECT_OR_QUALIFY
4,3G6_RJ05,Missing competition/state observations are silently imputed or pooled.,HIGH,REJECT
5,3G6_RJ06,T20 and T20I benchmarks are pooled without justified validation.,HIGH,REJECT
6,3G6_RJ07,Fallback estimates are published without valid inputs and stability checks.,CRITICAL,REJECT
7,3G6_RJ08,"Arbitrary weights, normalization, pressure bands, or thresholds are introduced.",CRITICAL,REJECT
8,3G6_RJ09,Current-delivery or post-match outcomes enter pre-delivery pressure inputs.,CRITICAL,REJECT
9,3G6_RJ10,Equal numerical values across innings are interpreted as equal pressure without calibration.,HIGH,REJECT



VALIDATION


,check_id,check,passed,status
0,3G6_3_V01,All 14 acceptance areas are documented,True,PASSED
1,3G6_3_V02,Acceptance criterion IDs are unique,True,PASSED
2,3G6_3_V03,All 10 rejection conditions are documented,True,PASSED
3,3G6_3_V04,Rejection condition IDs are unique,True,PASSED
4,3G6_3_V05,Blocked scoring and resource criteria remain blocked,True,PASSED
5,3G6_3_V06,No numerical pressure scale is treated as accepted,True,PASSED
6,3G6_3_V07,Pre-delivery leakage rejection is explicit,True,PASSED



PASSED: 7
FAILED: 0

PHASE 3G.6.3 ACCEPTANCE CRITERIA: PASSED


#### Consolidated limitation register

In [193]:
# PHASE 3G.6.4 — CONSOLIDATED LIMITATION REGISTER

phase3G_6_4_limitations = pd.DataFrame([
    {
        "limitation_id": "3G6_L01",
        "area": "Cumulative scoring",
        "current_status": "UNAVAILABLE",
        "evidence_basis": "Source runs and extras semantics remain unresolved; no independent innings-total reconciliation.",
        "impact": "First-innings scoring position and second-innings runs required cannot be constructed reliably.",
        "resolution_evidence": "Validated scoring-field definitions and independent innings totals."
    },
    {
        "limitation_id": "3G6_L02",
        "area": "Legal-ball progression",
        "current_status": "UNAVAILABLE",
        "evidence_basis": "Delivery sequence fields are not validated legal-ball counters; unresolved legal-ball evidence remains.",
        "impact": "Remaining legal balls and resource progression cannot be calculated.",
        "resolution_evidence": "Validated legal-delivery classification and reconciled legal-ball counts."
    },
    {
        "limitation_id": "3G6_L03",
        "area": "Scheduled innings length",
        "current_status": "UNAVAILABLE",
        "evidence_basis": "Revised-overs fields are sparse and their units are unresolved.",
        "impact": "Scheduled resources and comparable innings lengths cannot be established.",
        "resolution_evidence": "Authoritative schedule definitions and validated revised-innings treatment."
    },
    {
        "limitation_id": "3G6_L04",
        "area": "Innings stage",
        "current_status": "UNAVAILABLE",
        "evidence_basis": "Legal-ball progression and scheduled innings length are not validated.",
        "impact": "Defensible stage-based benchmark grouping is unavailable.",
        "resolution_evidence": "Validated progression and documented stage boundaries."
    },
    {
        "limitation_id": "3G6_L05",
        "area": "Pre-delivery wicket states",
        "current_status": "CONDITIONAL",
        "evidence_basis": "Reconstructed states reconcile internally with prior recorded wicket indicators.",
        "impact": "Wicket-based state dimensions inherit source dismissal-event uncertainty.",
        "resolution_evidence": "Independent dismissal-event reconciliation and exceptional-dismissal review."
    },
    {
        "limitation_id": "3G6_L06",
        "area": "First-innings competition/state coverage",
        "current_status": "DESCRIPTIVE_ONLY",
        "evidence_basis": "119 unobserved competition/state combinations in the 0-9 wicket-fallen domain.",
        "impact": "Coverage gaps prevent assuming every competition has evidence for every state.",
        "resolution_evidence": "Additional valid observations or a separately validated fallback approach."
    },
    {
        "limitation_id": "3G6_L07",
        "area": "Second-innings target and chase state",
        "current_status": "UNVERIFIED_OR_UNAVAILABLE",
        "evidence_basis": "Target values are sparse and unverified; cumulative score is unavailable.",
        "impact": "Runs required and required run rate cannot be constructed.",
        "resolution_evidence": "Verified target semantics and validated cumulative second-innings scoring."
    },
    {
        "limitation_id": "3G6_L08",
        "area": "Expected scoring benchmarks",
        "current_status": "BLOCKED",
        "evidence_basis": "Required scoring, resource, schedule, and stage dimensions are unavailable.",
        "impact": "Neither innings-specific expected scoring benchmark can be constructed.",
        "resolution_evidence": "Validated inputs and comparable-state benchmark construction."
    },
    {
        "limitation_id": "3G6_L09",
        "area": "Adaptive fallback execution",
        "current_status": "POLICY_ONLY",
        "evidence_basis": "Fallback hierarchy is documented but its required state dimensions are not executable.",
        "impact": "No fallback benchmark estimates are approved.",
        "resolution_evidence": "Executable fallback logic with provenance, sample support, and stability checks."
    },
    {
        "limitation_id": "3G6_L10",
        "area": "Numerical pressure scales",
        "current_status": "UNAVAILABLE",
        "evidence_basis": "Core innings-specific measures and normalization inputs are blocked.",
        "impact": "No numerical first-innings, second-innings, or unified scale is accepted.",
        "resolution_evidence": "Validated components, justified normalization, and empirical validation."
    },
    {
        "limitation_id": "3G6_L11",
        "area": "Weights and normalization",
        "current_status": "NOT_ESTABLISHED",
        "evidence_basis": "No empirically supported component weights or normalization procedure has been established.",
        "impact": "A composite pressure score cannot be justified.",
        "resolution_evidence": "A documented and empirically evaluated combination methodology."
    },
    {
        "limitation_id": "3G6_L12",
        "area": "Pressure bands and thresholds",
        "current_status": "NOT_ESTABLISHED",
        "evidence_basis": "No validated numerical scale or empirical classification boundaries exist.",
        "impact": "Pressure categories and numerical thresholds cannot be assigned.",
        "resolution_evidence": "Validated scale and empirically justified classification boundaries."
    },
    {
        "limitation_id": "3G6_L13",
        "area": "Cross-format pooling",
        "current_status": "PROHIBITED_BY_DESIGN",
        "evidence_basis": "T20 and T20I benchmark populations are to remain separate.",
        "impact": "Cross-format fallback or pooling is not permitted by the current methodology.",
        "resolution_evidence": "A separately justified and empirically validated change to the methodology."
    },
    {
        "limitation_id": "3G6_L14",
        "area": "Cross-innings comparability",
        "current_status": "NOT_ESTABLISHED",
        "evidence_basis": "No validated innings-specific measures or cross-innings calibration exist.",
        "impact": "Equal values cannot be interpreted as equal pressure across innings.",
        "resolution_evidence": "Separate validated measures and empirical calibration."
    },
    {
        "limitation_id": "3G6_L15",
        "area": "Outcome leakage",
        "current_status": "SCHEMA_RULE_ESTABLISHED",
        "evidence_basis": "The pre-delivery feature boundary and prohibited outcome fields are documented.",
        "impact": "Any use of current-delivery or post-match outcomes would invalidate pre-delivery pressure claims.",
        "resolution_evidence": "Maintain and test the pre-delivery feature boundary in any future implementation."
    }
])

phase3G_6_4_validation = pd.DataFrame([
    {
        "check_id": "3G6_4_V01",
        "check": "All 15 consolidated limitations are documented",
        "passed": len(phase3G_6_4_limitations) == 15
    },
    {
        "check_id": "3G6_4_V02",
        "check": "Limitation IDs are unique",
        "passed": phase3G_6_4_limitations["limitation_id"].is_unique
    },
    {
        "check_id": "3G6_4_V03",
        "check": "Every limitation has evidence basis, impact, and resolution evidence",
        "passed": phase3G_6_4_limitations[
            ["evidence_basis", "impact", "resolution_evidence"]
        ].notna().all().all()
    },
    {
        "check_id": "3G6_4_V04",
        "check": "Scoring and resource limitations remain blocked or unavailable",
        "passed": phase3G_6_4_limitations.loc[
            phase3G_6_4_limitations["limitation_id"].isin(
                ["3G6_L01", "3G6_L02", "3G6_L03", "3G6_L04", "3G6_L08"]
            ),
            "current_status"
        ].isin(["UNAVAILABLE", "BLOCKED"]).all()
    },
    {
        "check_id": "3G6_4_V05",
        "check": "Wicket-state reliability remains conditional",
        "passed": phase3G_6_4_limitations.loc[
            phase3G_6_4_limitations["limitation_id"].eq("3G6_L05"),
            "current_status"
        ].eq("CONDITIONAL").all()
    },
    {
        "check_id": "3G6_4_V06",
        "check": "Cross-format pooling remains prohibited",
        "passed": phase3G_6_4_limitations.loc[
            phase3G_6_4_limitations["limitation_id"].eq("3G6_L13"),
            "current_status"
        ].eq("PROHIBITED_BY_DESIGN").all()
    },
    {
        "check_id": "3G6_4_V07",
        "check": "Outcome leakage prohibition remains explicit",
        "passed": phase3G_6_4_limitations.loc[
            phase3G_6_4_limitations["limitation_id"].eq("3G6_L15"),
            "current_status"
        ].eq("SCHEMA_RULE_ESTABLISHED").all()
    }
])

phase3G_6_4_validation["status"] = (
    phase3G_6_4_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

phase3G_6_4_limitations.to_csv(
    "/content/phase3_outputs/phase3G_6_4_consolidated_limitation_register.csv",
    index=False
)

phase3G_6_4_validation.to_csv(
    "/content/phase3_outputs/phase3G_6_4_validation.csv",
    index=False
)

print("CONSOLIDATED LIMITATION REGISTER")
display(phase3G_6_4_limitations)

print("\nVALIDATION")
display(phase3G_6_4_validation)

print("\nPASSED:", int(phase3G_6_4_validation["passed"].sum()))
print("FAILED:", int((~phase3G_6_4_validation["passed"]).sum()))

assert phase3G_6_4_validation["passed"].all()

print("\nPHASE 3G.6.4 LIMITATION REGISTER: PASSED")

CONSOLIDATED LIMITATION REGISTER


,limitation_id,area,current_status,evidence_basis,impact,resolution_evidence
0,3G6_L01,Cumulative scoring,UNAVAILABLE,Source runs and extras semantics remain unresolved; no independent innings-total reconciliation.,First-innings scoring position and second-innings runs required cannot be constructed reliably.,Validated scoring-field definitions and independent innings totals.
1,3G6_L02,Legal-ball progression,UNAVAILABLE,Delivery sequence fields are not validated legal-ball counters; unresolved legal-ball evidence remains.,Remaining legal balls and resource progression cannot be calculated.,Validated legal-delivery classification and reconciled legal-ball counts.
2,3G6_L03,Scheduled innings length,UNAVAILABLE,Revised-overs fields are sparse and their units are unresolved.,Scheduled resources and comparable innings lengths cannot be established.,Authoritative schedule definitions and validated revised-innings treatment.
3,3G6_L04,Innings stage,UNAVAILABLE,Legal-ball progression and scheduled innings length are not validated.,Defensible stage-based benchmark grouping is unavailable.,Validated progression and documented stage boundaries.
4,3G6_L05,Pre-delivery wicket states,CONDITIONAL,Reconstructed states reconcile internally with prior recorded wicket indicators.,Wicket-based state dimensions inherit source dismissal-event uncertainty.,Independent dismissal-event reconciliation and exceptional-dismissal review.
5,3G6_L06,First-innings competition/state coverage,DESCRIPTIVE_ONLY,119 unobserved competition/state combinations in the 0-9 wicket-fallen domain.,Coverage gaps prevent assuming every competition has evidence for every state.,Additional valid observations or a separately validated fallback approach.
6,3G6_L07,Second-innings target and chase state,UNVERIFIED_OR_UNAVAILABLE,Target values are sparse and unverified; cumulative score is unavailable.,Runs required and required run rate cannot be constructed.,Verified target semantics and validated cumulative second-innings scoring.
7,3G6_L08,Expected scoring benchmarks,BLOCKED,"Required scoring, resource, schedule, and stage dimensions are unavailable.",Neither innings-specific expected scoring benchmark can be constructed.,Validated inputs and comparable-state benchmark construction.
8,3G6_L09,Adaptive fallback execution,POLICY_ONLY,Fallback hierarchy is documented but its required state dimensions are not executable.,No fallback benchmark estimates are approved.,"Executable fallback logic with provenance, sample support, and stability checks."
9,3G6_L10,Numerical pressure scales,UNAVAILABLE,Core innings-specific measures and normalization inputs are blocked.,"No numerical first-innings, second-innings, or unified scale is accepted.","Validated components, justified normalization, and empirical validation."



VALIDATION


,check_id,check,passed,status
0,3G6_4_V01,All 15 consolidated limitations are documented,True,PASSED
1,3G6_4_V02,Limitation IDs are unique,True,PASSED
2,3G6_4_V03,"Every limitation has evidence basis, impact, and resolution evidence",True,PASSED
3,3G6_4_V04,Scoring and resource limitations remain blocked or unavailable,True,PASSED
4,3G6_4_V05,Wicket-state reliability remains conditional,True,PASSED
5,3G6_4_V06,Cross-format pooling remains prohibited,True,PASSED
6,3G6_4_V07,Outcome leakage prohibition remains explicit,True,PASSED



PASSED: 7
FAILED: 0

PHASE 3G.6.4 LIMITATION REGISTER: PASSED


#### Closure

In [195]:
# PHASE 3G.6 — CORRECT FAILED CLOSURE CHECK

phase3G_6_closure_validation.loc[
    phase3G_6_closure_validation["check_id"].eq("3G6_C06"),
    "passed"
] = (
    not bool(
        phase3G_6_closure_summary.loc[
            0, "numerical_pressure_scale_accepted"
        ]
    )
)

phase3G_6_closure_validation["status"] = (
    phase3G_6_closure_validation["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

phase3G_6_closure_validation.to_csv(
    "/content/phase3_outputs/phase3G_6_closure_validation.csv",
    index=False
)

print("CORRECTED CLOSURE VALIDATION")
display(phase3G_6_closure_validation)

print(
    "PASSED:",
    int(phase3G_6_closure_validation["passed"].sum())
)
print(
    "FAILED:",
    int((~phase3G_6_closure_validation["passed"]).sum())
)

assert phase3G_6_validation_register["passed"].all()
assert phase3G_6_closure_validation["passed"].all()

print("\nPHASE 3G.6 CLOSURE: PASSED")

CORRECTED CLOSURE VALIDATION


,check_id,check,passed,status
0,3G6_C01,All four 3G.6 validation registers are available,True,PASSED
1,3G6_C02,All component validation checks passed,True,PASSED
2,3G6_C03,All 19 evidence items are classified,True,PASSED
3,3G6_C04,All acceptance criteria and rejection conditions are retained,True,PASSED
4,3G6_C05,All consolidated limitations are retained,True,PASSED
5,3G6_C06,No numerical pressure scale is accepted,True,PASSED
6,3G6_C07,Conditional wicket and policy-only fallback limitations remain,True,PASSED


PASSED: 7
FAILED: 0

PHASE 3G.6 CLOSURE: PASSED


#### Consolidated report and closure

In [196]:
# PHASE 3G.7 — CONSOLIDATED REPORT AND FINAL CLOSURE

from pathlib import Path
from datetime import datetime

phase3G_output_dir = Path("/content/phase3_outputs")
phase3G_output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Subphase completion register
# ---------------------------------------------------------

phase3G_subphase_register = pd.DataFrame([
    {
        "subphase": "3G.1",
        "title": "Scope, objectives and validation-readiness audit",
        "status": "COMPLETE",
        "decision": "Validation scope and object readiness documented."
    },
    {
        "subphase": "3G.2",
        "title": "Source population and structural robustness",
        "status": "COMPLETE",
        "decision": "813 approved matches reconciled; 26 Super Over records documented as expected exclusions."
    },
    {
        "subphase": "3G.3",
        "title": "Delivery alignment and reconstructed-state robustness",
        "status": "COMPLETE",
        "decision": "Delivery/state alignment and arithmetic consistency passed; wicket-event truth remains conditional."
    },
    {
        "subphase": "3G.4",
        "title": "Competition and coverage robustness",
        "status": "COMPLETE",
        "decision": "Format separation passed; coverage gaps and thin competition-state support documented."
    },
    {
        "subphase": "3G.5",
        "title": "Methodological decision and limitation robustness",
        "status": "COMPLETE",
        "decision": "Methodological consistency and coverage-domain discrepancy reconciled."
    },
    {
        "subphase": "3G.6",
        "title": "Validation limitations and acceptance criteria",
        "status": "COMPLETE",
        "decision": "Claims classified; acceptance criteria, rejection conditions, and limitations consolidated."
    }
])

# ---------------------------------------------------------
# 2. Final decision register
# ---------------------------------------------------------

phase3G_final_decisions = pd.DataFrame([
    {
        "decision_area": "Approved match population",
        "final_status": "VERIFIED_WITHIN_SCOPE",
        "decision": "813-match approved population retained."
    },
    {
        "decision_area": "Delivery structure and keys",
        "final_status": "VERIFIED_WITHIN_SCOPE",
        "decision": "Integrated regulation-delivery keys and innings structure passed structural checks."
    },
    {
        "decision_area": "Super Over handling",
        "final_status": "RECONCILED",
        "decision": "26 raw-only delivery records across four matches classified as Super Over exclusions."
    },
    {
        "decision_area": "Pre-delivery wicket states",
        "final_status": "CONDITIONAL",
        "decision": "Internally consistent with recorded wicket indicators; independent dismissal truth not established."
    },
    {
        "decision_area": "Wickets remaining",
        "final_status": "CONDITIONAL_ARITHMETIC_VERIFICATION",
        "decision": "The 10 minus wickets-fallen derivation reconciles; input reliability remains conditional."
    },
    {
        "decision_area": "Competition separation",
        "final_status": "PASSED_WITH_COVERAGE_LIMITATIONS",
        "decision": "T20 and T20I remain separate; 208 competition-state combinations are unobserved across both innings."
    },
    {
        "decision_area": "Cumulative scoring",
        "final_status": "UNAVAILABLE",
        "decision": "Source scoring semantics and independent innings totals remain unresolved."
    },
    {
        "decision_area": "Legal-ball progression",
        "final_status": "UNAVAILABLE",
        "decision": "Sequence fields are not approved as legal-ball counters."
    },
    {
        "decision_area": "Scheduled innings length and stage",
        "final_status": "UNAVAILABLE",
        "decision": "Sparse revised fields and sequence positions do not establish schedule or stage."
    },
    {
        "decision_area": "Expected scoring benchmarks",
        "final_status": "BLOCKED",
        "decision": "Required scoring and resource inputs are unavailable."
    },
    {
        "decision_area": "Adaptive fallback",
        "final_status": "POLICY_ONLY",
        "decision": "Fallback hierarchy documented; execution not approved."
    },
    {
        "decision_area": "Numerical pressure scales",
        "final_status": "UNAVAILABLE",
        "decision": "No first-innings, second-innings, or unified numerical scale accepted."
    },
    {
        "decision_area": "Weights, normalization, bands and thresholds",
        "final_status": "NOT_ESTABLISHED",
        "decision": "No empirically justified composite methodology or pressure thresholds established."
    },
    {
        "decision_area": "Cross-innings comparability",
        "final_status": "NOT_ESTABLISHED",
        "decision": "No empirical calibration supporting equal-pressure interpretation across innings."
    },
    {
        "decision_area": "Outcome leakage",
        "final_status": "SCHEMA_RULE_ESTABLISHED",
        "decision": "Pre-delivery feature boundary and prohibited outcome fields documented."
    }
])

# ---------------------------------------------------------
# 3. Validation consolidation
# ---------------------------------------------------------

phase3G_validation_sources = [
    ("3G.6 component checks", phase3G_6_validation_register),
    ("3G.6 closure checks", phase3G_6_closure_validation)
]

phase3G_validation_rollup = []

for validation_name, validation_df in phase3G_validation_sources:
    phase3G_validation_rollup.append({
        "validation_group": validation_name,
        "checks": len(validation_df),
        "passed": int(validation_df["passed"].sum()),
        "failed": int((~validation_df["passed"]).sum())
    })

phase3G_validation_rollup = pd.DataFrame(phase3G_validation_rollup)

phase3G_total_checks = int(
    phase3G_validation_rollup["checks"].sum()
)
phase3G_passed_checks = int(
    phase3G_validation_rollup["passed"].sum()
)
phase3G_failed_checks = int(
    phase3G_validation_rollup["failed"].sum()
)

# ---------------------------------------------------------
# 4. Final completion summary
# ---------------------------------------------------------

phase3G_completion_summary = pd.DataFrame([{
    "phase": "Phase 3G",
    "title": "Robustness and validation",
    "subphases_completed": len(phase3G_subphase_register),
    "approved_matches": 813,
    "integrated_regulation_deliveries": 192235,
    "first_innings_deliveries": 100333,
    "second_innings_deliveries": 91902,
    "confirmed_super_over_exclusions": 26,
    "competition_state_combinations_unobserved": 208,
    "first_innings_missing_states_0_to_9": 119,
    "evidence_items_classified": 19,
    "acceptance_criteria": 14,
    "rejection_conditions": 10,
    "consolidated_limitations": 15,
    "validation_checks_in_final_rollup": phase3G_total_checks,
    "passed_checks_in_final_rollup": phase3G_passed_checks,
    "failed_checks_in_final_rollup": phase3G_failed_checks,
    "numerical_pressure_scale_accepted": False,
    "fallback_execution": "POLICY_ONLY",
    "wicket_state_reliability": "CONDITIONAL",
    "phase_status": (
        "COMPLETE"
        if phase3G_failed_checks == 0
        else "REQUIRES_REVIEW"
    )
}])

# ---------------------------------------------------------
# 5. Consolidated report
# ---------------------------------------------------------

phase3G_report_path = (
    phase3G_output_dir / "SupeRR_Selector_S2_Phase_3G_Report.md"
)

phase3G_report = f"""# SupeRR Selector Season 2
# Phase 3G — Robustness and Validation

## 1. Purpose and scope

Phase 3G audited the structural, reconstructed-state, competition,
coverage, methodological, and validation limitations of the Phase 3
pressure-state work.

The phase did not construct player metrics, pressure benchmarks,
numerical pressure scores, rankings, or pressure bands.

## 2. Subphase completion

{phase3G_subphase_register.to_markdown(index=False)}

## 3. Population and structural findings

- Approved match population: 813 matches.
- Integrated regulation-delivery population: 192,235 records.
- First-innings delivery records: 100,333.
- Second-innings delivery records: 91,902.
- All integrated delivery keys passed the documented structural checks.
- All 813 matches contain two integrated regulation innings.
- 26 raw-only records across four matches were reconciled as Super Over
  exclusions using match-result context.
- Delivery sequence continuity does not establish legal-ball progression.

## 4. Reconstructed-state findings

- Pre-delivery wicket-state continuity reconciles with prior recorded
  wicket indicators.
- Wickets remaining reconciles arithmetically as 10 minus reconstructed
  pre-delivery wickets fallen.
- These checks establish internal consistency, not independent
  verification of dismissal-event truth.
- First-innings and second-innings phase tables reconcile with the
  corresponding full reconstructed-state records.

## 5. Competition and coverage findings

- T20 population: 362 matches.
- T20I population: 451 matches.
- T20 and T20I remain separate benchmark populations.
- Across both innings, 208 competition-by-wicket-state combinations
  are unobserved.
- In the first-innings 0–9 wicket-fallen domain, 119 combinations
  are unobserved.
- Thin single-match support and unequal competition sample sizes remain
  limitations; no minimum sample threshold was invented.

## 6. Methodological findings

- First-innings and second-innings measures remain separate.
- Cross-format fallback is prohibited by the current methodology.
- Fallback hierarchy is documented but not executable with the current
  inputs.
- Required-rate conditioning is not approved for the initial expected
  scoring-rate benchmark.
- No numerical weights, normalization, composite scale, pressure bands,
  or thresholds are established.
- Pre-delivery information boundaries and outcome-leakage prohibitions
  remain mandatory.

## 7. Validation limitations

The following inputs remain unavailable or unverified:

1. Cumulative innings scoring.
2. Legal-ball progression and remaining legal balls.
3. Scheduled innings length and revised-innings treatment.
4. Innings-stage classification.
5. Verified second-innings target and cumulative chase score.
6. Executable expected scoring benchmarks.
7. Empirically validated fallback estimates.
8. Independently verified wicket-state source truth.

## 8. Final decision

No numerical first-innings, second-innings, or unified pressure scale
is accepted.

The shared conceptual framework may be retained with limitations.
Wicket-state reliability remains conditional. Fallback execution remains
policy-only. Pressure bands and thresholds are not established.

Structural and arithmetic consistency checks must not be represented
as proof of source semantic correctness or predictive validity.

## 9. Validation rollup

- Checks included in this final rollup: {phase3G_total_checks}
- Passed: {phase3G_passed_checks}
- Failed: {phase3G_failed_checks}

## 10. Phase decision

**Phase 3G status: {
    "COMPLETE" if phase3G_failed_checks == 0 else "REQUIRES REVIEW"
}**

This is completion of the robustness and validation audit only.
It is not approval of a numerical pressure model or authorization
to proceed to player-level modelling.
"""

with open(phase3G_report_path, "w", encoding="utf-8") as report_file:
    report_file.write(phase3G_report)

# ---------------------------------------------------------
# 6. Output audit
# ---------------------------------------------------------

phase3G_expected_outputs = [
    "SupeRR_Selector_S2_Phase_3G_Report.md",
    "phase3G_subphase_completion_register.csv",
    "phase3G_final_decision_register.csv",
    "phase3G_validation_rollup.csv",
    "phase3G_completion_summary.csv",
    "phase3G_final_validation_register.csv"
]

phase3G_output_audit = pd.DataFrame([
    {
        "file_name": file_name,
        "exists": (phase3G_output_dir / file_name).is_file(),
        "file_size_bytes": (
            (phase3G_output_dir / file_name).stat().st_size
            if (phase3G_output_dir / file_name).is_file()
            else None
        )
    }
    for file_name in phase3G_expected_outputs
])

# ---------------------------------------------------------
# 7. Final validation register
# ---------------------------------------------------------

phase3G_final_validation_register = pd.DataFrame([
    {
        "check_id": "3G7_V01",
        "check": "All six Phase 3G subphases are recorded complete",
        "passed": (
            len(phase3G_subphase_register) == 6
            and phase3G_subphase_register["status"].eq("COMPLETE").all()
        )
    },
    {
        "check_id": "3G7_V02",
        "check": "Final decision register contains all 15 decision areas",
        "passed": len(phase3G_final_decisions) == 15
    },
    {
        "check_id": "3G7_V03",
        "check": "Final validation rollup has no failed checks",
        "passed": phase3G_failed_checks == 0
    },
    {
        "check_id": "3G7_V04",
        "check": "Numerical pressure scale remains unaccepted",
        "passed": not bool(
            phase3G_completion_summary.loc[
                0, "numerical_pressure_scale_accepted"
            ]
        )
    },
    {
        "check_id": "3G7_V05",
        "check": "Wicket-state reliability remains conditional",
        "passed": (
            phase3G_completion_summary.loc[
                0, "wicket_state_reliability"
            ] == "CONDITIONAL"
        )
    },
    {
        "check_id": "3G7_V06",
        "check": "Fallback execution remains policy-only",
        "passed": (
            phase3G_completion_summary.loc[
                0, "fallback_execution"
            ] == "POLICY_ONLY"
        )
    },
    {
        "check_id": "3G7_V07",
        "check": "All six final output files exist",
        "passed": phase3G_output_audit["exists"].all()
    },
    {
        "check_id": "3G7_V08",
        "check": "Report file is non-empty",
        "passed": (
            (phase3G_output_dir / "SupeRR_Selector_S2_Phase_3G_Report.md").is_file()
            and (phase3G_output_dir / "SupeRR_Selector_S2_Phase_3G_Report.md").stat().st_size > 0
        )
    }
])

phase3G_final_validation_register["status"] = (
    phase3G_final_validation_register["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

# ---------------------------------------------------------
# 8. Export final registers
# ---------------------------------------------------------

phase3G_subphase_register.to_csv(
    phase3G_output_dir / "phase3G_subphase_completion_register.csv",
    index=False
)

phase3G_final_decisions.to_csv(
    phase3G_output_dir / "phase3G_final_decision_register.csv",
    index=False
)

phase3G_validation_rollup.to_csv(
    phase3G_output_dir / "phase3G_validation_rollup.csv",
    index=False
)

phase3G_completion_summary.to_csv(
    phase3G_output_dir / "phase3G_completion_summary.csv",
    index=False
)

phase3G_output_audit.to_csv(
    phase3G_output_dir / "phase3G_final_output_audit.csv",
    index=False
)

phase3G_final_validation_register.to_csv(
    phase3G_output_dir / "phase3G_final_validation_register.csv",
    index=False
)

# Refresh output audit after all CSV exports.
phase3G_output_audit = pd.DataFrame([
    {
        "file_name": file_name,
        "exists": (phase3G_output_dir / file_name).is_file(),
        "file_size_bytes": (
            (phase3G_output_dir / file_name).stat().st_size
            if (phase3G_output_dir / file_name).is_file()
            else None
        )
    }
    for file_name in phase3G_expected_outputs
])

phase3G_output_audit.to_csv(
    phase3G_output_dir / "phase3G_final_output_audit.csv",
    index=False
)

# Recheck output existence after exports.
phase3G_final_validation_register.loc[
    phase3G_final_validation_register["check_id"].eq("3G7_V07"),
    "passed"
] = phase3G_output_audit["exists"].all()

phase3G_final_validation_register["status"] = (
    phase3G_final_validation_register["passed"]
    .map({True: "PASSED", False: "FAILED"})
)

phase3G_final_validation_register.to_csv(
    phase3G_output_dir / "phase3G_final_validation_register.csv",
    index=False
)

# ---------------------------------------------------------
# 9. Display and assertions
# ---------------------------------------------------------

print("PHASE 3G SUBPHASE COMPLETION")
display(phase3G_subphase_register)

print("\nFINAL DECISION REGISTER")
display(phase3G_final_decisions)

print("\nVALIDATION ROLLUP")
display(phase3G_validation_rollup)

print("\nCOMPLETION SUMMARY")
display(phase3G_completion_summary)

print("\nOUTPUT AUDIT")
display(phase3G_output_audit)

print("\nFINAL VALIDATION")
display(phase3G_final_validation_register)

print(
    "\nFINAL CHECKS PASSED:",
    int(phase3G_final_validation_register["passed"].sum()),
    "/",
    len(phase3G_final_validation_register)
)

print(
    "FINAL CHECKS FAILED:",
    int((~phase3G_final_validation_register["passed"]).sum())
)

assert phase3G_final_validation_register["passed"].all()

print("\nPHASE 3G.7 CONSOLIDATED REPORT AND CLOSURE: PASSED")

PHASE 3G SUBPHASE COMPLETION


,subphase,title,status,decision
0,3G.1,"Scope, objectives and validation-readiness audit",COMPLETE,Validation scope and object readiness documented.
1,3G.2,Source population and structural robustness,COMPLETE,813 approved matches reconciled; 26 Super Over records documented as expected exclusions.
2,3G.3,Delivery alignment and reconstructed-state robustness,COMPLETE,Delivery/state alignment and arithmetic consistency passed; wicket-event truth remains conditional.
3,3G.4,Competition and coverage robustness,COMPLETE,Format separation passed; coverage gaps and thin competition-state support documented.
4,3G.5,Methodological decision and limitation robustness,COMPLETE,Methodological consistency and coverage-domain discrepancy reconciled.
5,3G.6,Validation limitations and acceptance criteria,COMPLETE,"Claims classified; acceptance criteria, rejection conditions, and limitations consolidated."



FINAL DECISION REGISTER


,decision_area,final_status,decision
0,Approved match population,VERIFIED_WITHIN_SCOPE,813-match approved population retained.
1,Delivery structure and keys,VERIFIED_WITHIN_SCOPE,Integrated regulation-delivery keys and innings structure passed structural checks.
2,Super Over handling,RECONCILED,26 raw-only delivery records across four matches classified as Super Over exclusions.
3,Pre-delivery wicket states,CONDITIONAL,Internally consistent with recorded wicket indicators; independent dismissal truth not established.
4,Wickets remaining,CONDITIONAL_ARITHMETIC_VERIFICATION,The 10 minus wickets-fallen derivation reconciles; input reliability remains conditional.
5,Competition separation,PASSED_WITH_COVERAGE_LIMITATIONS,T20 and T20I remain separate; 208 competition-state combinations are unobserved across both innings.
6,Cumulative scoring,UNAVAILABLE,Source scoring semantics and independent innings totals remain unresolved.
7,Legal-ball progression,UNAVAILABLE,Sequence fields are not approved as legal-ball counters.
8,Scheduled innings length and stage,UNAVAILABLE,Sparse revised fields and sequence positions do not establish schedule or stage.
9,Expected scoring benchmarks,BLOCKED,Required scoring and resource inputs are unavailable.



VALIDATION ROLLUP


,validation_group,checks,passed,failed
0,3G.6 component checks,25,25,0
1,3G.6 closure checks,7,7,0


... (3 outputs left out)

,file_name,exists,file_size_bytes
0,SupeRR_Selector_S2_Phase_3G_Report.md,True,5096
1,phase3G_subphase_completion_register.csv,True,889
2,phase3G_final_decision_register.csv,True,1811
3,phase3G_validation_rollup.csv,True,94
4,phase3G_completion_summary.csv,True,651
5,phase3G_final_validation_register.csv,True,525



FINAL VALIDATION


,check_id,check,passed,status
0,3G7_V01,All six Phase 3G subphases are recorded complete,True,PASSED
1,3G7_V02,Final decision register contains all 15 decision areas,True,PASSED
2,3G7_V03,Final validation rollup has no failed checks,True,PASSED
3,3G7_V04,Numerical pressure scale remains unaccepted,True,PASSED
4,3G7_V05,Wicket-state reliability remains conditional,True,PASSED
5,3G7_V06,Fallback execution remains policy-only,True,PASSED
6,3G7_V07,All six final output files exist,True,PASSED
7,3G7_V08,Report file is non-empty,True,PASSED



FINAL CHECKS PASSED: 8 / 8
FINAL CHECKS FAILED: 0

PHASE 3G.7 CONSOLIDATED REPORT AND CLOSURE: PASSED


# Report and approval

In [197]:
# PHASE 3H.1 — EXPORT READINESS AUDIT

from pathlib import Path
import pandas as pd

phase3H_output_dir = Path("/content/phase3_outputs")
phase3H_output_dir.mkdir(parents=True, exist_ok=True)

# Core in-memory datasets required for handoff.
phase3H_required_objects = [
    "df_deliveries",
    "df_source_matches",
    "df_source_ball",
    "phase3D_first_innings_population",
    "phase3E_second_innings_wicket_alignment",
    "phase3G_final_decisions",
    "phase3G_completion_summary",
    "phase3G_final_validation_register",
    "phase3G_output_audit"
]

phase3H_object_inventory = []

for object_name in phase3H_required_objects:
    obj = globals().get(object_name)

    phase3H_object_inventory.append({
        "object_name": object_name,
        "available": isinstance(obj, pd.DataFrame),
        "rows": len(obj) if isinstance(obj, pd.DataFrame) else None,
        "columns": len(obj.columns) if isinstance(obj, pd.DataFrame) else None
    })

phase3H_object_inventory = pd.DataFrame(phase3H_object_inventory)

# Check the key existing exports without assuming they all exist.
phase3H_expected_files = [
    "phase2_integrated_cleaned_deliveries.parquet",
    "phase2_integrated_cleaned_deliveries.csv",
    "phase3D_first_innings_population.parquet",
    "SupeRR_Selector_S2_Phase_3G_Report.md",
    "phase3G_completion_summary.csv",
    "phase3G_final_decision_register.csv",
    "phase3G_final_validation_register.csv",
    "phase3G_final_output_audit.csv",
    "phase3G_subphase_completion_register.csv",
    "phase3G_validation_rollup.csv"
]

phase3H_file_inventory = pd.DataFrame([
    {
        "file_name": name,
        "exists": (phase3H_output_dir / name).is_file(),
        "size_bytes": (
            (phase3H_output_dir / name).stat().st_size
            if (phase3H_output_dir / name).is_file()
            else None
        )
    }
    for name in phase3H_expected_files
])

# Locate source CSVs in the common Colab locations.
phase3H_source_names = [
    "matches.csv",
    "playing_eleven.csv",
    "competition_team_player_mapping.csv",
    "teams.csv",
    "competitions.csv"
]

phase3H_source_inventory = []

for name in phase3H_source_names:
    candidates = [
        Path("/content") / name,
        Path("/content/drive/MyDrive") / name
    ]

    found = [str(p) for p in candidates if p.is_file()]

    phase3H_source_inventory.append({
        "source_file": name,
        "found": bool(found),
        "path": found[0] if found else None
    })

phase3H_source_inventory = pd.DataFrame(phase3H_source_inventory)

print("IN-MEMORY DATASET INVENTORY")
display(phase3H_object_inventory)

print("\nEXISTING PHASE 3 OUTPUT FILES")
display(phase3H_file_inventory)

print("\nSOURCE CSV INVENTORY")
display(phase3H_source_inventory)

print("\nOUTPUT DIRECTORY")
print(phase3H_output_dir)

print("\nMISSING IN-MEMORY OBJECTS:")
print(
    phase3H_object_inventory.loc[
        ~phase3H_object_inventory["available"], "object_name"
    ].tolist()
)

print("\nMISSING EXISTING EXPORTS:")
print(
    phase3H_file_inventory.loc[
        ~phase3H_file_inventory["exists"], "file_name"
    ].tolist()
)

IN-MEMORY DATASET INVENTORY


,object_name,available,rows,columns
0,df_deliveries,True,192235,61
1,df_source_matches,True,883,22
2,df_source_ball,True,200999,33
3,phase3D_first_innings_population,True,100333,69
4,phase3E_second_innings_wicket_alignment,True,91902,64
5,phase3G_final_decisions,True,15,3
6,phase3G_completion_summary,True,1,21
7,phase3G_final_validation_register,True,8,4
8,phase3G_output_audit,True,6,3



EXISTING PHASE 3 OUTPUT FILES


,file_name,exists,size_bytes
0,phase2_integrated_cleaned_deliveries.parquet,False,NaN
1,phase2_integrated_cleaned_deliveries.csv,False,NaN
2,phase3D_first_innings_population.parquet,True,1942312.0
3,SupeRR_Selector_S2_Phase_3G_Report.md,True,5096.0
4,phase3G_completion_summary.csv,True,651.0
5,phase3G_final_decision_register.csv,True,1811.0
6,phase3G_final_validation_register.csv,True,524.0
7,phase3G_final_output_audit.csv,True,302.0
8,phase3G_subphase_completion_register.csv,True,889.0
9,phase3G_validation_rollup.csv,True,94.0



SOURCE CSV INVENTORY


,source_file,found,path
0,matches.csv,True,/content/matches.csv
1,playing_eleven.csv,True,/content/playing_eleven.csv
2,competition_team_player_mapping.csv,True,/content/competition_team_player_mapping.csv
3,teams.csv,True,/content/teams.csv
4,competitions.csv,True,/content/competitions.csv



OUTPUT DIRECTORY
/content/phase3_outputs

MISSING IN-MEMORY OBJECTS:
[]

MISSING EXISTING EXPORTS:
['phase2_integrated_cleaned_deliveries.parquet', 'phase2_integrated_cleaned_deliveries.csv']


In [198]:
# PHASE 3H.2 — CANONICAL DATASET EXPORT AND VERIFICATION

from pathlib import Path
import shutil
import pandas as pd

phase3H_package_dir = Path(
    "/content/SupeRR_Selector_S2_Phase3H_Handoff"
)
phase3H_data_dir = phase3H_package_dir / "datasets"
phase3H_source_dir = phase3H_package_dir / "source_data"

phase3H_data_dir.mkdir(parents=True, exist_ok=True)
phase3H_source_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Define canonical analytical datasets
# ---------------------------------------------------------

phase3H_datasets = {
    "integrated_regulation_deliveries": df_deliveries,
    "first_innings_population": phase3D_first_innings_population,
    "second_innings_wicket_alignment": (
        phase3E_second_innings_wicket_alignment
    )
}

phase3H_key_columns = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num"
]

phase3H_export_records = []
phase3H_dataset_checks = []

for dataset_name, dataset in phase3H_datasets.items():

    missing_columns = [
        col for col in phase3H_key_columns
        if col not in dataset.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_name} missing key columns: {missing_columns}"
        )

    key_duplicates = int(dataset.duplicated(
        subset=phase3H_key_columns
    ).sum())

    missing_keys = int(
        dataset[phase3H_key_columns].isna().any(axis=1).sum()
    )

    parquet_path = phase3H_data_dir / f"{dataset_name}.parquet"

    dataset.to_parquet(parquet_path, index=False)

    # Read back and verify.
    reloaded = pd.read_parquet(parquet_path)

    dimensions_match = (
        len(reloaded) == len(dataset)
        and list(reloaded.columns) == list(dataset.columns)
    )

    reloaded_duplicate_keys = int(
        reloaded.duplicated(subset=phase3H_key_columns).sum()
    )

    reloaded_missing_keys = int(
        reloaded[phase3H_key_columns].isna().any(axis=1).sum()
    )

    check_passed = (
        dimensions_match
        and key_duplicates == 0
        and missing_keys == 0
        and reloaded_duplicate_keys == 0
        and reloaded_missing_keys == 0
    )

    phase3H_dataset_checks.append({
        "dataset": dataset_name,
        "rows": len(dataset),
        "columns": len(dataset.columns),
        "duplicate_keys_before_export": key_duplicates,
        "missing_key_rows_before_export": missing_keys,
        "readback_dimensions_match": dimensions_match,
        "duplicate_keys_after_readback": reloaded_duplicate_keys,
        "missing_key_rows_after_readback": reloaded_missing_keys,
        "passed": check_passed
    })

    phase3H_export_records.append({
        "file_name": parquet_path.name,
        "file_path": str(parquet_path),
        "file_type": "PARQUET",
        "dataset": dataset_name,
        "rows": len(reloaded),
        "columns": len(reloaded.columns),
        "size_bytes": parquet_path.stat().st_size
    })

phase3H_dataset_checks = pd.DataFrame(phase3H_dataset_checks)

# ---------------------------------------------------------
# 2. Export raw ball-by-ball source table
# ---------------------------------------------------------

phase3H_raw_ball_path = (
    phase3H_source_dir / "ball_by_ball.csv"
)

df_source_ball.to_csv(phase3H_raw_ball_path, index=False)

phase3H_export_records.append({
    "file_name": phase3H_raw_ball_path.name,
    "file_path": str(phase3H_raw_ball_path),
    "file_type": "SOURCE_CSV_FROM_IN_MEMORY_SOURCE",
    "dataset": "df_source_ball",
    "rows": len(df_source_ball),
    "columns": len(df_source_ball.columns),
    "size_bytes": phase3H_raw_ball_path.stat().st_size
})

# ---------------------------------------------------------
# 3. Copy the five original reference source CSVs
# ---------------------------------------------------------

phase3H_reference_files = [
    "matches.csv",
    "playing_eleven.csv",
    "competition_team_player_mapping.csv",
    "teams.csv",
    "competitions.csv"
]

phase3H_source_copy_checks = []

for file_name in phase3H_reference_files:

    source_path = Path("/content") / file_name
    destination_path = phase3H_source_dir / file_name

    if not source_path.is_file():
        phase3H_source_copy_checks.append({
            "file_name": file_name,
            "source_exists": False,
            "copied": False,
            "size_bytes": None
        })
        continue

    shutil.copy2(source_path, destination_path)

    phase3H_source_copy_checks.append({
        "file_name": file_name,
        "source_exists": True,
        "copied": destination_path.is_file(),
        "size_bytes": (
            destination_path.stat().st_size
            if destination_path.is_file()
            else None
        )
    })

phase3H_source_copy_checks = pd.DataFrame(
    phase3H_source_copy_checks
)

# ---------------------------------------------------------
# 4. Export manifest
# ---------------------------------------------------------

phase3H_export_manifest = pd.DataFrame(phase3H_export_records)

phase3H_export_manifest.to_csv(
    phase3H_package_dir / "phase3H_dataset_manifest.csv",
    index=False
)

phase3H_dataset_checks.to_csv(
    phase3H_package_dir / "phase3H_dataset_validation.csv",
    index=False
)

phase3H_source_copy_checks.to_csv(
    phase3H_package_dir / "phase3H_source_file_audit.csv",
    index=False
)

# ---------------------------------------------------------
# 5. Display results
# ---------------------------------------------------------

print("CANONICAL DATASET EXPORTS")
display(phase3H_export_manifest)

print("\nDATASET VALIDATION")
display(phase3H_dataset_checks)

print("\nSOURCE FILE COPY AUDIT")
display(phase3H_source_copy_checks)

print("\nPACKAGE DIRECTORY")
print(phase3H_package_dir)

print("\nDATASET CHECKS PASSED:",
      int(phase3H_dataset_checks["passed"].sum()),
      "/", len(phase3H_dataset_checks))

print("\nSOURCE FILES COPIED:",
      int(phase3H_source_copy_checks["copied"].sum()),
      "/", len(phase3H_source_copy_checks))

assert phase3H_dataset_checks["passed"].all()
assert phase3H_source_copy_checks["copied"].all()

print("\nPHASE 3H.2 CANONICAL DATASET EXPORT: PASSED")

CANONICAL DATASET EXPORTS


,file_name,file_path,file_type,dataset,rows,columns,size_bytes
0,integrated_regulation_deliveries.parquet,/content/SupeRR_Selector_S2_Phase3H_Handoff/datasets/integrated_regulation_deliveries.parquet,PARQUET,integrated_regulation_deliveries,192235,61,3456892
1,first_innings_population.parquet,/content/SupeRR_Selector_S2_Phase3H_Handoff/datasets/first_innings_population.parquet,PARQUET,first_innings_population,100333,69,1942312
2,second_innings_wicket_alignment.parquet,/content/SupeRR_Selector_S2_Phase3H_Handoff/datasets/second_innings_wicket_alignment.parquet,PARQUET,second_innings_wicket_alignment,91902,64,1801491
3,ball_by_ball.csv,/content/SupeRR_Selector_S2_Phase3H_Handoff/source_data/ball_by_ball.csv,SOURCE_CSV_FROM_IN_MEMORY_SOURCE,df_source_ball,200999,33,40547435



DATASET VALIDATION


,dataset,rows,columns,duplicate_keys_before_export,missing_key_rows_before_export,readback_dimensions_match,duplicate_keys_after_readback,missing_key_rows_after_readback,passed
0,integrated_regulation_deliveries,192235,61,0,0,True,0,0,True
1,first_innings_population,100333,69,0,0,True,0,0,True
2,second_innings_wicket_alignment,91902,64,0,0,True,0,0,True



SOURCE FILE COPY AUDIT


,file_name,source_exists,copied,size_bytes
0,matches.csv,True,True,123864
1,playing_eleven.csv,True,True,966287
2,competition_team_player_mapping.csv,True,True,462969
3,teams.csv,True,True,6922
4,competitions.csv,True,True,7624



PACKAGE DIRECTORY
/content/SupeRR_Selector_S2_Phase3H_Handoff

DATASET CHECKS PASSED: 3 / 3

SOURCE FILES COPIED: 5 / 5

PHASE 3H.2 CANONICAL DATASET EXPORT: PASSED


In [199]:
from pathlib import Path
import pandas as pd

root = Path("/content")
phase3_dir = root / "phase3_outputs"
package_dir = root / "SupeRR_Selector_S2_Phase3H_Handoff"

search_roots = [
    root,
    phase3_dir,
    package_dir
]

seen = set()
records = []

for search_root in search_roots:
    if not search_root.exists():
        continue

    for path in search_root.rglob("*"):
        if not path.is_file():
            continue

        if package_dir in path.parents:
            continue

        suffix = path.suffix.lower()
        name = path.name.lower()

        relevant = (
            suffix in [".md", ".docx", ".pdf"]
            and ("report" in name or "phase" in name or "handoff" in name)
        ) or (
            suffix == ".csv"
            and (
                "validation" in name
                or "decision" in name
                or "completion" in name
                or "output_audit" in name
                or "exception" in name
                or "coverage" in name
                or "register" in name
            )
        )

        if relevant:
            resolved = str(path.resolve())
            if resolved not in seen:
                seen.add(resolved)
                records.append({
                    "file_name": path.name,
                    "file_path": resolved,
                    "file_type": suffix,
                    "size_bytes": path.stat().st_size
                })

inventory = (
    pd.DataFrame(records)
    .sort_values(["file_type", "file_name"])
    .reset_index(drop=True)
    if records else
    pd.DataFrame(columns=[
        "file_name", "file_path", "file_type", "size_bytes"
    ])
)

print("REPORT AND AUDIT FILE INVENTORY")
display(inventory)

print("\nINVENTORY SUMMARY")
print("Files found:", len(inventory))
print("Markdown:", int((inventory["file_type"] == ".md").sum()))
print("DOCX:", int((inventory["file_type"] == ".docx").sum()))
print("PDF:", int((inventory["file_type"] == ".pdf").sum()))
print("CSV registers:", int((inventory["file_type"] == ".csv").sum()))
print("\nPHASE 3H.3 INVENTORY COMPLETE")

REPORT AND AUDIT FILE INVENTORY


,file_name,file_path,file_type,size_bytes
0,phase2_completion_summary.csv,/content/phase2_completion_summary.csv,.csv,423
1,phase2_exception_register.csv,/content/phase2_exception_register.csv,.csv,6829
2,phase2_unexplained_extended_over_register.csv,/content/phase2_unexplained_extended_over_register.csv,.csv,4500
3,phase3A_54_excluded_matches_validation.csv,/content/phase3_outputs/phase3A_54_excluded_matches_validation.csv,.csv,262
4,phase3A_54_investigation_validation.csv,/content/phase3_outputs/phase3A_54_investigation_validation.csv,.csv,413
...,...,...,...,...
119,SupeRR_Selector_S2_Phase_3C_Report.md,/content/phase3_outputs/SupeRR_Selector_S2_Phase_3C_Report.md,.md,6830
120,SupeRR_Selector_S2_Phase_3D_Report.md,/content/phase3_outputs/SupeRR_Selector_S2_Phase_3D_Report.md,.md,3371
121,SupeRR_Selector_S2_Phase_3E_Report.md,/content/phase3_outputs/SupeRR_Selector_S2_Phase_3E_Report.md,.md,3316
122,SupeRR_Selector_S2_Phase_3F_Report.md,/content/phase3_outputs/SupeRR_Selector_S2_Phase_3F_Report.md,.md,7352



INVENTORY SUMMARY
Files found: 124
Markdown: 8
DOCX: 1
PDF: 0
CSV registers: 115

PHASE 3H.3 INVENTORY COMPLETE


In [200]:
from pathlib import Path
import pandas as pd
import re

search_roots = [
    Path("/content"),
    Path("/content/phase3_outputs")
]

package_dir = Path("/content/SupeRR_Selector_S2_Phase3H_Handoff")

records = []
seen = set()

for search_root in search_roots:
    if not search_root.exists():
        continue

    for path in search_root.rglob("*"):
        if not path.is_file():
            continue
        if package_dir in path.parents:
            continue

        name = path.name.lower()
        suffix = path.suffix.lower()

        relevant = (
            suffix in [".md", ".docx", ".pdf"]
            or (
                suffix == ".csv"
                and any(term in name for term in [
                    "validation", "decision", "completion",
                    "output_audit", "exception", "coverage",
                    "register", "summary", "audit"
                ])
            )
        )

        if not relevant:
            continue

        resolved = str(path.resolve())
        if resolved in seen:
            continue
        seen.add(resolved)

        match = re.search(r"phase\s*([0-9]+[A-Z]?)", name)
        phase = match.group(1).upper() if match else "UNASSIGNED"

        records.append({
            "phase": phase,
            "file_name": path.name,
            "file_type": suffix,
            "file_path": resolved,
            "size_bytes": path.stat().st_size
        })

inventory = pd.DataFrame(records)

if not inventory.empty:
    inventory = inventory.sort_values(
        ["phase", "file_type", "file_name"]
    ).reset_index(drop=True)

print("COMPLETE REPORT AND REGISTER INVENTORY")
print(inventory.to_string(index=False))

print("\nPHASE COUNTS")
print(
    inventory.groupby(["phase", "file_type"])
    .size()
    .to_string()
)

print("\nREPORT DOCUMENTS ONLY")
reports = inventory[
    inventory["file_type"].isin([".md", ".docx", ".pdf"])
]
print(reports.to_string(index=False))

print("\nINVENTORY ROW COUNT:", len(inventory))

COMPLETE REPORT AND REGISTER INVENTORY
     phase                                              file_name file_type                                                                      file_path  size_bytes
         0                                      Phase 0 report.md       .md                                                     /content/Phase 0 report.md       17463
         2                          phase2_completion_summary.csv      .csv                                         /content/phase2_completion_summary.csv         423
         2                          phase2_exception_register.csv      .csv                                         /content/phase2_exception_register.csv        6829
         2          phase2_unexplained_extended_over_register.csv      .csv                         /content/phase2_unexplained_extended_over_register.csv        4500
         2                                      Phase 2 report.md       .md                                                   

In [201]:
from pathlib import Path
import shutil
import pandas as pd

package = Path("/content/SupeRR_Selector_S2_Phase3H_Handoff")
reports_dir = package / "reports"
audit_dir = package / "phase3_audit_registers"

reports_dir.mkdir(parents=True, exist_ok=True)
audit_dir.mkdir(parents=True, exist_ok=True)

# Verified Phase 0, 1 and 2 reports
report_sources = [
    Path("/content/Phase 0 report.md"),
    Path("/content/SupeRR_Selector_S2_Phase_1_Report.docx"),
    Path("/content/Phase 2 report.md"),
]

# All Phase 3 Markdown reports and CSV audit registers
phase3_source = Path("/content/phase3_outputs")
phase3_reports = sorted(phase3_source.glob("*.md"))
phase3_registers = sorted(phase3_source.glob("*.csv"))

# Phase 2 supporting registers
phase2_registers = [
    Path("/content/phase2_completion_summary.csv"),
    Path("/content/phase2_exception_register.csv"),
    Path("/content/phase2_unexplained_extended_over_register.csv"),
]

copied_records = []
missing_sources = []

def copy_verified_file(source, destination_dir, category):
    if not source.exists() or not source.is_file():
        missing_sources.append(str(source))
        return

    destination = destination_dir / source.name
    shutil.copy2(source, destination)

    copied_records.append({
        "category": category,
        "file_name": source.name,
        "source_path": str(source),
        "package_path": str(destination),
        "size_bytes": destination.stat().st_size,
        "copy_size_matches": (
            source.stat().st_size == destination.stat().st_size
        )
    })

for source in report_sources:
    copy_verified_file(source, reports_dir, "PROJECT_PHASE_REPORT")

for source in phase3_reports:
    copy_verified_file(source, reports_dir, "PHASE3_REPORT")

for source in phase2_registers:
    copy_verified_file(source, audit_dir, "PHASE2_AUDIT_REGISTER")

for source in phase3_registers:
    copy_verified_file(source, audit_dir, "PHASE3_AUDIT_REGISTER")

copy_audit = pd.DataFrame(copied_records)

print("DOCUMENT AND AUDIT REGISTER COPY AUDIT")
display(copy_audit)

print("\nCOPY SUMMARY")
print("Project reports copied:", len([
    x for x in copied_records
    if x["category"] == "PROJECT_PHASE_REPORT"
]))
print("Phase 3 reports copied:", len(phase3_reports))
print("Phase 2 registers copied:", len(phase2_registers) - len([
    x for x in missing_sources if "phase2_" in x
]))
print("Phase 3 registers copied:", len(phase3_registers))
print("Missing source files:", len(missing_sources))
print("Copy size mismatches:", int(
    (~copy_audit["copy_size_matches"]).sum()
) if not copy_audit.empty else 0)

if missing_sources:
    print("\nMISSING SOURCES")
    for item in missing_sources:
        print(item)

assert not missing_sources, "Some expected source documents are missing."
assert not copy_audit.empty
assert copy_audit["copy_size_matches"].all()

print("\nPHASE 3H.5 DOCUMENT AND AUDIT COPY: PASSED")

DOCUMENT AND AUDIT REGISTER COPY AUDIT


,category,file_name,source_path,package_path,size_bytes,copy_size_matches
0,PROJECT_PHASE_REPORT,Phase 0 report.md,/content/Phase 0 report.md,/content/SupeRR_Selector_S2_Phase3H_Handoff/reports/Phase 0 report.md,17463,True
1,PROJECT_PHASE_REPORT,SupeRR_Selector_S2_Phase_1_Report.docx,/content/SupeRR_Selector_S2_Phase_1_Report.docx,/content/SupeRR_Selector_S2_Phase3H_Handoff/reports/SupeRR_Selector_S2_Phase_1_Report.docx,44819,True
2,PROJECT_PHASE_REPORT,Phase 2 report.md,/content/Phase 2 report.md,/content/SupeRR_Selector_S2_Phase3H_Handoff/reports/Phase 2 report.md,9393,True
3,PHASE3_REPORT,SupeRR_Selector_S2_Phase_3B_Report.md,/content/phase3_outputs/SupeRR_Selector_S2_Phase_3B_Report.md,/content/SupeRR_Selector_S2_Phase3H_Handoff/reports/SupeRR_Selector_S2_Phase_3B_Report.md,2973,True
4,PHASE3_REPORT,SupeRR_Selector_S2_Phase_3C_Report.md,/content/phase3_outputs/SupeRR_Selector_S2_Phase_3C_Report.md,/content/SupeRR_Selector_S2_Phase3H_Handoff/reports/SupeRR_Selector_S2_Phase_3C_Report.md,6830,True
...,...,...,...,...,...,...
252,PHASE3_AUDIT_REGISTER,phase3G_wicket_event_quality.csv,/content/phase3_outputs/phase3G_wicket_event_quality.csv,/content/SupeRR_Selector_S2_Phase3H_Handoff/phase3_audit_registers/phase3G_wicket_event_quality.csv,96,True
253,PHASE3_AUDIT_REGISTER,phase3G_wickets_remaining_distribution_comparison.csv,/content/phase3_outputs/phase3G_wickets_remaining_distribution_comparison.csv,/content/SupeRR_Selector_S2_Phase3H_Handoff/phase3_audit_registers/phase3G_wickets_remaining_distribution_comparison.csv,851,True
254,PHASE3_AUDIT_REGISTER,phase3G_wickets_remaining_mismatches.csv,/content/phase3_outputs/phase3G_wickets_remaining_mismatches.csv,/content/SupeRR_Selector_S2_Phase3H_Handoff/phase3_audit_registers/phase3G_wickets_remaining_mismatches.csv,170,True
255,PHASE3_AUDIT_REGISTER,phase3G_wickets_remaining_quality.csv,/content/phase3_outputs/phase3G_wickets_remaining_quality.csv,/content/SupeRR_Selector_S2_Phase3H_Handoff/phase3_audit_registers/phase3G_wickets_remaining_quality.csv,244,True



COPY SUMMARY
Project reports copied: 3
Phase 3 reports copied: 6
Phase 2 registers copied: 3
Phase 3 registers copied: 245
Missing source files: 0
Copy size mismatches: 0

PHASE 3H.5 DOCUMENT AND AUDIT COPY: PASSED


In [202]:
from pathlib import Path

package = Path("/content/SupeRR_Selector_S2_Phase3H_Handoff")
(package / "reports").mkdir(parents=True, exist_ok=True)

readme = r"""
# SupeRR Selector Season 2 — Phase 3H Handoff Package

## Purpose
This package transfers the verified project evidence and datasets to a new working chat for Phases 4–7.

## Start here
1. Read `SupeRR_Selector_S2_Master_Handoff.md`.
2. Read the Phase 0–3 reports in `reports/`.
3. Read the Phase 3 decision, validation and limitation registers in `phase3_audit_registers/`.
4. Inspect the canonical Parquet datasets and source CSVs.
5. Use `SupeRR_Selector_S2_Phase4_Continuation_Prompt.md` as the first prompt in the new chat.

## Canonical datasets
| File | Rows | Columns | Description |
|---|---:|---:|---|
| `datasets/integrated_regulation_deliveries.parquet` | 192,235 | 61 | Integrated regulation-delivery population |
| `datasets/first_innings_population.parquet` | 100,333 | 69 | First-innings analytical population |
| `datasets/second_innings_wicket_alignment.parquet` | 91,902 | 64 | Second-innings wicket-state alignment |
| `source_data/ball_by_ball.csv` | 200,999 | 33 | Source ball-by-ball table exported from in-memory `df_source_ball` |

The three Parquet files passed export/readback dimension checks and delivery-key uniqueness/missing-key checks. The CSV is a source-table export from the in-memory source DataFrame, not a verified byte-for-byte copy of an original CSV file.

## Source/reference CSVs
- `source_data/matches.csv`
- `source_data/playing_eleven.csv`
- `source_data/competition_team_player_mapping.csv`
- `source_data/teams.csv`
- `source_data/competitions.csv`

## Documents
- `SupeRR_Selector_S2_Master_Handoff.md`
- `SupeRR_Selector_S2_Phase4_Continuation_Prompt.md`
- `reports/`: Phase 0, 1, 2 and Phase 3B–3G reports, plus Phase 3H report.
- `phase3_audit_registers/`: copied Phase 2 and Phase 3 CSV registers.

## Known report gap
No Phase 3A Markdown report was present in the inventoried Colab paths. Available Phase 3A audit CSVs were copied. Do not invent or imply that a Phase 3A report exists.

## Approval status
Phase 3H dataset exports and document copying passed their reported checks. Formal Phase 3H user approval remains pending until the user explicitly signs off.
"""

master_handoff = r"""
# SupeRR Selector Season 2 — Master Project Handoff

**Handoff purpose:** Independent continuation of Phases 4–7 in a new chat
**Current project boundary:** Phase 3H — Report and approval
**Phase 3H approval:** PENDING USER SIGN-OFF

---

## 1. Project objective

The project objective is to identify the top 10 batters and top 10 bowlers based on performance under pressure in IPL and men's T20I cricket from 2022 onward. The recent two seasons are supplementary context and must not silently replace or alter the historical evaluation population.

The project must distinguish a player's general performance from performance under a defensible, pre-delivery pressure definition. Do not call ordinary aggregate performance a pressure metric.

## 2. Core methodology decisions from Phase 0

- Pressure describes the difficulty of the team's match situation immediately before a delivery.
- Pressure-state inputs must be pre-delivery. Delivery outcomes and future match outcomes must not leak into the pressure definition.
- Batter and bowler evaluations use the same underlying match-state classification; their performance outcomes are evaluated separately.
- First-innings pressure concept:
  1. Scoring position relative to expected cumulative scoring.
  2. Wickets remaining.
  3. Balls remaining.
- Raw scoring difference is actual cumulative score minus expected cumulative score.
- Standardized difference is the difference divided by historical standard deviation for comparable situations.
- A negative difference means below expectation; a positive difference means above expectation. This deviation is not by itself a pressure score and is not a normal-distribution probability.
- Wickets remaining and balls remaining are separate dimensions initially. Do not invent an arbitrary composite resource index.
- Second-innings expected scoring-rate benchmarks should adapt to competition, innings stage, wickets remaining and remaining resources, if those inputs can be validated.
- Required-rate difficulty should initially be evaluated without conditioning the expected-rate benchmark on required rate itself. Conditioning would require evidence of improved out-of-sample performance and stability.
- Fallback hierarchy: exact comparable states; nearby scheduled innings lengths; adjacent wicket states using observation-weighted averages; symmetric expansion; broader stage-based fallback only if defensible; otherwise unavailable.
- If exact and pooled estimates materially disagree relative to uncertainty or practical significance, mark the estimate unavailable.
- Preserve benchmark provenance, sample size, schedule-length range, progression range and wicket-state matching.
- IPL/T20 and T20I must remain separate. No cross-competition fallback unless separately justified and approved.
- Phase 0 did not approve an operational numerical pressure scale. First-innings scoring deviation and second-innings required-rate difficulty are distinct indicators, not an assumed unified score.

## 3. Population and Phase 1

- Source match inventory: 883 matches.
- Approved target population: 853 matches (371 T20 and 482 T20I).
- Eligible integrated regulation population: 813 matches (362 T20 and 451 T20I).
- Documented eligibility exclusions: 40 matches (9 T20 and 31 T20I).
- Out-of-scope competition matches: 30:
  - Competition 2670: Men's T20 Emerging Teams Asia Cup 2024 (15).
  - Competition 3289: Asia Cup Rising Stars 2025 (15).
- Reconciliation: 883 = 853 + 30; 853 = 813 + 40.
- The 40 exclusions reconcile to 16 abandoned matches with no deliveries, 3 abandoned matches with partial deliveries, 19 No Result matches with one innings and 2 No Result matches with two innings.
- Phase 1 approved.

## 4. Phase 2 integrated delivery data and data-quality findings

- Integrated regulation-delivery dataset: 192,235 records, 813 matches and 1,626 innings.
- Wicket events: 10,392.
- One wicket event lacks `dismissed_player_id` (Retire Out, match 123543, innings 2, sequence 2).
- Eleven non-wicket records have dismissed-player IDs.
- Dismissal categories: Caught 7,264; Bowled 1,675; LBW 663; RunOut 567; Stump 198; Hit Wicket 17; Retire Out 6; Obstructing field 2 (one raw spelling typo).
- Legal-ball evidence:
  - `NO_RECORDED_WIDE_OR_OVERSTEP`: 184,002 (95.7172%).
  - `NON_LEGAL_EVIDENCE`: 7,976 (4.1491%).
  - `UNRESOLVED`: 257 (0.1337%).
  - 7,419 wides, 557 oversteps and 257 unresolved records (256 other plus 1 penalty).
- There are 147 unexplained extended overs with no recorded wide/overstep, covering 1,036 records.
- `over_ball_num` is a delivery-sequence position, not a legal-ball counter.
- 6,450 innings-over groups have more than six delivery records, covering 46,821 records across 810 matches.
- `over_or_around` labels: Around 43,219 (22.482378%) and Over 148,620 (77.311624%). These labels do not prove legal-ball progression.
- `runs` semantics remain unresolved: do not assume it is total delivery runs or sum it as such.
- `extras1_runs` is a categorical code, not a numeric extras amount: 1 Overstep, 2 Wide, 3 Legbyes, 4 Byes, 5 Other, 6 No extras, 7 Penalty.
- `extras_2_runs` semantics remain unresolved.
- 23 boundary-flag mismatches were identified.
- `innings_ball_num` is a continuous innings sequence, not a legal-ball counter.
- Scheduled innings length is not available as a normal validated scheduled-overs field. Revised fields are sparse and their units unresolved.
- All 813 integrated matches have exactly two regulation innings, correct team reversal and no mid-innings team change.
- Phase 2 report and supporting exception registers are included in the package.

## 5. Phase 3B — Data audit and reconstruction feasibility

Phase 3B completed its data audit and reconstruction-feasibility assessment.

Feasibility classification:
- USABLE: competition population; innings/team structure.
- CONDITIONAL: pre-delivery wickets fallen; dismissal event identity; over structure.
- UNAVAILABLE: cumulative scoring; legal-ball progression; scheduled innings length; first-innings scoring position; second-innings scoring demand; historical pressure benchmarks.

No numerical pressure model was created.

## 6. Phase 3C — Pre-delivery state specification

Phase 3C specified the pre-delivery state boundary; it did not score pressure.

The 12-field state schema:
- `match_id`
- `innings`
- `comp_id`
- `comp_name`
- `match_format`
- `season`
- `batting_team_id`
- `bowling_team_id`
- `overid`
- `over_ball_num`
- `innings_ball_num`
- `pre_delivery_wickets_fallen`

The state table contains 192,235 rows and 65 columns. Delivery key alignment passed across all 192,235 records with no duplicates, missing keys or unmatched rows.

Pre-delivery wickets fallen was reconstructed from prior recorded wicket events. Continuity checks found zero mismatches, but the result remains conditional because the source dismissal truth is not independently established.

Prohibited leakage fields include `post_delivery_wickets_fallen`, `is_wicket`, `runs`, `extras1_runs`, `extras_2_runs` and `win_team_id`.

## 7. Phase 3D — First-innings benchmark feasibility

- First-innings population: 100,333 delivery rows, 813 matches and 69 columns.
- IPL: 362 matches and 45,139 records.
- T20I: 451 matches and 55,194 records.
- Expected cumulative scoring benchmark was classified NOT_CONSTRUCTIBLE.
- Scoring reconstruction was blocked because run semantics could not be validated.
- Legal balls, schedule, stage and scoring deviation remain blocked.
- Wicket-state availability is conditional.
- Competition/wicket coverage is audit-only; coverage is not evidence of benchmark reliability.
- The fallback hierarchy was specified, but its numerical fallback steps were not executable.
- Phase 3D closed as: `CLOSED — AUDITS COMPLETE; BENCHMARK CONSTRUCTION BLOCKED`.

## 8. Phase 3E — Second-innings feasibility

- Second-innings population/alignment: 91,902 delivery rows across 813 matches.
- `revised_target` appears in 22 of 813 second innings (1,616 delivery rows), with 22 distinct values and no within-match conflicts; its units and semantics remain unverified.
- No validated cumulative score exists; runs required cannot be constructed.
- Second-innings legal-ball evidence: 88,026 records with no recorded wide/overstep, 3,755 with recorded non-legal evidence and 121 unresolved.
- `second_inn_revised_overs` appears in 22 of 813 matches, has 10 distinct values in the observed subset, range 30–114, and unverified units.
- Sequence fields are not legal-ball counters.
- Wickets remaining was computed as 10 minus reconstructed pre-delivery wickets fallen; this is conditional on the reconstructed wicket input.
- Expected scoring rate, required run rate and second-innings pressure score were not constructed.
- The fallback policy was specified, not executed.
- Phase 3E closed with 34/34 checks and documented blockers. It is not a working second-innings pressure model.

## 9. Phase 3F — Pressure scale validation

Final decisions:
- First-innings numerical scale: unavailable.
- Second-innings numerical scale: unavailable.
- Unified numerical scale: not justified.
- Shared conceptual framework: retained with limitations.
- Wicket state: conditional.
- Pressure bands, weights and thresholds: not established.
- No normalization or composite score was approved.
- Cross-innings calibration could not be tested.

## 10. Phase 3G — Robustness and validation

Phase 3G.1–3G.7 completed and closed.

Key results:
- 883 source matches, 813 approved integrated matches and 70 outside the integrated population.
- 192,235 integrated records with 192,235 unique complete delivery keys.
- All 813 integrated matches have two regulation innings.
- 1,626 innings sequences are continuous, but sequence continuity does not establish legal-ball counts.
- Raw source data contains 26 records absent from the integrated regulation population. These are innings 3/4 Super Over records across four matches and were confirmed as expected exclusions:
  - Match 34429: India tied Sri Lanka; India won the Super Over by 2 wickets.
  - Match 40725: Delhi Capitals tied Rajasthan Royals; Delhi Capitals won the Super Over.
  - Match 45512: India won the Super Over after a tied match.
  - Match 50359: Kolkata Knight Riders won by Super Over after a tied match.
- All 192,235 integrated delivery rows aligned to pre-delivery states with no key mismatch, duplicate or missing key.
- Wicket continuity recomputation had zero mismatches; this is not independent dismissal truth.
- First- and second-innings subsets aligned to their respective state tables.
- 136 competition groups: T20 has 362 matches across 5 competitions; T20I has 451 matches across 131 competitions.
- 208 unobserved competition × wicket × innings combinations: first innings 119, second innings 89; all in T20I.
- 461 observed competition-state combinations are single-match dependent (of 2,512 observed combinations).
- No sample-size threshold was invented and benchmark reliability was not established.
- A prior first-innings missing-state register of 255 rows reconciles to 119 shared rows plus 136 state-10 rows outside the actual first-innings state domain 0–9. These are not directly comparable counts.
- Final Phase 3G decision: no first-, second- or unified numerical pressure scale; conceptual framework retained with limitations; wicket state conditional; no pressure bands or thresholds.

## 11. Phase 3H — Report and approval

Phase 3H scope: export datasets, document limitations and obtain sign-off.

Completed in the current Colab handoff work:
- Exported three canonical Parquet datasets:
  - integrated regulation deliveries: 192,235 × 61.
  - first-innings population: 100,333 × 69.
  - second-innings wicket alignment: 91,902 × 64.
- All three Parquet exports passed readback dimension checks, duplicate-key checks and missing-key checks using the delivery key `match_id`, `innings`, `overid`, `over_ball_num`, `innings_ball_num`.
- Exported `ball_by_ball.csv` from in-memory `df_source_ball`: 200,999 × 33.
- Copied five reference CSVs: `matches.csv`, `playing_eleven.csv`, `competition_team_player_mapping.csv`, `teams.csv`, `competitions.csv`.
- Copied 3 original project reports (Phases 0, 1 and 2), 6 Phase 3 reports (3B–3G), 3 Phase 2 audit registers and 245 Phase 3 CSV registers.
- File-size checks matched for all 257 copied documents/registers.

Important caveats:
- `ball_by_ball.csv` in this package is an export from the in-memory `df_source_ball`, not a verified byte-for-byte copy of the original source file.
- No Phase 3A Markdown report was found in the Colab inventory. Available Phase 3A audit CSVs were copied.
- Formal Phase 3H user sign-off has not yet been given. Do not mark the project approved until the user explicitly approves.

## 12. Critical limitations that Phase 4 must inherit

1. Run semantics are unresolved. Do not sum `runs` and `extras_2_runs` or treat `extras1_runs` as numeric runs.
2. Cumulative innings score is unavailable.
3. Legal-ball progression is unavailable/unresolved; delivery sequence columns are not legal-ball counters.
4. Scheduled innings length and stage are unavailable or unverified.
5. First-innings scoring position and expected cumulative scoring benchmark are unavailable.
6. Second-innings runs required and required run rate are not constructible; `revised_target` is sparse and unverified.
7. Historical expected-rate benchmarks and fallback estimates were not constructed.
8. Pre-delivery wickets fallen is conditional; wickets remaining inherits that limitation.
9. T20I competition-state coverage is sparse in some wicket states; single-match support is common enough to require explicit robustness treatment.
10. No numerical pressure scale, weights, bands or thresholds are approved.
11. No cross-format fallback or cross-innings calibration is approved.
12. The 26 Super Over records are excluded from the regulation-delivery population and retained as an audit trail.

## 13. Phase 4 starting boundary

Phase 4 must begin with a feasibility and dependency audit, not player scoring.

It must determine:
- Which player-level outcomes can be derived from the available delivery and player mapping data.
- Whether batter and bowler identities and team/competition joins are valid.
- Which metrics are ordinary performance measures versus pressure-conditioned measures.
- Whether any pressure-conditioned player analysis is possible without the missing score, legal-ball, schedule and benchmark inputs.
- What additional source data or approved reconstruction is required.
- The exact definitions, denominators, exclusions, missingness and validation checks for any proposed metric.

Do not produce final player rankings if the pressure state cannot be defensibly constructed. Any fallback to general performance requires explicit approval and must not be labelled pressure performance.

## 14. Phase 5–7 gates

- Phase 5: only after Phase 4 methodology and data feasibility are approved. Ranking uncertainty and sensitivity analysis must reflect actual defensible metrics. Do not invent bootstrap design, weights or thresholds.
- Phase 6: report only validated and supported results, with limitations and reproducible evidence.
- Phase 7: final submission only if the ranking methodology is defensible. If not, report the blockers transparently rather than manufacture a top 10.

## 15. Working protocol for the next chat

- First verify the handoff files and manifest; report missing or inconsistent items.
- Treat the package reports and registers as source of truth. Do not rely on the previous chat's memory.
- Work in Google Colab, one cell at a time.
- Explain the purpose, provide one cell, then wait for the user's complete output before continuing.
- If an error occurs, fix only that error and wait for output again.
- Inspect actual schemas, dtypes, values and object names before writing transformations.
- Do not assume columns or variable names.
- Do not modify raw DataFrames; use derived copies.
- Use explicit merge validation, row-count checks and assertions.
- Do not silently drop rows, overwrite data or impute unavailable values.
- Export analytical datasets as Parquet and audit registers as CSV.
- Keep outputs organized and reproducible.
- Do not claim a cell ran or a validation passed until the user shares the output.

---
"""

continuation_prompt = r"""
# SupeRR Selector Season 2 — Phase 4–7 Continuation Instructions

You are continuing an existing cricket analytics project. The authoritative handoff package is `SupeRR_Selector_S2_Phase3H_Handoff`.

## First task — verify the handoff
1. Read `SupeRR_Selector_S2_Master_Handoff.md` completely.
2. Inspect `README.md`, the Phase 0–3 reports, the decision/validation/limitation registers and the canonical datasets.
3. Verify the package manifest, dataset dimensions, delivery keys, and file availability. Do not assume a file exists because it is named in a document.
4. Report missing, inconsistent or ambiguous items before doing analysis.
5. Treat the package as the source of truth. Do not rely on another chat's memory or invent project decisions.

## Project objective
The project aims to identify the top 10 batters and top 10 bowlers based on performance under pressure in IPL and men's T20I from 2022 onward. The recent two seasons are supplementary context and must not silently alter the historical evaluation population.

## Non-negotiable methodology constraints
- Pressure must be defined using information available immediately before a delivery.
- Prevent outcome leakage. Do not use the current delivery's runs, extras, wicket result, post-delivery state or match result to define pre-delivery pressure.
- Batter and bowler analysis must use the same underlying pressure-state classification, with their outcomes evaluated separately.
- Keep IPL/T20 and T20I separate unless an explicit, evidence-based approval changes this.
- Do not invent run semantics, legal-ball counts, scheduled overs, target semantics, pressure weights, bands, thresholds or fallback estimates.
- Preserve player identity and competition mapping provenance.
- Do not modify raw DataFrames. Use derived copies, explicit join validation and row-count/key assertions.
- Work in Google Colab one cell at a time. Explain the purpose, provide exactly one cell, wait for the user's complete output, inspect it, and only then continue. If an error occurs, fix only that error and wait for output again.
- Never claim code ran or a check passed unless the user has shared the output.

## Critical inherited Phase 3 blockers
- `runs` semantics are unresolved. Do not assume it represents total delivery runs or sum it blindly.
- `extras1_runs` is a categorical code, not a numeric extras amount.
- `extras_2_runs` semantics are unresolved.
- 23 boundary-flag mismatches were identified.
- Cumulative innings score is unavailable.
- Legal-ball progression is unavailable/unresolved. `over_ball_num` and `innings_ball_num` are sequence fields, not legal-ball counters.
- Scheduled innings length and innings stage are unavailable or unverified; revised fields are sparse and units unresolved.
- First-innings scoring position and expected cumulative scoring benchmark are unavailable.
- Second-innings runs required and required run rate are not constructible. `revised_target` is sparse and unverified.
- No historical expected-rate benchmark or executable fallback estimates were constructed.
- `pre_delivery_wickets_fallen` was reconstructed from prior recorded wicket events; internal continuity passed but independent dismissal truth was not established. Wickets remaining inherits this conditional limitation.
- T20I competition-state coverage is sparse in some wicket states. Single-match support exists in 461 of 2,512 observed competition-state combinations.
- No numerical pressure scale, weights, bands or thresholds were approved.
- No cross-format fallback or cross-innings calibration was approved.
- 26 raw Super Over records were excluded from the regulation population and retained in an audit trail.

## Phase 4 — feasibility and metric design, not premature scoring
Start with a data and methodology feasibility audit. Inspect the actual files and schemas first.

Determine:
1. Whether player-level batting and bowling outcomes can be reconstructed reliably.
2. Whether player IDs, team IDs, competition IDs, seasons and innings can be joined without duplication or leakage.
3. Which conventional player metrics are calculable from validated fields.
4. Which metrics can genuinely be conditioned on a defensible pre-delivery pressure state.
5. Which required fields are absent or ambiguous and what external source/reconstruction would be needed.
6. Definitions, denominators, exclusions, missingness, merge validation and independent validation for each proposed metric.
7. Whether a valid pressure-conditioned analysis is feasible at all under the inherited Phase 3 constraints.

Do not silently replace pressure-conditioned performance with general performance. If a general-performance fallback is proposed, label it accurately and ask for approval before adopting it. Do not finalize metric weights or ranking thresholds without evidence and explicit approval.

## Phase 5 — ranking and uncertainty
Proceed only after Phase 4 definitions and feasibility are approved. Use defensible uncertainty and sensitivity analysis. Do not invent a bootstrap design, weighting scheme, eligibility rule or threshold. Do not produce top-10 rankings if the pressure metric is not valid.

## Phase 6 — evidence and reporting
Report only validated results. Preserve definitions, source provenance, sample sizes, uncertainty, robustness findings and limitations. Distinguish descriptive results from inferential claims.

## Phase 7 — final submission
Prepare final submission materials only after the evidence is complete and the ranking method is defensible. If blockers remain, state them plainly rather than manufacturing rankings.

## Required first response
Do not start calculating player metrics or rankings. First confirm the handoff package can be inspected, list the exact files you can verify, identify any missing documents or datasets, and propose the first Phase 4 feasibility-audit step. Then wait for approval to begin the first Colab cell.
"""

phase3h_report = r"""
# SupeRR Selector Season 2 — Phase 3H Report

## Scope
Phase 3H: Report and approval. Export datasets, document limitations and prepare the project handoff for Phases 4–7.

## Canonical dataset exports

| Dataset | File | Rows | Columns |
|---|---|---:|---:|
| Integrated regulation deliveries | `datasets/integrated_regulation_deliveries.parquet` | 192,235 | 61 |
| First-innings population | `datasets/first_innings_population.parquet` | 100,333 | 69 |
| Second-innings wicket alignment | `datasets/second_innings_wicket_alignment.parquet` | 91,902 | 64 |
| Source ball-by-ball table | `source_data/ball_by_ball.csv` | 200,999 | 33 |

The three Parquet exports passed readback dimension checks, duplicate delivery-key checks and missing delivery-key checks. The delivery key used was `match_id`, `innings`, `overid`, `over_ball_num`, `innings_ball_num`.

The `ball_by_ball.csv` file was exported from the in-memory `df_source_ball` object. It is not certified as a byte-for-byte copy of an original source CSV.

## Reference source files copied

- `matches.csv`
- `playing_eleven.csv`
- `competition_team_player_mapping.csv`
- `teams.csv`
- `competitions.csv`

All five source files were present and copied successfully.

## Reports and audit registers copied

- 3 project reports: Phase 0, Phase 1 and Phase 2.
- 6 Phase 3 reports: Phase 3B through Phase 3G.
- 3 Phase 2 audit registers.
- 245 Phase 3 CSV audit registers.

Total copied report/register files: 257. All copied files passed source-to-destination file-size comparisons; no missing source files or size mismatches were reported.

No Phase 3A Markdown report was present in the inventoried Colab paths. Available Phase 3A CSV audit evidence was copied. This report gap is disclosed and has not been filled by fabrication.

## Inherited project limitations

Phase 3 did not produce a validated numerical pressure model.

Key unresolved limitations:
- Run-field semantics are unresolved.
- `extras1_runs` is categorical, not a numeric extras amount.
- `extras_2_runs` semantics are unresolved.
- Cumulative score is unavailable.
- Legal-ball progression is unavailable/unresolved.
- Scheduled innings length and stage are unavailable or unverified.
- First-innings scoring position and expected cumulative scoring benchmark are unavailable.
- Second-innings runs required and required run rate are not constructible.
- Wicket-state reconstruction is conditional.
- No pressure scale, weights, bands or thresholds are approved.
- No cross-format fallback or cross-innings calibration is approved.

See the Phase 0–3 reports and the copied audit registers for details.

## Phase 4–7 handoff

The package includes:
- `SupeRR_Selector_S2_Master_Handoff.md`
- `SupeRR_Selector_S2_Phase4_Continuation_Prompt.md`
- `README.md`

Phase 4 must begin with a feasibility/dependency audit. Player-level metrics must not be labelled pressure-conditioned unless the required pre-delivery pressure state can be validly reconstructed and approved.

## Validation summary

| Check | Result |
|---|---|
| Integrated Parquet readback dimensions | PASS |
| First-innings Parquet readback dimensions | PASS |
| Second-innings Parquet readback dimensions | PASS |
| Delivery-key duplicate/missing checks | PASS |
| Five reference CSV copies | PASS |
| 257 report/register copy-size checks | PASS |

## Approval

**Phase 3H status: PENDING USER APPROVAL**

This report records export and handoff preparation. It does not claim formal user sign-off, nor does it change the Phase 3 feasibility decisions.
"""

documents = {
    package / "README.md": readme,
    package / "SupeRR_Selector_S2_Master_Handoff.md": master_handoff,
    package / "SupeRR_Selector_S2_Phase4_Continuation_Prompt.md": continuation_prompt,
    package / "reports" / "SupeRR_Selector_S2_Phase_3H_Report.md": phase3h_report,
}

for path, content in documents.items():
    path.write_text(content.strip() + "\n", encoding="utf-8")

print("HANDOFF DOCUMENT CREATION AUDIT")
for path in documents:
    print({
        "file": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None
    })

assert all(path.exists() and path.stat().st_size > 0 for path in documents)

print("\nDOCUMENTS CREATED:", len(documents))
print("PHASE 3H.6 DOCUMENT CREATION: PASSED")

HANDOFF DOCUMENT CREATION AUDIT
{'file': '/content/SupeRR_Selector_S2_Phase3H_Handoff/README.md', 'exists': True, 'size_bytes': 2128}
{'file': '/content/SupeRR_Selector_S2_Phase3H_Handoff/SupeRR_Selector_S2_Master_Handoff.md', 'exists': True, 'size_bytes': 16392}
{'file': '/content/SupeRR_Selector_S2_Phase3H_Handoff/SupeRR_Selector_S2_Phase4_Continuation_Prompt.md', 'exists': True, 'size_bytes': 5958}
{'file': '/content/SupeRR_Selector_S2_Phase3H_Handoff/reports/SupeRR_Selector_S2_Phase_3H_Report.md', 'exists': True, 'size_bytes': 3550}

DOCUMENTS CREATED: 4
PHASE 3H.6 DOCUMENT CREATION: PASSED
